# Support figures — reorganized to match the article

This notebook is a re-ordered view of `Demo_post_processing.ipynb`.
Section 0 holds all data preparation (imports, helpers, data load, feature table,
pooling, large 2.5 mM example, PCA reference space, WT clustering setup, Q estimation).
Every other plotting cell is grouped under the figure (Section N == Figure N)
or supplementary (Section S{N}) it serves.
Each code cell carries a leading `# Figure X panel Y` annotation comment.
PDF output names follow `Fig{N}_{panels}_{slug}.pdf` (or `FigS{N}_…`).

## Table of contents

- Section 0 — Data preparation (imports, helpers, data load, feature table,
  pooling, large 2.5 mM example, PCA reference space, WT clustering setup)
- Section 1 — Figure 1: Representative bouton readouts and quantal size
- Section 2 — Figure 2: Four clusters of short-term plasticity
- Section 3 — Figure 3: Heterogeneity & quantal release (N, P, refilling)
- Section 4 — Figure 4: Calcium dependence (1.5 vs 4 mM)
- Section 5 — Figure 5: Frequency dependence (50 Hz)
- Section 6 — Figure 6: Synapsin II perturbation
- Section 7 — Figure 7: Diversity along single PFs
- Section 8 — Figure 8: Postsynaptic target identity
- Section S2 — Supp Fig S2: NNLS method overview
- Section S6 — Supp Fig S6: Stability of glutamate release with time
- Section S7 — Supp Fig S7: HCPC dendrogram & Random Forest validation
- Section S8 — Supp Fig S8: Sex distribution control
- Section S9 — Supp Fig S9: PPR vs A1 & Fail correlations across Ca²⁺
- Section S10 — Supp Fig S10: Sensor saturation control
- Supp — Quantal & failure detection method (technical figures)
- Extras — Additional WT descriptive analyses
- Extras — Synapsin II follow-up analyses (not in main figures)


## Section 0 — Data preparation (imports, loading, PCA & clustering setup)


# Bouton-Specific Diversity of Glutamate Release from Single Axons


This notebook assembles the analyses and figure panels supporting the manuscript on bouton-specific diversity of glutamate release along cerebellar parallel fibers. WT boutons define the reference description of release strength and short-term plasticity, and perturbation datasets are interpreted relative to that common framework.

The notebook is organized to follow the logic of the Results section: establish the WT reference space, define WT bouton classes, interpret those classes in mechanistic terms, then examine fiber organization, target identity, calcium, stimulation frequency, Synapsin II loss, and recording stability.


### Shared resources and reusable tables

The opening section loads the processed traces, bouton-level metrics, target annotations, and shared plotting utilities used throughout the notebook. These cells establish the common data structures that support the WT reference analyses and all later perturbation comparisons.

Methodological note: the analyses combine bouton-level summary metrics extracted from linescan recordings with normalized average traces and trial-level event tables. This makes it possible to move from descriptive fluorescence features to mechanistic interpretations of release probability, apparent release-site occupancy, and short-term plasticity.


#### General imports

The import cell collects the numerical, statistical, geometric, and plotting tools needed for the full notebook. Optional dependencies are handled explicitly so that the main manuscript analyses remain executable even when secondary plotting packages are unavailable.


In [ ]:
# === Helper — General imports ===

# Import standard library, numerical, statistics, PCA, clustering, and plotting dependencies used throughout the notebook.
import json
import os, re
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

# Scientific computing and data analysis
import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage, set_link_color_palette
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from scipy.spatial.distance import cdist
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

# Data visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from matplotlib.colors import BoundaryNorm, ListedColormap, to_hex
from matplotlib.cm import Set1
from matplotlib.patches import Ellipse
from matplotlib.widgets import Button

# Geometric and statistical analysis tools
try:
    import alphashape
except ModuleNotFoundError:
    alphashape = None

from shapely.geometry import MultiPolygon, Point, Polygon as ShapelyPolygon
from shapely.affinity import scale as shp_scale
from shapely.prepared import prep

try:
    from statannotations.Annotator import Annotator
except ModuleNotFoundError:
    Annotator = None


#### Data sources and paths

The file paths connect the notebook to the processed feature tables, trace summaries, and target-annotation tables generated upstream. Keeping the paths explicit makes it clear which tables define the WT reference dataset and which tables are only used for perturbation overlays.


In [ ]:
# === Helper — Data sources and paths ===

# Define input filenames, source directories, and shared path constants for all downstream analyses.
#BASE_DIR            = Path(r"C:\Users\Anthime.PERROT\PPR_DATA_AND_CODE\Stability_After_temp_t_delete_later")
BASE_DIR            = Path(r"C:\Users\Antoine.Valera\Desktop\PPR_DATA_FINAL")  # Base directory for data and code

PPR_FILENAME        = 'summary.xlsx'           # PCA features file
PPR_TRIALS_FILENAME = 'summary_trials.xlsx'  # Per-trial failure data
PPR_TRIALS_NNLS_NULL_FILENAME = 'summary_trials_nnls_null.xlsx'  # Packed per-trial sliding-NNLS baseline amplitudes
TRACES_FILENAME     = 'summary_traces.xlsx'  # Preprocessed average traces (generated by extract_metrics)
TIMES_FILENAME      = 'summary_times.xlsx'   # Time vectors for traces (generated by extract_metrics)
TARGET_MAP_FILENAME = 'Target_WT_pooled.xlsx'  # Bouton target identity mapping
OUTPUT_DIR          = BASE_DIR / 'output'

# ------------------------------------------------------------------
# Central condition registry (single source of truth)
# ------------------------------------------------------------------
STABILITY_BEFORE_CONDITIONS = ['Stability_Before', 'Stability_Before_05']
STABILITY_AFTER_CONDITIONS = ['Stability_After', 'Stability_After_05']
WT_2_5_20HZ_CONDITIONS = ['WT_Theo', 'WT_Theo_1scd', 'WT_Anthime'] + STABILITY_BEFORE_CONDITIONS
STABILITY_2_5_20HZ_CONDITIONS = STABILITY_BEFORE_CONDITIONS + STABILITY_AFTER_CONDITIONS
STABILITY_OUTLIER_IDS = ['241212_Fibre1_PortionB_bouton9', '241212_Fibre1_PortionB_Bouton_9_bis']
CALCIUM_2_5_20HZ_ALL_CONDITIONS = WT_2_5_20HZ_CONDITIONS + STABILITY_AFTER_CONDITIONS
CALCIUM_LEVEL_ALIASES = {
    '1.5': '1.5mM',
    '2.5': '2.5mM',
    '4': '4mM',
    '4.0': '4mM',
}


def _ordered_unique(seq):
    return list(dict.fromkeys(seq))


def _duplicates(seq):
    seen = set()
    dup = []
    for item in seq:
        if item in seen and item not in dup:
            dup.append(item)
        seen.add(item)
    return dup


def _assert_no_duplicates(group_name, seq):
    dup = _duplicates(seq)
    if dup:
        raise ValueError(f"Duplicate conditions in '{group_name}': {dup}")


def combine_condition_groups(*groups):
    merged = []
    for group in groups:
        merged.extend(list(group))
    _assert_no_duplicates('combined groups', merged)
    return merged


_assert_no_duplicates('WT_2_5_20HZ_CONDITIONS', WT_2_5_20HZ_CONDITIONS)
_assert_no_duplicates('STABILITY_BEFORE_CONDITIONS', STABILITY_BEFORE_CONDITIONS)
_assert_no_duplicates('STABILITY_AFTER_CONDITIONS', STABILITY_AFTER_CONDITIONS)
_assert_no_duplicates('CALCIUM_2_5_20HZ_ALL_CONDITIONS', CALCIUM_2_5_20HZ_ALL_CONDITIONS)

CALCIUM_CONDITION_GROUPS = {
    '1.5mM': {'20Hz': ['Theo_1_5Ca'], '50Hz': ['Theo_1_5_50Hz']},
    '2.5mM': {
        '20Hz': CALCIUM_2_5_20HZ_ALL_CONDITIONS,
        '50Hz': ['Theo_2_5_50Hz'],
    },
    '4mM': {'20Hz': ['Theo_4Ca'], '50Hz': ['Theo_4_50Hz']},
}
SYNAPSIN_CONDITIONS = ['SynII']

# ------------------------------------------------------------------
# Central display palette (single source of truth)
# ------------------------------------------------------------------
WT_CA_COLORS = {
    '1.5mM': "#59baff",
    '2.5mM': "#ffa759",
    '4mM': "#ff3030",
}
FREQ_50_COLORS = {
    '1.5mM': "#2048fa",
    '2.5mM': "#c97018",
    '4mM': "#c00000",
}
SYNII_COLOR = "#af37ff"
WT_THEO_COLOR = "#727272"
WT_ANTHIME_COLOR = "#b9b9b9"
WT_THEO_1SCD_COLOR = WT_CA_COLORS['2.5mM']
STABILITY_BEFORE_COLOR = WT_CA_COLORS['2.5mM']
STABILITY_AFTER_COLOR = "#ffd037"
PC_TARGET_COLOR = "#13ca02"
PC_TARGET_MEDIAN_COLOR = "#0fb400"
IN_TARGET_COLOR = "#ba67bd"
IN_TARGET_MEDIAN_COLOR = "#955597"

def _normalize_calcium_key(calcium_level):
    calcium_raw = str(calcium_level).strip().replace(' ', '')
    calcium_key = calcium_raw.lower()
    for suffix in ('mmca', 'mm'):
        if calcium_key.endswith(suffix):
            calcium_key = calcium_key[:-len(suffix)]
            break
    return CALCIUM_LEVEL_ALIASES.get(calcium_key, calcium_raw)

def _normalize_calcium(calcium_level):
    return _normalize_calcium_key(calcium_level)

def get_wt_ca_color(calcium_level):
    return WT_CA_COLORS[_normalize_calcium_key(calcium_level)]

def get_50hz_ca_color(calcium_level):
    return FREQ_50_COLORS[_normalize_calcium_key(calcium_level)]

def get_source_condition_color(condition_name):
    source_colors = {
        'WT_Theo': WT_THEO_COLOR,
        'WT_Anthime': WT_ANTHIME_COLOR,
        'WT_Theo_1scd': WT_THEO_1SCD_COLOR,
        'SynII': SYNII_COLOR,
    }
    return source_colors[condition_name]

def get_stability_condition_color(condition_name):
    if 'Before' in str(condition_name):
        return STABILITY_BEFORE_COLOR
    if 'After' in str(condition_name):
        return STABILITY_AFTER_COLOR
    raise KeyError(condition_name)

def get_target_identity_color(target_name):
    target_key = str(target_name).strip().upper()
    if target_key == 'PC' or target_key.endswith('_PC') or target_key.startswith('PC_'):
        return PC_TARGET_COLOR
    if target_key == 'IN' or target_key.endswith('_IN') or target_key.startswith('IN_'):
        return IN_TARGET_COLOR
    raise KeyError(target_name)

def get_calcium_conditions(calcium_level, frequency='all'):
    calcium_raw = str(calcium_level).strip().replace(' ', '')
    calcium_key = calcium_raw.lower()
    for suffix in ('mmca', 'mm'):
        if calcium_key.endswith(suffix):
            calcium_key = calcium_key[:-len(suffix)]
            break

    calcium_norm = CALCIUM_LEVEL_ALIASES.get(calcium_key, calcium_raw)
    if calcium_norm not in CALCIUM_CONDITION_GROUPS:
        raise KeyError(f"Unknown calcium level: {calcium_level}")

    freq_key = str(frequency).strip().lower().replace(' ', '')
    if freq_key in ('all', '*', 'any'):
        return _ordered_unique(
            CALCIUM_CONDITION_GROUPS[calcium_norm]['20Hz']
            + CALCIUM_CONDITION_GROUPS[calcium_norm]['50Hz']
        )
    if freq_key in ('20hz', '20'):
        return list(CALCIUM_CONDITION_GROUPS[calcium_norm]['20Hz'])
    if freq_key in ('50hz', '50'):
        return list(CALCIUM_CONDITION_GROUPS[calcium_norm]['50Hz'])
    raise KeyError(f"Unknown frequency selector: {frequency}")


def get_synapsin_conditions():
    return list(SYNAPSIN_CONDITIONS)


# ------------------------------------------------------------------
# Central condition/group aliases
# ------------------------------------------------------------------
CONDITION_POOLS = {
    'WT_pooled': WT_2_5_20HZ_CONDITIONS,
    'stability_before': STABILITY_BEFORE_CONDITIONS,
    'stability_after': STABILITY_AFTER_CONDITIONS,
}

FRIENDLY_CONDITION_GROUPS = {
    '1.5mM_20Hz': get_calcium_conditions('1.5mM', '20Hz'),
    '1.5mM_50Hz': get_calcium_conditions('1.5mM', '50Hz'),
    '1.5mM_all': get_calcium_conditions('1.5mM', 'all'),
    '2.5mM_20Hz': get_calcium_conditions('2.5mM', '20Hz'),
    '2.5mM_20Hz_WT': WT_2_5_20HZ_CONDITIONS,
    '2.5mM_20Hz_Stability': STABILITY_2_5_20HZ_CONDITIONS,
    '2.5mM_50Hz': get_calcium_conditions('2.5mM', '50Hz'),
    '2.5mM_all': get_calcium_conditions('2.5mM', 'all'),
    '4mM_20Hz': get_calcium_conditions('4mM', '20Hz'),
    '4mM_50Hz': get_calcium_conditions('4mM', '50Hz'),
    '4mM_all': get_calcium_conditions('4mM', 'all'),
    'Synapsin': get_synapsin_conditions(),
}

def get_friendly_conditions(group_name):
    if group_name not in FRIENDLY_CONDITION_GROUPS:
        raise KeyError(f"Unknown group '{group_name}'")
    return list(FRIENDLY_CONDITION_GROUPS[group_name])

for _name, _group in FRIENDLY_CONDITION_GROUPS.items():
    _assert_no_duplicates(_name, _group)


def filter_df_by_conditions(dataframe, conditions, condition_col='Condition'):
    return dataframe[dataframe[condition_col].isin(list(conditions))].copy()


def filter_df_by_calcium(dataframe, calcium_level, frequency='all', include_synapsin=False, condition_col='Condition'):
    selected = get_calcium_conditions(calcium_level, frequency)
    if include_synapsin:
        selected = _ordered_unique(selected + get_synapsin_conditions())
    return filter_df_by_conditions(dataframe, selected, condition_col=condition_col)



# ------------------------------------------------------------------
# DataFrame enrichment and selection utilities
# ------------------------------------------------------------------

def _build_condition_metadata_map():
    metadata = {}

    for cond in get_calcium_conditions('1.5mM', '20Hz'):
        metadata[cond] = {'Ca_mM': 1.5, 'Freq_Hz': 20, 'ConditionFamily': 'Calcium'}
    for cond in get_calcium_conditions('1.5mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 1.5, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in WT_2_5_20HZ_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'WT'}
    for cond in STABILITY_BEFORE_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityBefore'}
    for cond in STABILITY_AFTER_CONDITIONS:
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityAfter'}
    for cond in get_calcium_conditions('2.5mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in get_calcium_conditions('4mM', '20Hz'):
        metadata[cond] = {'Ca_mM': 4.0, 'Freq_Hz': 20, 'ConditionFamily': 'Calcium'}
    for cond in get_calcium_conditions('4mM', '50Hz'):
        metadata[cond] = {'Ca_mM': 4.0, 'Freq_Hz': 50, 'ConditionFamily': 'Calcium'}

    for cond in get_synapsin_conditions():
        metadata[cond] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'Synapsin'}

    # Pooled labels
    metadata['WT_pooled'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'WTPooled'}
    metadata['stability_before'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityBeforePooled'}
    metadata['stability_after'] = {'Ca_mM': 2.5, 'Freq_Hz': 20, 'ConditionFamily': 'StabilityAfterPooled'}
    return metadata


def _normalize_bouton_id(value):
    if pd.isna(value):
        return value
    return str(value).strip().replace('_traces_converted', '')


def _extract_fiber_id(value, n_chars=22):
    if pd.isna(value):
        return value
    s = _normalize_bouton_id(value)
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    if m:
        s = m.group(1).rstrip('_ ')
    s = re.sub(r'[_\s]*Portion[A-Za-z0-9]+\s*$', '', str(s), flags=re.IGNORECASE).rstrip('_ ')
    return str(s)


def add_condition_metadata(dataframe, condition_col='Condition'):
    d = dataframe.copy()
    condition_meta = _build_condition_metadata_map()
    cond_series = d[condition_col].astype(str)
    d['Ca_mM'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('Ca_mM', np.nan))
    d['Freq_Hz'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('Freq_Hz', np.nan))
    d['ConditionFamily'] = cond_series.map(lambda c: condition_meta.get(c, {}).get('ConditionFamily', 'Unknown'))
    return d


def enrich_features_dataframe(dataframe):
    d = dataframe.copy()
    d['ID'] = d['ID'].astype(str).str.strip()
    d['BaseID'] = d['ID'].map(_normalize_bouton_id)
    d['FiberID'] = d['ID'].map(_extract_fiber_id)
    d = add_condition_metadata(d, condition_col='Condition')
    return d


def enrich_traces_dataframe(dataframe, time_already_aligned=False):
    d = dataframe.copy()
    d['ID'] = d['ID'].astype(str).str.strip()
    d['BaseID'] = d['ID'].map(_normalize_bouton_id)
    d['FiberID'] = d['ID'].map(_extract_fiber_id)
    d = add_condition_metadata(d, condition_col='Condition')
    baseline_offset = float(globals().get('EXCEPTIONAL_BASELINE_OFFSET', 0.5))
    d['TrainStart_s'] = d['Condition'].map(lambda c: 1.0 - baseline_offset if c in EXCEPTIONAL_CONDITIONS else 1.0)
    d['TimeRaw'] = d['Time'].map(lambda values: np.asarray(values, dtype=float))
    if time_already_aligned:
        d['TimeOffset'] = d['TimeRaw'].map(lambda values: np.asarray(values, dtype=float))
    else:
        d['TimeOffset'] = [
            np.asarray(values, dtype=float) + (1.0 - float(train_start))
            for values, train_start in zip(d['TimeRaw'], d['TrainStart_s'])
        ]
    return d


def build_condition_views(dataframe, condition_col='Condition'):
    return {
        cond: sub.copy()
        for cond, sub in dataframe.groupby(condition_col, sort=False)
    }


def make_pooled_features_dataframe(features_raw, pools):
    d = features_raw.copy()
    existing_conditions = set(d['Condition'].unique())
    for pool_name, source_conditions in pools.items():
        if pool_name in existing_conditions:
            continue
        available_sources = [c for c in source_conditions if c in existing_conditions]
        if not available_sources:
            continue
        pooled_parts = []
        seen_ids = set()
        for source_condition in available_sources:
            source_rows = d[d['Condition'] == source_condition].copy()
            source_norm_ids = source_rows['ID'].map(_normalize_bouton_id)
            keep_mask = ~source_norm_ids.isin(seen_ids)
            source_rows = source_rows[keep_mask].copy()
            seen_ids.update(source_rows['ID'].map(_normalize_bouton_id))
            pooled_parts.append(source_rows)
        if len(pooled_parts) == 0:
            continue
        pooled_data = pd.concat(pooled_parts, ignore_index=True)
        pooled_data['Condition'] = pool_name
        d = pd.concat([d, pooled_data], ignore_index=True)
        existing_conditions.add(pool_name)
    return d


def recompute_failure_summary_from_trials(trials_df, n_fail_cols=3, threshold_col='thr_shared'):
    """Temporary notebook-side recomputation of %Fail from summary_trials.xlsx."""
    d = trials_df.copy()
    if 'condition' not in d.columns or 'file' not in d.columns:
        raise KeyError("Trials dataframe must contain 'condition' and 'file' columns")

    d['Condition'] = d['condition'].astype(str).str.strip()
    d['BaseID'] = d['file'].map(_normalize_bouton_id)

    rows = []
    for (condition_name, base_id), grp in d.groupby(['Condition', 'BaseID'], sort=False):
        out = {'Condition': condition_name, 'BaseID': base_id}
        for k in range(1, n_fail_cols + 1):
            corr_col = f'AMP{k}_CORR'
            uncorr_col = f'AMP{k}_UNCORR'
            n_fail = 0
            n_valid = 0
            for _, row in grp.iterrows():
                thr = pd.to_numeric(pd.Series([row.get(threshold_col, np.nan)]), errors='coerce').iloc[0]
                if not np.isfinite(thr):
                    continue
                vals = []
                if corr_col in grp.columns:
                    v_corr = pd.to_numeric(pd.Series([row.get(corr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_corr):
                        vals.append(float(v_corr))
                if uncorr_col in grp.columns:
                    v_uncorr = pd.to_numeric(pd.Series([row.get(uncorr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_uncorr):
                        vals.append(float(v_uncorr))
                if len(vals) == 0:
                    continue
                n_valid += 1
                if np.min(vals) < float(thr):
                    n_fail += 1
            out[f'%Fail{k}'] = round((100.0 * n_fail / n_valid), 2) if n_valid else np.nan
        rows.append(out)
    return pd.DataFrame(rows)


def recompute_event_failure_rates_from_trials(trials_df, max_pulse_number=10, threshold_col='thr_shared'):
    """Compute canonical eventwise failure rates (0-1) from trials for pulses 1..max_pulse_number."""
    d = trials_df.copy()
    if 'condition' not in d.columns or 'file' not in d.columns:
        raise KeyError("Trials dataframe must contain 'condition' and 'file' columns")

    d['Condition'] = d['condition'].astype(str).str.strip()
    d['BaseID'] = d['file'].map(_normalize_bouton_id)

    rows = []
    for (condition_name, base_id), grp in d.groupby(['Condition', 'BaseID'], sort=False):
        out = {'Condition': condition_name, 'BaseID': base_id}
        for k in range(1, max_pulse_number + 1):
            corr_col = f'AMP{k}_CORR'
            uncorr_col = f'AMP{k}_UNCORR'
            fail_count = 0
            valid_count = 0
            for _, row in grp.iterrows():
                thr = pd.to_numeric(pd.Series([row.get(threshold_col, np.nan)]), errors='coerce').iloc[0]
                if not np.isfinite(thr):
                    continue
                vals = []
                if corr_col in grp.columns:
                    v_corr = pd.to_numeric(pd.Series([row.get(corr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_corr):
                        vals.append(float(v_corr))
                if uncorr_col in grp.columns:
                    v_uncorr = pd.to_numeric(pd.Series([row.get(uncorr_col, np.nan)]), errors='coerce').iloc[0]
                    if np.isfinite(v_uncorr):
                        vals.append(float(v_uncorr))
                if len(vals) == 0:
                    continue
                valid_count += 1
                if np.min(vals) < float(thr):
                    fail_count += 1
            out[f'FailRate{k}'] = (fail_count / valid_count) if valid_count else np.nan
        rows.append(out)
    return pd.DataFrame(rows)


def attach_event_failure_rates_to_features(features_df, trials_df=None, max_pulse_number=10, verbose=True):
    """Attach canonical FailRate1..N columns to the feature dataframe without touching %Fail columns."""
    d = features_df.copy()
    if 'Condition' not in d.columns:
        raise KeyError("Features dataframe must contain a 'Condition' column")
    if 'BaseID' not in d.columns:
        if 'ID' not in d.columns:
            raise KeyError("Features dataframe must contain either 'BaseID' or 'ID'")
        d['BaseID'] = d['ID'].map(_normalize_bouton_id)

    if trials_df is None:
        trials_df = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)

    fail_rates = recompute_event_failure_rates_from_trials(
        trials_df,
        max_pulse_number=max_pulse_number,
    )
    d = d.merge(fail_rates, on=['Condition', 'BaseID'], how='left')

    if verbose:
        rate_cols = [f'FailRate{k}' for k in range(1, max_pulse_number + 1)]
        matched = pd.to_numeric(d[rate_cols[0]], errors='coerce').notna().sum() if rate_cols[0] in d.columns else 0
        print('Canonical trial-derived failure rates attached to features:')
        print(f'  matched feature rows: {matched}/{len(d)}')
        print(f'  added columns: {rate_cols[0]} .. {rate_cols[-1]}')

    return d, fail_rates


def apply_recomputed_failures_to_features(features_df, trials_df=None, n_fail_cols=3, verbose=True):
    """Patch summary-derived feature rows with trial-recomputed %Fail columns."""
    d = features_df.copy()
    if 'Condition' not in d.columns:
        raise KeyError("Features dataframe must contain a 'Condition' column")
    if 'BaseID' not in d.columns:
        if 'ID' not in d.columns:
            raise KeyError("Features dataframe must contain either 'BaseID' or 'ID'")
        d['BaseID'] = d['ID'].map(_normalize_bouton_id)

    if trials_df is None:
        trials_df = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)

    fail_summary = recompute_failure_summary_from_trials(
        trials_df,
        n_fail_cols=n_fail_cols,
    )

    before_cols = {col: d[col].copy() for col in [f'%Fail{k}' for k in range(1, n_fail_cols + 1)] if col in d.columns}
    d = d.merge(fail_summary, on=['Condition', 'BaseID'], how='left', suffixes=('', '_recomputed'))

    updated_counts = {}
    for k in range(1, n_fail_cols + 1):
        col = f'%Fail{k}'
        new_col = f'{col}_recomputed'
        if new_col not in d.columns:
            continue
        old_vals = before_cols.get(col, pd.Series(np.nan, index=d.index))
        has_new = np.isfinite(pd.to_numeric(d[new_col], errors='coerce'))
        d[col] = d[new_col].where(has_new, d[col] if col in d.columns else np.nan)
        updated_counts[col] = int(np.sum(has_new & (~np.isclose(pd.to_numeric(old_vals, errors='coerce'), pd.to_numeric(d[col], errors='coerce'), equal_nan=True))))
        d = d.drop(columns=[new_col])

    if verbose:
        n_match = int(d[['Condition', 'BaseID']].merge(fail_summary[['Condition', 'BaseID']].drop_duplicates(), on=['Condition', 'BaseID'], how='inner').shape[0])
        print("Temporary failure correction from trials:")
        print(f"  matched feature rows: {n_match}/{len(d)}")
        for col in sorted(updated_counts):
            print(f"  {col}: updated {updated_counts[col]} rows")

    return d, fail_summary


def get_ppr_columns(dataframe, max_pulse=10):
    return [
        f'PPR{i}/1'
        for i in range(2, max_pulse + 1)
        if f'PPR{i}/1' in dataframe.columns
    ]


# Data filtering and analysis parameters
EXCEPTIONAL_CONDITIONS = combine_condition_groups(
    STABILITY_BEFORE_CONDITIONS[1:],
    STABILITY_AFTER_CONDITIONS[1:],
    get_calcium_conditions('1.5mM', 'all'),
    get_calcium_conditions('4mM', 'all'),
    get_calcium_conditions('2.5mM', '50Hz'),
    ['WT_Theo'],
)
PCA_KEEP_COLS          = ['AMP1', 'AMP2', '%Fail1', '%Fail2', 'PPR2/1', 'PPR3/1', 'PPR4/1', 'PPR5/1', 'PPR6/1', 'PPR7/1', 'PPR8/1', 'PPR9/1', 'PPR10/1']
N_CLUSTERS             = 4                         # Number of clusters for analysis --> obtained from final RF analysis and kept fixed for all clustering methods for consistency

# Trace processing parameters (used for visualization only - traces are pre-processed)
STIM_SHIFT  = 0.5                        # Stimulus time offset (seconds)
CROP_END    = 2.0                          # Trace duration to keep (seconds)
SAMPLE_RATE = 1000                      # Target sampling rate (Hz)
N_SAMPLES   = int(CROP_END * SAMPLE_RATE) + 1
COMMON_TIME = np.linspace(0, CROP_END, N_SAMPLES)  # Standardized time vector

# Shared trace-source switches
TRACE_MEAN_SOURCE = 'raw_nearest'   # 'normalized' or 'raw_nearest'
TRACE_SINGLE_SOURCE = 'raw'         # 'raw' or 'normalized'
TRACE_RAW_NEAREST_TOL_FACTOR = 0.51
TRACE_XLIM_20HZ = (0.8, 1.7)


#### Output directory and shared helpers

Figure and table export helpers are defined once here so that all later panels are saved reproducibly. The output names are aligned to the notebook order to simplify manuscript assembly and figure curation.


In [ ]:
# === Helper — Output directory and shared filesystem helpers ===
# Create the export directory and define small helper functions for saving figures and resolving local paths.
OUTPUT_FALLBACK_DIR = Path.cwd() / "_exported_figures"

def _probe_writable_directory(directory):
    directory = Path(directory)
    try:
        directory.mkdir(parents=True, exist_ok=True)
        probe = directory / '.__write_probe__.tmp'
        probe.write_text('ok', encoding='utf-8')
        probe.unlink()
        return True
    except Exception:
        return False


def _timestamped_path(path_obj, tag='autosave'):
    path_obj = Path(path_obj)
    stem = path_obj.stem
    suffix = path_obj.suffix
    timestamp = pd.Timestamp.now().strftime('%Y%m%d_%H%M%S')
    candidate = path_obj.with_name(f'{stem}__{tag}_{timestamp}{suffix}')
    counter = 1
    while candidate.exists():
        candidate = path_obj.with_name(f'{stem}__{tag}_{timestamp}_{counter}{suffix}')
        counter += 1
    return candidate


def _as_pathlike_target(target):
    try:
        return Path(target)
    except Exception:
        return None


def _replace_savefig_target(args, kwargs, new_target):
    args = list(args)
    kwargs = dict(kwargs)
    if len(args) >= 1:
        args[0] = new_target
    else:
        kwargs['fname'] = new_target
    return tuple(args), kwargs


def _permission_retry_path(target, prefer_same_dir=True):
    path_obj = _as_pathlike_target(target)
    if path_obj is None:
        return None
    if prefer_same_dir:
        try:
            path_obj.parent.mkdir(parents=True, exist_ok=True)
            return _timestamped_path(path_obj, tag='retry')
        except Exception:
            pass
    fallback_dir = OUTPUT_FALLBACK_DIR
    fallback_dir.mkdir(parents=True, exist_ok=True)
    return _timestamped_path(fallback_dir / path_obj.name, tag='fallback')


def write_excel_sheets(target, sheets, *, engine='openpyxl', index=False):
    resolved_target = target
    try:
        with pd.ExcelWriter(resolved_target, engine=engine) as writer:
            for sheet_name, dataframe in sheets.items():
                dataframe.to_excel(writer, sheet_name=sheet_name, index=index)
        return resolved_target
    except PermissionError as exc:
        raise PermissionError(f'Cannot save {target}: file open, close it and retry.') from exc


if not _probe_writable_directory(OUTPUT_DIR):
    OUTPUT_FALLBACK_DIR.mkdir(parents=True, exist_ok=True)
    print(f'OUTPUT_DIR not writable, switching to fallback: {OUTPUT_FALLBACK_DIR}')
    OUTPUT_DIR = OUTPUT_FALLBACK_DIR
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def is_bouton_file(file_path: Path) -> bool:
    """Check if Excel file contains bouton trace data (excludes metadata files)"""
    metadata_files = {PPR_FILENAME.lower(), PPR_TRIALS_FILENAME.lower(), TARGET_MAP_FILENAME.lower()}
    return (file_path.suffix.lower() == '.xlsx' and 
            not file_path.name.startswith('~$') and 
            file_path.name.lower() not in metadata_files)

def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')


#### Load preprocessed traces

The trace-loading step reconstructs the average glutamate transients used for the trace panels throughout the notebook. These traces retain the timing structure of the stimulation trains and provide the most direct view of bouton-level glutamate release dynamics.


In [ ]:
# === Helper — Load preprocessed traces ===
# Load preprocessed mean-trace tables and aligned time vectors exported by the feature-extraction pipeline.
## (generated by export_folders_to_excel with save_traces=True)

traces_file = BASE_DIR / TRACES_FILENAME
times_file = BASE_DIR / TIMES_FILENAME

if not traces_file.exists():
    raise FileNotFoundError(
        f"Traces file not found: {traces_file}\n"
        "Run extract_metrics.export_folders_to_excel() with save_traces=True to generate it."
    )
if not times_file.exists():
    raise FileNotFoundError(
        f"Times file not found: {times_file}\n"
        "Run extract_metrics.export_folders_to_excel() with save_traces=True to generate it."
    )

# Load all sheets (one per condition) from both Excel files
excel_traces = pd.ExcelFile(traces_file)
excel_times = pd.ExcelFile(times_file)
experimental_conditions = excel_traces.sheet_names
raw_traces_data = []

for condition_name in experimental_conditions:
    trace_df = pd.read_excel(traces_file, sheet_name=condition_name)
    time_df = pd.read_excel(times_file, sheet_name=condition_name)
    
    # All columns are bouton IDs (no Time column in traces file)
    bouton_columns = list(trace_df.columns)
    
    print(f"Loading {condition_name}: {len(bouton_columns)} boutons")
    
    for bouton_id in bouton_columns:
        avg_trace = trace_df[bouton_id].to_numpy(float)
        # Get corresponding time vector from times file
        if bouton_id in time_df.columns:
            time_array = time_df[bouton_id].to_numpy(float)
        else:
            # Fallback: use index as time (should not happen with consistent files)
            print(f"Warning: {bouton_id} not found in times file, using index")
            time_array = np.arange(len(avg_trace)) / SAMPLE_RATE
        
        raw_traces_data.append({
            'ID': str(bouton_id),
            'Condition': condition_name,
            'Time': time_array.tolist(),
            'Avg': avg_trace.tolist(),
            'n_trials': 1,  # Already averaged
            'FilePath': str(traces_file),
        })

excel_traces.close()
excel_times.close()
RAW_TRACES_DF = pd.DataFrame(raw_traces_data)
CONDITIONS = experimental_conditions

print(f"\n=== Loaded {len(RAW_TRACES_DF)} preprocessed traces from {TRACES_FILENAME} and {TIMES_FILENAME} ===")


#### Plotting and statistical helpers

Shared formatting, PCA, and PPR helper functions are centralized here so that later figures are stylistically consistent and analytically comparable. This cell also contains the small utilities that make the same computations reusable across WT, calcium, frequency, SynII, and stability sections.


In [ ]:
# === Helper — Figure defaults and style ===
# Define manuscript-style figure sizes, plotting defaults, and shared style utilities used across panels.
PNAS_FIGURE_WIDTH_CM = {'single': 8.7, 'one_half': 11.4, 'double': 17.8}
PNAS_MAX_HEIGHT_CM = 22.5
PNAS_FIGURE_WIDTH_IN = {k: v / 2.54 for k, v in PNAS_FIGURE_WIDTH_CM.items()}
PNAS_MAX_HEIGHT_IN = PNAS_MAX_HEIGHT_CM / 2.54
PNAS_PANEL_ASPECT = {
    'simple': 0.78,
    'elongated': 0.40,
    'pca': 0.78,
    'ppr': 0.60,
    'trace': 0.42,
    'hist': 0.72,
    'boxplot_compact': 1.2580645161,
}
PNAS_PANEL_WIDTH_SCALE = {
    'simple': 1.00,
    'elongated': 1.00,
    'pca': 1.00,
    'ppr': 1.00,
    'trace': 1.00,
    'hist': 1.00,
    'boxplot_compact': 0.62,
}

# Canonical grid unit: one data panel should match the current visual size of
# the single-panel hierarchical-cluster PPR figure. Legends and inter-panel
# spacing are budgeted separately, never by shrinking the panel footprint.
GRID_PANEL_UNIT_W_IN = float(PNAS_FIGURE_WIDTH_IN['single'])
GRID_PANEL_UNIT_H_IN = GRID_PANEL_UNIT_W_IN * float(PNAS_PANEL_ASPECT['simple'])
GRID_WSPACE_IN = 0.32
GRID_HSPACE_IN = 0.26
GRID_MARGIN_LEFT_IN = 0.48
GRID_MARGIN_RIGHT_IN = 0.10
GRID_MARGIN_BOTTOM_IN = 0.42
GRID_MARGIN_TOP_IN = 0.18
GRID_LEGEND_STRIP_IN = 0.92
GRID_SUPTITLE_IN = 0.24
PNAS_FONT_MIN_PT = 6.0
PNAS_FONT_MAX_PT = 12.0
PNAS_LINEWIDTH_MIN_PT = 0.25
PNAS_LINEWIDTH_MAX_PT = 1.5
PNAS_MARKER_MIN_PT = 4.0
PNAS_MARKER_MAX_PT = 11.0
PNAS_SCATTER_AREA_MIN = PNAS_MARKER_MIN_PT ** 2
PNAS_SCATTER_AREA_MAX = 30.0
PNAS_SCATTER_AREA_DEFAULT = 30.0

def _clip_style_value(value, vmin, vmax):
    arr = np.asarray(value, dtype=float)
    clipped = np.clip(arr, vmin, vmax)
    if np.isscalar(value):
        return float(clipped)
    return clipped

def clamp_fontsize(value):
    return _clip_style_value(value, PNAS_FONT_MIN_PT, PNAS_FONT_MAX_PT)

def clamp_linewidth(value):
    return _clip_style_value(value, PNAS_LINEWIDTH_MIN_PT, PNAS_LINEWIDTH_MAX_PT)

def clamp_markersize(value):
    return _clip_style_value(value, PNAS_MARKER_MIN_PT, PNAS_MARKER_MAX_PT)

def clamp_scatter_area(value):
    """Pass-through by default: marker area remains user-controlled."""
    return value

def compute_protected_ppr_from_amplitudes(amps, noise_threshold=np.nan):
    """Floor amplitudes to a noise threshold, then compute PPR relative to A1."""
    arr = np.asarray(amps, dtype=float).copy()
    if np.isfinite(noise_threshold):
        finite_mask = np.isfinite(arr)
        arr[finite_mask] = np.maximum(arr[finite_mask], float(noise_threshold))
    if arr.size == 0:
        return arr
    a1 = float(arr[0])
    return (arr / a1) if np.isfinite(a1) and abs(a1) > 1e-12 else np.full_like(arr, np.nan)

PPR_NOISE_PROTECTION = False

def apply_protected_ppr_inplace(dataframe, amp_prefix='AMP', thr_col='NOISE_THR_MEDIAN'):
    """Overwrite existing PPRn/1 columns in place using notebook-side noise protection."""
    d = dataframe.copy()
    amp_cols = []
    idx = 1
    while f'{amp_prefix}{idx}' in d.columns:
        amp_cols.append(f'{amp_prefix}{idx}')
        idx += 1
    if not amp_cols:
        raise ValueError(f'No amplitude columns found with prefix {amp_prefix!r}.')
    if thr_col not in d.columns:
        return d
    for row_idx, (_, row) in enumerate(d.iterrows()):
        amps = pd.to_numeric(row[amp_cols], errors='coerce').to_numpy(dtype=float)
        thr = pd.to_numeric(pd.Series([row[thr_col]]), errors='coerce').iloc[0]
        ppr_vals = compute_protected_ppr_from_amplitudes(amps, noise_threshold=thr)
        for pulse_idx in range(2, len(amp_cols) + 1):
            key = f'PPR{pulse_idx}/1'
            d.at[d.index[row_idx], key] = float(ppr_vals[pulse_idx - 1]) if pulse_idx - 1 < len(ppr_vals) else np.nan
    return d

def _guess_panel_kind(panel_kind=None, nrows=1, ncols=1, figsize=None):
    if panel_kind is not None:
        return panel_kind
    if figsize is not None:
        try:
            w, h = float(figsize[0]), float(figsize[1])
            if h > 0 and (w / h) >= 2.2:
                return 'elongated'
        except Exception:
            pass
    if int(ncols) >= 4:
        return 'elongated'
    return 'simple'


def grid_figsize(nrows=1, ncols=1, panel_kind='simple', *, has_legend=False, has_suptitle=False):
    """Return a figure size whose data-panel grid is built from one fixed panel unit."""
    nrows = max(1, int(nrows))
    ncols = max(1, int(ncols))
    kind = panel_kind if panel_kind in PNAS_PANEL_ASPECT else 'simple'
    panel_w = GRID_PANEL_UNIT_W_IN * float(PNAS_PANEL_WIDTH_SCALE.get(kind, 1.0))
    panel_h = panel_w * float(PNAS_PANEL_ASPECT.get(kind, PNAS_PANEL_ASPECT['simple']))

    total_w = (panel_w * ncols) + (GRID_WSPACE_IN * max(0, ncols - 1))
    total_w += GRID_MARGIN_LEFT_IN + GRID_MARGIN_RIGHT_IN
    if has_legend:
        total_w += GRID_LEGEND_STRIP_IN

    total_h = (panel_h * nrows) + (GRID_HSPACE_IN * max(0, nrows - 1))
    total_h += GRID_MARGIN_BOTTOM_IN + GRID_MARGIN_TOP_IN
    if has_suptitle:
        total_h += GRID_SUPTITLE_IN

    return total_w, total_h


# Backward-compatible names used later in notebook.
def pnas_figsize(width='single', aspect=0.70, nrows=1):
    if isinstance(width, (int, float)):
        w = float(width)
        h = w * float(aspect) * max(1, int(nrows))
        return w, h
    return grid_figsize(nrows=max(1, int(nrows)), ncols=1, panel_kind='simple')


def recommended_panel_figsize(nrows=1, ncols=1, panel_kind='simple'):
    return grid_figsize(nrows=nrows, ncols=ncols, panel_kind=panel_kind)


def clamp_figsize_to_pnas(figsize, *, nrows=1, ncols=1, panel_kind=None):
    # Intentionally ignore any ad-hoc requested figsize: only grid helper is authoritative.
    return grid_figsize(nrows=nrows, ncols=ncols, panel_kind=_guess_panel_kind(panel_kind, nrows=nrows, ncols=ncols, figsize=figsize))


def _parse_subplots_shape(args, kwargs):
    if len(args) >= 2:
        nrows, ncols = args[0], args[1]
    elif len(args) == 1:
        nrows, ncols = args[0], 1
    else:
        nrows, ncols = kwargs.get('nrows', 1), kwargs.get('ncols', 1)
    nrows = kwargs.get('nrows', nrows)
    ncols = kwargs.get('ncols', ncols)
    return int(nrows), int(ncols)


def _figure_grid_shape(fig):
    """Infer logical panel grid from non-colorbar axes."""
    row_starts = []
    col_starts = []

    for axis in getattr(fig, 'axes', []):
        # Ignore colorbar axes: they are layout helpers, not data panels.
        if str(getattr(axis, 'get_label', lambda: '')()) == '<colorbar>':
            continue
        try:
            ss = axis.get_subplotspec()
            row_starts.append(int(ss.rowspan.start))
            col_starts.append(int(ss.colspan.start))
        except Exception:
            pass

    if row_starts and col_starts:
        nrows = len(sorted(set(row_starts)))
        ncols = len(sorted(set(col_starts)))
        return max(1, int(nrows)), max(1, int(ncols))

    # Fallback
    return 1, 1


def _outer_axis_label_visibility(ax):
    """Hide inner-axis labels only when that axis is truly shared."""
    try:
        ss = ax.get_subplotspec()
        gs = ss.get_gridspec()
        x_siblings = [sib for sib in ax.get_shared_x_axes().get_siblings(ax) if sib is not ax]
        y_siblings = [sib for sib in ax.get_shared_y_axes().get_siblings(ax) if sib is not ax]
        x_is_shared = len(x_siblings) > 0
        y_is_shared = len(y_siblings) > 0
        show_xlabel = (not x_is_shared) or (int(ss.rowspan.stop) == int(gs.nrows))
        show_ylabel = (not y_is_shared) or (int(ss.colspan.start) == 0)
        return show_xlabel, show_ylabel
    except Exception:
        return True, True

def _display_layout_rect(fig, *, has_external_legend=None, has_suptitle=None):
    """Return a tight-layout rect from fixed inch budgets instead of scaled fractions."""
    if fig is None:
        return [0.03, 0.03, 0.98, 0.99]
    if has_external_legend is None:
        has_external_legend = len(getattr(fig, 'legends', [])) > 0
    if has_suptitle is None:
        has_suptitle = getattr(fig, '_suptitle', None) is not None

    w, h = fig.get_size_inches()
    w = max(float(w), 1e-6)
    h = max(float(h), 1e-6)

    left = GRID_MARGIN_LEFT_IN / w
    bottom = GRID_MARGIN_BOTTOM_IN / h
    right = 1.0 - ((GRID_MARGIN_RIGHT_IN + (GRID_LEGEND_STRIP_IN if has_external_legend else 0.0)) / w)
    top = 1.0 - ((GRID_MARGIN_TOP_IN + (GRID_SUPTITLE_IN if has_suptitle else 0.0)) / h)

    left = min(max(left, 0.01), 0.45)
    bottom = min(max(bottom, 0.01), 0.30)
    right = min(max(right, left + 0.20), 0.99)
    top = min(max(top, bottom + 0.20), 0.99)
    return [left, bottom, right, top]

def prepare_figure_for_display(fig):
    """Apply one shared layout pass for both notebook display and export."""
    if fig is None:
        return None
    apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
    rect = _display_layout_rect(fig)
    if hasattr(type(fig), '_grid_original_tight_layout'):
        try:
            type(fig)._grid_original_tight_layout(fig, rect=rect)
        except Exception:
            pass
    try:
        fig.canvas.draw()
    except Exception:
        pass
    return fig


def apply_grid_size(fig, panel_kind='simple'):
    if fig is None:
        return None
    nrows, ncols = _figure_grid_shape(fig)
    if panel_kind is None:
        panel_kind = getattr(fig, '_grid_panel_kind', 'simple')
    w, h = grid_figsize(
        nrows=nrows,
        ncols=ncols,
        panel_kind=panel_kind,
        has_legend=len(getattr(fig, 'legends', [])) > 0,
        has_suptitle=getattr(fig, '_suptitle', None) is not None,
    )
    fig.set_size_inches(w, h, forward=True)
    return fig.get_size_inches()


def shrink_figure_to_pnas(fig, panel_kind='simple'):
    # Kept for compatibility with existing code paths.
    return apply_grid_size(fig, panel_kind=panel_kind)


def _install_pnas_matplotlib_sizers():
    from matplotlib.axes import Axes as _MplAxes
    from matplotlib.figure import Figure as _MplFigure

    if not hasattr(plt, '_grid_original_subplots'):
        plt._grid_original_subplots = plt.subplots
        plt._grid_original_figure = plt.figure
        plt._grid_original_show = plt.show

        def _grid_subplots(*args, **kwargs):
            panel_kind = kwargs.pop('panel_kind', None)
            nrows, ncols = _parse_subplots_shape(args, kwargs)
            kind = _guess_panel_kind(panel_kind, nrows=nrows, ncols=ncols)

            # Enforce single helper for all figure sizes.
            kwargs.pop('figsize', None)
            kwargs['figsize'] = grid_figsize(nrows=nrows, ncols=ncols, panel_kind=kind)

            if (nrows * ncols) > 1:
                gkw = dict(kwargs.get('gridspec_kw', {}))
                gkw.setdefault('wspace', GRID_WSPACE_IN / max(GRID_PANEL_UNIT_W_IN, 1e-6))
                gkw.setdefault('hspace', GRID_HSPACE_IN / max(GRID_PANEL_UNIT_H_IN, 1e-6))
                kwargs['gridspec_kw'] = gkw

            prev = getattr(plt, '_grid_in_subplots', False)
            plt._grid_in_subplots = True
            try:
                fig, axes = plt._grid_original_subplots(*args, **kwargs)
            finally:
                plt._grid_in_subplots = prev

            fig._grid_helper_applied = True
            fig._grid_panel_kind = kind
            return fig, axes

        def _grid_figure(*args, **kwargs):
            # If called via subplots(), keep the subplots-computed figsize.
            called_from_subplots = getattr(plt, '_grid_in_subplots', False)
            panel_kind = kwargs.pop('panel_kind', None)
            if called_from_subplots:
                return plt._grid_original_figure(*args, **kwargs)

            # Standalone figure(): use standard single-panel base size.
            kwargs.pop('figsize', None)
            fig = plt._grid_original_figure(*args, **kwargs)
            kind = _guess_panel_kind(panel_kind, nrows=1, ncols=1)
            fig.set_size_inches(*grid_figsize(1, 1, panel_kind=kind), forward=True)
            fig._grid_panel_kind = kind
            return fig

        def _grid_show(*args, **kwargs):
            try:
                from matplotlib._pylab_helpers import Gcf
                for manager in Gcf.get_all_fig_managers():
                    fig = manager.canvas.figure
                    apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
                    if '_style_boxplot_axis' in globals():
                        for ax in getattr(fig, 'axes', []):
                            _style_boxplot_axis(ax)
                    if 'apply_external_legend' in globals():
                        apply_external_legend(fig)
                    if 'sanitize_figure_text' in globals():
                        sanitize_figure_text(fig)
                    if 'prepare_figure_for_display' in globals():
                        prepare_figure_for_display(fig)
            except Exception:
                pass
            return plt._grid_original_show(*args, **kwargs)

        plt.subplots = _grid_subplots
        plt.figure = _grid_figure
        plt.show = _grid_show

    if not hasattr(_MplFigure, '_grid_original_add_subplot'):
        _MplFigure._grid_original_add_subplot = _MplFigure.add_subplot

        def _grid_add_subplot(self, *args, **kwargs):
            ax = _MplFigure._grid_original_add_subplot(self, *args, **kwargs)
            try:
                apply_grid_size(self, panel_kind=getattr(self, '_grid_panel_kind', 'simple'))
            except Exception:
                pass
            return ax

        _MplFigure.add_subplot = _grid_add_subplot

    if not hasattr(plt, '_pnas_original_scatter'):
        plt._pnas_original_scatter = plt.scatter
        plt._pnas_original_axes_scatter = _MplAxes.scatter

        def _pnas_plt_scatter(*args, **kwargs):
            kwargs = dict(kwargs)
            kwargs.setdefault('s', PNAS_SCATTER_AREA_DEFAULT)
            if 'lw' in kwargs and 'linewidths' not in kwargs and 'linewidth' not in kwargs:
                kwargs['linewidths'] = kwargs.pop('lw')
            kwargs.setdefault('linewidths', 0.0)
            return plt._pnas_original_scatter(*args, **kwargs)

        def _pnas_axes_scatter(self, *args, **kwargs):
            kwargs = dict(kwargs)
            kwargs.setdefault('s', PNAS_SCATTER_AREA_DEFAULT)
            if 'lw' in kwargs and 'linewidths' not in kwargs and 'linewidth' not in kwargs:
                kwargs['linewidths'] = kwargs.pop('lw')
            kwargs.setdefault('linewidths', 0.0)
            return plt._pnas_original_axes_scatter(self, *args, **kwargs)

        plt.scatter = _pnas_plt_scatter
        _MplAxes.scatter = _pnas_axes_scatter

    def _pnas_boxplot_defaults(kwargs=None):
        kw = dict(kwargs or {})
        kw.setdefault('patch_artist', True)
        kw.setdefault('showmeans', False)
        kw.setdefault('meanline', False)
        kw.setdefault('showcaps', False)
        kw.setdefault('showfliers', False)
        kw.setdefault('whis', (10, 90))

        boxprops = dict(kw.get('boxprops', {}))
        boxprops.setdefault('edgecolor', 'none')
        boxprops.setdefault('linewidth', 0.0)
        kw['boxprops'] = boxprops

        medianprops = dict(kw.get('medianprops', {}))
        medianprops.setdefault('color', 'none')
        medianprops.setdefault('linewidth', 0.0)
        kw['medianprops'] = medianprops

        meanprops = dict(kw.get('meanprops', {}))
        meanprops.setdefault('color', 'white')
        meanprops.setdefault('linewidth', 0.0)
        meanprops.setdefault('linestyle', '-')
        kw['meanprops'] = meanprops

        whiskerprops = dict(kw.get('whiskerprops', {}))
        whiskerprops.setdefault('linewidth', 1.0)
        kw['whiskerprops'] = whiskerprops

        capprops = dict(kw.get('capprops', {}))
        capprops.setdefault('linewidth', 0.0)
        capprops.setdefault('alpha', 0.0)
        kw['capprops'] = capprops

        flierprops = dict(kw.get('flierprops', {}))
        flierprops.setdefault('marker', 'o')
        flierprops.setdefault('markersize', 0.0)
        flierprops.setdefault('alpha', 0.0)
        kw['flierprops'] = flierprops
        return kw

    if not hasattr(plt, '_pnas_original_boxplot'):
        plt._pnas_original_boxplot = plt.boxplot
        plt._pnas_original_axes_boxplot = _MplAxes.boxplot
        plt._pnas_original_axes_bxp = _MplAxes.bxp

        def _pnas_plt_boxplot(*args, **kwargs):
            return plt._pnas_original_boxplot(*args, **_pnas_boxplot_defaults(kwargs))

        def _pnas_axes_boxplot(self, *args, **kwargs):
            return plt._pnas_original_axes_boxplot(self, *args, **_pnas_boxplot_defaults(kwargs))

        def _pnas_axes_bxp(self, *args, **kwargs):
            bxp_kwargs = _pnas_boxplot_defaults(kwargs)
            bxp_kwargs.pop('whis', None)  # bxp() does not accept 'whis'; only boxplot() does
            return plt._pnas_original_axes_bxp(self, *args, **bxp_kwargs)

        plt.boxplot = _pnas_plt_boxplot
        _MplAxes.boxplot = _pnas_axes_boxplot
        _MplAxes.bxp = _pnas_axes_bxp

    if not hasattr(_MplFigure, '_grid_original_tight_layout'):
        _MplFigure._grid_original_tight_layout = _MplFigure.tight_layout

        def _grid_tight_layout(self, *args, **kwargs):
            try:
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
                if 'prepare_figure_for_display' in globals():
                    prepare_figure_for_display(self)
                    return self
            except Exception:
                pass
            return _MplFigure._grid_original_tight_layout(self, *args, **kwargs)

        _MplFigure.tight_layout = _grid_tight_layout

    if not hasattr(_MplFigure, '_grid_original_savefig'):
        _MplFigure._grid_original_savefig = _MplFigure.savefig

        def _grid_savefig(self, *args, **kwargs):
            try:
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
            except Exception:
                pass

            kwargs = dict(kwargs)
            extra_artists = list(kwargs.get('bbox_extra_artists', []))
            extra_artists.extend([lg for lg in getattr(self, 'legends', []) if lg is not None])
            if extra_artists:
                kwargs['bbox_extra_artists'] = extra_artists

            try:
                return _MplFigure._grid_original_savefig(self, *args, **kwargs)
            except PermissionError as exc:
                target = args[0] if len(args) >= 1 else kwargs.get('fname', None)
                retry_path = _permission_retry_path(target, prefer_same_dir=True) if '_permission_retry_path' in globals() else None
                if retry_path is not None:
                    print(f'Permission denied saving to {target}; retrying at {retry_path}')
                    retry_args, retry_kwargs = _replace_savefig_target(args, kwargs, retry_path)
                    try:
                        return _MplFigure._grid_original_savefig(self, *retry_args, **retry_kwargs)
                    except PermissionError:
                        fallback_path = _permission_retry_path(target, prefer_same_dir=False) if '_permission_retry_path' in globals() else None
                        if fallback_path is not None:
                            print(f'Permission denied saving to {target}; using fallback export {fallback_path}')
                            fallback_args, fallback_kwargs = _replace_savefig_target(args, kwargs, fallback_path)
                            return _MplFigure._grid_original_savefig(self, *fallback_args, **fallback_kwargs)
                raise exc

        _MplFigure.savefig = _grid_savefig

    if not hasattr(_MplFigure, '_grid_original_show'):
        _MplFigure._grid_original_show = _MplFigure.show

        def _grid_figure_show(self, *args, **kwargs):
            try:
                apply_grid_size(self, panel_kind=getattr(self, '_grid_panel_kind', 'simple'))
                if '_style_boxplot_axis' in globals():
                    for ax in getattr(self, 'axes', []):
                        _style_boxplot_axis(ax)
                if 'apply_external_legend' in globals():
                    apply_external_legend(self)
                if 'sanitize_figure_text' in globals():
                    sanitize_figure_text(self)
                if 'prepare_figure_for_display' in globals():
                    prepare_figure_for_display(self)
            except Exception:
                pass
            return _MplFigure._grid_original_show(self, *args, **kwargs)

        _MplFigure.show = _grid_figure_show


def _boxplot_face_rgba(patch):
    if not hasattr(patch, 'get_facecolor'):
        return None
    fc = patch.get_facecolor()
    if fc is None:
        return None
    if hasattr(fc, '__len__') and len(np.shape(fc)) > 1:
        fc = fc[0]
    if len(fc) < 4:
        return None
    return tuple(fc)


def _boxplot_center_x(patch):
    try:
        verts = np.asarray(patch.get_path().vertices, dtype=float)
    except Exception:
        return np.nan
    if verts.ndim != 2 or verts.shape[1] < 2:
        return np.nan
    x = verts[:, 0]
    x = x[np.isfinite(x)]
    if x.size == 0:
        return np.nan
    return float(np.nanmean(x))


def _style_boxplot_axis(ax):
    if ax is None:
        return
    patches = [p for p in getattr(ax, 'patches', []) if _boxplot_face_rgba(p) is not None]
    if not patches:
        return

    for patch in patches:
        patch.set_edgecolor('none')
        patch.set_linewidth(0.0)

    line_groups = {}
    for line in getattr(ax, 'lines', []):
        try:
            x = np.asarray(line.get_xdata(), dtype=float)
            y = np.asarray(line.get_ydata(), dtype=float)
        except Exception:
            continue
        if x.size == 0 or y.size == 0:
            continue
        if not np.isfinite(np.nanmean(x)):
            continue
        center = round(float(np.nanmean(x)), 3)
        line_groups.setdefault(center, []).append(line)

    centers = sorted(line_groups)
    for patch in patches:
        color = _boxplot_face_rgba(patch)
        if color is None or not centers:
            continue
        patch_center = _boxplot_center_x(patch)
        if not np.isfinite(patch_center):
            continue
        nearest_center = min(centers, key=lambda c: abs(float(c) - patch_center))
        group = line_groups.get(nearest_center, [])
        horiz = []
        vert = []
        for line in group:
            x = np.asarray(line.get_xdata(), dtype=float)
            y = np.asarray(line.get_ydata(), dtype=float)
            xr = float(np.nanmax(x) - np.nanmin(x)) if x.size else 0.0
            yr = float(np.nanmax(y) - np.nanmin(y)) if y.size else 0.0
            if xr >= yr:
                horiz.append((line, xr))
            else:
                vert.append(line)

        for line in vert:
            line.set_color(color)
            line.set_linewidth(1.0)
            line.set_linestyle('-')

        if horiz:
            horiz = sorted(horiz, key=lambda item: float(item[1]), reverse=True)
            keep_width = horiz[0][1] if horiz else 0.0
            kept = False
            for line, xr in horiz:
                if keep_width > 0 and xr < (0.75 * keep_width):
                    line.set_visible(False)
                    continue
                if not kept:
                    line.set_color('white')
                    line.set_linewidth(1.0)
                    line.set_linestyle('-')
                    kept = True
                else:
                    line.set_visible(False)


def _enforce_pnas_artist_style(ax):
    if '_style_boxplot_axis' in globals():
        _style_boxplot_axis(ax)
    for line in ax.lines:
        if hasattr(line, 'get_linewidth') and line.get_linewidth() is not None:
            line.set_linewidth(clamp_linewidth(line.get_linewidth()))
        if hasattr(line, 'get_markersize') and line.get_markersize() is not None:
            line.set_markersize(clamp_markersize(line.get_markersize()))

    for collection in ax.collections:
        if hasattr(collection, 'get_linewidths'):
            widths = collection.get_linewidths()
            if widths is not None and len(widths) > 0:
                collection.set_linewidths(clamp_linewidth(widths))
        if hasattr(collection, 'get_sizes'):
            sizes = collection.get_sizes()
            if sizes is not None and len(sizes) > 0:
                collection.set_sizes(clamp_scatter_area(sizes))

if 'plt' in globals():
    plt.rcParams['axes.grid'] = False
    plt.rcParams['axes.spines.top'] = False
    plt.rcParams['axes.spines.right'] = False
    plt.rcParams['legend.frameon'] = False
    plt.rcParams['pdf.fonttype'] = 42
    plt.rcParams['ps.fonttype'] = 42
    plt.rcParams['svg.fonttype'] = 'none'
    plt.rcParams['font.family'] = 'sans-serif'
    plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
    plt.rcParams['font.size'] = clamp_fontsize(8.0)
    plt.rcParams['axes.labelsize'] = clamp_fontsize(8.0)
    plt.rcParams['axes.titlesize'] = clamp_fontsize(8.0)
    plt.rcParams['xtick.labelsize'] = clamp_fontsize(6.0)
    plt.rcParams['ytick.labelsize'] = clamp_fontsize(6.0)
    plt.rcParams['legend.fontsize'] = clamp_fontsize(6.0)
    plt.rcParams['lines.linewidth'] = clamp_linewidth(0.9)
    plt.rcParams['lines.markersize'] = clamp_markersize(5.0)
    plt.rcParams['contour.linewidth'] = clamp_linewidth(0.8)
    _install_pnas_matplotlib_sizers()

# Helper function for cleaning bouton IDs
def clean_bouton_id(raw_bouton_id: str) -> str:
    """Remove file suffixes from bouton IDs for consistent matching"""
    return raw_bouton_id.replace('_traces_converted', '')

# Savitzky-Golay smoothing function (for additional analysis if needed)
def sg_smooth(y: np.ndarray, window_length: int = 9, polyorder: int = 2) -> np.ndarray:
    """Apply Savitzky-Golay filter, handling NaN gracefully."""
    from scipy.signal import savgol_filter
    if np.all(np.isnan(y)):
        return y.copy()
    valid = np.isfinite(y)
    if valid.sum() < max(window_length, polyorder + 2):
        return y.copy()
    result = y.copy()
    result[valid] = savgol_filter(y[valid], min(window_length, valid.sum() // 2 * 2 - 1), polyorder)
    return result

# Unified trace selection / stats / plotting API

def _normalize_trace_source(source=None):
    src = TRACE_MEAN_SOURCE if source is None else source
    src = str(src).strip().lower()
    if src not in {'normalized', 'raw', 'raw_nearest'}:
        raise ValueError(f'Unsupported trace source: {source}')
    return src


def _trace_source_dataframe(source=None):
    return NORM_TRACES_DATAFRAME if _normalize_trace_source(source) == 'normalized' else RAW_TRACES_DF


def _trace_row_time(trace_row, aligned=True):
    preferred_col = 'TimeOffset' if aligned else 'TimeRaw'
    if preferred_col in trace_row.index:
        values = trace_row[preferred_col]
    elif aligned and 'Time' in trace_row.index:
        values = trace_row['Time']
    elif 'Time' in trace_row.index:
        values = trace_row['Time']
    else:
        raise KeyError('Trace row does not contain a usable time vector')
    return np.asarray(values, dtype=float)


def _normalize_trace_alignment(alignment=None, source=None):
    src = _normalize_trace_source(source)
    align = 'auto' if alignment is None else str(alignment).strip().lower()
    if align == 'auto':
        return 'nearest' if src == 'raw_nearest' else 'grid'
    if align not in {'grid', 'nearest'}:
        raise ValueError(f'Unsupported trace alignment: {alignment}')
    return align


def select_traces(trace_ids=None, condition_names=None, rows=None, source=None, filters=None):
    selected = rows.copy() if rows is not None else _trace_source_dataframe(source).copy()
    if 'BaseID' not in selected.columns and 'ID' in selected.columns:
        selected['BaseID'] = selected['ID'].map(_normalize_bouton_id)
    if trace_ids is not None:
        ids = trace_ids if isinstance(trace_ids, (list, tuple, set, pd.Index, np.ndarray, pd.Series)) else [trace_ids]
        norm_ids = pd.Index([_normalize_bouton_id(v) for v in ids])
        selected = selected[selected['BaseID'].astype(str).isin(norm_ids.astype(str))].copy()
    if condition_names is not None:
        conds = [condition_names] if isinstance(condition_names, str) else list(condition_names)
        selected = selected[selected['Condition'].isin(conds)].copy()
        if trace_ids is not None and 'BaseID' in selected.columns:
            cond_priority = {cond: idx for idx, cond in enumerate(conds)}
            selected['_cond_priority'] = selected['Condition'].map(lambda c: cond_priority.get(c, len(cond_priority)))
            selected = selected.sort_values(['_cond_priority']).drop_duplicates(subset='BaseID', keep='first').drop(columns=['_cond_priority']).reset_index(drop=True)
    if filters is not None:
        if callable(filters):
            selected = selected[np.asarray(filters(selected), dtype=bool)].copy()
        elif isinstance(filters, dict):
            for col, expected in filters.items():
                if callable(expected):
                    selected = selected[np.asarray(expected(selected[col]), dtype=bool)].copy()
                elif isinstance(expected, (list, tuple, set, pd.Index, np.ndarray, pd.Series)):
                    selected = selected[selected[col].isin(list(expected))].copy()
                else:
                    selected = selected[selected[col] == expected].copy()
        else:
            raise ValueError('filters must be a callable or dict')
    return selected


def _trace_pairs(rows, source=None):
    src = _normalize_trace_source(source)
    pairs = []
    for _, row in rows.iterrows():
        t = _trace_row_time(row, aligned=True)
        y = np.asarray(row['Avg'], float)
        pairs.append((t, y))
    return pairs


def _reference_time(pairs, resample=1.0):
    scale = 1.0 if resample is None else float(resample)
    if not np.isfinite(scale) or scale <= 0:
        raise ValueError(f'resample must be positive, got {resample!r}')
    for t_src, _ in pairs:
        t_src = np.asarray(t_src, float)
        t_valid = t_src[np.isfinite(t_src)]
        if t_valid.size < 2:
            continue
        diffs = np.diff(t_valid)
        diffs = diffs[np.isfinite(diffs) & (diffs > 0)]
        if diffs.size:
            dt = float(np.nanmedian(diffs)) / scale
            return np.arange(t_valid[0], t_valid[-1] + dt / 2.0, dt, dtype=float)
    raise ValueError('No valid time vector available for trace resampling')


def _trace_matrix(rows, source=None, alignment='auto', resample=1.0, tol_factor=None):
    pairs = _trace_pairs(rows, source=source)
    if not pairs:
        return None, None
    t_ref = _reference_time(pairs, resample=resample)
    align = _normalize_trace_alignment(alignment, source=source)
    if align == 'grid':
        def _interp(t_src, y_src):
            finite = np.isfinite(t_src) & np.isfinite(y_src)
            if finite.sum() < 2:
                return np.full_like(t_ref, np.nan, dtype=float)
            t_valid = t_src[finite]
            y_valid = y_src[finite]
            out = np.interp(t_ref, t_valid, y_valid)
            out[(t_ref < t_valid[0]) | (t_ref > t_valid[-1])] = np.nan
            return out
        return t_ref, np.vstack([_interp(np.asarray(t, float), np.asarray(y, float)) for t, y in pairs])
    tol_factor = TRACE_RAW_NEAREST_TOL_FACTOR if tol_factor is None else float(tol_factor)
    diffs = [np.diff(np.asarray(t, float)) for t, _ in pairs if len(t) > 1]
    diffs = [d[np.isfinite(d) & (d > 0)] for d in diffs if np.any(np.isfinite(d))]
    tol = tol_factor * float(np.nanmedian(np.concatenate(diffs))) if diffs else np.inf
    mat = np.full((len(pairs), len(t_ref)), np.nan, dtype=float)
    for i, (t_src, y_src) in enumerate(pairs):
        t_src = np.asarray(t_src, float)
        y_src = np.asarray(y_src, float)
        finite = np.isfinite(t_src) & np.isfinite(y_src)
        if finite.sum() == 0:
            continue
        t_valid = t_src[finite]
        y_valid = y_src[finite]
        idx = np.searchsorted(t_valid, t_ref)
        idx0 = np.clip(idx - 1, 0, len(t_valid) - 1)
        idx1 = np.clip(idx, 0, len(t_valid) - 1)
        d0 = np.abs(t_valid[idx0] - t_ref)
        d1 = np.abs(t_valid[idx1] - t_ref)
        use1 = d1 < d0
        best = np.where(use1, idx1, idx0)
        dist = np.where(use1, d1, d0)
        valid = dist <= tol
        mat[i, valid] = y_valid[best[valid]]
    return t_ref, mat


def _aggregate_trace_matrix(matrix, aggregation='mean'):
    agg = 'mean' if aggregation is None else str(aggregation).strip().lower()
    if agg == 'median':
        return np.nanmedian(matrix, axis=0)
    if agg == 'max':
        return np.nanmax(matrix, axis=0)
    if agg == 'std':
        return np.nanstd(matrix, axis=0)
    return np.nanmean(matrix, axis=0)


def compute_trace_stats(trace_ids=None, condition_names=None, rows=None, source=None, *, filters=None, alignment='auto', resample=1, aggregation='mean', smooth=None, tol_factor=None, return_matrix=False):
    selected = select_traces(trace_ids=trace_ids, condition_names=condition_names, rows=rows, source=source, filters=filters)
    if selected is None or len(selected) == 0:
        raise ValueError('No traces matched the requested selection')
    t_ref, matrix = _trace_matrix(selected, source=source, alignment=alignment, resample=resample, tol_factor=tol_factor)
    average = _aggregate_trace_matrix(matrix, aggregation=aggregation)
    if smooth:
        cfg = smooth if isinstance(smooth, dict) else {}
        average = sg_smooth(average, window_length=int(cfg.get('window_length', cfg.get('window', 9))), polyorder=int(cfg.get('polyorder', 2)))
    n_eff = np.sum(np.isfinite(matrix), axis=0)
    sem = np.nanstd(matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
    return {'time': t_ref, 'average': average, 'sem': sem, 'n': int(matrix.shape[0]), 'rows': selected, 'matrix': matrix if return_matrix else None}


def plot_traces(ax=None, trace_ids=None, condition_names=None, rows=None, source=None, *, filters=None, show_average=True, show_sem=True, show_individuals=False, alignment='auto', resample=1, aggregation='mean', smooth=None, tol_factor=None, color='k', individual_color=None, sem_color=None, label=None, linewidth=2.0, linestyle='-', sem_alpha=0.25, individual_alpha=0.12, individual_lw=0.6, stim_times=None, stim_kwargs=None, zero_line=False, zero_kwargs=None, hlines=None, hline_kwargs=None, event_time=None, event_kwargs=None, xlim=None, ylim=None, xlabel=None, ylabel=None, title=None, legend=False, legend_kwargs=None, style_axis=True, return_data=False):
    """Unified trace plotting helper.

    Data selection:
    - provide one of `rows`, `trace_ids`, or `condition_names`
    - `source`: None | 'normalized' | 'raw' | 'raw_nearest'
    - `filters`: optional extra row filter(s) passed to `select_traces`

    Trace computation:
    - `alignment`: 'auto' | 'grid' | 'nearest'
    - `resample`: grid resampling factor
    - `aggregation`: 'mean' | 'median' | 'robust_mean' | 'max' | 'std'
    - `smooth`: optional post-aggregation smoothing spec
    - `tol_factor`: nearest-sample tolerance multiplier

    What to draw:
    - `show_average`, `show_sem`, `show_individuals`
    - `color`, `individual_color`, `sem_color`, `label`
    - `linewidth`, `linestyle`, `sem_alpha`, `individual_alpha`, `individual_lw`

    Common trace decorators:
    - `stim_times` with `stim_kwargs`
    - `zero_line` with `zero_kwargs`
    - `hlines` with `hline_kwargs`
    - `event_time` with `event_kwargs`
    - axis labels / limits / title / legend through `xlim`, `ylim`, `xlabel`, `ylabel`, `title`, `legend`, `legend_kwargs`

    Return value:
    - returns `ax` by default
    - returns the computed stats dict when `return_data=True`
    """
    if ax is None:
        _, ax = make_figure_grid(panel_kind='trace')
    stats = compute_trace_stats(trace_ids=trace_ids, condition_names=condition_names, rows=rows, source=source, filters=filters, alignment=alignment, resample=resample, aggregation=aggregation, smooth=smooth, tol_factor=tol_factor, return_matrix=show_individuals)
    stats['ax'] = ax
    plot_lw = clamp_linewidth(linewidth)
    ind_lw = clamp_linewidth(individual_lw)
    ind_color = individual_color if individual_color is not None else color
    fill_color = sem_color if sem_color is not None else color
    if show_individuals and stats['matrix'] is not None:
        for trace in stats['matrix']:
            ax.plot(stats['time'], trace, color=ind_color, alpha=individual_alpha, linewidth=ind_lw)
    if show_sem and stats['sem'] is not None:
        ax.fill_between(stats['time'], stats['average'] - stats['sem'], stats['average'] + stats['sem'], color=fill_color, alpha=sem_alpha)
    if show_average:
        ax.plot(stats['time'], stats['average'], color=color, linewidth=plot_lw, linestyle=linestyle, label=label)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if stim_times is not None:
        add_stimulus_ticks(ax, stim_times, **(stim_kwargs or {}))
    if zero_line:
        zero_defaults = {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1}
        zero_defaults.update(zero_kwargs or {})
        ax.axhline(0, **zero_defaults)
    if hlines is not None:
        ref_defaults = {'color': 'black', 'linestyle': ':', 'alpha': 0.5, 'linewidth': 1}
        ref_defaults.update(hline_kwargs or {})
        for y_value in hlines:
            ax.axhline(y_value, **ref_defaults)
    if event_time is not None:
        event_defaults = {'color': 'gray', 'linestyle': '--', 'alpha': 0.5, 'linewidth': 1}
        event_defaults.update(event_kwargs or {})
        ax.axvline(event_time, **event_defaults)
    if style_axis:
        style_trace_axis(ax)
    if legend:
        add_legend(ax, frameon=False, **(legend_kwargs or {}))
    return stats if return_data else ax


def build_trace_lookup(traces_dataframe, id_col='ID', time_col='TimeOffset', avg_col='Avg'):
    lookup = {}
    if traces_dataframe is None or len(traces_dataframe) == 0:
        return lookup
    for _, trace_row in traces_dataframe.iterrows():
        resolved_time_col = time_col if time_col in trace_row.index else ('Time' if 'Time' in trace_row.index else time_col)
        lookup[str(trace_row[id_col])] = {'Time': trace_row[resolved_time_col], 'Avg': trace_row[avg_col]}
    return lookup


def build_trace_lookup_from_source(source=None, condition_names=None):
    src = _normalize_trace_source(source)
    rows = select_traces(condition_names=condition_names, source=src)
    lookup = {}
    for _, trace_row in rows.iterrows():
        time_values = _trace_row_time(trace_row, aligned=True)
        lookup[str(trace_row['ID'])] = {'Time': time_values, 'Avg': trace_row['Avg'], 'Condition': trace_row.get('Condition', None)}
    return lookup

print("✓ Helper functions loaded")

# Shared trace helper utilities

def add_stimulus_ticks(
    ax,
    stim_times,
    *,
    mode='top',
    y_span=None,
    tick_ratio=0.03,
    color='black',
    linewidth=1.5,
):
    """Add stimulus ticks either at top of axes or at a fixed y-span."""
    if mode == 'fixed':
        if y_span is None:
            raise ValueError("y_span must be provided when mode='fixed'")
        y0, y1 = y_span
        clip_on = True
    else:
        y_min, y_max = ax.get_ylim()
        tick_height = (y_max - y_min) * tick_ratio
        y0, y1 = y_max - tick_height, y_max
        clip_on = False

    tick_lw = clamp_linewidth(linewidth) if 'clamp_linewidth' in globals() else linewidth
    for stim_time in stim_times:
        ax.plot([stim_time, stim_time], [y0, y1], color=color, linewidth=tick_lw, clip_on=clip_on)

def _apply_clean_axes_style(ax, panel_kind='simple'):
    """Apply no-grid and no top/right spines style."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine_name in ('left', 'bottom'):
        if spine_name in ax.spines:
            ax.spines[spine_name].set_linewidth(1.0)
    show_xlabel, show_ylabel = (_outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True))
    tick_w = 1.0
    ax.tick_params(direction='out', length=4, width=tick_w, labelsize=6)
    if not show_xlabel:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    if not show_ylabel:
        ax.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind=panel_kind)
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)

def style_trace_axis(ax):
    """Apply consistent style for trace plots."""
    _apply_clean_axes_style(ax, panel_kind='trace')
    legend = ax.get_legend()
    if legend is not None:
        legend.set_frame_on(False)

def style_hist_axis(ax):
    """Apply consistent style for histogram plots."""
    _apply_clean_axes_style(ax, panel_kind='hist')
    legend = ax.get_legend()
    if legend is not None:
        legend.set_frame_on(False)



# ===== Shared PPR Helpers (moved from later cell) =====
def ppr_profile_stats(condition_dataframe, ppr_column_names=None, max_pulse_number=10):
    """Return pulse numbers, means, SEMs, and n for a condition dataframe."""
    if ppr_column_names is None:
        ppr_column_names = [
            f'PPR{pulse_num}/1'
            for pulse_num in range(2, max_pulse_number + 1)
            if f'PPR{pulse_num}/1' in condition_dataframe.columns
        ]

    # Keep only columns present in dataframe, then drop columns with no finite values.
    ppr_column_names = [col for col in ppr_column_names if col in condition_dataframe.columns]
    if len(ppr_column_names) == 0:
        return [1], [1.0], [0.0], len(condition_dataframe)

    ppr_data = condition_dataframe[ppr_column_names].apply(pd.to_numeric, errors='coerce')
    valid_cols = [col for col in ppr_column_names if ppr_data[col].notna().any()]
    if len(valid_cols) == 0:
        return [1], [1.0], [0.0], len(condition_dataframe)

    means = [1.0] + ppr_data[valid_cols].mean().tolist()
    sems = [0.0] + ppr_data[valid_cols].sem().fillna(0.0).tolist()
    pulse_numbers = list(range(1, len(valid_cols) + 2))
    return pulse_numbers, means, sems, len(condition_dataframe)


def extract_ppr_profile(condition_dataframe, max_pulse_number=10):
    """Backward-compatible wrapper used in older cells."""
    pulse_numbers, means, sems, _ = ppr_profile_stats(
        condition_dataframe,
        max_pulse_number=max_pulse_number,
    )
    return means, sems, len(pulse_numbers)


def ppr_profiles_matrix(condition_dataframe, ppr_column_names=None, max_pulse_number=10):
    """Return matrix with pulse 1 forced to 1.0, then valid PPR2..PPRn columns."""
    if ppr_column_names is None:
        ppr_column_names = [
            f'PPR{pulse_num}/1'
            for pulse_num in range(2, max_pulse_number + 1)
            if f'PPR{pulse_num}/1' in condition_dataframe.columns
        ]

    ppr_column_names = [col for col in ppr_column_names if col in condition_dataframe.columns]
    n_rows = len(condition_dataframe)
    if len(ppr_column_names) == 0:
        return np.ones((n_rows, 1), dtype=float) if n_rows > 0 else np.empty((0, 1), dtype=float)

    ppr_data = condition_dataframe[ppr_column_names].apply(pd.to_numeric, errors='coerce')
    valid_cols = [col for col in ppr_column_names if ppr_data[col].notna().any()]
    if len(valid_cols) == 0:
        return np.ones((n_rows, 1), dtype=float) if n_rows > 0 else np.empty((0, 1), dtype=float)

    arr = ppr_data[valid_cols].to_numpy(dtype=float)
    return np.hstack([np.ones((arr.shape[0], 1), dtype=float), arr])




def failure_profiles_matrix(
    condition_dataframe,
    trials_df=None,
    *,
    threshold_col='thr_shared',
    max_pulse_number=10,
    min_trials=1,
):
    """Return bouton × pulse failure-rate matrix using explicit per-event failures."""
    if condition_dataframe is None or len(condition_dataframe) == 0:
        return np.empty((0, int(max_pulse_number)), dtype=float)

    failrate_cols = [f'FailRate{k}' for k in range(1, int(max_pulse_number) + 1)]
    if all(col in condition_dataframe.columns for col in failrate_cols):
        return condition_dataframe[failrate_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)

    if trials_df is None:
        if 'trials_all' not in globals():
            raise RuntimeError('trials_all is required to build explicit failure-rate profiles.')
        trials_df = trials_all

    if 'ID' not in condition_dataframe.columns:
        raise ValueError("condition_dataframe must contain an 'ID' column")

    cond_col = 'Condition' if 'Condition' in condition_dataframe.columns else None
    trial_df = trials_df.copy()

    if cond_col is not None and 'condition' in trial_df.columns:
        wanted_conditions = (
            condition_dataframe[cond_col]
            .dropna()
            .astype(str)
            .str.strip()
            .unique()
            .tolist()
        )
        trial_df = trial_df[trial_df['condition'].astype(str).str.strip().isin(wanted_conditions)]

    fail_lookup = defaultdict(list)
    for _, row in trial_df.iterrows():
        if 'file' not in row.index:
            continue
        trial_id = extract_base_name(str(row['file']).strip())
        trial_cond = str(row['condition']).strip() if 'condition' in row.index else ''
        fail_mask, _ = get_failure_mask_row(row, threshold_col=threshold_col)
        fail_mask = np.asarray(fail_mask, dtype=float)[:max_pulse_number]
        if not np.isfinite(fail_mask).any():
            continue
        key = (trial_cond, trial_id) if cond_col is not None else trial_id
        fail_lookup[key].append(fail_mask)

    profiles = []
    for _, row in condition_dataframe.iterrows():
        bouton_id = extract_base_name(str(row['ID']).strip())
        bouton_cond = str(row[cond_col]).strip() if cond_col is not None else ''
        key = (bouton_cond, bouton_id) if cond_col is not None else bouton_id
        trial_masks = fail_lookup.get(key, [])
        if len(trial_masks) < int(min_trials):
            profiles.append(np.full(int(max_pulse_number), np.nan, dtype=float))
            continue
        mat = np.asarray(trial_masks, dtype=float)
        profiles.append(np.nanmean(mat, axis=0))

    if len(profiles) == 0:
        return np.empty((0, int(max_pulse_number)), dtype=float)
    return np.vstack(profiles)

def plot_ppr_mean_sem(
    ax,
    pulse_numbers,
    means,
    sems,
    *,
    color,
    marker,
    label,
    linewidth=2.0,
    markersize=6,
    sem_alpha=0.2,
):
    """Plot one mean PPR trajectory with SEM shading (robust to NaN segments)."""
    x = np.asarray(pulse_numbers, dtype=float)
    means_arr = np.asarray(means, dtype=float)
    sems_arr = np.asarray(sems, dtype=float)

    if len(x) != len(means_arr):
        n = min(len(x), len(means_arr), len(sems_arr))
        x, means_arr, sems_arr = x[:n], means_arr[:n], sems_arr[:n]

    valid_line = np.isfinite(x) & np.isfinite(means_arr)
    if not np.any(valid_line):
        return

    linewidth = clamp_linewidth(linewidth) if 'clamp_linewidth' in globals() else linewidth
    markersize = clamp_markersize(markersize) if 'clamp_markersize' in globals() else markersize

    ax.plot(
        x[valid_line],
        means_arr[valid_line],
        marker=marker,
        color=color,
        linewidth=linewidth,
        markersize=markersize,
        label=label,
    )

    lower = means_arr - sems_arr
    upper = means_arr + sems_arr
    valid_fill = np.isfinite(x) & np.isfinite(lower) & np.isfinite(upper)
    if np.any(valid_fill):
        ax.fill_between(x, lower, upper, where=valid_fill, alpha=sem_alpha, color=color)


def plot_ppr_profiles_overlay(
    ax,
    profiles_matrix,
    *,
    color,
    label='Mean',
    individual_alpha=0.25,
    individual_lw=1.0,
    mean_lw=2.5,
    sem_alpha=0.18,
):
    """Plot individual PPR profiles + mean ± SEM."""
    if profiles_matrix is None or profiles_matrix.size == 0:
        return None

    pulse_numbers = list(range(1, profiles_matrix.shape[1] + 1))
    x = np.asarray(pulse_numbers, dtype=float)

    individual_lw = clamp_linewidth(individual_lw) if 'clamp_linewidth' in globals() else individual_lw
    mean_lw = clamp_linewidth(mean_lw) if 'clamp_linewidth' in globals() else mean_lw

    for i in range(profiles_matrix.shape[0]):
        ax.plot(pulse_numbers, profiles_matrix[i], color=color, alpha=individual_alpha, linewidth=individual_lw)

    mean_profile = np.nanmean(profiles_matrix, axis=0)
    n_eff = np.sum(np.isfinite(profiles_matrix), axis=0)
    if profiles_matrix.shape[0] > 1:
        sem_profile = np.nanstd(profiles_matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
        sem_profile[n_eff < 2] = 0.0
    else:
        sem_profile = np.zeros_like(mean_profile)

    valid_line = np.isfinite(x) & np.isfinite(mean_profile)
    if np.any(valid_line):
        ax.plot(x[valid_line], mean_profile[valid_line], color=color, linewidth=mean_lw, label=label)

    lower = mean_profile - sem_profile
    upper = mean_profile + sem_profile
    valid_fill = np.isfinite(x) & np.isfinite(lower) & np.isfinite(upper)
    if np.any(valid_fill):
        ax.fill_between(x, lower, upper, where=valid_fill, color=color, alpha=sem_alpha)

    return pulse_numbers, mean_profile, sem_profile


def finalize_ppr_axis(
    ax,
    pulse_numbers,
    *,
    title=None,
    xlabel='Stimulus',
    ylabel='PPR (A_n/A_1)',
    show_xlabel='auto',
    show_ylabel='auto',
    ylim=None,
    unity_line=True,
    unity_kwargs=None,
    legend=True,
    legend_loc='best',
    legend_fontsize=None,
    legend_frame=False,
    grid=True,
    grid_alpha=0.3,
    xlabel_fontsize=None,
    ylabel_fontsize=None,
    title_fontsize=None,
    title_fontweight=None,
):
    """Apply shared PPR axis formatting across notebook figures."""
    if unity_line:
        line_kwargs = dict(color='gray', linestyle='dotted', linewidth=1)
        if unity_kwargs:
            line_kwargs.update(unity_kwargs)
        if 'clamp_linewidth' in globals() and 'linewidth' in line_kwargs:
            line_kwargs['linewidth'] = clamp_linewidth(line_kwargs['linewidth'])
        ax.axhline(1.0, **line_kwargs)

    if show_xlabel == 'auto' or show_ylabel == 'auto':
        auto_x, auto_y = _outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True)
        if show_xlabel == 'auto':
            show_xlabel = auto_x
        if show_ylabel == 'auto':
            show_ylabel = auto_y

    if xlabel is not None:
        if show_xlabel:
            if xlabel_fontsize is None:
                ax.set_xlabel(xlabel)
            else:
                fs = clamp_fontsize(xlabel_fontsize) if 'clamp_fontsize' in globals() else xlabel_fontsize
                ax.set_xlabel(xlabel, fontsize=fs)
        else:
            ax.set_xlabel('')

    if ylabel is not None:
        if show_ylabel:
            if ylabel_fontsize is None:
                ax.set_ylabel(ylabel)
            else:
                fs = clamp_fontsize(ylabel_fontsize) if 'clamp_fontsize' in globals() else ylabel_fontsize
                ax.set_ylabel(ylabel, fontsize=fs)
        else:
            ax.set_ylabel('')

    if pulse_numbers is not None:
        ax.set_xticks(list(pulse_numbers))

    if ylim is not None:
        ax.set_ylim(ylim)
    elif ylabel is not None and 'PPR' in str(ylabel):
        y_min, y_max = ax.get_ylim()
        ax.set_ylim(0, y_max)

    if title is not None:
        title_kwargs = {}
        if title_fontsize is not None:
            title_kwargs['fontsize'] = clamp_fontsize(title_fontsize) if 'clamp_fontsize' in globals() else title_fontsize
        if title_fontweight is not None:
            title_kwargs['fontweight'] = title_fontweight
        ax.set_title(title, **title_kwargs)

    if legend:
        handles, _labels = ax.get_legend_handles_labels()
        if len(handles) > 0:
            legend_kwargs = dict(loc=legend_loc, frameon=legend_frame)
            if legend_fontsize is not None:
                legend_kwargs['fontsize'] = clamp_fontsize(legend_fontsize) if 'clamp_fontsize' in globals() else legend_fontsize
            lg = ax.legend(**legend_kwargs)
            if lg is not None:
                for txt in lg.get_texts():
                    txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine_name in ('left', 'bottom'):
        if spine_name in ax.spines:
            ax.spines[spine_name].set_linewidth(1.0)
    tick_w = 1.0
    ax.tick_params(direction='out', length=4, width=tick_w, labelsize=6)
    if not show_xlabel:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    if not show_ylabel:
        ax.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind='ppr')
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)


def _coerce_profile_matrix(curves):
    """Coerce one curve or a curve matrix to a 2D float array."""
    arr = np.asarray(curves, dtype=float)
    if arr.ndim == 1:
        arr = arr[np.newaxis, :]
    if arr.ndim != 2:
        raise ValueError('Each profile input must be a 1D curve or a 2D subject-by-pulse matrix.')
    return arr


def _shared_profile_width(profile_matrices):
    widths = [mat.shape[1] for mat in profile_matrices if mat.size > 0]
    if not widths:
        return 0
    return min(widths)


def _prepare_profile_inputs(group_inputs, pulse_numbers=None):
    """Normalize profile inputs to a shared width and pulse-number vector."""
    prepared = []
    for spec in group_inputs:
        if 'matrix' not in spec:
            raise KeyError("Each group spec must contain a 'matrix' entry.")
        mat = _coerce_profile_matrix(spec['matrix'])
        prepared.append(dict(spec, matrix=mat))

    shared_width = _shared_profile_width([spec['matrix'] for spec in prepared])
    if shared_width <= 0:
        return prepared, np.array([], dtype=float)

    if pulse_numbers is None:
        pulse_arr = np.arange(1, shared_width + 1, dtype=float)
    else:
        pulse_arr = np.asarray(pulse_numbers, dtype=float).ravel()
        if pulse_arr.size < shared_width:
            raise ValueError('pulse_numbers is shorter than the shared profile width.')
        pulse_arr = pulse_arr[:shared_width]

    for spec in prepared:
        spec['matrix'] = spec['matrix'][:, :shared_width]
    return prepared, pulse_arr


def _compute_repeated_measures_components(data_matrix, group_codes):
    """Compute mixed repeated-measures ANOVA components from complete cases."""
    from scipy.stats import f as f_dist

    data = np.asarray(data_matrix, dtype=float)
    codes = np.asarray(group_codes)
    if data.ndim != 2:
        raise ValueError('data_matrix must be 2D.')

    finite_rows = np.isfinite(data).all(axis=1)
    data = data[finite_rows]
    codes = codes[finite_rows]

    unique_codes = pd.unique(codes)
    n_groups = len(unique_codes)
    n_subjects, n_pulses = data.shape
    if n_groups < 2 or n_subjects <= n_groups or n_pulses < 2:
        return None

    group_mats = []
    group_sizes = []
    for code in unique_codes:
        mat = data[codes == code]
        if mat.shape[0] == 0:
            continue
        group_mats.append(mat)
        group_sizes.append(mat.shape[0])

    if len(group_mats) < 2:
        return None

    n_total = int(sum(group_sizes))
    grand_mean = float(np.mean(data))
    group_means = np.array([np.mean(mat) for mat in group_mats], dtype=float)
    pulse_means = np.mean(data, axis=0)
    cell_means = [np.mean(mat, axis=0) for mat in group_mats]

    ss_group = n_pulses * sum(n_g * (g_mean - grand_mean) ** 2 for n_g, g_mean in zip(group_sizes, group_means))
    ss_subjects_within = 0.0
    for mat, g_mean in zip(group_mats, group_means):
        ss_subjects_within += n_pulses * np.sum((np.mean(mat, axis=1) - g_mean) ** 2)

    ss_pulse = n_total * np.sum((pulse_means - grand_mean) ** 2)
    ss_interaction = 0.0
    for n_g, g_mean, gp_mean in zip(group_sizes, group_means, cell_means):
        ss_interaction += n_g * np.sum((gp_mean - g_mean - pulse_means + grand_mean) ** 2)

    ss_total = float(np.sum((data - grand_mean) ** 2))
    ss_error = ss_total - ss_group - ss_subjects_within - ss_pulse - ss_interaction
    if ss_error < 0 and abs(ss_error) < 1e-10:
        ss_error = 0.0

    df_group = len(group_mats) - 1
    df_subjects_within = n_total - len(group_mats)
    df_pulse = n_pulses - 1
    df_interaction = (len(group_mats) - 1) * (n_pulses - 1)
    df_error = (n_total - len(group_mats)) * (n_pulses - 1)
    if min(df_group, df_subjects_within, df_pulse, df_interaction, df_error) <= 0:
        return None

    ms_group = ss_group / df_group if df_group > 0 else np.nan
    ms_subjects_within = ss_subjects_within / df_subjects_within if df_subjects_within > 0 else np.nan
    ms_pulse = ss_pulse / df_pulse if df_pulse > 0 else np.nan
    ms_interaction = ss_interaction / df_interaction if df_interaction > 0 else np.nan
    ms_error = ss_error / df_error if df_error > 0 else np.nan

    f_group = ms_group / ms_subjects_within if np.isfinite(ms_group) and np.isfinite(ms_subjects_within) and ms_subjects_within > 0 else np.nan
    f_pulse = ms_pulse / ms_error if np.isfinite(ms_pulse) and np.isfinite(ms_error) and ms_error > 0 else np.nan
    f_interaction = ms_interaction / ms_error if np.isfinite(ms_interaction) and np.isfinite(ms_error) and ms_error > 0 else np.nan

    p_group = 1 - f_dist.cdf(f_group, df_group, df_subjects_within) if np.isfinite(f_group) else np.nan
    p_pulse = 1 - f_dist.cdf(f_pulse, df_pulse, df_error) if np.isfinite(f_pulse) else np.nan
    p_interaction = 1 - f_dist.cdf(f_interaction, df_interaction, df_error) if np.isfinite(f_interaction) else np.nan

    eta2_group = ss_group / (ss_group + ss_subjects_within) if (ss_group + ss_subjects_within) > 0 else np.nan
    eta2_pulse = ss_pulse / (ss_pulse + ss_error) if (ss_pulse + ss_error) > 0 else np.nan
    eta2_interaction = ss_interaction / (ss_interaction + ss_error) if (ss_interaction + ss_error) > 0 else np.nan

    return {
        'n_total': n_total,
        'n_pulses': n_pulses,
        'group_sizes': group_sizes,
        'SS_between': ss_group,
        'SS_subjects_within': ss_subjects_within,
        'SS_within': ss_pulse,
        'SS_interaction': ss_interaction,
        'SS_error': ss_error,
        'df_between': df_group,
        'df_subjects_within': df_subjects_within,
        'df_within': df_pulse,
        'df_interaction': df_interaction,
        'df_error': df_error,
        'MS_between': ms_group,
        'MS_subjects_within': ms_subjects_within,
        'MS_within': ms_pulse,
        'MS_interaction': ms_interaction,
        'MS_error': ms_error,
        'F_group': f_group,
        'F_pulse': f_pulse,
        'F_interaction': f_interaction,
        'p_group': p_group,
        'p_pulse': p_pulse,
        'p_interaction': p_interaction,
        'eta2_group': eta2_group,
        'eta2_pulse': eta2_pulse,
        'eta2_interaction': eta2_interaction,
    }


def _permutation_profile_interaction_test(data_matrix, group_codes, n_permutations=5000, random_state=42):
    """Non-parametric omnibus test using subject-level label permutations."""
    observed = _compute_repeated_measures_components(data_matrix, group_codes)
    if observed is None or not np.isfinite(observed['F_interaction']):
        return None

    rng = np.random.default_rng(random_state)
    codes = np.asarray(group_codes)
    obs_group = observed['F_group']
    obs_inter = observed['F_interaction']
    ge_group = 1
    ge_inter = 1

    for _ in range(int(n_permutations)):
        shuffled = rng.permutation(codes)
        perm = _compute_repeated_measures_components(data_matrix, shuffled)
        if perm is None:
            continue
        if np.isfinite(obs_group) and np.isfinite(perm['F_group']) and perm['F_group'] >= obs_group:
            ge_group += 1
        if np.isfinite(obs_inter) and np.isfinite(perm['F_interaction']) and perm['F_interaction'] >= obs_inter:
            ge_inter += 1

    denom = int(n_permutations) + 1
    observed['omnibus_test'] = 'Permutation test on Group × Pulse interaction statistic'
    observed['p_group'] = ge_group / denom if np.isfinite(obs_group) else np.nan
    observed['p_pulse'] = np.nan
    observed['p_interaction'] = ge_inter / denom if np.isfinite(obs_inter) else np.nan
    return observed


def _compute_paired_difference_profile_anova(diff_matrix):
    """One-factor repeated-measures ANOVA on paired difference curves across pulses."""
    from scipy.stats import f as f_dist

    diff = np.asarray(diff_matrix, dtype=float)
    if diff.ndim != 2:
        raise ValueError('diff_matrix must be 2D.')
    finite_rows = np.isfinite(diff).all(axis=1)
    diff = diff[finite_rows]
    n_subjects, n_pulses = diff.shape
    if n_subjects < 2 or n_pulses < 2:
        return None

    grand_mean = float(np.mean(diff))
    subject_means = np.mean(diff, axis=1)
    pulse_means = np.mean(diff, axis=0)
    ss_subject = n_pulses * np.sum((subject_means - grand_mean) ** 2)
    ss_pulse = n_subjects * np.sum((pulse_means - grand_mean) ** 2)
    ss_total = float(np.sum((diff - grand_mean) ** 2))
    ss_error = ss_total - ss_subject - ss_pulse
    if ss_error < 0 and abs(ss_error) < 1e-10:
        ss_error = 0.0

    df_subject = n_subjects - 1
    df_pulse = n_pulses - 1
    df_error = (n_subjects - 1) * (n_pulses - 1)
    if min(df_subject, df_pulse, df_error) <= 0:
        return None

    ms_pulse = ss_pulse / df_pulse
    ms_error = ss_error / df_error if df_error > 0 else np.nan
    f_interaction = ms_pulse / ms_error if np.isfinite(ms_error) and ms_error > 0 else np.nan
    p_interaction = 1 - f_dist.cdf(f_interaction, df_pulse, df_error) if np.isfinite(f_interaction) else np.nan
    eta2_interaction = ss_pulse / (ss_pulse + ss_error) if (ss_pulse + ss_error) > 0 else np.nan

    return {
        'n_subjects': n_subjects,
        'n_pulses': n_pulses,
        'SS_subject': ss_subject,
        'SS_pulse': ss_pulse,
        'SS_error': ss_error,
        'df_subject': df_subject,
        'df_pulse': df_pulse,
        'df_error': df_error,
        'MS_pulse': ms_pulse,
        'MS_error': ms_error,
        'F_interaction': f_interaction,
        'p_interaction': p_interaction,
        'eta2_interaction': eta2_interaction,
    }


def _friedman_paired_difference_profile_test(diff_matrix):
    """Non-parametric paired omnibus test on pulse-varying differences."""
    from scipy.stats import friedmanchisquare

    diff = np.asarray(diff_matrix, dtype=float)
    if diff.ndim != 2:
        raise ValueError('diff_matrix must be 2D.')
    finite_rows = np.isfinite(diff).all(axis=1)
    diff = diff[finite_rows]
    n_subjects, n_pulses = diff.shape
    if n_subjects < 2 or n_pulses < 2:
        return None

    stat, p_value = friedmanchisquare(*[diff[:, j] for j in range(n_pulses)])
    return {
        'n_subjects': n_subjects,
        'n_pulses': n_pulses,
        'F_interaction': stat,
        'p_interaction': p_value,
        'df_pulse': n_pulses - 1,
        'df_error': np.nan,
        'eta2_interaction': np.nan,
    }


def _per_pulse_profile_tests(group_matrices, group_labels, pulse_numbers, *, parametric_stat=True, alpha=0.05):
    """Pulse-wise follow-up tests with FDR correction."""
    from scipy.stats import f_oneway, kruskal, mannwhitneyu, ttest_ind
    from statsmodels.stats.multitest import multipletests

    rows = []
    for pulse_idx, pulse_num in enumerate(pulse_numbers):
        values = []
        for mat in group_matrices:
            vals = np.asarray(mat[:, pulse_idx], dtype=float)
            vals = vals[np.isfinite(vals)]
            values.append(vals)

        if sum(len(vals) > 0 for vals in values) < 2:
            continue

        if parametric_stat:
            if len(values) == 2:
                statistic, p_value = ttest_ind(values[0], values[1], equal_var=False, nan_policy='omit')
                test_name = 'Welch t-test'
                statistic_name = 't_statistic'
            else:
                statistic, p_value = f_oneway(*values)
                test_name = 'One-way ANOVA'
                statistic_name = 'F_statistic'
        else:
            if len(values) == 2:
                statistic, p_value = mannwhitneyu(values[0], values[1], alternative='two-sided')
                test_name = 'Mann-Whitney U'
                statistic_name = 'U_statistic'
            else:
                statistic, p_value = kruskal(*values)
                test_name = 'Kruskal-Wallis'
                statistic_name = 'H_statistic'

        row = {
            'Pulse': pulse_num,
            'test': test_name,
            statistic_name: statistic,
            'p_value_raw': p_value,
        }
        for label, vals in zip(group_labels, values):
            row[f'{label}_mean'] = float(np.mean(vals)) if len(vals) else np.nan
            row[f'{label}_sem'] = float(np.std(vals, ddof=1) / np.sqrt(len(vals))) if len(vals) > 1 else 0.0
            row[f'{label}_n'] = int(len(vals))
        rows.append(row)

    per_pulse_df = pd.DataFrame(rows)
    if per_pulse_df.empty:
        return per_pulse_df

    _, p_corr, _, _ = multipletests(per_pulse_df['p_value_raw'], alpha=alpha, method='fdr_bh')
    per_pulse_df['p_value_corrected'] = p_corr
    per_pulse_df['significant'] = per_pulse_df['p_value_corrected'] < alpha
    return per_pulse_df


def _paired_per_pulse_profile_tests(group_a, group_b, pulse_numbers, *, parametric_stat=True, alpha=0.05):
    """Pulse-wise paired follow-up tests with FDR correction."""
    from scipy.stats import ttest_rel, wilcoxon
    from statsmodels.stats.multitest import multipletests

    a = np.asarray(group_a, dtype=float)
    b = np.asarray(group_b, dtype=float)
    rows = []
    for pulse_idx, pulse_num in enumerate(pulse_numbers):
        vals_a = a[:, pulse_idx]
        vals_b = b[:, pulse_idx]
        mask = np.isfinite(vals_a) & np.isfinite(vals_b)
        vals_a = vals_a[mask]
        vals_b = vals_b[mask]
        if len(vals_a) < 2:
            continue

        if parametric_stat:
            statistic, p_value = ttest_rel(vals_a, vals_b, nan_policy='omit')
            test_name = 'Paired t-test'
            statistic_name = 't_statistic'
        else:
            statistic, p_value = wilcoxon(vals_a, vals_b, alternative='two-sided', zero_method='wilcox')
            test_name = 'Wilcoxon signed-rank'
            statistic_name = 'W_statistic'

        rows.append({
            'Pulse': pulse_num,
            'test': test_name,
            statistic_name: statistic,
            'p_value_raw': p_value,
            'GroupA_mean': float(np.mean(vals_a)),
            'GroupA_sem': float(np.std(vals_a, ddof=1) / np.sqrt(len(vals_a))) if len(vals_a) > 1 else 0.0,
            'GroupA_n': int(len(vals_a)),
            'GroupB_mean': float(np.mean(vals_b)),
            'GroupB_sem': float(np.std(vals_b, ddof=1) / np.sqrt(len(vals_b))) if len(vals_b) > 1 else 0.0,
            'GroupB_n': int(len(vals_b)),
        })

    per_pulse_df = pd.DataFrame(rows)
    if per_pulse_df.empty:
        return per_pulse_df

    _, p_corr, _, _ = multipletests(per_pulse_df['p_value_raw'], alpha=alpha, method='fdr_bh')
    per_pulse_df['p_value_corrected'] = p_corr
    per_pulse_df['significant'] = per_pulse_df['p_value_corrected'] < alpha
    return per_pulse_df


def _pvalue_to_stars(p_value):
    if not np.isfinite(p_value):
        return ''
    if p_value < 0.001:
        return '***'
    if p_value < 0.01:
        return '**'
    if p_value < 0.05:
        return '*'
    return ''


def _strip_group_count_suffix(label):
    text = str(label)
    return re.sub(r'\s*\(n\s*=.*?\)\s*$', '', text, flags=re.IGNORECASE).strip()




def _should_print_wt_profile_summary(group_labels, summary_label):
    labels_lower = [str(label).lower() for label in group_labels]
    summary_lower = str(summary_label).lower()
    return any('wt' in label for label in labels_lower) or 'wt' in summary_lower


def _profile_significant_pulse_string(per_pulse_df):
    sig_pulses = []
    if per_pulse_df is not None and not per_pulse_df.empty and 'significant' in per_pulse_df.columns:
        sig_pulses = per_pulse_df.loc[per_pulse_df['significant'], 'Pulse'].tolist()
    pulse_str = ','.join(str(int(p)) if float(p).is_integer() else f'{p:g}' for p in sig_pulses) if sig_pulses else 'none'
    return pulse_str, sig_pulses


def _compute_profile_level_test(group_matrices, *, omnibus=None, paired_groups=False, parametric_stat=True):
    from scipy.stats import f_oneway, kruskal, mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

    matrices = [np.asarray(mat, dtype=float) for mat in group_matrices]
    if not paired_groups and parametric_stat and omnibus is not None and np.isfinite(omnibus.get('p_group', np.nan)):
        return {
            'effect_label': 'Overall level',
            'test_name': 'Group main effect',
            'statistic_label': 'F',
            'statistic': omnibus.get('F_group', np.nan),
            'df1': omnibus.get('df_between', np.nan),
            'df2': omnibus.get('df_subjects_within', np.nan),
            'p_value': omnibus.get('p_group', np.nan),
            'effect_size': omnibus.get('eta2_group', np.nan),
            'source_key': 'group_main_effect',
        }

    subject_means = []
    for mat in matrices:
        if mat.ndim != 2 or mat.size == 0:
            subject_means.append(np.asarray([], dtype=float))
            continue
        means = np.nanmean(mat, axis=1)
        subject_means.append(means[np.isfinite(means)])

    if paired_groups:
        if len(subject_means) != 2 or subject_means[0].size != subject_means[1].size or subject_means[0].size < 2:
            return None
        if parametric_stat:
            statistic, p_value = ttest_rel(subject_means[0], subject_means[1], nan_policy='omit')
            return {
                'effect_label': 'Overall level',
                'test_name': 'Paired t-test on profile means',
                'statistic_label': 't',
                'statistic': statistic,
                'df1': subject_means[0].size - 1,
                'df2': np.nan,
                'p_value': p_value,
                'effect_size': np.nan,
                'source_key': 'paired_mean_test',
            }
        diffs = subject_means[1] - subject_means[0]
        diffs = diffs[np.isfinite(diffs)]
        if diffs.size == 0 or np.allclose(diffs, 0.0):
            statistic, p_value = 0.0, 1.0
        else:
            statistic, p_value = wilcoxon(subject_means[0], subject_means[1])
        return {
            'effect_label': 'Overall level',
            'test_name': 'Wilcoxon signed-rank on profile means',
            'statistic_label': 'W',
            'statistic': statistic,
            'df1': subject_means[0].size,
            'df2': np.nan,
            'p_value': p_value,
            'effect_size': np.nan,
            'source_key': 'paired_mean_test',
        }

    analyzable = [arr for arr in subject_means if arr.size > 0]
    if len(analyzable) < 2:
        return None

    if len(analyzable) == 2:
        if parametric_stat:
            statistic, p_value = ttest_ind(analyzable[0], analyzable[1], equal_var=False, nan_policy='omit')
            return {
                'effect_label': 'Overall level',
                'test_name': 'Welch t-test on profile means',
                'statistic_label': 't',
                'statistic': statistic,
                'df1': np.nan,
                'df2': np.nan,
                'p_value': p_value,
                'effect_size': np.nan,
                'source_key': 'mean_profile_test',
            }
        statistic, p_value = mannwhitneyu(analyzable[0], analyzable[1], alternative='two-sided')
        return {
            'effect_label': 'Overall level',
            'test_name': 'Mann-Whitney U on profile means',
            'statistic_label': 'U',
            'statistic': statistic,
            'df1': np.nan,
            'df2': np.nan,
            'p_value': p_value,
            'effect_size': np.nan,
            'source_key': 'mean_profile_test',
        }

    if parametric_stat:
        statistic, p_value = f_oneway(*analyzable)
        total_n = int(sum(arr.size for arr in analyzable))
        return {
            'effect_label': 'Overall level',
            'test_name': 'One-way ANOVA on profile means',
            'statistic_label': 'F',
            'statistic': statistic,
            'df1': len(analyzable) - 1,
            'df2': total_n - len(analyzable),
            'p_value': p_value,
            'effect_size': np.nan,
            'source_key': 'mean_profile_test',
        }
    statistic, p_value = kruskal(*analyzable)
    return {
        'effect_label': 'Overall level',
        'test_name': 'Kruskal-Wallis on profile means',
        'statistic_label': 'H',
        'statistic': statistic,
        'df1': len(analyzable) - 1,
        'df2': np.nan,
        'p_value': p_value,
        'effect_size': np.nan,
        'source_key': 'mean_profile_test',
    }


def _format_profile_effect_text(test_result):
    if not test_result:
        return 'overall level test unavailable'
    label = str(test_result.get('effect_label', 'Effect')).strip()
    statistic_label = str(test_result.get('statistic_label', 'stat')).strip()
    statistic = test_result.get('statistic', np.nan)
    p_value = test_result.get('p_value', np.nan)
    df1 = test_result.get('df1', np.nan)
    df2 = test_result.get('df2', np.nan)
    effect_size = test_result.get('effect_size', np.nan)

    if statistic_label == 'F' and np.isfinite(df1) and np.isfinite(df2):
        text = f"{label} F({df1:.0f},{df2:.0f})={statistic:.3f}, p={p_value:.3g}"
    elif statistic_label == 't' and np.isfinite(df1):
        text = f"{label} t({df1:.0f})={statistic:.3f}, p={p_value:.3g}"
    elif statistic_label in {'U', 'W', 'H'}:
        text = f"{label} {statistic_label}={statistic:.3f}, p={p_value:.3g}"
    else:
        text = f"{label} {statistic_label}={statistic:.3f}, p={p_value:.3g}"

    if np.isfinite(effect_size):
        text += f", η²p={effect_size:.3f}"
    return text


def _format_profile_shape_text(omnibus, *, paired_groups=False, parametric_stat=True):
    effect_label = 'Condition×Pulse' if paired_groups else 'Group×Pulse'
    statistic = omnibus.get('F_interaction', np.nan)
    p_value = omnibus.get('p_interaction', np.nan)
    effect_size = omnibus.get('eta2_interaction', np.nan)
    df1 = omnibus.get('df_pulse', np.nan) if paired_groups else omnibus.get('df_interaction', np.nan)
    df2 = omnibus.get('df_error', np.nan)

    if parametric_stat:
        text = f"{effect_label} F({df1:.0f},{df2:.0f})={statistic:.3f}, p={p_value:.3g}"
        if np.isfinite(effect_size):
            text += f", η²p={effect_size:.3f}"
        return text
    return f"{effect_label} statistic={statistic:.3f}, p={p_value:.3g}"


def _format_two_group_profile_summary(group_labels, complete_matrices, omnibus, per_pulse_df, *, level_test=None, overall_curve_significant=False, paired_groups=False, parametric_stat=True):
    if len(group_labels) != 2 or len(complete_matrices) != 2:
        return None
    left_label = _strip_group_count_suffix(group_labels[0])
    right_label = _strip_group_count_suffix(group_labels[1])
    pulse_str, _ = _profile_significant_pulse_string(per_pulse_df)
    n_left = int(complete_matrices[0].shape[0])
    n_right = int(complete_matrices[1].shape[0])
    overall_text = 'yes' if overall_curve_significant else 'no'
    return (
        f"WT-style curve summary: {left_label} vs {right_label} | "
        f"overall difference={overall_text} | "
        f"{_format_profile_effect_text(level_test)} | "
        f"{_format_profile_shape_text(omnibus, paired_groups=paired_groups, parametric_stat=parametric_stat)} | "
        f"FDR-significant pulses={pulse_str} | n={n_left} vs {n_right}"
    )


def summarize_profile_group_stats(
    group_inputs,
    *,
    pulse_numbers=None,
    stats_start_index=0,
    paired_groups=False,
    parametric_stat=True,
    alpha=0.05,
    n_permutations=5000,
    summary_label='Profile comparison',
    print_summary=True,
):
    """Shared omnibus + pulse-wise stats for profile-shaped data."""
    prepared, pulse_arr = _prepare_profile_inputs(group_inputs, pulse_numbers=pulse_numbers)
    if pulse_arr.size == 0:
        return {
            'summary_line': f'{summary_label}: no valid profile data.',
            'omnibus_df': pd.DataFrame(),
            'per_pulse_df': pd.DataFrame(),
            'stats_pulse_numbers': np.array([], dtype=float),
        }

    if int(stats_start_index) < 0 or int(stats_start_index) >= pulse_arr.size:
        raise ValueError('stats_start_index must point to a valid column within the plotted profiles.')

    group_labels = [str(spec.get('label', f'Group {i + 1}')) for i, spec in enumerate(prepared)]
    group_matrices = [spec['matrix'] for spec in prepared]
    tested_pulses = pulse_arr[int(stats_start_index):]
    tested_matrices = [mat[:, int(stats_start_index):] for mat in group_matrices]

    if paired_groups:
        if len(tested_matrices) != 2:
            raise ValueError('paired_groups=True requires exactly two aligned groups.')
        if tested_matrices[0].shape != tested_matrices[1].shape:
            raise ValueError('Paired profile matrices must have identical shapes.')
        pair_mask = np.isfinite(tested_matrices[0]).all(axis=1) & np.isfinite(tested_matrices[1]).all(axis=1)
        tested_a = tested_matrices[0][pair_mask]
        tested_b = tested_matrices[1][pair_mask]
        if tested_a.shape[0] < 2 or tested_pulses.size < 2:
            summary_line = f'{summary_label}: not enough paired complete curves for profile statistics.'
            if print_summary:
                print(summary_line)
            return {
                'summary_line': summary_line,
                'omnibus_df': pd.DataFrame(),
                'per_pulse_df': pd.DataFrame(),
                'stats_pulse_numbers': tested_pulses,
                'group_labels': group_labels,
                'complete_case_n': {group_labels[0]: int(tested_a.shape[0]), group_labels[1]: int(tested_b.shape[0])},
            }

        diff_matrix = tested_b - tested_a
        if parametric_stat:
            omnibus = _compute_paired_difference_profile_anova(diff_matrix)
            omnibus_test_name = 'Paired repeated-measures ANOVA on condition differences'
            pulse_test_name = 'Paired t-test'
        else:
            omnibus = _friedman_paired_difference_profile_test(diff_matrix)
            omnibus_test_name = 'Friedman test on condition differences'
            pulse_test_name = 'Wilcoxon signed-rank'

        if omnibus is None:
            summary_line = f'{summary_label}: paired profile-shape test unavailable.'
            if print_summary:
                print(summary_line)
            return {
                'summary_line': summary_line,
                'omnibus_df': pd.DataFrame(),
                'per_pulse_df': pd.DataFrame(),
                'stats_pulse_numbers': tested_pulses,
                'group_labels': group_labels,
                'complete_case_n': {group_labels[0]: int(tested_a.shape[0]), group_labels[1]: int(tested_b.shape[0])},
            }

        per_pulse_df = _paired_per_pulse_profile_tests(
            tested_a,
            tested_b,
            tested_pulses,
            parametric_stat=parametric_stat,
            alpha=alpha,
        )
        level_test = _compute_profile_level_test([tested_a, tested_b], paired_groups=True, parametric_stat=parametric_stat)
        level_p = level_test.get('p_value', np.nan) if level_test else np.nan
        shape_p = omnibus.get('p_interaction', np.nan)
        overall_curve_significant = bool((np.isfinite(level_p) and level_p < alpha) or (np.isfinite(shape_p) and shape_p < alpha))
        pulse_str, _ = _profile_significant_pulse_string(per_pulse_df)
        n_summary = f'paired n={tested_a.shape[0]}'

        omnibus_df = pd.DataFrame({
            'Source': ['Overall level', 'Subjects', 'Condition × Pulse', 'Error'],
            'SS': [np.nan, omnibus.get('SS_subject', np.nan), omnibus.get('SS_pulse', np.nan), omnibus.get('SS_error', np.nan)],
            'df': [level_test.get('df1', np.nan) if level_test else np.nan, omnibus.get('df_subject', np.nan), omnibus.get('df_pulse', np.nan), omnibus.get('df_error', np.nan)],
            'MS': [np.nan, np.nan, omnibus.get('MS_pulse', np.nan), omnibus.get('MS_error', np.nan)],
            'F': [level_test.get('statistic', np.nan) if level_test and level_test.get('statistic_label') == 'F' else np.nan, np.nan, omnibus.get('F_interaction', np.nan), np.nan],
            'p_value': [level_p, np.nan, shape_p, np.nan],
            'partial_eta2': [level_test.get('effect_size', np.nan) if level_test else np.nan, np.nan, omnibus.get('eta2_interaction', np.nan), np.nan],
            'Significant': [bool(np.isfinite(level_p) and level_p < alpha), np.nan, bool(np.isfinite(shape_p) and shape_p < alpha), np.nan],
        })

        summary_line = (
            f"{summary_label}: overall curve difference={'yes' if overall_curve_significant else 'no'}; "
            f"{_format_profile_effect_text(level_test)}; "
            f"{_format_profile_shape_text(omnibus, paired_groups=True, parametric_stat=parametric_stat)}; "
            f"FDR-significant pulses ({pulse_test_name})={pulse_str}; {n_summary}"
        )

        if print_summary:
            print(summary_line)

        return {
            'summary_line': summary_line,
            'omnibus_df': omnibus_df,
            'per_pulse_df': per_pulse_df,
            'stats_pulse_numbers': tested_pulses,
            'group_labels': group_labels,
            'complete_case_n': {group_labels[0]: int(tested_a.shape[0]), group_labels[1]: int(tested_b.shape[0])},
            'omnibus_test_name': omnibus_test_name,
            'pulse_test_name': pulse_test_name,
            'omnibus_raw': omnibus,
            'overall_level_test': level_test,
            'overall_curve_significant': overall_curve_significant,
            'overall_level_p': level_p,
            'overall_shape_p': shape_p,
        }

    complete_matrices = [mat[np.isfinite(mat).all(axis=1)] for mat in tested_matrices]
    analyzable = [(label, mat) for label, mat in zip(group_labels, complete_matrices) if mat.shape[0] > 0]

    if len(analyzable) < 2 or tested_pulses.size < 2:
        summary_line = f'{summary_label}: not enough complete curves for profile statistics.'
        if print_summary:
            print(summary_line)
        return {
            'summary_line': summary_line,
            'omnibus_df': pd.DataFrame(),
            'per_pulse_df': pd.DataFrame(),
            'stats_pulse_numbers': tested_pulses,
            'group_labels': group_labels,
            'complete_case_n': {label: int(mat.shape[0]) for label, mat in zip(group_labels, complete_matrices)},
        }

    group_labels = [label for label, _ in analyzable]
    complete_matrices = [mat for _, mat in analyzable]
    data_matrix = np.vstack(complete_matrices)
    group_codes = np.concatenate([np.full(mat.shape[0], idx, dtype=int) for idx, mat in enumerate(complete_matrices)])

    if parametric_stat:
        omnibus = _compute_repeated_measures_components(data_matrix, group_codes)
        omnibus_test_name = 'Two-way repeated-measures ANOVA'
        pulse_test_name = 'Welch t-test' if len(complete_matrices) == 2 else 'One-way ANOVA'
    else:
        omnibus = _permutation_profile_interaction_test(data_matrix, group_codes, n_permutations=n_permutations)
        omnibus_test_name = 'Permutation Group × Pulse test'
        pulse_test_name = 'Mann-Whitney U' if len(complete_matrices) == 2 else 'Kruskal-Wallis'

    if omnibus is None:
        summary_line = f'{summary_label}: profile-shape test unavailable.'
        if print_summary:
            print(summary_line)
        return {
            'summary_line': summary_line,
            'omnibus_df': pd.DataFrame(),
            'per_pulse_df': pd.DataFrame(),
            'stats_pulse_numbers': tested_pulses,
            'group_labels': group_labels,
            'complete_case_n': {label: int(mat.shape[0]) for label, mat in zip(group_labels, complete_matrices)},
        }

    per_pulse_df = _per_pulse_profile_tests(
        tested_matrices,
        [str(spec.get('label', f'Group {i + 1}')) for i, spec in enumerate(prepared)],
        tested_pulses,
        parametric_stat=parametric_stat,
        alpha=alpha,
    )
    level_test = _compute_profile_level_test(complete_matrices, omnibus=omnibus, paired_groups=False, parametric_stat=parametric_stat)
    level_p = level_test.get('p_value', np.nan) if level_test else np.nan
    shape_p = omnibus.get('p_interaction', np.nan)
    overall_curve_significant = bool((np.isfinite(level_p) and level_p < alpha) or (np.isfinite(shape_p) and shape_p < alpha))
    pulse_str, _ = _profile_significant_pulse_string(per_pulse_df)
    n_summary = ', '.join(f'{label} n={mat.shape[0]}' for label, mat in zip(group_labels, complete_matrices))

    omnibus_df = pd.DataFrame({
        'Source': ['Overall level', 'Subjects(Group)', 'Pulse', 'Group × Pulse', 'Error'],
        'SS': [omnibus.get('SS_between', np.nan), omnibus.get('SS_subjects_within', np.nan), omnibus.get('SS_within', np.nan), omnibus.get('SS_interaction', np.nan), omnibus.get('SS_error', np.nan)],
        'df': [omnibus.get('df_between', np.nan), omnibus.get('df_subjects_within', np.nan), omnibus.get('df_within', np.nan), omnibus.get('df_interaction', np.nan), omnibus.get('df_error', np.nan)],
        'MS': [omnibus.get('MS_between', np.nan), omnibus.get('MS_subjects_within', np.nan), omnibus.get('MS_within', np.nan), omnibus.get('MS_interaction', np.nan), omnibus.get('MS_error', np.nan)],
        'F': [omnibus.get('F_group', np.nan), np.nan, omnibus.get('F_pulse', np.nan), omnibus.get('F_interaction', np.nan), np.nan],
        'p_value': [level_p, np.nan, omnibus.get('p_pulse', np.nan), shape_p, np.nan],
        'partial_eta2': [level_test.get('effect_size', np.nan) if level_test else np.nan, np.nan, omnibus.get('eta2_pulse', np.nan), omnibus.get('eta2_interaction', np.nan), np.nan],
        'Significant': [bool(np.isfinite(level_p) and level_p < alpha), np.nan, bool(np.isfinite(omnibus.get('p_pulse', np.nan)) and omnibus.get('p_pulse', np.nan) < alpha), bool(np.isfinite(shape_p) and shape_p < alpha), np.nan],
    })

    summary_line = (
        f"{summary_label}: overall curve difference={'yes' if overall_curve_significant else 'no'}; "
        f"{_format_profile_effect_text(level_test)}; "
        f"{_format_profile_shape_text(omnibus, paired_groups=False, parametric_stat=parametric_stat)}; "
        f"FDR-significant pulses ({pulse_test_name})={pulse_str}; {n_summary}"
    )

    if print_summary:
        print(summary_line)
        if _should_print_wt_profile_summary(group_labels, summary_label):
            wt_line = _format_two_group_profile_summary(
                group_labels,
                complete_matrices,
                omnibus,
                per_pulse_df,
                level_test=level_test,
                overall_curve_significant=overall_curve_significant,
                paired_groups=False,
                parametric_stat=parametric_stat,
            )
            if wt_line is not None:
                print(wt_line)

    return {
        'summary_line': summary_line,
        'omnibus_df': omnibus_df,
        'per_pulse_df': per_pulse_df,
        'stats_pulse_numbers': tested_pulses,
        'group_labels': group_labels,
        'complete_case_n': {label: int(mat.shape[0]) for label, mat in zip(group_labels, complete_matrices)},
        'omnibus_test_name': omnibus_test_name,
        'pulse_test_name': pulse_test_name,
        'omnibus_raw': omnibus,
        'overall_level_test': level_test,
        'overall_curve_significant': overall_curve_significant,
        'overall_level_p': level_p,
        'overall_shape_p': shape_p,
    }


def annotate_profile_significance(ax, stats_result, *, color='black', fontsize=5, pad_fraction=0.08):
    """Annotate pulses that survive FDR correction."""
    if stats_result is None:
        return
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame())
    if per_pulse_df is None or per_pulse_df.empty or 'significant' not in per_pulse_df.columns:
        return

    sig_df = per_pulse_df[per_pulse_df['significant']].copy()
    if sig_df.empty:
        return

    y_min, y_max = ax.get_ylim()
    y_span = y_max - y_min
    if not np.isfinite(y_span) or y_span <= 0:
        y_span = 1.0
    y_text = y_max + pad_fraction * y_span
    y_top = y_max

    for _, row in sig_df.iterrows():
        stars = _pvalue_to_stars(row.get('p_value_corrected', np.nan))
        if not stars:
            stars = '•'
        txt = ax.text(row['Pulse'], y_text, stars, ha='center', va='bottom', color=color, fontsize=fontsize, fontweight='bold', clip_on=False)
        setattr(txt, '_keep_in_panel_annotation', True)
        y_top = max(y_top, y_text)

    ax.set_ylim(y_min, y_top + pad_fraction * y_span)


def plot_profile_group_comparison(
    ax,
    group_inputs,
    *,
    pulse_numbers=None,
    stats_start_index=0,
    show_stats=False,
    paired_groups=False,
    parametric_stat=True,
    alpha=0.05,
    n_permutations=5000,
    summary_label='Profile comparison',
    title=None,
    xlabel='Stimulus',
    ylabel='PPR (A_n/A_1)',
    finalize_kwargs=None,
):
    """Shared plotting entry point for PPR-like or cumulative profile comparisons."""
    prepared, pulse_arr = _prepare_profile_inputs(group_inputs, pulse_numbers=pulse_numbers)
    for spec in prepared:
        matrix = spec['matrix']
        if matrix.size == 0:
            continue
        mean_profile = np.nanmean(matrix, axis=0)
        n_eff = np.sum(np.isfinite(matrix), axis=0)
        if matrix.shape[0] > 1:
            sem_profile = np.nanstd(matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
            sem_profile[n_eff < 2] = 0.0
        else:
            sem_profile = np.zeros_like(mean_profile)
        plot_ppr_mean_sem(
            ax,
            pulse_arr,
            mean_profile,
            sem_profile,
            color=spec.get('color', 'black'),
            marker=spec.get('marker', 'o'),
            label=spec.get('label', 'Group'),
            linewidth=spec.get('linewidth', 2.0),
            markersize=spec.get('markersize', 6),
            sem_alpha=spec.get('sem_alpha', 0.2),
        )

    finalize_kwargs = dict(finalize_kwargs or {})
    finalize_kwargs.setdefault('title', title)
    finalize_kwargs.setdefault('xlabel', xlabel)
    finalize_kwargs.setdefault('ylabel', ylabel)
    finalize_ppr_axis(ax, pulse_arr, **finalize_kwargs)

    stats_result = None
    if len(prepared) >= 2:
        stats_result = summarize_profile_group_stats(
            prepared,
            pulse_numbers=pulse_arr,
            stats_start_index=stats_start_index,
            paired_groups=paired_groups,
            parametric_stat=parametric_stat,
            alpha=alpha,
            n_permutations=n_permutations,
            summary_label=summary_label,
            print_summary=True,
        )
        if show_stats:
            annotate_profile_significance(ax, stats_result)

    return stats_result


# ===== Shared PCA Helpers (moved from later cell) =====
def get_cluster_color(cluster_id):
    cluster_id = int(cluster_id)
    if 'cluster_color_lookup' in globals() and cluster_id in cluster_color_lookup:
        return cluster_color_lookup[cluster_id]
    cmap = plt.get_cmap('tab10')
    return cmap((cluster_id - 1) % 10)

def get_cluster_colors(labels):
    return [get_cluster_color(cid) for cid in labels]

def get_cluster_hex_color(cluster_id):
    import matplotlib.colors as mcolors

    cluster_id = int(cluster_id)
    if 'cluster_hex_colors' in globals() and 1 <= cluster_id <= len(cluster_hex_colors):
        return cluster_hex_colors[cluster_id - 1]
    return mcolors.to_hex(get_cluster_color(cluster_id))

def plot_pca_background(
    ax,
    *,
    coords=None,
    labels=None,
    use_cluster_colors=True,
    color='gray',
    alpha=0.3,
    s=30,
    marker='o',
    edgecolors='none',
    linewidths=0.0,
    label='WT pooled',
    zorder=1,
):
    """Plot WT-like PCA background cloud with optional cluster colors."""
    if coords is None:
        coords = pca_coordinates
    if coords is None or len(coords) == 0:
        return None

    if use_cluster_colors:
        if labels is None:
            labels = cluster_assignments
        colors = get_cluster_colors(labels)
    else:
        colors = color

    s = clamp_scatter_area(s) if 'clamp_scatter_area' in globals() else s
    linewidths = clamp_linewidth(linewidths) if ('clamp_linewidth' in globals() and linewidths is not None) else linewidths
    kwargs = dict(c=colors, alpha=alpha, s=s, marker=marker, edgecolors=edgecolors, linewidths=linewidths, zorder=zorder)
    if label is not None:
        kwargs['label'] = label
    return ax.scatter(coords[:, 0], coords[:, 1], **kwargs)


def plot_pca_overlay_points(
    ax,
    coords,
    *,
    y=None,
    color='red',
    marker='o',
    s=30,
    alpha=0.8,
    edgecolors='black',
    linewidths=0.0,
    label=None,
    zorder=3,
    **kwargs,
):
    """Plot overlay points in PCA space from Nx2 coords or x/y vectors."""
    if coords is None:
        return None

    if y is not None:
        x_vals = np.asarray(coords)
        y_vals = np.asarray(y)
        if x_vals.size == 0 or y_vals.size == 0:
            return None
        coords = np.column_stack([x_vals.reshape(-1), y_vals.reshape(-1)])
    else:
        coords = np.asarray(coords)
        if coords.size == 0:
            return None
        if coords.ndim == 1 and coords.size == 2:
            coords = coords.reshape(1, 2)

    scatter_kwargs = dict(
        marker=marker,
        s=s,
        alpha=alpha,
        zorder=zorder,
    )
    if edgecolors is not None and 'edgecolors' not in kwargs and 'edgecolor' not in kwargs:
        scatter_kwargs['edgecolors'] = edgecolors
    if linewidths is not None and 'linewidths' not in kwargs and 'linewidth' not in kwargs and 'lw' not in kwargs:
        scatter_kwargs['linewidths'] = linewidths
    if color is not None and 'c' not in kwargs and 'color' not in kwargs:
        scatter_kwargs['c'] = color
    scatter_kwargs.update(kwargs)

    # Normalize matplotlib aliases to avoid conflicts (e.g., lw + linewidths)
    if 'lw' in scatter_kwargs:
        if 'linewidths' not in scatter_kwargs and 'linewidth' not in scatter_kwargs:
            scatter_kwargs['linewidths'] = scatter_kwargs['lw']
        scatter_kwargs.pop('lw', None)
    if label is not None:
        scatter_kwargs['label'] = label
    if 'clamp_scatter_area' in globals() and 's' in scatter_kwargs:
        scatter_kwargs['s'] = clamp_scatter_area(scatter_kwargs['s'])
    if 'clamp_linewidth' in globals():
        if 'linewidths' in scatter_kwargs and scatter_kwargs['linewidths'] is not None:
            scatter_kwargs['linewidths'] = clamp_linewidth(scatter_kwargs['linewidths'])
        if 'linewidth' in scatter_kwargs and scatter_kwargs['linewidth'] is not None:
            scatter_kwargs['linewidth'] = clamp_linewidth(scatter_kwargs['linewidth'])
    return ax.scatter(coords[:, 0], coords[:, 1], **scatter_kwargs)


def get_reference_pca_limits(pad=0.5, fallback_x=(-7.0, 10.0), fallback_y=(-6.0, 6.0), pad_frac=0.05):
    """Return the WT reference PCA window from the WT cloud itself, with a small margin."""
    coords = None
    if 'pca_data' in globals() and isinstance(globals().get('pca_data'), dict) and 'WT_pooled' in globals().get('pca_data', {}):
        arr = np.asarray(globals()['pca_data']['WT_pooled'], float)
        if arr.ndim == 2 and arr.shape[0] > 0 and arr.shape[1] >= 2:
            coords = arr[:, :2]
    if coords is None and 'pca_coordinates' in globals() and globals().get('pca_coordinates') is not None:
        arr = np.asarray(globals().get('pca_coordinates'), float)
        if arr.ndim == 2 and arr.shape[0] > 0 and arr.shape[1] >= 2:
            coords = arr[:, :2]
    if coords is None:
        return tuple(float(v) for v in fallback_x), tuple(float(v) for v in fallback_y)
    x = coords[:, 0][np.isfinite(coords[:, 0])]
    y = coords[:, 1][np.isfinite(coords[:, 1])]
    if len(x) == 0 or len(y) == 0:
        return tuple(float(v) for v in fallback_x), tuple(float(v) for v in fallback_y)
    xmn, xmx = float(np.min(x)), float(np.max(x))
    ymn, ymx = float(np.min(y)), float(np.max(y))
    xpad = max(float(pad), float((xmx - xmn) * pad_frac)) if np.isfinite(xmx - xmn) else float(pad)
    ypad = max(float(pad), float((ymx - ymn) * pad_frac)) if np.isfinite(ymx - ymn) else float(pad)
    return (xmn - xpad, xmx + xpad), (ymn - ypad, ymx + ypad)

def _draw_reference_pca_box(ax, ref_xlim, ref_ylim):
    """Show the WT pooled reference PCA footprint when current axes extend beyond it."""
    for ln in list(ax.lines):
        try:
            if ln.get_gid() == 'wt_ref_pca_box':
                ln.remove()
        except Exception:
            pass
    cur_xlim = ax.get_xlim()
    cur_ylim = ax.get_ylim()
    bigger = (
        (cur_xlim[0] < ref_xlim[0] - 1e-9) or (cur_xlim[1] > ref_xlim[1] + 1e-9)
        or (cur_ylim[0] < ref_ylim[0] - 1e-9) or (cur_ylim[1] > ref_ylim[1] + 1e-9)
    )
    if not bigger:
        return None
    keep_xlim = ax.get_xlim()
    keep_ylim = ax.get_ylim()
    xs = [ref_xlim[0], ref_xlim[1], ref_xlim[1], ref_xlim[0], ref_xlim[0]]
    ys = [ref_ylim[0], ref_ylim[0], ref_ylim[1], ref_ylim[1], ref_ylim[0]]
    line, = ax.plot(xs, ys, ls=':', lw=0.9, color='0.55', alpha=0.8, zorder=2)
    line.set_gid('wt_ref_pca_box')
    ax.set_xlim(keep_xlim)
    ax.set_ylim(keep_ylim)
    return line

def _pca_artist_data_limits(ax):
    """Return finite PCA point/curve bounds from plotted artists only."""
    xs = []
    ys = []
    for coll in getattr(ax, 'collections', []):
        try:
            offs = coll.get_offsets()
        except Exception:
            offs = None
        if offs is None:
            continue
        offs = np.asarray(offs, float)
        if offs.ndim != 2 or offs.shape[1] < 2 or offs.size == 0:
            continue
        ok = np.isfinite(offs[:, 0]) & np.isfinite(offs[:, 1])
        if np.any(ok):
            xs.append(offs[ok, 0])
            ys.append(offs[ok, 1])
    for ln in getattr(ax, 'lines', []):
        try:
            if ln.get_gid() == 'wt_ref_pca_box':
                continue
        except Exception:
            pass
        try:
            xdat = np.asarray(ln.get_xdata(orig=False), float)
            ydat = np.asarray(ln.get_ydata(orig=False), float)
        except Exception:
            continue
        ok = np.isfinite(xdat) & np.isfinite(ydat)
        if np.any(ok):
            xs.append(xdat[ok])
            ys.append(ydat[ok])
    if not xs or not ys:
        return None
    x = np.concatenate(xs)
    y = np.concatenate(ys)
    if x.size == 0 or y.size == 0:
        return None
    return float(np.min(x)), float(np.max(x)), float(np.min(y)), float(np.max(y))

def style_pca_axes(
    ax,
    *,
    title=None,
    xlim=None,
    ylim=None,
    explained_variance=None,
    show_variance_text=False,
    grid=False,
    legend=True,
    legend_frame=False,
    show_xlabel='auto',
    show_ylabel='auto',
    extend_limits=True,
    align_to_background=True,
):
    """Apply consistent PCA axis formatting with extend-only limits."""
    if explained_variance is None and 'PCA_RESULTS' in globals():
        explained_variance = PCA_RESULTS.get('explained_variance')

    image_extent = None
    if align_to_background and len(ax.images) > 0:
        ext = ax.images[0].get_extent()
        if ext is not None and len(ext) == 4:
            image_extent = (
                float(min(ext[0], ext[1])),
                float(max(ext[0], ext[1])),
                float(min(ext[2], ext[3])),
                float(max(ext[2], ext[3])),
            )

    ref_xlim, ref_ylim = get_reference_pca_limits(pad=0.5, fallback_x=(-7.0, 10.0), fallback_y=(-6.0, 6.0))
    if xlim is None:
        xlim = tuple(ref_xlim)
    if ylim is None:
        ylim = tuple(ref_ylim)
    if image_extent is not None:
        xlim = (float(image_extent[0]), float(image_extent[1]))
        ylim = (float(image_extent[2]), float(image_extent[3]))

    if show_xlabel == 'auto' or show_ylabel == 'auto':
        auto_x, auto_y = _outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True)
        if show_xlabel == 'auto':
            show_xlabel = auto_x
        if show_ylabel == 'auto':
            show_ylabel = auto_y

    if show_variance_text and explained_variance is not None and len(explained_variance) >= 2:
        if show_xlabel:
            ax.set_xlabel(f"PC1 ({explained_variance[0]:.1%} variance)")
        else:
            ax.set_xlabel('')
        if show_ylabel:
            ax.set_ylabel(f"PC2 ({explained_variance[1]:.1%} variance)")
        else:
            ax.set_ylabel('')
    else:
        if show_xlabel:
            ax.set_xlabel('PC1')
        else:
            ax.set_xlabel('')
        if show_ylabel:
            ax.set_ylabel('PC2')
        else:
            ax.set_ylabel('')

    if xlim is not None:
        x0, x1 = float(xlim[0]), float(xlim[1])
        x0 = min(x0, float(ref_xlim[0]))
        x1 = max(x1, float(ref_xlim[1]))
        if extend_limits:
            data_limits = _pca_artist_data_limits(ax)
            if data_limits is not None:
                x0 = min(x0, float(data_limits[0]))
                x1 = max(x1, float(data_limits[1]))
        ax.set_xlim((x0, x1))

    if ylim is not None:
        y0, y1 = float(ylim[0]), float(ylim[1])
        y0 = min(y0, float(ref_ylim[0]))
        y1 = max(y1, float(ref_ylim[1]))
        if extend_limits:
            data_limits = _pca_artist_data_limits(ax)
            if data_limits is not None:
                y0 = min(y0, float(data_limits[2]))
                y1 = max(y1, float(data_limits[3]))
        ax.set_ylim((y0, y1))

    _draw_reference_pca_box(ax, ref_xlim, ref_ylim)

    if title is not None:
        title_fs = clamp_fontsize(8.0) if 'clamp_fontsize' in globals() else None
        if title_fs is None:
            ax.set_title(title)
        else:
            ax.set_title(title, fontsize=title_fs)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine_name in ('left', 'bottom'):
        if spine_name in ax.spines:
            ax.spines[spine_name].set_linewidth(1.0)
    tick_w = 1.0
    ax.tick_params(direction='out', length=4, width=tick_w, labelsize=6)
    if not show_xlabel:
        ax.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
    if not show_ylabel:
        ax.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax.grid(False)
    if 'shrink_figure_to_pnas' in globals():
        shrink_figure_to_pnas(ax.figure, panel_kind='pca')
    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)
    if legend:
        lg = ax.legend(frameon=legend_frame)
        if lg is not None:
            lg.set_frame_on(legend_frame)

def plot_pca_value_overlay(
    ax,
    coords,
    mask,
    values,
    *,
    cmap,
    vmin,
    vmax,
    s=30,
    edgecolors='none',
    zorder=3,
):
    """Overlay PCA points colored by scalar values on top of a map."""
    if coords is None or values is None:
        return None
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0 or np.sum(mask) == 0:
        return None
    point_size = clamp_scatter_area(s) if 'clamp_scatter_area' in globals() else s
    return ax.scatter(
        coords[mask, 0],
        coords[mask, 1],
        c=np.asarray(values)[mask],
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=point_size,
        edgecolors=edgecolors,
        zorder=zorder,
    )

def make_figure(panel_kind='simple', **kwargs):
    """Unified single-figure factory using the notebook sizing rules."""
    kwargs = dict(kwargs)
    kwargs.pop('figsize', None)
    return plt.figure(panel_kind=panel_kind, **kwargs)

def make_figure_grid(nrows=1, ncols=1, panel_kind='simple', **kwargs):
    """Unified subplot factory using the notebook grid sizing rules."""
    kwargs = dict(kwargs)
    kwargs.pop('figsize', None)
    return plt.subplots(nrows, ncols, panel_kind=panel_kind, **kwargs)

def finalize_figure(fig, title=None, *, rect=None, save_path=None, dpi=300, show=True, tight_layout_kwargs=None):
    """Apply common suptitle, layout, save, and display handling."""
    if title:
        fig.suptitle(title, fontsize=clamp_fontsize(8.0), fontweight='bold')
    if rect is None and title is not None:
        rect = [0, 0, 1, 0.97]
    if 'apply_grid_size' in globals():
        apply_grid_size(fig, panel_kind=getattr(fig, '_grid_panel_kind', 'simple'))
    if 'apply_external_legend' in globals():
        apply_external_legend(fig)
    if 'sanitize_figure_text' in globals():
        sanitize_figure_text(fig)
    tl_kwargs = dict(tight_layout_kwargs or {})
    if rect is not None and 'rect' not in tl_kwargs:
        tl_kwargs['rect'] = rect
    fig.tight_layout(**tl_kwargs)
    if save_path is not None:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    return fig

def _display_cmap(cmap, bad_color='#e6e6e6'):
    """Return a colormap copy with unsupported / missing regions shown in light gray."""
    cm = plt.get_cmap(cmap).copy() if isinstance(cmap, str) else cmap.copy()
    cm.set_bad(bad_color)
    return cm

def add_scalar_colorbar(fig, axes, *, cmap, vmin, vmax, label, shrink=0.7, ticks=None):
    """Attach one shared scalar colorbar to one axis or a group of axes."""
    axes_arr = np.atleast_1d(np.asarray(axes, dtype=object)).ravel().tolist()
    sm = plt.cm.ScalarMappable(cmap=_display_cmap(cmap), norm=plt.Normalize(vmin, vmax))
    return fig.colorbar(sm, ax=axes_arr, shrink=shrink, label=label, ticks=ticks)

def render_pca_scalar_panel(
    ax,
    xy,
    arr,
    *,
    cmap,
    vmin,
    vmax,
    title,
    pad=0.5,
    point_size=14,
    min_points=3,
    empty_label='too few',
    add_colorbar=False,
    colorbar_label=None,
    colorbar_shrink=0.7,
    colorbar_ticks=None,
):
    """Render one PCA scalar map panel with consistent smoothing and fallback text."""
    xy = np.asarray(xy) if xy is not None else np.empty((0, 2))
    arr = np.asarray(arr, dtype=float)
    ok = np.isfinite(arr)
    n_ok = int(np.sum(ok))

    if xy.size:
        _, _, ext = setup_pca_grid(xy, pad=pad)
    else:
        ext = (-7, 10, -6, 6)

    if n_ok < min_points:
        ax.text(0.5, 0.5, f'{title}\n{empty_label}', ha='center', va='center',
                transform=ax.transAxes, fontsize=clamp_fontsize(5.0))
        ax.set_xlim(ext[0], ext[1])
        ax.set_ylim(ext[2], ext[3])
        style_pca_axes(ax, title=title, legend=False)
        return n_ok

    gx, gy, ext = setup_pca_grid(xy, pad=pad)
    sg, _ = smooth_field(xy, arr, ok, gx, gy)
    ax.imshow(sg, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap(cmap),
              vmin=vmin, vmax=vmax, interpolation='bilinear', alpha=0.7)
    plot_pca_value_overlay(
        ax,
        xy,
        ok,
        arr,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        s=point_size,
        edgecolors='none',
        zorder=3,
    )
    style_pca_axes(ax, title=title, legend=False)
    if add_colorbar:
        add_scalar_colorbar(ax.figure, [ax], cmap=cmap, vmin=vmin, vmax=vmax,
                            label=(colorbar_label or ''), shrink=colorbar_shrink, ticks=colorbar_ticks)
    return n_ok

def plot_mean_sem_trace(
    ax,
    x,
    values,
    *,
    color,
    label,
    marker='o',
    linestyle='-',
    ms=4,
    lw=1.2,
    fill_alpha=0.12,
    min_points=2,
):
    """Plot mean ± SEM trajectory across boutons or trials."""
    values = np.asarray(values, dtype=float)
    nv = np.sum(np.isfinite(values), axis=0)
    mn = np.nanmean(values, axis=0)
    se = np.nanstd(values, axis=0) / np.sqrt(np.clip(nv, 1, None))
    mask = (nv >= min_points) & np.isfinite(mn)
    if not np.any(mask):
        return 0
    n0 = int(np.sum(np.isfinite(values[:, 0]))) if values.ndim == 2 else int(np.sum(np.isfinite(values)))
    ax.plot(np.asarray(x)[mask], mn[mask], marker=marker, ls=linestyle, color=color, ms=ms, lw=lw,
            label=label.format(n=n0))
    ax.fill_between(np.asarray(x)[mask], mn[mask] - se[mask], mn[mask] + se[mask],
                    alpha=fill_alpha, color=color)
    return n0

# ===== High-Level Analysis API =====
def amplitude_profiles_matrix(condition_dataframe, amp_column_names=None, max_pulse_number=10):
    """Return bouton × pulse amplitude matrix from AMP columns or AMP1 + PPR columns."""
    if condition_dataframe is None or len(condition_dataframe) == 0:
        return np.empty((0, int(max_pulse_number)), dtype=float)
    if amp_column_names is None:
        amp_column_names = [
            f'AMP{pulse_num}'
            for pulse_num in range(1, int(max_pulse_number) + 1)
            if f'AMP{pulse_num}' in condition_dataframe.columns
        ]
    amp_column_names = [col for col in amp_column_names if col in condition_dataframe.columns]
    if amp_column_names:
        return condition_dataframe[amp_column_names].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
    if 'AMP1' not in condition_dataframe.columns:
        raise ValueError("Could not build amplitudes: missing AMP columns and missing 'AMP1'.")
    a1 = pd.to_numeric(condition_dataframe['AMP1'], errors='coerce').to_numpy(dtype=float)
    profiles = [a1]
    for pulse_num in range(2, int(max_pulse_number) + 1):
        ppr_col = f'PPR{pulse_num}/1'
        if ppr_col not in condition_dataframe.columns:
            break
        ppr_vals = pd.to_numeric(condition_dataframe[ppr_col], errors='coerce').to_numpy(dtype=float)
        profiles.append(a1 * ppr_vals)
    return np.column_stack(profiles) if profiles else np.empty((len(condition_dataframe), 0), dtype=float)


def build_profile_matrix(
    *,
    dataframe=None,
    matrix=None,
    profile_kind='ppr',
    ppr_column_names=None,
    amp_column_names=None,
    max_pulse_number=10,
    trials_df=None,
    threshold_col='thr_shared',
    min_trials=1,
    cumulative=False,
    normalize=None,
):
    """Build a profile matrix from a dataframe or a precomputed matrix."""
    kind = str(profile_kind).strip().lower()
    if matrix is None:
        if dataframe is None:
            raise ValueError('Provide either dataframe or matrix.')
        if kind == 'ppr':
            matrix = ppr_profiles_matrix(dataframe, ppr_column_names=ppr_column_names, max_pulse_number=max_pulse_number)
        elif kind == 'failure':
            matrix = failure_profiles_matrix(dataframe, trials_df=trials_df, threshold_col=threshold_col, max_pulse_number=max_pulse_number, min_trials=min_trials)
        elif kind == 'amplitude':
            matrix = amplitude_profiles_matrix(dataframe, amp_column_names=amp_column_names, max_pulse_number=max_pulse_number)
        else:
            raise ValueError(f'Unsupported profile_kind: {profile_kind!r}')
    arr = _coerce_profile_matrix(matrix)
    if cumulative:
        arr = np.cumsum(arr, axis=1)
    norm = None if normalize is None else str(normalize).strip().lower()
    if norm in (None, '', 'none'):
        return arr
    if norm in ('first', 'a1'):
        denom = arr[:, [0]]
        return np.divide(arr, denom, out=np.full_like(arr, np.nan), where=np.isfinite(denom) & (denom != 0))
    if norm in ('peak', 'max'):
        denom = np.nanmax(np.abs(arr), axis=1, keepdims=True)
        return np.divide(arr, denom, out=np.full_like(arr, np.nan), where=np.isfinite(denom) & (denom != 0))
    raise ValueError(f'Unsupported profile normalization: {normalize!r}')


def make_profile_group(
    label,
    *,
    dataframe=None,
    matrix=None,
    color='black',
    marker='o',
    linewidth=2.0,
    markersize=6,
    sem_alpha=0.2,
    profile_kind='ppr',
    ppr_column_names=None,
    amp_column_names=None,
    max_pulse_number=10,
    trials_df=None,
    threshold_col='thr_shared',
    min_trials=1,
    cumulative=False,
    normalize=None,
):
    """Return one high-level profile-group specification."""
    return {
        'matrix': build_profile_matrix(
            dataframe=dataframe,
            matrix=matrix,
            profile_kind=profile_kind,
            ppr_column_names=ppr_column_names,
            amp_column_names=amp_column_names,
            max_pulse_number=max_pulse_number,
            trials_df=trials_df,
            threshold_col=threshold_col,
            min_trials=min_trials,
            cumulative=cumulative,
            normalize=normalize,
        ),
        'label': str(label),
        'color': color,
        'marker': marker,
        'linewidth': linewidth,
        'markersize': markersize,
        'sem_alpha': sem_alpha,
    }


def plot_profile_analysis(
    ax,
    groups,
    *,
    pulse_numbers=None,
    show_individuals=False,
    individual_alpha=0.12,
    individual_lw=0.8,
    show_stats=False,
    paired_groups=False,
    parametric_stat=True,
    alpha=0.05,
    n_permutations=5000,
    summary_label='Profile analysis',
    title=None,
    xlabel='Stimulus',
    ylabel='Profile value',
    finalize_kwargs=None,
):
    """High-level API for single or multi profile plots."""
    prepared = []
    for i, group in enumerate(groups):
        spec = dict(group)
        if 'matrix' not in spec:
            raise KeyError(f'Profile group #{i + 1} is missing a matrix.')
        spec['matrix'] = _coerce_profile_matrix(spec['matrix'])
        prepared.append(spec)
    prepared, pulse_arr = _prepare_profile_inputs(prepared, pulse_numbers=pulse_numbers)
    if show_individuals:
        for spec in prepared:
            plot_ppr_profiles_overlay(ax, spec['matrix'], color=spec.get('color', 'gray'), label='_nolegend_', individual_alpha=individual_alpha, individual_lw=individual_lw, mean_lw=0.0, sem_alpha=0.0)
    return plot_profile_group_comparison(ax, prepared, pulse_numbers=pulse_arr, stats_start_index=0, show_stats=show_stats, paired_groups=paired_groups, parametric_stat=parametric_stat, alpha=alpha, n_permutations=n_permutations, summary_label=summary_label, title=title, xlabel=xlabel, ylabel=ylabel, finalize_kwargs=finalize_kwargs)


def make_box_group(label, *, values=None, dataframe=None, column=None, color='gray'):
    """Return one boxplot group spec from values or a dataframe column."""
    if values is None:
        if dataframe is None or column is None:
            raise ValueError('Provide either values or (dataframe, column).')
        values = dataframe[column]
    arr = pd.to_numeric(pd.Series(values), errors='coerce').dropna().to_numpy(dtype=float)
    return {'label': str(label), 'values': arr, 'color': color}


def _boxplot_stat_summary(groups, *, paired=False, parametric_stat=False):
    from scipy.stats import f_oneway, kruskal, mannwhitneyu, ttest_ind, ttest_rel, wilcoxon

    labels = [g['label'] for g in groups]
    arrays = [np.asarray(g['values'], dtype=float) for g in groups]
    arrays = [arr[np.isfinite(arr)] for arr in arrays]
    if len(arrays) < 2:
        line = 'Boxplot analysis: need at least two groups.'
        print(line)
        return {'summary_line': line, 'p_value': np.nan, 'test_name': None}
    if paired:
        if len(arrays) != 2:
            raise ValueError('paired=True currently supports exactly two groups.')
        n = min(len(arrays[0]), len(arrays[1]))
        x = np.asarray(arrays[0][:n], dtype=float)
        y = np.asarray(arrays[1][:n], dtype=float)
        valid = np.isfinite(x) & np.isfinite(y)
        x = x[valid]
        y = y[valid]
        n = len(x)
        if n == 0:
            line = f"Boxplot analysis: no valid paired data for {labels[0]} vs {labels[1]}."
            print(line)
            return {'summary_line': line, 'p_value': np.nan, 'test_name': None}
        if parametric_stat:
            stat, p = ttest_rel(x, y, nan_policy='omit')
            test_name, stat_label = 'Paired t-test', 't'
        else:
            stat, p = wilcoxon(x, y, alternative='two-sided', zero_method='wilcox')
            test_name, stat_label = 'Wilcoxon signed-rank', 'W'
        line = f"Boxplot analysis: {test_name}, {labels[0]} vs {labels[1]}, {stat_label}={stat:.3f}, p={p:.3g}, paired n={n}"
        print(line)
        return {'summary_line': line, 'p_value': p, 'test_name': test_name, 'statistic': stat}
    if len(arrays) == 2:
        if parametric_stat:
            stat, p = ttest_ind(arrays[0], arrays[1], equal_var=False, nan_policy='omit')
            test_name, stat_label = 'Welch t-test', 't'
        else:
            stat, p = mannwhitneyu(arrays[0], arrays[1], alternative='two-sided')
            test_name, stat_label = 'Mann-Whitney U', 'U'
        line = f"Boxplot analysis: {test_name}, {labels[0]} vs {labels[1]}, {stat_label}={stat:.3f}, p={p:.3g}, n={len(arrays[0])}/{len(arrays[1])}"
        print(line)
        return {'summary_line': line, 'p_value': p, 'test_name': test_name, 'statistic': stat}
    if parametric_stat:
        stat, p = f_oneway(*arrays)
        test_name, stat_label = 'One-way ANOVA', 'F'
    else:
        stat, p = kruskal(*arrays)
        test_name, stat_label = 'Kruskal-Wallis', 'H'
    n_str = ', '.join(f'{lbl} n={len(arr)}' for lbl, arr in zip(labels, arrays))
    line = f"Boxplot analysis: {test_name}, {stat_label}={stat:.3f}, p={p:.3g}; {n_str}"
    print(line)
    return {'summary_line': line, 'p_value': p, 'test_name': test_name, 'statistic': stat}


BOXPLOT_GROUP_SPACING_DEFAULT = 0.25
BOXPLOT_X_PADDING_DEFAULT = 0.10

def plot_boxplot_analysis(ax, groups, *, paired=False, show_points=True, show_stats=True, parametric_stat=False, title=None, xlabel=None, ylabel=None, point_alpha=0.5, point_size=24, group_spacing=BOXPLOT_GROUP_SPACING_DEFAULT, x_padding=BOXPLOT_X_PADDING_DEFAULT, box_width=None):
    """High-level API for single or multi boxplots with a condensed stat summary."""
    prepared = [dict(g) for g in groups]
    values = [np.asarray(g['values'], dtype=float) for g in prepared]
    spacing_value = float(group_spacing)
    positions = np.arange(len(values), dtype=float) * spacing_value
    box_width = min(0.36, max(0.10, 0.55 * spacing_value)) if box_width is None else float(box_width)
    bp = ax.boxplot(values, positions=positions, widths=box_width, patch_artist=True, showfliers=False, whis=(10, 90))
    for patch, group in zip(bp['boxes'], prepared):
        patch.set_facecolor(group.get('color', 'gray'))
        patch.set_alpha(0.55)
    if show_points:
        point_radius_x = min(0.015, 0.0025 * np.sqrt(max(float(point_size), 1.0)))
        jitter_half_width = min(0.07, max(0.015, 0.5 * box_width - 2.0 * point_radius_x))
        for pos, vals, group in zip(positions, values, prepared):
            jitter = np.ones(len(vals)) * pos + np.random.uniform(-jitter_half_width, jitter_half_width, len(vals))
            ax.scatter(jitter, vals, c=group.get('color', 'gray'), s=point_size, alpha=point_alpha, edgecolors='black', linewidths=0.0)
    display_labels = [re.sub(r'\s*\n?\s*\(n\s*=\s*\d+\)\s*$', '', str(g['label'])).strip() for g in prepared]
    ax.set_xticks(positions)
    ax.set_xticklabels(display_labels, rotation=45, ha='right')
    ax.set_xlim(positions[0] - float(x_padding), positions[-1] + float(x_padding))
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if '_style_boxplot_axis' in globals():
        _style_boxplot_axis(ax)
    else:
        _apply_clean_axes_style(ax, panel_kind='simple')
    stats_result = _boxplot_stat_summary(prepared, paired=paired, parametric_stat=parametric_stat)
    if show_stats and len(prepared) == 2 and np.isfinite(stats_result.get('p_value', np.nan)):
        y_hi = np.nanmax([np.nanmax(v) for v in values if len(v)]) if any(len(v) for v in values) else 1.0
        y_lo = np.nanmin([np.nanmin(v) for v in values if len(v)]) if any(len(v) for v in values) else 0.0
        y_span = max(1e-6, y_hi - y_lo)
        y_bar = y_hi + 0.08 * y_span
        ax.plot([positions[0], positions[1]], [y_bar, y_bar], color='black', linewidth=1.2)
        stat_text = ax.text(np.mean(positions[:2]), y_bar + 0.02 * y_span, _pvalue_to_stars(stats_result['p_value']) or 'ns', ha='center', va='bottom')
        setattr(stat_text, '_keep_in_panel_annotation', True)
        ax.set_ylim(top=y_bar + 0.08 * y_span)
    ppr_tokens = [str(title or ''), str(ylabel or '')] + [str(g.get('label', '')) for g in prepared]
    if any('PPR' in token.upper() for token in ppr_tokens):
        _ymin, _ymax = ax.get_ylim()
        ax.set_ylim(0, _ymax)
    return stats_result


def add_significance_bar(ax, x1, x2, p_value, *, y=None, h=None, color='black', linewidth=1.0, fontsize=5, y_pad_frac=0.05, h_frac=0.02, text_pad_frac=0.015, expand_ylim=True):
    y_min, y_max = ax.get_ylim()
    span = max(y_max - y_min, 1e-9)
    y_line = y_max + y_pad_frac * span if y is None else float(y)
    bar_h = h_frac * span if h is None else float(h)
    y_text = y_line + text_pad_frac * span
    ax.plot([x1, x1, x2, x2], [y_line, y_line + bar_h, y_line + bar_h, y_line], color=color, lw=linewidth)
    text_obj = ax.text((x1 + x2) / 2, y_text + bar_h, _pvalue_to_stars(p_value), ha='center', va='bottom', fontsize=fontsize)
    setattr(text_obj, '_keep_in_panel_annotation', True)
    if expand_ylim:
        ax.set_ylim(y_min, max(y_max, y_text + bar_h + 0.05 * span))
    return text_obj


def make_pca_group(coords, *, label='Data', color='black', marker='o', s=30, alpha=0.8, edgecolors='none', linewidths=0.0, zorder=3):
    """Return one PCA overlay-group spec."""
    return {'coords': np.asarray(coords, dtype=float), 'label': str(label), 'color': color, 'marker': marker, 's': s, 'alpha': alpha, 'edgecolors': edgecolors, 'linewidths': linewidths, 'zorder': zorder}


def plot_pca_analysis(ax, *, background=None, overlays=None, title=None, legend=False, legend_frame=False, xlim=None, ylim=None, explained_variance=None, show_variance_text=False):
    """High-level API for PCA panels with optional colored background and overlays."""
    bg = dict(background or {})
    overlays = [] if overlays is None else list(overlays)
    if bg:
        plot_pca_background(ax, coords=bg.get('coords', None), labels=bg.get('labels', None), use_cluster_colors=bool(bg.get('use_cluster_colors', False)), color=bg.get('color', 'gray'), alpha=bg.get('alpha', 0.3), s=bg.get('s', 30), marker=bg.get('marker', 'o'), edgecolors=bg.get('edgecolors', 'none'), linewidths=bg.get('linewidths', 0.0), label=bg.get('label', 'WT pooled'), zorder=bg.get('zorder', 1))
    for spec in overlays:
        plot_pca_overlay_points(ax, np.asarray(spec['coords'], dtype=float), c=spec.get('color', 'black'), marker=spec.get('marker', 'o'), s=spec.get('s', 30), alpha=spec.get('alpha', 0.8), edgecolors=spec.get('edgecolors', 'none'), linewidths=spec.get('linewidths', 0.0), label=spec.get('label', 'Data'), zorder=spec.get('zorder', 3))
    style_pca_axes(ax, title=title, legend=legend, legend_frame=legend_frame, xlim=xlim, ylim=ylim, explained_variance=explained_variance, show_variance_text=show_variance_text)
    return ax


def _scale_trace_matrix(matrix, mode=None):
    """Scale trace matrix row-wise for display only."""
    arr = np.asarray(matrix, dtype=float)
    if arr.ndim != 2:
        raise ValueError('matrix must be 2D.')
    mode = None if mode is None else str(mode).strip().lower()
    if mode in (None, '', 'none'):
        return arr
    if mode == 'baseline_zero':
        baseline = np.nanmean(arr[:, :max(1, min(10, arr.shape[1]))], axis=1, keepdims=True)
        return arr - baseline
    if mode in ('peak', 'max'):
        denom = np.nanmax(np.abs(arr), axis=1, keepdims=True)
        return np.divide(arr, denom, out=np.full_like(arr, np.nan), where=np.isfinite(denom) & (denom != 0))
    if mode == 'zscore':
        mu = np.nanmean(arr, axis=1, keepdims=True)
        sigma = np.nanstd(arr, axis=1, keepdims=True)
        return np.divide(arr - mu, sigma, out=np.full_like(arr, np.nan), where=np.isfinite(sigma) & (sigma != 0))
    raise ValueError(f'Unsupported trace scale mode: {mode!r}')


def plot_trace_analysis(ax=None, *, trace_ids=None, condition_names=None, rows=None, source=None, filters=None, display='mean_sem', show_individuals=False, scale_mode=None, alignment='auto', resample=1, aggregation='mean', smooth=None, tol_factor=None, color='k', individual_color=None, sem_color=None, label=None, linewidth=2.0, linestyle='-', sem_alpha=0.25, individual_alpha=0.12, individual_lw=0.6, stim_times=None, stim_kwargs=None, zero_line=False, zero_kwargs=None, hlines=None, hline_kwargs=None, event_time=None, event_kwargs=None, xlim=None, ylim=None, xlabel=None, ylabel=None, title=None, legend=False, legend_kwargs=None, style_axis=True):
    """High-level API for single, multiple, or mean ± SEM trace panels."""
    if ax is None:
        _, ax = make_figure_grid(panel_kind='trace')
    display_mode = str(display).strip().lower()
    stats = compute_trace_stats(trace_ids=trace_ids, condition_names=condition_names, rows=rows, source=source, filters=filters, alignment=alignment, resample=resample, aggregation=aggregation, smooth=smooth, tol_factor=tol_factor, return_matrix=True)
    matrix = _scale_trace_matrix(stats['matrix'], mode=scale_mode)
    average = _aggregate_trace_matrix(matrix, aggregation=aggregation)
    n_eff = np.sum(np.isfinite(matrix), axis=0)
    sem = np.nanstd(matrix, axis=0, ddof=1) / np.sqrt(np.maximum(n_eff, 1))
    ind_color = individual_color if individual_color is not None else color
    fill_color = sem_color if sem_color is not None else color
    if display_mode in ('multiple', 'all', 'individuals'):
        for trace in matrix:
            ax.plot(stats['time'], trace, color=ind_color, alpha=individual_alpha, linewidth=clamp_linewidth(individual_lw))
    elif display_mode == 'single':
        ax.plot(stats['time'], matrix[0], color=color, linewidth=clamp_linewidth(linewidth), linestyle=linestyle, label=label)
    else:
        if show_individuals:
            for trace in matrix:
                ax.plot(stats['time'], trace, color=ind_color, alpha=individual_alpha, linewidth=clamp_linewidth(individual_lw))
        if display_mode in ('mean_sem', 'mean+sem', 'mean_with_sem') and sem is not None:
            ax.fill_between(stats['time'], average - sem, average + sem, color=fill_color, alpha=sem_alpha)
        ax.plot(stats['time'], average, color=color, linewidth=clamp_linewidth(linewidth), linestyle=linestyle, label=label)
    if xlim is not None:
        ax.set_xlim(*xlim)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if xlabel is not None:
        ax.set_xlabel(xlabel)
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    if stim_times is not None:
        add_stimulus_ticks(ax, stim_times, **(stim_kwargs or {}))
    if zero_line:
        zero_defaults = {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1}
        zero_defaults.update(zero_kwargs or {})
        ax.axhline(0, **zero_defaults)
    if hlines is not None:
        ref_defaults = {'color': 'black', 'linestyle': ':', 'alpha': 0.5, 'linewidth': 1}
        ref_defaults.update(hline_kwargs or {})
        for y_value in hlines:
            ax.axhline(y_value, **ref_defaults)
    if event_time is not None:
        event_defaults = {'color': 'gray', 'linestyle': '--', 'alpha': 0.5, 'linewidth': 1}
        event_defaults.update(event_kwargs or {})
        ax.axvline(event_time, **event_defaults)
    if style_axis:
        style_trace_axis(ax)
    if legend:
        add_legend(ax, frameon=False, **(legend_kwargs or {}))
    stats.update({'ax': ax, 'matrix': matrix, 'average': average, 'sem': sem})
    return stats

def _data_axes_for_layout(fig):
    axes = []
    for ax in getattr(fig, 'axes', []):
        if str(getattr(ax, 'get_label', lambda: '')()) == '<colorbar>':
            continue
        axes.append(ax)
    return axes

def _legend_entries(ax):
    legend = ax.get_legend()
    if legend is not None:
        handles = getattr(legend, 'legend_handles', None)
        if handles is None:
            handles = getattr(legend, 'legendHandles', None)
        labels = [txt.get_text() for txt in legend.get_texts()]
        if handles is not None and len(labels) == len(handles):
            pairs = [(h, l) for h, l in zip(handles, labels) if l and not str(l).startswith('_')]
            if pairs:
                return [h for h, _ in pairs], [l for _, l in pairs]
    handles, labels = ax.get_legend_handles_labels()
    pairs = [(h, l) for h, l in zip(handles, labels) if l and not str(l).startswith('_')]
    return [h for h, _ in pairs], [l for _, l in pairs]

def _dedupe_legend_entries(handles, labels):
    seen = set()
    out_h, out_l = [], []
    for h, l in zip(handles, labels):
        key = str(l)
        if key in seen:
            continue
        seen.add(key)
        out_h.append(h)
        out_l.append(l)
    return out_h, out_l

MOJIBAKE_REPLACEMENTS = {
    'Ca²⁺': 'Ca2+',
    'Ca²⁺': 'Ca2+',
    '⁺': '+',
    'A₁': 'A1',
    'A₁': 'A1',
    '₀': '0',
    '₁': '1',
    'ΔF/F₀': 'ΔF/F0',
    'ΔF/F₀': 'ΔF/F0',
    '╝': '',
}

def _replace_mojibake_text(text):
    s = '' if text is None else str(text)
    for bad, good in MOJIBAKE_REPLACEMENTS.items():
        s = s.replace(bad, good)
    return s

def _compact_legend_label(text):
    s = _replace_mojibake_text(text)
    s = re.sub(r'\bCluster\s+(\d+)\b', r'C\1', s, flags=re.I)
    return s

def _strip_result_numbers(text):
    s = _replace_mojibake_text(text)
    if not s:
        return s
    s = re.sub(r'\n(?:paired\s+)?(?:Wilcoxon|Mann-Whitney|Welch|Bootstrap|95% CI|CI|p\s*=|r\s*=|n\s*=|Mean\b|Median\b).*$', '', s, flags=re.I)
    s = re.sub(r'\s*\((?=[^)]*(?:n\s*=|σ\s*=|p\s*=|r\s*=|mean|median|RRP|P0|refill|κ|U\s*=|t\s*=|CI)).*?\)\s*$', '', s, flags=re.I)
    s = re.sub(r'\n\((?=[^)]*(?:n\s*=|σ\s*=|p\s*=|r\s*=|mean|median|RRP|P0|refill|κ|U\s*=|t\s*=|CI)).*?\)\s*$', '', s, flags=re.I)
    s = re.sub(r'\s*\(n\s*=\s*[^)]*\)', '', s, flags=re.I)
    s = re.sub(r'(avg Q|mean|med|median|RRP|P0|refill|κ|U|t|p|r)\s*=\s*[^,;\n]+', r'\1', s, flags=re.I)
    s = re.sub(r'(95% CI|CI)\s*:\s*\[[^\]]+\]', r'\1', s, flags=re.I)
    s = re.sub(r'(Mean|Median)\s*:\s*[-+]?\d*\.?\d+%?', r'\1', s)
    s = re.sub(r'\s{2,}', ' ', s).strip(' ,;')
    return s

def _is_result_annotation_text(text):
    s = _replace_mojibake_text(text).strip()
    if not s:
        return False
    if re.search(r'too few|missing|no data|available', s, flags=re.I):
        return False
    if re.fullmatch(r'[+\-]?\d+(?:\.\d+)?%?', s):
        return True
    if re.search(r'\d+(?:\.\d+)?%', s):
        return True
    if re.fullmatch(r'\*+|ns', s, flags=re.I):
        return True
    patterns = [
        r'\bp\s*=', r'\br\s*=', r'\bn\s*=', r'RRP\s*=', r'P0\s*=', r'refill', r'Stable \(', r'Unstable \(',
        r'mean\s*=', r'median\s*=', r'κ\s*=', r'\bU\s*=', r'\bt\s*=', r'CI', r'Peak response'
    ]
    return any(re.search(pat, s, flags=re.I) for pat in patterns)

def sanitize_figure_text(fig):
    """Normalize text encoding and remove result annotations from figure interiors."""
    if fig is None:
        return None
    removed_notes = []
    if getattr(fig, '_suptitle', None) is not None:
        fig._suptitle.set_text(_strip_result_numbers(fig._suptitle.get_text()))
    for ax in getattr(fig, 'axes', []):
        ax.set_title(_strip_result_numbers(ax.get_title()))
        ax.set_xlabel(_replace_mojibake_text(ax.get_xlabel()))
        ax.set_ylabel(_replace_mojibake_text(ax.get_ylabel()))
        for txt in list(getattr(ax, 'texts', [])):
            if getattr(txt, '_keep_in_panel_annotation', False):
                txt.set_text(_replace_mojibake_text(txt.get_text()))
            elif _is_result_annotation_text(txt.get_text()):
                removed_notes.append(_replace_mojibake_text(txt.get_text()))
                try:
                    txt.remove()
                except Exception:
                    pass
            else:
                txt.set_text(_replace_mojibake_text(txt.get_text()))
        lg = ax.get_legend()
        if lg is not None:
            for txt in lg.get_texts():
                txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
            if lg.get_title() is not None:
                lg.get_title().set_text(_replace_mojibake_text(lg.get_title().get_text()))
    for lg in list(getattr(fig, 'legends', [])):
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
        if lg.get_title() is not None:
            lg.get_title().set_text(_replace_mojibake_text(lg.get_title().get_text()))
    if removed_notes and not getattr(fig, '_sanitized_result_notes_printed', False):
        fig._sanitized_result_notes_printed = True
    return fig

def apply_external_legend(fig, *, max_in_axes_items=3, min_repeated_panels=2):
    """Move dense or repeated legends outside the panel grid, with optional per-axis override."""
    if fig is None:
        return None
    legend_specs = []
    for ax in _data_axes_for_layout(fig):
        handles, labels = _legend_entries(ax)
        if labels:
            labels = [_compact_legend_label(_strip_result_numbers(lbl)) for lbl in labels]
            legend_specs.append({
                'ax': ax,
                'handles': handles,
                'labels': labels,
                'force_outside': bool(getattr(ax, '_legend_force_outside', False)),
                'force_inside': bool(getattr(ax, '_legend_force_inside', False)),
            })
    if not legend_specs:
        return None

    candidate_specs = [spec for spec in legend_specs if not spec['force_inside']]
    if not candidate_specs:
        return None
    has_forced_outside = any(spec['force_outside'] for spec in candidate_specs)

    repeated_counts = {}
    for spec in candidate_specs:
        sig = tuple(spec['labels'])
        repeated_counts[sig] = repeated_counts.get(sig, 0) + 1
    has_repeated = any(len(sig) > 0 and count >= min_repeated_panels for sig, count in repeated_counts.items())
    data_axes = _data_axes_for_layout(fig)
    has_large = any(len(spec['labels']) > max_in_axes_items for spec in candidate_specs)
    if not has_forced_outside and not has_repeated and not has_large:
        return None

    all_handles, all_labels = [], []
    for spec in candidate_specs:
        all_handles.extend(spec['handles'])
        all_labels.extend(spec['labels'])
    all_handles, all_labels = _dedupe_legend_entries(all_handles, all_labels)
    if not all_labels:
        return None

    for spec in candidate_specs:
        lg_ax = spec['ax'].get_legend()
        if lg_ax is not None:
            lg_ax.remove()
    for lg in list(getattr(fig, 'legends', [])):
        try:
            lg.remove()
        except Exception:
            pass
    if data_axes:
        if 'grid_figsize' in globals() and '_figure_grid_shape' in globals():
            nrows, ncols = _figure_grid_shape(fig)
            w, h = grid_figsize(
                nrows=nrows,
                ncols=ncols,
                panel_kind=getattr(fig, '_grid_panel_kind', 'simple'),
                has_legend=True,
                has_suptitle=getattr(fig, '_suptitle', None) is not None,
            )
            fig.set_size_inches(w, h, forward=True)
        rect = _display_layout_rect(fig, has_external_legend=True) if '_display_layout_rect' in globals() else [0.03, 0.03, 0.84, 0.96]
        x_anchor = min(0.985, rect[2] + 0.008)
        y_anchor = 0.5 * (min(float(ax.get_position().y0) for ax in data_axes) + max(float(ax.get_position().y1) for ax in data_axes))
    else:
        x_anchor, y_anchor = 0.985, 0.5
    fig._external_legend_labels = tuple(all_labels)
    lg = fig.legend(
        all_handles,
        all_labels,
        loc='center left',
        bbox_to_anchor=(x_anchor, y_anchor),
        bbox_transform=fig.transFigure,
        borderaxespad=0.0,
        frameon=False,
        fontsize=clamp_fontsize(6.0),
    )
    try:
        lg.set_in_layout(True)
    except Exception:
        pass
    try:
        lg.set_clip_on(False)
    except Exception:
        pass
    return lg

def add_legend(ax, *args, **kw):
    kw = dict(kw)
    outside = kw.pop('outside', None)
    kw.setdefault('frameon', False)
    kw.setdefault('fontsize', clamp_fontsize(6.0))
    if outside is True:
        setattr(ax, '_legend_force_outside', True)
        setattr(ax, '_legend_force_inside', False)
    elif outside is False:
        setattr(ax, '_legend_force_outside', False)
        setattr(ax, '_legend_force_inside', True)
    lg = ax.legend(*args, **kw)
    if lg is not None:
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
    if outside is True and 'apply_external_legend' in globals():
        try:
            apply_external_legend(ax.figure, max_in_axes_items=0, min_repeated_panels=1)
        except Exception:
            pass
    return lg


#### Assemble the bouton feature matrix

Bouton-level scalar descriptors are gathered here from the processed feature sheets. These variables form the basis of the WT reference PCA, clustering, and the direct genotype or perturbation comparisons shown later in the notebook.


In [ ]:
# === Helper — Load bouton feature matrix ===

# Load the bouton-level feature tables from the exported multi-sheet summary workbook.
pca_features_file = BASE_DIR / PPR_FILENAME
excel_data        = pd.ExcelFile(pca_features_file)

# Only process conditions that have corresponding feature sheets
available_conditions = [cond for cond in experimental_conditions if cond in excel_data.sheet_names]
CONDITIONS           = available_conditions

feature_dataframes = []
for condition_name in CONDITIONS:
    condition_features = pd.read_excel(pca_features_file, sheet_name=condition_name)
    
    # Standardize ID column name (handle various naming conventions)
    id_column_names = ['id', 'bouton', 'bouton_id', 'name']
    for column in condition_features.columns:
        if str(column).strip().lower() in id_column_names:
            condition_features = condition_features.rename(columns={column: 'ID'})
            break
    
    # Clean bouton IDs and add condition label
    condition_features['ID'] = condition_features['ID'].apply(
        lambda x: clean_bouton_id(str(x)) if pd.notnull(x) else x
    )
    condition_features['Condition'] = condition_name
    feature_dataframes.append(condition_features)

excel_data.close()
FEATURES_DATAFRAME = pd.concat(feature_dataframes, ignore_index=True)
if globals().get('PPR_NOISE_PROTECTION', False):
    FEATURES_DATAFRAME = apply_protected_ppr_inplace(FEATURES_DATAFRAME)

# Identify and drop boutons with missing critical amplitude/PPR values
def _is_missing_critical(value):
    if isinstance(value, str):
        return value.strip() == ''
    return pd.isna(value)


def _is_critical_column(column_name: str) -> bool:
    column_name = str(column_name)
    if column_name.startswith('AMP') and column_name[3:].isdigit():
        return True
    if column_name.startswith('PPR') and '/1' in column_name:
        numerator = column_name[3:].split('/')[0]
        return numerator.isdigit()
    return False


critical_feature_columns = [
    col for col in FEATURES_DATAFRAME.columns if _is_critical_column(col)
]
invalid_feature_indices = []
invalid_feature_ids = set()

for row_idx, feature_row in FEATURES_DATAFRAME.iterrows():
    missing_columns = [
        col for col in critical_feature_columns
        if _is_missing_critical(feature_row.get(col))
    ]
    if missing_columns:
        raw_id = feature_row.get("ID")
        bouton_id = (
            clean_bouton_id(str(raw_id))
            if pd.notnull(raw_id) else f"row_{row_idx}"
        )
        condition_label = feature_row.get("Condition", "Unknown")
        print(
            f"! Skipping ID {bouton_id} (Condition {condition_label}) "
            f"due to missing values in {', '.join(missing_columns)}"
        )
        invalid_feature_indices.append(row_idx)
        if pd.notnull(raw_id):
            invalid_feature_ids.add(bouton_id)

if invalid_feature_indices:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(
        index=invalid_feature_indices
    ).reset_index(drop=True)

    if invalid_feature_ids and "RAW_TRACES_DF" in globals():
        before_trace_count = len(RAW_TRACES_DF)
        RAW_TRACES_DF = (
            RAW_TRACES_DF[~RAW_TRACES_DF['ID'].isin(invalid_feature_ids)]
            .reset_index(drop=True)
        )
        dropped_traces = before_trace_count - len(RAW_TRACES_DF)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")

        if "raw_traces_data" in globals():
            kept_ids = set(RAW_TRACES_DF['ID'])
            raw_traces_data = [
                entry for entry in raw_traces_data
                if entry['ID'] in kept_ids
            ]

    elif invalid_feature_ids and "raw_traces_data" in globals():
        before_trace_count = len(raw_traces_data)
        raw_traces_data = [
            entry for entry in raw_traces_data
            if entry['ID'] not in invalid_feature_ids
        ]
        dropped_traces = before_trace_count - len(raw_traces_data)
        if dropped_traces:
            print(f"! Ignoring {dropped_traces} raw trace files linked to invalid IDs")
        RAW_TRACES_DF = pd.DataFrame(raw_traces_data)



# Remove specified amplitude columns from analysis
amplitude_columns_to_drop = [f'AMP{i}' for i in range(3, 11) if f'AMP{i}' in FEATURES_DATAFRAME.columns]
if amplitude_columns_to_drop:
    FEATURES_DATAFRAME = FEATURES_DATAFRAME.drop(columns=amplitude_columns_to_drop)


In [ ]:
# === Helper — Manual 50 Hz corrections ===

# Apply curated corrections to problematic 50 Hz recordings before pooled analyses are rebuilt.
manual_fix_file = BASE_DIR / 'Manual_50Hz_fix_last_event.xlsx'

if manual_fix_file.exists():
    print(f"Loading manual 50Hz fixes from {manual_fix_file.name}...")
    
    # Load the manual fix file
    manual_fixes = pd.read_excel(manual_fix_file)
    
    # Column E contains IDs (0-indexed: column index 4), Column D contains corrected AMP10 (index 3)
    # Assuming column E is the 5th column (index 4) and column D is the 4th column (index 3)
    fixes_df = manual_fixes.iloc[:, [4, 3]].copy()
    fixes_df.columns = ['ID', 'AMP10_corrected']
    
    # Remove any NaN rows
    fixes_df = fixes_df.dropna()
    
    # Convert IDs to string and strip whitespace for matching
    fixes_df['ID'] = fixes_df['ID'].astype(str).str.strip()
    
    # Track updates
    updated_count = 0
    
    # Apply corrections to FEATURES_DATAFRAME
    for idx, fix_row in fixes_df.iterrows():
        fix_id = fix_row['ID']
        new_amp10 = fix_row['AMP10_corrected']
        
        # Find matching rows in FEATURES_DATAFRAME
        mask = FEATURES_DATAFRAME['ID'] == fix_id
        
        if mask.any():
            # Update AMP10
            FEATURES_DATAFRAME.loc[mask, 'AMP10'] = new_amp10
            
            # Recalculate PPR10/1 = AMP10 / AMP1
            amp1_value = FEATURES_DATAFRAME.loc[mask, 'AMP1'].values[0]
            if amp1_value != 0:
                new_ppr10_1 = new_amp10 / amp1_value
                FEATURES_DATAFRAME.loc[mask, 'PPR10/1'] = new_ppr10_1
                
                updated_count += 1
                print(f"  ✓ Updated ID {fix_id}: AMP10={new_amp10:.4f}, PPR10/1={new_ppr10_1:.4f}")
            else:
                print(f"  ⚠ Warning: ID {fix_id} has AMP1=0, cannot calculate PPR10/1")
        else:
            print(f"  ✗ Warning: ID {fix_id} not found in FEATURES_DATAFRAME")
    
    if globals().get('PPR_NOISE_PROTECTION', False):
        FEATURES_DATAFRAME = apply_protected_ppr_inplace(FEATURES_DATAFRAME)
    print(f"\nManual corrections applied: {updated_count}/{len(fixes_df)} recordings updated")
    if 'FEATURES_DATAFRAME_RAW' in globals():
        for idx, fix_row in fixes_df.iterrows():
            fix_id = str(fix_row['ID']).strip()
            new_amp10 = fix_row['AMP10_corrected']
            mask_raw = FEATURES_DATAFRAME_RAW['ID'].astype(str).str.strip() == fix_id
            if mask_raw.any():
                FEATURES_DATAFRAME_RAW.loc[mask_raw, 'AMP10'] = new_amp10
                amp1_raw = FEATURES_DATAFRAME_RAW.loc[mask_raw, 'AMP1'].values[0]
                if amp1_raw != 0:
                    FEATURES_DATAFRAME_RAW.loc[mask_raw, 'PPR10/1'] = new_amp10 / amp1_raw
    if 'build_condition_views' in globals():
        FEATURES_VIEWS = build_condition_views(FEATURES_DATAFRAME)
        PCA_Data_WT_Pooled = FEATURES_VIEWS.get('WT_pooled', pd.DataFrame()).copy()
        PCA_Data_WT_Theo = FEATURES_VIEWS.get('WT_Theo', pd.DataFrame()).copy()
        PCA_Data_WT_Anthime = FEATURES_VIEWS.get('WT_Anthime', pd.DataFrame()).copy()
        PCA_Data_SynII = filter_df_by_conditions(FEATURES_DATAFRAME, get_synapsin_conditions())
        PCA_Data_WT_Low_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '20Hz')
        PCA_Data_WT_High_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '20Hz')
        PCA_Data_Stability_Before = _exclude_ids(FEATURES_VIEWS.get('stability_before', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY) if '_exclude_ids' in globals() else FEATURES_VIEWS.get('stability_before', pd.DataFrame()).copy()
        PCA_Data_Stability_After = _exclude_ids(FEATURES_VIEWS.get('stability_after', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY) if '_exclude_ids' in globals() else FEATURES_VIEWS.get('stability_after', pd.DataFrame()).copy()
        PCA_Data_50Hz_1_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '50Hz')
        PCA_Data_50Hz_4_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '50Hz')
        PCA_Data_50Hz_2_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '2.5mM', '50Hz')
else:
    print(f"No manual fix file found at {manual_fix_file.name}, skipping manual corrections")


#### Target identity annotation

Target labels derived from the L7-tdTomato experiments are merged here so that WT bouton classes can later be tested against postsynaptic identity without redefining the WT state space. The target labels are therefore treated as an overlay on the WT reference rather than as primary clustering features.


In [ ]:
# === Helper — Target identity annotation ===

# Load and merge the PC/IN/UN target-identity map onto the bouton feature dataframe.
target_mapping_file          = BASE_DIR / TARGET_MAP_FILENAME
target_identity_data         = pd.read_excel(target_mapping_file).iloc[:, :2].copy()
target_identity_data.columns = ['ID', 'Target']

# Clean and standardize target data
target_identity_data['ID'] = target_identity_data['ID'].map(lambda x: _normalize_bouton_id(str(x).strip()))
target_identity_data['Target'] = target_identity_data['Target'].astype(str).str.strip().str.upper()

# Set invalid targets to 'UN' (undefined)
valid_targets = ['PC', 'IN', 'UN']
target_identity_data['Target'] = target_identity_data['Target'].where(
    target_identity_data['Target'].isin(valid_targets), 'UN'
)

# Merge target identities into feature data
FEATURES_DATAFRAME['ID'] = FEATURES_DATAFRAME['ID'].astype(str).str.strip()
FEATURES_DATAFRAME = FEATURES_DATAFRAME.merge(target_identity_data, on='ID', how='left')
FEATURES_DATAFRAME['Target'] = FEATURES_DATAFRAME['Target'].fillna('UN')  # Missing targets → undefined

# Enrich features and raw traces with reusable metadata columns
FEATURES_DATAFRAME = enrich_features_dataframe(FEATURES_DATAFRAME)
RAW_TRACES_DF = enrich_traces_dataframe(RAW_TRACES_DF, time_already_aligned=False)

# Attach canonical eventwise trial-derived failure rates without altering %Fail used for PCA
FEATURES_DATAFRAME, TEMP_FAILRATE_RECOMP_DF = attach_event_failure_rates_to_features(
    FEATURES_DATAFRAME,
    max_pulse_number=10,
)

# Keep explicit raw snapshots for later reuse
FEATURES_DATAFRAME_RAW = FEATURES_DATAFRAME.copy()
RAW_TRACES_DF_RAW = RAW_TRACES_DF.copy()

# Cached indexes for fast lookup (avoid repeated per-row filtering)
FEATURES_BY_COND_ID = FEATURES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
FEATURES_BY_ID = FEATURES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
FEATURES_VIEWS = build_condition_views(FEATURES_DATAFRAME)
PCA_Data_WT_Pooled = FEATURES_VIEWS.get('WT_pooled', pd.DataFrame()).copy()
PCA_Data_WT_Theo = FEATURES_VIEWS.get('WT_Theo', pd.DataFrame()).copy()
PCA_Data_WT_Anthime = FEATURES_VIEWS.get('WT_Anthime', pd.DataFrame()).copy()
PCA_Data_SynII = filter_df_by_conditions(FEATURES_DATAFRAME, get_synapsin_conditions())
PCA_Data_WT_Low_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '20Hz')
PCA_Data_WT_High_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '20Hz')
PCA_Data_50Hz_1_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '50Hz')
PCA_Data_50Hz_4_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '50Hz')
PCA_Data_50Hz_2_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '2.5mM', '50Hz')

# Data loading summary
print("\n=== DATA LOADING SUMMARY ===")
print(f"Conditions: {len(CONDITIONS)} ({', '.join(CONDITIONS)})")
print(f"Traces: {len(RAW_TRACES_DF)} | Features: {len(FEATURES_DATAFRAME)} | Target mappings: {len(target_identity_data)}")
target_counts = FEATURES_DATAFRAME['Target'].value_counts().sort_index()
print('Target counts: ' + '  '.join(f"{k}={int(v)}" for k, v in target_counts.items()))
print(f"✓ Successfully processed {len(raw_traces_data)} bouton files")


#### Organize traces by experimental condition

Trace tables are reorganized by condition so that mean responses, SEM envelopes, and single-fiber examples can be generated consistently. This step also harmonizes the time base across recordings, which is critical when comparing train responses across calcium levels, frequencies, or genotypes.


In [ ]:
# === Helper — Organize traces by condition ===

# Reorganize trace tables by condition and align them to the shared analysis time base when needed.
if len(raw_traces_data) > 0:
    # Use the time vector from the first trace as reference
    first_time = np.array(raw_traces_data[0]['Time'])
    if len(first_time) > 0:
        COMMON_TIME = first_time
        N_SAMPLES = len(COMMON_TIME)
        CROP_END = float(COMMON_TIME[-1])
        print(f"Updated COMMON_TIME from traces: {N_SAMPLES} samples, {CROP_END:.2f}s duration")

# Per-condition ISI mapping (for visualization/annotation)
CONDITION_ISI = {
    c: 0.02 for c in (
        get_calcium_conditions('1.5mM', '50Hz')
        + get_calcium_conditions('2.5mM', '50Hz')
        + get_calcium_conditions('4mM', '50Hz')
    )
}
DEFAULT_ISI = 0.05
DEFAULT_TRAIN_START = 1.0

# Baseline offset for exceptional conditions (0.5s baseline instead of 1s)
EXCEPTIONAL_BASELINE_OFFSET = 0.5

# Build traces dictionary organized by condition
traces_by_condition = {}
all_processed_traces = []

for _, row in RAW_TRACES_DF.iterrows():
    condition = row['Condition']
    avg_trace = np.array(row['Avg'])
    time_vec = np.array(row['Time'])
    
    # Check if this is an exceptional condition with short baseline
    is_exceptional = condition in EXCEPTIONAL_CONDITIONS
    
    if is_exceptional:
        # For exceptional conditions: shift time by 0.5s and prepend NaN values
        # Original traces start at 0s with 0.5s baseline, we shift them to start at 0.5s
        shifted_time = time_vec + EXCEPTIONAL_BASELINE_OFFSET
        
        # Create NaN-padded trace aligned to COMMON_TIME
        # Points before 0.5s get NaN, points after get interpolated from shifted trace
        padded_trace = np.full_like(COMMON_TIME, np.nan)
        
        # Find indices in COMMON_TIME that are >= 0.5s (where we have actual data)
        valid_mask = COMMON_TIME >= EXCEPTIONAL_BASELINE_OFFSET
        
        # Interpolate the original trace values onto the valid portion of COMMON_TIME
        valid_times = COMMON_TIME[valid_mask]
        padded_trace[valid_mask] = np.interp(valid_times, shifted_time, avg_trace)
        
        avg_trace = padded_trace
    else:
        # Standard case: resample to COMMON_TIME if needed
        if len(time_vec) != len(COMMON_TIME) or not np.allclose(time_vec, COMMON_TIME):
            avg_trace = np.interp(COMMON_TIME, time_vec, avg_trace)
    
    processed = {
        'ID': row['ID'],
        'Condition': condition,
        'Time': COMMON_TIME,
        'Avg': avg_trace,
    }
    all_processed_traces.append(processed)
    
    # Clean ID by removing '_traces_converted' suffix if present
    if processed['ID'].endswith('_traces_converted'):
        processed['ID'] = processed['ID'][:-len('_traces_converted')]

    if condition not in traces_by_condition:
        traces_by_condition[condition] = []
    traces_by_condition[condition].append(avg_trace)

print(f"✓ Organized {len(all_processed_traces)} traces across {len(traces_by_condition)} conditions")
exceptional_count = sum(1 for t in all_processed_traces if t['Condition'] in EXCEPTIONAL_CONDITIONS)
if exceptional_count > 0:
    print(f"  → {exceptional_count} traces from exceptional conditions (0.5s baseline, NaN-padded 0-0.5s)")


#### Build the normalized trace table

The normalized trace table is the common time-domain representation used for the trace figures. It links each bouton ID to a standardized trace, enabling later cluster-wise, fiber-wise, and condition-wise averaging without rebuilding the trace structure each time.


In [ ]:
# === Helper — Build normalized trace table ===

# Assemble the normalized trace dataframe used for shared trace plotting across conditions and figures.
for condition_name, condition_traces in traces_by_condition.items():
    print(f"Condition {condition_name}: {len(condition_traces)} traces")

# Create final processed traces dataframe
NORM_TRACES_DATAFRAME = pd.DataFrame(all_processed_traces)
NORM_TRACES_DATAFRAME = enrich_traces_dataframe(NORM_TRACES_DATAFRAME, time_already_aligned=True)

# Cached trace indexes/lookups to avoid repeated recomputation
TRACE_BY_COND_ID = NORM_TRACES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
TRACE_BY_ID = NORM_TRACES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
lookup_source = str(TRACE_SINGLE_SOURCE).strip().lower()
lookup_df = NORM_TRACES_DATAFRAME if lookup_source == 'normalized' else RAW_TRACES_DF
resampled_trace_lookup = {}
for _, trace_row in lookup_df.iterrows():
    time_values = _trace_row_time(trace_row, aligned=True)
    resampled_trace_lookup[str(trace_row['ID'])] = {'Time': time_values, 'Avg': trace_row['Avg']}

print(f"\n✓ Created NORM_TRACES_DATAFRAME with {len(NORM_TRACES_DATAFRAME)} traces")


#### Exclude the outlier 50 Hz recording

The outlier exclusion is kept explicit because the 50 Hz analyses are sensitive to rare recordings with atypical final events or partial saturation. Keeping this decision visible in the notebook makes the downstream high-frequency comparisons easier to interpret.


In [ ]:
# === Helper — Rebuild pooled views and derived datasets ===
# Refresh pooled condition tables, derived subsets, and cached dataframe views after upstream data updates.
import numpy as np
import matplotlib.pyplot as plt
cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
cond_2_5_50hz = get_calcium_conditions('2.5mM', '50Hz')[0]

# Filter traces, times, and IDs for each condition
traces_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['Avg']
times_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['Time']
trace_ids_4_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_4_50hz]['ID']

traces_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['Avg']
times_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['Time']
trace_ids_2_5_50Hz = NORM_TRACES_DATAFRAME[NORM_TRACES_DATAFRAME['Condition'] == cond_2_5_50hz]['ID']

# Function to get time corresponding to the maximum
def get_max_and_time(traces, times):
    max_values = []
    max_times = []
    for trace, time in zip(traces, times):
        max_val = np.nanmax(trace)
        max_idx = np.nanargmax(trace)
        max_values.append(max_val)
        max_times.append(time[max_idx])
    return max_values, max_times

# Compute maxima and corresponding times
max_values_4_50Hz, max_times_4_50Hz = get_max_and_time(traces_4_50Hz, times_4_50Hz)
max_values_2_5_50Hz, max_times_2_5_50Hz = get_max_and_time(traces_2_5_50Hz, times_2_5_50Hz)

# Define threshold for outlier exclusion
threshold = 5

def _trace_peak_exclusions_by_condition(trace_df, threshold_value, condition_names=None):
    """Return per-condition peak-based exclusions using the current normalized trace table."""
    if trace_df is None or len(trace_df) == 0:
        return {}, pd.DataFrame(columns=['Condition', 'ID', 'PeakValue'])
    if condition_names is not None:
        allowed = {str(name).strip() for name in condition_names}
        trace_df = trace_df[trace_df['Condition'].astype(str).str.strip().isin(allowed)].copy()
        if len(trace_df) == 0:
            return {}, pd.DataFrame(columns=['Condition', 'ID', 'PeakValue'])

    records = []
    for _, row in trace_df.iterrows():
        avg_vals = np.asarray(row.get('Avg', []), dtype=float)
        if avg_vals.size == 0 or not np.isfinite(avg_vals).any():
            peak_val = np.nan
        else:
            peak_val = float(np.nanmax(avg_vals))
        records.append({
            'Condition': row.get('Condition', ''),
            'ID': row.get('ID', ''),
            'PeakValue': peak_val,
        })

    peak_df = pd.DataFrame(records)
    hit_mask = np.isfinite(peak_df['PeakValue']) & (peak_df['PeakValue'] > float(threshold_value))
    peak_hits = peak_df.loc[hit_mask].copy()
    peak_hits['Condition'] = peak_hits['Condition'].astype(str).str.strip()
    peak_hits['ID'] = peak_hits['ID'].astype(str).str.strip()

    grouped = {
        cond: grp.sort_values(['PeakValue', 'ID'], ascending=[False, True])['ID'].tolist()
        for cond, grp in peak_hits.groupby('Condition', sort=True)
    }
    return grouped, peak_hits.sort_values(['Condition', 'PeakValue', 'ID'], ascending=[True, False, True]).reset_index(drop=True)


THRESHOLD_FILTER_CONDITIONS = (
    get_calcium_conditions('1.5mM', '50Hz')
    + get_calcium_conditions('2.5mM', '50Hz')
    + get_calcium_conditions('4mM', '50Hz')
)
OUTLIER_IDS_BY_CONDITION, OUTLIER_TRACE_SUMMARY = _trace_peak_exclusions_by_condition(
    NORM_TRACES_DATAFRAME,
    threshold,
    condition_names=THRESHOLD_FILTER_CONDITIONS,
)
OUTLIER_IDS_50HZ = sorted({bouton_id for ids in OUTLIER_IDS_BY_CONDITION.values() for bouton_id in ids})
OUTLIER_IDS_STABILITY = list(globals().get('STABILITY_OUTLIER_IDS', []))

# Keep explicit pre-filter snapshots before exclusion
NORM_TRACES_DATAFRAME_PRE_50HZ_FILTER = NORM_TRACES_DATAFRAME.copy()
FEATURES_DATAFRAME_PRE_50HZ_FILTER = FEATURES_DATAFRAME.copy()
if 'FEATURES_DATAFRAME_RAW' in globals():
    FEATURES_DATAFRAME_RAW_PRE_50HZ_FILTER = FEATURES_DATAFRAME_RAW.copy()
if 'RAW_TRACES_DF' in globals():
    RAW_TRACES_DF_PRE_50HZ_FILTER = RAW_TRACES_DF.copy()
if 'RAW_TRACES_DF_RAW' in globals():
    RAW_TRACES_DF_RAW_PRE_50HZ_FILTER = RAW_TRACES_DF_RAW.copy()


def _exclude_ids(df, excluded_ids):
    if df is None or len(excluded_ids) == 0:
        return df
    if 'ID' not in df.columns:
        return df
    excluded_norm = {_normalize_bouton_id(x) for x in excluded_ids}
    keep_mask = ~df['ID'].map(_normalize_bouton_id).isin(excluded_norm)
    return df[keep_mask].reset_index(drop=True)


def _refresh_shared_datasets():
    """Refresh cached indexes, condition views, and standard derived datasets after any shared filtering."""
    global TRACE_BY_COND_ID, TRACE_BY_ID, resampled_trace_lookup
    global FEATURES_BY_COND_ID, FEATURES_BY_ID, FEATURES_VIEWS, NORM_TRACES_VIEWS
    global PCA_Data_WT_Pooled, PCA_Data_WT_Theo, PCA_Data_WT_Anthime, PCA_Data_SynII
    global PCA_Data_WT_Low_Ca, PCA_Data_WT_High_Ca, PCA_Data_Stability_Before, PCA_Data_Stability_After
    global PCA_Data_50Hz_1_5_Ca, PCA_Data_50Hz_4_Ca, PCA_Data_50Hz_2_5_Ca

    TRACE_BY_COND_ID = NORM_TRACES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
    TRACE_BY_ID = NORM_TRACES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
    resampled_trace_lookup = build_trace_lookup_from_source(TRACE_SINGLE_SOURCE)
    NORM_TRACES_VIEWS = build_condition_views(NORM_TRACES_DATAFRAME)

    FEATURES_BY_COND_ID = FEATURES_DATAFRAME.set_index(['Condition', 'ID'], drop=False)
    FEATURES_BY_ID = FEATURES_DATAFRAME.drop_duplicates('ID').set_index('ID', drop=False)
    FEATURES_VIEWS = build_condition_views(FEATURES_DATAFRAME)

    PCA_Data_WT_Pooled = FEATURES_VIEWS.get('WT_pooled', pd.DataFrame()).copy()
    PCA_Data_WT_Theo = FEATURES_VIEWS.get('WT_Theo', pd.DataFrame()).copy()
    PCA_Data_WT_Anthime = FEATURES_VIEWS.get('WT_Anthime', pd.DataFrame()).copy()
    PCA_Data_SynII = filter_df_by_conditions(FEATURES_DATAFRAME, get_synapsin_conditions())
    PCA_Data_WT_Low_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '20Hz')
    PCA_Data_WT_High_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '20Hz')
    PCA_Data_Stability_Before = _exclude_ids(FEATURES_VIEWS.get('stability_before', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY)
    PCA_Data_Stability_After = _exclude_ids(FEATURES_VIEWS.get('stability_after', pd.DataFrame()).copy(), OUTLIER_IDS_STABILITY)
    PCA_Data_50Hz_1_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '1.5mM', '50Hz')
    PCA_Data_50Hz_4_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '4mM', '50Hz')
    PCA_Data_50Hz_2_5_Ca = filter_df_by_calcium(FEATURES_DATAFRAME, '2.5mM', '50Hz')

def _count_remaining_excluded_ids(df, excluded_ids):
    if df is None or len(excluded_ids) == 0 or 'ID' not in df.columns:
        return 0
    excluded_norm = {_normalize_bouton_id(x) for x in excluded_ids}
    return int(df['ID'].map(_normalize_bouton_id).isin(excluded_norm).sum())


def _print_exclusion_sanity(excluded_ids, label='excluded IDs'):
    print(f"Sanity check after refresh for {label}:")
    for _name in [
        'NORM_TRACES_DATAFRAME',
        'FEATURES_DATAFRAME',
        'FEATURES_DATAFRAME_RAW',
        'RAW_TRACES_DF',
        'RAW_TRACES_DF_RAW',
        'PCA_Data_WT_Pooled',
        'PCA_Data_WT_Theo',
        'PCA_Data_WT_Anthime',
        'PCA_Data_SynII',
        'PCA_Data_WT_Low_Ca',
        'PCA_Data_WT_High_Ca',
        'PCA_Data_50Hz_1_5_Ca',
        'PCA_Data_50Hz_4_Ca',
        'PCA_Data_50Hz_2_5_Ca',
    ]:
        if _name in globals():
            _remaining = _count_remaining_excluded_ids(globals()[_name], excluded_ids)
            print(f"  {_name}: remaining={_remaining}")


print(f"Peak threshold exclusion summary for 50Hz conditions only (threshold = {threshold} ΔF/F0)")
if OUTLIER_IDS_BY_CONDITION:
    for _cond in sorted(OUTLIER_IDS_BY_CONDITION):
        _ids = OUTLIER_IDS_BY_CONDITION[_cond]
        print(f"  {_cond}: {len(_ids)} ID(s) filtered")
        for _id in _ids:
            _peak_match = OUTLIER_TRACE_SUMMARY.loc[(OUTLIER_TRACE_SUMMARY['Condition'] == _cond) & (OUTLIER_TRACE_SUMMARY['ID'] == _id), 'PeakValue']
            _peak_str = f" ({float(_peak_match.iloc[0]):.3f})" if len(_peak_match) else ''
            print(f"    {_id}{_peak_str}")
else:
    print("  No 50Hz IDs exceeded the threshold.")

# Apply threshold outlier exclusion globally; keep stability-pair exclusions local to stability-only views
NORM_TRACES_DATAFRAME = _exclude_ids(NORM_TRACES_DATAFRAME, OUTLIER_IDS_50HZ)
FEATURES_DATAFRAME = _exclude_ids(FEATURES_DATAFRAME, OUTLIER_IDS_50HZ)

if 'FEATURES_DATAFRAME_RAW' in globals():
    FEATURES_DATAFRAME_RAW = _exclude_ids(FEATURES_DATAFRAME_RAW, OUTLIER_IDS_50HZ)
if 'RAW_TRACES_DF' in globals():
    RAW_TRACES_DF = _exclude_ids(RAW_TRACES_DF, OUTLIER_IDS_50HZ)
if 'RAW_TRACES_DF_RAW' in globals():
    RAW_TRACES_DF_RAW = _exclude_ids(RAW_TRACES_DF_RAW, OUTLIER_IDS_50HZ)

# Refresh cached indexes/lookups after filtering
_refresh_shared_datasets()

print(f"Excluded {len(OUTLIER_IDS_50HZ)} 50Hz threshold outlier ID(s) from all reusable dataframes; 20Hz datasets were not targeted by this filter; stability-pair exclusions remain local to stability-only views ({len(OUTLIER_IDS_STABILITY)} IDs)")
_print_exclusion_sanity(OUTLIER_IDS_50HZ, label='50Hz peak-threshold IDs')


#### Pool conditions for the main comparisons

Related acquisition conditions are pooled here to define the core WT, calcium, frequency, SynII, and stability datasets used later in the notebook. The pooled WT dataset is the reference anchor for the PCA and hierarchical clustering analyses.


In [ ]:
# === Helper — Pool conditions for the main comparisons ===

# Build the pooled WT, calcium, frequency, SynII, and stability dataframes used throughout the manuscript figures.
FEATURES_DATAFRAME = make_pooled_features_dataframe(FEATURES_DATAFRAME_RAW, CONDITION_POOLS)
FEATURES_DATAFRAME = enrich_features_dataframe(FEATURES_DATAFRAME)

# Refresh feature caches after pooling
_refresh_shared_datasets()

# Global reusable feature lists
PPR_COLS_GLOBAL = get_ppr_columns(FEATURES_DATAFRAME)

print(f"✓ RAW features (cleaned): {len(FEATURES_DATAFRAME_RAW)} boutons")
print(f"✓ POOLED features: {len(FEATURES_DATAFRAME)} boutons across {len(FEATURES_DATAFRAME['Condition'].unique())} conditions")
print(f"✓ Normalized traces (cleaned): {len(NORM_TRACES_DATAFRAME)}")
print(f"Key datasets: WT_pooled({len(PCA_Data_WT_Pooled)}), SynII({len(PCA_Data_SynII)}), 1.5Ca({len(PCA_Data_WT_Low_Ca)}), 4Ca({len(PCA_Data_WT_High_Ca)})")

# ---------------------------------------------------------------------------
# Per-bouton sex annotation (ID_and_sex.csv): merge a 'Sexe' column onto the
# main feature dataframes so downstream sex-stratified analyses (Supp S8) can
# pull it directly from PCA_Data_WT_Pooled. The CSV has columns
# (condition, ID, sex) with sex in {M, F, UN, NaN}; we keep the existing
# 'Sexe' column name for backward compatibility.
# ---------------------------------------------------------------------------
SEX_FILENAME = 'ID_and_sex.csv'
_sex_path = BASE_DIR / SEX_FILENAME
if _sex_path.exists():
    _sex_df = pd.read_csv(_sex_path)
    _sex_df.columns = [c.strip() for c in _sex_df.columns]
    _id_col = _sex_df.columns[1]
    _sex_col = _sex_df.columns[2]
    _sex_map = (_sex_df[[_id_col, _sex_col]]
                .dropna(subset=[_id_col])
                .drop_duplicates(subset=[_id_col])
                .rename(columns={_id_col: 'ID', _sex_col: 'Sexe'}))
    _sex_map['ID'] = _sex_map['ID'].astype(str).str.strip()
    for _name in ('FEATURES_DATAFRAME', 'FEATURES_DATAFRAME_RAW'):
        if _name in globals():
            _df = globals()[_name]
            _df['ID'] = _df['ID'].astype(str).str.strip()
            if 'Sexe' in _df.columns:
                _df = _df.drop(columns=['Sexe'])
            globals()[_name] = _df.merge(_sex_map, on='ID', how='left')
    _refresh_shared_datasets()
    print(f"Loaded sex annotation for {len(_sex_map)} IDs from {_sex_path.name}")
else:
    print(f"[warn] {_sex_path} not found; 'Sexe' column will be missing")

# ---------------------------------------------------------------------------
# Outlier rejection: WT boutons that fall on the extreme right of PC1 (>8)
# bias the reference PCA / clustering. Detect them on a preliminary fit of
# the WT_pooled features, then drop the offending IDs from every reusable
# dataframe BEFORE the main PCA is fitted downstream.
# ---------------------------------------------------------------------------
HIGH_PC1_THRESHOLD = 8.0

def _detect_high_pc1_outliers(reference_df, threshold=HIGH_PC1_THRESHOLD):
    if reference_df is None or len(reference_df) == 0:
        return []
    from sklearn.decomposition import PCA as _PCA
    from sklearn.preprocessing import StandardScaler as _Scaler
    feats = reference_df[[c for c in PCA_KEEP_COLS if c in reference_df.columns]].copy()
    feats = feats.dropna(axis=0)
    if len(feats) == 0:
        return []
    coords = _PCA(n_components=2).fit_transform(_Scaler().fit_transform(feats))
    mask = coords[:, 0] > threshold
    # feats.index carries the original row labels of reference_df; use .loc
    return reference_df.loc[feats.index[mask], 'ID'].tolist()


OUTLIER_IDS_HIGH_PC1 = sorted(set(_detect_high_pc1_outliers(PCA_Data_WT_Pooled)))

if OUTLIER_IDS_HIGH_PC1:
    FEATURES_DATAFRAME = _exclude_ids(FEATURES_DATAFRAME, OUTLIER_IDS_HIGH_PC1)
    NORM_TRACES_DATAFRAME = _exclude_ids(NORM_TRACES_DATAFRAME, OUTLIER_IDS_HIGH_PC1)
    if 'FEATURES_DATAFRAME_RAW' in globals():
        FEATURES_DATAFRAME_RAW = _exclude_ids(FEATURES_DATAFRAME_RAW, OUTLIER_IDS_HIGH_PC1)
    if 'RAW_TRACES_DF' in globals():
        RAW_TRACES_DF = _exclude_ids(RAW_TRACES_DF, OUTLIER_IDS_HIGH_PC1)
    if 'RAW_TRACES_DF_RAW' in globals():
        RAW_TRACES_DF_RAW = _exclude_ids(RAW_TRACES_DF_RAW, OUTLIER_IDS_HIGH_PC1)
    # refresh derived tables
    _refresh_shared_datasets()

print(f"Excluded {len(OUTLIER_IDS_HIGH_PC1)} high-PC1 (>{HIGH_PC1_THRESHOLD}) outlier IDs from all reusable dataframes:")
for _id in OUTLIER_IDS_HIGH_PC1:
    print(f"  {_id}")
_print_exclusion_sanity(OUTLIER_IDS_HIGH_PC1, label='high-PC1 IDs')

# ---------------------------------------------------------------------------
# Early cohort summary for manuscript reporting.
# "Animals" are approximated here by unique recording days extracted from the
# leading YYYYMMDD token in the bouton/fiber IDs.
# ---------------------------------------------------------------------------
_COHORT_CA_SUFFIXES = re.compile(
    r'[_\s]*(1[._]5[_\s]?Ca|4[_\s]?Ca|2[._]5[_\s]?Ca'
    r'|1[._]5[_\s]?50Hz|2[._]5[_\s]?50Hz|4[_\s]?50Hz'
    r'|50Hz|20Hz)\s*$', re.IGNORECASE
)

def _cohort_strip_calcium(name):
    s = str(name).strip()
    prev = None
    while s != prev:
        prev = s
        s = _COHORT_CA_SUFFIXES.sub('', s).rstrip('_ ')
    return s

def _cohort_pair_base_id(value):
    bid = _normalize_bouton_id(value)
    bid = _cohort_strip_calcium(bid)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_set\d+', '', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_+$', '', bid)
    bid = re.sub(r'_+', '_', bid)
    return bid

def _cohort_recording_day(value):
    match = re.match(r'^(\d{8})', str(value).strip())
    return match.group(1) if match else np.nan

def _cohort_prepare(df):
    d = df.copy()
    if 'ID' not in d.columns:
        raise KeyError("Cohort summary requires an 'ID' column")
    d['ID'] = d['ID'].astype(str).str.strip()
    d['PairBaseID'] = d['ID'].map(_cohort_pair_base_id)
    d['PairFiberID'] = d['PairBaseID'].map(_extract_fiber_id)
    d['PairDay'] = d['PairBaseID'].map(_cohort_recording_day)
    return d

def _cohort_counts(df):
    d = _cohort_prepare(df)
    return {
        'raw_rows': int(len(d)),
        'unique_exact_ids': int(d['ID'].nunique()),
        'unique_boutons': int(d['PairBaseID'].nunique()),
        'unique_fibers': int(d['PairFiberID'].nunique()),
        'unique_days': int(d['PairDay'].dropna().nunique()),
    }

def _paired_overlap_counts(df_a, df_b):
    a = _cohort_prepare(df_a)
    b = _cohort_prepare(df_b)
    pair_ids = sorted(set(a['PairBaseID']).intersection(set(b['PairBaseID'])))
    pair_df = pd.DataFrame({'PairBaseID': pair_ids})
    if len(pair_df) == 0:
        return {'paired_boutons': 0, 'paired_fibers': 0, 'paired_days': 0}
    pair_df['PairFiberID'] = pair_df['PairBaseID'].map(_extract_fiber_id)
    pair_df['PairDay'] = pair_df['PairBaseID'].map(_cohort_recording_day)
    return {
        'paired_boutons': int(len(pair_df)),
        'paired_fibers': int(pair_df['PairFiberID'].nunique()),
        'paired_days': int(pair_df['PairDay'].dropna().nunique()),
    }

ALL_EXPERIMENTAL_FEATURES_FINAL = FEATURES_DATAFRAME_RAW.copy()
WT_2P5_REFERENCE_FINAL = PCA_Data_WT_Pooled.copy()
ALL_2P5_20HZ_FINAL = filter_df_by_conditions(FEATURES_DATAFRAME_RAW, CALCIUM_2_5_20HZ_ALL_CONDITIONS)

DATASET_N_SUMMARY = {
    'wt_2p5_reference': _cohort_counts(WT_2P5_REFERENCE_FINAL),
    'all_2p5_20hz_including_stability_after': _cohort_counts(ALL_2P5_20HZ_FINAL),
    'all_experimental_conditions': _cohort_counts(ALL_EXPERIMENTAL_FEATURES_FINAL),
    'paired_calcium_1p5_vs_4': _paired_overlap_counts(PCA_Data_WT_Low_Ca, PCA_Data_WT_High_Ca),
    'paired_stability_before_vs_after': _paired_overlap_counts(PCA_Data_Stability_Before, PCA_Data_Stability_After),
}

print("\n=== DATASET N SUMMARY (EARLY NOTEBOOK COHORT COUNTS) ===")
print("Counts below use the final reusable dataframes after the shared exclusions above.")
print("Definitions:")
print("  table rows              = rows present in the current summary dataframe")
print("  unique bouton IDs       = unique bouton names exactly as stored in the table")
print("  paired-collapsed boutons= same bouton merged across repeat-condition recordings")
print("  n fibers                = unique fibers after the same collapsing rule")
print("  n days                  = unique recording days (YYYYMMDD prefix in IDs; used here as animal proxy)")
print("  paired subset counts    = overlap between the two matched conditions only")

_wt = DATASET_N_SUMMARY['wt_2p5_reference']
print(
    f"WT pooled 2.5 mM / 20 Hz reference\n"
    f"  table rows={_wt['raw_rows']} | unique bouton IDs={_wt['unique_exact_ids']} | paired-collapsed boutons={_wt['unique_boutons']} | n fibers={_wt['unique_fibers']} | n days={_wt['unique_days']}"
)

_all25 = DATASET_N_SUMMARY['all_2p5_20hz_including_stability_after']
print(
    f"All 2.5 mM / 20 Hz datasets including Stability After\n"
    f"  table rows={_all25['raw_rows']} | unique bouton IDs={_all25['unique_exact_ids']} | paired-collapsed boutons={_all25['unique_boutons']} | n fibers={_all25['unique_fibers']} | n days={_all25['unique_days']}"
)

_all = DATASET_N_SUMMARY['all_experimental_conditions']
print(
    f"All experimental conditions together\n"
    f"  table rows={_all['raw_rows']} | unique bouton IDs={_all['unique_exact_ids']} | paired-collapsed boutons={_all['unique_boutons']} | n fibers={_all['unique_fibers']} | n days={_all['unique_days']}"
)

_ca_pair = DATASET_N_SUMMARY['paired_calcium_1p5_vs_4']
print(
    f"Paired low-vs-high calcium subset (1.5 vs 4 mM, 20 Hz)\n"
    f"  paired boutons={_ca_pair['paired_boutons']} | paired fibers={_ca_pair['paired_fibers']} | paired days={_ca_pair['paired_days']}"
)

_stab_pair = DATASET_N_SUMMARY['paired_stability_before_vs_after']
print(
    f"Paired stability subset (Before vs After)\n"
    f"  paired boutons={_stab_pair['paired_boutons']} | paired fibers={_stab_pair['paired_fibers']} | paired days={_stab_pair['paired_days']}"
)


## Section 1 — Figure 1: Representative bouton readouts and quantal size

An example high-release bouton (2.5 mM Ca²⁺, 20 Hz) illustrates single-trial Glu-Ts and NNLS amplitude decomposition. A representative PF recording then shows mean responses across boutons from the same fiber, followed by the quantal size distribution (Q).

In [ ]:
# === Fig 1c — Example NNLS trace ===
# Large 2.5 mM bouton example before PCA: first three events + NNLS + decay summary

import os
import subprocess
import sys
from pathlib import Path

from scipy.optimize import curve_fit
from scipy.signal import savgol_filter

EXAMPLE_LARGE_2P5_BOUTON_ID = '20210128_linescan1_20Hz_10pulses_2.5mMCa_bouton1_traces_converted'#'20210125_linescan1_20Hz_10pulses_2.5mMCa_bouton2_traces_converted'
EXAMPLE_LARGE_2P5_MIN_TRIALS = 5
EXAMPLE_LARGE_2P5_MAX_FAIL1 = 20.0
EXAMPLE_LARGE_2P5_TRIAL_INPUT_COL_1BASED = 1
EXAMPLE_LARGE_2P5_PULSES = (1, 2, 3)
EXAMPLE_LARGE_2P5_XLIM = (0.985, 1.145)
EXAMPLE_LARGE_2P5_DECAY_GUARD_S = 0.003
EXAMPLE_LARGE_2P5_AMP_LEFT_SHIFT_S = 0.004
EXAMPLE_LARGE_2P5_FIT_UPSAMPLE = 1
EXAMPLE_LARGE_2P5_TRACE_DISPLAY = 'raw'  # 'raw' or 'savgol'
EXAMPLE_LARGE_2P5_RESIDUAL_OFFSET = -1.0


def _sanitize_token(value):
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))


def _overlay_npz_matches(npz_path, condition, file_stem, trial_input_col_1based):
    if not npz_path.exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        required = {
            'condition', 'file_stem', 'time_s', 'train_start_s', 'isi_s',
            'average_yproc', 'average_yhat', 'average_amp_nnls_corr', 'average_ppr_nnls_corr',
            'trial_input_col_1based'
        }
        ok = (
            required.issubset(set(d.files))
            and str(d['condition']) == str(condition)
            and str(d['file_stem']) == str(file_stem)
            and int(d['trial_input_col_1based']) == int(trial_input_col_1based)
        )
        d.close()
        return ok
    except Exception:
        return False


def _event_failure_mask_local(row, n_pulses=10):
    thr = float(row['thr_shared']) if np.isfinite(row['thr_shared']) else np.nan
    fail_mask = np.full(int(n_pulses), np.nan)
    for k in range(int(n_pulses)):
        vals = []
        for col in (f'AMP{k+1}_CORR', f'AMP{k+1}_UNCORR'):
            if col in row.index and np.isfinite(row[col]):
                vals.append(float(row[col]))
        if np.isfinite(thr) and vals:
            fail_mask[k] = 1.0 if np.min(vals) < thr else 0.0
    return fail_mask


def _candidate_workbook_stems_local(bouton_id):
    requested_id = str(bouton_id).strip().removesuffix('.xlsx')
    requested_base = _normalize_bouton_id(requested_id)
    return list(dict.fromkeys([requested_id, requested_base, f'{requested_base}_traces_converted']))


def _resolve_workbook_stem_local(condition_name, bouton_id, trials_df):
    candidates = _candidate_workbook_stems_local(bouton_id)
    requested_base = _normalize_bouton_id(bouton_id)
    trial_matches = trials_df[
        (trials_df['condition'].astype(str) == str(condition_name))
        & (trials_df['file'].map(_normalize_bouton_id) == requested_base)
    ]['file'].dropna().astype(str).tolist()
    for file_stem in trial_matches:
        if (BASE_DIR / condition_name / f'{file_stem}.xlsx').exists():
            return file_stem
    for file_stem in candidates:
        if (BASE_DIR / condition_name / f'{file_stem}.xlsx').exists():
            return file_stem
    raise FileNotFoundError(f'No raw workbook found for {condition_name} / {bouton_id}')


def _upsample_plot_series_local(x, y, factor=1, kind='cubic'):
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    factor = int(factor)
    valid = np.isfinite(x) & np.isfinite(y)
    if factor <= 1 or np.count_nonzero(valid) < 2:
        return x[valid], y[valid]
    x_valid = x[valid]
    y_valid = y[valid]
    interp_kind = kind if (kind != 'cubic' or x_valid.size >= 4) else 'linear'
    interp_fn = interp1d(x_valid, y_valid, kind=interp_kind, bounds_error=False, fill_value='extrapolate')
    x_dense = np.linspace(float(x_valid[0]), float(x_valid[-1]), max(x_valid.size * factor, x_valid.size))
    y_dense = np.asarray(interp_fn(x_dense), float)
    return x_dense, y_dense


example_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
example_trials = example_trials[example_trials['condition'].isin(WT_2_5_20HZ_CONDITIONS)].copy()
source_candidates = FEATURES_DATAFRAME[
    FEATURES_DATAFRAME['Condition'].isin(WT_2_5_20HZ_CONDITIONS)
].copy()
source_candidates['_source_priority'] = source_candidates['Condition'].map(
    lambda cond: WT_2_5_20HZ_CONDITIONS.index(cond) if cond in WT_2_5_20HZ_CONDITIONS else len(WT_2_5_20HZ_CONDITIONS)
)
source_candidates['_BaseID'] = source_candidates['ID'].map(_normalize_bouton_id)
source_candidates = source_candidates.sort_values(['_source_priority', 'AMP1'], ascending=[True, False])
source_candidates = source_candidates.drop_duplicates('_BaseID', keep='first').copy()
trial_counts = example_trials.groupby(['condition', 'file']).size().rename('n_trials').reset_index()
pooled_candidates = source_candidates.merge(
    trial_counts,
    left_on=['Condition', 'ID'],
    right_on=['condition', 'file'],
    how='left',
)
pooled_candidates['n_trials'] = pooled_candidates['n_trials'].fillna(0).astype(int)
pooled_candidates['xlsx_exists'] = [
    any((BASE_DIR / cond / f'{candidate}.xlsx').exists() for candidate in _candidate_workbook_stems_local(bouton_id))
    for cond, bouton_id in zip(pooled_candidates['Condition'], pooled_candidates['ID'])
]

if EXAMPLE_LARGE_2P5_BOUTON_ID is not None:
    requested_id = str(EXAMPLE_LARGE_2P5_BOUTON_ID).strip().removesuffix('.xlsx')
    requested_base = _normalize_bouton_id(requested_id)
    selected_candidates = pooled_candidates[
        pooled_candidates['ID'].astype(str).str.strip().isin({requested_id, requested_base, f'{requested_base}_traces_converted'})
        | (pooled_candidates['_BaseID'].astype(str) == str(requested_base))
    ].copy()
    if selected_candidates.empty:
        raise ValueError(f"Requested example bouton not found in pooled 2.5 mM data: {EXAMPLE_LARGE_2P5_BOUTON_ID}")
    example_row = selected_candidates.sort_values('AMP1', ascending=False).iloc[0]
else:
    auto_candidates = pooled_candidates[
        pooled_candidates['xlsx_exists']
        & (pooled_candidates['n_trials'] >= EXAMPLE_LARGE_2P5_MIN_TRIALS)
    ].copy()
    preferred_candidates = auto_candidates[auto_candidates['%Fail1'] <= EXAMPLE_LARGE_2P5_MAX_FAIL1].copy()
    if preferred_candidates.empty:
        preferred_candidates = auto_candidates
    if preferred_candidates.empty:
        raise RuntimeError('No pooled 2.5 mM bouton with matching raw workbook and trial rows was found.')
    example_row = preferred_candidates.sort_values(['AMP1', 'PPR2/1'], ascending=[False, False]).iloc[0]

example_condition = str(example_row['Condition'])
example_base_id = _normalize_bouton_id(example_row['ID'])
example_file_stem = _resolve_workbook_stem_local(example_condition, example_row['ID'], example_trials)

REPO_ROOT = Path.cwd()
helper_script = REPO_ROOT / 'Feature_extraction' / 'tmp_selected_recording_nnls_overlay.py'
overlay_npz = REPO_ROOT / (
    f"_tmp_large2p5_overlay__{_sanitize_token(example_condition)}__"
    f"{_sanitize_token(example_file_stem)}__trial{int(EXAMPLE_LARGE_2P5_TRIAL_INPUT_COL_1BASED)}.npz"
)

if not _overlay_npz_matches(overlay_npz, example_condition, example_file_stem, EXAMPLE_LARGE_2P5_TRIAL_INPUT_COL_1BASED):
    env = dict(os.environ)
    env['PYTHONIOENCODING'] = 'utf-8'
    subprocess.run(
        [
            sys.executable,
            str(helper_script),
            '--condition', example_condition,
            '--file', f'{example_file_stem}.xlsx',
            '--trial', str(int(EXAMPLE_LARGE_2P5_TRIAL_INPUT_COL_1BASED)),
            '--out', str(overlay_npz),
        ],
        check=True,
        cwd=str(REPO_ROOT),
        env=env,
    )

overlay_data = np.load(overlay_npz, allow_pickle=True)
overlay_time = np.asarray(overlay_data['time_s'], float)
train_start_s = float(overlay_data['train_start_s'])
isi_s = float(overlay_data['isi_s'])
average_yproc = np.asarray(overlay_data['average_yproc'], float)
average_yhat = np.asarray(overlay_data['average_yhat'], float)
average_amp_nnls_corr = np.asarray(overlay_data['average_amp_nnls_corr'], float)
average_ppr_nnls_corr = np.asarray(overlay_data['average_ppr_nnls_corr'], float)
overlay_data.close()

plot_time = overlay_time + (1.0 - train_start_s)
stim_times = 1.0 + isi_s * np.arange(10)
avg_sg = savgol_filter(average_yproc, 9, 2, mode='interp')
trace_display_mode = str(EXAMPLE_LARGE_2P5_TRACE_DISPLAY).strip().lower()
if trace_display_mode == 'raw':
    average_trace_plot = average_yproc
    average_trace_label = 'Raw average trace'
elif trace_display_mode == 'savgol':
    average_trace_plot = avg_sg
    average_trace_label = 'SavGol average trace'
else:
    raise ValueError("EXAMPLE_LARGE_2P5_TRACE_DISPLAY must be 'raw' or 'savgol'")
plot_time_fit, average_yhat_plot = _upsample_plot_series_local(plot_time, average_yhat, factor=EXAMPLE_LARGE_2P5_FIT_UPSAMPLE)
residual_raw_minus_nnls = np.asarray(average_yproc - average_yhat, float)
plot_time_resid, residual_plot = _upsample_plot_series_local(plot_time, residual_raw_minus_nnls, factor=EXAMPLE_LARGE_2P5_FIT_UPSAMPLE)
residual_plot_shifted = residual_plot + float(EXAMPLE_LARGE_2P5_RESIDUAL_OFFSET)

bouton_trials = example_trials[
    (example_trials['condition'].astype(str) == example_condition)
    & (example_trials['file'].map(_normalize_bouton_id) == example_base_id)
].copy().sort_values('trial_input_col_1based')
if bouton_trials.empty:
    raise RuntimeError(f'No trial rows found for {example_condition} / {example_file_stem}')

all_fail_masks = np.asarray([_event_failure_mask_local(row, n_pulses=10) for _, row in bouton_trials.iterrows()], dtype=float)
fail_pct_by_pulse = 100.0 * np.nanmean(all_fail_masks == 1.0, axis=0)

first_three_peak_times = []
for pulse_number in EXAMPLE_LARGE_2P5_PULSES:
    stim_time = stim_times[pulse_number - 1]
    win = (plot_time >= stim_time - 0.001) & (plot_time <= stim_time + 0.020)
    if not np.any(win):
        raise RuntimeError(f'No average-fit samples found around pulse {pulse_number}')
    local_idx = np.nanargmax(average_yhat[win])
    local_times = plot_time[win]
    first_three_peak_times.append(float(local_times[local_idx]))

first_three_peak_times = np.asarray(first_three_peak_times, float)
example_amp_values = np.asarray([average_amp_nnls_corr[pulse_number - 1] for pulse_number in EXAMPLE_LARGE_2P5_PULSES], dtype=float)
if not np.isfinite(example_amp_values[0]) or example_amp_values[0] <= 0:
    raise RuntimeError('The first fitted event amplitude is not valid for PPR computation.')
example_ppr_values = np.asarray([average_ppr_nnls_corr[pulse_number - 1] for pulse_number in EXAMPLE_LARGE_2P5_PULSES], dtype=float)
if np.any(~np.isfinite(example_ppr_values)):
    raise RuntimeError('The corrected NNLS PPR values are not valid for the requested pulses.')

next_stim_time = stim_times[len(EXAMPLE_LARGE_2P5_PULSES)]
decay_start = first_three_peak_times[-1]
decay_stop = next_stim_time - EXAMPLE_LARGE_2P5_DECAY_GUARD_S
decay_mask = (plot_time >= decay_start) & (plot_time <= decay_stop) & np.isfinite(average_yhat)
if np.count_nonzero(decay_mask) < 6:
    raise RuntimeError('Not enough samples to fit the post-A3 decay segment.')

decay_t = plot_time[decay_mask]
decay_y = average_yhat[decay_mask]
decay_t0 = float(decay_t[0])


def _exp_decay(t, amp, tau, offset):
    return offset + amp * np.exp(-(t - decay_t0) / tau)


p0 = [max(float(decay_y[0] - np.nanmin(decay_y)), 1e-3), 0.015, max(float(np.nanmin(decay_y)), 0.0)]
bounds = ([0.0, 0.001, -0.2], [10.0, 0.200, 1.0])
fit_params, _ = curve_fit(_exp_decay, decay_t, decay_y, p0=p0, bounds=bounds, maxfev=20000)
decay_tau_ms = float(fit_params[1] * 1000.0)
decay_fit = _exp_decay(decay_t, *fit_params)
decay_t_plot, decay_fit_plot = _upsample_plot_series_local(decay_t, decay_fit, factor=EXAMPLE_LARGE_2P5_FIT_UPSAMPLE)

fail_vals_first_three = fail_pct_by_pulse[:len(EXAMPLE_LARGE_2P5_PULSES)]

fig, ax_trace = make_figure_grid(figsize=(8, 5))
trace_color = '#8f8f8f'
fit_color = '#f28e2b'
decay_color = '#d62728'
amp_color = 'black'
residual_color = '#9467bd'

ax_trace.plot(plot_time, average_trace_plot, color=trace_color, lw=1.6, alpha=0.95, label=average_trace_label)
ax_trace.plot(plot_time_fit, average_yhat_plot, color=fit_color, lw=2.4, label='NNLS fit')
ax_trace.plot(decay_t_plot, decay_fit_plot, color=decay_color, lw=1.8, ls='--', label=f'Decay fit: τ={decay_tau_ms:.1f} ms')
ax_trace.axhline(EXAMPLE_LARGE_2P5_RESIDUAL_OFFSET, color=residual_color, lw=0.8, ls=':', alpha=0.7)
ax_trace.plot(plot_time_resid, residual_plot_shifted, color=residual_color, lw=1.0, alpha=0.95, label='Residual (raw - NNLS)')
ax_trace.axhline(0.0, color='0.55', lw=0.9, ls=':')

trace_ymin = float(np.nanmin(np.r_[average_trace_plot, average_yhat_plot, residual_plot_shifted, [EXAMPLE_LARGE_2P5_RESIDUAL_OFFSET]]))
trace_ymax = float(np.nanmax(np.r_[average_trace_plot, average_yhat_plot, decay_fit_plot]))
ypad = max((trace_ymax - trace_ymin) * 0.12, 0.05)
ax_trace.set_ylim(trace_ymin - 0.05 * ypad, trace_ymax + ypad)
ax_trace.set_xlim(*EXAMPLE_LARGE_2P5_XLIM)

stim_tick_low = ax_trace.get_ylim()[1] - 0.10 * (ax_trace.get_ylim()[1] - ax_trace.get_ylim()[0])
stim_tick_high = ax_trace.get_ylim()[1] - 0.04 * (ax_trace.get_ylim()[1] - ax_trace.get_ylim()[0])
for stim_time in stim_times[:len(EXAMPLE_LARGE_2P5_PULSES)]:
    ax_trace.plot([stim_time, stim_time], [stim_tick_low, stim_tick_high], color='black', lw=1.3, solid_capstyle='butt', zorder=5)

for idx, (stim_time, amp_v) in enumerate(zip(stim_times[:len(EXAMPLE_LARGE_2P5_PULSES)], example_amp_values), start=1):
    amp_x = stim_time - EXAMPLE_LARGE_2P5_AMP_LEFT_SHIFT_S
    ax_trace.vlines(amp_x, 0.0, amp_v, color=amp_color, lw=1.5, alpha=0.95, zorder=6)
    ax_trace.plot([amp_x, stim_time - 0.0008], [amp_v, amp_v], color=amp_color, lw=1.1, alpha=0.95, zorder=6)
    ax_trace.annotate(
        f'A{idx}',
        xy=(amp_x, amp_v),
        xytext=(0, 11),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=5,
        color=amp_color,
        fontweight='bold',
    )
    ax_trace.annotate(
        f'{amp_v:.2f}',
        xy=(amp_x, amp_v),
        xytext=(0, -16),
        textcoords='offset points',
        ha='center',
        va='top',
        fontsize=5,
        color=amp_color,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.75, pad=0.15),
    )

summary_text = (
    f"{example_condition}\n{example_file_stem}\n"
    f"n={len(bouton_trials)} trials\n"
    f"A1={example_amp_values[0]:.2f}  A2={example_amp_values[1]:.2f}  A3={example_amp_values[2]:.2f}\n"
    f"PPR2/1={example_ppr_values[1]:.2f}  PPR3/1={example_ppr_values[2]:.2f}\n"
    f"Fail1={fail_vals_first_three[0]:.0f}%  Fail2={fail_vals_first_three[1]:.0f}%  Fail3={fail_vals_first_three[2]:.0f}%"
)
ax_trace.text(
    0.015,
    0.98,
    summary_text,
    transform=ax_trace.transAxes,
    ha='left',
    va='top',
    fontsize=8,
    bbox=dict(facecolor='white', edgecolor='0.85', boxstyle='round,pad=0.28'),
)
ax_trace.set_xlabel('Time (s)')
ax_trace.set_ylabel('ΔF/F₀')
ax_trace.set_title('Representative bouton NNLS trace', fontweight='bold')
if 'style_trace_axis' in globals():
    style_trace_axis(ax_trace)
add_legend(ax_trace, loc='upper left', fontsize=6, frameon=False, outside=True)
output_file = OUTPUT_DIR / 'Fig1_c_example_nnls_trace.pdf'
finalize_figure(fig, save_path=output_file, show=True)

print(f"✓ Selected pooled 2.5 mM bouton: {example_file_stem} ({example_condition})")
print(f"✓ A1={example_amp_values[0]:.3f}, A2={example_amp_values[1]:.3f}, A3={example_amp_values[2]:.3f}")
print(f"✓ PPR2/1={example_ppr_values[1]:.3f}, PPR3/1={example_ppr_values[2]:.3f}")
print(f"✓ Fail1={fail_vals_first_three[0]:.1f}%, Fail2={fail_vals_first_three[1]:.1f}%, Fail3={fail_vals_first_three[2]:.1f}%")
print(f"✓ Local post-A3 decay tau={decay_tau_ms:.1f} ms")
print(f"✓ Saved to {output_file}")


In [ ]:
# === Fig 1d — Example PF bouton responses ===
# Plot mean responses from all boutons detected in one representative PF recording.

from scipy.signal import savgol_filter
import re

FIG1_FIBER_PREFIX = '20220426_linescan2'
FIG1_FIBER_CONDITION = 'WT_Theo'
FIG1_FIBER_TRAIN_START_S = 0.552
FIG1_FIBER_XLIM = (0.49, 1.06)
FIG1_FIBER_WINDOW = 9
FIG1_FIBER_POLY = 2
FIG1_TRACE_COLOR = '#1b8a5a'
FIG1_ROW_STEP = 1.0
FIG1_SCALE_DFF = 0.8
FIG1_SCALE_TIME_S = 0.1

fiber_rows = RAW_TRACES_DF[
    (RAW_TRACES_DF['Condition'].astype(str) == FIG1_FIBER_CONDITION)
    & (RAW_TRACES_DF['ID'].astype(str).str.startswith(FIG1_FIBER_PREFIX))
].copy()

if fiber_rows.empty:
    raise ValueError(f'No traces found for {FIG1_FIBER_PREFIX} in {FIG1_FIBER_CONDITION}')

def _bouton_sort_key_local(bouton_id):
    match = re.search(r'bouton(\d+)', str(bouton_id))
    return int(match.group(1)) if match else 10**9

fiber_rows['_bouton_order'] = fiber_rows['ID'].map(_bouton_sort_key_local)
fiber_rows = fiber_rows.sort_values('_bouton_order').reset_index(drop=True)

trial_summary = (
    pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
    .loc[lambda df: (df['condition'].astype(str) == FIG1_FIBER_CONDITION) & (df['file'].astype(str).str.startswith(FIG1_FIBER_PREFIX))]
    .groupby('file')
    .agg(
        n_trials=('trial', 'count'),
        n_success=('status', lambda s: int((s.astype(str) == 'success').sum())),
    )
    .reset_index()
)

trace_cache = []
for _, row in fiber_rows.iterrows():
    time = np.asarray(row['Time'], dtype=float)
    avg = np.asarray(row['Avg'], dtype=float)
    if np.isfinite(avg).sum() >= 3:
        win_len = min(FIG1_FIBER_WINDOW, len(avg))
        if win_len % 2 == 0:
            win_len -= 1
        if win_len >= 3:
            smooth = savgol_filter(avg, window_length=win_len, polyorder=min(FIG1_FIBER_POLY, win_len - 1), mode='interp')
        else:
            smooth = avg.copy()
    else:
        smooth = avg.copy()
    trace_cache.append((str(row['ID']), time, smooth))

fig, ax = make_figure_grid(figsize=(3.2, 3.2))
ax = np.asarray(ax).reshape(-1)[0]

row_offsets = FIG1_ROW_STEP * np.arange(len(trace_cache) - 1, -1, -1, dtype=float)
stim_times = [FIG1_FIBER_TRAIN_START_S + i * 0.05 for i in range(10)]
tick_y0 = row_offsets[0] + 0.18
tick_y1 = row_offsets[0] + 0.32

for idx, ((bouton_id, time, smooth), offset) in enumerate(zip(trace_cache, row_offsets), start=1):
    ax.plot(time, smooth + offset, color=FIG1_TRACE_COLOR, linewidth=0.85)
    ax.text(
        FIG1_FIBER_XLIM[0] - 0.018,
        offset + 0.02,
        f'{idx}',
        ha='right',
        va='center',
        fontsize=8,
        fontweight='bold',
        color='black',
    )

for stim_t in stim_times:
    ax.plot([stim_t, stim_t], [tick_y0, tick_y1], color='black', lw=0.9, solid_capstyle='butt')

top_center = 0.5 * (stim_times[0] + stim_times[-1])
ax.text(top_center, tick_y1 + 0.08, '20 Hz', ha='center', va='bottom', fontsize=8, color='black')

scale_x0 = FIG1_FIBER_XLIM[0] + 0.015
scale_y0 = row_offsets[0] + 0.08
ax.plot([scale_x0, scale_x0 + FIG1_SCALE_TIME_S], [scale_y0, scale_y0], color='black', lw=1.0)
ax.plot([scale_x0, scale_x0], [scale_y0, scale_y0 + FIG1_SCALE_DFF], color='black', lw=1.0)
ax.text(scale_x0 + FIG1_SCALE_TIME_S * 0.5, scale_y0 - 0.10, '100 ms', ha='center', va='top', fontsize=7)
ax.text(scale_x0 + 0.012, scale_y0 + FIG1_SCALE_DFF * 0.5, '0.8\nΔF/F₀', ha='left', va='center', fontsize=7, linespacing=0.9)

ax.set_xlim(*FIG1_FIBER_XLIM)
ax.set_ylim(-0.25, row_offsets[0] + 1.02)
ax.axis('off')
fig.subplots_adjust(left=0.16, right=0.98, top=0.92, bottom=0.08)

output_file = OUTPUT_DIR / 'Fig1_d_example_pf_bouton_responses.pdf'
finalize_figure(fig, save_path=output_file)

print(f'✓ Plotted {len(fiber_rows)} boutons from {FIG1_FIBER_PREFIX}')
print(f'✓ Saved to {output_file}')
for bouton_id in fiber_rows['ID'].astype(str):
    trial_row = trial_summary[trial_summary['file'].astype(str) == bouton_id]
    if trial_row.empty:
        print(f'  {bouton_id}: trial summary unavailable')
    else:
        print(f"  {bouton_id}: n={int(trial_row['n_trials'].iloc[0])} trials, successes={int(trial_row['n_success'].iloc[0])}")


In [ ]:
# === Helper — Q estimation setup ===


# Configure the quantal-size estimation inputs and per-trial tables used by the Q-related analyses.
# CELL 0 : Shared utilities, style, smoothing engine
# Assumes upstream: trials_all, PCA_Data_WT_Pooled_clustered,
#   pca_coordinates, cluster_assignments, sorted_clusters,
#   cluster_color_lookup, FEATURES_DATAFRAME, strip_calcium,
#   ppr_cols, num_pulses, OUTPUT_DIR, BASE_DIR, PPR_TRIALS_FILENAME

import re, json, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import linregress, norm, spearmanr, kruskal, pearsonr
from scipy.special import gammaln
from scipy.spatial.distance import cdist
from collections import defaultdict, OrderedDict
from pathlib import Path

# Global figure defaults (PNAS-aligned): no grid, no top/right spines, no legend box
if 'clamp_fontsize' not in globals():
    def clamp_fontsize(value):
        return float(np.clip(float(value), 6.0, 12.0))
if 'clamp_linewidth' not in globals():
    def clamp_linewidth(value):
        return float(np.clip(float(value), 0.25, 1.5))
if 'clamp_markersize' not in globals():
    def clamp_markersize(value):
        return float(np.clip(float(value), 3.0, 9.0))
if '_enforce_pnas_artist_style' not in globals():
    def _enforce_pnas_artist_style(ax):
        for line in ax.lines:
            line.set_linewidth(clamp_linewidth(line.get_linewidth()))
            line.set_markersize(clamp_markersize(line.get_markersize()))
        for collection in ax.collections:
            if hasattr(collection, 'get_linewidths'):
                widths = collection.get_linewidths()
                if widths is not None and len(widths) > 0:
                    collection.set_linewidths(np.clip(widths, 0.25, 1.5))

plt.rcParams['axes.grid'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['legend.frameon'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
plt.rcParams['svg.fonttype'] = 'none'
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']
plt.rcParams['font.size'] = clamp_fontsize(8.0)
plt.rcParams['axes.labelsize'] = clamp_fontsize(8.0)
plt.rcParams['axes.titlesize'] = clamp_fontsize(8.0)
plt.rcParams['xtick.labelsize'] = clamp_fontsize(6.0)
plt.rcParams['ytick.labelsize'] = clamp_fontsize(6.0)
plt.rcParams['legend.fontsize'] = clamp_fontsize(6.0)
plt.rcParams['contour.linewidth'] = clamp_linewidth(0.8)
if '_install_pnas_matplotlib_sizers' in globals():
    _install_pnas_matplotlib_sizers()

warnings.filterwarnings('ignore', category=RuntimeWarning)

# All conditions recorded at 2.5 mM Ca, 20 Hz (for quantal size estimation AND normalization)
conditions_2_5_20Hz = get_friendly_conditions('2.5mM_20Hz_WT')

# All conditions recorded at 2.5 mM Ca (any frequency): for grouping later
conditions_2_5_all = get_calcium_conditions('2.5mM', 'all')

# Map each condition to its [Ca] and frequency for later grouping
condition_meta = {}
for cond in get_calcium_conditions('1.5mM', '20Hz'):
    condition_meta[cond] = {'Ca': 1.5, 'Freq': 20}
for cond in get_calcium_conditions('1.5mM', '50Hz'):
    condition_meta[cond] = {'Ca': 1.5, 'Freq': 50}
for cond in get_calcium_conditions('2.5mM', '20Hz'):
    condition_meta[cond] = {'Ca': 2.5, 'Freq': 20}
for cond in get_calcium_conditions('2.5mM', '50Hz'):
    condition_meta[cond] = {'Ca': 2.5, 'Freq': 50}
for cond in get_calcium_conditions('4mM', '20Hz'):
    condition_meta[cond] = {'Ca': 4.0, 'Freq': 20}
for cond in get_calcium_conditions('4mM', '50Hz'):
    condition_meta[cond] = {'Ca': 4.0, 'Freq': 50}


# == Upstream dependency guard ====================================
# These should already be defined in your notebook from earlier cells.
# Uncomment / adjust if running this file standalone.
# OUTPUT_DIR = Path('figures')
# BASE_DIR   = Path('data')
# PPR_TRIALS_FILENAME = 'PPR_trials.xlsx'
if 'ppr_cols' not in dir():
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11)]
if 'num_pulses' not in dir():
    num_pulses = 10

# == strip_calcium: normalise bouton IDs across Ca²⁺ conditions ==
# Removes calcium/frequency suffixes so the same bouton recorded
# under different conditions maps to a single base ID.
_CA_SUFFIXES = re.compile(
    r'[_\s]*(1[._]5[_\s]?Ca|4[_\s]?Ca|2[._]5[_\s]?Ca'
    r'|1[._]5[_\s]?50Hz|2[._]5[_\s]?50Hz|4[_\s]?50Hz'
    r'|50Hz|20Hz)\s*$', re.IGNORECASE)

def strip_calcium(name: str) -> str:
    """Remove trailing Ca²⁺ / frequency tags from a bouton file-ID."""
    s = str(name).strip()
    prev = None
    while s != prev:          # iterative : handles stacked suffixes
        prev = s
        s = _CA_SUFFIXES.sub('', s).rstrip('_ ')
    return s

# == Publication style ============================================
def style_ax(ax, xlabel=None, ylabel=None, title=None):
    """Clean axes style; enforce unified limits when plotting PCA space."""
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    for spine_name in ('left', 'bottom'):
        if spine_name in ax.spines:
            ax.spines[spine_name].set_linewidth(1.0)
    ax.tick_params(direction='out', length=4, width=1.0, labelsize=6)
    ax.grid(False)
    show_x, show_y = (_outer_axis_label_visibility(ax) if '_outer_axis_label_visibility' in globals() else (True, True))

    if xlabel == 'PC1' and 'style_pca_axes' in globals():
        xlim = (-7, 10)
        ylim = (-6, 6)
        if len(ax.images) > 0:
            ext = ax.images[0].get_extent()
            if ext is not None and len(ext) == 4:
                xlim = (float(min(ext[0], ext[1])), float(max(ext[0], ext[1])))
                ylim = (float(min(ext[2], ext[3])), float(max(ext[2], ext[3])))

        style_pca_axes(
            ax,
            title=title,
            xlim=xlim,
            ylim=ylim,
legend=False,
            show_xlabel=show_x,
            show_ylabel=(show_y and bool(ylabel)),
            extend_limits=True,
            align_to_background=True,
        )
        if '_enforce_pnas_artist_style' in globals():
            _enforce_pnas_artist_style(ax)
        return

    if xlabel and show_x:
        ax.set_xlabel(xlabel)
    elif xlabel is not None:
        ax.set_xlabel('')

    if ylabel and show_y:
        ax.set_ylabel(ylabel)
    elif ylabel is not None:
        ax.set_ylabel('')

    if ylabel is not None and 'N_k' in str(ylabel):
        ax.set_ylim(bottom=0)

    if title:
        ax.set_title(title, fontsize=clamp_fontsize(8.0), fontweight='bold')

    if '_enforce_pnas_artist_style' in globals():
        _enforce_pnas_artist_style(ax)

def add_legend(ax, *args, **kw):
    kw = dict(kw)
    outside = kw.pop('outside', None)
    kw.setdefault('frameon', False)
    kw.setdefault('fontsize', clamp_fontsize(6.0))
    if outside is True:
        setattr(ax, '_legend_force_outside', True)
        setattr(ax, '_legend_force_inside', False)
    elif outside is False:
        setattr(ax, '_legend_force_outside', False)
        setattr(ax, '_legend_force_inside', True)
    lg = ax.legend(*args, **kw)
    if lg is not None:
        for txt in lg.get_texts():
            txt.set_text(_compact_legend_label(_strip_result_numbers(txt.get_text())))
    if outside is True and 'apply_external_legend' in globals():
        try:
            apply_external_legend(ax.figure, max_in_axes_items=0, min_repeated_panels=1)
        except Exception:
            pass
    return lg


def grid_size(nrows=1, ncols=1, panel_kind='simple'):
    """Notebook-local bridge to the global grid sizing helper."""
    nrows = max(1, int(nrows))
    ncols = max(1, int(ncols))
    if "recommended_panel_figsize" in globals():
        return recommended_panel_figsize(nrows=nrows, ncols=ncols, panel_kind=panel_kind)
    base_w = 8.7 / 2.54
    base_h = base_w * 0.78
    return base_w * ncols, base_h * nrows

# == Smoothing engine (flags set in Cell C, used by C:E) ==========
# Defaults : overridden in Cell C
SMOOTH_ROBUST      = True    # clip outliers before smoothing
SMOOTH_SIGMA_FACTOR   = 2.0    # sigma = median_NN_dist × factor
SMOOTH_SUPPORT_RADIUS = 3.0    # keep grid points within this many sigma of a datapoint
SMOOTH_CLIP_PCT       = 5      # percentile for robust clipping
GRID_RES              = 80     # grid resolution for heatmaps

def smooth_field(xy, vals, mask, gx, gy):
    """Gaussian-kernel smoothing with support masking.
    Grid points farther than one smoothing sigma from every datapoint are set to NaN.
    Returns (grid_image, sigma_used).
    """
    pts, v = xy[mask], vals[mask]
    Xg, Yg = np.meshgrid(gx, gy)
    if len(pts) < 3:
        return np.full(Xg.shape, np.nan), np.nan
    if SMOOTH_ROBUST:
        lo, hi = np.nanpercentile(v, [SMOOTH_CLIP_PCT, 100-SMOOTH_CLIP_PCT])
        v = np.clip(v, lo, hi)
    d = cdist(pts, pts); np.fill_diagonal(d, np.inf)
    sig = np.median(np.min(d, axis=1)) * SMOOTH_SIGMA_FACTOR
    gp = np.column_stack([Xg.ravel(), Yg.ravel()])
    dist_gp = cdist(gp, pts)
    w = np.exp(-0.5 * (dist_gp / sig)**2)
    ws = w.sum(1)
    sg = np.full(len(gp), np.nan)
    ok = ws > 1e-12
    support = np.min(dist_gp, axis=1) <= (SMOOTH_SUPPORT_RADIUS * sig)
    ok = ok & support
    sg[ok] = (w[ok] * v).sum(1) / ws[ok]
    return sg.reshape(Xg.shape), sig

# == PCA grid helpers (set once, reused everywhere) ===============
def setup_pca_grid(coords, pad=0.5, at_least_reference=True):
    coords = np.asarray(coords, float)
    xmn, xmx = coords[:,0].min()-pad, coords[:,0].max()+pad
    ymn, ymx = coords[:,1].min()-pad, coords[:,1].max()+pad
    if at_least_reference and 'get_reference_pca_limits' in globals():
        ref_xlim, ref_ylim = get_reference_pca_limits(pad=pad)
        xmn = min(float(xmn), float(ref_xlim[0]))
        xmx = max(float(xmx), float(ref_xlim[1]))
        ymn = min(float(ymn), float(ref_ylim[0]))
        ymx = max(float(ymx), float(ref_ylim[1]))
    gx = np.linspace(xmn, xmx, GRID_RES)
    gy = np.linspace(ymn, ymx, GRID_RES)
    ext = [xmn, xmx, ymn, ymx]
    return gx, gy, ext

# == Trial amplitude extractor ====================================
_amp_corr   = ['AMP1_UNCORR'] + [f'AMP{k}_CORR'   for k in range(2, 11)]
_amp_uncorr = ['AMP1_UNCORR'] + [f'AMP{k}_UNCORR' for k in range(2, 11)]
N_STIM = 10

def get_amp_row(row):
    """Extract 10-pulse amplitude vector; CORR<UNCORR → use UNCORR."""
    v = np.full(N_STIM, np.nan)
    for k in range(N_STIM):
        cc, uc = _amp_corr[k], _amp_uncorr[k]
        if cc not in row.index: continue
        vc = row[cc]
        if k == 0:
            v[k] = row[uc] if uc in row.index else vc
        else:
            if uc in row.index:
                vu = row[uc]
                if np.isfinite(vc) and np.isfinite(vu) and vc < vu:
                    vc = vu
            v[k] = vc
    return v

THR_SHARED_COL_MLE = 'thr_shared'

def get_thr_shared_value(row, threshold_col=THR_SHARED_COL_MLE):
    """Return the trial-wise shared threshold computed from the pre-train baseline."""
    if threshold_col not in row.index:
        return np.nan
    thr = pd.to_numeric(pd.Series([row[threshold_col]]), errors='coerce').iloc[0]
    return float(thr) if np.isfinite(thr) else np.nan

def get_failure_mask_row(row, threshold_col=THR_SHARED_COL_MLE):
    """Return per-event failures using the trial-wise thr_shared threshold."""
    fail_mask = np.full(N_STIM, np.nan)
    thr = get_thr_shared_value(row, threshold_col=threshold_col)
    if not np.isfinite(thr):
        return fail_mask, np.nan
    for k in range(N_STIM):
        cc, uc = _amp_corr[k], _amp_uncorr[k]
        vals = []
        if cc in row.index and np.isfinite(row[cc]):
            vals.append(float(row[cc]))
        if uc in row.index and np.isfinite(row[uc]):
            vals.append(float(row[uc]))
        if len(vals) == 0:
            continue
        is_failure = np.min(vals) < thr
        fail_mask[k] = 1.0 if is_failure else 0.0
    return fail_mask, thr

def get_count_row_with_failures(row, q_value, threshold_col=THR_SHARED_COL_MLE):
    """Convert amplitudes to quantal counts using thr_shared-defined failures."""
    counts = np.full(N_STIM, np.nan)
    amps = get_amp_row(row)
    fail_mask, _ = get_failure_mask_row(row, threshold_col=threshold_col)
    q_use = q_value if np.isfinite(q_value) and q_value > 0 else Global_Q
    for k in range(N_STIM):
        if not np.isfinite(amps[k]) or not np.isfinite(fail_mask[k]):
            continue
        if fail_mask[k] > 0:
            counts[k] = 0
        else:
            counts[k] = max(0, int(round(amps[k] / q_use)))
    return counts

def build_trial_dict(trials_df, conditions, min_trials=5, extra_conds=None):
    """bouton_id → list of amplitude vectors.  Optionally top-up from extra_conds."""
    td = defaultdict(list)
    for _, r in trials_df[trials_df['condition'].isin(conditions)].iterrows():
        td[strip_calcium(str(r['file']).strip())].append(get_amp_row(r))
    if extra_conds:
        for _, r in trials_df[trials_df['condition'].isin(extra_conds)].iterrows():
            bid = strip_calcium(str(r['file']).strip())
            if len(td[bid]) < min_trials:
                td[bid].append(get_amp_row(r))
    return td

# == Binomial MLE engine =========================================
def log_binom_pmf(k, N, P):
    k = np.asarray(k, float)
    return (gammaln(N+1) - gammaln(k+1) - gammaln(N-k+1)
            + k * (np.log(P) if P > 0 else -np.inf)
            + (N-k) * (np.log(1-P) if P < 1 else -np.inf))

def fit_binom_mle(counts, N_max=20):
    """MLE for Binomial(N,P).  Returns (N, P, loglik).

    Vectorised across all candidate N in [max(k_max, 1), N_max] :
    the inner Python loop is replaced by a single broadcast over N,
    which makes per-call cost dominated by one gammaln() evaluation.
    """
    counts = np.asarray(counts, int)
    if len(counts) < 3:
        return np.nan, np.nan, -np.inf
    k_max = int(counts.max())
    k_mean = float(counts.mean())
    if k_mean < 1e-10:
        return np.nan, np.nan, -np.inf
    N_lo = max(k_max, 1)
    if N_lo > N_max:
        return np.nan, np.nan, -np.inf
    Ns = np.arange(N_lo, N_max + 1, dtype=float)            # shape (M,)
    Ps = k_mean / Ns                                        # shape (M,)
    valid = (Ps > 0.0) & (Ps < 1.0)
    if not np.any(valid):
        return np.nan, np.nan, -np.inf
    Ns_v = Ns[valid][:, None]                               # (Mv, 1)
    Ps_v = Ps[valid][:, None]                               # (Mv, 1)
    k = counts.astype(float)[None, :]                       # (1, T)
    log_pmf = (gammaln(Ns_v + 1.0)
               - gammaln(k + 1.0)
               - gammaln(Ns_v - k + 1.0)
               + k * np.log(Ps_v)
               + (Ns_v - k) * np.log1p(-Ps_v))             # (Mv, T)
    ll = log_pmf.sum(axis=1)                                # (Mv,)
    finite = np.isfinite(ll)
    if not np.any(finite):
        return np.nan, np.nan, -np.inf
    ll[~finite] = -np.inf
    j = int(np.argmax(ll))
    return float(Ns_v[j, 0]), float(Ps_v[j, 0]), float(ll[j])

# == SN cumulative fit ============================================
def sn_cumulative(amps_matrix, n_fit_last=4):
    """Schneggenburger-Neher fit on (n_boutons × n_pulses) array.
    Returns dict with cum_mean, cum_sem, RRP, P0, slope, r2, n."""
    nb, npuls = amps_matrix.shape
    cum = np.cumsum(amps_matrix, axis=1)
    cm = np.nanmean(cum, axis=0)
    cs = np.nanstd(cum, axis=0) / np.sqrt(nb)
    px = np.arange(npuls)
    sl, ic, r, _, _ = linregress(px[-n_fit_last:], cm[-n_fit_last:])
    p0 = np.nanmean(amps_matrix[:,0]) / ic if ic > 0 else np.nan
    return dict(cum_mean=cm, cum_sem=cs, RRP=ic, P0=p0,
                slope=sl, r2=r**2, n=nb,
                x_fit=px, y_fit=sl*px+ic)

def cumulative_profiles_matrix(amps_matrix):
    """Return per-bouton cumulative-release curves from an amplitude matrix."""
    arr = np.asarray(amps_matrix, dtype=float)
    if arr.ndim != 2:
        raise ValueError('amps_matrix must be a 2D bouton-by-pulse array.')
    return np.cumsum(arr, axis=1)

def reconstruct_amps_qnorm(df, Q_dict, Q_fallback):
    """Summary dataframe → (n_boutons, num_pulses) Q-normalised amplitudes."""
    ids = df['ID'].astype(str).str.strip().values
    q = np.array([Q_dict.get(strip_calcium(b), Q_fallback) for b in ids])
    q[q <= 0] = Q_fallback
    a1 = df['AMP1'].values / q
    return np.column_stack([a1] + [df[c].values * a1 for c in ppr_cols])

def extract_base_name(bouton_id):
    """Extract base name (date + linescan/fibre + bouton) for pairing
    across conditions.  Handles BOTH naming conventions:

      20210721_linescan1_50Hz_10pulses_4mMCa_bouton1_traces_converted
        → 20210721_linescan1_bouton1

      250113_Fibre2_Bouton_4
        → 250113_Fibre2_Bouton_4   (unchanged : no Hz/Ca to strip)
    """
    bid = str(bouton_id).strip()
    bid = re.sub(r'\.xlsx?$', '', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_traces_converted$', '', bid)
    bid = re.sub(r'_\d+Hz_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+pulses_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+\.?\d*mMCa_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_\d+[_.]?\d*Ca_?', '_', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_set\d+', '', bid, flags=re.IGNORECASE)
    bid = re.sub(r'_+$', '', bid)
    bid = re.sub(r'_+', '_', bid)
    return bid

def find_valid_pairs(ids_1, ids_2, coords_1=None, coords_2=None):
    """Find paired boutons between two ID sets using normalized base names.

    Returns tuple: (indices_in_first, indices_in_second, base_names).
    """
    base_1 = {i: extract_base_name(bid) for i, bid in enumerate(ids_1)}
    base_2 = {i: extract_base_name(bid) for i, bid in enumerate(ids_2)}

    lookup_2 = {}
    for i, base in base_2.items():
        if base not in lookup_2:
            lookup_2[base] = i

    valid_1, valid_2, names = [], [], []
    for i1, base in base_1.items():
        if base in lookup_2:
            valid_1.append(i1)
            valid_2.append(lookup_2[base])
            names.append(base)

    return valid_1, valid_2, names


In [ ]:
# === Helper — Per-bouton Q estimates ===
# Estimate bouton-specific quantal size values from low-calcium success distributions.

USE_EMPIRICAL_BAYES = False
MIN_TRIALS_FOR_RAW  = 1

# == Condition→Ca mapping (SynII excluded) ========================
CA_1_5_CONDITIONS = get_calcium_conditions('1.5mM', 'all')
CA_4_CONDITIONS   = get_calcium_conditions('4mM', 'all')
CA_2_5_CONDITIONS = get_calcium_conditions('2.5mM', 'all')
ALL_Q_CONDITIONS = CA_1_5_CONDITIONS + CA_4_CONDITIONS + CA_2_5_CONDITIONS
ca_map = {**{c:'1.5Ca' for c in CA_1_5_CONDITIONS},
          **{c:'4Ca'   for c in CA_4_CONDITIONS},
          **{c:'2.5Ca' for c in CA_2_5_CONDITIONS}}

# == Load & filter trials =========================================
trials_for_q = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
trials_for_q = trials_for_q[trials_for_q['condition'].isin(ALL_Q_CONDITIONS)].copy()
trials_for_q['CaGroup'] = trials_for_q['condition'].map(ca_map)
trials_for_q['base_id'] = trials_for_q['file'].apply(strip_calcium)
trials_for_q['ID'] = trials_for_q['file']
status_norm = trials_for_q['status'].astype(str).str.strip().str.lower()
t15_succ = trials_for_q[(trials_for_q['CaGroup']=='1.5Ca') & (status_norm=='success')]

# == Raw Q per bouton (median of successes) =======================
Q_raw, Q_n_trials = {}, {}
for bid, grp in t15_succ.groupby('base_id'):
    amps = grp['AMP1_UNCORR'].dropna()
    if len(amps) >= MIN_TRIALS_FOR_RAW:
        # get the median excluding failures
        
        Q_raw[bid] = float(amps.median())
        Q_n_trials[bid] = len(amps)

# == Empirical Bayes shrinkage ====================================
if USE_EMPIRICAL_BAYES and len(Q_raw) > 2:
    q_raw_vals  = np.array(list(Q_raw.values()))
    n_trials_vals = np.array([Q_n_trials[b] for b in Q_raw])
    Q_global_raw = float(np.nanmean(q_raw_vals))

    # Pooled within-bouton variance
    pooled_ss, pooled_df = 0.0, 0
    for bid, grp in t15_succ.groupby('base_id'):
        a = grp['AMP1_UNCORR'].dropna()
        if len(a) >= 2:
            pooled_ss += float(np.sum((a - a.mean())**2))
            pooled_df += len(a) - 1
    sigma2_obs = pooled_ss / pooled_df if pooled_df > 0 else float(np.var(q_raw_vals))

    sigma2_total = float(np.nanvar(q_raw_vals))
    avg_noise    = float(np.mean(sigma2_obs / n_trials_vals))
    tau2         = max(sigma2_total - avg_noise, 1e-10)
    kappa        = sigma2_obs / tau2

    Q_estimates = {}
    for bid, qr in Q_raw.items():
        ni = Q_n_trials[bid]
        w  = ni / (ni + kappa)
        Q_estimates[bid] = w * qr + (1-w) * Q_global_raw
else:
    Q_estimates = Q_raw.copy()
    kappa = np.nan

average_Q = float(np.nanmean(list(Q_estimates.values())))
median_Q  = float(np.nanmedian(list(Q_estimates.values())))
Global_Q = median_Q
print(f"Q: {len(Q_estimates)} boutons | mean={average_Q:.4f} | median={median_Q:.4f} | Global_Q={Global_Q:.4f}"
      + (f" | κ={kappa:.2f}" if np.isfinite(kappa) else ""))

# == Build global trials_all with Ca/Q columns ====================
trials_all = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
trials_all = trials_all[trials_all['condition'].isin(ALL_Q_CONDITIONS)].copy()
trials_all['CaGroup']  = trials_all['condition'].map(ca_map)
trials_all['base_id']  = trials_all['file'].apply(strip_calcium)
trials_all['Q_bouton'] = trials_all['base_id'].map(Q_estimates).fillna(Global_Q)
trials_all['amp_norm'] = trials_all['AMP1_UNCORR'] / trials_all['Q_bouton']

# Enrich trial-level table once (avoid repeated recomputation downstream)
trials_all = add_condition_metadata(trials_all.rename(columns={'condition': 'Condition'}), condition_col='Condition').rename(columns={'Condition': 'condition'})
TRIALS_DATAFRAME = trials_all.copy()

# == FIX: Build Q_by_ebn HERE so norm_summary can use it =========
Q_by_ebn = {}
for k, v in Q_estimates.items():
    Q_by_ebn.setdefault(extract_base_name(k), v)

# == Normalize summary dataframes =================================
def norm_summary(df, paired):
    """Add AMP1_norm, AMP2_norm columns."""
    d = df.copy()
    d['base_id'] = d['ID'].apply(extract_base_name)
    if paired:
        q = d['base_id'].map(Q_by_ebn).fillna(Global_Q)
    else:
        q = pd.Series(Global_Q, index=d.index)
    d['Q'] = q; d['AMP1_norm'] = d['AMP1']/q; d['AMP2_norm'] = d['AMP2']/q
    return d

# Apply to all summary dataframes that exist
for vn, paired in [('PCA_Data_WT_Low_Ca',True),('PCA_Data_50Hz_1_5_Ca',True),
                    ('PCA_Data_WT_High_Ca',True),('PCA_Data_50Hz_4_Ca',True),
                    ('PCA_Data_WT_Pooled',False),('PCA_Data_WT_Theo',False),
                    ('PCA_Data_WT_Anthime',False),('PCA_Data_Stability_Before',False),
                    ('PCA_Data_Stability_After',False),('PCA_Data_50Hz_2_5_Ca',False)]:
    if vn in globals():
        globals()[vn] = norm_summary(globals()[vn], paired)


In [ ]:
# === Fig 1e — Quantal size estimate ===
# Plot the quantal-size distribution inferred from 1.5 mM success amplitudes and its fitted summary.

q_vals = np.array(list(Q_estimates.values()))
fig_a1, ax = make_figure_grid(figsize=(5, 3.5))
hist_bin_width = np.nan
mu = np.nan
std = np.nan
if q_vals.size:
    hist_weights = np.ones_like(q_vals, dtype=float) / q_vals.size
    _, hist_edges, _ = ax.hist(
        q_vals,
        bins=9,
        weights=hist_weights,
        color='steelblue',
        edgecolor='none',
        alpha=0.8,
    )
    if len(hist_edges) > 1:
        hist_bin_width = hist_edges[1] - hist_edges[0]
# Gaussian fit (exclude outliers > 0.75)
fit_data = q_vals[q_vals <= 0.75]
if len(fit_data) > 1:
    mu, std = norm.fit(fit_data)
    xf = np.linspace(0, q_vals.max()*1.2, 200)
    ax.plot(xf, norm.pdf(xf, mu, std) * hist_bin_width if np.isfinite(hist_bin_width) else norm.pdf(xf, mu, std), color='darkred', lw=1.5,
            label=f'Gauss μ={mu:.4f} σ={std:.4f}')
ax.vlines(median_Q, ymin=-0.01, ymax=ax.get_ylim()[1], color='orange', lw=1.2, ls='-.', label=f'med={median_Q:.4f}')
ax.plot(q_vals, -0.12*np.ones_like(q_vals), '|k', ms=6, mew=0.8)
ax.set_xlim(0, q_vals.max()*1.2)
ax.set_ylim(bottom=-0.01)
style_ax(ax, 'Quantal size Q (ΔF/F₀)', 'Probability',
         f'Quantal size estimate from 1.5 mM successes ({"EB" if USE_EMPIRICAL_BAYES else "raw"}) : n={len(q_vals)}')
add_legend(ax)
if 'style_hist_axis' in globals():
    style_hist_axis(ax)
fig_a1.tight_layout()
fig_a1.savefig(OUTPUT_DIR / 'Fig1_e_quantal_size_estimate.pdf', dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ Fig 1e quantal estimate: Q={median_Q:.4f} ΔF/F₀ | n={len(q_vals)} boutons')
if np.isfinite(mu) and np.isfinite(std):
    print(f'✓ Gaussian fit (Q ≤ 0.75): μ={mu:.4f}, σ={std:.4f}')


### WT reference PCA space

Release variables (AMP1, AMP2, PPR2/1–PPR10/1, Fail1, Fail2) are reduced onto a PCA space defined on WT pooled boutons alone, so that calcium, frequency, and SynII conditions can be projected without influencing the reference axes.

In [ ]:
# === Helper — PCA feature preparation ===
# Define the feature-preparation helper that selects numeric bouton metrics and standardizes them for PCA.
def prepare_pca_features(dataframe, reference_columns=None):
    """Return PCA-ready numeric features aligned to PCA_KEEP_COLS (or reference_columns)."""
    if reference_columns is None:
        keep = [col for col in PCA_KEEP_COLS if col in dataframe.columns]
        missing_keep = [col for col in PCA_KEEP_COLS if col not in dataframe.columns]
        if missing_keep:
            print(f'[warn] PCA_KEEP_COLS columns absent from dataset: {missing_keep}')
        return dataframe[keep].copy()

    missing_columns = [col for col in reference_columns if col not in dataframe.columns]
    if missing_columns:
        raise ValueError(
            f"{len(missing_columns)} PCA feature columns missing in dataset: {missing_columns}"
        )
    return dataframe[reference_columns].copy()


# Prepare reference dataset for PCA
WT_pooled_for_pca = prepare_pca_features(PCA_Data_WT_Pooled)
PCA_FEATURE_COLUMNS = list(WT_pooled_for_pca.columns)

forbidden_pca_cols = {'Ca_mM', 'Freq_Hz'}
forbidden_present = [col for col in PCA_FEATURE_COLUMNS if col in forbidden_pca_cols]
if forbidden_present:
    raise ValueError(f"Forbidden metadata columns included in PCA features: {forbidden_present}")

if len(PCA_FEATURE_COLUMNS) == 0:
    raise ValueError("No numeric PCA features found in PCA_Data_WT_Pooled after metadata filtering")

# Fit StandardScaler on WT_pooled reference dataset
scaler = StandardScaler()
scaled_data = {'WT_pooled': scaler.fit_transform(WT_pooled_for_pca)}

# Transform all other datasets using WT_pooled scaling parameters
datasets_to_scale = {
    'WT_Theo': PCA_Data_WT_Theo,
    'WT_Anthime': PCA_Data_WT_Anthime,
    'SynII': PCA_Data_SynII,
    'WT_1_5Ca': PCA_Data_WT_Low_Ca,
    'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before,
    'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca': PCA_Data_50Hz_1_5_Ca,
    '50Hz_4Ca': PCA_Data_50Hz_4_Ca,
    '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca,
}

for dataset_name, dataset_df in datasets_to_scale.items():
    pca_features = prepare_pca_features(dataset_df, reference_columns=PCA_FEATURE_COLUMNS)
    scaled_data[dataset_name] = scaler.transform(pca_features)

print(f"✓ PCA feature columns ({len(PCA_FEATURE_COLUMNS)}): {PCA_FEATURE_COLUMNS}")
print(f"✓ Removed metadata columns and non-numeric columns before scaling")
print(f"✓ Scaled {len(scaled_data)} datasets using WT_pooled reference parameters")


In [ ]:
# === Helper — Fit WT reference PCA ===

# Fit the WT pooled reference PCA and project all reused datasets into the same low-dimensional space.
pca                   = PCA(n_components=2)
pca_data              = {}
pca_data['WT_pooled'] = pca.fit_transform(scaled_data['WT_pooled'])

# Transform all other datasets using same PCA axes from WT_pooled
for dataset_name in datasets_to_scale.keys():
    pca_data[dataset_name] = pca.transform(scaled_data[dataset_name])

# Display PCA results summary
variance_pc1, variance_pc2 = pca.explained_variance_ratio_
print(f"✓ PCA transformation complete")
print(f"✓ PC1 explains {variance_pc1:.1%} of variance, PC2 explains {variance_pc2:.1%}")
print(f"✓ Total variance explained: {variance_pc1 + variance_pc2:.1%}")
print(f"✓ Transformed {len(pca_data)} datasets using WT_pooled PCA axes")


In [ ]:
# === Supp S7a / S7b — PCA variance summary ===
# Compute the full PCA spectrum to summarize explained variance and component retention.
pca_full = PCA(n_components=None)
pca_full.fit(scaled_data['WT_pooled'])

explained_variance = pca_full.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)
n_components = len(explained_variance)
component_idx = np.arange(1, n_components + 1)

fig, axes = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
ax_var, ax_cum = axes[0, 0], axes[0, 1]

ax_var.bar(component_idx, explained_variance, alpha=0.7, align='center', color='steelblue', label='Individual variance')
ax_var.set_xlabel('Number of principal components')
ax_var.set_ylabel('Explained variance')
ax_var.set_title('Elbow Plot: Individual variance')
ax_var.spines['top'].set_visible(False)
ax_var.spines['right'].set_visible(False)
add_legend(ax_var)
ax_var.grid(False)

ax_cum.plot(component_idx, cumulative_variance, 'r-', linewidth=2, marker='o', ms=4, label='Cumulative variance')
ax_cum.set_xlabel('Number of principal components')
ax_cum.set_ylabel('Explained variance')
ax_cum.set_title('Elbow Plot: Cumulative variance')
ax_cum.spines['top'].set_visible(False)
ax_cum.spines['right'].set_visible(False)
add_legend(ax_cum)
ax_cum.grid(False)

fig.tight_layout()
output_file = OUTPUT_DIR / "FigS7_ab_wt_pca_variance_summary.pdf"
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved Supp S7a/b PCA variance summary to {output_file}")
print(f"✓ Variance explained by PC1 + PC2: {cumulative_variance[1]:.1%}")
print(f"✓ Number of components shown: {n_components}")


In [ ]:
# === Helper — PCA coordinate tables ===

# Convert projected PCA coordinates into labeled dataframes for plotting and downstream joins.
principal_component_columns = ['PC1', 'PC2']
pca_dfs = {}

for dataset_name, dataset_coords in pca_data.items():
    pca_dfs[f'transformed_{dataset_name}'] = pd.DataFrame(dataset_coords, columns=principal_component_columns)

# Metadata-enriched PCA tables for downstream stats/plots
pca_source_map = {
    'WT_pooled': PCA_Data_WT_Pooled,
    'WT_Theo': PCA_Data_WT_Theo,
    'WT_Anthime': PCA_Data_WT_Anthime,
    'SynII': PCA_Data_SynII,
    'WT_1_5Ca': PCA_Data_WT_Low_Ca,
    'WT_4Ca': PCA_Data_WT_High_Ca,
    'stab_before': PCA_Data_Stability_Before,
    'stab_after': PCA_Data_Stability_After,
    '50Hz_1_5Ca': PCA_Data_50Hz_1_5_Ca,
    '50Hz_4Ca': PCA_Data_50Hz_4_Ca,
    '50Hz_2_5Ca': PCA_Data_50Hz_2_5_Ca,
}

PCA_METADATA_DFS = {}
for key, coord_df in pca_dfs.items():
    short_name = key.replace('transformed_', '')
    source_df = pca_source_map.get(short_name)
    if source_df is not None and len(source_df) == len(coord_df):
        PCA_METADATA_DFS[short_name] = pd.concat([
            coord_df.reset_index(drop=True),
            source_df.reset_index(drop=True)
        ], axis=1)

# Create PCA components table showing feature contributions to each PC
feature_names = list(WT_pooled_for_pca.columns)
df_components = pd.DataFrame(pca.components_, columns=feature_names, index=['PC1', 'PC2'])

# Display top feature contributors for each principal component
print("PCA Components Analysis (top 3 contributors per PC):")
print("-" * 50)
for pc_name in ['PC1', 'PC2']:
    top_contributing_features = df_components.loc[pc_name].abs().nlargest(3)
    feature_contributions = [f'{feature_name}({contribution:.3f})' for feature_name, contribution in top_contributing_features.items()]
    print(f"  {pc_name}: {', '.join(feature_contributions)}")

print(f"\n✓ Created {len(pca_dfs)} PCA coordinate DataFrames")
print(f"✓ Created {len(PCA_METADATA_DFS)} metadata-enriched PCA DataFrames")
print(f"✓ PCA components table shape: {df_components.shape}")


In [ ]:
# === Helper — PCA feature correlation table ===

# Combine PCA coordinates with original bouton metrics to support correlation and loading analyses.
combined_pca_FEATURES_DATAFRAME = pd.concat([
    pca_dfs['transformed_WT_pooled'].reset_index(drop=True),
    WT_pooled_for_pca.reset_index(drop=True)
], axis=1)

# Calculate correlation matrix between principal components and original features
full_correlation_matrix = combined_pca_FEATURES_DATAFRAME.corr(numeric_only=True)
pc_feature_correlations = full_correlation_matrix.iloc[:2, 2:]  # Extract PC1,PC2 vs features

# Display strongest correlations for interpretability
print("Strongest PC-Feature Correlations:")
print("-" * 40)
for pc_name in ['PC1', 'PC2']:
    strongest_correlations = pc_feature_correlations.loc[pc_name].abs().nlargest(3)
    correlation_strings = [
        f'{feature_name}({correlation_value:.3f})'
        for feature_name, correlation_value in strongest_correlations.items()
    ]
    print(f"  {pc_name}: {', '.join(correlation_strings)}")

# Manuscript-facing PC2 correlation summary.
# PCA sign is arbitrary, so store both the raw PC2 correlations and an oriented synaptic-weight axis.
amp_pc2_features = [c for c in ['AMP1', 'AMP2'] if c in pc_feature_correlations.columns]
fail_pc2_features = [c for c in ['%Fail1', '%Fail2', '%Fail3'] if c in pc_feature_correlations.columns]

def _pc2_group_summary(columns):
    vals = pd.to_numeric(pc_feature_correlations.loc['PC2', columns], errors='coerce').dropna()
    return {
        'features': list(vals.index),
        'r_values': vals.to_dict(),
        'mean_r': float(vals.mean()) if len(vals) else np.nan,
        'sd_r': float(vals.std(ddof=1)) if len(vals) > 1 else np.nan,
        'n_features': int(len(vals)),
    }

pc2_amp_summary = _pc2_group_summary(amp_pc2_features)
pc2_fail_summary = _pc2_group_summary(fail_pc2_features)
_pc2_orientation = 1.0
if np.isfinite(pc2_amp_summary['mean_r']) and pc2_amp_summary['mean_r'] < 0:
    _pc2_orientation = -1.0

def _orient_summary(summary, orientation):
    out = dict(summary)
    out['r_values'] = {k: float(v) * orientation for k, v in summary['r_values'].items()}
    out['mean_r'] = float(summary['mean_r']) * orientation if np.isfinite(summary['mean_r']) else np.nan
    out['sd_r'] = summary['sd_r']
    return out

PCA_PC2_MANUSCRIPT_STATS = {
    'raw_PC2': {
        'amplitudes': pc2_amp_summary,
        'failures': pc2_fail_summary,
    },
    'synaptic_weight_axis': {
        'orientation_multiplier': _pc2_orientation,
        'amplitudes': _orient_summary(pc2_amp_summary, _pc2_orientation),
        'failures': _orient_summary(pc2_fail_summary, _pc2_orientation),
    },
}

print("\nPC2 manuscript correlation summary:")
for group_name, summary in [('amplitudes AMP1/AMP2', pc2_amp_summary), ('failure rates %Fail1-%Fail3', pc2_fail_summary)]:
    detail = ', '.join(f'{k}={v:.3f}' for k, v in summary['r_values'].items())
    print(f"  raw PC2 {group_name}: mean r={summary['mean_r']:.3f} ± {summary['sd_r']:.3f}; {detail}")
if _pc2_orientation < 0:
    amp_or = PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['amplitudes']
    fail_or = PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['failures']
    print("  Synaptic-weight axis uses -PC2 so amplitudes are positive by convention:")
    print(f"    amplitudes: mean r={amp_or['mean_r']:.3f} ± {amp_or['sd_r']:.3f}")
    print(f"    failures:   mean r={fail_or['mean_r']:.3f} ± {fail_or['sd_r']:.3f}")

pd.DataFrame({
    'feature': amp_pc2_features + fail_pc2_features,
    'raw_PC2_r': [pc2_amp_summary['r_values'].get(c, np.nan) for c in amp_pc2_features] + [pc2_fail_summary['r_values'].get(c, np.nan) for c in fail_pc2_features],
    'synaptic_weight_axis_r': [PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['amplitudes']['r_values'].get(c, np.nan) for c in amp_pc2_features] + [PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['failures']['r_values'].get(c, np.nan) for c in fail_pc2_features],
}).to_excel(OUTPUT_DIR / 'FigS8_pc2_feature_correlations_for_manuscript.xlsx', index=False)

# Store comprehensive PCA results for downstream analysis
PCA_RESULTS = {
    'pca_model': pca,                           # Fitted PCA transformer
    'scaler': scaler,                           # Fitted StandardScaler
    'pca_dataframes': pca_dfs,                  # PCA coordinates for all datasets
    'components': df_components,                # Feature contributions to PCs
    'correlations': pc_feature_correlations,    # PC-feature correlation matrix
    'explained_variance': pca.explained_variance_ratio_,  # Variance explained by each PC
    'pc2_manuscript_stats': PCA_PC2_MANUSCRIPT_STATS,
}

print(f"\n✓ Correlation analysis complete")
print(f"✓ PCA results stored in PCA_RESULTS dictionary with {len(PCA_RESULTS)} components")

In [ ]:
# === Helper — PCA smoothing and grid utilities ===
# Define smoothing, grid-building, and scalar-field helpers for PCA-space heatmaps and contours.
SMOOTH_SIGMA_FACTOR = 2.0
SMOOTH_SUPPORT_RADIUS = 3.0
SMOOTH_CLIP_PCT = 5
GRID_RES = 80

def smooth_field(xy, vals, mask, gx, gy):
    """Gaussian-kernel smoothing with support masking."""
    pts, v = xy[mask], vals[mask]
    Xg, Yg = np.meshgrid(gx, gy)
    if len(pts) < 3:
        return np.full(Xg.shape, np.nan), np.nan
    if SMOOTH_ROBUST:
        lo, hi = np.nanpercentile(v, [SMOOTH_CLIP_PCT, 100 - SMOOTH_CLIP_PCT])
        v = np.clip(v, lo, hi)
    d = cdist(pts, pts)
    np.fill_diagonal(d, np.inf)
    sig = np.median(np.min(d, axis=1)) * SMOOTH_SIGMA_FACTOR
    gp = np.column_stack([Xg.ravel(), Yg.ravel()])
    dist_gp = cdist(gp, pts)
    w = np.exp(-0.5 * (dist_gp / sig) ** 2)
    ws = w.sum(1)
    sg = np.full(len(gp), np.nan)
    ok = ws > 1e-12
    support = np.min(dist_gp, axis=1) <= (SMOOTH_SUPPORT_RADIUS * sig)
    ok = ok & support
    sg[ok] = (w[ok] * v).sum(1) / ws[ok]
    return sg.reshape(Xg.shape), sig

def setup_pca_grid(coords, pad=0.5, at_least_reference=True):
    coords = np.asarray(coords, float)
    xmn, xmx = coords[:, 0].min() - pad, coords[:, 0].max() + pad
    ymn, ymx = coords[:, 1].min() - pad, coords[:, 1].max() + pad
    if at_least_reference and 'get_reference_pca_limits' in globals():
        ref_xlim, ref_ylim = get_reference_pca_limits(pad=pad)
        xmn = min(float(xmn), float(ref_xlim[0]))
        xmx = max(float(xmx), float(ref_xlim[1]))
        ymn = min(float(ymn), float(ref_ylim[0]))
        ymx = max(float(ymx), float(ref_ylim[1]))
    gx = np.linspace(xmn, xmx, GRID_RES)
    gy = np.linspace(ymn, ymx, GRID_RES)
    ext = [xmn, xmx, ymn, ymx]
    return gx, gy, ext


In [ ]:
# === Aux — WT parameter maps on PCA space ===
# Render auxiliary PCA-space maps showing how selected WT parameters vary across the reference manifold.
if 'scaled_data' not in globals():
    raise RuntimeError('Run the PCA preparation cells first.')

wt_coords = pca_data['WT_pooled']
wt_feature_names = list(WT_pooled_for_pca.columns)
wt_scaled = np.asarray(scaled_data['WT_pooled'], float)

n_features = len(wt_feature_names)
ncols = 4
nrows = int(np.ceil(n_features / ncols))

fig, axes = make_figure_grid(nrows, ncols, panel_kind='pca', squeeze=False)
axes_flat = axes.ravel()

for i, feature_name in enumerate(wt_feature_names):
    ax = axes_flat[i]

    feature_idx = wt_feature_names.index(feature_name)
    feature_vals = wt_scaled[:, feature_idx]
    ok = np.isfinite(feature_vals)

    if ok.sum() < 3:
        ax.text(0.5, 0.5, f'{feature_name}\nnot enough data', ha='center', va='center',
                transform=ax.transAxes, fontsize=5)
        style_pca_axes(ax, title=feature_name, legend=False)
        continue

    gx, gy, ext = setup_pca_grid(wt_coords[ok], pad=0.5)
    sg, _ = smooth_field(wt_coords[ok], feature_vals[ok], np.ones(ok.sum(), dtype=bool), gx, gy)

    vmax = np.nanmax(np.abs(feature_vals[ok]))
    vmin = -vmax

    ax.imshow(
        sg,
        extent=ext,
        origin='lower',
        aspect='equal',
        cmap='coolwarm',
        vmin=vmin,
        vmax=vmax,
        interpolation='bilinear',
        alpha=0.75,
    )

    plot_pca_value_overlay(
        ax,
        wt_coords[ok],
        np.ones(ok.sum(), dtype=bool),
        feature_vals[ok],
        cmap='coolwarm',
        vmin=vmin,
        vmax=vmax,
        s=16,
        edgecolors='none',
        zorder=3,
    )

    sm = plt.cm.ScalarMappable(
        cmap='coolwarm',
        norm=plt.Normalize(vmin, vmax)
    )
    plt.colorbar(sm, ax=ax, shrink=0.75)

    style_pca_axes(ax, title=feature_name, legend=False)
    ax.set_aspect('equal', adjustable='box')

for j in range(n_features, len(axes_flat)):
    axes_flat[j].axis('off')

fig.suptitle('WT PCA maps of standardized parameter values', fontsize=8, fontweight='bold')
fig.tight_layout(rect=[0, 0, 0.96, 0.95], pad=0.12, w_pad=0.08, h_pad=0.08)

output_file = OUTPUT_DIR / 'Aux_wt_parameter_value_maps.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"Saved to {output_file}")
print(f"Mapped parameters: {', '.join(wt_feature_names)}")


#### WT reference check

Auxiliary panels verify that the pooled WT reference space is not dominated by a single acquisition subset and that the principal axes reflect genuine physiological variation.

In [ ]:
# === Supp S4 — Basal fluorescence control from workbook F0 sheets ===
# Test whether basal fluorescence (proxy for iGluSnFR expression level) explains A1 or PPR2/1.
if 'FEATURES_DATAFRAME_RAW' not in globals():
    raise RuntimeError('Run the feature-table loading cells first.')
from pathlib import Path as _Path_F0
F0_SOURCE_CONDITIONS = WT_2_5_20HZ_CONDITIONS
def _read_workbook_f0_mean(condition_name, bouton_id):
    workbook_path = BASE_DIR / str(condition_name) / f'{str(bouton_id).strip()}.xlsx'
    if not workbook_path.exists():
        return np.nan, 'missing workbook'
    try:
        f0_df = pd.read_excel(workbook_path, sheet_name='F0')
    except Exception as exc:
        return np.nan, f'F0 read failed: {exc}'
    numeric_values = pd.to_numeric(f0_df.to_numpy().ravel(), errors='coerce')
    numeric_values = numeric_values[np.isfinite(numeric_values)]
    if numeric_values.size == 0:
        return np.nan, 'empty F0 sheet'
    return float(np.nanmedian(numeric_values)), 'ok'
def _linear_fit_with_ci(x, y, confidence=0.95):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.size < 3 or np.allclose(x, x[0]):
        return None
    slope, intercept = np.polyfit(x, y, deg=1)
    xs = np.linspace(x.min(), x.max(), 200)
    y_fit = slope * xs + intercept
    residuals = y - (slope * x + intercept)
    dof = x.size - 2
    if dof <= 0:
        return {'xs': xs, 'y_fit': y_fit, 'lower': None, 'upper': None}
    ssx = np.sum((x - x.mean()) ** 2)
    if ssx <= 0:
        return {'xs': xs, 'y_fit': y_fit, 'lower': None, 'upper': None}
    mse = np.sum(residuals ** 2) / dof
    alpha = 1.0 - confidence
    tcrit = stats.t.ppf(1.0 - alpha / 2.0, dof)
    se_fit = np.sqrt(mse * (1.0 / x.size + ((xs - x.mean()) ** 2) / ssx))
    return {
        'xs': xs,
        'y_fit': y_fit,
        'lower': y_fit - tcrit * se_fit,
        'upper': y_fit + tcrit * se_fit,
    }
f0_rows = []
for _, row in FEATURES_DATAFRAME_RAW[FEATURES_DATAFRAME_RAW['Condition'].isin(F0_SOURCE_CONDITIONS)].iterrows():
    condition_name = str(row['Condition']).strip()
    bouton_id = str(row['ID']).strip()
    f0_value, f0_status = _read_workbook_f0_mean(condition_name, bouton_id)
    f0_rows.append({
        'Condition': condition_name,
        'ID': bouton_id,
        'BaseID': _normalize_bouton_id(bouton_id),
        'F0': f0_value,
        'F0_status': f0_status,
    })
F0_REFERENCE_TABLE = pd.DataFrame(f0_rows)
if not F0_REFERENCE_TABLE.empty:
    F0_REFERENCE_TABLE = F0_REFERENCE_TABLE.drop_duplicates('BaseID', keep='first')
sup4_df = PCA_Data_WT_Pooled.copy().reset_index(drop=True)
sup4_df['BaseID'] = sup4_df['ID'].map(_normalize_bouton_id)
sup4_df = sup4_df.merge(F0_REFERENCE_TABLE[['BaseID', 'F0', 'F0_status']], on='BaseID', how='left')
SUPP_S4_F0_STATS = {}
for metric_name in ['AMP1', 'PPR2/1']:
    valid = sup4_df[['F0', metric_name]].apply(pd.to_numeric, errors='coerce').dropna()
    if len(valid) >= 3:
        pearson_r, pearson_p = stats.pearsonr(valid['F0'], valid[metric_name])
        spearman_r, spearman_p = stats.spearmanr(valid['F0'], valid[metric_name])
    else:
        pearson_r = pearson_p = spearman_r = spearman_p = np.nan
    SUPP_S4_F0_STATS[metric_name] = {
        'n': int(len(valid)),
        'pearson_r': float(pearson_r) if np.isfinite(pearson_r) else np.nan,
        'pearson_p': float(pearson_p) if np.isfinite(pearson_p) else np.nan,
        'spearman_r': float(spearman_r) if np.isfinite(spearman_r) else np.nan,
        'spearman_p': float(spearman_p) if np.isfinite(spearman_p) else np.nan,
    }
fig, axes = make_figure_grid(1, 2, figsize=(7.0, 3.2))
for ax, metric_name, ylabel in zip(axes, ['AMP1', 'PPR2/1'], ['A1 ΔF/F0', 'PPR2/1']):
    plot_df = sup4_df[['F0', metric_name]].apply(pd.to_numeric, errors='coerce').dropna()
    ax.scatter(plot_df['F0'], plot_df[metric_name], s=18, color=get_wt_ca_color('2.5mM'), alpha=0.72, edgecolors='none')
    fit_result = _linear_fit_with_ci(plot_df['F0'], plot_df[metric_name])
    if fit_result is not None:
        if fit_result['lower'] is not None and fit_result['upper'] is not None:
            ax.fill_between(
                fit_result['xs'],
                fit_result['lower'],
                fit_result['upper'],
                color='black',
                alpha=0.12,
                linewidth=0,
                zorder=1,
            )
        ax.plot(
            fit_result['xs'],
            fit_result['y_fit'],
            color='black',
            lw=1.1,
            ls=':',
            zorder=2,
        )
    stat = SUPP_S4_F0_STATS[metric_name]
    ax.text(
        0.02, 0.98,
        f"r={stat['pearson_r']:.3f}\np={stat['pearson_p']:.3g}\nn={stat['n']}",
        transform=ax.transAxes,
        ha='left', va='top', fontsize=6,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.78, pad=1.5),
    )
    style_ax(ax, 'Basal fluorescence F0', ylabel, None)
fig.tight_layout()
sup4_pdf = OUTPUT_DIR / 'FigS4_basal_fluorescence_vs_a1_ppr.pdf'
fig.savefig(sup4_pdf, dpi=300, bbox_inches='tight')
plt.show()
sup4_stats_df = pd.DataFrame([
    {'metric': metric_name, **stat_values}
    for metric_name, stat_values in SUPP_S4_F0_STATS.items()
])
sup4_stats_xlsx = OUTPUT_DIR / 'FigS4_basal_fluorescence_correlation_statistics.xlsx'
sup4_stats_df.to_excel(sup4_stats_xlsx, index=False)
print('Supp S4 basal fluorescence control:')
print(f"  F0 sheets found: {(F0_REFERENCE_TABLE['F0_status'] == 'ok').sum()} / {len(F0_REFERENCE_TABLE)} WT source boutons")
for metric_name, stat in SUPP_S4_F0_STATS.items():
    print(f"  F0 vs {metric_name}: Pearson r={stat['pearson_r']:.3f}, p={stat['pearson_p']:.4g}, n={stat['n']}; Spearman rho={stat['spearman_r']:.3f}, p={stat['spearman_p']:.4g}")
print(f'✓ Saved Supp S4 panel to {sup4_pdf}')
print(f'✓ Saved Supp S4 statistics to {sup4_stats_xlsx}')


In [ ]:
# _failure_traces = show_trial_overlays(
#     conditions=WT_2_5_20HZ_CONDITIONS,
#     classes=['1Q'],
#     xlim=A1_TRACE_XLIM,
#     ylim=(-1.0, 2.0),
#     q_value=Global_Q,
#     savgol=True,
#     sg_window=19,
#     sg_poly=6,
#     plot_figures=True,
# )



In [ ]:
# === Aux — WT reference parameter boxplots ===
# Compare selected WT pooled scalar parameters with paired statistical summaries as an auxiliary reference check.
from scipy.stats import wilcoxon
fig, axes = make_figure_grid(1, 3, panel_kind='boxplot_compact')

# Prepare data for comparison
wt_data = PCA_Data_WT_Pooled.copy()

def paired_wilcoxon(df, col_a, col_b):
    sub = df[[col_a, col_b]].replace([np.inf, -np.inf], np.nan).dropna()
    n = len(sub)
    if n < 3:
        return {'n': n, 'W': np.nan, 'p': np.nan, 'mean_diff': np.nan, 'median_diff': np.nan}
    d = sub[col_b] - sub[col_a]
    try:
        W, p = wilcoxon(sub[col_a], sub[col_b], alternative='two-sided', zero_method='wilcox')
    except ValueError:
        W, p = np.nan, np.nan
    return {
        'n': n,
        'W': W,
        'p': p,
        'mean_diff': d.mean(),
        'median_diff': d.median()
    }

# Plot 1: AMP1 vs AMP2
data_amp = pd.concat([
    pd.DataFrame({'Value': wt_data['AMP1'], 'Amplitude': 'AMP1'}),
    pd.DataFrame({'Value': wt_data['AMP2'], 'Amplitude': 'AMP2'})
])
plot_boxplot_analysis(axes[0], [make_box_group('AMP1', values=wt_data['AMP1'], color='lightgray'), make_box_group('AMP2', values=wt_data['AMP2'], color='gray')], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='AMP1 vs AMP2', ylabel='Amplitude (ΔF/F)')

amp_stats = paired_wilcoxon(wt_data, 'AMP1', 'AMP2')

# Plot 2: %Fail1 vs %Fail2
fail_stats = None
if '%Fail1' in wt_data.columns and '%Fail2' in wt_data.columns:
    data_fail = pd.concat([
        pd.DataFrame({'Value': wt_data['%Fail1'], 'Failure Rate': '%Fail1'}),
        pd.DataFrame({'Value': wt_data['%Fail2'], 'Failure Rate': '%Fail2'})
    ])
    plot_boxplot_analysis(axes[1], [make_box_group('%Fail1', values=wt_data['%Fail1'], color='lightgray'), make_box_group('%Fail2', values=wt_data['%Fail2'], color='gray')], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='%Fail1 vs %Fail2', ylabel='Failure Rate (%)')

    fail_stats = paired_wilcoxon(wt_data, '%Fail1', '%Fail2')
else:
    axes[1].axis('off')
    axes[1].text(0.5, 0.5, 'Missing %Fail1/%Fail2 columns', ha='center', va='center')

# Plot 3: PPR2/1 vs PPR3/1
data_ppr = pd.concat([
    pd.DataFrame({'Value': wt_data['PPR2/1'], 'PPR': 'PPR2/1'}),
    pd.DataFrame({'Value': wt_data['PPR3/1'], 'PPR': 'PPR3/1'})
])
plot_boxplot_analysis(axes[2], [make_box_group('PPR2/1', values=wt_data['PPR2/1'], color='lightgray'), make_box_group('PPR3/1', values=wt_data['PPR3/1'], color='gray')], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='PPR2/1 vs PPR3/1', ylabel='PPR (A_n/A_1)')
axes[2].axhline(1.0, color='gray', linestyle='--', alpha=0.7, linewidth=1)
axes[2].set_ylim(bottom=0)

ppr_stats = paired_wilcoxon(wt_data, 'PPR2/1', 'PPR3/1')

plt.tight_layout()
plt.suptitle('WT Pooled Parameter Comparisons', fontsize=8, fontweight='bold', y=1.02)

output_file = OUTPUT_DIR / "Aux_wt_reference_parameter_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved parameter comparison boxplots to {output_file}")

# Print summary statistics
print("\n=== WT Pooled Parameter Statistics ===")
print(f"AMP1: mean={wt_data['AMP1'].mean():.4f}, median={wt_data['AMP1'].median():.4f}")
print(f"AMP2: mean={wt_data['AMP2'].mean():.4f}, median={wt_data['AMP2'].median():.4f}")
if '%Fail1' in wt_data.columns and '%Fail2' in wt_data.columns:
    print(f"%Fail1: mean={wt_data['%Fail1'].mean():.4f}, median={wt_data['%Fail1'].median():.4f}")
    print(f"%Fail2: mean={wt_data['%Fail2'].mean():.4f}, median={wt_data['%Fail2'].median():.4f}")
print(f"PPR2/1: mean={wt_data['PPR2/1'].mean():.4f}, median={wt_data['PPR2/1'].median():.4f}")
print(f"PPR3/1: mean={wt_data['PPR3/1'].mean():.4f}, median={wt_data['PPR3/1'].median():.4f}")

# Paired stats report
print("\n=== Paired Statistics (Wilcoxon signed-rank) ===")
print(f"AMP2-AMP1: n={amp_stats['n']}, W={amp_stats['W']:.3g}, p={amp_stats['p']:.3g}, "
      f"meanΔ={amp_stats['mean_diff']:.4f}, medianΔ={amp_stats['median_diff']:.4f}")
if fail_stats is not None:
    print(f"%Fail2-%Fail1: n={fail_stats['n']}, W={fail_stats['W']:.3g}, p={fail_stats['p']:.3g}, "
          f"meanΔ={fail_stats['mean_diff']:.4f}, medianΔ={fail_stats['median_diff']:.4f}")
print(f"PPR3/1-PPR2/1: n={ppr_stats['n']}, W={ppr_stats['W']:.3g}, p={ppr_stats['p']:.3g}, "
      f"meanΔ={ppr_stats['mean_diff']:.4f}, medianΔ={ppr_stats['median_diff']:.4f}")


### WT clustering and release phenotypes

Hierarchical clustering in the WT PCA space separates boutons with distinct combinations of amplitude, reliability, and train dynamics, defining the release classes characterized in subsequent figures.

In [ ]:
# === Helper — WT hierarchical clustering ===

# Perform hierarchical clustering on the WT pooled PCA coordinates to define the reference bouton classes.
pca_coordinates = pca_data['WT_pooled']  # Shape: (n_samples, 2)

# Perform hierarchical clustering using Ward linkage method
linkage_matrix      = linkage(pca_coordinates, method='ward')
cluster_assignments_raw = fcluster(linkage_matrix, N_CLUSTERS, criterion='maxclust')

# Compute mean AMP1 for each cluster and reorder labels accordingly
amp1_values = PCA_Data_WT_Pooled['AMP1'].values
cluster_amp1_means = {}
for cid in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments_raw == cid
    cluster_amp1_means[cid] = np.nanmean(amp1_values[mask])

# Sort clusters by ascending mean AMP1
sorted_clusters = sorted(cluster_amp1_means.keys(), key=lambda c: cluster_amp1_means[c])
old_to_new = {old: new for new, old in enumerate(sorted_clusters, start=1)}
cluster_assignments = np.array([old_to_new[c] for c in cluster_assignments_raw])

# Add cluster labels to original dataframe
PCA_Data_WT_Pooled_clustered               = PCA_Data_WT_Pooled.copy()
PCA_Data_WT_Pooled_clustered['HC_Cluster'] = cluster_assignments
# Save clustered data to Excel
clustered_output_file = OUTPUT_DIR / f"PCA_Data_WT_Pooled_clustered_{N_CLUSTERS}.xlsx"
PCA_Data_WT_Pooled_clustered.to_excel(clustered_output_file, index=False)
print(f"✓ Saved clustered data to {clustered_output_file}")


# Visualize clusters in PCA space using plot_pca_nice
pc1_variance = PCA_RESULTS["explained_variance"][0]
pc2_variance = PCA_RESULTS["explained_variance"][1]

# Generate consistent Set1 colors for clusters
set1_colors          = Set1(np.linspace(0, 1, N_CLUSTERS))
cluster_rgba_colors  = [tuple(color) for color in set1_colors]
cluster_hex_colors   = [to_hex(color) for color in cluster_rgba_colors]
cluster_palette      = ListedColormap(cluster_rgba_colors, name='cluster_palette')
cluster_color_lookup = {cid: cluster_rgba_colors[cid - 1] for cid in range(1, N_CLUSTERS + 1)}

if 'get_cluster_color' not in globals() or 'style_pca_axes' not in globals() or 'plot_pca_background' not in globals():
    raise RuntimeError('Run the main shared helper cell near the beginning of the notebook first.')



In [ ]:
# === Helper — WT clustering outlier sanity check ===
# Verify that the excluded high-PC1 WT outliers remain absent from the clustering input as intended.
high_pc1_ids = []
for cluster_id in range(1, N_CLUSTERS + 1):
    mask = cluster_assignments == cluster_id
    high_pc1_mask = (pca_coordinates[mask, 0] > HIGH_PC1_THRESHOLD)
    cluster_indices = np.where(mask)[0]
    high_pc1_ids.extend(PCA_Data_WT_Pooled.iloc[cluster_indices[high_pc1_mask]]['ID'].tolist())
print(f"PC1 > {HIGH_PC1_THRESHOLD} sanity check: {len(high_pc1_ids)} remaining ID(s) "
      f"(should be 0 after upstream exclusion of {len(OUTLIER_IDS_HIGH_PC1)} bouton(s)).")
for _id in high_pc1_ids:
    print(f"  STILL PRESENT: {_id}")


### Supplementary Figure S2 - NNLS dataset construction workflow

Detailed schematic of the processing path used to convert trial trains into per-pulse amplitudes: train preprocessing, upsampled event recuts, kinetic/template fitting, SavGol-smoothed tau progression, constrained NNLS refit, residual checks, and final peak/amplitude extraction.

In [ ]:
# === Supplementary Figure S2 - how the NNLS / template pipeline builds the dataset (real data) ===
# Method figure: each panel answers "what we do" and "why we do it".
# Workflow: noisy trials -> bouton template from recuts -> kinetics that slow smoothly
#           across the train -> templates fixed at stimulus times -> non-negative solve
#           (deconvolution) -> per-event amplitudes.

from pathlib import Path
from scipy.optimize import curve_fit, nnls
from scipy.signal import savgol_filter
import numpy as np
import matplotlib.pyplot as plt

# Save into OUTPUT_DIR (defined by the setup cells), exactly like every other figure cell.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGS2_OUTPUT_PDF = OUTPUT_DIR / 'FigS2_NNLS_dataset_construction_workflow.pdf'
FIGS2_OUTPUT_PNG = OUTPUT_DIR / 'FigS2_NNLS_dataset_construction_workflow.png'

_repo_root = Path(globals().get('REPO_ROOT', Path.cwd()))
_overlay_candidates = [
    _repo_root / '_tmp_nnls_overlay__WT_Anthime__241212_Fibre1_PortionD_bouton4__trial1.npz',
    _repo_root / '_tmp_large2p5_overlay__WT_Theo__20210128_linescan1_20Hz_10pulses_2_5mMCa_bouton1_traces_converted__trial1.npz',
    _repo_root / '_tmp_nnls_overlay__Theo_1_5Ca__20200909_linescan1_20Hz_10pulses_1_5mMCa_bouton4_traces_converted.npz',
]
if 'FIGS2_METHOD_NPZ' in globals():
    _overlay_candidates.insert(0, Path(FIGS2_METHOD_NPZ))

FIGS2_METHOD_NPZ = next((p for p in _overlay_candidates if Path(p).exists()), None)
if FIGS2_METHOD_NPZ is None:
    raise FileNotFoundError('No cached real NNLS overlay NPZ found for Fig. S2 method panel.')

_overlay = np.load(FIGS2_METHOD_NPZ, allow_pickle=True)
_required_keys = {
    'time_s', 'train_start_s', 'isi_s', 'all_trial_yproc', 'all_trial_yhat',
    'average_yproc', 'average_yhat', 'average_amp_nnls_corr',
}
_missing_keys = _required_keys - set(_overlay.files)
if _missing_keys:
    raise KeyError(f'{FIGS2_METHOD_NPZ} is missing keys required for the real-data S2 figure: {sorted(_missing_keys)}')

time_s = np.asarray(_overlay['time_s'], float).reshape(-1)
train_start_s = float(np.asarray(_overlay['train_start_s']).item())
isi_s = float(np.asarray(_overlay['isi_s']).item())
all_trial_yproc = np.asarray(_overlay['all_trial_yproc'], float)
average_yproc = np.asarray(_overlay['average_yproc'], float).reshape(-1)
average_yhat = np.asarray(_overlay['average_yhat'], float).reshape(-1)
average_amp_nnls_corr = np.asarray(_overlay['average_amp_nnls_corr'], float).reshape(-1)
average_ppr_nnls_corr = (
    np.asarray(_overlay['average_ppr_nnls_corr'], float).reshape(-1)
    if 'average_ppr_nnls_corr' in _overlay.files else None
)
condition_label = str(np.asarray(_overlay['condition']).item()) if 'condition' in _overlay.files else 'real data'
file_stem = str(np.asarray(_overlay['file_stem']).item()) if 'file_stem' in _overlay.files else Path(FIGS2_METHOD_NPZ).stem

if all_trial_yproc.ndim != 2:
    raise ValueError(f'all_trial_yproc must be 2D trial x time; got shape {all_trial_yproc.shape}')
if all_trial_yproc.shape[-1] != time_s.size:
    raise ValueError(f'Time/trial shape mismatch: time has {time_s.size}, trials have {all_trial_yproc.shape[-1]}')

n_trials = int(all_trial_yproc.shape[0])
n_pulses = int(min(10, average_amp_nnls_corr.size))
stim_times_s = train_start_s + isi_s * np.arange(n_pulses)
dt_s = float(np.nanmedian(np.diff(time_s)))
sg_window, sg_poly = 9, 2
peak_window_s = min(0.010, 0.45 * isi_s)
recut_pre_s = 0.006
recut_post_s = min(0.040, max(0.008, isi_s - 0.006))
recut_oversample = 20
plot_xlim_s = (max(time_s[0], train_start_s - 0.025), min(time_s[-1], stim_times_s[-1] + isi_s + 0.040))

# --- Baseline subtraction (pre-train) ---
baseline_mask = time_s < (train_start_s - 0.005)
if baseline_mask.sum() < 3:
    baseline_mask = time_s < train_start_s

def _baseline_subtract_matrix(y2d):
    y2d = np.asarray(y2d, float)
    baseline = np.nanmedian(y2d[:, baseline_mask], axis=1)
    return y2d - baseline[:, None]

all_proc = _baseline_subtract_matrix(all_trial_yproc)
average_baseline = float(np.nanmedian(average_yproc[baseline_mask])) if np.any(baseline_mask) else 0.0
average_proc = average_yproc - average_baseline

_sg_window = min(sg_window, average_proc.size if average_proc.size % 2 else average_proc.size - 1)
average_sg = (average_proc.copy() if _sg_window < 5
              else savgol_filter(average_proc, _sg_window, min(sg_poly, _sg_window - 1), mode='interp'))

# =============================================================================
# Real upsampled recuts: project every (trial x pulse) post-stimulus snippet
# onto a fine grid using its exact timing, then median-average.
# =============================================================================
recut_time_s = np.arange(-recut_pre_s, recut_post_s + dt_s / recut_oversample, dt_s / recut_oversample)
recut_stack, recut_by_pulse = [], []
for stim_time in stim_times_s:
    snippets = []
    target_time = stim_time + recut_time_s
    for trial_idx in range(n_trials):
        snippet = np.interp(target_time, time_s, all_proc[trial_idx], left=np.nan, right=np.nan)
        pre_mask = recut_time_s < -0.001
        if np.any(pre_mask):
            snippet = snippet - np.nanmedian(snippet[pre_mask])
        snippets.append(snippet)
        recut_stack.append(snippet)
    recut_by_pulse.append(np.nanmedian(np.vstack(snippets), axis=0))
recut_stack = np.vstack(recut_stack)
recut_by_pulse = np.vstack(recut_by_pulse)
recut_median_raw = np.nanmedian(recut_stack, axis=0)
recut_scale = float(np.nanmax(recut_median_raw)) if np.nanmax(recut_median_raw) > 0 else 1.0
recut_median = recut_median_raw / recut_scale
recut_by_pulse_norm = recut_by_pulse / recut_scale
recut_stack_norm = recut_stack / recut_scale

# =============================================================================
# iGluSnFR template (matches Model_Calibration/event_models.model_iglusnfr_tri):
# common rise time, fast + slow + super-slow decay; fit rise AND onset so the
# template lands on the real peak (no post-hoc cross-correlation shift).
# =============================================================================
TAU_SUPER_MS = 65.0  # super-slow constant (from late-train / last-event decay)

def _iglusnfr_tri_kernel(t_rel_s, tau_rise_ms, tau_fast_ms, tau_slow_ms,
                         frac_fast, frac_slow, t_onset_ms=0.0,
                         tau_super_ms=TAU_SUPER_MS, normalize=True):
    x_ms = np.asarray(t_rel_s, float) * 1000.0 - float(t_onset_ms)
    pos = x_ms > 0.0
    xp = np.where(pos, x_ms, 0.0)
    rise = 1.0 - np.exp(-xp / max(tau_rise_ms, 1e-6))
    ff = float(np.clip(frac_fast, 0.0, 1.0))
    fs = float(np.clip(frac_slow, 0.0, 1.0 - ff))
    fss = max(0.0, 1.0 - ff - fs)
    fast = np.where(pos, rise * ff * np.exp(-xp / max(tau_fast_ms, 1e-6)), 0.0)
    slow = np.where(pos, rise * fs * np.exp(-xp / max(tau_slow_ms, 1e-6)), 0.0)
    super_slow = np.where(pos, rise * fss * np.exp(-xp / max(tau_super_ms, 1e-6)), 0.0)
    total = fast + slow + super_slow
    if normalize and np.nanmax(total) > 0:
        s = np.nanmax(total)
        return total / s, fast / s, slow / s, super_slow / s
    return total, fast, slow, super_slow

def _template_total(t_rel_s, tau_rise_ms, tau_fast_ms, tau_slow_ms, frac_fast, frac_slow, t_onset_ms):
    return _iglusnfr_tri_kernel(t_rel_s, tau_rise_ms, tau_fast_ms, tau_slow_ms,
                                frac_fast, frac_slow, t_onset_ms)[0]

# Bounds: tau_rise 0.5-8 ms, fast 3-10 ms, slow 10-35 ms, onset 0-6 ms. Rise and onset
# are free so the template peak lands on the real peak for both fast- and slow-rising
# boutons; frac_fast capped < 1 so the slow/super-slow components stay identifiable.
_p0 = [3.0, 5.0, 18.0, 0.55, 0.30, 1.5]
_lo = [0.5, 3.0, 10.0, 0.40, 0.15, 0.0]
_hi = [8.0, 10.0, 35.0, 0.75, 0.45, 6.0]
fit_mask = (recut_time_s >= -0.003) & (recut_time_s <= min(recut_post_s, 0.038)) & np.isfinite(recut_median)
try:
    popt, _ = curve_fit(_template_total, recut_time_s[fit_mask], recut_median[fit_mask],
                        p0=_p0, bounds=(_lo, _hi), maxfev=40000)
except Exception:
    popt = np.array(_p0, float)
(fit_tau_rise_ms, fit_tau_fast_ms, fit_tau_slow_ms,
 fit_frac_fast, fit_frac_slow, fit_t_onset_ms) = [float(x) for x in popt]

template_fit, fast_component, slow_component, super_component = _iglusnfr_tri_kernel(
    recut_time_s, fit_tau_rise_ms, fit_tau_fast_ms, fit_tau_slow_ms,
    fit_frac_fast, fit_frac_slow, fit_t_onset_ms)

recut_peak_time_ms = float(recut_time_s[int(np.nanargmax(recut_median))] * 1000.0)
template_peak_time_ms = float(recut_time_s[int(np.nanargmax(template_fit))] * 1000.0)
print(f"Panel B alignment: recut peak={recut_peak_time_ms:.2f} ms, template peak={template_peak_time_ms:.2f} ms, "
      f"diff={recut_peak_time_ms - template_peak_time_ms:.2f} ms (rise={fit_tau_rise_ms:.2f}, onset={fit_t_onset_ms:.2f})")

# =============================================================================
# Per-pulse kinetics: fit each pulse's median recut (rise & onset fixed from the
# global fit). Decay constants are kept for the per-pulse design templates.
# =============================================================================
raw_tau_fast_ms = np.full(n_pulses, np.nan)
raw_tau_slow_ms = np.full(n_pulses, np.nan)
raw_frac_fast = np.full(n_pulses, np.nan)
raw_frac_slow = np.full(n_pulses, np.nan)

def _pulse_fun(t_rel_s, tau_fast_ms, tau_slow_ms, frac_fast, frac_slow):
    return _template_total(t_rel_s, fit_tau_rise_ms, tau_fast_ms, tau_slow_ms,
                           frac_fast, frac_slow, fit_t_onset_ms)

for pulse_idx in range(n_pulses):
    y_pulse = recut_by_pulse_norm[pulse_idx]
    valid = fit_mask & np.isfinite(y_pulse)
    if valid.sum() < 8 or np.nanmax(y_pulse[valid]) <= 0:
        continue
    y_norm = y_pulse / np.nanmax(y_pulse[valid])
    try:
        pp, _ = curve_fit(_pulse_fun, recut_time_s[valid], y_norm[valid],
                          p0=[fit_tau_fast_ms, fit_tau_slow_ms, fit_frac_fast, fit_frac_slow],
                          bounds=([3.0, 10.0, 0.40, 0.15], [10.0, 35.0, 0.75, 0.45]), maxfev=20000)
        raw_tau_fast_ms[pulse_idx], raw_tau_slow_ms[pulse_idx], raw_frac_fast[pulse_idx], raw_frac_slow[pulse_idx] = pp
    except Exception:
        pass

def _fill(values, fallback):
    arr = np.asarray(values, float).copy()
    x = np.arange(arr.size)
    finite = np.isfinite(arr)
    if finite.sum() >= 2:
        arr[~finite] = np.interp(x[~finite], x[finite], arr[finite])
    else:
        arr[~finite] = fallback
    return arr

_sw = min(7, n_pulses if n_pulses % 2 else n_pulses - 1)
def _smooth(a):
    return savgol_filter(a, _sw, 2, mode='interp') if _sw >= 5 else a

# Decay constants change only smoothly and monotonically across the train.
smooth_tau_fast_ms = np.maximum.accumulate(np.clip(_smooth(_fill(raw_tau_fast_ms, fit_tau_fast_ms)), 3.0, 10.0))
smooth_tau_slow_ms = np.maximum.accumulate(np.clip(_smooth(_fill(raw_tau_slow_ms, fit_tau_slow_ms)), 10.0, 35.0))

# Component fractions evolve smoothly and monotonically across the train, exactly as
# in the extract_metrics variant model: the super-slow share accumulates ~linearly
# (persistent fluorescence builds up), the fast share falls, and the slow share takes
# the remainder. Event 1 is anchored on the bouton-average (global) template fit.
_ramp = np.linspace(0.0, 1.0, n_pulses) if n_pulses > 1 else np.zeros(1)
SUPER_FRAC_MAX = 0.12   # super-slow fraction reached by the last event
FAST_FRAC_END = 0.40    # fast fraction reached by the last event
frac_super_evo = SUPER_FRAC_MAX * _ramp
frac_fast_evo = np.clip(fit_frac_fast - (fit_frac_fast - FAST_FRAC_END) * _ramp, 0.0, 1.0)
frac_slow_evo = np.clip(1.0 - frac_fast_evo - frac_super_evo, 0.0, 1.0)
_frac_tot = frac_fast_evo + frac_slow_evo + frac_super_evo
_frac_tot[_frac_tot <= 0] = 1.0
frac_fast_evo, frac_slow_evo, frac_super_evo = (frac_fast_evo / _frac_tot,
                                                frac_slow_evo / _frac_tot,
                                                frac_super_evo / _frac_tot)

# =============================================================================
# Design matrix: one template per pulse, fixed at the stimulus time, using the
# per-pulse kinetics. Solve non-negative least squares on the SavGol-weighted
# average trace (peak region weighted; robust to noise) -> per-event amplitudes.
# =============================================================================
design = np.zeros((time_s.size, n_pulses))
for pulse_idx, stim_time in enumerate(stim_times_s):
    kernel, _, _, _ = _iglusnfr_tri_kernel(
        time_s - stim_time, fit_tau_rise_ms, smooth_tau_fast_ms[pulse_idx], smooth_tau_slow_ms[pulse_idx],
        frac_fast_evo[pulse_idx], frac_slow_evo[pulse_idx], t_onset_ms=fit_t_onset_ms)
    design[:, pulse_idx] = kernel

weights = np.abs(average_sg)
weights = weights - np.nanmin(weights)
weights = weights / np.nanmax(weights) if np.nanmax(weights) > 0 else np.ones_like(weights)
weights = np.clip(weights, 0.05, 1.0)
fit_region = (time_s >= train_start_s - 0.002) & (time_s <= stim_times_s[-1] + isi_s)
refit_amps, _ = nnls(design[fit_region] * weights[fit_region, None], average_proc[fit_region] * weights[fit_region])
component_matrix = design * refit_amps[None, :]
refit_yhat = design @ refit_amps
residual = average_proc - refit_yhat

# --- Peak readout: SavGol peak vs model peak per event ---
savgol_peak_amp = np.full(n_pulses, np.nan)
savgol_peak_time = np.full(n_pulses, np.nan)
savgol_peak_y = np.full(n_pulses, np.nan)
model_peak_time = np.full(n_pulses, np.nan)
model_peak_y = np.full(n_pulses, np.nan)
for pulse_idx, stim_time in enumerate(stim_times_s):
    win = (time_s >= stim_time) & (time_s < stim_time + peak_window_s)
    pre = (time_s >= stim_time - 0.004) & (time_s < stim_time - 0.001)
    local_baseline = float(np.nanmedian(average_sg[pre])) if np.any(pre) else 0.0
    if np.any(win):
        idx_win = np.flatnonzero(win)
        peak_idx = idx_win[int(np.nanargmax(average_sg[win]))]
        savgol_peak_time[pulse_idx] = time_s[peak_idx]
        savgol_peak_y[pulse_idx] = average_sg[peak_idx]
        savgol_peak_amp[pulse_idx] = average_sg[peak_idx] - local_baseline
        model_idx = idx_win[int(np.nanargmax(refit_yhat[win]))]
        model_peak_time[pulse_idx] = time_s[model_idx]
        model_peak_y[pulse_idx] = refit_yhat[model_idx]

# =============================================================================
# Plotting helpers
# =============================================================================
COL_TRACE, COL_AVG, COL_SG, COL_FIT = '0.80', '#1b4f72', '#d35400', '#c0392b'
# Component palette matches the extract_metrics parameter-evolution figure.
COL_FAST, COL_SLOW, COL_SUPER = '#1f77b4', '#ff7f0e', '#2ca02c'

def _panel_label(ax, label):
    ax.text(-0.16, 1.10, label, transform=ax.transAxes, ha='left', va='top',
            fontsize=12, fontweight='bold')

def _why(ax, text):
    ax.text(0.5, 1.15, text, transform=ax.transAxes, ha='center', va='bottom',
            fontsize=7.0, style='italic', color='0.4')

def _stim_ticks(ax, color='0.6', alpha=0.4):
    for stim_time in stim_times_s:
        ax.axvline(stim_time * 1000.0, color=color, lw=0.6, ls=':', alpha=alpha)

def _norm_to_first(values):
    arr = np.asarray(values, float)
    finite = arr[np.isfinite(arr)]
    first = finite[0] if finite.size else np.nan
    return arr / first if np.isfinite(first) and abs(first) > 1e-12 else arr

plt.rcParams.update({
    'font.size': 8, 'axes.titlesize': 9.5, 'axes.labelsize': 8,
    'xtick.labelsize': 7, 'ytick.labelsize': 7, 'legend.fontsize': 6.6,
})

fig_s2 = plt.figure(figsize=(12.4, 8.6), constrained_layout=False)
gs = fig_s2.add_gridspec(2, 3, hspace=0.52, wspace=0.30,
                         left=0.06, right=0.985, top=0.88, bottom=0.10)
ax_a = fig_s2.add_subplot(gs[0, 0])
ax_b = fig_s2.add_subplot(gs[0, 1])
ax_c = fig_s2.add_subplot(gs[0, 2])
ax_d = fig_s2.add_subplot(gs[1, 0])
# Panel E (fit + residual) is a nested 2-row cell.
gs_e = gs[1, 1].subgridspec(2, 1, height_ratios=[3.0, 1.0], hspace=0.08)
ax_e = fig_s2.add_subplot(gs_e[0])
ax_e_res = fig_s2.add_subplot(gs_e[1], sharex=ax_e)
ax_f = fig_s2.add_subplot(gs[1, 2])

# --- A. The measurement problem ---
for trial_idx in range(n_trials):
    ax_a.plot(time_s * 1000.0, all_proc[trial_idx], color=COL_TRACE, lw=0.55, alpha=0.7,
              label='Single trials' if trial_idx == 0 else None)
ax_a.plot(time_s * 1000.0, average_proc, color=COL_AVG, lw=1.5, label=f'Trial average (n={n_trials})')
_stim_ticks(ax_a)
ax_a.set_xlim(plot_xlim_s[0] * 1000.0, plot_xlim_s[1] * 1000.0)
ax_a.set_title('Noisy trials, overlapping responses')
ax_a.set_xlabel('Time (ms)')
ax_a.set_ylabel('dF/F0')
ax_a.legend(loc='upper right', frameon=False)
_panel_label(ax_a, 'A')
_why(ax_a, 'Single-trial peaks are unreliable; responses overlap in time')

# --- B. Bouton template from recuts (recuts + median + fitted template & components) ---
rng = np.random.default_rng(14)
sample_idx = rng.choice(recut_stack_norm.shape[0], size=min(90, recut_stack_norm.shape[0]), replace=False)
for j, idx in enumerate(sample_idx):
    ax_b.plot(recut_time_s * 1000.0, recut_stack_norm[idx], color=COL_TRACE, lw=0.4, alpha=0.4,
              label='Oversampled recuts' if j == 0 else None)
ax_b.plot(recut_time_s * 1000.0, recut_median, color=COL_AVG, lw=1.8, label='Recut median')
ax_b.plot(recut_time_s * 1000.0, template_fit, color=COL_FIT, lw=1.8, label='iGluSnFR template')
ax_b.plot(recut_time_s * 1000.0, fast_component, color=COL_FAST, lw=0.9, ls='--', label='Fast')
ax_b.plot(recut_time_s * 1000.0, slow_component, color=COL_SLOW, lw=0.9, ls='--', label='Slow')
ax_b.plot(recut_time_s * 1000.0, super_component, color=COL_SUPER, lw=0.9, ls='--', label='Super-slow')
ax_b.axvline(0, color='0.4', lw=0.7, ls='--')
ax_b.set_xlim(-recut_pre_s * 1000.0, recut_post_s * 1000.0)
ax_b.set_title('Bouton template from averaged recuts')
ax_b.set_xlabel('Time from stimulus (ms)')
ax_b.set_ylabel('Normalized response')
ax_b.text(0.97, 0.55,
          f'tau_rise={fit_tau_rise_ms:.1f} ms\ntau_fast={fit_tau_fast_ms:.1f} ms\ntau_slow={fit_tau_slow_ms:.1f} ms',
          transform=ax_b.transAxes, ha='right', va='top', fontsize=6.6, color='0.25')
ax_b.legend(loc='upper right', frameon=False, ncol=1)
_panel_label(ax_b, 'B')
_why(ax_b, 'Averaging recuts gives a stable estimate of bouton kinetics')

# --- C. Component fractions evolve smoothly across the train (extract_metrics style) ---
event_idx = np.arange(1, n_pulses + 1)
ax_c.stackplot(event_idx, frac_fast_evo, frac_slow_evo, frac_super_evo,
               labels=['frac_fast', 'frac_slow', 'frac_superslow'],
               colors=[COL_FAST, COL_SLOW, COL_SUPER], alpha=0.7)
ax_c.plot(event_idx, frac_fast_evo, 'o-', color=COL_FAST, ms=4, lw=1.4)
ax_c.plot(event_idx, frac_fast_evo + frac_slow_evo, 's-', color=COL_SLOW, ms=4, lw=1.4)
ax_c.set_xlim(1, n_pulses)
ax_c.set_ylim(0, 1.0)
ax_c.set_xticks(event_idx)
ax_c.set_title('Component fractions evolve along the train')
ax_c.set_xlabel('Event #')
ax_c.set_ylabel('Cumulative fraction')
ax_c.legend(loc='lower left', frameon=False, ncol=3, columnspacing=1.0, handlelength=1.2)
_panel_label(ax_c, 'C')
_why(ax_c, 'Contributions vary only smoothly and monotonically between events')

# --- D. Templates fixed at the stimulus times (deconvolution basis) ---
window = (time_s >= plot_xlim_s[0]) & (time_s <= plot_xlim_s[1])
basis_colors = plt.cm.viridis(np.linspace(0.10, 0.88, n_pulses))
for pulse_idx in range(n_pulses):
    ax_d.plot(time_s[window] * 1000.0, design[window, pulse_idx], color=basis_colors[pulse_idx], lw=0.9)
ax_d.plot(time_s[window] * 1000.0, design[window].sum(axis=1), color='0.35', lw=0.8, ls='--',
          label='Overlapping sum')
_stim_ticks(ax_d)
ax_d.set_xlim(plot_xlim_s[0] * 1000.0, plot_xlim_s[1] * 1000.0)
ax_d.set_title('Templates fixed at the stimulus times')
ax_d.set_xlabel('Time (ms)')
ax_d.set_ylabel('Unit template')
ax_d.legend(loc='upper right', frameon=False)
_panel_label(ax_d, 'D')
_why(ax_d, 'NNLS scales these fixed shapes; their tails explain later events')

# --- E. Solve the full train: fit, deconvolved components, residual, peaks ---
for pulse_idx in range(n_pulses):
    ax_e.plot(time_s * 1000.0, component_matrix[:, pulse_idx], color=basis_colors[pulse_idx],
              lw=0.7, alpha=0.85)
ax_e.plot(time_s * 1000.0, average_proc, color='0.6', lw=0.9, label='Average trace')
ax_e.plot(time_s * 1000.0, refit_yhat, color=COL_FIT, lw=1.5, label='NNLS fit (sum)')
ax_e.plot(savgol_peak_time * 1000.0, model_peak_y, ls='none', marker='o', ms=3.8,
          color=COL_FIT, mec='white', mew=0.4, label='Per-event amplitude')
_stim_ticks(ax_e)
ax_e.set_xlim(plot_xlim_s[0] * 1000.0, plot_xlim_s[1] * 1000.0)
ax_e.set_title('Non-negative fit of the whole train')
ax_e.set_ylabel('dF/F0')
ax_e.legend(loc='upper right', frameon=False)
ax_e.tick_params(labelbottom=False)
_panel_label(ax_e, 'E')
_why(ax_e, 'One robust fit deconvolves overlap into per-event amplitudes')
ax_e_res.axhline(0, color='0.6', lw=0.6)
ax_e_res.plot(time_s * 1000.0, residual, color='#34495e', lw=0.7)
_stim_ticks(ax_e_res)
ax_e_res.set_xlim(plot_xlim_s[0] * 1000.0, plot_xlim_s[1] * 1000.0)
ax_e_res.set_xlabel('Time (ms)')
ax_e_res.set_ylabel('Resid.')

# --- F. Output: per-event amplitude series (one bouton's dataset row) ---
amp_abs = average_amp_nnls_corr[:n_pulses]
amp_norm = _norm_to_first(amp_abs)
A1 = float(amp_abs[0]) if np.isfinite(amp_abs[0]) else np.nan
ax_f.plot(event_idx, amp_norm, 'o-', color=COL_FIT, ms=5, lw=1.8)
ax_f.axhline(1.0, color='0.5', lw=0.8, ls='--')
ax_f.set_xlim(1, n_pulses)
ax_f.set_ylim(0, max(1.15, float(np.nanmax(amp_norm)) * 1.12))
ax_f.set_xticks(event_idx)
ax_f.set_title('Per-event amplitudes (one dataset row)')
ax_f.set_xlabel('Stimulus')
ax_f.set_ylabel('Amplitude / A1   (= PPR)')
if np.isfinite(A1):
    ax_f.text(0.97, 0.95, f'A1 = {A1:.2f} dF/F0', transform=ax_f.transAxes,
              ha='right', va='top', fontsize=7.0, color='0.25')
_panel_label(ax_f, 'F')
_why(ax_f, 'These amplitudes (and A_n/A1 ratios) become one row of the dataset')

fig_s2.suptitle('Supplementary Figure S2. Template-matching NNLS pipeline used to build the per-bouton dataset',
                fontsize=12.5, fontweight='bold', y=0.965)
fig_s2.text(
    0.5, 0.022,
    f'Real example: {condition_label}, {file_stem}, n={n_trials} trials. '
    'Trial-averaged recuts give a bouton-specific iGluSnFR template (rise + fast/slow/super-slow decay). '
    'Templates are fixed at the stimulus times and their non-negative amplitudes solved jointly (robust, '
    'peak-weighted), deconvolving overlap into the per-event amplitude series.',
    ha='center', va='bottom', fontsize=7.4)

fig_s2.savefig(FIGS2_OUTPUT_PDF, dpi=300, bbox_inches='tight')
fig_s2.savefig(FIGS2_OUTPUT_PNG, dpi=300, bbox_inches='tight')
plt.show()

print('Supplementary Figure S2 (method) exported:')
print(f'  Source NPZ: {FIGS2_METHOD_NPZ}')
print(f'  PDF: {FIGS2_OUTPUT_PDF}')
print(f'  PNG: {FIGS2_OUTPUT_PNG}')
print(f'  Template: rise={fit_tau_rise_ms:.2f} ms, fast={fit_tau_fast_ms:.2f} ms, '
      f'slow={fit_tau_slow_ms:.2f} ms, frac_fast={fit_frac_fast:.2f}, onset={fit_t_onset_ms:.2f} ms')


## Section 2 — Figure 2: Four clusters of short-term plasticity


#### WT release property survey

Once clusters are defined, the next figures examine how each class differs in train profile, first-response amplitude, failure rate, and mean trace shape. Together these panels establish the physiological identity of the WT bouton classes that underlie the rest of the manuscript.


In [ ]:
# === Fig 2a — Mean WT PPR profile ===
# Plot the WT pooled mean PPR profile with individual bouton profiles in the background.

if 'plot_profile_group_comparison' not in globals() or 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled)

fig, ax = make_figure_grid(figsize=(10, 6))

# Individual overlays via shared helper (mean hidden for this pass)
plot_ppr_profiles_overlay(
    ax,
    wt_profiles,
    color='gray',
    label='_nolegend_',
    individual_alpha=0.05,
    individual_lw=3.0,
    mean_lw=0.0,
    sem_alpha=0.0,
)

# Mean ± SEM via shared helper
pulse_numbers = np.arange(1, wt_profiles.shape[1] + 1, dtype=float)
plot_profile_group_comparison(
    ax,
    [{
        'matrix': wt_profiles,
        'color': 'black',
        'marker': 'o',
        'label': f'WT pooled mean (n={wt_profiles.shape[0]})',
        'linewidth': 2.5,
        'markersize': 6,
        'sem_alpha': 0.25,
    }],
    pulse_numbers=pulse_numbers,
    title='Mean PPR profile',
    finalize_kwargs={
        'ylim': (0, 3.5),
        'unity_kwargs': {'color': 'gray', 'linestyle': '--', 'linewidth': 1, 'label': 'No facilitation'},
        'legend': True,
        'legend_loc': 'best',
    },
)
plt.tight_layout()

output_file = OUTPUT_DIR / 'Fig2_a_wt_mean_ppr_profile.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved to {output_file}')
print(f'✓ Plotted {wt_profiles.shape[0]} individual profiles with mean ± SEM')


In [ ]:
# === Aux — Top/bottom 10% extremes (parameter description) ===
# Plot PCA with top 10% AND bottom 10% highlighted for AMP1, PPR2/1, PPR5/1, and %Fail1
fig, axes = make_figure_grid(2, 2, figsize=(14, 12))
axes = axes.flatten()
metrics = ['AMP1', 'PPR2/1', 'PPR5/1', '%Fail1']
titles = ['Top/Bottom 10% AMP1', 'Top/Bottom 10% PPR2/1', 'Top/Bottom 10% PPR5/1', 'Top/Bottom 10% %Fail1']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx]
    
    # Plot all WT pooled with cluster colors (background)
    plot_pca_background(ax, alpha=0.3,  label='WT pooled')
    
    # Get top 10% and bottom 10% for this metric
    if metric in PCA_Data_WT_Pooled.columns:
        top_threshold = PCA_Data_WT_Pooled[metric].quantile(0.90)
        bottom_threshold = PCA_Data_WT_Pooled[metric].quantile(0.10)

        top_mask = PCA_Data_WT_Pooled[metric] >= top_threshold
        bottom_mask = PCA_Data_WT_Pooled[metric] <= bottom_threshold

        top_coords = pca_coordinates[top_mask.values]
        bottom_coords = pca_coordinates[bottom_mask.values]

        n_top = len(top_coords)
        n_bottom = len(bottom_coords)
        
        # Plot bottom 10% as orange stars (drawn first, behind top 10%)
        plot_pca_overlay_points(ax, bottom_coords[:],
                   marker='*',  c='orange', alpha=0.9,
                   edgecolors='black', linewidths=0.0,
                   label=f'Bottom 10% {metric} (n={n_bottom})')

        # Plot top 10% as purple stars (drawn on top)
        plot_pca_overlay_points(ax, top_coords[:],
                   marker='*',  c='purple', alpha=0.9,
                   edgecolors='black', linewidths=0.0,
                   label=f'Top 10% {metric} (n={n_top})')
    
    # Format subplot
    style_pca_axes(ax, title=title,  legend=False)
    add_legend(ax, loc='upper right', fontsize=6, frameon=False)

plt.tight_layout()
plt.suptitle('PCA: Top & Bottom 10% by Metric', fontsize=8, fontweight='bold', y=1.02)
output_file = OUTPUT_DIR / "Aux_wt_cluster_metric_extremes.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")

for metric in metrics:
    if metric in PCA_Data_WT_Pooled.columns:
        top_threshold = PCA_Data_WT_Pooled[metric].quantile(0.90)
        bottom_threshold = PCA_Data_WT_Pooled[metric].quantile(0.10)
        print(f"  {metric}: bottom threshold = {bottom_threshold:.4f} | top threshold = {top_threshold:.4f}")


In [ ]:
# === Aux — PCA scatter (unclustered) ===
# Plot PCA without cluster colors - all points in gray

make_figure(figsize=(8, 6))

# Plot all WT pooled points in gray
plot_pca_background(plt.gca(), use_cluster_colors=False, color='gray', alpha=1,  edgecolors='black', linewidths=0.0, label=None)

# Format plot
style_pca_axes(plt.gca(), title='PCA Projection: WT Pooled (unclustered)',  legend=False, show_variance_text=True)
plt.tight_layout()



# Save and display
output_file = OUTPUT_DIR / "Aux_wt_pca_unclustered_scatter.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved unclustered PCA plot to {output_file}")
print(f"✓ Total samples: {len(pca_coordinates)}")


In [ ]:
# === Fig 2b — PCA cluster ellipses ===
# Plot the WT PCA clusters with confidence ellipses and tolerance zones.

ELLIPSE_ALPHA = 0.8        # hard edge level
ELLIPSE_TOL   = 1.0       # inflate ellipse threshold by (1 + ELLIPSE_TOL)

import numpy as np, matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
from scipy.stats import chi2

# --- build models ---
def _ellipse_models(X, y, n_clusters, ridge=1e-6):
    models=[]
    for k in range(1, n_clusters+1):
        pts = X[y==k]
        if len(pts)>2:
            mu=pts.mean(axis=0); cov=np.cov(pts.T)+np.eye(2)*ridge; inv=np.linalg.pinv(cov)
            models.append({'cluster':k,'center':mu,'cov':cov,'inv_cov':inv})
    return models
def _md2(x, m): 
    d=x-m['center']; return float(d.T @ m['inv_cov'] @ d)

ellipse_models = _ellipse_models(pca_coordinates, cluster_assignments, N_CLUSTERS)
thr_hard       = chi2.ppf(ELLIPSE_ALPHA, df=2)
thr_tol        = thr_hard*(1.0 + ELLIPSE_TOL)
print(f"Thresholds - Hard: {thr_hard:.2f}, Tolerance: {thr_tol:.2f}")

# --- plot PCA with hard + tolerance ---
fig, ax = make_figure_grid(figsize=(8,6))
plot_pca_background(ax, alpha=0.9, marker='o', edgecolors='black', linewidths=0.0, label=None)
uniq = np.unique(cluster_assignments)

def _draw_ellipse(ax, mean, cov, thr, color, fa, lw, ls='-'):
    U,s,_=np.linalg.svd(cov); ang=np.degrees(np.arctan2(U[1,0],U[0,0])); w,h=2*np.sqrt(thr*s)
    ax.add_patch(Ellipse(mean, w, h, angle=ang, facecolor=color, edgecolor=color, alpha=fa, lw=lw, ls=ls))

for cid in uniq:
    pts = pca_coordinates[cluster_assignments==cid]
    if len(pts)<3:
        continue
    m     = [mm for mm in ellipse_models if mm['cluster']==cid][0]
    color = get_cluster_color(cid)
    _draw_ellipse(ax, m['center'], m['cov'], thr_tol,  color, 0.10, 0.0)
    _draw_ellipse(ax, m['center'], m['cov'], thr_hard, color, 0.00, 2.0)

cluster_handles = [
    Line2D([0], [0], marker='o', color='none', markerfacecolor=get_cluster_color(cid),
           markeredgecolor='black', markeredgewidth=0.0, markersize=7, label=f'C{cid}')
    for cid in uniq
]
cluster_labels = [f'C{int(cid)}' for cid in uniq]

style_pca_axes(ax, title='PCA Clusters and Ellipses', legend=False, show_variance_text=True)
add_legend(ax, handles=cluster_handles, labels=cluster_labels, frameon=False, loc='upper right')
fig.tight_layout()
out = OUTPUT_DIR / "Fig2_b_wt_cluster_ellipses.pdf"
fig.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print(f"[Ellipses] PCA plot saved: {out}")


#### PCA correlation circle (supplementary)

Each release variable is plotted as a vector onto the first two WT components, making the principal axes interpretable in terms of amplitude, failures, and short-term plasticity dynamics.

In [ ]:
# === Fig 2 supp — PCA parameter circle ===
# Supplementary PCA correlation circle (biplot) showing feature contributions to principal components.

feature_pc_correlations = PCA_RESULTS['correlations'].T.values
scaling_factor = 1.0
correlation_vectors = feature_pc_correlations * scaling_factor

fig, ax = make_figure_grid(figsize=(8, 8))

ax.axhline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
ax.axvline(0, color='black', linestyle='--', alpha=0.3, linewidth=1)
unit_circle = plt.Circle((0, 0), 1, color='black', fill=False, linestyle='-', alpha=0.5)
ax.add_patch(unit_circle)

feature_names = list(WT_pooled_for_pca.columns)

for feature_idx, feature_name in enumerate(feature_names):
    pc1_correlation = correlation_vectors[feature_idx, 0]
    pc2_correlation = correlation_vectors[feature_idx, 1]

    ax.annotate(
        '',
        xy=(pc1_correlation, pc2_correlation),
        xytext=(0, 0),
        arrowprops=dict(
            arrowstyle='-|>',
            color='darkred',
            lw=3.0,
            mutation_scale=14,
            alpha=0.9,
            shrinkA=0,
            shrinkB=0
        )
    )

    label_x_position = pc1_correlation * 1.1
    label_y_position = pc2_correlation * 1.1
    ax.text(
        label_x_position,
        label_y_position,
        feature_name,
        ha='center',
        va='center',
        fontsize=5,
        weight='bold'
    )

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']
total_variance = pc1_variance + pc2_variance

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_aspect('equal')
ax.set_title(f'PCA Parameter Circle\n({total_variance:.1%} total variance explained)')
ax.grid(False)

plt.tight_layout()

output_file = OUTPUT_DIR / "Fig2_supp_pca_parameter_circle.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print("Strongest Feature-PC Correlations:")
print("=" * 45)
pc_feature_correlations = PCA_RESULTS['correlations']

for pc_name in ['PC1', 'PC2']:
    strongest_features = pc_feature_correlations.loc[pc_name].abs().nlargest(5)
    print(f"\n{pc_name} (strongest contributors):")
    for feature_name, correlation_magnitude in strongest_features.items():
        correlation_value = pc_feature_correlations.loc[pc_name, feature_name]
        correlation_direction = "+" if correlation_value > 0 else "-"
        print(f"  {correlation_direction} {feature_name}: {correlation_magnitude:.3f}")

print(f"\n✓ Saved correlation circle to {output_file}")


##### Mean traces by WT cluster

The mean-trace panels show the time-domain signature of each WT class and make the cluster interpretation easier to relate to the original fluorescence data. They are especially useful when comparing classes that overlap partly in one scalar metric but diverge across the full train.


In [ ]:
# === Fig 2c — Mean traces per WT cluster ===
# Plot all cluster mean traces on one panel in reverse cluster order, separated by each trace spread.

fig, ax = make_figure_grid(figsize=(7.4, 4.8), panel_kind='trace')
cluster_trace_ids = {cid: [] for cid in range(1, N_CLUSTERS + 1)}
for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    cluster_trace_ids[row['HC_Cluster']].append(row['ID'])
cluster_stats = {cid: compute_trace_stats(trace_ids=cluster_trace_ids[cid], source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS) for cid in range(1, N_CLUSTERS + 1) if cluster_trace_ids[cid]}
cluster_order = sorted(cluster_stats, reverse=True)
cluster_spreads = {}
for cid in cluster_order:
    stats = cluster_stats[cid]
    sem_trace = np.asarray(stats['sem'], float)
    lower = np.asarray(stats['average'], float) - sem_trace
    upper = np.asarray(stats['average'], float) + sem_trace
    cluster_spreads[cid] = float(np.nanmax(upper) - np.nanmin(lower))
cluster_offsets = {}
running_offset = 0.0
prev_spread = None
for cid in cluster_order:
    if prev_spread is not None:
        running_offset += prev_spread
    cluster_offsets[cid] = running_offset
    prev_spread = cluster_spreads[cid]
stim_times = [1.0 + i * 0.05 for i in range(10)]
all_upper = []
all_lower = []
for cid in cluster_order:
    stats = cluster_stats[cid]
    offset = cluster_offsets[cid]
    mean_trace = np.asarray(stats['average'], float) + offset
    sem_trace = np.asarray(stats['sem'], float)
    all_upper.append(mean_trace + sem_trace)
    all_lower.append(mean_trace - sem_trace)
y_min_global = np.nanmin([np.nanmin(vals) for vals in all_lower])
y_max_global = np.nanmax([np.nanmax(vals) for vals in all_upper])
y_margin = max((y_max_global - y_min_global) * 0.06, 0.08)
y_lim = (y_min_global - y_margin, y_max_global + y_margin)
for cid in cluster_order:
    stats = cluster_stats[cid]
    color = get_cluster_color(cid)
    offset = cluster_offsets[cid]
    mean_trace = np.asarray(stats['average'], float) + offset
    sem_trace = np.asarray(stats['sem'], float)
    ax.fill_between(stats['time'], mean_trace - sem_trace, mean_trace + sem_trace, color=color, alpha=0.18)
    ax.plot(stats['time'], mean_trace, color=color, linewidth=clamp_linewidth(2.0), label=f'C{cid} (n={stats["n"]})')
    ax.axhline(offset, color=color, linestyle=':', linewidth=0.8, alpha=0.35)
add_stimulus_ticks(ax, stim_times, mode='fixed', y_span=(y_lim[1] - 0.07 * (y_lim[1] - y_lim[0]), y_lim[1] - 0.02 * (y_lim[1] - y_lim[0])), linewidth=1.4)
ax.axvline(1.0, color='gray', linestyle='--', alpha=0.45, linewidth=1.0)
ax.set_xlim(*TRACE_XLIM_20HZ)
ax.set_ylim(*y_lim)
style_ax(ax, 'Time (s)', 'ΔF/F + offset', 'WT cluster mean traces')
if 'style_trace_axis' in globals():
    style_trace_axis(ax)
add_legend(ax, frameon=False)
plt.tight_layout()
output_file = OUTPUT_DIR / "Fig2_c_wt_cluster_mean_traces.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved cluster mean traces to {output_file}")

print("Traces per cluster and applied offsets:")
for cid in cluster_order:
    print(f"  Cluster {cid}: {cluster_stats[cid]['n']} traces | spread={cluster_spreads[cid]:.2f} | offset={cluster_offsets[cid]:.2f}")


##### Selected-cluster trace overlay

Direct overlay of selected cluster averages highlights that boutons with related train dynamics can still differ in absolute glutamate output, consistent with partially dissociable control of short-term plasticity and synaptic weight.


In [ ]:
# === Fig 2d — Selected-cluster trace overlay ===
# Overlay mean traces for first and last cluster

clusterA_ids = []
clusterB_ids = []
clusterA_id = 1#1
clusterB_id = 4#N_CLUSTERS
for _, row in PCA_Data_WT_Pooled_clustered.iterrows():
    if row['HC_Cluster'] == clusterA_id:
        clusterA_ids.append(row['ID'])
    elif row['HC_Cluster'] == clusterB_id:
        clusterB_ids.append(row['ID'])
clusterA_stats = compute_trace_stats(trace_ids=clusterA_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
clusterB_stats = compute_trace_stats(trace_ids=clusterB_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
fig, ax = make_figure_grid(figsize=(10, 6))
plot_trace_analysis(ax=ax, rows=clusterB_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_cluster_color(clusterB_id), label=f'C{clusterB_id} (n={clusterB_stats["n"]})', display='mean_sem', style_axis=False)
plot_trace_analysis(ax=ax, rows=clusterA_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_cluster_color(clusterA_id), label=f'C{clusterA_id} (n={clusterA_stats["n"]})', display='mean_sem', zero_line=True, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, xlabel='Time (s)', ylabel='ΔF/F', title=f'Selected Cluster Overlay: C{clusterA_id} vs C{clusterB_id}', legend=True)
plt.tight_layout()
output_file = OUTPUT_DIR / "Fig2_d_wt_selected_cluster_mean_trace_overlay.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")
print(f"  C{clusterA_id}: {clusterA_stats['n']} traces")
print(f"  C{clusterB_id}: {clusterB_stats['n']} traces")

from scipy.stats import mannwhitneyu as _selected_cluster_mannwhitneyu

def _selected_cluster_bh_fdr(p_values):
    p_values = np.asarray(p_values, dtype=float)
    corrected = np.full(p_values.shape, np.nan, dtype=float)
    valid_mask = np.isfinite(p_values)
    valid_p = p_values[valid_mask]
    if valid_p.size == 0:
        return corrected
    order = np.argsort(valid_p)
    sorted_p = valid_p[order]
    ranks = np.arange(1, sorted_p.size + 1, dtype=float)
    sorted_q = np.minimum.accumulate((sorted_p * sorted_p.size / ranks)[::-1])[::-1]
    sorted_q = np.clip(sorted_q, 0.0, 1.0)
    valid_q = np.empty_like(valid_p)
    valid_q[order] = sorted_q
    corrected[valid_mask] = valid_q
    return corrected

def _selected_cluster_format_p(p_value):
    if not np.isfinite(p_value):
        return 'nan'
    return f'{p_value:.2g}' if p_value < 0.01 else f'{p_value:.2f}'.rstrip('0').rstrip('.')

selected_cluster_metric_labels = [
    ('AMP1', 'AMP1'),
    ('%Fail1', '%Fail1'),
    ('PPR2/1', 'PPR2/1'),
]
all_cluster_ids_for_stats = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].dropna().unique())
selected_cluster_stat_rows = []
for column, label in selected_cluster_metric_labels:
    metric_pair_rows = []
    raw_p_values = []
    for idx_a, cluster_a in enumerate(all_cluster_ids_for_stats):
        values_a = pd.to_numeric(PCA_Data_WT_Pooled_clustered.loc[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_a, column], errors='coerce').dropna().to_numpy(dtype=float)
        for cluster_b in all_cluster_ids_for_stats[idx_a + 1:]:
            values_b = pd.to_numeric(PCA_Data_WT_Pooled_clustered.loc[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_b, column], errors='coerce').dropna().to_numpy(dtype=float)
            stat, p_value = _selected_cluster_mannwhitneyu(values_a, values_b, alternative='two-sided')
            metric_pair_rows.append({
                'label': label,
                'Cluster_A': cluster_a,
                'Cluster_B': cluster_b,
                'n_a': int(values_a.size),
                'n_b': int(values_b.size),
                'mean_a': float(np.nanmean(values_a)),
                'mean_b': float(np.nanmean(values_b)),
                'sd_a': float(np.nanstd(values_a, ddof=1)),
                'sd_b': float(np.nanstd(values_b, ddof=1)),
                'U': float(stat),
                'p_raw': float(p_value),
            })
            raw_p_values.append(float(p_value))
    corrected_p_values = _selected_cluster_bh_fdr(raw_p_values)
    for row, corrected_p in zip(metric_pair_rows, corrected_p_values):
        row['p_fdr'] = float(corrected_p)
        if row['Cluster_A'] == clusterA_id and row['Cluster_B'] == clusterB_id:
            selected_cluster_stat_rows.append(row)

print(f"C{clusterA_id} vs C{clusterB_id} selected-cluster stats (Mann-Whitney U; FDR across all cluster pairs per metric):")
for row in selected_cluster_stat_rows:
    print(f"  {row['label']}: C{clusterA_id}={row['mean_a']:.3g} +/- {row['sd_a']:.3g} (n={row['n_a']}), C{clusterB_id}={row['mean_b']:.3g} +/- {row['sd_b']:.3g} (n={row['n_b']}), U={row['U']:.3g}, FDR p={_selected_cluster_format_p(row['p_fdr'])}")


##### Cluster PPR trajectories

The normalized train profiles make the short-term plasticity phenotype of each WT class explicit. These are the profiles that motivate the interpretation of classes in terms of facilitation, depression, and mixed modes of release regulation.


In [ ]:
# === Fig 2e — Mean PPR per cluster ===
# PPR profiles by hierarchical cluster

if 'plot_profile_group_comparison' not in globals() or 'ppr_profiles_matrix' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())

fig, ax = make_figure_grid(figsize=(10, 6))
show_stats = True
parametric_stat = True
group_inputs = []
pulse_numbers = None

for cluster in clusters:
    cluster_data = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster]
    profiles = ppr_profiles_matrix(cluster_data, ppr_column_names=ppr_cols)
    pulse_numbers = np.arange(1, profiles.shape[1] + 1, dtype=float)
    color = get_cluster_color(cluster)
    group_inputs.append({
        'matrix': profiles,
        'color': color,
        'marker': 'o',
        'label': f'Cluster {cluster} (n={profiles.shape[0]})',
        'linewidth': 2.0,
        'sem_alpha': 0.2,
    })

plot_profile_group_comparison(
    ax,
    group_inputs,
    pulse_numbers=pulse_numbers,
    stats_start_index=1,
    show_stats=show_stats,
    parametric_stat=parametric_stat,
    summary_label='PPR profiles by hierarchical cluster',
    title='Average PPR Profiles by Cluster',
    ylabel='Mean PPR (A_n/A_1)',
    finalize_kwargs={'legend': True, 'legend_loc': 'upper right', 'ylim': (0, None)},
)
plt.tight_layout()

output_file = OUTPUT_DIR / 'Fig2_e_wt_cluster_ppr_profiles.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# === Helper — Cluster boxplot helper ===
# Define the reusable cluster-wise boxplot helper before the figure cells that call it.
from scipy.stats import kruskal, mannwhitneyu
from itertools import combinations
from statsmodels.stats.multitest import multipletests

def cluster_boxplot_analysis(data, column, title, output_prefix, *, group_spacing=BOXPLOT_GROUP_SPACING_DEFAULT, x_padding=BOXPLOT_X_PADDING_DEFAULT):
    'Create boxplot by cluster with pairwise Mann-Whitney U tests and Benjamini-Hochberg (FDR) correction.'

    clusters = sorted(data['HC_Cluster'].unique())
    data_per_cluster = {c: data.loc[data['HC_Cluster'] == c, column].dropna().values for c in clusters}

    make_figure(panel_kind='boxplot_compact')
    ax = plt.gca()
    groups = [make_box_group(f'Cluster {cluster_id}', values=data_per_cluster[cluster_id], color=get_cluster_color(cluster_id)) for cluster_id in clusters]
    plot_boxplot_analysis(ax, groups, paired=False, show_points=True, show_stats=False, parametric_stat=False, title=title, ylabel=column, point_alpha=0.5, point_size=18, group_spacing=group_spacing, x_padding=x_padding)

    if 'PPR' in column:
        ax.axhline(1, color='gray', linestyle='dotted', linewidth=1)
        y_top = ax.get_ylim()[1]
        ax.set_ylim(0, y_top)
    elif 'AMP' in column or 'Fail' in column:
        ax.set_ylim(bottom=0)
    ax.get_xaxis().set_visible(False)

    handles = [plt.Line2D([0], [0], color=get_cluster_color(cluster_id), lw=4) for cluster_id in clusters]
    labels = [f'Cluster {cluster_id}' for cluster_id in clusters]
    add_legend(ax, handles, labels, loc='upper right')

    # Statistical tests

    try:
        kw_stat, kw_p = kruskal(*data_per_cluster.values())
    except ValueError:
        kw_stat, kw_p = float('nan'), float('nan')

    pairs = list(combinations(clusters, 2))
    results = []
    p_values = []
    for a, b in pairs:
        x, y = data_per_cluster[a], data_per_cluster[b]
        try:
            stat, p = mannwhitneyu(x, y, alternative='two-sided')
        except ValueError:
            stat, p = float('nan'), float('nan')
        results.append({
            'Cluster_A': a,
            'Cluster_B': b,
            'n_A': len(x),
            'n_B': len(y),
            'U_statistic': stat,
            'p_value_raw': p,
            'p_value_corrected': float('nan')  # Placeholder, filled after FDR correction
        })
        p_values.append(p)

    # Apply Benjamini-Hochberg correction (FDR)
    _, corrected_p_values, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')

    # Update corrected p-values in the results
    for idx, result in enumerate(results):
        result['p_value_corrected'] = corrected_p_values[idx] if not pd.isna(p_values[idx]) else pd.NA

    # Sort results by corrected p-value
    results.sort(key=lambda d: d['p_value_corrected'] if not pd.isna(d['p_value_corrected']) else 1)

    # Create a DataFrame for the results
    results_df = pd.DataFrame(results)

    # Add Kruskal-Wallis test results
    kw_results = pd.DataFrame({
        'Test': ['Kruskal-Wallis'],
        'Statistic': [kw_stat],
        'p_value': [kw_p],
        'Number_of_comparisons': [len(pairs)]
    })

    # Export vers Excel
    with pd.ExcelWriter(OUTPUT_DIR / f"{output_prefix}_statistics.xlsx") as writer:
        results_df.to_excel(writer, sheet_name='Pairwise_Comparisons', index=False)
        kw_results.to_excel(writer, sheet_name='Kruskal_Wallis', index=False)

    # Save boxplot
    plt.tight_layout()
    output_file = OUTPUT_DIR / f"{output_prefix}.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Analysis saved to {output_file} and {OUTPUT_DIR / f'{output_prefix}_statistics.xlsx'}.")

##### Cluster-level parameter comparisons

These comparisons quantify how amplitude, failure rate, and early PPR differ from one WT class to another. They support the idea that bouton diversity is not reducible to a single monotonic axis of synaptic strength.


In [ ]:
# === Fig 2f — Per-cluster summary boxplots ===
# Assemble the four cluster-wise summary metrics into one composite panel.
cluster_group_spacing = 0.25
cluster_x_padding = 0.10
clusters = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].unique())
metrics = [
    ('AMP1', 'A1 amplitude', 'A1 (ΔF/F₀)'),
    ('PPR2/1', 'PPR2/A1', 'PPR2/A1'),
    ('PPR10/1', 'PPR10/A1', 'PPR10/A1'),
    ('%Fail1', 'Failure peak 1', 'Failure peak 1 (%)'),
]

fig, axes = make_figure_grid(1, len(metrics), panel_kind='boxplot_compact', squeeze=False)
stats_workbook = OUTPUT_DIR / 'Fig2_f_wt_cluster_summary_boxplots_statistics.xlsx'
summary_lines = []

with pd.ExcelWriter(stats_workbook) as writer:
    for ax, (column, panel_title, ylabel) in zip(axes[0], metrics):
        data_per_cluster = {cid: PCA_Data_WT_Pooled_clustered.loc[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cid, column].dropna().values for cid in clusters}
        groups = [make_box_group(f'C{cid}', values=data_per_cluster[cid], color=get_cluster_color(cid)) for cid in clusters]
        plot_boxplot_analysis(
            ax,
            groups,
            paired=False,
            show_points=True,
            show_stats=False,
            parametric_stat=False,
            title=panel_title,
            ylabel=ylabel if column == 'AMP1' else None,
            point_alpha=0.5,
            point_size=18,
            group_spacing=cluster_group_spacing,
            x_padding=cluster_x_padding,
        )
        if 'PPR' in column:
            ax.axhline(1, color='gray', linestyle='dotted', linewidth=1)
            y_top = ax.get_ylim()[1]
            ax.set_ylim(0, y_top)
        else:
            ax.set_ylim(bottom=0)

        try:
            kw_stat, kw_p = kruskal(*data_per_cluster.values())
        except ValueError:
            kw_stat, kw_p = float('nan'), float('nan')

        pairs = list(combinations(clusters, 2))
        pairwise_rows = []
        raw_p_values = []
        for a, b in pairs:
            x = data_per_cluster[a]
            y = data_per_cluster[b]
            try:
                stat, p_value = mannwhitneyu(x, y, alternative='two-sided')
            except ValueError:
                stat, p_value = float('nan'), float('nan')
            pairwise_rows.append({
                'Cluster_A': a,
                'Cluster_B': b,
                'n_A': len(x),
                'n_B': len(y),
                'U_statistic': stat,
                'p_value_raw': p_value,
            })
            raw_p_values.append(p_value)

        _, corrected_p_values, _, _ = multipletests(raw_p_values, alpha=0.05, method='fdr_bh')
        for idx, row in enumerate(pairwise_rows):
            row['p_value_corrected'] = corrected_p_values[idx] if not pd.isna(raw_p_values[idx]) else pd.NA

        pairwise_df = pd.DataFrame(pairwise_rows)
        pairwise_df.to_excel(writer, sheet_name=column.replace('/', '_')[:31], index=False)
        summary_lines.append(
            f"{column}: Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.3g}; "
            + ', '.join([f'C{cid} n={len(data_per_cluster[cid])}' for cid in clusters])
        )

fig.tight_layout()
output_file = OUTPUT_DIR / 'Fig2_f_wt_cluster_summary_boxplots.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ Saved composite cluster summary boxplots to {output_file}')
print(f'✓ Saved statistics workbook to {stats_workbook}')
for line in summary_lines:
    print(line)

In [ ]:
# === Fig 2f addendum - Candidate C1/C2/C3 pair checks ===
# Check whether any requested pair supports "similar A1 and %Fail1, but different PPR".
if 'PCA_Data_WT_Pooled_clustered' not in globals():
    raise RuntimeError('Run the WT clustering cells before this addendum.')
if 'mannwhitneyu' not in globals() or 'multipletests' not in globals():
    raise RuntimeError('Run the cluster statistics helper cell before this addendum.')

candidate_pairs = [(1, 2), (1, 3), (2, 3)]
candidate_metrics = [
    ('AMP1', 'A1'),
    ('%Fail1', '%Fail1'),
    ('PPR2/1', 'PPR2/1'),
    ('PPR10/1', 'PPR10/1'),
]
all_cluster_ids = sorted(PCA_Data_WT_Pooled_clustered['HC_Cluster'].dropna().unique())

def _candidate_pair_format_p(p_value):
    if not np.isfinite(p_value):
        return 'nan'
    if p_value < 0.001:
        return f'{p_value:.2g}'
    if p_value < 0.01:
        return f'{p_value:.3f}'.rstrip('0').rstrip('.')
    return f'{p_value:.2f}'.rstrip('0').rstrip('.')

def _candidate_pair_format_u(value):
    if not np.isfinite(value):
        return 'nan'
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    return f'{value:.1f}'

def _candidate_pair_format_value(metric, value):
    if not np.isfinite(value):
        return 'nan'
    return f'{value:.1f}' if metric == '%Fail1' else f'{value:.2f}'

candidate_pair_rows = []
for column, label in candidate_metrics:
    metric_rows = []
    raw_p_values = []
    for idx_a, cluster_a in enumerate(all_cluster_ids):
        values_a = pd.to_numeric(
            PCA_Data_WT_Pooled_clustered.loc[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_a, column],
            errors='coerce',
        ).dropna().to_numpy(dtype=float)
        for cluster_b in all_cluster_ids[idx_a + 1:]:
            values_b = pd.to_numeric(
                PCA_Data_WT_Pooled_clustered.loc[PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cluster_b, column],
                errors='coerce',
            ).dropna().to_numpy(dtype=float)
            stat, p_value = mannwhitneyu(values_a, values_b, alternative='two-sided')
            metric_rows.append({
                'metric': label,
                'Cluster_A': int(cluster_a),
                'Cluster_B': int(cluster_b),
                'n_A': int(values_a.size),
                'n_B': int(values_b.size),
                'mean_A': float(np.nanmean(values_a)),
                'mean_B': float(np.nanmean(values_b)),
                'sd_A': float(np.nanstd(values_a, ddof=1)),
                'sd_B': float(np.nanstd(values_b, ddof=1)),
                'U': float(stat),
                'p_raw': float(p_value),
            })
            raw_p_values.append(float(p_value))
    _, corrected_p_values, _, _ = multipletests(raw_p_values, alpha=0.05, method='fdr_bh')
    for row, p_fdr in zip(metric_rows, corrected_p_values):
        row['p_fdr'] = float(p_fdr)
        if (row['Cluster_A'], row['Cluster_B']) in candidate_pairs:
            candidate_pair_rows.append(row)

CANDIDATE_PAIR_STATS_FIG2 = pd.DataFrame(candidate_pair_rows)
print('Candidate-pair check for "similar A1 and %Fail1, but different PPR":')
for cluster_a, cluster_b in candidate_pairs:
    pair_rows = CANDIDATE_PAIR_STATS_FIG2[
        (CANDIDATE_PAIR_STATS_FIG2['Cluster_A'] == cluster_a) &
        (CANDIDATE_PAIR_STATS_FIG2['Cluster_B'] == cluster_b)
    ]
    similarity_by_metric = {row['metric']: bool(row['p_fdr'] >= 0.05) for _, row in pair_rows.iterrows()}
    supports_claim = (
        similarity_by_metric.get('A1', False) and
        similarity_by_metric.get('%Fail1', False) and
        not similarity_by_metric.get('PPR2/1', True) and
        not similarity_by_metric.get('PPR10/1', True)
    )
    verdict = 'supports exact claim' if supports_claim else 'does NOT support exact claim'
    print(f'C{cluster_a} vs C{cluster_b}: {verdict}')
    for _, row in pair_rows.iterrows():
        metric = row['metric']
        mean_a = _candidate_pair_format_value(metric, row['mean_A'])
        sd_a = _candidate_pair_format_value(metric, row['sd_A'])
        mean_b = _candidate_pair_format_value(metric, row['mean_B'])
        sd_b = _candidate_pair_format_value(metric, row['sd_B'])
        print(
            f"  {metric}: C{cluster_a}={mean_a} +/- {sd_a} (n={int(row['n_A'])}), "
            f"C{cluster_b}={mean_b} +/- {sd_b} (n={int(row['n_B'])}), "
            f"U={_candidate_pair_format_u(row['U'])}, FDR p={_candidate_pair_format_p(row['p_fdr'])}"
        )


## Section 3 — Figure 3: Heterogeneity & quantal release (N, P, refilling)


### WT quantal, N, P, and refilling analyses

Trial-level responses are normalized to an empirical single-vesicle estimate to interpret WT bouton responses in terms of quantal size, effective release-site occupancy, effective release probability, and refilling dynamics. This section provides a mechanistic bridge between the WT clustering results and the calcium/frequency perturbations.

Several panels in this block extend beyond the wording of the current manuscript draft and should be read as mechanistic elaborations of the same N-Pr framework. The guiding idea is unchanged: bouton classes differ not only in apparent release probability, but also in the occupancy and rapid reuse of release sites during trains.

Methodological note: quantal calibration is anchored to successes at 1.5 mM Ca²⁺, cumulative release is summarized with Schneggenburger-Neher style plots, and N/P trajectories are estimated with binomial or bootstrap-based procedures. The more recent refilling and occupancy figures are included here because they sharpen the interpretation of sustained facilitation at high release drive.


#### First-response amplitude distribution

The first transient amplitude is examined directly because it provides the most immediate readout of initial synaptic weight at the start of the train. Its change with Ca²⁺ is later interpreted jointly with failures and PPR.


In [ ]:
# === Fig 3a — A1 amplitude distribution ===
# Compare AMP1 distributions between calcium concentrations
AMP1_HIST_BIN_WIDTH = 0.1


def plot_calcium_amp1_comparison(bin_width=AMP1_HIST_BIN_WIDTH):
    """Compare AMP1 distributions between 1.5mM, 2.5mM and 4.0mM calcium."""
    
    # Get AMP1 data for all three conditions
    amp1_standard = PCA_Data_WT_Pooled['AMP1'].dropna()
    amp1_low_ca   = PCA_Data_WT_Low_Ca['AMP1'].dropna()
    amp1_high_ca  = PCA_Data_WT_High_Ca['AMP1'].dropna()
    
    # Calculate common bins for fair comparison
    all_amp1_values = pd.concat([amp1_standard, amp1_low_ca, amp1_high_ca])
    amp1_min = float(all_amp1_values.min())
    amp1_max = float(all_amp1_values.max())
    bin_edges = np.arange(amp1_min, amp1_max + bin_width, bin_width, dtype=float)
    if bin_edges.size < 2:
        bin_edges = np.array([amp1_min - bin_width / 2.0, amp1_max + bin_width / 2.0], dtype=float)
    elif bin_edges[-1] < amp1_max:
        bin_edges = np.append(bin_edges, bin_edges[-1] + bin_width)
    
    # Create figure
    make_figure(figsize=(10, 6))
    
    # Calculate weights for percentage display
    weights_standard = np.ones(len(amp1_standard)) * (100.0 / len(amp1_standard))
    weights_low_ca   = np.ones(len(amp1_low_ca)) * (100.0 / len(amp1_low_ca))
    weights_high_ca  = np.ones(len(amp1_high_ca)) * (100.0 / len(amp1_high_ca))
    
    # Plot histograms
    plt.hist(amp1_standard, bins=bin_edges, alpha=0.4, color=get_wt_ca_color('2.5mM'), 
             weights=weights_standard, edgecolor='black', linewidth=0.8,
             label=f'WT 2.5mM Ca (n={len(amp1_standard)})')
    plt.hist(amp1_low_ca, bins=bin_edges, alpha=0.4, color=get_wt_ca_color('1.5mM'), 
             weights=weights_low_ca, edgecolor='black', linewidth=0.8,
             label=f'WT 1.5mM Ca (n={len(amp1_low_ca)})')
    plt.hist(amp1_high_ca, bins=bin_edges, alpha=0.35, color=get_wt_ca_color('4mM'), 
             weights=weights_high_ca, edgecolor='black', linewidth=0.8,
             label=f'WT 4.0mM Ca (n={len(amp1_high_ca)})')
    
    # Add vertical lines for means
    mean_standard = amp1_standard.mean()
    mean_low_ca = amp1_low_ca.mean()
    mean_high_ca = amp1_high_ca.mean()
    
    plt.axvline(mean_standard, color=get_wt_ca_color('2.5mM'), linestyle='--', linewidth=1, alpha=0.9,
                label=f'Mean 2.5mM: {mean_standard:.3f}')
    plt.axvline(mean_low_ca, color=get_wt_ca_color('1.5mM'), linestyle='--', linewidth=1, alpha=0.9,
                label=f'Mean 1.5mM: {mean_low_ca:.3f}')
    plt.axvline(mean_high_ca, color=get_wt_ca_color('4mM'), linestyle='--', linewidth=1, alpha=0.9,
                label=f'Mean 4.0mM: {mean_high_ca:.3f}')
    
    # Format plot
    plt.xlabel('AMP1 (Amplitude)')
    plt.ylabel('Proportion (%)')
    plt.title('AMP1 Distribution Across Calcium Conditions')
    add_legend(plt.gca(), frameon=False, outside=True)
    if 'style_hist_axis' in globals():
        style_hist_axis(plt.gca())
    plt.grid(False)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "Fig3_a_calcium_amp1_histogram_comparison.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return amp1_standard, amp1_low_ca, amp1_high_ca, output_file

# Run analysis
amp1_standard, amp1_low_ca, amp1_high_ca, output_file = plot_calcium_amp1_comparison()

# Statistical comparison
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

# Summary statistics
print(f"=== AMP1 CALCIUM COMPARISON ===")
print(f"2.5mM Ca (standard): {amp1_standard.mean():.3f} ± {amp1_standard.std():.3f} (n={len(amp1_standard)})")
print(f"1.5mM Ca (low):      {amp1_low_ca.mean():.3f} ± {amp1_low_ca.std():.3f} (n={len(amp1_low_ca)})")
print(f"4.0mM Ca (high):     {amp1_high_ca.mean():.3f} ± {amp1_high_ca.std():.3f} (n={len(amp1_high_ca)})")

print(f"\nPairwise Mann-Whitney tests:")
for pair_label, a, b in [
    ('1.5mM vs 2.5mM', amp1_low_ca, amp1_standard),
    ('2.5mM vs 4.0mM', amp1_standard, amp1_high_ca),
    ('1.5mM vs 4.0mM', amp1_low_ca, amp1_high_ca),
]:
    mw_stat, mw_p = mannwhitneyu(a, b, alternative='two-sided')
    print(f"{pair_label:<15}: U={mw_stat:.1f}, p={mw_p:.4g}")

print(f"\n✓ Saved comparison to {output_file}")


In [ ]:
# === Helper - Classical multimodal quantal fits ===
# One 2x3 figure only:
#   top row    = one bouton across 1.5 / 2.5 / 4 mM when available
#   bottom row = pooled distributions for the same calcium conditions
# Model:
#   failures:  N(0, sigma_B^2) after offsetting the failure pedestal by mu0
#   releases:  sum_{k>=1} w_k * N(k, sigma_B^2 + k * sigma_Q^2)
# Failures are shown separately by default, so the failure pedestal can be offset
# without shifting the quantal ladder away from K = 1, 2, 3, ...

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

DATA_SOURCE = 'A1'          # 'A1' or 'all'
SEPARATE_FAILURES = True    # failures in gray + separate noise Gaussian
BIN_WIDTH_Q = 0.25
MAX_K = 8
XLIM_Q = (-1.0, 8.0)
EXAMPLE_BASE_ID = globals().get('EXAMPLE_MLE_OVERRIDE', None)
MIN_GROUP_SAMPLES = 3

required_globals = [
    'trials_all',
    'Q_estimates',
    'Global_Q',
    'extract_base_name',
    'get_amp_row',
    'get_failure_mask_row',
    'get_count_row_with_failures',
    'get_calcium_conditions',
    'fit_binom_mle',
]
missing_globals = [name for name in required_globals if name not in globals()]
if missing_globals:
    raise RuntimeError('Run the upstream quantal helper cells first; missing: ' + ', '.join(missing_globals))

if DATA_SOURCE not in {'A1', 'all'}:
    raise ValueError("DATA_SOURCE must be 'A1' or 'all'.")

CA_SPECS = [
    ('1.5mM', '1.5 mM', get_wt_ca_color('1.5mM')),
    ('2.5mM', '2.5 mM', get_wt_ca_color('2.5mM')),
    ('4mM', '4 mM', get_wt_ca_color('4mM')),
]


def _condition_names(ca_key):
    names = []
    for freq in ('20Hz', '50Hz'):
        try:
            names.extend(list(get_calcium_conditions(ca_key, freq)))
        except Exception:
            pass
    return set(str(name).strip() for name in names)


CA_CONDITIONS = {ca_key: _condition_names(ca_key) for ca_key, _label, _color in CA_SPECS}

trials_plot = trials_all.copy()
trials_plot['condition'] = trials_plot['condition'].astype(str).str.strip()
trials_plot['base_id'] = trials_plot['file'].astype(str).map(lambda s: extract_base_name(str(s).strip()))
trials_plot['CaKey'] = None
for ca_key, cond_names in CA_CONDITIONS.items():
    if cond_names:
        trials_plot.loc[trials_plot['condition'].isin(cond_names), 'CaKey'] = ca_key
trials_plot = trials_plot[trials_plot['CaKey'].isin([ca_key for ca_key, _label, _color in CA_SPECS])].copy()
if trials_plot.empty:
    raise RuntimeError('No trial rows matched the calcium-condition groups.')


def _q_for_base(base_id):
    q_value = Q_estimates.get(base_id, np.nan)
    if not np.isfinite(q_value) or q_value <= 0:
        q_value = Global_Q
    return float(q_value)


def _collect_panel(df):
    amps_all = []
    fail_all = []
    count_all = []
    for _, row in df.iterrows():
        q_value = _q_for_base(str(row['base_id']))
        amps = np.asarray(get_amp_row(row), dtype=float)
        fail_mask, _ = get_failure_mask_row(row)
        fail_mask = np.asarray(fail_mask, dtype=bool)
        counts = np.asarray(get_count_row_with_failures(row, q_value), dtype=float)
        if DATA_SOURCE == 'A1':
            amps = amps[:1]
            fail_mask = fail_mask[:1]
            counts = counts[:1]
        keep = np.isfinite(amps) & np.isfinite(counts)
        if not np.any(keep):
            continue
        amps = amps[keep] / q_value
        fail_mask = fail_mask[keep]
        counts = np.rint(counts[keep]).astype(int)
        amps_all.append(amps)
        fail_all.append(fail_mask)
        count_all.append(counts)
    if not amps_all:
        empty = np.array([], dtype=float)
        empty_i = np.array([], dtype=int)
        return {
            'samples': empty,
            'fail_mask': np.array([], dtype=bool),
            'counts': empty_i,
            'fail_samples': empty,
            'nonfail_samples': empty,
            'nonfail_counts': empty_i,
            'n_rows': int(len(df)),
            'n_boutons': int(df['base_id'].nunique()) if len(df) else 0,
        }
    samples = np.concatenate(amps_all)
    fail_mask = np.concatenate(fail_all)
    counts = np.concatenate(count_all)
    return {
        'samples': samples,
        'fail_mask': fail_mask,
        'counts': counts,
        'fail_samples': samples[fail_mask],
        'nonfail_samples': samples[~fail_mask],
        'nonfail_counts': counts[~fail_mask],
        'n_rows': int(len(df)),
        'n_boutons': int(df['base_id'].nunique()),
    }


coverage = (
    trials_plot.groupby(['base_id', 'CaKey']).size().unstack(fill_value=0)
    .reindex(columns=[ca_key for ca_key, _label, _color in CA_SPECS], fill_value=0)
    .rename_axis(index=None, columns=None)
)
if coverage.empty:
    raise RuntimeError('No bouton ids available after calcium grouping.')

if EXAMPLE_BASE_ID is not None:
    example_base_id = extract_base_name(str(EXAMPLE_BASE_ID).strip())
else:
    coverage_summary = pd.DataFrame({
        'n_conditions': (coverage > 0).sum(axis=1),
        'n_rows': coverage.sum(axis=1),
        'base_id': coverage.index.astype(str),
    }).sort_values(['n_conditions', 'n_rows', 'base_id'], ascending=[False, False, True])
    example_base_id = str(coverage_summary.iloc[0]['base_id'])

panels = []
for row_kind in ('example', 'pooled'):
    for ca_key, ca_label, color in CA_SPECS:
        if row_kind == 'example':
            df_use = trials_plot[(trials_plot['CaKey'] == ca_key) & (trials_plot['base_id'] == example_base_id)].copy()
            title = f'{ca_label} example - {example_base_id}'
        else:
            df_use = trials_plot[trials_plot['CaKey'] == ca_key].copy()
            title = f'{ca_label} pooled'
        panels.append({
            'row_kind': row_kind,
            'ca_key': ca_key,
            'ca_label': ca_label,
            'color': color,
            'title': title,
            'payload': _collect_panel(df_use),
        })

all_failure_samples = np.concatenate([
    panel['payload']['fail_samples']
    for panel in panels
    if panel['payload']['fail_samples'].size
]) if any(panel['payload']['fail_samples'].size for panel in panels) else np.array([], dtype=float)


def _robust_sigma(values, fallback=0.20):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size < 2:
        return float(fallback)
    center = float(np.nanmedian(values))
    mad = float(np.nanmedian(np.abs(values - center)))
    sigma = 1.4826 * mad if np.isfinite(mad) else np.nan
    if not np.isfinite(sigma) or sigma <= 1e-6:
        sigma = float(np.nanstd(values, ddof=1))
    if not np.isfinite(sigma) or sigma <= 1e-6:
        sigma = float(fallback)
    return float(np.clip(sigma, 0.03, 1.50))


def _failure_mode(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0
    sigma_guess = _robust_sigma(values, fallback=0.20)
    if not np.isfinite(sigma_guess) or sigma_guess <= 0:
        sigma_guess = 0.20
    bin_width_mode = max(0.05, 0.35 * sigma_guess)
    lo = float(np.nanpercentile(values, 2))
    hi = float(np.nanpercentile(values, 98))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return float(np.nanmedian(values))
    edges = np.arange(lo, hi + bin_width_mode, bin_width_mode, dtype=float)
    if edges.size < 2:
        return float(np.nanmedian(values))
    hist, edges = np.histogram(values, bins=edges)
    if hist.size == 0 or np.nanmax(hist) <= 0:
        return float(np.nanmedian(values))
    idx = int(np.nanargmax(hist))
    return float(0.5 * (edges[idx] + edges[idx + 1]))


def _normal_pdf(x, mu, sigma):
    sigma = max(float(sigma), 1e-6)
    z = (np.asarray(x, dtype=float) - float(mu)) / sigma
    return np.exp(-0.5 * z * z) / (sigma * np.sqrt(2.0 * np.pi))


def _release_stats(counts):
    counts = np.asarray(counts, dtype=float)
    counts = counts[np.isfinite(counts)]
    if counts.size == 0:
        return None
    counts = np.rint(counts).astype(int)
    counts = np.clip(counts, 0, MAX_K)
    support_max = int(min(MAX_K, max(int(np.nanmax(counts)), 1)))
    support_trim = np.arange(0, support_max + 1, dtype=int)
    weights_trim = np.array([(counts == k_val).sum() for k_val in support_trim], dtype=float)
    weights_trim /= max(weights_trim.sum(), 1.0)
    support_pos = support_trim[1:]
    weights_pos = weights_trim[1:].copy()
    if weights_pos.sum() > 0:
        weights_pos /= weights_pos.sum()
    n_hat, p_hat, _ = fit_binom_mle(counts, int(max(globals().get('N_MAX_D', 20), support_max)))
    return {
        'N_hat': float(n_hat) if np.isfinite(n_hat) else np.nan,
        'P_hat': float(p_hat) if np.isfinite(p_hat) else np.nan,
        'P0_model': float(weights_trim[0]),
        'support_trim': support_trim,
        'weights_trim': weights_trim,
        'support_pos': support_pos,
        'weights_pos_cond': weights_pos,
    }


def _sigma_q_candidates(nonfail_samples, nonfail_counts, sigma_b):
    samples = np.asarray(nonfail_samples, dtype=float)
    counts = np.asarray(nonfail_counts, dtype=int)
    candidates = []
    weights = []
    for k_val in np.unique(counts):
        if int(k_val) < 1 or int(k_val) > MAX_K:
            continue
        vals = samples[counts == int(k_val)]
        vals = vals[np.isfinite(vals)]
        if vals.size < MIN_GROUP_SAMPLES:
            continue
        var_k = float(np.nanvar(vals, ddof=1))
        sigma_q_sq = max((var_k - sigma_b ** 2) / float(k_val), 0.0)
        candidates.append(np.sqrt(sigma_q_sq))
        weights.append(vals.size - 1)
    return np.asarray(candidates, dtype=float), np.asarray(weights, dtype=float)


mu0_shared = _failure_mode(all_failure_samples) if all_failure_samples.size else 0.0
sigma_B_shared = _robust_sigma(all_failure_samples, fallback=0.20)

bin_edges = np.arange(XLIM_Q[0], XLIM_Q[1] + BIN_WIDTH_Q, BIN_WIDTH_Q, dtype=float)
bin_width = float(np.median(np.diff(bin_edges)))
x_grid = np.linspace(XLIM_Q[0], XLIM_Q[1], 1600)

for panel in panels:
    payload = panel['payload']
    panel['release_fit'] = _release_stats(payload['counts'])

sigma_q_pool = []
sigma_q_w = []
for panel in panels:
    payload = panel['payload']
    cand, w = _sigma_q_candidates(payload['nonfail_samples'], payload['nonfail_counts'], sigma_B_shared)
    if cand.size:
        sigma_q_pool.append(cand)
        sigma_q_w.append(w)

if sigma_q_pool:
    sigma_q_pool = np.concatenate(sigma_q_pool)
    sigma_q_w = np.concatenate(sigma_q_w)
    sigma_Q_shared = float(np.average(sigma_q_pool, weights=sigma_q_w))
else:
    sigma_Q_shared = 0.20
sigma_Q_shared = float(np.clip(sigma_Q_shared, 0.03, 1.50))

fig, axes = plt.subplots(2, 3, figsize=(14, 7.8), sharex=True, sharey=False, constrained_layout=True)
summary_rows = []

for ax, panel in zip(axes.ravel(), panels):
    payload = panel['payload']
    release_fit = panel['release_fit']

    ax.set_xlim(*XLIM_Q)
    ax.grid(False)

    if payload['samples'].size == 0:
        ax.text(0.5, 0.5, 'Not available', transform=ax.transAxes, ha='center', va='center', fontsize=10)
        if 'style_ax' in globals():
            style_ax(ax, 'Amplitude / Q (K units)', 'Probability', panel['title'])
        else:
            ax.set_title(panel['title'])
            ax.set_xlabel('Amplitude / Q (K units)')
            ax.set_ylabel('Probability')
        summary_rows.append({'panel': panel['title'], 'samples': 0, 'rows': payload['n_rows'], 'boutons': payload['n_boutons']})
        continue

    fail_curve = np.zeros_like(x_grid, dtype=float)
    nonfail_curve = np.zeros_like(x_grid, dtype=float)
    hist_max = 0.0

    fail_fraction_emp = payload['fail_samples'].size / max(payload['samples'].size, 1)
    nonfail_fraction_emp = payload['nonfail_samples'].size / max(payload['samples'].size, 1)

    if SEPARATE_FAILURES and payload['fail_samples'].size:
        fail_display = payload['fail_samples'] - mu0_shared
        fail_weights = np.full(payload['fail_samples'].size, 1.0 / max(payload['samples'].size, 1), dtype=float)
        fail_hist, _ = np.histogram(fail_display, bins=bin_edges, weights=fail_weights)
        hist_max = max(hist_max, float(np.nanmax(fail_hist)) if fail_hist.size else 0.0)
        ax.hist(fail_display, bins=bin_edges, weights=fail_weights, color='0.75', alpha=0.95, edgecolor='none')
        fail_curve = _normal_pdf(x_grid, 0.0, sigma_B_shared) * bin_width * fail_fraction_emp
        ax.plot(x_grid, fail_curve, color='0.35', lw=1.6)
        plot_samples = payload['nonfail_samples']
        plot_denominator = max(payload['samples'].size, 1)
    else:
        plot_samples = payload['samples']
        plot_denominator = max(plot_samples.size, 1)

    if plot_samples.size:
        plot_weights = np.full(plot_samples.size, 1.0 / plot_denominator, dtype=float)
        plot_hist, _ = np.histogram(plot_samples, bins=bin_edges, weights=plot_weights)
        hist_max = max(hist_max, float(np.nanmax(plot_hist)) if plot_hist.size else 0.0)
        ax.hist(plot_samples, bins=bin_edges, weights=plot_weights, color=panel['color'], alpha=0.45, edgecolor='none')

    if release_fit is not None:
        if SEPARATE_FAILURES:
            support = release_fit['support_pos']
            weights = release_fit['weights_pos_cond']
            scale_fraction = nonfail_fraction_emp
            means = support.astype(float)
        else:
            support = release_fit['support_trim']
            weights = release_fit['weights_trim']
            scale_fraction = 1.0
            means = mu0_shared + support
        sigmas = np.sqrt(np.maximum(sigma_B_shared ** 2 + support * (sigma_Q_shared ** 2), 1e-8))
        for k_val, weight, mean_k, sigma_k in zip(support, weights, means, sigmas):
            component_curve = _normal_pdf(x_grid, mean_k, sigma_k) * bin_width * float(weight) * scale_fraction
            nonfail_curve += component_curve
            ax.plot(x_grid, component_curve, color=panel['color'], lw=1.0, ls='--', alpha=0.75)
        ax.plot(x_grid, fail_curve + nonfail_curve, color='black', lw=2.0)

    y_max = max(hist_max, 1e-6)
    ax.set_ylim(0.0, 1.05 * y_max)

    info_lines = [
        f'rows={payload["n_rows"]} | boutons={payload["n_boutons"]}',
        f'samples={payload["samples"].size}',
        f'mu0={mu0_shared:.2f}K | sigmaB={sigma_B_shared:.2f}K',
        f'sigmaQ={sigma_Q_shared:.2f}K',
    ]
    if release_fit is not None:
        info_lines.append(f'N={release_fit["N_hat"]:.1f} | P={release_fit["P_hat"]:.2f} | P0={release_fit["P0_model"]:.2f}')
    if SEPARATE_FAILURES:
        info_lines.append(f'fail={payload["fail_samples"].size} | nonfail={payload["nonfail_samples"].size}')
    ax.text(0.98, 0.98, '\n'.join(info_lines), transform=ax.transAxes, ha='right', va='top', fontsize=7, bbox=dict(boxstyle='round,pad=0.28', fc='white', ec='0.82', alpha=0.92))

    if 'style_ax' in globals():
        style_ax(ax, 'Amplitude / Q (K units)', 'Probability', panel['title'])
    else:
        ax.set_title(panel['title'])
        ax.set_xlabel('Amplitude / Q (K units)')
        ax.set_ylabel('Probability')

    summary_rows.append({
        'panel': panel['title'],
        'samples': int(payload['samples'].size),
        'rows': int(payload['n_rows']),
        'boutons': int(payload['n_boutons']),
        'failures': int(payload['fail_samples'].size),
        'nonfailures': int(payload['nonfail_samples'].size),
        'N_hat': float(release_fit['N_hat']) if release_fit is not None else np.nan,
        'P_hat': float(release_fit['P_hat']) if release_fit is not None else np.nan,
        'P0_model': float(release_fit['P0_model']) if release_fit is not None else np.nan,
        'y_max_hist': float(y_max),
    })

for ax in axes[0, :]:
    ax.tick_params(labelbottom=False)
for ax in axes.ravel():
    ax.tick_params(labelleft=True)

fig.suptitle(
    'Classical multimodal quantal fits of trial amplitudes\n'
    f'DATA_SOURCE={DATA_SOURCE} | SEPARATE_FAILURES={SEPARATE_FAILURES} | example bouton={example_base_id} | '
    f'mu0={mu0_shared:.2f}K | sigmaB={sigma_B_shared:.2f}K | sigmaQ={sigma_Q_shared:.2f}K',
    fontsize=11,
)
legend_items = [Line2D([0], [0], color='black', lw=2.0, label='Mixture from release statistics')]
if SEPARATE_FAILURES:
    legend_items.append(Line2D([0], [0], color='0.35', lw=1.6, label='Failure Gaussian'))
fig.legend(handles=legend_items, loc='center right', frameon=False)

save_name = f'Fig3_helper_classical_multimodal_single_{DATA_SOURCE}_{"separate_failures" if SEPARATE_FAILURES else "joint"}'
out_pdf = OUTPUT_DIR / f'{save_name}.pdf'
out_png = OUTPUT_DIR / f'{save_name}.png'
fig.savefig(out_pdf, dpi=300, bbox_inches='tight')
fig.savefig(out_png, dpi=300, bbox_inches='tight')
plt.show()

summary_df = pd.DataFrame(summary_rows)
print('=== Classical multimodal quantal fits ===')
print(f'Example bouton: {example_base_id}')
print(f'mu0 (failure mode offset): {mu0_shared:.4f} K')
print(f'sigma_B (failure width): {sigma_B_shared:.4f} K')
print(f'sigma_Q (shared quantal width): {sigma_Q_shared:.4f} K')
print('Model: failures recentered to 0; releases at K = 1, 2, 3, ... with sigma_k^2 = sigma_B^2 + k * sigma_Q^2')
print(summary_df.to_string(index=False))
print(f'Saved to {out_pdf}')
print(f'Saved to {out_png}')

In [ ]:
# =============================================================================
#  Quantal amplitude analysis - parallel-fibre boutons, SF-iGluSnFR imaging
# -----------------------------------------------------------------------------
#  We model each trial's response amplitude as coming from a release of k quanta
#  (k = 0 is a failure). In units of the quantal size q, the k-quantum response
#  is a Gaussian centred on k with a width that grows with k:
#
#        amplitude ~ Normal( mean = k * spacing ,  sd = sqrt(sigma_B^2 + k*sigma_Q^2) )
#
#    sigma_B  = baseline (recording) noise        -> measured on the FAILURES
#    sigma_Q  = quantal variability (per quantum)  -> measured on the RELEASES
#    spacing  = the quantal size in q-units (~1)    -> measured where release is
#               sparse (low Ca), so the single-quantum peak is clean
#
#  Two facts drive the design:
#    1. The failure peak does not sit at 0 - there is a real additive pedestal
#       (Gaussian noise, not NNLS rectification). We measure it from the
#       failure-masked trials, per bouton, and subtract it, so noise is centred.
#    2. We KNOW which trials are failures (the mask), so the failure probability
#       is observed, not fitted. Only the release weights are inferred.
#
#  PULSE_INDEX lets us look at one pulse of the train at a time. The first pulse
#  is free of short-term plasticity / sensor desensitisation; pooling all pulses
#  mixes pulses with different effective quantal size and inflates sigma_Q.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.special import logsumexp

# ----------------------------- choices to make -------------------------------
PULSE_INDEX = 1            # 1 = first pulse (A1), 2 = second, ... ; None or 0 = pool all pulses

BIN_WIDTH_Q = 0.10         # histogram bin width, in q-units
MAX_K       = 20            # highest quantal level drawn
XLIM_Q      = (-1.0, 8.0)  # x-range; left of 0 so the centred Gaussian noise shows its tail
EXAMPLE_BASE_ID = None     # None -> auto-pick the bouton present in the most Ca conditions

CENTER_ON_FAILURES      = True   # subtract each bouton's failure pedestal (recommended)
FAILURE_GAUSSIAN_SOURCE = 'ref'  # 'ref' = low-Ca failures only ; 'all' = failures from every Ca
BASELINE_SIGMA_K        = None   # set this (q-units) if you measured pre-stim noise; fixes sigma_B

Q_REF_CA = '1.5mM'         # low-Ca reference condition used to estimate the quantum
K_REF    = 1               # quanta expected at low Ca; caps the comb so it cannot over-fragment
S_BOUNDS = (0.2, 2.4)      # plausible spacing range; hitting a bound = a q-calibration problem

# robustness thresholds (rarely need changing)
MIN_FAILS_FOR_OWN_PEDESTAL = 50   # a bouton needs this many failures to use its own pedestal
MIN_RELEASES_FOR_COMB      = 30  # below this at the reference, fall back to all-Ca releases
# -----------------------------------------------------------------------------

needed = ['trials_all', 'Q_estimates', 'Global_Q', 'extract_base_name',
          'get_amp_row', 'get_failure_mask_row', 'get_count_row_with_failures',
          'get_calcium_conditions', 'fit_binom_mle', 'get_wt_ca_color', 'OUTPUT_DIR']
absent = [g for g in needed if g not in globals()]
if absent:
    raise RuntimeError('Run the upstream quantal cells first; missing: ' + ', '.join(absent))
if FAILURE_GAUSSIAN_SOURCE not in {'ref', 'all'}:
    raise ValueError("FAILURE_GAUSSIAN_SOURCE must be 'ref' or 'all'.")


# ----------------------------- small numerics --------------------------------
def gaussian_logpdf(values, means, sigmas):
    """log N(values | means, sigmas) for every (value, component) pair."""
    sigmas = np.maximum(sigmas, 1e-9)
    z = (values[:, None] - means[None, :]) / sigmas[None, :]
    return -0.5 * z ** 2 - np.log(sigmas)[None, :] - 0.5 * np.log(2 * np.pi)


def gaussian_pdf(x, mean, sigma):
    sigma = max(float(sigma), 1e-9)
    return np.exp(-0.5 * ((x - mean) / sigma) ** 2) / (sigma * np.sqrt(2 * np.pi))


def robust_std(values, fallback=0.20):
    """MAD-based SD, so a stray failure amplitude doesn't blow up sigma_B."""
    values = np.asarray(values, float)
    values = values[np.isfinite(values)]
    if values.size < 2:
        return fallback
    mad = np.median(np.abs(values - np.median(values)))
    sigma = 1.4826 * mad if mad > 0 else np.std(values, ddof=1)
    return float(np.clip(sigma if sigma > 0 else fallback, 0.03, 1.50))


def take_pulse(values):
    """Select the pulse(s) of interest. PULSE_INDEX is 1-based; 0/None = all."""
    values = np.asarray(values)
    if PULSE_INDEX:                       # 1, 2, 3, ...
        i = PULSE_INDEX - 1
        return values[i:i + 1] if i < values.shape[0] else values[:0]
    return values


# ----------------------------- the two fits ----------------------------------
def fit_quantal_comb(release_amplitudes, baseline_sigma, max_quanta,
                     spacing_bounds=None, n_iter=600, tol=1e-8):
    """Estimate the quantal spacing and quantal width sigma_Q from RELEASE
    amplitudes only (failures excluded - they carry no quantum).

    Peaks live at k*spacing (k = 1..max_quanta) and widen as
    sqrt(sigma_B^2 + k*sigma_Q^2); sigma_B is held fixed (it comes from the
    failures). Weights are free here because we only want the comb geometry.
    """
    amps = np.asarray(release_amplitudes, float)
    amps = amps[np.isfinite(amps)]
    n = amps.size
    k = np.arange(1, max_quanta + 1, dtype=float)
    if n == 0:
        return 1.0, 0.20
    spacing = float(np.clip(1.0, *spacing_bounds)) if spacing_bounds else 1.0
    quantal_sigma = 0.20
    weights = np.full(k.size, 1.0 / k.size)
    previous_loglik = -np.inf

    for _ in range(n_iter):
        means = k * spacing
        sigmas = np.sqrt(baseline_sigma ** 2 + k * quantal_sigma ** 2)

        # E-step: how much each release event belongs to each quantal level
        log_resp = gaussian_logpdf(amps, means, sigmas) + np.log(weights + 1e-300)
        normaliser = logsumexp(log_resp, axis=1)
        loglik = float(normaliser.sum())
        responsibility = np.exp(log_resp - normaliser[:, None])

        # M-step
        weights = responsibility.sum(0) / n

        # spacing: precision-weighted regression of amplitude on quantum number,
        # forced through the origin (mean of level k must equal k*spacing)
        precision = 1.0 / (baseline_sigma ** 2 + k * quantal_sigma ** 2)
        numerator = (responsibility * precision * k * amps[:, None]).sum()
        denominator = (responsibility * precision * k ** 2).sum()
        if denominator > 0:
            spacing = numerator / denominator
        if spacing_bounds:
            spacing = float(np.clip(spacing, spacing_bounds[0], spacing_bounds[1]))

        # quantal width: leftover spread after removing the baseline noise, since
        # E[(a - k*spacing)^2] = sigma_B^2 + k*sigma_Q^2
        means = k * spacing
        spread = (responsibility * (amps[:, None] - means) ** 2).sum()
        weighted_k = (responsibility * k).sum()
        if weighted_k > 0:
            quantal_sigma = np.sqrt(max((spread - baseline_sigma ** 2 * n) / weighted_k, 1e-6))

        if loglik - previous_loglik < tol * max(1.0, abs(previous_loglik)):
            break
        previous_loglik = loglik

    return spacing, quantal_sigma


def fit_release_weights(release_amplitudes, means, sigmas, n_iter=500):
    """Probability of each quantal level (k = 1..K) GIVEN that release happened,
    estimated from the release amplitudes. Returns a length-K vector summing to 1."""
    amps = np.asarray(release_amplitudes, float)
    amps = amps[np.isfinite(amps)]
    n_levels = len(means) - 1
    if amps.size == 0:
        return np.zeros(n_levels)
    log_component = gaussian_logpdf(amps, means[1:], sigmas[1:])   # k = 1..K
    weights = np.full(n_levels, 1.0 / n_levels)
    for _ in range(n_iter):
        log_resp = log_component + np.log(weights + 1e-300)
        normaliser = logsumexp(log_resp, axis=1)
        updated = np.exp(log_resp - normaliser[:, None]).mean(0)
        if np.abs(updated - weights).max() < 1e-10:
            weights = updated
            break
        weights = updated
    return weights


# ----------------------------- group trials by Ca ----------------------------
CA_CONDITIONS = [('1.5mM', '1.5 mM', get_wt_ca_color('1.5mM')),
                 ('2.5mM', '2.5 mM', get_wt_ca_color('2.5mM')),
                 ('4mM',   '4 mM',   get_wt_ca_color('4mM'))]


def condition_labels(ca_key):
    labels = set()
    for freq in ('20Hz', '50Hz'):
        try:
            labels.update(str(name).strip() for name in get_calcium_conditions(ca_key, freq))
        except Exception:
            pass
    return labels


trials = trials_all.copy()
trials['condition'] = trials['condition'].astype(str).str.strip()
trials['base_id'] = trials['file'].astype(str).map(lambda s: extract_base_name(str(s).strip()))
trials['CaKey'] = None
for ca_key, _label, _colour in CA_CONDITIONS:
    names = condition_labels(ca_key)
    if names:
        trials.loc[trials['condition'].isin(names), 'CaKey'] = ca_key
trials = trials[trials['CaKey'].isin([c for c, _l, _col in CA_CONDITIONS])].copy()
if trials.empty:
    raise RuntimeError('No trials matched the calcium conditions.')


def quantal_size_for(bouton):
    q = Q_estimates.get(bouton, np.nan)
    return float(q) if (np.isfinite(q) and q > 0) else float(Global_Q)


# --- pedestal: each bouton's baseline offset, read off its own failures ------
# (The pedestal is a recording artefact, independent of calcium, so we pool a
#  bouton's failures across all conditions to estimate it.)
failures_by_bouton = {}
for _, trial in trials.iterrows():
    q = quantal_size_for(str(trial['base_id']))
    amp = take_pulse(get_amp_row(trial)).astype(float)
    is_failure = take_pulse(np.asarray(get_failure_mask_row(trial)[0], bool))
    finite = np.isfinite(amp)
    failure_amps = (amp[finite] / q)[is_failure[finite]]
    if failure_amps.size:
        failures_by_bouton.setdefault(str(trial['base_id']), []).append(failure_amps)

pooled_failures = [np.concatenate(v) for v in failures_by_bouton.values()]
global_pedestal = float(np.median(np.concatenate(pooled_failures))) if pooled_failures else 0.0
pedestal_by_bouton = {}
for bouton, chunks in failures_by_bouton.items():
    own = np.concatenate(chunks)
    pedestal_by_bouton[bouton] = (float(np.median(own))
                                  if own.size >= MIN_FAILS_FOR_OWN_PEDESTAL else global_pedestal)
if not CENTER_ON_FAILURES:
    pedestal_by_bouton, global_pedestal = {}, 0.0


def collect_amplitudes(panel_trials):
    """All amplitudes (in q-units, pedestal removed), the failure mask, and the
    integer quantal counts for one panel."""
    amps_all, fail_all, count_all = [], [], []
    for _, trial in panel_trials.iterrows():
        bouton = str(trial['base_id'])
        q = quantal_size_for(bouton)
        pedestal = pedestal_by_bouton.get(bouton, global_pedestal) if CENTER_ON_FAILURES else 0.0
        amp = take_pulse(get_amp_row(trial)).astype(float)
        is_failure = take_pulse(np.asarray(get_failure_mask_row(trial)[0], bool))
        count = take_pulse(np.asarray(get_count_row_with_failures(trial, q), float))
        valid = np.isfinite(amp) & np.isfinite(count)
        if not valid.any():
            continue
        amps_all.append(amp[valid] / q - pedestal)
        fail_all.append(is_failure[valid])
        count_all.append(np.rint(count[valid]).astype(int))

    if not amps_all:
        empty = np.array([], float)
        return dict(amplitudes=empty, failure_amps=empty, release_amps=empty,
                    counts=np.array([], int), n_rows=len(panel_trials),
                    n_boutons=panel_trials['base_id'].nunique() if len(panel_trials) else 0)

    amplitudes = np.concatenate(amps_all)
    failure_mask = np.concatenate(fail_all)
    counts = np.concatenate(count_all)
    return dict(amplitudes=amplitudes,
                failure_amps=amplitudes[failure_mask],
                release_amps=amplitudes[~failure_mask],
                counts=counts,
                n_rows=int(len(panel_trials)),
                n_boutons=int(panel_trials['base_id'].nunique()))


# pick the example bouton (best calcium coverage) unless one is given
if EXAMPLE_BASE_ID is not None:
    example_bouton = extract_base_name(str(EXAMPLE_BASE_ID).strip())
else:
    coverage = (trials.groupby(['base_id', 'CaKey']).size().unstack(fill_value=0)
                .reindex(columns=[c for c, _l, _col in CA_CONDITIONS], fill_value=0))
    coverage.index = coverage.index.astype(str)
    ranked = pd.DataFrame({'n_conditions': (coverage > 0).sum(axis=1),
                           'n_rows': coverage.sum(axis=1)},
                          index=coverage.index).sort_index()
    ranked = ranked.sort_values(['n_conditions', 'n_rows'], ascending=[False, False], kind='stable')
    example_bouton = str(ranked.index[0])

# build the six panels: example bouton (top) and pooled (bottom), per calcium
panels = []
for kind in ('example', 'pooled'):
    for ca_key, ca_label, colour in CA_CONDITIONS:
        if kind == 'example':
            subset = trials[(trials['CaKey'] == ca_key) & (trials['base_id'] == example_bouton)]
            title = f'{ca_label} example - {example_bouton}'
        else:
            subset = trials[trials['CaKey'] == ca_key]
            title = f'{ca_label} pooled'
        panels.append(dict(kind=kind, ca_key=ca_key, colour=colour, title=title,
                           data=collect_amplitudes(subset.copy())))


# ----------------------------- noise width sigma_B ---------------------------
reference = next((p for p in panels if p['kind'] == 'pooled' and p['ca_key'] == Q_REF_CA), None)
reference_failures = reference['data']['failure_amps'] if reference else np.array([], float)
reference_releases = reference['data']['release_amps'] if reference else np.array([], float)

all_failures = np.concatenate([p['data']['failure_amps'] for p in panels
                               if p['kind'] == 'pooled' and p['data']['failure_amps'].size] or [np.array([], float)])
all_releases = np.concatenate([p['data']['release_amps'] for p in panels
                               if p['kind'] == 'pooled' and p['data']['release_amps'].size] or [np.array([], float)])

if FAILURE_GAUSSIAN_SOURCE == 'all':
    failures_for_sigmaB = all_failures
else:
    failures_for_sigmaB = reference_failures if reference_failures.size >= 10 else all_failures

if BASELINE_SIGMA_K:
    baseline_sigma = float(BASELINE_SIGMA_K)
else:
    baseline_sigma = robust_std(failures_for_sigmaB)
failure_centre = float(np.mean(failures_for_sigmaB)) if failures_for_sigmaB.size else 0.0


# ----------------------------- the quantum (low Ca) --------------------------
if reference_releases.size >= MIN_RELEASES_FOR_COMB:
    comb_releases, comb_source = reference_releases, f'reference {Q_REF_CA}'
else:
    comb_releases, comb_source = all_releases, 'all-Ca (fallback)'

spacing, quantal_sigma = fit_quantal_comb(comb_releases, baseline_sigma, K_REF, spacing_bounds=S_BOUNDS)
spacing_at_bound = abs(spacing - S_BOUNDS[0]) < 1e-6 or abs(spacing - S_BOUNDS[1]) < 1e-6

k_levels = np.arange(MAX_K + 1, dtype=float)
peak_means = k_levels * spacing
peak_sigmas = np.sqrt(baseline_sigma ** 2 + k_levels * quantal_sigma ** 2)


# ----------------------------- figure ----------------------------------------
bin_edges = np.arange(XLIM_Q[0], XLIM_Q[1] + BIN_WIDTH_Q, BIN_WIDTH_Q)
bin_width = float(np.median(np.diff(bin_edges)))
x = np.linspace(XLIM_Q[0], XLIM_Q[1], 1600)
xlabel = '(Amp - pedestal) / Q (K units)' if CENTER_ON_FAILURES else 'Amplitude / Q (K units)'

fig, axes = plt.subplots(2, 3, figsize=(14, 7.8), sharex=True, sharey=False, constrained_layout=True)
summary = []

for ax, panel in zip(axes.ravel(), panels):
    data = panel['data']
    ax.set_xlim(*XLIM_Q)
    ax.axvline(0.0, color='0.85', lw=0.8, zorder=0)

    def label_axes():
        if 'style_ax' in globals():
            style_ax(ax, xlabel, 'Probability', panel['title'])
        else:
            ax.set_title(panel['title']); ax.set_xlabel(xlabel); ax.set_ylabel('Probability')

    if data['amplitudes'].size == 0:
        ax.text(0.5, 0.5, 'Not available', transform=ax.transAxes, ha='center', va='center')
        label_axes()
        summary.append(dict(panel=panel['title'], n=0, rows=data['n_rows'], boutons=data['n_boutons']))
        continue

    n_total = data['amplitudes'].size
    n_failures = data['failure_amps'].size

    # failure probability is OBSERVED (we have the mask); release weights are fitted
    failure_rate = n_failures / n_total
    release_weights = fit_release_weights(data['release_amps'], peak_means, peak_sigmas)
    weights = np.empty(MAX_K + 1)
    weights[0] = failure_rate
    weights[1:] = (1.0 - failure_rate) * release_weights
    mean_quanta_given_release = float((k_levels[1:] * release_weights).sum())

    # histograms: failures (grey) and releases (colour), both on the same scale
    per_sample = 1.0 / n_total
    grey = np.full(n_failures, per_sample)
    colour = np.full(data['release_amps'].size, per_sample)
    h_fail, _ = np.histogram(data['failure_amps'], bins=bin_edges, weights=grey)
    h_rel, _ = np.histogram(data['release_amps'], bins=bin_edges, weights=colour)
    if n_failures:
        ax.hist(data['failure_amps'], bins=bin_edges, weights=grey, color='0.75', edgecolor='none')
    ax.hist(data['release_amps'], bins=bin_edges, weights=colour, color=panel['colour'], alpha=0.45, edgecolor='none')

    # model curves
    failure_curve = (failure_rate * gaussian_pdf(x, peak_means[0], peak_sigmas[0]) * bin_width
                     if n_failures else np.zeros_like(x))
    if n_failures:
        ax.plot(x, failure_curve, color='0.35', lw=1.6)
    release_curve = np.zeros_like(x)
    for k in range(1, MAX_K + 1):
        component = weights[k] * gaussian_pdf(x, peak_means[k], peak_sigmas[k]) * bin_width
        release_curve += component
        if weights[k] > 1e-4:
            ax.plot(x, component, color=panel['colour'], lw=1.0, ls='--', alpha=0.75)
    fit_curve = failure_curve + release_curve
    ax.plot(x, fit_curve, color='black', lw=2.0)
    y_max = max(h_fail.max(initial=0), h_rel.max(initial=0), failure_curve.max(initial=0), release_curve.max(initial=0), fit_curve.max(initial=0), 1e-6)
    ax.set_ylim(0.0, 1.05 * y_max)

    try:
        n_hat, p_hat, _ = fit_binom_mle(data['counts'], int(max(globals().get('N_MAX_D', 20), MAX_K)))
    except Exception:
        n_hat, p_hat = np.nan, np.nan

    ax.text(0.98, 0.98, '\n'.join([
        f"rows={data['n_rows']} | boutons={data['n_boutons']}",
        f"n={n_total} (fail={n_failures}, rel={data['release_amps'].size})",
        f"sigmaB={baseline_sigma:.2f} ({FAILURE_GAUSSIAN_SOURCE}) | sigmaQ={quantal_sigma:.2f}",
        f"spacing={spacing:.2f}",
        f"fail_rate={failure_rate:.2f} | <k|rel>={mean_quanta_given_release:.2f}",
        f"N={n_hat:.0f} | P={p_hat:.2f} (binom, annot.)",
    ]), transform=ax.transAxes, ha='right', va='top', fontsize=7,
        bbox=dict(boxstyle='round,pad=0.28', fc='white', ec='0.82', alpha=0.92))

    label_axes()
    summary.append(dict(panel=panel['title'], n=n_total, rows=data['n_rows'], boutons=data['n_boutons'],
                        failures=n_failures, releases=int(data['release_amps'].size),
                        fail_rate=failure_rate, mean_quanta_given_release=mean_quanta_given_release,
                        N_hat=float(n_hat), P_hat=float(p_hat)))

for ax in axes[0, :]:
    ax.tick_params(labelbottom=False)
for ax in axes.ravel():
    ax.tick_params(labelleft=True)

pulse_label = 'all pulses' if not PULSE_INDEX else f'pulse {PULSE_INDEX}'
fig.suptitle(
    f'Quantal mixture fits ({pulse_label}; Gaussian noise, pedestal removed)\n'
    f'q from {comb_source} | pedestal global={global_pedestal:.2f} | '
    f'sigmaB={baseline_sigma:.2f} [{FAILURE_GAUSSIAN_SOURCE}] | sigmaQ={quantal_sigma:.2f} | spacing={spacing:.2f}'
    + ('  [spacing AT BOUND - check Q_estimates]' if spacing_at_bound else ''),
    fontsize=11)
fig.legend(handles=[Line2D([0], [0], color='black', lw=2.0, label='EM mixture'),
                    Line2D([0], [0], color=CA_CONDITIONS[0][2], lw=1.0, ls='--', label='Quantal components'),
                    Line2D([0], [0], color='0.35', lw=1.6, label='Failure (Gaussian, centred)')],
           loc='lower center', ncol=3, frameon=False, bbox_to_anchor=(0.5, -0.03))

stem = f'Fig3_quantal_{pulse_label.replace(" ", "")}_{FAILURE_GAUSSIAN_SOURCE}_qref{Q_REF_CA}'
fig.savefig(OUTPUT_DIR / f'{stem}.pdf', dpi=300, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / f'{stem}.png', dpi=300, bbox_inches='tight')
plt.show()

print('=== Quantal mixture fits ===')
print(f'Pulse                 : {pulse_label}')
print(f'Example bouton        : {example_bouton}')
print(f'Pedestal removed      : {CENTER_ON_FAILURES} | global = {global_pedestal:.4f} q '
      f'| {len(pedestal_by_bouton)} boutons with own pedestal')
print(f'Failure Gaussian      : source={FAILURE_GAUSSIAN_SOURCE}, n_fail={failures_for_sigmaB.size}, '
      f'centre={failure_centre:.4f} q (should be ~0), sigmaB={baseline_sigma:.4f} q')
print(f'Quantum (low Ca)      : from {comb_source} (n={comb_releases.size}) -> '
      f'spacing={spacing:.4f}{"  [AT BOUND]" if spacing_at_bound else ""}, sigmaQ={quantal_sigma:.4f} q')
print('Failure rate is observed (mask), not fitted. Model: mean_k = k*spacing, var_k = sigmaB^2 + k*sigmaQ^2')
print(pd.DataFrame(summary).to_string(index=False))
print(f'Saved {stem}.pdf / .png to {OUTPUT_DIR}')

In [ ]:
# === Helper — Bootstrap MLE config + N/P arrays ===
# Build the shared bootstrap-MLE arrays of apparent N and P used by the WT quantal analyses.

USE_BOOTSTRAP = True          # bootstrap for cleaner estimates
N_BOOT        = 200           # number of resamples
N_MAX_D       = 20            # max N in grid search
MIN_TRIALS_D  = 5
STIM_SHOW     = [1, 2, 5, 10]  # stimulus positions to map
# Override colour axis (None = auto from data)
N_VLIM        = (1, 8)       # (vmin, vmax) for N maps
P_VLIM        = (0, 1.0)     # (vmin, vmax) for P maps
# ╚##############################################################╝

Q_D = Global_Q

_pca_ids = PCA_Data_WT_Pooled_clustered['ID'].astype(str).str.strip().values

# == Build count dict keyed by extract_base_name ====================
# Event failures are defined per pulse from the lower of corrected /
# uncorrected amplitudes compared against the row-specific thr_shared.
# Non-failure amplitudes are then rounded to quantal counts.
td_D = defaultdict(list)
for _, r in trials_all[trials_all['condition'].isin(conditions_2_5_20Hz)].iterrows():
    td_D[extract_base_name(str(r['file']).strip())].append(get_count_row_with_failures(r, Q_D))

# == Precompute valid count matrices ==============================
_pca_ebn_D = [extract_base_name(s) for s in _pca_ids]
n_pca = len(_pca_ids)
bouton_counts_D = {}
for i in range(n_pca):
    bid = _pca_ebn_D[i]
    if bid not in td_D: continue
    qc = np.array(td_D[bid])
    ok = np.all(np.isfinite(qc), axis=1)
    qc = qc[ok]
    if len(qc) >= MIN_TRIALS_D:
        bouton_counts_D[i] = qc.astype(int)

print(f"Boutons with ≥{MIN_TRIALS_D} trials: {len(bouton_counts_D)}")

# == Fit: point or bootstrap =====================================
D_N = np.full((n_pca, N_STIM), np.nan)
D_P = np.full((n_pca, N_STIM), np.nan)

from concurrent.futures import ThreadPoolExecutor
import os as _os
_n_workers_D = max(1, min((_os.cpu_count() or 4), 8))

def _fit_bouton_bootstrap_D(item):
    """Bootstrap (N, P) per stim for a single bouton. Thread-safe."""
    i, counts = item
    ntr = len(counts)
    rng_l = np.random.default_rng(42 + int(i))
    boot_N = np.full((N_STIM, N_BOOT), np.nan)
    boot_P = np.full((N_STIM, N_BOOT), np.nan)
    for b in range(N_BOOT):
        qc_b = counts[rng_l.integers(0, ntr, ntr)]
        for k in range(N_STIM):
            qc = qc_b[:, k].astype(int)
            Nf, Pf, _ll = fit_binom_mle(qc, N_MAX_D)
            if np.isfinite(Nf) and Nf < N_MAX_D:
                boot_N[k, b] = Nf
                boot_P[k, b] = Pf
    out_N = np.full(N_STIM, np.nan)
    out_P = np.full(N_STIM, np.nan)
    for k in range(N_STIM):
        if np.isfinite(boot_N[k]).sum() >= N_BOOT * 0.5:
            out_N[k] = np.nanmedian(boot_N[k])
            out_P[k] = np.nanmedian(boot_P[k])
    return i, out_N, out_P

def _fit_bouton_point_D(item):
    """Point-MLE (N, P) per stim for a single bouton. Thread-safe."""
    i, counts = item
    out_N = np.full(N_STIM, np.nan)
    out_P = np.full(N_STIM, np.nan)
    for k in range(N_STIM):
        qc = counts[:, k].astype(int)
        Nf, Pf, _ll = fit_binom_mle(qc, N_MAX_D)
        if np.isfinite(Nf) and Nf < N_MAX_D:
            out_N[k] = Nf
            out_P[k] = Pf
    return i, out_N, out_P

_worker_D = _fit_bouton_bootstrap_D if USE_BOOTSTRAP else _fit_bouton_point_D
with ThreadPoolExecutor(max_workers=_n_workers_D) as _ex_D:
    for i, out_N, out_P in _ex_D.map(_worker_D, list(bouton_counts_D.items())):
        D_N[i, :] = out_N
        D_P[i, :] = out_P

ok_D = np.isfinite(D_N[:, 0])
print(f"Valid (stim 1): {ok_D.sum()} | N med={np.nanmedian(D_N[ok_D,0]):.1f} | P med={np.nanmedian(D_P[ok_D,0]):.3f}")
print(f"MLE failure rule: per event, min(corrected, uncorrected) < {THR_SHARED_COL_MLE} -> count 0")


In [ ]:
# === Helper ? Review WT 20 Hz example candidates ===
# Review one bouton per row with trial overlays, count heatmaps, and bouton-level binomial summaries.

MIN_TRIALS_EXAMPLE_REVIEW = 10
EXAMPLE_REVIEW_TRACE_XLIM = (0.8, 1.6)
EXAMPLE_REVIEW_TRACE_ALPHA = 0.30
EXAMPLE_REVIEW_PULSE = 1
EXAMPLE_REVIEW_N_BINS = 10
EXAMPLE_REVIEW_ROWS_PER_PAGE = 8
EXAMPLE_REVIEW_HEATMAP_CAP = 10
ENABLE_FIG3B_CANDIDATE_REVIEW_HELPER = False


def _fiber_name_from_bouton_id_local(bouton_id):
    base = extract_base_name(str(bouton_id).strip())
    return re.sub(r'_bouton\d+$', '', base, flags=re.IGNORECASE)


def _collect_single_trial_rows_local(bouton_id, condition_names):
    rows = RAW_TRACES_DF.copy()
    if 'BaseID' not in rows.columns and 'ID' in rows.columns:
        rows['BaseID'] = rows['ID'].map(_normalize_bouton_id)
    base_id = extract_base_name(str(bouton_id).strip())
    cond_priority = {cond: idx for idx, cond in enumerate(condition_names)}
    rows = rows[rows['BaseID'].astype(str) == base_id].copy()
    rows = rows[rows['Condition'].isin(condition_names)].copy()
    if rows.empty:
        return rows
    rows['_cond_priority'] = rows['Condition'].map(lambda cond: cond_priority.get(cond, len(cond_priority)))
    sort_cols = ['_cond_priority']
    for optional_col in ['Recording', 'Trial', 'Sweep', 'file', 'ID']:
        if optional_col in rows.columns:
            sort_cols.append(optional_col)
    return rows.sort_values(sort_cols).drop(columns=['_cond_priority']).reset_index(drop=True)


eligible_example_candidates = []
if not ENABLE_FIG3B_CANDIDATE_REVIEW_HELPER:
    print('Skipping WT 20 Hz example-candidate review helper: ENABLE_FIG3B_CANDIDATE_REVIEW_HELPER = False')
else:
    for i, counts in sorted(bouton_counts_D.items()):
        n_repeats = int(len(counts))
        if n_repeats < MIN_TRIALS_EXAMPLE_REVIEW:
            continue
        bouton_id = str(_pca_ids[i]).strip()
        base_bouton_id = extract_base_name(bouton_id)
        trace_stats = compute_trace_stats(
            trace_ids=[bouton_id],
            condition_names=WT_2_5_20HZ_CONDITIONS,
            source=TRACE_MEAN_SOURCE,
        )
        single_trial_rows = _collect_single_trial_rows_local(base_bouton_id, WT_2_5_20HZ_CONDITIONS)
        fit_N = np.full(N_STIM, np.nan)
        fit_P = np.full(N_STIM, np.nan)
        fit_LL = np.full(N_STIM, np.nan)
        for k in range(N_STIM):
            n_hat_k, p_hat_k, ll_hat_k = fit_binom_mle(np.asarray(counts[:, k], dtype=int), N_MAX_D)
            fit_N[k] = n_hat_k
            fit_P[k] = p_hat_k
            fit_LL[k] = ll_hat_k

        time_vals = np.asarray(trace_stats['time'], float)
        avg_vals = np.asarray(trace_stats['average'], float)
        review_mask = np.isfinite(time_vals) & (time_vals >= EXAMPLE_REVIEW_TRACE_XLIM[0]) & (time_vals <= EXAMPLE_REVIEW_TRACE_XLIM[1])
        peak_df_f = float(np.nanmax(avg_vals[review_mask])) if np.any(review_mask) else float(np.nanmax(avg_vals))
        expected_counts = fit_N * fit_P
        observed_counts = np.nanmean(np.asarray(counts, float), axis=0)
        fit_rmse = float(np.sqrt(np.nanmean((observed_counts - expected_counts) ** 2))) if np.any(np.isfinite(expected_counts)) else np.nan
        review_pulse = int(np.clip(EXAMPLE_REVIEW_PULSE, 1, counts.shape[1]))
        pulse_idx = review_pulse - 1
        pulse_counts = np.asarray(counts[:, pulse_idx], dtype=int)
        pulse_n_hat = fit_N[pulse_idx]
        pulse_p_hat = fit_P[pulse_idx]
        k_support = np.arange(EXAMPLE_REVIEW_N_BINS, dtype=int)
        obs_freq = np.array([(pulse_counts == kk).mean() for kk in k_support], dtype=float)
        fit_pmf = np.array([
            np.exp(log_binom_pmf(kk, int(round(pulse_n_hat)), float(pulse_p_hat)))
            if np.isfinite(pulse_n_hat) and np.isfinite(pulse_p_hat) and kk <= int(round(pulse_n_hat)) else 0.0
            for kk in k_support
        ], dtype=float)

        eligible_example_candidates.append({
            'pca_index': int(i),
            'bouton_id': bouton_id,
            'base_bouton_id': base_bouton_id,
            'fiber_name': _fiber_name_from_bouton_id_local(bouton_id),
            'n_repeats': n_repeats,
            'trace_stats': trace_stats,
            'single_trial_rows': single_trial_rows,
            'counts': np.asarray(counts, dtype=int),
            'fit_N': fit_N,
            'fit_P': fit_P,
            'fit_LL': fit_LL,
            'peak_df_f': peak_df_f,
            'fit_rmse': fit_rmse,
            'review_pulse': review_pulse,
            'pulse_n_hat': pulse_n_hat,
            'pulse_p_hat': pulse_p_hat,
            'obs_freq': obs_freq,
            'fit_pmf': fit_pmf,
            'k_support': k_support,
        })

    if not eligible_example_candidates:
        raise RuntimeError(
            f'No WT 20 Hz bouton has at least {MIN_TRIALS_EXAMPLE_REVIEW} valid repeats for Fig 3b review.'
        )

    eligible_example_candidates = sorted(
        eligible_example_candidates,
        key=lambda item: (
            -float(item['pulse_n_hat']) if np.isfinite(item['pulse_n_hat']) else np.inf,
            -float(item['pulse_p_hat']) if np.isfinite(item['pulse_p_hat']) else np.inf,
            -item['n_repeats'],
            -item['peak_df_f'],
            item['fit_rmse'] if np.isfinite(item['fit_rmse']) else np.inf,
            item['base_bouton_id'],
        )
    )

    review_stim_times = [1.0 + 0.05 * i for i in range(N_STIM)]
    heatmap_cap = int(EXAMPLE_REVIEW_HEATMAP_CAP)
    heatmap_bounds = np.arange(-0.5, heatmap_cap + 1.5, 1)
    heatmap_colors = ['#d9d9d9'] + [plt.cm.rainbow(x) for x in np.linspace(0.02, 0.98, heatmap_cap)]
    heatmap_cmap = ListedColormap(heatmap_colors)
    heatmap_norm = BoundaryNorm(heatmap_bounds, heatmap_cmap.N)
    summary_rows = []
    rows_per_page = max(1, int(EXAMPLE_REVIEW_ROWS_PER_PAGE))
    page_paths = []
    n_pages = int(np.ceil(len(eligible_example_candidates) / rows_per_page))
    for page_idx in range(n_pages):
        start = page_idx * rows_per_page
        stop = min(len(eligible_example_candidates), (page_idx + 1) * rows_per_page)
        page_items = eligible_example_candidates[start:stop]
        fig_review, axes_review = plt.subplots(
            len(page_items),
            3,
            figsize=(13.5, max(2.4 * len(page_items), 6.0)),
            sharex='col',
            squeeze=False,
            gridspec_kw={'width_ratios': [1.75, 1.0, 1.15]},
        )
        heatmap_im = None
        for local_idx, item in enumerate(page_items, start=1):
            idx = start + local_idx
            ax_trace, ax_counts, ax_fit = axes_review[local_idx - 1]
            stats = item['trace_stats']
            counts = item['counts']
            fit_N = item['fit_N']
            fit_P = item['fit_P']
            fit_LL = item['fit_LL']
            time_vals = np.asarray(stats['time'], float)
            avg_vals = np.asarray(stats['average'], float)
            sem_vals = np.asarray(stats['sem'], float) if stats.get('sem') is not None else np.full_like(avg_vals, np.nan)
            review_pulse = item['review_pulse']
            pulse_n_hat = item['pulse_n_hat']
            pulse_p_hat = item['pulse_p_hat']
            obs_freq = item['obs_freq']
            fit_pmf = item['fit_pmf']
            k_support = item['k_support']

            trial_bounds = [np.array([0.0])]
            for _, trial_row in item['single_trial_rows'].iterrows():
                trial_t = np.asarray(_trace_row_time(trial_row, aligned=True), float)
                trial_y = np.asarray(trial_row['Avg'], float)
                keep = np.isfinite(trial_t) & np.isfinite(trial_y) & (trial_t >= EXAMPLE_REVIEW_TRACE_XLIM[0]) & (trial_t <= EXAMPLE_REVIEW_TRACE_XLIM[1])
                if not np.any(keep):
                    continue
                trial_bounds.append(trial_y[keep])
                ax_trace.plot(trial_t[keep], trial_y[keep], color='0.55', alpha=EXAMPLE_REVIEW_TRACE_ALPHA, lw=0.8)

            mean_keep = np.isfinite(time_vals) & np.isfinite(avg_vals) & (time_vals >= EXAMPLE_REVIEW_TRACE_XLIM[0]) & (time_vals <= EXAMPLE_REVIEW_TRACE_XLIM[1])
            if np.any(mean_keep):
                ax_trace.plot(time_vals[mean_keep], avg_vals[mean_keep], color='black', lw=1.5, label='Mean')
                trial_bounds.append(avg_vals[mean_keep])
                if np.any(np.isfinite(sem_vals[mean_keep])):
                    ax_trace.fill_between(
                        time_vals[mean_keep],
                        avg_vals[mean_keep] - sem_vals[mean_keep],
                        avg_vals[mean_keep] + sem_vals[mean_keep],
                        color='black',
                        alpha=0.10,
                        linewidth=0.0,
                    )
            ax_trace.axhline(0.0, color='0.55', lw=0.8, ls=':')
            if 'add_stimulus_ticks' in globals():
                add_stimulus_ticks(ax_trace, review_stim_times, mode='top', linewidth=1.0)
            else:
                for stim_time in review_stim_times:
                    ax_trace.axvline(stim_time, color='0.65', lw=0.6, ls=':')
            trace_bounds = np.concatenate([vals[np.isfinite(vals)] for vals in trial_bounds if np.any(np.isfinite(vals))])
            y_min = float(np.nanmin(trace_bounds))
            y_max = float(np.nanmax(trace_bounds))
            y_pad = max(0.05, 0.08 * (y_max - y_min if y_max > y_min else 1.0))
            ax_trace.set_xlim(*EXAMPLE_REVIEW_TRACE_XLIM)
            ax_trace.set_ylim(y_min - 0.10 * y_pad, y_max + y_pad)
            style_ax(
                ax_trace,
                'Time (s)' if local_idx == len(page_items) else None,
                '?F/F',
                f"{idx}. {item['fiber_name']} | {item['base_bouton_id']} | n={item['n_repeats']}"
            )
            if local_idx != len(page_items):
                ax_trace.tick_params(labelbottom=False)

            counts_plot = np.clip(counts.T, 0, heatmap_cap)
            heatmap_im = ax_counts.imshow(counts_plot, aspect='auto', origin='lower', cmap=heatmap_cmap, norm=heatmap_norm)
            style_ax(ax_counts, 'Trial' if local_idx == len(page_items) else None, 'Stimulus', 'Observed quantal counts')
            ax_counts.set_yticks(np.arange(N_STIM))
            ax_counts.set_yticklabels(np.arange(1, N_STIM + 1))
            if local_idx != len(page_items):
                ax_counts.tick_params(labelbottom=False)

            ax_fit.bar(k_support - 0.18, obs_freq, width=0.36, color='0.70', label='Observed')
            ax_fit.bar(k_support + 0.18, fit_pmf, width=0.36, color='#2ca02c', alpha=0.85, label='Binomial fit')
            fit_upper = np.nanmax(np.r_[obs_freq, fit_pmf, [0.05]])
            ax_fit.set_xlim(-0.5, EXAMPLE_REVIEW_N_BINS - 0.5)
            ax_fit.set_xticks(k_support)
            ax_fit.set_ylim(0, fit_upper + max(0.02, 0.10 * fit_upper))
            fit_title = (
                f"Stim {review_pulse}: N={int(round(pulse_n_hat))}, P={pulse_p_hat:.2f}"
                if np.isfinite(pulse_n_hat) and np.isfinite(pulse_p_hat) else f'Stim {review_pulse}: fit unavailable'
            )
            style_ax(ax_fit, 'Quantal count k' if local_idx == len(page_items) else None, 'P(k)', fit_title)
            if local_idx != len(page_items):
                ax_fit.tick_params(labelbottom=False)
            fit_note = f"RMSE={item['fit_rmse']:.2f}" if np.isfinite(item['fit_rmse']) else 'RMSE=nan'
            ax_fit.text(0.98, 0.96, fit_note, transform=ax_fit.transAxes, ha='right', va='top', fontsize=6)
            if page_idx == 0 and local_idx == 1:
                add_legend(ax_fit, loc='upper left', frameon=False)

            summary_rows.append({
                'rank': idx,
                'fiber_name': item['fiber_name'],
                'bouton_id': item['base_bouton_id'],
                'n_repeats': item['n_repeats'],
                'peak_df_f_0p8_1p6s': item['peak_df_f'],
                'fit_rmse': item['fit_rmse'],
                'review_pulse': review_pulse,
                'pulse_fit_N': float(pulse_n_hat) if np.isfinite(pulse_n_hat) else np.nan,
                'pulse_fit_P': float(pulse_p_hat) if np.isfinite(pulse_p_hat) else np.nan,
                'mean_fit_N': float(np.nanmean(fit_N)),
                'mean_fit_P': float(np.nanmean(fit_P)),
            })

        if heatmap_im is not None:
            cbar = fig_review.colorbar(heatmap_im, ax=axes_review[:, 1], fraction=0.018, pad=0.02, label='Quantal count')
            cbar.set_ticks(np.arange(0, heatmap_cap + 1))
        fig_review.tight_layout(h_pad=1.0, w_pad=1.0)
        output_review_page = OUTPUT_DIR / f'Fig3b_helper_wt20hz_candidate_review_triptych_page{page_idx + 1:02d}.png'
        fig_review.savefig(output_review_page, dpi=200, bbox_inches='tight')
        page_paths.append(output_review_page)
        plt.show()

    output_review = page_paths[0] if page_paths else None

    example_review_df = pd.DataFrame(summary_rows)
    display(example_review_df)
    print(f"WT 20 Hz candidate review: {len(eligible_example_candidates)} boutons with at least {MIN_TRIALS_EXAMPLE_REVIEW} repeats")
    print(f"Right panel uses observed vs binomial histogram for stimulus {EXAMPLE_REVIEW_PULSE}")
    print(f"Sorted by stimulus {EXAMPLE_REVIEW_PULSE} fitted N, then fitted P, both descending")
    print(f"Saved helper review pages: {len(page_paths)}")
    for page_path in page_paths:
        print(f"  - {page_path}")


In [ ]:
# === Fig 3b ? Selected fibres: smoothed binomial evolution across events ===
# Pick a few representative WT 20 Hz / 2.5 mM boutons and show how the smoothed binomial fit changes across the train.

FIG3B_BINOMIAL_SUPPORT_MAX = 10
FIG3B_BINOMIAL_SMOOTH_WINDOW = 5
FIG3B_BINOMIAL_SMOOTH_POLYORDER = 2
ENABLE_FIG3B_SELECTED_FIBRES = False


def _smooth_binomial_trajectory_local(values, *, clip=None, round_int=False):
    arr = np.asarray(values, dtype=float).copy()
    if arr.ndim != 1:
        arr = np.ravel(arr)
    finite = np.isfinite(arr)
    if finite.sum() == 0:
        return arr
    if finite.sum() == 1:
        arr[~finite] = arr[finite][0]
    elif finite.sum() < len(arr):
        x = np.arange(len(arr), dtype=float)
        arr[~finite] = np.interp(x[~finite], x[finite], arr[finite])
    win = min(int(FIG3B_BINOMIAL_SMOOTH_WINDOW), len(arr))
    if win % 2 == 0:
        win -= 1
    poly = min(int(FIG3B_BINOMIAL_SMOOTH_POLYORDER), max(win - 1, 0))
    if win >= 3 and poly >= 1:
        arr = sg_smooth(arr, window_length=win, polyorder=poly)
    if clip is not None:
        arr = np.clip(arr, clip[0], clip[1])
    if round_int:
        arr = np.clip(np.rint(arr), 1, N_MAX_D)
    return arr


def _candidate_np_summary_local(item):
    idx = int(item['pca_index'])
    n_vals = np.asarray(D_N[idx], dtype=float) if idx < D_N.shape[0] and np.isfinite(D_N[idx]).sum() >= 3 else np.asarray(item['fit_N'], dtype=float)
    p_vals = np.asarray(D_P[idx], dtype=float) if idx < D_P.shape[0] and np.isfinite(D_P[idx]).sum() >= 3 else np.asarray(item['fit_P'], dtype=float)
    sm_n = _smooth_binomial_trajectory_local(n_vals, clip=(1, N_MAX_D), round_int=True)
    sm_p = _smooth_binomial_trajectory_local(p_vals, clip=(1e-4, 0.999), round_int=False)
    return {
        'item': item,
        'fiber_name': item['fiber_name'],
        'bouton_id': item['base_bouton_id'],
        'smoothed_N': sm_n,
        'smoothed_P': sm_p,
        'n_start': float(sm_n[0]) if np.isfinite(sm_n[0]) else np.nan,
        'n_end': float(sm_n[-1]) if np.isfinite(sm_n[-1]) else np.nan,
        'n_delta': float(sm_n[-1] - sm_n[0]) if np.isfinite(sm_n[0]) and np.isfinite(sm_n[-1]) else np.nan,
        'n_range': float(np.nanmax(sm_n) - np.nanmin(sm_n)) if np.isfinite(sm_n).any() else np.nan,
        'n_std': float(np.nanstd(sm_n)) if np.isfinite(sm_n).any() else np.nan,
        'p_start': float(sm_p[0]) if np.isfinite(sm_p[0]) else np.nan,
        'p_end': float(sm_p[-1]) if np.isfinite(sm_p[-1]) else np.nan,
        'p_delta': float(sm_p[-1] - sm_p[0]) if np.isfinite(sm_p[0]) and np.isfinite(sm_p[-1]) else np.nan,
        'p_range': float(np.nanmax(sm_p) - np.nanmin(sm_p)) if np.isfinite(sm_p).any() else np.nan,
        'p_std': float(np.nanstd(sm_p)) if np.isfinite(sm_p).any() else np.nan,
    }


def _pick_best_candidate_local(pool, used_fibers, predicate, score_func):
    matches = [entry for entry in pool if entry['fiber_name'] not in used_fibers and predicate(entry)]
    if not matches:
        matches = [entry for entry in pool if entry['fiber_name'] not in used_fibers]
    if not matches:
        return None
    return max(matches, key=score_func)


if not ENABLE_FIG3B_SELECTED_FIBRES:
    print('Skipping Fig 3b selected-fibres plot: ENABLE_FIG3B_SELECTED_FIBRES = False')
else:
    binomial_pool = [_candidate_np_summary_local(item) for item in eligible_example_candidates]
    binomial_pool = [
        entry for entry in binomial_pool
        if np.isfinite(entry['smoothed_N']).sum() >= N_STIM and np.isfinite(entry['smoothed_P']).sum() >= N_STIM
    ]
    if len(binomial_pool) < 5:
        raise RuntimeError('Not enough finite N/P trajectories to build the selected binomial-evolution figure.')

    n_start_vals = np.array([entry['n_start'] for entry in binomial_pool], dtype=float)
    n_range_vals = np.array([entry['n_range'] for entry in binomial_pool], dtype=float)
    p_range_vals = np.array([entry['p_range'] for entry in binomial_pool], dtype=float)
    low_n_cut = float(np.nanpercentile(n_start_vals, 35))
    high_n_cut = float(np.nanpercentile(n_start_vals, 65))
    stable_n_cut = max(1.0, float(np.nanpercentile(n_range_vals, 40)))
    stable_p_cut = max(0.10, float(np.nanpercentile(p_range_vals, 35)))

    selection_specs = [
        {
            'label': 'Low N, N increases',
            'predicate': lambda entry: np.isfinite(entry['n_start']) and np.isfinite(entry['n_delta']) and entry['n_start'] <= low_n_cut and entry['n_delta'] >= 1.0,
            'score': lambda entry: (4.0 * entry['n_delta']) - entry['n_start'] + entry['p_range'],
        },
        {
            'label': 'High N, N decreases',
            'predicate': lambda entry: np.isfinite(entry['n_start']) and np.isfinite(entry['n_delta']) and entry['n_start'] >= high_n_cut and entry['n_delta'] <= -1.0,
            'score': lambda entry: entry['n_start'] + abs(entry['n_delta']) + (0.5 * entry['p_range']),
        },
        {
            'label': 'Stable N, P increases',
            'predicate': lambda entry: np.isfinite(entry['n_range']) and np.isfinite(entry['p_delta']) and entry['n_range'] <= stable_n_cut and entry['p_delta'] >= 0.12,
            'score': lambda entry: (5.0 * entry['p_delta']) - entry['n_range'],
        },
        {
            'label': 'Stable N, stable P',
            'predicate': lambda entry: np.isfinite(entry['n_range']) and np.isfinite(entry['p_range']) and entry['n_range'] <= stable_n_cut and entry['p_range'] <= stable_p_cut,
            'score': lambda entry: -(entry['n_range'] + entry['p_range']),
        },
        {
            'label': 'Mixed remodeling',
            'predicate': lambda entry: np.isfinite(entry['n_range']) and np.isfinite(entry['p_range']),
            'score': lambda entry: entry['n_range'] + (2.0 * entry['p_range']) + abs(entry['n_delta']) + abs(entry['p_delta']),
        },
    ]

    selected_binomial_examples = []
    used_fibers = set()
    for spec in selection_specs:
        chosen = _pick_best_candidate_local(binomial_pool, used_fibers, spec['predicate'], spec['score'])
        if chosen is None:
            continue
        selected_binomial_examples.append((spec['label'], chosen))
        used_fibers.add(chosen['fiber_name'])

    if len(selected_binomial_examples) != len(selection_specs):
        raise RuntimeError('Could not find enough distinct fibres/boutons for the selected binomial-evolution figure.')

    support = np.arange(0, FIG3B_BINOMIAL_SUPPORT_MAX + 1, dtype=int)
    event_colors = [plt.cm.rainbow(x) for x in np.linspace(0.02, 0.98, N_STIM)]
    fig_dyn, axes_dyn = plt.subplots(1, 5, figsize=(18, 4.2), squeeze=False)
    axes_dyn = axes_dyn[0]

    for panel_idx, (ax, (panel_label, entry)) in enumerate(zip(axes_dyn, selected_binomial_examples), start=1):
        max_prob = 0.0
        for stim_idx, (n_val, p_val, color) in enumerate(zip(entry['smoothed_N'], entry['smoothed_P'], event_colors), start=1):
            n_use = int(np.clip(np.rint(n_val), 1, FIG3B_BINOMIAL_SUPPORT_MAX))
            p_use = float(np.clip(p_val, 1e-4, 0.999))
            pmf = np.array([
                np.exp(log_binom_pmf(k_val, n_use, p_use)) if k_val <= n_use else 0.0
                for k_val in support
            ], dtype=float)
            max_prob = max(max_prob, float(np.nanmax(pmf)))
            ax.plot(support, pmf, color=color, lw=1.6, alpha=0.95)
        ax.set_xlim(0, FIG3B_BINOMIAL_SUPPORT_MAX)
        ax.set_xticks(np.arange(0, FIG3B_BINOMIAL_SUPPORT_MAX + 1, 2))
        ax.set_ylim(0, max_prob + max(0.03, 0.08 * max_prob))
        display_fiber = entry['fiber_name'].replace('_', ' ')
        bouton_short = entry['bouton_id'].split('_')[-1]
        title = (
            f"{panel_idx}. {panel_label}\n"
            f"{display_fiber} {bouton_short}\n"
            f"N {entry['n_start']:.0f}→{entry['n_end']:.0f} | P {entry['p_start']:.2f}→{entry['p_end']:.2f}"
        )
        style_ax(ax, 'Quantal count k', 'P(k)' if panel_idx == 1 else None, title)

    sm = plt.cm.ScalarMappable(cmap=ListedColormap(event_colors), norm=plt.Normalize(1, N_STIM))
    sm.set_array([])
    cbar = fig_dyn.colorbar(sm, ax=axes_dyn, fraction=0.020, pad=0.02)
    cbar.set_label('Stimulus')
    cbar.set_ticks(np.arange(1, N_STIM + 1))

    fig_dyn.suptitle('WT 20 Hz / 2.5 mM: selected fibres, smoothed binomial fits across the train', y=1.03)
    fig_dyn.tight_layout(rect=(0, 0, 1, 0.95), w_pad=1.0)
    output_dyn = OUTPUT_DIR / 'Fig3b_selected_fibres_smoothed_binomial_event_evolution.pdf'
    output_dyn_png = output_dyn.with_suffix('.png')
    fig_dyn.savefig(output_dyn, dpi=300, bbox_inches='tight')
    fig_dyn.savefig(output_dyn_png, dpi=300, bbox_inches='tight')
    plt.show()

    selected_binomial_df = pd.DataFrame([
        {
            'panel': idx,
            'pattern': label,
            'fiber_name': entry['fiber_name'],
            'bouton_id': entry['bouton_id'],
            'n_repeats': entry['item']['n_repeats'],
            'N1': entry['n_start'],
            'N10': entry['n_end'],
            'P1': entry['p_start'],
            'P10': entry['p_end'],
            'N_range': entry['n_range'],
            'P_range': entry['p_range'],
        }
        for idx, (label, entry) in enumerate(selected_binomial_examples, start=1)
    ])
    display(selected_binomial_df)
    print('=== Selected fibres/boutons: smoothed binomial evolution ===')
    print(selected_binomial_df.to_string(index=False))
    print(f'Saved to {output_dyn}')
    print(f'Saved to {output_dyn_png}')


In [ ]:
# === Fig 3b ? Example bouton binomial MLE fit ===
# Pick any bouton from the helper cell above.

EXAMPLE_MLE_OVERRIDE = '20240227_linescan1_bouton8'  # e.g. '20210128_linescan1_bouton1'
EXAMPLE_MLE_INDEX = 0        # fallback among boutons sorted by descending repeat count
EXAMPLE_MLE_PULSE = 1        # which stimulus to visualize in the histogram panel
EXAMPLE_MLE_N_BINS = 10      # force a fixed 10-bin support regardless of fitted N
EXAMPLE_MLE_TRACE_XLIM = (0.9, 1.6)
EXAMPLE_MLE_EVENT1_XLIM = (0.96, 1.150)
EXAMPLE_MLE_EVENT1_OFFSET = 1.0
EXAMPLE_MLE_EVENT1_USE_SAVGOL = True
EXAMPLE_MLE_EVENT1_SAVGOL_WINDOW = 9
EXAMPLE_MLE_EVENT1_SAVGOL_POLYORDER = 2

eligible_items = sorted(td_D.items(), key=lambda kv: len(kv[1]), reverse=True)
if not eligible_items:
    raise RuntimeError('No 20 Hz / 2.5 mM boutons available for the Fig 3b example.')

default_example_id = extract_base_name(str(globals().get('EXAMPLE_LARGE_2P5_BOUTON_ID', eligible_items[0][0])).strip())

example_mle_id = None
example_counts = None
if EXAMPLE_MLE_OVERRIDE is not None:
    requested_base = extract_base_name(str(EXAMPLE_MLE_OVERRIDE).strip())
    for bid, mats in eligible_items:
        if bid == requested_base:
            example_mle_id = bid
            example_counts = mats
            break
    if example_counts is None:
        raise ValueError(f'EXAMPLE_MLE_OVERRIDE not found in td_D: {EXAMPLE_MLE_OVERRIDE}')
elif default_example_id in td_D:
    example_mle_id = default_example_id
    example_counts = td_D[default_example_id]
else:
    safe_idx = int(np.clip(EXAMPLE_MLE_INDEX, 0, len(eligible_items) - 1))
    example_mle_id, example_counts = eligible_items[safe_idx]

example_counts = np.asarray(example_counts, dtype=int)
example_counts = example_counts[np.all(np.isfinite(example_counts), axis=1)]
if len(example_counts) < MIN_TRIALS_D:
    raise RuntimeError(f'Example bouton {example_mle_id} has only {len(example_counts)} valid trials; need at least {MIN_TRIALS_D}.')

example_fit_N = np.full(N_STIM, np.nan)
example_fit_P = np.full(N_STIM, np.nan)
example_fit_LL = np.full(N_STIM, np.nan)
for k in range(N_STIM):
    n_hat_k, p_hat_k, ll_hat_k = fit_binom_mle(example_counts[:, k], N_MAX_D)
    example_fit_N[k] = n_hat_k
    example_fit_P[k] = p_hat_k
    example_fit_LL[k] = ll_hat_k

example_pulse = int(np.clip(EXAMPLE_MLE_PULSE, 1, example_counts.shape[1]))
pulse_idx = example_pulse - 1
counts_k = example_counts[:, pulse_idx].astype(int)
n_hat = int(round(example_fit_N[pulse_idx])) if np.isfinite(example_fit_N[pulse_idx]) else np.nan
p_hat = float(example_fit_P[pulse_idx]) if np.isfinite(example_fit_P[pulse_idx]) else np.nan
k_support = np.arange(EXAMPLE_MLE_N_BINS, dtype=int)
obs_freq = np.array([(counts_k == kk).mean() for kk in k_support], dtype=float)
fit_pmf = np.array([
    np.exp(log_binom_pmf(kk, n_hat, p_hat)) if np.isfinite(n_hat) and np.isfinite(p_hat) and kk <= n_hat else 0.0
    for kk in k_support
], dtype=float)

example_trace_id = next(
    (str(trace_id).strip() for trace_id in _pca_ids if extract_base_name(str(trace_id).strip()) == example_mle_id),
    example_mle_id,
)
trace_stats = compute_trace_stats(
    trace_ids=[example_trace_id],
    condition_names=WT_2_5_20HZ_CONDITIONS,
    source=TRACE_MEAN_SOURCE,
)
trace_time = np.asarray(trace_stats['time'], float)
trace_avg = np.asarray(trace_stats['average'], float)
example_trial_rows = trials_all[
    trials_all['condition'].isin(WT_2_5_20HZ_CONDITIONS) &
    (trials_all['file'].astype(str).map(lambda s: extract_base_name(str(s).strip())) == example_mle_id)
].copy()
if example_trial_rows.empty:
    raise ValueError(f'No trial rows found for example bouton: {example_mle_id}')
example_trial_rows = example_trial_rows.sort_values('trial_input_col_1based')
example_trace_condition = str(example_trial_rows.iloc[0]['condition']).strip()
example_trace_file = str(example_trial_rows.iloc[0]['file']).strip()
example_trace_path = BASE_DIR / example_trace_condition / f'{example_trace_file}.xlsx'
example_trace_filename = example_trace_path.name
example_trace_df = pd.read_excel(example_trace_path, sheet_name='Traces DF_F0')
trace_time_raw = np.asarray(example_trace_df['Time'], float)
trace_time_shift = 0.5
trace_trials = []
for trial_1based in example_trial_rows['trial_input_col_1based'].to_numpy(int):
    trial_col = str(int(trial_1based) - 1)
    if trial_col not in example_trace_df.columns:
        continue
    trial_trace = np.asarray(example_trace_df[trial_col], float)
    keep = np.isfinite(trace_time_raw) & np.isfinite(trial_trace)
    if not np.any(keep):
        continue
    trace_trials.append((trace_time_raw[keep] + trace_time_shift, trial_trace[keep]))
if not trace_trials:
    raise ValueError(f'No workbook trial traces found for example bouton: {example_mle_id}')

REPO_ROOT = Path.cwd()
helper_script = REPO_ROOT / 'Feature_extraction' / 'tmp_selected_recording_nnls_overlay.py'
overlay_npz = OUTPUT_DIR / f'Fig3_b_{_sanitize_token(example_trace_condition)}_{_sanitize_token(example_trace_file)}_trial_overlay.npz'
example_trial_1based = int(example_trial_rows.iloc[0]['trial_input_col_1based'])
if not _overlay_npz_matches(overlay_npz, example_trace_condition, example_trace_file, example_trial_1based):
    if not helper_script.exists():
        raise FileNotFoundError(f'Helper script not found: {helper_script}')
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    subprocess.run(
        [
            sys.executable,
            str(helper_script),
            '--condition', example_trace_condition,
            '--file', example_trace_filename,
            '--trial', str(example_trial_1based),
            '--out', str(overlay_npz),
        ],
        check=True,
        cwd=str(REPO_ROOT),
        env=env,
    )
overlay_data = np.load(overlay_npz, allow_pickle=False)
required_overlay_keys = {'time_s', 'train_start_s', 'all_trial_yproc', 'all_trial_yhat'}
missing_overlay_keys = sorted(required_overlay_keys - set(overlay_data.files))
if missing_overlay_keys:
    raise KeyError(f'Missing keys in trial overlay NPZ: {missing_overlay_keys}')
overlay_time = np.asarray(overlay_data['time_s'], float)
overlay_train_start = float(np.asarray(overlay_data['train_start_s']).reshape(-1)[0])
overlay_plot_time = overlay_time + (1.0 - overlay_train_start)
overlay_trial_yproc = np.asarray(overlay_data['all_trial_yproc'], float)
overlay_trial_yhat = np.asarray(overlay_data['all_trial_yhat'], float)
if overlay_trial_yproc.shape != overlay_trial_yhat.shape:
    raise ValueError(
        f'Overlay trial arrays shape mismatch: {overlay_trial_yproc.shape} vs {overlay_trial_yhat.shape}'
    )

fig_mle, axes_mle = make_figure_grid(1, 4, panel_kind='elongated', squeeze=False)
ax_trace, ax_zoom, ax_hist, ax_fit = axes_mle[0, 0], axes_mle[0, 1], axes_mle[0, 2], axes_mle[0, 3]

trace_bounds = [trace_avg[np.isfinite(trace_avg)], np.array([0.0])]
for trial_t, trial_y in trace_trials:
    ax_trace.plot(trial_t, trial_y, color='0.55', lw=0.9, alpha=0.3, zorder=1)
    trace_bounds.append(trial_y[np.isfinite(trial_y)])
ax_trace.plot(trace_time, trace_avg, color='k', lw=1.6, label='Mean', zorder=3)
ax_trace.axhline(0.0, color='0.55', lw=0.8, ls=':')
trace_stim_times = [1.0 + 0.05 * i for i in range(example_counts.shape[1])]
if 'add_stimulus_ticks' in globals():
    add_stimulus_ticks(ax_trace, trace_stim_times, mode='top', linewidth=1.1)
else:
    for stim_time in trace_stim_times:
        ax_trace.axvline(stim_time, color='0.65', lw=0.6, ls=':')
trace_xlim = EXAMPLE_MLE_TRACE_XLIM
trace_bounds = np.concatenate([vals for vals in trace_bounds if vals.size > 0])
trace_ymin = float(np.nanmin(trace_bounds))
trace_ymax = float(np.nanmax(trace_bounds))
trace_ypad = max(0.05, 0.08 * (trace_ymax - trace_ymin if trace_ymax > trace_ymin else 1.0))
ax_trace.set_xlim(*trace_xlim)
ax_trace.set_ylim(trace_ymin - 0.10 * trace_ypad, trace_ymax + trace_ypad)
style_ax(ax_trace, 'Time (s)', '?F/F', 'Trace overlay')
add_legend(ax_trace, loc='upper right', frameon=False)

zoom_mask = np.isfinite(overlay_plot_time) & (overlay_plot_time >= EXAMPLE_MLE_EVENT1_XLIM[0]) & (overlay_plot_time <= EXAMPLE_MLE_EVENT1_XLIM[1])
if not np.any(zoom_mask):
    raise ValueError(f'No trial overlay samples inside zoom window {EXAMPLE_MLE_EVENT1_XLIM}')
zoom_time = overlay_plot_time[zoom_mask]
zoom_lo = np.inf
zoom_hi = -np.inf
zoom_trial_stack = np.full((overlay_trial_yproc.shape[0], zoom_time.size), np.nan, dtype=float)
for trial_idx in range(overlay_trial_yproc.shape[0]):
    trial_proc = overlay_trial_yproc[trial_idx, zoom_mask]
    trial_fit = overlay_trial_yhat[trial_idx, zoom_mask]
    keep = np.isfinite(zoom_time) & np.isfinite(trial_proc) & np.isfinite(trial_fit)
    if not np.any(keep):
        continue
    trial_offset = trial_idx * EXAMPLE_MLE_EVENT1_OFFSET
    trial_time_plot = zoom_time[keep]
    trial_proc_plot = np.asarray(trial_proc[keep], float).copy()
    trial_fit_plot = np.asarray(trial_fit[keep], float)
    if EXAMPLE_MLE_EVENT1_USE_SAVGOL and trial_proc_plot.size >= 5:
        sg_window = min(int(EXAMPLE_MLE_EVENT1_SAVGOL_WINDOW), int(trial_proc_plot.size))
        if sg_window % 2 == 0:
            sg_window -= 1
        sg_poly = min(int(EXAMPLE_MLE_EVENT1_SAVGOL_POLYORDER), sg_window - 1)
        if sg_window >= 3 and sg_poly >= 1:
            trial_proc_plot = savgol_filter(trial_proc_plot, sg_window, sg_poly, mode='interp')
    if trial_proc_plot.size >= 1:
        trial_baseline = trial_fit_plot[0]
        trial_proc_plot[0] = trial_baseline
        trial_proc_plot[-1] = trial_baseline
    zoom_trial_stack[trial_idx, np.flatnonzero(keep)] = trial_proc_plot
    ax_zoom.plot(trial_time_plot, trial_proc_plot + trial_offset, color='0.45', lw=0.9, alpha=0.95, zorder=1)
    ax_zoom.plot(trial_time_plot, trial_fit_plot + trial_offset, color='#ff7f0e', lw=1.0, alpha=0.95, zorder=2)
    zoom_lo = min(zoom_lo, float(np.nanmin(np.minimum(trial_proc_plot, trial_fit_plot) + trial_offset)))
    zoom_hi = max(zoom_hi, float(np.nanmax(np.maximum(trial_proc_plot, trial_fit_plot) + trial_offset)))
if not np.isfinite(zoom_lo) or not np.isfinite(zoom_hi):
    raise ValueError('No finite trial overlay values available for the Fig 3b event-1 zoom panel.')
zoom_avg = np.nanmean(zoom_trial_stack, axis=0)
zoom_avg_keep = np.isfinite(zoom_time) & np.isfinite(zoom_avg)
if np.any(zoom_avg_keep):
    avg_offset = -EXAMPLE_MLE_EVENT1_OFFSET
    ax_zoom.plot(zoom_time[zoom_avg_keep], zoom_avg[zoom_avg_keep] + avg_offset, color='k', lw=1.4, zorder=3)
    zoom_lo = min(zoom_lo, float(np.nanmin(zoom_avg[zoom_avg_keep] + avg_offset)))
    zoom_hi = max(zoom_hi, float(np.nanmax(zoom_avg[zoom_avg_keep] + avg_offset)))
ax_zoom.axhline(0.0, color='0.80', lw=0.7, ls=':')
ax_zoom.axvline(1.0, color='0.55', lw=0.8, ls=':')
zoom_ypad = max(0.04, 0.08 * (zoom_hi - zoom_lo if zoom_hi > zoom_lo else 1.0))
ax_zoom.set_xlim(*EXAMPLE_MLE_EVENT1_XLIM)
ax_zoom.set_ylim(zoom_lo - 0.25 * zoom_ypad, zoom_hi + zoom_ypad)
style_ax(ax_zoom, 'Time (s)', '?F/F + offset', 'Event 1 trial stack + NNLS')
zoom_handles = [
    plt.Line2D([0], [0], color='0.45', lw=0.9, label='Trial'),
    plt.Line2D([0], [0], color='#ff7f0e', lw=1.0, label='NNLS fit'),
    plt.Line2D([0], [0], color='k', lw=1.4, label='Average'),
]
add_legend(ax_zoom, handles=zoom_handles, labels=['Trial', 'NNLS fit', 'Average'], loc='upper right', frameon=False)

x_stim = np.arange(1, N_STIM + 1)
ax_fit.plot(x_stim, example_fit_P, color='#d62728', marker='o', lw=1.4, label='P?')
ax_fit.set_ylim(0, 1)
style_ax(ax_fit, 'Stimulus', '$P_k$', 'Bouton-level MLE trajectory')
ax_fit_n = ax_fit.twinx()
ax_fit_n.plot(x_stim, example_fit_N, color='#1f77b4', marker='s', lw=1.4, label='N?')
ax_fit_n.set_ylabel('$N_k$')
ax_fit_n.set_ylim(bottom=0)
handles_fit = [
    plt.Line2D([0], [0], color='#d62728', marker='o', lw=1.4, label='P?'),
    plt.Line2D([0], [0], color='#1f77b4', marker='s', lw=1.4, label='N?'),
]
add_legend(ax_fit, handles=handles_fit, labels=['P?', 'N?'], loc='upper right', frameon=False)

ax_hist.bar(k_support - 0.18, obs_freq, width=0.36, color='0.70', label='Observed')
ax_hist.bar(k_support + 0.18, fit_pmf, width=0.36, color='#2ca02c', alpha=0.85, label='Binomial fit')
style_ax(ax_hist, 'Quantal count k', 'Probability', f'Stim {example_pulse}: N?={n_hat}, P?={p_hat:.2f}')
ax_hist.set_xticks(k_support)
ax_hist.set_xlim(-0.5, EXAMPLE_MLE_N_BINS - 0.5)
add_legend(ax_hist, loc='upper right', frameon=False)

output_file = OUTPUT_DIR / 'Fig3_b_example_binomial_fit.pdf'
finalize_figure(fig_mle, title='Example bouton binomial MLE fit', save_path=output_file)

print('=== Example bouton binomial MLE fit ===')
print(f'Bouton: {example_mle_id}')
print(f'Valid trials: {len(example_counts)}')
print(f'Illustrative pulse: {example_pulse} | N?={n_hat} | P?={p_hat:.4f} | logL={example_fit_LL[pulse_idx]:.2f}')
print(f'Fixed histogram bins: {EXAMPLE_MLE_N_BINS}')
print(f'? Saved to {output_file}')


In [ ]:
# === Fig 3cdef — N and P maps at selected stimuli + per-cluster trajectories ===
# Render WT N/P maps across PCA space and summarize their per-cluster trajectories across the train.

_coords_D = pca_coordinates.copy()
_clust_D  = cluster_assignments.copy()
gx_D, gy_D, ext_D = setup_pca_grid(_coords_D)

n_show = len(STIM_SHOW)
meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'

fig_c, axes_c = make_figure_grid(1, n_show, panel_kind='pca', squeeze=False)
for col, sk in enumerate(STIM_SHOW):
    k = sk - 1
    vals = D_N[:, k]
    ok = np.isfinite(vals)
    ax = axes_c[0, col]
    nv, nxv = N_VLIM if N_VLIM else (np.nanpercentile(vals[ok], 2), np.nanpercentile(vals[ok], 98))
    render_pca_scalar_panel(
        ax,
        _coords_D,
        vals,
        cmap='Spectral_r',
        vmin=nv,
        vmax=nxv,
        title=f'N : stim {sk} (n={ok.sum()})',
        point_size=14,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
add_scalar_colorbar(fig_c, axes_c[0, :], cmap='Spectral_r', vmin=nv, vmax=nxv, label='N', shrink=0.7)
finalize_figure(
    fig_c,
    title=f'Binomial {meth} : N maps',
    rect=[0, 0, 0.94, 0.93],
    tight_layout_kwargs={'pad': 0.15, 'w_pad': 0.10, 'h_pad': 0.08},
    save_path=OUTPUT_DIR / 'Fig3_c_n_maps_selected_stimuli.pdf',
)

fig_d, axes_d = make_figure_grid(1, n_show, panel_kind='pca', squeeze=False)
for col, sk in enumerate(STIM_SHOW):
    k = sk - 1
    vals = D_P[:, k]
    ok = np.isfinite(vals)
    ax = axes_d[0, col]
    pv, pxv = 0.0, 1.0
    render_pca_scalar_panel(
        ax,
        _coords_D,
        vals,
        cmap='coolwarm',
        vmin=pv,
        vmax=pxv,
        title=f'P : stim {sk} (n={ok.sum()})',
        point_size=14,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
add_scalar_colorbar(fig_d, axes_d[0, :], cmap='coolwarm', vmin=pv, vmax=pxv, label='P', shrink=0.7, ticks=np.linspace(0.0, 1.0, 6))
finalize_figure(
    fig_d,
    title=f'Binomial {meth} : P maps',
    rect=[0, 0, 0.94, 0.93],
    tight_layout_kwargs={'pad': 0.15, 'w_pad': 0.10, 'h_pad': 0.08},
    save_path=OUTPUT_DIR / 'Fig3_d_p_maps_selected_stimuli.pdf',
)

# == Cluster trajectories ========================================
def _format_profile_pvalues_cluster(stats_result):
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
    if per_pulse_df.empty:
        return 'none'
    vals = []
    for _, row in per_pulse_df.iterrows():
        p_corr = row.get('p_value_corrected', np.nan)
        if np.isfinite(p_corr):
            vals.append(f"P{int(row['Pulse'])}={p_corr:.3g}")
    return ', '.join(vals) if vals else 'none'

x_stim = np.arange(1, N_STIM+1)
cluster_groups_p = []
cluster_groups_n = []
cluster_plot_rows = []
for cid in sorted_clusters:
    idx = np.where(_clust_D == cid)[0]
    vals_p = np.asarray(D_P[idx])
    vals_n = np.asarray(D_N[idx])
    n_fit = int(np.isfinite(vals_n[:, 0]).sum()) if vals_n.ndim == 2 and vals_n.shape[1] else 0
    if n_fit == 0:
        continue
    color = cluster_color_lookup.get(cid, 'gray')
    cluster_groups_p.append({'cid': cid, 'label': f'Cluster {cid}', 'matrix': vals_p, 'color': color, 'n_fit': n_fit})
    cluster_groups_n.append({'cid': cid, 'label': f'Cluster {cid}', 'matrix': vals_n, 'color': color, 'n_fit': n_fit})
    valid_idx = idx[np.isfinite(vals_n[:, 0])]
    cluster_plot_rows.append((
        cid,
        n_fit,
        np.nanmean(D_P[valid_idx, 0]),
        np.nanmean(D_P[valid_idx, 1]),
        np.nanmean(D_P[valid_idx, 4]),
        np.nanmean(D_P[valid_idx, 9]),
        np.nanmean(D_N[valid_idx, 0]),
        np.nanmean(D_N[valid_idx, 1]),
        np.nanmean(D_N[valid_idx, 4]),
        np.nanmean(D_N[valid_idx, 9]),
    ))

stats_cluster_p = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['matrix']} for item in cluster_groups_p],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Cluster P trajectories',
    print_summary=False,
)
stats_cluster_n = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['matrix']} for item in cluster_groups_n],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Cluster N trajectories',
    print_summary=False,
)

fig_e, ax_p = make_figure_grid(figsize=(6.5, 5))
for item in cluster_groups_p:
    plot_mean_sem_trace(
        ax_p,
        x_stim,
        item['matrix'],
        color=item['color'],
        label=f"C{item['cid']} ({item['n_fit']})",
        marker='o',
        linestyle='-',
        ms=3,
        lw=1,
        fill_alpha=0.12,
    )
ax_p.set_ylim(0, 1)
style_ax(ax_p, 'Stimulus', '$P_k$', 'P trajectory per cluster')
add_legend(ax_p, loc='best')
annotate_profile_significance(ax_p, stats_cluster_p)
out_cluster_p = OUTPUT_DIR / 'Fig3_e_p_cluster_trajectories.pdf'
finalize_figure(fig_e, title=f'Binomial {meth} : P per cluster', save_path=out_cluster_p)

fig_f, ax_n = make_figure_grid(figsize=(6.5, 5))
for item in cluster_groups_n:
    plot_mean_sem_trace(
        ax_n,
        x_stim,
        item['matrix'],
        color=item['color'],
        label=f"C{item['cid']} ({item['n_fit']})",
        marker='o',
        linestyle='-',
        ms=3,
        lw=1,
        fill_alpha=0.12,
    )
style_ax(ax_n, 'Stimulus', '$N_k$', 'N trajectory per cluster')
ax_n.set_ylim(bottom=0)
add_legend(ax_n, loc='best')
annotate_profile_significance(ax_n, stats_cluster_n)
out_cluster_n = OUTPUT_DIR / 'Fig3_f_n_cluster_trajectories.pdf'
finalize_figure(fig_f, title=f'Binomial {meth} : N per cluster', save_path=out_cluster_n)

print(stats_cluster_p.get('summary_line', 'Cluster P trajectories: no statistics available.'))
print(f"Cluster P trajectories FDR-corrected p values: {_format_profile_pvalues_cluster(stats_cluster_p)}")
print(stats_cluster_n.get('summary_line', 'Cluster N trajectories: no statistics available.'))
print(f"Cluster N trajectories FDR-corrected p values: {_format_profile_pvalues_cluster(stats_cluster_n)}")
print(f"{'Cl':>4} {'n':>4} {'P1':>6} {'P2':>6} {'P5':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N5':>6} {'N10':>6}")
for cid, n_fit, p1, p2, p5, p10, n1, n2, n5, n10 in cluster_plot_rows:
    vals = [p1, p2, p5, p10, n1, n2, n5, n10]
    print(f"C{cid:>3} {n_fit:>4} " + ' '.join(f"{v:>6.3f}" if np.isfinite(v) and v < 2 else (f"{v:>6.1f}" if np.isfinite(v) else '   nan') for v in vals))
print(f"✓ Saved P trajectories to {out_cluster_p}")
print(f"✓ Saved N trajectories to {out_cluster_n}")


In [ ]:
# === Helper — N_FIT_LAST and group definitions ===
# Define the end-of-train fitting window and the WT cluster groups used by cumulative-release analyses.

N_FIT_LAST = 4

# == Condition groups =============================================
COND_GROUPS_20 = OrderedDict([
    ('1.5 mM', {'conds': get_calcium_conditions('1.5mM', '20Hz'), 'color': get_wt_ca_color('1.5mM')}),
    ('2.5 mM', {'conds': get_calcium_conditions('2.5mM', '20Hz'), 'color': get_wt_ca_color('2.5mM')}),
    ('4.0 mM', {'conds': get_calcium_conditions('4mM', '20Hz'), 'color': get_wt_ca_color('4mM')}),
])
COND_GROUPS_50 = OrderedDict([
    ('1.5 mM', {'conds': get_calcium_conditions('1.5mM', '50Hz'), 'color': get_50hz_ca_color('1.5mM')}),
    ('2.5 mM', {'conds': get_calcium_conditions('2.5mM', '50Hz'), 'color': get_50hz_ca_color('2.5mM')}),
    ('4.0 mM', {'conds': get_calcium_conditions('4mM', '50Hz'), 'color': get_50hz_ca_color('4mM')}),
])

# Get summary DataFrames per condition
for groups in [COND_GROUPS_20, COND_GROUPS_50]:
    for lab, info in groups.items():
        mask = FEATURES_DATAFRAME['Condition'].isin(info['conds'])
        info['df'] = FEATURES_DATAFRAME[mask].copy()


In [ ]:
# === Fig 3g — Cumulative quantal release per cluster ===
# Compare cumulative quantal-release curves across WT clusters using the shared Q-normalized amplitudes.

ncl = len(sorted_clusters)
px = np.arange(num_pulses)
pulse_numbers = np.arange(1, num_pulses + 1, dtype=float)
show_stats = False
parametric_stat = True
fig_b3, ax = make_figure_grid(figsize=(8, 5))
clust_sn = {}
curve_inputs_b3 = []

for cid in sorted_clusters:
    mask = PCA_Data_WT_Pooled_clustered['HC_Cluster'] == cid
    df_cl = PCA_Data_WT_Pooled_clustered[mask]
    if len(df_cl) < 2:
        continue
    amps = reconstruct_amps_qnorm(df_cl, Q_estimates, Global_Q)
    r = sn_cumulative(amps, N_FIT_LAST)
    clust_sn[cid] = r
    c = cluster_color_lookup.get(cid, 'gray')
    curve_inputs_b3.append({
        'matrix': cumulative_profiles_matrix(amps),
        'color': c,
        'marker': 'o',
        'label': f"C{cid} (n={r['n']})",
        'linewidth': 1.0,
        'markersize': 4,
        'sem_alpha': 0.16,
    })

plot_profile_group_comparison(
    ax,
    curve_inputs_b3,
    pulse_numbers=pulse_numbers,
    stats_start_index=0,
    show_stats=show_stats,
    parametric_stat=parametric_stat,
    summary_label='SN cumulative 20 Hz 2.5 mM by hierarchical cluster',
    title='SN 20 Hz 2.5 mM : per cluster',
    xlabel='Stimulus',
    ylabel='Cumulative release (quanta)',
    finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'upper left'},
)
for cid in sorted_clusters:
    if cid not in clust_sn:
        continue
    c = cluster_color_lookup.get(cid, 'gray')
    r = clust_sn[cid]
    ax.plot(r['x_fit'] + 1, r['y_fit'], '--', color=c, alpha=0.4, lw=1)
ax.axhline(0, color='gray', lw=0.5, ls=':')
fig_b3.tight_layout()
fig_b3.savefig(OUTPUT_DIR / 'Fig3_g_sn_cumulative_per_wt_cluster.pdf', dpi=300, bbox_inches='tight')
plt.show()

print('\n== SN Summary ==')
print(f"{'Label':>16} {'n':>4} {'RRP':>7} {'P0':>6} {'slope':>8} {'r2':>6}")
for cid in sorted_clusters:
    if cid in clust_sn:
        r = clust_sn[cid]
        print(f"{'C'+str(cid):>16} {r['n']:>4} {r['RRP']:>7.2f} {r['P0']:>6.3f} {r['slope']:>8.3f} {r['r2']:>6.3f}")


In [ ]:
# === Fig 3 manuscript stats addendum ===
# Collect manuscript-ready Fig. 3 support tables without touching figure outputs.

from scipy.stats import wilcoxon

_required_fig3_stats = ['PCA_Data_WT_Pooled_clustered', 'cluster_assignments', 'sorted_clusters', 'D_N', 'D_P', 'clust_sn']
_missing_fig3_stats = [name for name in _required_fig3_stats if name not in globals()]
if _missing_fig3_stats:
    raise RuntimeError(f"Run the upstream Fig. 3 cells first; missing: {_missing_fig3_stats}")

_fig3_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
_fig3_valid_stim1 = np.isfinite(D_N[:, 0]) & np.isfinite(D_P[:, 0])


def _fig3_paired_stat(label, a, b):
    mask = np.isfinite(a) & np.isfinite(b)
    aa = np.asarray(a[mask], dtype=float)
    bb = np.asarray(b[mask], dtype=float)
    row = {
        'Comparison': label,
        'n': int(aa.size),
        'W': np.nan,
        'p': np.nan,
        'A_mean': float(np.nanmean(aa)) if aa.size else np.nan,
        'B_mean': float(np.nanmean(bb)) if bb.size else np.nan,
        'A_median': float(np.nanmedian(aa)) if aa.size else np.nan,
        'B_median': float(np.nanmedian(bb)) if bb.size else np.nan,
    }
    if aa.size == 0 or np.count_nonzero(np.abs(bb - aa) > 0) == 0:
        return row
    res = wilcoxon(aa, bb, zero_method='wilcox', alternative='two-sided')
    row['W'] = float(res.statistic)
    row['p'] = float(res.pvalue)
    return row


FIG3_MANUSCRIPT_GLOBAL_SUMMARY = pd.DataFrame([
    {
        'Metric': 'Pr stimulus 1',
        'Min': float(np.nanmin(D_P[_fig3_valid_stim1, 0])),
        'Max': float(np.nanmax(D_P[_fig3_valid_stim1, 0])),
        'Mean': float(np.nanmean(D_P[_fig3_valid_stim1, 0])),
        'n': int(_fig3_valid_stim1.sum()),
    },
    {
        'Metric': 'N stimulus 1',
        'Min': float(np.nanmin(D_N[_fig3_valid_stim1, 0])),
        'Max': float(np.nanmax(D_N[_fig3_valid_stim1, 0])),
        'Mean': float(np.nanmean(D_N[_fig3_valid_stim1, 0])),
        'n': int(_fig3_valid_stim1.sum()),
    },
])

FIG3_MANUSCRIPT_TRAJ_STATS = pd.DataFrame([
    _fig3_paired_stat('Pr1 vs Pr2', D_P[:, 0], D_P[:, 1]),
    _fig3_paired_stat('N1 vs N2', D_N[:, 0], D_N[:, 1]),
    _fig3_paired_stat('Pr2 vs Pr10', D_P[:, 1], D_P[:, 9]),
    _fig3_paired_stat('N2 vs N10', D_N[:, 1], D_N[:, 9]),
    _fig3_paired_stat('N1 vs N10', D_N[:, 0], D_N[:, 9]),
])

_fig3_cluster_rows = []
for cid in sorted_clusters:
    idx = np.where(cluster_assignments == cid)[0]
    valid_idx = idx[np.isfinite(D_N[idx, 0]) & np.isfinite(D_P[idx, 0])]
    rr_sv_ms = np.nan
    rr_per_stim = np.nan
    if cid in clust_sn:
        rr_per_stim = float(clust_sn[cid]['slope'])
        rr_sv_ms = rr_per_stim / 50.0
    _fig3_cluster_rows.append({
        'Cluster': f'C{int(cid)}',
        'n_profile': int(_fig3_cluster_counts.get(cid, 0)),
        'n_mle': int(valid_idx.size),
        'Pr1_mean': float(np.nanmean(D_P[valid_idx, 0])) if valid_idx.size else np.nan,
        'Pr2_mean': float(np.nanmean(D_P[valid_idx, 1])) if valid_idx.size else np.nan,
        'Pr10_mean': float(np.nanmean(D_P[valid_idx, 9])) if valid_idx.size else np.nan,
        'N1_mean': float(np.nanmean(D_N[valid_idx, 0])) if valid_idx.size else np.nan,
        'N2_mean': float(np.nanmean(D_N[valid_idx, 1])) if valid_idx.size else np.nan,
        'N10_mean': float(np.nanmean(D_N[valid_idx, 9])) if valid_idx.size else np.nan,
        'RR_per_stim': rr_per_stim,
        'RR_SV_per_ms': rr_sv_ms,
    })
FIG3_MANUSCRIPT_CLUSTER_TABLE = pd.DataFrame(_fig3_cluster_rows)

print('Fig. 3 cluster sizes (full WT clustered dataset):')
print(', '.join(f"C{int(cid)}={int(n)}" for cid, n in _fig3_cluster_counts.items()))
print('')
print('Fig. 3 global bootstrap-MLE summary (stimulus 1):')
print(FIG3_MANUSCRIPT_GLOBAL_SUMMARY.round({'Min': 3, 'Max': 3, 'Mean': 3}).to_string(index=False))
print('')
print('Fig. 3 paired bootstrap-MLE comparisons:')
def _fig3_format_p_for_report(p):
    if not np.isfinite(p):
        return 'nan'
    return f'{p:.2g}' if p < 0.01 else f'{p:.2f}'
_fig3_display = FIG3_MANUSCRIPT_TRAJ_STATS.round({'W': 3, 'A_mean': 3, 'B_mean': 3, 'A_median': 3, 'B_median': 3})
_fig3_display['p'] = FIG3_MANUSCRIPT_TRAJ_STATS['p'].map(_fig3_format_p_for_report)
print(_fig3_display.to_string(index=False))
print('')
print('Fig. 3 per-cluster summary:')
print(FIG3_MANUSCRIPT_CLUSTER_TABLE.round({'Pr1_mean': 3, 'Pr2_mean': 3, 'Pr10_mean': 3, 'N1_mean': 3, 'N2_mean': 3, 'N10_mean': 3, 'RR_per_stim': 3, 'RR_SV_per_ms': 4}).to_string(index=False))


In [ ]:
# === Helper — Refilling slope setup ===
# Compute the per-bouton refilling-slope quantities that feed the WT slope-map and comparison panels.

SMOOTH_ROBUST       = True     # clip outlier percentiles before smoothing
SMOOTH_SIGMA_FACTOR = 2.0      # sigma = median_NN_distance × this
SMOOTH_CLIP_PCT     = 5        # percentile for clipping (each tail)
GRID_RES            = 80       # heatmap grid resolution

# == Per-bouton SN slope (no rejection : ALL boutons) =============
_ppr_cols = [c for c in [f'PPR{i}/1' for i in range(2, 11)]
             if c in PCA_Data_WT_Pooled_clustered.columns]
_npulses = 1 + len(_ppr_cols)
_df = PCA_Data_WT_Pooled_clustered.copy()
_ids = _df['ID'].astype(str).str.strip().values
_q = np.array([Q_estimates.get(strip_calcium(b), Global_Q) for b in _ids])
_q[_q <= 0] = Global_Q
_a1 = _df['AMP1'].values / _q
_amps = np.column_stack([_a1] + [_df[c].values * _a1 for c in _ppr_cols])
_nb = len(_amps)
_px_map = np.arange(1, _npulses+1)
_nfit = min(N_FIT_LAST, _npulses)

_slopes = np.full(_nb, np.nan)
for i in range(_nb):
    ar = _amps[i]
    if np.any(~np.isfinite(ar)): continue
    cum = np.cumsum(ar)
    sl, ic, r, _, _ = linregress(_px_map[-_nfit:], cum[-_nfit:])
    if np.isfinite(sl):
        _slopes[i] = sl

_slope_over_a1 = np.where(np.isfinite(_slopes) & (_a1 > 0),
                           _slopes / _a1, np.nan)
_coords = pca_coordinates.copy()
_clust  = cluster_assignments.copy()
gx, gy, ext = setup_pca_grid(_coords)


In [ ]:
# === Fig 3h — Refilling slope normalised map ===
# Map the A1-normalized refilling slope across WT PCA space as a continuous mechanistic summary.

ok_sa = np.isfinite(_slope_over_a1)
v3, v4 = np.nanpercentile(_slope_over_a1[ok_sa], [2, 98])

fig_c2, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
sg2, sig2 = smooth_field(_coords, _slope_over_a1, ok_sa, gx, gy)
Xg, Yg = np.meshgrid(gx, gy)
im = ax.imshow(sg2, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap('RdBu_r'),
               vmin=v3, vmax=v4, interpolation='bilinear', zorder=1, alpha=0.8)
# Isoline at norm_rate = 1
if np.any(np.isfinite(sg2)):
    ax.contour(Xg, Yg, sg2, levels=[1.0], colors='black', linewidths=0.0, zorder=2)
for cid in sorted_clusters:
    m = (_clust == cid) & ok_sa
    c = cluster_color_lookup.get(cid, 'gray')
    plot_pca_overlay_points(ax, _coords[m], c=[c]*m.sum(),
              s=18, edgecolors='none', zorder=3, label=f'C{cid}')
plt.colorbar(im, ax=ax, shrink=0.7, label='slope / A₁')
style_pca_axes(ax, title=f'Slope/A₁ (σ={sig2:.2f}, n={ok_sa.sum()})', legend=False)
add_legend(ax, loc='upper right', markerscale=1.5)
fig_c2.tight_layout()
fig_c2.savefig(OUTPUT_DIR / 'Fig3_h_refilling_slope_norm_map.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# === Aux — Q projection on WT PCA ===
# Aux Q maps + Q vs PCA axes (1.5 Ca projected, Q matched via extract_base_name)

# Get coordinates + match Q per bouton
low_ca_coords = pca_data['WT_1_5Ca']
low_ca_ids    = PCA_Data_WT_Low_Ca['ID'].astype(str).str.strip().values
q_vals_a2     = np.array([Q_by_ebn.get(extract_base_name(s), np.nan) for s in low_ca_ids])
has_q         = np.isfinite(q_vals_a2)

print(f"[A2] Matched: {has_q.sum()}/{len(low_ca_ids)}")
print(f"     Q range: {np.nanmin(q_vals_a2[has_q]):.4f} : {np.nanmax(q_vals_a2[has_q]):.4f}")

shared_ylim = (-2, 2)

fig_a2, axes_a2 = plt.subplots(1, 3, figsize=(15, 4))
ax0, ax1, ax2 = axes_a2

# Panel 0: PCA map of Q
ok = has_q
if ok.sum() >= 3:
    gx_a2, gy_a2, ext_a2 = setup_pca_grid(low_ca_coords[ok], pad=0.5)
    sg_a2, _ = smooth_field(low_ca_coords[ok], q_vals_a2[ok], np.ones(ok.sum(), dtype=bool), gx_a2, gy_a2)
    ax0.imshow(
        sg_a2,
        extent=ext_a2,
        origin='lower',
        aspect='auto',
        cmap='viridis',
        interpolation='bilinear',
        alpha=0.7,
    )
    plot_pca_value_overlay(
        ax0,
        low_ca_coords[ok],
        np.ones(ok.sum(), dtype=bool),
        q_vals_a2[ok],
        cmap='viridis',
        vmin=np.nanmin(q_vals_a2[ok]),
        vmax=np.nanmax(q_vals_a2[ok]),
        s=18,
        edgecolors='none',
        zorder=3,
    )
    sm = plt.cm.ScalarMappable(
        cmap='viridis',
        norm=plt.Normalize(np.nanmin(q_vals_a2[ok]), np.nanmax(q_vals_a2[ok]))
    )
    plt.colorbar(sm, ax=ax0, shrink=0.8, label='Q')
else:
    ax0.text(0.5, 0.5, 'Too few Q-matched boutons', ha='center', va='center', transform=ax0.transAxes)

style_pca_axes(ax0, title=f'Q map (n={ok.sum()})', legend=False)

# Panels 1-2: Q vs PC1 / PC2
for ax, pc_idx, pc_lab in [(ax1, 0, 'PC1'), (ax2, 1, 'PC2')]:
    x = low_ca_coords[has_q, pc_idx]
    y = q_vals_a2[has_q]

    ax.scatter(x, y, s=20, alpha=0.6, c='steelblue', edgecolors='none')

    if len(x) >= 2:
        z = np.polyfit(x, y, 1)
        xl = np.linspace(x.min(), x.max(), 200)
        ax.plot(xl, np.polyval(z, xl), 'r--', lw=1.2)

    ax.axhline(Global_Q, color='orange', ls=':', lw=1, label=f'Global Q={Global_Q:.4f}')

    style_ax(ax, pc_lab, 'Q (ΔF/F₀)')
    add_legend(ax)

    ax.set_xlim(x.min(), x.max())
    ax.set_ylim(*shared_ylim)

fig_a2.suptitle(f'Q in PCA space and Q vs PCA axes : 1.5 mM Ca projected (n={has_q.sum()})',
                fontsize=8, fontweight='bold')
fig_a2.tight_layout()
fig_a2.savefig(OUTPUT_DIR / 'Aux_q_projection_on_wt_pca.pdf', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# === Aux — Refilling slope map ===
# Map the raw refilling slope across WT PCA space as an auxiliary comparison panel.

ok_s = np.isfinite(_slopes)
v1, v2 = np.nanpercentile(_slopes[ok_s], [2, 98])

fig_c1, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
sg, sig = smooth_field(_coords, _slopes, ok_s, gx, gy)
ax.imshow(sg, extent=ext, origin='lower', aspect='auto', cmap=_display_cmap('viridis'),
          vmin=v1, vmax=v2, interpolation='bilinear', zorder=1, alpha=0.8)
# Cluster-coloured dots on top
for cid in sorted_clusters:
    m = (_clust == cid) & ok_s
    c = cluster_color_lookup.get(cid, 'gray')
    plot_pca_overlay_points(ax, _coords[m], c=[c]*m.sum(),
              s=18, edgecolors='none', zorder=3, label=f'C{cid}')
sm = plt.cm.ScalarMappable(cmap=_display_cmap('viridis'), norm=plt.Normalize(v1, v2))
plt.colorbar(sm, ax=ax, shrink=0.7, label='slope (q/stim)')
style_pca_axes(ax, title=f'Refilling slope (σ={sig:.2f}, n={ok_s.sum()})', legend=False)
add_legend(ax, loc='upper right', markerscale=1.5)
fig_c1.tight_layout()
fig_c1.savefig(OUTPUT_DIR / 'Aux_refilling_slope_map.pdf', dpi=300, bbox_inches='tight')
plt.show()

## Section 4 — Figure 4: Calcium dependence (1.5 vs 4 mM)


### Calcium perturbation overview

Changes in extracellular calcium are used to probe how release strength, apparent failures, and train dynamics shift relative to the WT reference organization. These analyses provide the main physiological perturbation of the WT bouton state space.

In the manuscript, the calcium manipulations are interpreted as a way to test whether bouton trajectories follow only a release-probability axis or whether they also recruit changes consistent with altered release-site occupancy and refilling.


#### Calcium effects on mean trace shape

The mean-trace overlays preserve the temporal structure of the calcium effects across the full train. They make it easier to see whether increased release remains sustained after the first response or collapses into rapid depression.


In [ ]:
# === Fig 4ab — Mean traces 1.5 vs 4 mM ===
# Compare mean traces between calcium concentrations and export raw + A1-normalized panels separately.

low_ca_20hz_conditions = set(get_calcium_conditions('1.5mM', '20Hz'))
high_ca_20hz_conditions = set(get_calcium_conditions('4mM', '20Hz'))
low_ca_stats = compute_trace_stats(condition_names=low_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
high_ca_stats = compute_trace_stats(condition_names=high_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
low_ca_rows = select_traces(condition_names=low_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
high_ca_rows = select_traces(condition_names=high_ca_20hz_conditions, source=TRACE_MEAN_SOURCE)
low_raw_stats = compute_trace_stats(rows=low_ca_rows, source=TRACE_MEAN_SOURCE)
high_raw_stats = compute_trace_stats(rows=high_ca_rows, source=TRACE_MEAN_SOURCE)
idx_low_1s = np.argmin(np.abs(low_raw_stats['time'] - 1.0))
idx_high_1s = np.argmin(np.abs(high_raw_stats['time'] - 1.0))
low_dt = np.diff(np.asarray(low_raw_stats['time'], float)); low_dt = low_dt[np.isfinite(low_dt) & (low_dt > 0)]
high_dt = np.diff(np.asarray(high_raw_stats['time'], float)); high_dt = high_dt[np.isfinite(high_dt) & (high_dt > 0)]
low_window_samples = max(1, int(np.ceil(0.05 / np.nanmedian(low_dt)))) if low_dt.size else 1
high_window_samples = max(1, int(np.ceil(0.05 / np.nanmedian(high_dt)))) if high_dt.size else 1
low_ca_peak_1s = np.nanmax(low_raw_stats['average'][idx_low_1s:idx_low_1s + low_window_samples])
high_ca_peak_1s = np.nanmax(high_raw_stats['average'][idx_high_1s:idx_high_1s + high_window_samples])
low_ca_rows_norm = low_ca_rows.copy(); high_ca_rows_norm = high_ca_rows.copy()
low_ca_rows_norm['Avg'] = [np.asarray(trace, float) / low_ca_peak_1s for trace in low_ca_rows_norm['Avg']]
high_ca_rows_norm['Avg'] = [np.asarray(trace, float) / high_ca_peak_1s for trace in high_ca_rows_norm['Avg']]
low_norm_stats = compute_trace_stats(rows=low_ca_rows_norm, source=TRACE_MEAN_SOURCE)
high_norm_stats = compute_trace_stats(rows=high_ca_rows_norm, source=TRACE_MEAN_SOURCE)
raw_bounds = np.concatenate([
    low_ca_stats['average'] - low_ca_stats['sem'],
    low_ca_stats['average'] + low_ca_stats['sem'],
    high_ca_stats['average'] - high_ca_stats['sem'],
    high_ca_stats['average'] + high_ca_stats['sem'],
])
raw_y_min, raw_y_max = np.nanmin(raw_bounds), np.nanmax(raw_bounds)
raw_y_pad = 0.05 * (raw_y_max - raw_y_min)
raw_y_lim = (raw_y_min - raw_y_pad, raw_y_max + raw_y_pad)
norm_bounds = np.concatenate([
    low_norm_stats['average'] - low_norm_stats['sem'],
    low_norm_stats['average'] + low_norm_stats['sem'],
    high_norm_stats['average'] - high_norm_stats['sem'],
    high_norm_stats['average'] + high_norm_stats['sem'],
    np.array([1.0]),
])
norm_y_min, norm_y_max = np.nanmin(norm_bounds), np.nanmax(norm_bounds)
norm_y_pad = 0.05 * (norm_y_max - norm_y_min)
norm_y_lim = (norm_y_min - norm_y_pad, norm_y_max + norm_y_pad)
stim_times = [t for t in [1.0 + 0.05 * i for i in range(10)] if t <= 2.0]

fig_raw, ax_raw = make_figure_grid(figsize=(7, 5))
plot_trace_analysis(ax=ax_raw, rows=low_ca_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('1.5mM'), label=f'1.5mM Ca (n={low_ca_stats["n"]})', display='mean_sem', style_axis=False)
plot_trace_analysis(ax=ax_raw, rows=high_ca_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('4mM'), label=f'4mM Ca (n={high_ca_stats["n"]})', display='mean_sem', stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.5}, zero_line=True, xlim=TRACE_XLIM_20HZ, ylim=raw_y_lim, xlabel='Time (s)', ylabel='ΔF/F', title='Average trace: 1.5 vs 4 mM', legend=True)
output_file_raw = OUTPUT_DIR / "Fig4_a_calcium_mean_traces_raw.pdf"
finalize_figure(fig_raw, title='Calcium mean traces', save_path=output_file_raw)

fig_norm, ax_norm = make_figure_grid(figsize=(7, 5))
plot_trace_analysis(ax=ax_norm, rows=low_norm_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('1.5mM'), label=f'1.5mM Ca (n={low_norm_stats["n"]})', display='mean_sem', style_axis=False)
plot_trace_analysis(ax=ax_norm, rows=high_norm_stats['rows'], source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('4mM'), label=f'4mM Ca (n={high_norm_stats["n"]})', display='mean_sem', stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.5}, zero_line=True, hlines=[1], hline_kwargs={'color': 'black', 'linestyle': ':', 'linewidth': 1, 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=norm_y_lim, xlabel='Time (s)', ylabel='ΔF/F (normalized)', title='A1-normalized trace: 1.5 vs 4 mM', legend=True)
output_file_norm = OUTPUT_DIR / "Fig4_b_calcium_mean_traces_a1_normalized.pdf"
finalize_figure(fig_norm, title='Calcium mean traces normalized to A1', save_path=output_file_norm)
print(f"✓ Calcium trace comparison: 1.5mM (n={low_ca_stats['n']}) vs 4mM (n={high_ca_stats['n']})")
print(f"✓ Calcium trace comparison (normalized to 1st peak = 1):")
print(f"  1.5mM Ca: n={low_norm_stats['n']}, peak at 1s={low_ca_peak_1s:.3f}")
print(f"  4mM Ca:   n={high_norm_stats['n']}, peak at 1s={high_ca_peak_1s:.3f}")
print(f"✓ Saved raw trace panel to {output_file_raw}")
print(f"✓ Saved normalized trace panel to {output_file_norm}")



#### Calcium effects on summary release metrics

These grouped comparisons combine amplitude, failures, and plasticity ratios to show that calcium changes do not map onto a single scalar descriptor. The joint presentation is important because the manuscript interprets calcium as affecting both Pr and apparent N. 


In [ ]:
# === Fig 4c — Direct calcium boxplot comparisons ===
# Compare AMP1, %Fail1, and PPR2/1 between low and high calcium with paired bouton overlays.

# Prepare data for plotting (unchanged)
amp1_comparison = pd.DataFrame({
    'AMP1': pd.concat([PCA_Data_WT_Low_Ca['AMP1'], PCA_Data_WT_High_Ca['AMP1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

fail1_comparison = pd.DataFrame({
    '%Fail1': pd.concat([PCA_Data_WT_Low_Ca['%Fail1'], PCA_Data_WT_High_Ca['%Fail1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

ppr2_comparison = pd.DataFrame({
    'PPR2/1': pd.concat([PCA_Data_WT_Low_Ca['PPR2/1'], PCA_Data_WT_High_Ca['PPR2/1']], ignore_index=True),
    'Condition': ['1.5mM Ca'] * len(PCA_Data_WT_Low_Ca) + ['4mM Ca'] * len(PCA_Data_WT_High_Ca)
})

idx_low_ca, idx_high_ca, paired_ca_ids = find_valid_pairs(
    PCA_Data_WT_Low_Ca['ID'].astype(str).tolist(),
    PCA_Data_WT_High_Ca['ID'].astype(str).tolist(),
)
paired_low_ca = PCA_Data_WT_Low_Ca.iloc[idx_low_ca].reset_index(drop=True).copy()
paired_high_ca = PCA_Data_WT_High_Ca.iloc[idx_high_ca].reset_index(drop=True).copy()

FIG4_CA_GROUP_SPACING = cluster_group_spacing if 'cluster_group_spacing' in globals() else BOXPLOT_GROUP_SPACING_DEFAULT
FIG4_CLUSTER_EQUIV_COUNT = 4
FIG4_CA_X_PADDING = 0.5 * ((FIG4_CLUSTER_EQUIV_COUNT - 1) * float(FIG4_CA_GROUP_SPACING) + 2.0 * float(cluster_x_padding if 'cluster_x_padding' in globals() else BOXPLOT_X_PADDING_DEFAULT) - float(FIG4_CA_GROUP_SPACING))
FIG4_CA_BOX_WIDTH = min(0.36, max(0.10, 0.55 * float(FIG4_CA_GROUP_SPACING)))
FIG4_CA_POSITIONS = np.arange(2, dtype=float) * float(FIG4_CA_GROUP_SPACING)

def _add_paired_lines(ax, low_values, high_values, positions=FIG4_CA_POSITIONS, color='gray', alpha=0.35, linewidth=0.9):
    low_arr = pd.to_numeric(pd.Series(low_values), errors='coerce').to_numpy(dtype=float)
    high_arr = pd.to_numeric(pd.Series(high_values), errors='coerce').to_numpy(dtype=float)
    valid_mask = np.isfinite(low_arr) & np.isfinite(high_arr)
    for low_val, high_val in zip(low_arr[valid_mask], high_arr[valid_mask]):
        ax.plot([positions[0], positions[1]], [low_val, high_val], color=color, alpha=alpha, linewidth=linewidth, zorder=1)

# Create plots
fig, (ax1, ax2, ax3) = make_figure_grid(1, 3, figsize=(7.8, 3.0))

# Comparaison AMP1
plot_boxplot_analysis(ax1, [make_box_group('1.5mM Ca', values=paired_low_ca['AMP1'], color=get_wt_ca_color('1.5mM')), make_box_group('4mM Ca', values=paired_high_ca['AMP1'], color=get_wt_ca_color('4mM'))], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='AMP1: Low vs High Calcium', ylabel='AMP1', group_spacing=FIG4_CA_GROUP_SPACING, x_padding=FIG4_CA_X_PADDING, box_width=FIG4_CA_BOX_WIDTH)
_add_paired_lines(ax1, paired_low_ca['AMP1'], paired_high_ca['AMP1'])
ax1.set_ylim(bottom=0)

# Comparaison %Fail1
plot_boxplot_analysis(ax2, [make_box_group('1.5mM Ca', values=paired_low_ca['%Fail1'], color=get_wt_ca_color('1.5mM')), make_box_group('4mM Ca', values=paired_high_ca['%Fail1'], color=get_wt_ca_color('4mM'))], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='%Fail1: Low vs High Calcium', ylabel='%Fail1', group_spacing=FIG4_CA_GROUP_SPACING, x_padding=FIG4_CA_X_PADDING, box_width=FIG4_CA_BOX_WIDTH)
_add_paired_lines(ax2, paired_low_ca['%Fail1'], paired_high_ca['%Fail1'])

# Comparaison PPR2/1
plot_boxplot_analysis(ax3, [make_box_group('1.5mM Ca', values=paired_low_ca['PPR2/1'], color=get_wt_ca_color('1.5mM')), make_box_group('4mM Ca', values=paired_high_ca['PPR2/1'], color=get_wt_ca_color('4mM'))], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='PPR2/1: Low vs High Calcium', ylabel='PPR2/1', group_spacing=FIG4_CA_GROUP_SPACING, x_padding=FIG4_CA_X_PADDING, box_width=FIG4_CA_BOX_WIDTH)
_add_paired_lines(ax3, paired_low_ca['PPR2/1'], paired_high_ca['PPR2/1'])
ax3.axhline(1, color='gray', linestyle='dotted', linewidth=1)
ppr2_vals = pd.concat([paired_low_ca['PPR2/1'], paired_high_ca['PPR2/1']], ignore_index=True)
ppr2_vals = pd.to_numeric(ppr2_vals, errors='coerce')
ppr2_max = float(np.nanmax(ppr2_vals)) if np.isfinite(ppr2_vals).any() else 1.0
ax3.set_ylim(0, max(1.0, ppr2_max * 1.05))

# Statistical tests with Wilcoxon on valid paired rows
amp1_paired = pd.DataFrame({'low': pd.to_numeric(paired_low_ca['AMP1'], errors='coerce'), 'high': pd.to_numeric(paired_high_ca['AMP1'], errors='coerce')}).dropna()
fail1_paired = pd.DataFrame({'low': pd.to_numeric(paired_low_ca['%Fail1'], errors='coerce'), 'high': pd.to_numeric(paired_high_ca['%Fail1'], errors='coerce')}).dropna()
ppr2_paired = pd.DataFrame({'low': pd.to_numeric(paired_low_ca['PPR2/1'], errors='coerce'), 'high': pd.to_numeric(paired_high_ca['PPR2/1'], errors='coerce')}).dropna()
amp1_stat, amp1_p = wilcoxon(amp1_paired['low'], amp1_paired['high']) if len(amp1_paired) >= 1 else (np.nan, np.nan)
fail1_stat, fail1_p = wilcoxon(fail1_paired['low'], fail1_paired['high']) if len(fail1_paired) >= 1 else (np.nan, np.nan)
ppr2_stat, ppr2_p = wilcoxon(ppr2_paired['low'], ppr2_paired['high']) if len(ppr2_paired) >= 1 else (np.nan, np.nan)

raw_pvals = np.array([amp1_p, fail1_p, ppr2_p], dtype=float)
valid_p_mask = np.isfinite(raw_pvals)
fdr_pvals = np.full(raw_pvals.shape, np.nan, dtype=float)
if valid_p_mask.any():
    _, fdr_valid, _, _ = multipletests(raw_pvals[valid_p_mask], method='fdr_bh')
    fdr_pvals[valid_p_mask] = fdr_valid
amp1_p_fdr, fail1_p_fdr, ppr2_p_fdr = fdr_pvals.tolist()

plt.tight_layout()

# Save plots
output_file = OUTPUT_DIR / "Fig4_c_calcium_direct_comparison_boxplots.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display results
print("1.5mM vs 4mM Calcium Comparison:")
print("-" * 40)
print(f"Paired boutons included: n={len(paired_ca_ids)}")
print(f"AMP1:")
print(f"  1.5mM: {amp1_paired['low'].mean():.3f} ± {amp1_paired['low'].std():.3f}")
print(f"  4mM:   {amp1_paired['high'].mean():.3f} ± {amp1_paired['high'].std():.3f}")
print(f"  Wilcoxon W = {amp1_stat:.1f}, raw p = {amp1_p:.4g}, FDR p = {amp1_p_fdr:.4g}")

print(f"%Fail1:")
print(f"  1.5mM: {fail1_paired['low'].mean():.1f}% ± {fail1_paired['low'].std():.1f}%")
print(f"  4mM:   {fail1_paired['high'].mean():.1f}% ± {fail1_paired['high'].std():.1f}%")
print(f"  Wilcoxon W = {fail1_stat:.1f}, raw p = {fail1_p:.4g}, FDR p = {fail1_p_fdr:.4g}")

print(f"PPR2/1:")
print(f"  1.5mM: {ppr2_paired['low'].mean():.3f} ± {ppr2_paired['low'].std():.3f}")
print(f"  4mM:   {ppr2_paired['high'].mean():.3f} ± {ppr2_paired['high'].std():.3f}")
print(f"  Wilcoxon W = {ppr2_stat:.1f}, raw p = {ppr2_p:.4g}, FDR p = {ppr2_p_fdr:.4g}")

# Save statistics
stats_file = OUTPUT_DIR / "FIG4DEF_calcium_comparison_statistics.txt"
with open(stats_file, 'w') as f:
    f.write("1.5mM vs 4mM Calcium Statistical Comparison\n")
    f.write("=" * 45 + "\n\n")
    f.write(f"Paired boutons included: n={len(paired_ca_ids)}\n")
    f.write(f"AMP1: 1.5mM={amp1_paired['low'].mean():.6f}±{amp1_paired['low'].std():.6f}; 4mM={amp1_paired['high'].mean():.6f}±{amp1_paired['high'].std():.6f}; Wilcoxon stat={amp1_stat:.1f}, raw_p={amp1_p:.6g}, fdr_p={amp1_p_fdr:.6g}\n")
    f.write(f"%Fail1: 1.5mM={fail1_paired['low'].mean():.6f}±{fail1_paired['low'].std():.6f}; 4mM={fail1_paired['high'].mean():.6f}±{fail1_paired['high'].std():.6f}; Wilcoxon stat={fail1_stat:.1f}, raw_p={fail1_p:.6g}, fdr_p={fail1_p_fdr:.6g}\n")
    f.write(f"PPR2/1: 1.5mM={ppr2_paired['low'].mean():.6f}±{ppr2_paired['low'].std():.6f}; 4mM={ppr2_paired['high'].mean():.6f}±{ppr2_paired['high'].std():.6f}; Wilcoxon stat={ppr2_stat:.1f}, raw_p={ppr2_p:.6g}, fdr_p={ppr2_p_fdr:.6g}\n")

print(f"Saved to {output_file} and {stats_file}")


#### Calcium modulation in WT PCA space

Low- and high-calcium boutons are projected onto the WT reference PCA to visualize how each bouton moves in the state space when release drive is changed. This is the geometric summary of the calcium manipulation. 


In [ ]:
# === Fig 4d — Calcium PCA trajectories ===
# Analyze calcium concentration effects on bouton properties in PCA space

def plot_calcium_trajectories():
    """Plot how calcium concentration changes affect PCA positioning."""
    
    make_figure(figsize=(10, 8))
    
    # Low calcium (1.5mM) - blue triangles pointing down
    low_ca_coords = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    plot_pca_analysis(
        plt.gca(),
        background={'alpha': 0.4, 'label': 'WT pooled (2.5mM Ca)'},
        overlays=[
            make_pca_group(low_ca_coords, label=f'1.5mM Ca (n={len(low_ca_coords)})', color=get_wt_ca_color('1.5mM'), marker='v', alpha=0.8, edgecolors='black', linewidths=0.0),
            make_pca_group(high_ca_coords, label=f'4mM Ca (n={len(high_ca_coords)})', color=get_wt_ca_color('4mM'), marker='^', alpha=0.8, edgecolors='black', linewidths=0.0),
        ],
        title=None,
        legend=False,
    )

    # Calculate centroids
    center_pooled = np.mean(pca_coordinates, axis=0)
    center_low_ca = np.mean(low_ca_coords, axis=0)
    center_high_ca = np.mean(high_ca_coords, axis=0)
    
    # Plot centroids
    plot_pca_analysis(plt.gca(), overlays=[make_pca_group(np.array([center_low_ca]), label='1.5mM centroid', color=get_wt_ca_color('1.5mM'), marker='X', edgecolors='black', linewidths=0.0, zorder=5), make_pca_group(np.array([center_high_ca]), label='4mM centroid', color=get_wt_ca_color('4mM'), marker='X', edgecolors='black', linewidths=0.0, zorder=5)], title=None, legend=False)
    
    # Draw single arrow from low calcium to high calcium centroid
    plt.arrow(center_low_ca[0], center_low_ca[1],
              center_high_ca[0] - center_low_ca[0], center_high_ca[1] - center_low_ca[1],
              color='purple', width=0.02, head_width=0.25, length_includes_head=True, alpha=0.9)
        
    # Connect paired boutons between conditions (assuming matched order)
    n_pairs = min(len(low_ca_coords), len(high_ca_coords))
    if n_pairs > 0:
        for i in range(n_pairs):
            plt.plot([low_ca_coords[i, 0], high_ca_coords[i, 0]],
                     [low_ca_coords[i, 1], high_ca_coords[i, 1]],
                     color='gray', alpha=0.4, linewidth=1)
        print(f"Connected {n_pairs} bouton pairs between calcium conditions")
    
    # Format plot
    style_pca_axes(plt.gca(), title='Calcium Concentration Effects on Bouton Properties',  legend=False)
    add_legend(plt.gca(), frameon=False)
    plt.tight_layout()
    
    # Save figure
    output_file = OUTPUT_DIR / "Fig4_d_calcium_pca_trajectories.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    
    return center_pooled, center_low_ca, center_high_ca, n_pairs

def calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs):
    """Calculate movement statistics for calcium concentration changes."""
    
    # Distances from standard condition to each calcium level
    dist_to_low  = np.linalg.norm(center_low_ca - center_pooled)
    dist_to_high = np.linalg.norm(center_high_ca - center_pooled)
    
    # Individual bouton movements
    low_ca_coords  = pca_data['WT_1_5Ca']
    high_ca_coords = pca_data['WT_4Ca']
    
    # Movement from standard to low calcium
    movements_to_low  = []
    n_low_comparisons = min(len(pca_coordinates), len(low_ca_coords))
    for i in range(n_low_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - low_ca_coords[i])
        movements_to_low.append(dist)
    
    # Movement from standard to high calcium
    movements_to_high  = []
    n_high_comparisons = min(len(pca_coordinates), len(high_ca_coords))
    for i in range(n_high_comparisons):
        dist = np.linalg.norm(pca_coordinates[i] - high_ca_coords[i])
        movements_to_high.append(dist)
    
    # Movement between calcium conditions (paired boutons)
    calcium_range_movements = []
    for i in range(n_pairs):
        dist = np.linalg.norm(low_ca_coords[i] - high_ca_coords[i])
        calcium_range_movements.append(dist)
    
    return {
        'centroid_distances': {'low': dist_to_low, 'high': dist_to_high},
        'individual_movements': {
            'to_low': movements_to_low,
            'to_high': movements_to_high,
            'between_ca': calcium_range_movements
        }
    }

# Run analysis
center_pooled, center_low_ca, center_high_ca, n_pairs = plot_calcium_trajectories()
movement_stats = calculate_calcium_movements(center_pooled, center_low_ca, center_high_ca, n_pairs)

# Display results
print(f"\n=== CALCIUM CONCENTRATION ANALYSIS ===")
print(f"Centroid coordinates:")
print(f"  WT pooled (2.5mM): ({center_pooled[0]:.3f}, {center_pooled[1]:.3f})")
print(f"  1.5mM Ca:          ({center_low_ca[0]:.3f}, {center_low_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['low']:.3f}")
print(f"  4mM Ca:            ({center_high_ca[0]:.3f}, {center_high_ca[1]:.3f}) - distance: {movement_stats['centroid_distances']['high']:.3f}")

print(f"\nIndividual bouton movements in PCA space:")
if movement_stats['individual_movements']['to_low']:
    low_moves = movement_stats['individual_movements']['to_low']
    print(f"2.5mM → 1.5mM Ca (n={len(low_moves)}): {np.mean(low_moves):.3f} ± {np.std(low_moves):.3f}")

if movement_stats['individual_movements']['to_high']:
    high_moves = movement_stats['individual_movements']['to_high']
    print(f"2.5mM → 4mM Ca (n={len(high_moves)}): {np.mean(high_moves):.3f} ± {np.std(high_moves):.3f}")

if movement_stats['individual_movements']['between_ca']:
    range_moves = movement_stats['individual_movements']['between_ca']
    print(f"1.5mM ↔ 4mM Ca (n={len(range_moves)}): {np.mean(range_moves):.3f} ± {np.std(range_moves):.3f}")

print(f"\n✓ Calcium trajectory analysis complete")


In [ ]:
# === Fig 4e — Cumulative release normalised to A1 ===
# Compare low- and high-calcium cumulative release after per-bouton normalization to quantal A1.

fig_b1, ax = make_figure_grid(figsize=(7, 5))
pulse_numbers = np.arange(1, num_pulses + 1, dtype=float)
show_stats = False
parametric_stat = True
res_b1_norm = {}
curve_inputs_b1_norm = []

for lab in ['1.5 mM', '4.0 mM']:
    info = COND_GROUPS_20[lab]
    df = info['df']
    if len(df) < 2:
        continue
    amps = reconstruct_amps_qnorm(df, Q_estimates, Global_Q)
    valid_rows = np.isfinite(amps).all(axis=1) & np.isfinite(amps[:, 0]) & (amps[:, 0] > 0)
    amps = amps[valid_rows]
    if len(amps) < 2:
        continue
    amps_norm = amps / amps[:, [0]]
    r = sn_cumulative(amps_norm, N_FIT_LAST)
    res_b1_norm[lab] = r
    curve_inputs_b1_norm.append({
        'matrix': cumulative_profiles_matrix(amps_norm),
        'color': info['color'],
        'marker': 'o',
        'label': f"{lab} (n={r['n']})",
        'linewidth': 1.2,
        'markersize': 4,
        'sem_alpha': 0.18,
    })

plot_profile_group_comparison(
    ax,
    curve_inputs_b1_norm,
    pulse_numbers=pulse_numbers,
    stats_start_index=0,
    show_stats=show_stats,
    parametric_stat=parametric_stat,
    summary_label='SN cumulative normalized to A1: low vs high calcium comparison',
    title='Cumulative release / A1 : 1.5 vs 4 mM',
    xlabel='Stimulus',
    ylabel='Normalized cumulative slope',
    finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'upper left'},
)
for lab in ['1.5 mM', '4.0 mM']:
    info = COND_GROUPS_20[lab]
    if lab not in res_b1_norm:
        continue
    r = res_b1_norm[lab]
    ax.plot(r['x_fit'] + 1, r['y_fit'], '--', color=info['color'], alpha=0.5, lw=1)
ax.axhline(0, color='gray', lw=0.5, ls=':')

fig_b1.tight_layout()
fig_b1.savefig(OUTPUT_DIR / 'Fig4_e_calcium_sn_cumulative_norm_a1_low_vs_high_20hz.pdf', dpi=300, bbox_inches='tight')
plt.show()

print("\nCumulative release / A1 summary (20 Hz, low vs high Ca):")
for lb in res_b1_norm:
    print(f"{lb}: RRP/A1={res_b1_norm[lb]['RRP']:.2f}  "
          f"P0_from_norm={res_b1_norm[lb]['P0']:.3f}  "
          f"refill/A1={res_b1_norm[lb]['slope']:.3f} per stim  "
          f"r2={res_b1_norm[lb]['r2']:.3f}")


In [ ]:
# === Fig 4e addendum — Cumulative release in Q-normalized SV equivalents ===
# Report the non-A1-normalized cumulative release and refill values used for manuscript placeholders.

pulse_numbers = np.arange(1, num_pulses + 1, dtype=float)
interstimulus_ms = 1000.0 / 20.0
FIG4_QNORM_CUMULATIVE_STATS = {}
fig_qnorm, ax_qnorm = make_figure_grid(figsize=(7, 5))
curve_inputs_qnorm = []

for lab in ['1.5 mM', '4.0 mM']:
    info = COND_GROUPS_20[lab]
    df = info['df'].copy().reset_index(drop=True)
    if len(df) < 2:
        continue
    amps_qnorm = reconstruct_amps_qnorm(df, Q_estimates, Global_Q)
    valid_rows = np.isfinite(amps_qnorm).all(axis=1)
    amps_qnorm = amps_qnorm[valid_rows]
    if len(amps_qnorm) < 2:
        continue

    cumulative_matrix = cumulative_profiles_matrix(amps_qnorm)
    r = sn_cumulative(amps_qnorm, N_FIT_LAST)
    final_release = cumulative_matrix[:, -1]
    slope_per_stim = float(r['slope'])
    slope_per_ms = slope_per_stim / interstimulus_ms

    FIG4_QNORM_CUMULATIVE_STATS[lab] = {
        'n': int(r['n']),
        'released_sv_mean': float(np.nanmean(final_release)),
        'released_sv_sd': float(np.nanstd(final_release, ddof=1)),
        'released_sv_sem': float(np.nanstd(final_release, ddof=1) / np.sqrt(np.isfinite(final_release).sum())),
        'rrp_sv': float(r['RRP']),
        'p0': float(r['P0']),
        'refill_sv_per_stim': slope_per_stim,
        'refill_sv_per_ms': slope_per_ms,
        'r2': float(r['r2']),
    }
    curve_inputs_qnorm.append({
        'matrix': cumulative_matrix,
        'color': info['color'],
        'marker': 'o',
        'label': f"{lab} (n={r['n']})",
        'linewidth': 1.2,
        'markersize': 4,
        'sem_alpha': 0.18,
    })
    ax_qnorm.plot(r['x_fit'] + 1, r['y_fit'], '--', color=info['color'], alpha=0.5, lw=1)

plot_profile_group_comparison(
    ax_qnorm,
    curve_inputs_qnorm,
    pulse_numbers=pulse_numbers,
    stats_start_index=0,
    show_stats=False,
    parametric_stat=True,
    summary_label='Q-normalized cumulative release: low vs high calcium comparison',
    title='Cumulative release: 1.5 vs 4 mM',
    xlabel='Stimulus',
    ylabel='Cumulative release (SV equivalents)',
    finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'upper left'},
)

fig_qnorm.tight_layout()
fig4_qnorm_pdf = OUTPUT_DIR / 'Fig4_e_addendum_calcium_cumulative_qnorm_sv_low_vs_high_20hz.pdf'
fig_qnorm.savefig(fig4_qnorm_pdf, dpi=300, bbox_inches='tight')
plt.show()

fig4_qnorm_stats_xlsx = OUTPUT_DIR / 'Fig4_e_addendum_calcium_cumulative_qnorm_sv_statistics.xlsx'
pd.DataFrame([
    {'condition': condition_name, **stats_dict}
    for condition_name, stats_dict in FIG4_QNORM_CUMULATIVE_STATS.items()
]).to_excel(fig4_qnorm_stats_xlsx, index=False)

print('\nCumulative release summary in Q-normalized SV equivalents (20 Hz, low vs high Ca):')
for lab, stat in FIG4_QNORM_CUMULATIVE_STATS.items():
    print(
        f"{lab}: released during train={stat['released_sv_mean']:.2f} ± {stat['released_sv_sd']:.2f} SV equivalents, "
        f"RRP={stat['rrp_sv']:.2f} SV, P0={stat['p0']:.3f}, "
        f"refill={stat['refill_sv_per_stim']:.3f} SV/stim = {stat['refill_sv_per_ms']:.5f} SV/ms, "
        f"n={stat['n']}, r2={stat['r2']:.3f}"
    )
print(f'✓ Saved Q-normalized cumulative panel to {fig4_qnorm_pdf}')
print(f'✓ Saved Q-normalized cumulative statistics to {fig4_qnorm_stats_xlsx}')

#### Calcium refilling normalized to A1

The cumulative-release slopes above are group-level fits in quantal units. This complementary plot normalizes each bouton's refilling slope by its own quantal A1 to show how sustained recruitment scales relative to the first response under each calcium condition.


In [ ]:
# === Fig 4f — Final slope norm ===
# Compare the end-of-train slope of the A1-normalized cumulative release curves between calcium conditions.

pulse_numbers = np.arange(1, num_pulses + 1, dtype=float)
slope_arrays = {}
ids_by_lab = {}

for lab in ['1.5 mM', '4.0 mM']:
    info = COND_GROUPS_20[lab]
    df = info['df'].copy().reset_index(drop=True)
    amps = reconstruct_amps_qnorm(df, Q_estimates, Global_Q)
    valid_rows = np.isfinite(amps).all(axis=1) & np.isfinite(amps[:, 0]) & (amps[:, 0] > 0)
    df = df.loc[valid_rows].copy().reset_index(drop=True)
    amps = amps[valid_rows]
    amps_norm = amps / amps[:, [0]]
    local_slopes = np.full(len(amps_norm), np.nan)
    for i in range(len(amps_norm)):
        cum_i = np.cumsum(amps_norm[i])
        sl, _, _, _, _ = linregress(pulse_numbers[-N_FIT_LAST:], cum_i[-N_FIT_LAST:])
        local_slopes[i] = sl
    slope_arrays[lab] = local_slopes
    ids_by_lab[lab] = df['ID'].astype(str).tolist()

idx_low_s, idx_high_s, paired_slope_ids = find_valid_pairs(ids_by_lab['1.5 mM'], ids_by_lab['4.0 mM'])
paired_low_slope = pd.Series(np.asarray(slope_arrays['1.5 mM'], float)[idx_low_s], name='1.5mM Ca')
paired_high_slope = pd.Series(np.asarray(slope_arrays['4.0 mM'], float)[idx_high_s], name='4mM Ca')
valid_slope = np.isfinite(paired_low_slope.to_numpy()) & np.isfinite(paired_high_slope.to_numpy())
paired_low_slope = paired_low_slope[valid_slope].reset_index(drop=True)
paired_high_slope = paired_high_slope[valid_slope].reset_index(drop=True)

FIG4_SLOPE_GROUP_SPACING = cluster_group_spacing if 'cluster_group_spacing' in globals() else BOXPLOT_GROUP_SPACING_DEFAULT
FIG4_SLOPE_CLUSTER_EQUIV_COUNT = 4
FIG4_SLOPE_X_PADDING = 0.5 * ((FIG4_SLOPE_CLUSTER_EQUIV_COUNT - 1) * float(FIG4_SLOPE_GROUP_SPACING) + 2.0 * float(cluster_x_padding if 'cluster_x_padding' in globals() else BOXPLOT_X_PADDING_DEFAULT) - float(FIG4_SLOPE_GROUP_SPACING))
FIG4_SLOPE_BOX_WIDTH = min(0.36, max(0.10, 0.55 * float(FIG4_SLOPE_GROUP_SPACING)))
FIG4_SLOPE_POSITIONS = np.arange(2, dtype=float) * float(FIG4_SLOPE_GROUP_SPACING)

def _add_paired_lines_local(ax, low_values, high_values, positions=FIG4_SLOPE_POSITIONS, color='gray', alpha=0.35, linewidth=0.9):
    low_arr = pd.to_numeric(pd.Series(low_values), errors='coerce').to_numpy(dtype=float)
    high_arr = pd.to_numeric(pd.Series(high_values), errors='coerce').to_numpy(dtype=float)
    valid_mask = np.isfinite(low_arr) & np.isfinite(high_arr)
    for low_val, high_val in zip(low_arr[valid_mask], high_arr[valid_mask]):
        ax.plot([positions[0], positions[1]], [low_val, high_val], color=color, alpha=alpha, linewidth=linewidth, zorder=1)

slope_stat, slope_p = wilcoxon(paired_low_slope, paired_high_slope) if len(paired_low_slope) >= 3 else (np.nan, np.nan)

make_figure(panel_kind='boxplot_compact')
fig_slope = plt.gcf()
ax_slope = plt.gca()
plot_boxplot_analysis(ax_slope, [make_box_group('1.5mM Ca', values=paired_low_slope, color=get_wt_ca_color('1.5mM')), make_box_group('4mM Ca', values=paired_high_slope, color=get_wt_ca_color('4mM'))], paired=True, show_points=True, show_stats=True, parametric_stat=False, title='Final slope norm', ylabel='Final slope norm', point_alpha=0.5, point_size=18, group_spacing=FIG4_SLOPE_GROUP_SPACING, x_padding=FIG4_SLOPE_X_PADDING, box_width=FIG4_SLOPE_BOX_WIDTH)
_add_paired_lines_local(ax_slope, paired_low_slope, paired_high_slope)
out_slope = OUTPUT_DIR / 'Fig4_f_calcium_refilling_slope_norm_boxplot.pdf'
finalize_figure(fig_slope, title='Final slope norm', save_path=out_slope)

print(f'Saved to {out_slope}')
print(f'Final slope norm paired boutons: n={len(paired_slope_ids)}')
print(f'  1.5mM: {paired_low_slope.mean():.3f} ± {paired_low_slope.std():.3f}')
print(f'  4mM:   {paired_high_slope.mean():.3f} ± {paired_high_slope.std():.3f}')
if np.isfinite(slope_p):
    print(f'Final slope comparison: Wilcoxon stat={slope_stat:.1f}, p={slope_p:.6g}')


In [ ]:
# === Helper — Bootstrap MLE condition definitions ===
# Define the pooled condition metadata and datasets used to compute bootstrap MLE N/P trajectories across perturbations.
# CELL E : Multi-condition MLE (calcium and frequency perturbations)
#
# Each condition has:
#   - trial data in trials_all (keyed by condition)
#   - a summary DF (row-aligned with projected PCA coordinates)
#   - projected PCA coordinates in pca_data[key]
#
# We match trial files → summary DF rows WITHIN each condition
# projected coords. This is the retained N/P method for calcium and
# frequency comparison figures shown after the WT-only section.
# ###################################################################

COND_DEFS_E = [
    # label              trial conditions                               colour     pca_key        summary_df_var
    ('1.5 mM 20 Hz',   ['Theo_1_5Ca'],                                 get_wt_ca_color('1.5mM'), 'WT_1_5Ca',    'PCA_Data_WT_Low_Ca'),
    ('4.0 mM 20 Hz',   ['Theo_4Ca'],                                   get_wt_ca_color('4mM'),   'WT_4Ca',      'PCA_Data_WT_High_Ca'),
    ('1.5 mM 50 Hz',   ['Theo_1_5_50Hz'],                              get_50hz_ca_color('1.5mM'), '50Hz_1_5Ca',  'PCA_Data_50Hz_1_5_Ca'),
    ('2.5 mM 50 Hz',   ['Theo_2_5_50Hz'],                              get_50hz_ca_color('2.5mM'), '50Hz_2_5Ca',  'PCA_Data_50Hz_2_5_Ca'),
    ('4.0 mM 50 Hz',   ['Theo_4_50Hz'],                                get_50hz_ca_color('4mM'),   '50Hz_4Ca',    'PCA_Data_50Hz_4_Ca'),
    ('2.5 mM 20 Hz',   get_calcium_conditions('2.5mM', '20Hz'),        get_wt_ca_color('2.5mM'), None,          'PCA_Data_WT_Pooled_clustered'),
]

# == Re-key Q with extract_base_name (same as A2 fix) ============
Q_by_ebn = {}
for k, v in Q_estimates.items():
    Q_by_ebn.setdefault(extract_base_name(k), v)

cond_results_E = {}

for (cond_label, cond_list, color, pca_key, sdf_var) in COND_DEFS_E:

    # == 1. Get projected coordinates =============================
    if pca_key is None:
        proj_xy = pca_coordinates.copy()
    else:
        if pca_key not in pca_data:
            print(f"  [{cond_label}] pca_data['{pca_key}'] missing : skip")
            continue
        proj_xy = np.array(pca_data[pca_key])

    if sdf_var not in globals():
        print(f"  [{cond_label}] {sdf_var} not defined : skip")
        continue
    sdf = globals()[sdf_var]
    n_pts = min(len(proj_xy), len(sdf))

    # == 2. Build count dict (keyed by extract_base_name) =========
    tg = defaultdict(list)
    sub = trials_all[trials_all['condition'].isin(cond_list)]
    for _, r in sub.iterrows():
        key = extract_base_name(str(r['file']).strip())
        tg[key].append(get_count_row_with_failures(r, Q_D))

    # == 3. Match summary DF rows → counts, run MLE ===============
    sdf_ids = sdf['ID'].astype(str).str.strip().values

    E_N = np.full((n_pts, N_STIM), np.nan)
    E_P = np.full((n_pts, N_STIM), np.nan)
    E_fail = np.full((n_pts, N_STIM), np.nan)
    n_matched, n_fit = 0, 0

    # Pre-collect valid (i, qc) tuples so the bootstrap can be parallelised.
    _items_E = []
    for i in range(n_pts):
        key = extract_base_name(sdf_ids[i])
        if key not in tg:
            continue
        qc = np.array(tg[key])
        ok_rows = np.all(np.isfinite(qc), axis=1)
        qc = qc[ok_rows].astype(int)
        if len(qc) < MIN_TRIALS_D:
            continue
        n_matched += 1
        E_fail[i, :] = np.mean(qc == 0, axis=0)
        _items_E.append((i, qc))

    def _fit_bouton_E(item):
        i, qc = item
        outN = np.full(N_STIM, np.nan)
        outP = np.full(N_STIM, np.nan)
        if USE_BOOTSTRAP:
            rng_e = np.random.default_rng(42 + int(i))
            ntr = len(qc)
            bN = np.full((N_STIM, N_BOOT), np.nan)
            bP = np.full((N_STIM, N_BOOT), np.nan)
            for b in range(N_BOOT):
                qc_b = qc[rng_e.integers(0, ntr, ntr)]
                for k in range(N_STIM):
                    qk = qc_b[:, k].astype(int)
                    Nf, Pf, _ll = fit_binom_mle(qk, N_MAX_D)
                    if np.isfinite(Nf) and Nf < N_MAX_D:
                        bN[k, b] = Nf
                        bP[k, b] = Pf
            for k in range(N_STIM):
                if np.isfinite(bN[k]).sum() >= N_BOOT * 0.5:
                    outN[k] = np.nanmedian(bN[k])
                    outP[k] = np.nanmedian(bP[k])
        else:
            for k in range(N_STIM):
                qk = qc[:, k].astype(int)
                Nf, Pf, _ll = fit_binom_mle(qk, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    outN[k] = Nf
                    outP[k] = Pf
        return i, outN, outP

    if _items_E:
        from concurrent.futures import ThreadPoolExecutor as _TPE_E
        import os as _os_E
        _n_w = max(1, min((_os_E.cpu_count() or 4), 8))
        with _TPE_E(max_workers=_n_w) as _ex_E:
            for i, outN, outP in _ex_E.map(_fit_bouton_E, _items_E):
                E_N[i, :] = outN
                E_P[i, :] = outP
                if np.isfinite(E_N[i, 0]):
                    n_fit += 1

    # Debug on zero matches
    if n_matched == 0 and len(tg) > 0:
        print(f"  [{cond_label:>16}]  !! 0 matches")
        print(f"     trial keys sample : {sorted(tg.keys())[:3]}")
        print(f"     sdf keys sample   : {[extract_base_name(s) for s in sdf_ids[:3]]}")
    else:
        print(f"  [{cond_label:>16}]  pts={n_pts}  matched={n_matched}  fit={n_fit}")

    cond_results_E[cond_label] = dict(
        E_N=E_N, E_P=E_P, E_fail=E_fail,
        proj_xy=proj_xy[:n_pts],
        color=color, n_fit=n_fit,
        n_pts=n_pts, n_matched=n_matched,
    )


In [ ]:
# === Fig 4gh — Bootstrap MLE trajectories ===
# Plot low- versus high-calcium bootstrap MLE trajectories for apparent P and N across the stimulus train.

import re

def _base_label(lbl):
    return re.sub(r"\s*\(\d+\)\s*$", "", lbl).strip()

def _cond_sort_key(base):
    mm = float(re.search(r"(\d+(?:\.\d+)?)\s*mM", base).group(1))
    hz = float(re.search(r"(\d+(?:\.\d+)?)\s*Hz", base).group(1))
    return (hz, mm)

def _force_ordered_legend(ax):
    handles, labels = ax.get_legend_handles_labels()
    by_base = {}
    for h, lbl in zip(handles, labels):
        if not lbl:
            continue
        base = _base_label(lbl)
        if base not in by_base:
            by_base[base] = (h, lbl)
    ordered_bases = sorted(by_base.keys(), key=_cond_sort_key)
    ordered_handles = [by_base[base][0] for base in ordered_bases]
    ordered_labels = [by_base[base][1] for base in ordered_bases]
    if ordered_labels:
        ax.legend(ordered_handles, ordered_labels, frameon=False)

def _format_profile_pvalues_lowhigh(stats_result):
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
    if per_pulse_df.empty:
        return 'none'
    vals = []
    for _, row in per_pulse_df.iterrows():
        p_corr = row.get('p_value_corrected', np.nan)
        if np.isfinite(p_corr):
            vals.append(f"P{int(row['Pulse'])}={p_corr:.3g}")
    return ', '.join(vals) if vals else 'none'

def _med(arr, k):
    ok = np.isfinite(arr[:, k])
    return np.nanmedian(arr[ok, k]) if ok.sum() else np.nan

selected_labels = ['1.5 mM 20 Hz', '4.0 mM 20 Hz']
selected_results = []
summary_rows_e1 = []
for cond_label in selected_labels:
    if cond_label not in cond_results_E:
        continue
    res = cond_results_E[cond_label]
    vals_p = np.asarray(res['E_P'])
    vals_n = np.asarray(res['E_N'])
    n_fit = int(np.isfinite(vals_n[:, 0]).sum()) if vals_n.ndim == 2 and vals_n.shape[1] else 0
    selected_results.append({'label': cond_label, 'color': res['color'], 'E_P': vals_p, 'E_N': vals_n, 'n_fit': n_fit})
    summary_rows_e1.append((cond_label, n_fit, _med(vals_p, 0), _med(vals_p, 1), _med(vals_p, 9), _med(vals_n, 0), _med(vals_n, 1), _med(vals_n, 9)))

meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'
stats_p_ca = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['E_P']} for item in selected_results],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Calcium P trajectories across conditions',
    print_summary=False,
)
stats_n_ca = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['E_N']} for item in selected_results],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Calcium N trajectories across conditions',
    print_summary=False,
)

fig_g, axp_e = make_figure_grid(figsize=(6.5, 5))
for item in selected_results:
    plot_mean_sem_trace(axp_e, x_stim, item['E_P'], color=item['color'], label=f"{item['label']} ({item['n_fit']})", marker='o', linestyle='-', ms=4, lw=1.2, fill_alpha=0.10)
axp_e.set_ylim(0, 1)
style_ax(axp_e, 'Stimulus', '$P_k$', 'P trajectory: 1.5 vs 4 mM')
_force_ordered_legend(axp_e)
annotate_profile_significance(axp_e, stats_p_ca)
save_path_g = OUTPUT_DIR / 'Fig4_g_bootstrap_mle_p_trajectories_low_vs_high_ca.pdf'
finalize_figure(fig_g, title=f'{meth}: P trajectories', save_path=save_path_g)

fig_h, axn_e = make_figure_grid(figsize=(6.5, 5))
for item in selected_results:
    plot_mean_sem_trace(axn_e, x_stim, item['E_N'], color=item['color'], label=f"{item['label']} ({item['n_fit']})", marker='o', linestyle='-', ms=4, lw=1.2, fill_alpha=0.10)
style_ax(axn_e, 'Stimulus', '$N_k$', 'N trajectory: 1.5 vs 4 mM')
axn_e.set_ylim(bottom=0)
_force_ordered_legend(axn_e)
annotate_profile_significance(axn_e, stats_n_ca)
save_path_h = OUTPUT_DIR / 'Fig4_h_bootstrap_mle_n_trajectories_low_vs_high_ca.pdf'
finalize_figure(fig_h, title=f'{meth}: N trajectories', save_path=save_path_h)

print('=== Low vs high calcium MLE trajectories ===')
print(stats_p_ca.get('summary_line', 'Calcium P trajectories across conditions: no statistics available.'))
print(f"Calcium P trajectories FDR-corrected p values: {_format_profile_pvalues_lowhigh(stats_p_ca)}")
print(stats_n_ca.get('summary_line', 'Calcium N trajectories across conditions: no statistics available.'))
print(f"Calcium N trajectories FDR-corrected p values: {_format_profile_pvalues_lowhigh(stats_n_ca)}")
print(f"{'Condition':<18} {'n':>4} {'P1':>6} {'P2':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N10':>6}")
for row in summary_rows_e1:
    cond_label, n_fit, p1, p2, p10, n1, n2, n10 = row
    vals = [p1, p2, p10, n1, n2, n10]
    formatted = ' '.join(f"{v:>6.3f}" if np.isfinite(v) and v < 2 else (f"{v:>6.1f}" if np.isfinite(v) else '   nan') for v in vals)
    print(f"{cond_label:<18} {n_fit:>4} {formatted}")
print(f"✓ Saved P trajectories to {save_path_g}")
print(f"✓ Saved N trajectories to {save_path_h}")


#### Calcium effects on paired-pulse facilitation

Paired-pulse behavior is interpreted jointly with the scalar comparisons and mean-trace views below, rather than as a separate standalone block.


In [ ]:
# === Aux — PPR profiles per Ca²⁺ at 20 Hz (background) ===
# Overlay all 20Hz PPR profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Use a shared y-axis max across all three plots
# Note: WT_pooled is the 20Hz / 2.5mM reference in this notebook

if 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining plot_profile_group_comparison).')

profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), 'Fig4_calcium_20hz_ppr_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), 'Fig4_calcium_20hz_ppr_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), 'Fig4_calcium_20hz_ppr_profiles_4p0mM.pdf'),
]

prepared_profiles = []
global_ymax = 0

for title, df_cond, color, save_name in profiles_by_condition:
    profiles = ppr_profiles_matrix(df_cond)

    if profiles.size == 0:
        prepared_profiles.append((title, profiles, color, save_name))
        continue

    mean_profile = np.nanmean(profiles, axis=0)
    sem_profile = np.nanstd(profiles, axis=0) / np.sqrt(profiles.shape[0])
    cond_ymax = np.nanmax(np.vstack([profiles, mean_profile + sem_profile]))
    global_ymax = max(global_ymax, cond_ymax)

    prepared_profiles.append((title, profiles, color, save_name))

global_ylim = (0, global_ymax * 1.05 if global_ymax > 0 else 1)

for title, profiles, color, save_name in prepared_profiles:
    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles,
        color=color,
        label='Mean',
        individual_alpha=0.25,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid PPR profiles')
        plt.close(fig)
        continue

    pulse_numbers, _, _ = result
    plot_profile_group_comparison(
        ax,
        [{
            'matrix': profiles,
            'color': color,
            'marker': 'o',
            'label': f'Mean (n={profiles.shape[0]})',
            'linewidth': 2.5,
            'sem_alpha': 0.18,
        }],
        pulse_numbers=np.asarray(pulse_numbers, dtype=float),
        title=(f'20Hz PPR Profiles - {title}\n' f'(n={profiles.shape[0]} boutons)'),
        ylabel='PPR (A_n/A_1)',
        finalize_kwargs={
            'ylim': global_ylim,
            'unity_kwargs': {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1},
            'legend': True,
            'legend_loc': 'best',
        },
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')


In [ ]:
# === Aux — Failure-rate profiles per Ca²⁺ at 20 Hz (background) ===
# Overlay all 20Hz failure-rate profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Explicit failures are defined per event from min(AMP_CORR, AMP_UNCORR) < thr_shared.

if 'failure_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared failure/PPR helper cell first (the cell defining plot_profile_group_comparison).')

failure_profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), 'Fig4_calcium_20hz_failure_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), 'Fig4_calcium_20hz_failure_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), 'Fig4_calcium_20hz_failure_profiles_4p0mM.pdf'),
]

for title, df_cond, color, save_name in failure_profiles_by_condition:
    profiles = failure_profiles_matrix(df_cond, trials_all, min_trials=5)
    n_valid = int(np.sum(np.isfinite(profiles).any(axis=1))) if profiles.size else 0

    if profiles.size == 0 or n_valid == 0:
        print(f'Skipping {title}: no valid failure-rate profiles')
        continue

    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles[np.isfinite(profiles).any(axis=1)],
        color=color,
        label='Mean',
        individual_alpha=0.20,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid failure-rate profiles')
        plt.close(fig)
        continue

    pulse_numbers, mean_profile, sem_profile = result
    valid_profiles = profiles[np.isfinite(profiles).any(axis=1)]
    plot_profile_group_comparison(
        ax,
        [{
            'matrix': valid_profiles,
            'color': color,
            'marker': 'o',
            'label': f'Mean (n={n_valid})',
            'linewidth': 2.5,
            'sem_alpha': 0.18,
        }],
        pulse_numbers=np.asarray(pulse_numbers, dtype=float),
        title=f'20Hz Failure Profiles - {title} (n={n_valid} boutons)',
        ylabel='Failure rate',
        finalize_kwargs={
            'ylim': (0, 1),
            'unity_line': False,
            'legend': True,
            'legend_loc': 'best',
        },
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')
    print(
        f"{title}: n={n_valid} | "
        f"Fail1={mean_profile[0]:.3f}±{sem_profile[0]:.3f} | "
        f"Fail2={mean_profile[1]:.3f}±{sem_profile[1]:.3f} | "
        f"Fail10={mean_profile[-1]:.3f}±{sem_profile[-1]:.3f}"
    )


#### Calcium dependence of failure rates

Failure-rate comparisons test whether increasing extracellular calcium primarily reduces apparent release failures, as expected if initial release probability rises. This provides one of the simplest physiological anchors for interpreting the PCA shifts. 


In [ ]:
# === Aux — Calcium %Fail1 comparison ===
# Compare failure rates between calcium concentrations
FAIL1_HIST_BIN_WIDTH = 10.0

fail1_standard = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_low_ca   = PCA_Data_WT_Low_Ca['%Fail1'].dropna()
fail1_high_ca  = PCA_Data_WT_High_Ca['%Fail1'].dropna()

# Plot histograms
all_fail1 = pd.concat([fail1_standard, fail1_low_ca, fail1_high_ca])
fail1_min = float(all_fail1.min())
fail1_max = float(all_fail1.max())
bins = np.arange(fail1_min, fail1_max + FAIL1_HIST_BIN_WIDTH, FAIL1_HIST_BIN_WIDTH, dtype=float)
if bins.size < 2:
    bins = np.array([fail1_min - FAIL1_HIST_BIN_WIDTH / 2.0, fail1_max + FAIL1_HIST_BIN_WIDTH / 2.0], dtype=float)
elif bins[-1] < fail1_max:
    bins = np.append(bins, bins[-1] + FAIL1_HIST_BIN_WIDTH)

make_figure(figsize=(10, 6))
weights_standard = np.ones(len(fail1_standard)) / len(fail1_standard) * 100
weights_low_ca   = np.ones(len(fail1_low_ca)) / len(fail1_low_ca) * 100
weights_high_ca  = np.ones(len(fail1_high_ca)) / len(fail1_high_ca) * 100

plt.hist(fail1_standard, bins=bins, alpha=0.4, color=get_wt_ca_color('2.5mM'), weights=weights_standard, 
         edgecolor='black', label=f'WT 2.5mM Ca (n={len(fail1_standard)})')
plt.hist(fail1_low_ca, bins=bins, alpha=0.4, color=get_wt_ca_color('1.5mM'), weights=weights_low_ca, 
         edgecolor='black', label=f'WT 1.5mM Ca (n={len(fail1_low_ca)})')
plt.hist(fail1_high_ca, bins=bins, alpha=0.35, color=get_wt_ca_color('4mM'), weights=weights_high_ca, 
         edgecolor='black', label=f'WT 4.0mM Ca (n={len(fail1_high_ca)})')

plt.axvline(fail1_standard.mean(), color=get_wt_ca_color('2.5mM'), linestyle='--', linewidth=1)
plt.axvline(fail1_low_ca.mean(), color=get_wt_ca_color('1.5mM'), linestyle='--', linewidth=1)
plt.axvline(fail1_high_ca.mean(), color=get_wt_ca_color('4mM'), linestyle='--', linewidth=1)

plt.xlabel('%Fail1')
plt.ylabel('Proportion (%)')
plt.title('Failure Rate: 1.5mM vs 2.5mM vs 4.0mM Calcium')
add_legend(plt.gca(), frameon=False, outside=True)
if 'style_hist_axis' in globals():
    style_hist_axis(plt.gca())
plt.grid(False)
plt.tight_layout()

output_file = OUTPUT_DIR / "Aux_calcium_fail1_comparison.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Stats
from scipy.stats import mannwhitneyu
print(f"1.5mM Ca: {fail1_low_ca.mean():.1f}% ± {fail1_low_ca.std():.1f}% (n={len(fail1_low_ca)})")
print(f"2.5mM Ca: {fail1_standard.mean():.1f}% ± {fail1_standard.std():.1f}% (n={len(fail1_standard)})")
print(f"4.0mM Ca: {fail1_high_ca.mean():.1f}% ± {fail1_high_ca.std():.1f}% (n={len(fail1_high_ca)})")
for pair_label, a, b in [
    ('1.5mM vs 2.5mM', fail1_low_ca, fail1_standard),
    ('2.5mM vs 4.0mM', fail1_standard, fail1_high_ca),
    ('1.5mM vs 4.0mM', fail1_low_ca, fail1_high_ca),
]:
    _, p_value = mannwhitneyu(a, b, alternative='two-sided')
    print(f"{pair_label:<15}: p = {p_value:.4g}")


In [ ]:
# === Aux — Bootstrap MLE condition maps (P, N) ===
# Render auxiliary per-condition PCA maps for the bootstrap-MLE estimates of N and P.

import re

E2_STIM = [1, 2, 10]

def _cond_sort_key(label):
    mm = float(re.search(r"(\d+(?:\.\d+)?)\s*mM", label).group(1))
    hz = float(re.search(r"(\d+(?:\.\d+)?)\s*Hz", label).group(1))
    return (hz, mm)

active_conds = sorted([c[0] for c in COND_DEFS_E if c[0] in cond_results_E], key=_cond_sort_key)
n_cond_plot = len(active_conds)
n_stim_e2 = len(E2_STIM)

def make_condition_map_figure(value_key, cmap, vlim, value_name, save_name):
    vmin_e, vmax_e = vlim
    fig_e2, axes_e2 = make_figure_grid(n_cond_plot, n_stim_e2, panel_kind='pca')

    for row_i, cond_label in enumerate(active_conds):
        res = cond_results_E[cond_label]
        xy = res['proj_xy']
        src = res[value_key]

        for si, sk in enumerate(E2_STIM):
            kidx = sk - 1
            arr = src[:, kidx]
            ok = np.isfinite(arr)
            ax = axes_e2[row_i, si]

            render_pca_scalar_panel(
                ax,
                xy,
                arr,
                cmap=cmap,
                vmin=vmin_e,
                vmax=vmax_e,
                title=f'{cond_label} : {value_name}$_{{{sk}}}$ (n={ok.sum()})',
                point_size=14,
                min_points=3,
                empty_label=f'n={ok.sum()} (too few)',
                add_colorbar=True,
                colorbar_label=value_name,
                colorbar_shrink=0.7,
                colorbar_ticks=np.linspace(0.0, 1.0, 6) if value_name == 'P' else None,
            )

    save_path = OUTPUT_DIR / save_name
    finalize_figure(
        fig_e2,
        title=f'Per-condition {value_name} maps at stim 1, 2, 10 (explicit-failure MLE)',
        rect=[0, 0, 0.96, 0.95],
        tight_layout_kwargs={'pad': 0.12, 'w_pad': 0.08, 'h_pad': 0.08},
        save_path=save_path,
    )
    print(f'✓ Saved {value_name} maps to {save_path}')
    return save_path

save_path_n = make_condition_map_figure('E_N', 'Spectral_r', N_VLIM, 'N', 'Aux_bootstrap_mle_condition_maps_N.pdf')
save_path_p = make_condition_map_figure('E_P', 'coolwarm', (0.0, 1.0), 'P', 'Aux_bootstrap_mle_condition_maps_P.pdf')

print(f"\n{'Condition':<18} {'n':>4} {'P1':>6} {'P2':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N10':>6}")
for cond_label in active_conds:
    res = cond_results_E[cond_label]
    N, P = res['E_N'], res['E_P']
    n = np.isfinite(N[:, 0]).sum()
    vals = []
    for k in [0, 1, 9]:
        ok_k = np.isfinite(P[:, k])
        vals.append(np.nanmedian(P[ok_k, k]) if ok_k.sum() else np.nan)
    for k in [0, 1, 9]:
        ok_k = np.isfinite(N[:, k])
        vals.append(np.nanmedian(N[ok_k, k]) if ok_k.sum() else np.nan)
    print(f"{cond_label:<18} {n:>4} " + ' '.join(f"{v:>6.3f}" if np.isfinite(v) and v < 2 else (f"{v:>6.1f}" if np.isfinite(v) else '   nan') for v in vals))


## Section 5 — Figure 5: Frequency dependence (50 Hz)


### High-frequency stimulation (50 Hz)

The 50 Hz experiments test how bouton properties shift when stimulation frequency is increased. These analyses are kept separate from the calcium block so that frequency-dependent trajectories can be interpreted explicitly against the WT 20 Hz reference.

In the manuscript logic, frequency mainly probes how boutons respond to stronger temporal demand. The key question is whether higher frequency pushes boutons along the same axis as calcium or instead produces a distinct trajectory in the WT reference space.


#### Example traces at 50 Hz

Example mean traces keep the temporal structure of the 50 Hz responses visible in the original fluorescence domain. They anchor the frequency comparison back to the measured signals before the PCA and MLE summaries.


In [ ]:
# === Fig 5a — Example traces across frequency ===
# Plot mean traces for all three 50Hz conditions from raw traces,
# using the shared nearest-sample raw alignment path.

condition_specs = [
    ('50Hz 1.5mM Ca', get_calcium_conditions('1.5mM', '50Hz')[0], get_50hz_ca_color('1.5mM')),
    ('50Hz 2.5mM Ca', get_calcium_conditions('2.5mM', '50Hz')[0], get_50hz_ca_color('2.5mM')),
    ('50Hz 4mM Ca', get_calcium_conditions('4mM', '50Hz')[0], get_50hz_ca_color('4mM')),
]
raw_stats = {}
fig, ax = make_figure_grid(figsize=(12, 6))
for idx, (label, condition_name, color) in enumerate(condition_specs):
    stats = plot_trace_analysis(ax=ax, condition_names=condition_name, source='raw_nearest', color=color, label=f'{label} (n={len(select_traces(condition_names=condition_name, source="raw"))})', display='mean_sem', zero_line=(idx == len(condition_specs) - 1), event_time=1.0 if idx == len(condition_specs) - 1 else None, event_kwargs={'color': 'gray', 'linestyle': '--', 'linewidth': 1} if idx == len(condition_specs) - 1 else None, stim_times=[t for t in [1.0 + 0.02 * i for i in range(10)] if t <= 1.4] if idx == len(condition_specs) - 1 else None, stim_kwargs={'mode': 'fixed', 'y_span': (-0.15, -0.10), 'linewidth': 2} if idx == len(condition_specs) - 1 else None, xlim=(0.898, 1.4) if idx == len(condition_specs) - 1 else None, xlabel='Time (s)' if idx == len(condition_specs) - 1 else None, ylabel='ΔF/F' if idx == len(condition_specs) - 1 else None, title='Mean Traces: 50Hz Stimulation at Different Calcium Concentrations (raw, stim-aligned, nearest-sample)' if idx == len(condition_specs) - 1 else None, legend=(idx == len(condition_specs) - 1), legend_kwargs={'fontsize': 10}, style_axis=(idx == len(condition_specs) - 1))
    raw_stats[label] = stats
plt.tight_layout()
output_file = OUTPUT_DIR / 'Fig5_a_frequency_example_traces_all_calcium_raw_nearest.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print('=== 50Hz RAW TRACE COMPARISON (nearest-sample average) ===')
for label in [s[0] for s in condition_specs]:
    stats = raw_stats[label]
    peak_idx = int(np.nanargmax(stats['average'])) if np.any(np.isfinite(stats['average'])) else None
    if peak_idx is None:
        print(f"{label}: n={stats['n']}, peak response unavailable")
    else:
        print(f"{label}: n={stats['n']}")
        print(f"  Peak response: {stats['average'][peak_idx]:.3f} ΔF/F at t={stats['time'][peak_idx]:.2f}s")
print(f'\n✓ Saved raw mean traces to {output_file}')


#### Mean PPR per condition

Mean PPR trajectories provide the train-normalized comparison between 20 Hz and 50 Hz at each calcium concentration. This panel is the direct frequency analogue of the calcium-dependent PPR panel in Figure 4.


In [ ]:
# === Fig 5b — Mean PPR per condition ===
# Compare PPR profiles across WT and 50Hz conditions at different calcium concentrations
if 'ppr_profile_stats' not in globals() or 'plot_ppr_mean_sem' not in globals() or 'finalize_ppr_axis' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining ppr_profile_stats).')

comparison_panels = [
    {
        'title': '1.5mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 1.5mM', PCA_Data_WT_Low_Ca, get_wt_ca_color('1.5mM'), 'o'),
            ('50Hz 1.5mM', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), 'v'),
        ],
    },
    {
        'title': '2.5mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 2.5mM', PCA_Data_WT_Pooled, get_wt_ca_color('2.5mM'), 'o'),
            ('50Hz 2.5mM', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), 'D'),
        ],
    },
    {
        'title': '4mM Ca: 20Hz vs 50Hz',
        'series': [
            ('20Hz 4mM', PCA_Data_WT_High_Ca, get_wt_ca_color('4mM'), 'o'),
            ('50Hz 4mM', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), '^'),
        ],
    },
]

fig, axes = make_figure_grid(1, 3, figsize=(18, 5))
show_stats = True
parametric_stat = True
summary_rows = []

for ax, panel in zip(axes, comparison_panels):
    group_inputs = []
    pulse_numbers = None
    for label, df_cond, color, marker in panel['series']:
        profiles = ppr_profiles_matrix(df_cond)
        pulse_numbers = np.arange(1, profiles.shape[1] + 1, dtype=float)
        group_inputs.append({
            'matrix': profiles,
            'color': color,
            'marker': marker,
            'label': f'{label} (n={profiles.shape[0]})',
            'linewidth': 2.5,
            'markersize': 5,
            'sem_alpha': 0.2,
        })
        means = np.nanmean(profiles, axis=0)
        sems = np.nanstd(profiles, axis=0, ddof=1) / np.sqrt(np.maximum(np.sum(np.isfinite(profiles), axis=0), 1)) if profiles.shape[0] > 1 else np.zeros(profiles.shape[1], dtype=float)
        if profiles.shape[0] > 1:
            sems[np.sum(np.isfinite(profiles), axis=0) < 2] = 0.0
        summary_rows.append((label, profiles.shape[0], means[1] if len(means) > 1 else np.nan, means[-1] if len(means) > 1 else np.nan,
                             sems[1] if len(sems) > 1 else np.nan, sems[-1] if len(sems) > 1 else np.nan))

    plot_profile_group_comparison(
        ax,
        group_inputs,
        pulse_numbers=pulse_numbers,
        stats_start_index=1,
        show_stats=show_stats,
        parametric_stat=parametric_stat,
        summary_label=panel['title'],
        title=panel['title'],
        ylabel='PPR (A_n/A_1)',
        finalize_kwargs={
            'ylim': (0, 3.0),
            'unity_kwargs': {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1},
            'legend': True,
            'legend_loc': 'best',
            'legend_fontsize': 6,
            'xlabel_fontsize': 11,
            'ylabel_fontsize': 11,
            'title_fontsize': 12,
            'title_fontweight': 'bold',
        },
    )

plt.tight_layout()
output_fig = OUTPUT_DIR / 'Fig5_b_frequency_ppr_20hz_vs_50hz_by_calcium.pdf'
plt.savefig(output_fig, dpi=300, bbox_inches='tight')
plt.show()

print('=== PPR PROFILE COMPARISON: 20Hz vs 50Hz ===')
for label, n, ppr2, ppr10, sem2, sem10 in summary_rows:
    print(f"{label:12} (n={n:3d}): PPR2/1={ppr2:.3f}±{sem2:.3f}, PPR10/1={ppr10:.3f}±{sem10:.3f}")

print(f'\n✓ Saved to {output_fig}')


#### PCA trajectories at 50 Hz

Frequency-dependent PCA trajectories summarize how raising the stimulation rate displaces bouton states across calcium conditions. This is the compact state-space view used to compare the 20 Hz and 50 Hz release regimes.


In [ ]:
# === Fig 5c — PCA trajectories across frequency ===
# Compare the PCA displacement between 20 Hz and 50 Hz across the tested calcium conditions.

conditions = [
    ('1.5mM', 'WT_1_5Ca',  '50Hz_1_5Ca', get_wt_ca_color('1.5mM'), get_50hz_ca_color('1.5mM'), 'v'),
    ('2.5mM', 'WT_pooled', '50Hz_2_5Ca', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM'), 'D'),
    ('4mM',   'WT_4Ca',    '50Hz_4Ca',   get_wt_ca_color('4mM'),   get_50hz_ca_color('4mM'),   '^'),
]

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']

fig, ax = plt.subplots(figsize=(10, 8))

for label, key20, key50, color20, color50, marker50 in conditions:
    coords20 = np.asarray(pca_data[key20])
    coords50 = np.asarray(pca_data[key50])

    center20 = coords20.mean(axis=0)
    center50 = coords50.mean(axis=0)

    ax.scatter(
        coords20[:, 0], coords20[:, 1],
        s=12, marker='o', facecolors='none', edgecolors=color20,
        alpha=0.35, linewidths=0.0,
        label=f'{label} 20Hz (n={len(coords20)})'
    )

    ax.scatter(
        coords50[:, 0], coords50[:, 1],
        s=24, marker=marker50, c=color50,
        alpha=0.80, edgecolors=color50, linewidths=0.0,
        label=f'{label} 50Hz (n={len(coords50)})'
    )

    ax.scatter(
        center20[0], center20[1],
        s=95, marker='o', c='white',
        edgecolors=color20, linewidths=0.0, zorder=5
    )

    ax.scatter(
        center50[0], center50[1],
        s=110, marker='X', c=color50,
        edgecolors='black', linewidths=0.0, zorder=6
    )

    ax.annotate(
        '',
        xy=(center50[0], center50[1]),
        xytext=(center20[0], center20[1]),
        arrowprops=dict(arrowstyle='->', color=color50, lw=2.3, mutation_scale=16),
        zorder=7
    )



ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_title('PCA Shift from 20Hz to 50Hz at Each Ca2+ Level')
ax.grid(False)
ax.legend(frameon=False, fontsize=6, ncol=2, bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()

output_file = OUTPUT_DIR / 'Fig5_c_frequency_pca_trajectories_all_calcium.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

for label, key20, key50, _, _, _ in conditions:
    center20 = np.asarray(pca_data[key20]).mean(axis=0)
    center50 = np.asarray(pca_data[key50]).mean(axis=0)
    print(f'{label}: 20Hz -> 50Hz centroid shift = {np.linalg.norm(center50 - center20):.3f}')

print(f'Saved to {output_file}')


#### Frequency-dependent scalar and cumulative summaries

These panels summarize how train-integrated release and end-of-train slope change with frequency across calcium conditions, complementing the PCA and MLE trajectory views.


In [ ]:
# === Fig 5d — A1-normalized cumulative release across frequency ===
# Compare 20 Hz and 50 Hz cumulative release after normalizing each bouton to its own A1 amplitude.
from scipy.stats import linregress

fig_freq_cum, axes = make_figure_grid(1, 3, figsize=(16, 4.5), sharey=True)
ca_labels = ['1.5 mM', '2.5 mM', '4.0 mM']
pulse_numbers = np.arange(1, num_pulses + 1, dtype=float)
show_stats = True
parametric_stat = True
res_freq_cum = {}

for ax, ca_lab in zip(axes, ca_labels):
    res_freq_cum[ca_lab] = {}
    curve_inputs = []
    for freq_lab, groups, marker in [('20 Hz', COND_GROUPS_20, 'o'), ('50 Hz', COND_GROUPS_50, 's')]:
        if ca_lab not in groups:
            continue
        info = groups[ca_lab]
        df = info['df']
        if len(df) < 2:
            continue
        amps = reconstruct_amps_qnorm(df, Q_estimates, Global_Q)
        if amps.size == 0:
            continue
        a1 = amps[:, [0]]
        valid_rows = np.isfinite(a1[:, 0]) & (np.abs(a1[:, 0]) > 1e-12)
        amps_norm = amps[valid_rows] / a1[valid_rows]
        if amps_norm.shape[0] < 2:
            continue
        cum_norm = np.cumsum(amps_norm, axis=1)
        x_tail = pulse_numbers[-N_FIT_LAST:]
        slopes = np.full(cum_norm.shape[0], np.nan)
        for row_i, row in enumerate(cum_norm):
            y_tail = row[-N_FIT_LAST:]
            ok = np.isfinite(y_tail)
            if ok.sum() >= 2:
                slopes[row_i] = linregress(x_tail[ok], y_tail[ok]).slope
        res_freq_cum[ca_lab][freq_lab] = {
            'matrix': cum_norm,
            'slopes': slopes,
            'n': int(cum_norm.shape[0]),
        }
        curve_inputs.append({
            'matrix': cum_norm,
            'color': info['color'],
            'marker': marker,
            'label': f"{freq_lab} (n={cum_norm.shape[0]})",
            'linewidth': 1.0,
            'markersize': 4,
            'sem_alpha': 0.16,
        })
    plot_profile_group_comparison(
        ax,
        curve_inputs,
        pulse_numbers=pulse_numbers,
        stats_start_index=0,
        show_stats=show_stats,
        parametric_stat=parametric_stat,
        summary_label=f'A1-normalized cumulative {ca_lab}: 20 Hz vs 50 Hz',
        title=ca_lab,
        xlabel='Stimulus',
        ylabel='Normalized cumulative slope' if ax is axes[0] else None,
        finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'best'},
    )
    ax.axhline(0, color='gray', lw=0.5, ls=':')

fig_freq_cum.suptitle('A1-normalized cumulative release: 20 Hz vs 50 Hz', fontsize=8, fontweight='bold')
out_freq_cum = OUTPUT_DIR / 'Fig5_d_frequency_sn_cumulative_norm_a1_20hz_vs_50hz.pdf'
fig_freq_cum.tight_layout()
fig_freq_cum.savefig(out_freq_cum, dpi=300, bbox_inches='tight')
plt.show()

print('=== A1-normalized cumulative release: 20 Hz versus 50 Hz by Ca²⁺ ===')
for ca_lab in ca_labels:
    for freq_lab in ['20 Hz', '50 Hz']:
        if freq_lab not in res_freq_cum.get(ca_lab, {}):
            continue
        slopes = res_freq_cum[ca_lab][freq_lab]['slopes']
        print(f"{ca_lab:<8} {freq_lab:<6} n={res_freq_cum[ca_lab][freq_lab]['n']:>3} slope={np.nanmedian(slopes):.3f}")
print(f"✓ Saved to {out_freq_cum}")

In [ ]:
# === Fig 5e — Final slope norm across frequency ===
# Boxplot comparison of the end-of-train slope from the A1-normalized cumulative curves.
fig_freq_slope, axes = make_figure_grid(1, 3, panel_kind='boxplot_compact', sharey=True)
fig_freq_slope.set_size_inches(7.8, 3.0, forward=True)
slope_summary_rows = []
slope_ylim_max = 0.0
slope_data = {}
for ca_lab in ['1.5 mM', '2.5 mM', '4.0 mM']:
    slope_data[ca_lab] = {}
    for freq_lab in ['20 Hz', '50 Hz']:
        entry = res_freq_cum.get(ca_lab, {}).get(freq_lab, {})
        vals = np.asarray(entry.get('slopes', []), dtype=float)
        vals = vals[np.isfinite(vals)]
        slope_data[ca_lab][freq_lab] = vals
        if vals.size:
            slope_ylim_max = max(slope_ylim_max, float(np.nanmax(vals)))
slope_ylim = (0, slope_ylim_max * 1.10 if slope_ylim_max > 0 else 1.0)
for ax, ca_lab in zip(axes, ['1.5 mM', '2.5 mM', '4.0 mM']):
    vals20 = slope_data[ca_lab].get('20 Hz', np.array([]))
    vals50 = slope_data[ca_lab].get('50 Hz', np.array([]))
    groups = [
        make_box_group(f'20 Hz\n(n={len(vals20)})', values=vals20, color=COND_GROUPS_20[ca_lab]['color']),
        make_box_group(f'50 Hz\n(n={len(vals50)})', values=vals50, color=COND_GROUPS_50[ca_lab]['color']),
    ]
    res = plot_boxplot_analysis(ax, groups, paired=False, show_points=True, show_stats=True, parametric_stat=False, title=ca_lab, ylabel='Final slope norm' if ax is axes[0] else None)
    ax.set_ylim(*slope_ylim)
    slope_summary_rows.append((ca_lab, len(vals20), len(vals50), res.get('p_value', np.nan)))
plt.tight_layout()
out_freq_slope = OUTPUT_DIR / 'Fig5_e_frequency_cumulative_slope_boxplots.pdf'
fig_freq_slope.savefig(out_freq_slope, dpi=300, bbox_inches='tight')
plt.show()
print('=== Final slope norm: 20 Hz versus 50 Hz ===')
for ca_lab, n20, n50, p_val in slope_summary_rows:
    print(f"{ca_lab:<8} n20={n20:>3} n50={n50:>3} p={p_val:.4g}" if np.isfinite(p_val) else f"{ca_lab:<8} n20={n20:>3} n50={n50:>3}")
print(f"✓ Saved to {out_freq_slope}")


In [ ]:
# === Fig 5f / 5g — Bootstrap MLE trajectories across frequency ===
# Panel f = P trajectories, panel g = N trajectories for 20 Hz vs 50 Hz at each calcium level.
freq_pairs = [
    ('1.5 mM', ('1.5 mM 20 Hz', get_wt_ca_color('1.5mM')), ('1.5 mM 50 Hz', get_50hz_ca_color('1.5mM'))),
    ('2.5 mM', ('2.5 mM 20 Hz', get_wt_ca_color('2.5mM')), ('2.5 mM 50 Hz', get_50hz_ca_color('2.5mM'))),
    ('4.0 mM', ('4.0 mM 20 Hz', get_wt_ca_color('4mM')), ('4.0 mM 50 Hz', get_50hz_ca_color('4mM'))),
]

def _format_profile_pvalues_frequency(stats_result):
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
    if per_pulse_df.empty:
        return 'none'
    vals = []
    for _, row in per_pulse_df.iterrows():
        p_corr = row.get('p_value_corrected', np.nan)
        if np.isfinite(p_corr):
            vals.append(f"P{int(row['Pulse'])}={p_corr:.3g}")
    return ', '.join(vals) if vals else 'none'

def _format_profile_rmanova_frequency(label, stats_result):
    omnibus = stats_result.get('omnibus_raw', {}) if isinstance(stats_result, dict) else {}
    if not omnibus:
        return f'{label} RM-ANOVA: unavailable'

    def _fmt_p(p_val):
        return f'{p_val:.3g}' if np.isfinite(p_val) else 'nan'

    def _term(name, f_key, p_key, df1_key, df2_key, eta_key):
        f_val = omnibus.get(f_key, np.nan)
        p_val = omnibus.get(p_key, np.nan)
        df1 = omnibus.get(df1_key, np.nan)
        df2 = omnibus.get(df2_key, np.nan)
        eta = omnibus.get(eta_key, np.nan)
        eta_txt = f', eta2p={eta:.3g}' if np.isfinite(eta) else ''
        return f'{name} F({df1:.0f},{df2:.0f})={f_val:.3g}, p={_fmt_p(p_val)}{eta_txt}'

    n_summary = ', '.join(
        f'{group_label} n={n}'
        for group_label, n in stats_result.get('complete_case_n', {}).items()
    )
    parts = [
        _term('Frequency', 'F_group', 'p_group', 'df_between', 'df_subjects_within', 'eta2_group'),
        _term('Stimulus', 'F_pulse', 'p_pulse', 'df_within', 'df_error', 'eta2_pulse'),
        _term('Frequency x Stimulus', 'F_interaction', 'p_interaction', 'df_interaction', 'df_error', 'eta2_interaction'),
    ]
    if n_summary:
        parts.append(n_summary)
    return f'{label} RM-ANOVA: ' + '; '.join(parts)

meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'
freq_specs_p = []
freq_specs_n = []
summary_rows_freq = []
for ca_lab, spec20, spec50 in freq_pairs:
    for cond_label, color in [spec20, spec50]:
        if cond_label not in cond_results_E:
            continue
        vals_p = np.asarray(cond_results_E[cond_label]['E_P'])
        vals_n = np.asarray(cond_results_E[cond_label]['E_N'])
        n_fit = int(np.isfinite(vals_n[:, 0]).sum()) if vals_n.ndim == 2 and vals_n.shape[1] else 0
        marker = 'o' if '20 Hz' in cond_label else 's'
        freq_specs_p.append({'label': cond_label, 'matrix': vals_p, 'color': color, 'marker': marker})
        freq_specs_n.append({'label': cond_label, 'matrix': vals_n, 'color': color, 'marker': marker})
        summary_rows_freq.append((cond_label, n_fit, np.nanmedian(vals_p[:, 0]), np.nanmedian(vals_p[:, 1]), np.nanmedian(vals_n[:, 0]), np.nanmedian(vals_n[:, 1])))

stats_freq_p = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['matrix']} for item in freq_specs_p],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Frequency P trajectories across calcium conditions',
    print_summary=False,
)
stats_freq_n = summarize_profile_group_stats(
    [{'label': item['label'], 'matrix': item['matrix']} for item in freq_specs_n],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='Frequency N trajectories across calcium conditions',
    print_summary=False,
)

fig_p, ax_p = make_figure_grid(figsize=(6.8, 5.2))
for item in freq_specs_p:
    matrix = np.asarray(item['matrix'], dtype=float)
    n_fit = int(np.isfinite(matrix[:, 0]).sum()) if matrix.ndim == 2 else 0
    plot_mean_sem_trace(ax_p, x_stim, item['matrix'], color=item['color'], label=f"{item['label']} ({n_fit})", marker=item['marker'], linestyle='-', ms=3, lw=1, fill_alpha=0.10)
ax_p.set_ylim(0, 1)
style_ax(ax_p, 'Stimulus', '$P_k$', 'P trajectories: 20 Hz vs 50 Hz')
add_legend(ax_p, loc='best')
annotate_profile_significance(ax_p, stats_freq_p)
out_p = OUTPUT_DIR / 'Fig5_f_bootstrap_mle_p_trajectories_20hz_vs_50hz.pdf'
finalize_figure(fig_p, title=f'{meth}: P trajectories', save_path=out_p)

fig_n, ax_n = make_figure_grid(figsize=(6.8, 5.2))
for item in freq_specs_n:
    matrix = np.asarray(item['matrix'], dtype=float)
    n_fit = int(np.isfinite(matrix[:, 0]).sum()) if matrix.ndim == 2 else 0
    plot_mean_sem_trace(ax_n, x_stim, item['matrix'], color=item['color'], label=f"{item['label']} ({n_fit})", marker=item['marker'], linestyle='-', ms=3, lw=1, fill_alpha=0.10)
style_ax(ax_n, 'Stimulus', '$N_k$', 'N trajectories: 20 Hz vs 50 Hz')
ax_n.set_ylim(bottom=0)
add_legend(ax_n, loc='best')
annotate_profile_significance(ax_n, stats_freq_n)
out_n = OUTPUT_DIR / 'Fig5_g_bootstrap_mle_n_trajectories_20hz_vs_50hz.pdf'
finalize_figure(fig_n, title=f'{meth}: N trajectories', save_path=out_n)

print('=== Frequency MLE trajectories ===')
print(stats_freq_p.get('summary_line', 'Frequency P trajectories across calcium conditions: no statistics available.'))
print(f"Frequency P trajectories FDR-corrected p values: {_format_profile_pvalues_frequency(stats_freq_p)}")
print(stats_freq_n.get('summary_line', 'Frequency N trajectories across calcium conditions: no statistics available.'))
print(f"Frequency N trajectories FDR-corrected p values: {_format_profile_pvalues_frequency(stats_freq_n)}")
for cond_label, n_fit, p1, p2, n1, n2 in summary_rows_freq:
    print(f"{cond_label:<16} n={n_fit:>3} P1={p1:.3f} P2={p2:.3f} N1={n1:.3f} N2={n2:.3f}")
for ca_lab, spec20, spec50 in freq_pairs:
    label20, _ = spec20
    label50, _ = spec50
    if label20 not in cond_results_E or label50 not in cond_results_E:
        continue
    pair_stats_p = summarize_profile_group_stats(
        [{'label': label20, 'matrix': cond_results_E[label20]['E_P']}, {'label': label50, 'matrix': cond_results_E[label50]['E_P']}],
        pulse_numbers=x_stim,
        paired_groups=False,
        parametric_stat=False,
        alpha=0.05,
        summary_label=f'{ca_lab} 20 Hz vs 50 Hz P trajectories',
        print_summary=False,
    )
    pair_stats_n = summarize_profile_group_stats(
        [{'label': label20, 'matrix': cond_results_E[label20]['E_N']}, {'label': label50, 'matrix': cond_results_E[label50]['E_N']}],
        pulse_numbers=x_stim,
        paired_groups=False,
        parametric_stat=False,
        alpha=0.05,
        summary_label=f'{ca_lab} 20 Hz vs 50 Hz N trajectories',
        print_summary=False,
    )
    pair_rmanova_p = summarize_profile_group_stats(
        [{'label': label20, 'matrix': cond_results_E[label20]['E_P']}, {'label': label50, 'matrix': cond_results_E[label50]['E_P']}],
        pulse_numbers=x_stim,
        paired_groups=False,
        parametric_stat=True,
        alpha=0.05,
        summary_label=f'{ca_lab} 20 Hz vs 50 Hz P trajectories RM-ANOVA',
        print_summary=False,
    )
    pair_rmanova_n = summarize_profile_group_stats(
        [{'label': label20, 'matrix': cond_results_E[label20]['E_N']}, {'label': label50, 'matrix': cond_results_E[label50]['E_N']}],
        pulse_numbers=x_stim,
        paired_groups=False,
        parametric_stat=True,
        alpha=0.05,
        summary_label=f'{ca_lab} 20 Hz vs 50 Hz N trajectories RM-ANOVA',
        print_summary=False,
    )
    print(pair_stats_p.get('summary_line', f'{ca_lab} 20 Hz vs 50 Hz P trajectories: no statistics available.'))
    print(f"{ca_lab} 20 Hz vs 50 Hz P trajectories FDR-corrected p values: {_format_profile_pvalues_frequency(pair_stats_p)}")
    print(_format_profile_rmanova_frequency(f'{ca_lab} 20 Hz vs 50 Hz P trajectories', pair_rmanova_p))
    print(pair_stats_n.get('summary_line', f'{ca_lab} 20 Hz vs 50 Hz N trajectories: no statistics available.'))
    print(f"{ca_lab} 20 Hz vs 50 Hz N trajectories FDR-corrected p values: {_format_profile_pvalues_frequency(pair_stats_n)}")
    print(_format_profile_rmanova_frequency(f'{ca_lab} 20 Hz vs 50 Hz N trajectories', pair_rmanova_n))
print(f"✓ Saved P trajectories to {out_p}")
print(f"✓ Saved N trajectories to {out_n}")


In [ ]:
# === Fig 5h — Frequency-dependent N fold increase across calcium ===
# Ratio analysis: compare N_50Hz / N_20Hz across calcium conditions per stimulus.
# The statistic is run on log ratios so equal fold changes are treated symmetrically around 1x.
from scipy.stats import chi2
from statsmodels.stats.multitest import multipletests

_RATIO_BOOTSTRAPS = 10000
_RATIO_RANDOM_STATE = 20260617
_RATIO_MIN_N = 3


def _positive_finite(values):
    arr = np.asarray(values, dtype=float)
    return arr[np.isfinite(arr) & (arr > 0)]


def _bootstrap_log_ratio_of_medians(values_20hz, values_50hz, *, n_boot=_RATIO_BOOTSTRAPS, seed=0):
    v20 = _positive_finite(values_20hz)
    v50 = _positive_finite(values_50hz)
    if len(v20) < _RATIO_MIN_N or len(v50) < _RATIO_MIN_N:
        return {
            'log_ratio': np.nan,
            'ratio': np.nan,
            'se_log_ratio': np.nan,
            'ci_low': np.nan,
            'ci_high': np.nan,
            'n_20hz': len(v20),
            'n_50hz': len(v50),
            'boot_log_ratios': np.array([], dtype=float),
        }

    rng = np.random.default_rng(seed)
    log_ratio = np.log(np.nanmedian(v50)) - np.log(np.nanmedian(v20))
    idx20 = rng.integers(0, len(v20), size=(int(n_boot), len(v20)))
    idx50 = rng.integers(0, len(v50), size=(int(n_boot), len(v50)))
    boot_log_ratios = np.log(np.nanmedian(v50[idx50], axis=1)) - np.log(np.nanmedian(v20[idx20], axis=1))
    boot_log_ratios = boot_log_ratios[np.isfinite(boot_log_ratios)]
    if boot_log_ratios.size < 2:
        se_log_ratio = np.nan
        ci_low = np.nan
        ci_high = np.nan
    else:
        se_log_ratio = float(np.nanstd(boot_log_ratios, ddof=1))
        ci_low, ci_high = np.nanpercentile(np.exp(boot_log_ratios), [2.5, 97.5])

    return {
        'log_ratio': float(log_ratio),
        'ratio': float(np.exp(log_ratio)),
        'se_log_ratio': se_log_ratio,
        'ci_low': float(ci_low) if np.isfinite(ci_low) else np.nan,
        'ci_high': float(ci_high) if np.isfinite(ci_high) else np.nan,
        'n_20hz': len(v20),
        'n_50hz': len(v50),
        'boot_log_ratios': boot_log_ratios,
    }


def _cochran_q_log_ratio_test(log_ratios, se_log_ratios):
    logs = np.asarray(log_ratios, dtype=float)
    ses = np.asarray(se_log_ratios, dtype=float)
    valid = np.isfinite(logs) & np.isfinite(ses) & (ses > 0)
    if valid.sum() < 2:
        return np.nan, np.nan, int(valid.sum()) - 1
    logs = logs[valid]
    weights = 1.0 / np.square(ses[valid])
    weighted_mean = np.sum(weights * logs) / np.sum(weights)
    q_stat = float(np.sum(weights * np.square(logs - weighted_mean)))
    df = int(valid.sum() - 1)
    p_value = float(chi2.sf(q_stat, df))
    return q_stat, p_value, df


ratio_rows = []
ratio_curve_specs = []
for ca_idx, (ca_lab, spec20, spec50) in enumerate(freq_pairs):
    label20, color20 = spec20
    label50, color50 = spec50
    if label20 not in cond_results_E or label50 not in cond_results_E:
        continue
    mat20 = np.asarray(cond_results_E[label20]['E_N'], dtype=float)
    mat50 = np.asarray(cond_results_E[label50]['E_N'], dtype=float)
    ratios = []
    ci_low = []
    ci_high = []
    n20 = []
    n50 = []
    log_ratios = []
    se_log_ratios = []
    for pulse_idx, pulse_num in enumerate(x_stim):
        res = _bootstrap_log_ratio_of_medians(
            mat20[:, pulse_idx],
            mat50[:, pulse_idx],
            seed=_RATIO_RANDOM_STATE + 1000 * ca_idx + int(pulse_num),
        )
        ratios.append(res['ratio'])
        ci_low.append(res['ci_low'])
        ci_high.append(res['ci_high'])
        n20.append(res['n_20hz'])
        n50.append(res['n_50hz'])
        log_ratios.append(res['log_ratio'])
        se_log_ratios.append(res['se_log_ratio'])
        ratio_rows.append({
            'Calcium': ca_lab,
            'Stimulus': int(pulse_num),
            'N50_over_N20': res['ratio'],
            'log_N50_over_N20': res['log_ratio'],
            'SE_log_ratio': res['se_log_ratio'],
            'CI95_low': res['ci_low'],
            'CI95_high': res['ci_high'],
            'n_20Hz': res['n_20hz'],
            'n_50Hz': res['n_50hz'],
        })
    ratio_curve_specs.append({
        'label': ca_lab,
        'color': color50,
        'ratios': np.asarray(ratios, dtype=float),
        'ci_low': np.asarray(ci_low, dtype=float),
        'ci_high': np.asarray(ci_high, dtype=float),
        'n_20hz': np.asarray(n20, dtype=int),
        'n_50hz': np.asarray(n50, dtype=int),
        'log_ratios': np.asarray(log_ratios, dtype=float),
        'se_log_ratios': np.asarray(se_log_ratios, dtype=float),
    })

ratio_fold_n_df = pd.DataFrame(ratio_rows)
ratio_stat_rows = []
for pulse_idx, pulse_num in enumerate(x_stim):
    logs = [spec['log_ratios'][pulse_idx] for spec in ratio_curve_specs]
    ses = [spec['se_log_ratios'][pulse_idx] for spec in ratio_curve_specs]
    q_stat, p_raw, df = _cochran_q_log_ratio_test(logs, ses)
    ratio_stat_rows.append({
        'Stimulus': int(pulse_num),
        'Q_log_ratio': q_stat,
        'df': df,
        'p_value_raw': p_raw,
    })
ratio_fold_n_stats_df = pd.DataFrame(ratio_stat_rows)
valid_p = ratio_fold_n_stats_df['p_value_raw'].notna().to_numpy()
ratio_fold_n_stats_df['p_value_corrected'] = np.nan
ratio_fold_n_stats_df['significant'] = False
if valid_p.any():
    _, p_corr, _, _ = multipletests(ratio_fold_n_stats_df.loc[valid_p, 'p_value_raw'], alpha=0.05, method='fdr_bh')
    ratio_fold_n_stats_df.loc[valid_p, 'p_value_corrected'] = p_corr
    ratio_fold_n_stats_df.loc[valid_p, 'significant'] = p_corr < 0.05

fig_ratio, ax_ratio = make_figure_grid(figsize=(6.8, 5.2))
for spec in ratio_curve_specs:
    y = spec['ratios']
    lo = spec['ci_low']
    hi = spec['ci_high']
    ax_ratio.plot(x_stim, y, color=spec['color'], marker='o', lw=1.4, ms=3.5, label=spec['label'])
    valid_band = np.isfinite(lo) & np.isfinite(hi)
    if valid_band.any():
        ax_ratio.fill_between(x_stim, lo, hi, where=valid_band, color=spec['color'], alpha=0.12, linewidth=0)
ax_ratio.axhline(1, color='0.35', lw=0.9, ls='--')
style_ax(ax_ratio, 'Stimulus', '$N_{50Hz} / N_{20Hz}$', 'Frequency-dependent N fold increase')
add_legend(ax_ratio, loc='best')

all_hi = np.concatenate([spec['ci_high'][np.isfinite(spec['ci_high'])] for spec in ratio_curve_specs if np.isfinite(spec['ci_high']).any()]) if ratio_curve_specs else np.array([])
all_lo = np.concatenate([spec['ci_low'][np.isfinite(spec['ci_low'])] for spec in ratio_curve_specs if np.isfinite(spec['ci_low']).any()]) if ratio_curve_specs else np.array([])
ymax = max(float(np.nanmax(all_hi)) if all_hi.size else 1.5, 1.2)
ymin = min(float(np.nanmin(all_lo)) if all_lo.size else 0.8, 1.0)
ypad = 0.12 * (ymax - ymin if ymax > ymin else ymax)
ax_ratio.set_ylim(max(0, ymin - ypad), ymax + 2.5 * ypad)
y_star = ymax + 0.55 * ypad
for _, row in ratio_fold_n_stats_df.iterrows():
    if bool(row.get('significant', False)):
        p_corr = row.get('p_value_corrected', np.nan)
        stars = '***' if p_corr < 0.001 else '**' if p_corr < 0.01 else '*'
        ax_ratio.text(row['Stimulus'], y_star, stars, ha='center', va='bottom', fontsize=9, fontweight='bold', clip_on=False)

out_ratio_n = OUTPUT_DIR / 'Fig5_h_bootstrap_mle_n_50hz_over_20hz_fold_change_across_ca.pdf'
finalize_figure(fig_ratio, title=f'{meth}: N 50 Hz / 20 Hz fold change', save_path=out_ratio_n)


def _format_ratio_stat_pvalues(stats_df):
    vals = []
    for _, row in stats_df.iterrows():
        p_corr = row.get('p_value_corrected', np.nan)
        if np.isfinite(p_corr):
            vals.append(f"S{int(row['Stimulus'])}={p_corr:.3g}")
    return ', '.join(vals) if vals else 'none'

print('=== Frequency-dependent N fold increase ratios ===')
for spec in ratio_curve_specs:
    ratio_txt = ', '.join(f"S{int(stim)}={ratio:.3g}" for stim, ratio in zip(x_stim, spec['ratios']))
    n20_min = int(np.nanmin(spec['n_20hz'])) if len(spec['n_20hz']) else 0
    n50_min = int(np.nanmin(spec['n_50hz'])) if len(spec['n_50hz']) else 0
    print(f"{spec['label']} N50/N20 fold-change: {ratio_txt}; minimum positive-N n: 20 Hz={n20_min}, 50 Hz={n50_min}")
print('Across-calcium ratio heterogeneity test per stimulus: Cochran Q on log(N50/N20), FDR-corrected across stimuli')
print(f"N50/N20 ratio heterogeneity FDR-corrected p values: {_format_ratio_stat_pvalues(ratio_fold_n_stats_df)}")
sig_stim = ratio_fold_n_stats_df.loc[ratio_fold_n_stats_df['significant'], 'Stimulus'].astype(int).tolist()
print(f"FDR-significant stimuli: {','.join(map(str, sig_stim)) if sig_stim else 'none'}")
print(f"✓ Saved N fold-change ratio trajectories to {out_ratio_n}")


In [ ]:
# === Aux — 50 Hz PCA shift at 2.5 mM ===
# Show the single-condition PCA shift between 20 Hz and 50 Hz at 2.5 mM calcium as an auxiliary panel.

wt_20_2_5_coords = np.asarray(pca_data['WT_pooled'])
hz50_2_5_coords = np.asarray(pca_data['50Hz_2_5Ca'])

center_20_2_5 = wt_20_2_5_coords.mean(axis=0)
center_50_2_5 = hz50_2_5_coords.mean(axis=0)

pc1_variance, pc2_variance = PCA_RESULTS['explained_variance']

fig, ax = plt.subplots(figsize=(8, 7))

ax.scatter(
    wt_20_2_5_coords[:, 0], wt_20_2_5_coords[:, 1],
    s=12, c='lightgray', alpha=0.35,
    label=f'20Hz 2.5mM / WT pooled (n={len(wt_20_2_5_coords)})'
)

ax.scatter(
    hz50_2_5_coords[:, 0], hz50_2_5_coords[:, 1],
    s=24, marker='D', c=get_50hz_ca_color('2.5mM'), alpha=0.80,
    edgecolors='#c9894f', linewidths=0.0,
    label=f'50Hz 2.5mM (n={len(hz50_2_5_coords)})'
)

ax.scatter(
    center_20_2_5[0], center_20_2_5[1],
    s=110, marker='o', c='white',
    edgecolors='black', linewidths=0.0, zorder=5,
    label='20Hz 2.5mM centroid'
)

ax.scatter(
    center_50_2_5[0], center_50_2_5[1],
    s=110, marker='X', c=get_50hz_ca_color('2.5mM'),
    edgecolors='black', linewidths=0.0, zorder=6,
    label='50Hz 2.5mM centroid'
)

ax.annotate(
    '',
    xy=(center_50_2_5[0], center_50_2_5[1]),
    xytext=(center_20_2_5[0], center_20_2_5[1]),
    arrowprops=dict(arrowstyle='->', color=get_50hz_ca_color('2.5mM'), lw=2.5, mutation_scale=18),
    zorder=7
)

ax.set_xlabel(f'PC1 ({pc1_variance:.1%} variance)')
ax.set_ylabel(f'PC2 ({pc2_variance:.1%} variance)')
ax.set_title('2.5mM Ca2+ PCA Shift: 20Hz → 50Hz')
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=6)
plt.tight_layout()

output_file = OUTPUT_DIR / 'Aux_50hz_pca_shift_2p5mM.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'20Hz 2.5mM centroid: ({center_20_2_5[0]:.3f}, {center_20_2_5[1]:.3f})')
print(f'50Hz 2.5mM centroid: ({center_50_2_5[0]:.3f}, {center_50_2_5[1]:.3f})')
print(f'Centroid shift: {np.linalg.norm(center_50_2_5 - center_20_2_5):.3f}')
print(f'Saved to {output_file}')


In [ ]:
# === Aux — PPR profiles per Ca²⁺ at 50 Hz ===
# Overlay all 50Hz PPR profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Use a shared y-axis max across all three plots

if 'ppr_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (the cell defining plot_profile_group_comparison).')

profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), 'Aux_50hz_ppr_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), 'Aux_50hz_ppr_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), 'Aux_50hz_ppr_profiles_4p0mM.pdf'),
]

prepared_profiles = []
global_ymax = 0

for title, df_cond, color, save_name in profiles_by_condition:
    profiles = ppr_profiles_matrix(df_cond)

    if profiles.size == 0:
        prepared_profiles.append((title, profiles, color, save_name))
        continue

    mean_profile = np.nanmean(profiles, axis=0)
    sem_profile = np.nanstd(profiles, axis=0) / np.sqrt(profiles.shape[0])
    cond_ymax = np.nanmax(np.vstack([profiles, mean_profile + sem_profile]))
    global_ymax = max(global_ymax, cond_ymax)

    prepared_profiles.append((title, profiles, color, save_name))

global_ylim = (0, global_ymax * 1.05 if global_ymax > 0 else 1)

for title, profiles, color, save_name in prepared_profiles:
    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles,
        color=color,
        label='Mean',
        individual_alpha=0.25,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid PPR profiles')
        plt.close(fig)
        continue

    pulse_numbers, _, _ = result
    plot_profile_group_comparison(
        ax,
        [{
            'matrix': profiles,
            'color': color,
            'marker': 'o',
            'label': f'Mean (n={profiles.shape[0]})',
            'linewidth': 2.5,
            'sem_alpha': 0.18,
        }],
        pulse_numbers=np.asarray(pulse_numbers, dtype=float),
        title=(f'50Hz PPR Profiles - {title}\n' f'(n={profiles.shape[0]} boutons)'),
        ylabel='PPR (A_n/A_1)',
        finalize_kwargs={
            'ylim': global_ylim,
            'unity_kwargs': {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1},
            'legend': True,
            'legend_loc': 'best',
        },
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')


In [ ]:
# === Aux — Failure-rate profiles per Ca²⁺ at 50 Hz ===
# Overlay all 50Hz failure-rate profiles, one plot per concentration (1.5, 2.5, and 4mM)
# Explicit failures are defined per event from min(AMP_CORR, AMP_UNCORR) < thr_shared.

if 'failure_profiles_matrix' not in globals() or 'plot_ppr_profiles_overlay' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared failure/PPR helper cell first (the cell defining plot_profile_group_comparison).')

failure_profiles_by_condition = [
    ('1.5 mM Ca', PCA_Data_50Hz_1_5_Ca, get_50hz_ca_color('1.5mM'), 'Aux_50hz_failure_profiles_1p5mM.pdf'),
    ('2.5 mM Ca', PCA_Data_50Hz_2_5_Ca, get_50hz_ca_color('2.5mM'), 'Aux_50hz_failure_profiles_2p5mM.pdf'),
    ('4 mM Ca', PCA_Data_50Hz_4_Ca, get_50hz_ca_color('4mM'), 'Aux_50hz_failure_profiles_4p0mM.pdf'),
]

for title, df_cond, color, save_name in failure_profiles_by_condition:
    profiles = failure_profiles_matrix(df_cond, trials_all, min_trials=5)
    n_valid = int(np.sum(np.isfinite(profiles).any(axis=1))) if profiles.size else 0

    if profiles.size == 0 or n_valid == 0:
        print(f'Skipping {title}: no valid failure-rate profiles')
        continue

    fig, ax = make_figure_grid(figsize=(7, 5))
    result = plot_ppr_profiles_overlay(
        ax,
        profiles[np.isfinite(profiles).any(axis=1)],
        color=color,
        label='Mean',
        individual_alpha=0.20,
        individual_lw=1.0,
        mean_lw=2.5,
        sem_alpha=0.18,
    )

    if result is None:
        print(f'Skipping {title}: no valid failure-rate profiles')
        plt.close(fig)
        continue

    pulse_numbers, mean_profile, sem_profile = result
    valid_profiles = profiles[np.isfinite(profiles).any(axis=1)]
    plot_profile_group_comparison(
        ax,
        [{
            'matrix': valid_profiles,
            'color': color,
            'marker': 'o',
            'label': f'Mean (n={n_valid})',
            'linewidth': 2.5,
            'sem_alpha': 0.18,
        }],
        pulse_numbers=np.asarray(pulse_numbers, dtype=float),
        title=f'50Hz Failure Profiles - {title} (n={n_valid} boutons)',
        ylabel='Failure rate',
        finalize_kwargs={
            'ylim': (0, 1),
            'unity_line': False,
            'legend': True,
            'legend_loc': 'best',
        },
    )
    fig.tight_layout()
    output_file = OUTPUT_DIR / save_name
    fig.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved to {output_file}')
    print(
        f"{title}: n={n_valid} | "
        f"Fail1={mean_profile[0]:.3f}±{sem_profile[0]:.3f} | "
        f"Fail2={mean_profile[1]:.3f}±{sem_profile[1]:.3f} | "
        f"Fail10={mean_profile[-1]:.3f}±{sem_profile[-1]:.3f}"
    )


#### Cluster composition at 50 Hz

The class-composition view asks whether raising frequency redistributes boutons across the WT classes or mainly shifts their continuous coordinates without a strong class reassignment. This complements the centroid-trajectory analysis with a discrete summary. 


In [ ]:
# === Aux — Cluster composition across 50 Hz conditions ===
# Compare WT-cluster composition across the 50 Hz conditions as an auxiliary frequency summary.

from sklearn.neighbors import KNeighborsClassifier

# Compare cluster distributions between 20Hz WT and 50Hz for each calcium concentration
# Three separate figures: 1.5mM Ca, 2.5mM Ca, 4mM Ca


# Get WT cluster counts (20Hz baseline)
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
all_clusters = sorted(wt_cluster_counts.index)

# Train kNN classifier on WT pooled data
knn = KNeighborsClassifier(n_neighbors=5, weights='distance')
knn.fit(pca_coordinates, cluster_assignments)

# Project 50Hz conditions and assign to clusters
hz50_1_5_assignments = knn.predict(pca_data['50Hz_1_5Ca'])
hz50_2_5_assignments = knn.predict(pca_data['50Hz_2_5Ca'])
hz50_4_assignments   = knn.predict(pca_data['50Hz_4Ca'])

hz50_1_5_counts = pd.Series(hz50_1_5_assignments).value_counts()
hz50_2_5_counts = pd.Series(hz50_2_5_assignments).value_counts()
hz50_4_counts   = pd.Series(hz50_4_assignments).value_counts()

#Project 20Hz calcium condition data and assign to clusters
hz20_1_5_assignments = knn.predict(pca_data['WT_1_5Ca'])
hz20_4_assignments   = knn.predict(pca_data['WT_4Ca'])

hz20_1_5_counts = pd.Series(hz20_1_5_assignments).value_counts()
hz20_4_counts   = pd.Series(hz20_4_assignments).value_counts()

# Ensure all clusters represented
hz50_1_5_complete = pd.Series([hz50_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_2_5_complete = pd.Series([hz50_2_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz50_4_complete   = pd.Series([hz50_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

hz20_1_5_complete = pd.Series([hz20_1_5_counts.get(c, 0) for c in all_clusters], index=all_clusters)
hz20_4_complete   = pd.Series([hz20_4_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)

hz50_1_5_percentages = 100 * hz50_1_5_complete / len(hz50_1_5_assignments)
hz50_2_5_percentages = 100 * hz50_2_5_complete / len(hz50_2_5_assignments)
hz50_4_percentages   = 100 * hz50_4_complete / len(hz50_4_assignments)

hz20_1_5_percentages = 100 * hz20_1_5_complete / len(hz20_1_5_assignments)
hz20_4_percentages   = 100 * hz20_4_complete / len(hz20_4_assignments)



In [ ]:
# === Aux — AMP1 boxplot 20 vs 50 Hz at 2.5 mM ===
# Compare AMP1 between 20 Hz and 50 Hz at 2.5 mM calcium as an auxiliary scalar panel.

# Boxplot comparing AMP1 between 20Hz and 50Hz at 2.5mM Ca

# Extract data
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()

# Create figure
fig, ax = make_figure_grid(panel_kind='boxplot_compact')

# Prepare data for boxplot
data = [amp1_20hz.values, amp1_50hz.values]
positions = [0, 1]
colors = [get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [f'AMP1 20Hz\n(n={len(amp1_20hz)})', f'AMP1 50Hz\n(n={len(amp1_50hz)})']

res_amp1 = plot_boxplot_analysis(ax, [make_box_group(labels[0], values=amp1_20hz, color=colors[0]), make_box_group(labels[1], values=amp1_50hz, color=colors[1])], paired=False, show_points=True, show_stats=True, parametric_stat=False, title='AMP1 Comparison: 20Hz vs 50Hz at 2.5mM Ca', ylabel='Amplitude (ΔF/F)')
p_amp1 = res_amp1['p_value']

plt.tight_layout()
output_file = OUTPUT_DIR / "Aux_50hz_50hz_amp1_20hz_vs_50hz_boxplot.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== AMP1 COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"AMP1 20Hz: {amp1_20hz.mean():.3f} ± {amp1_20hz.std():.3f} (n={len(amp1_20hz)})")
print(f"AMP1 50Hz: {amp1_50hz.mean():.3f} ± {amp1_50hz.std():.3f} (n={len(amp1_50hz)})")
print(f"  Mann-Whitney p = {p_amp1:.4g}")
print(f"\n✓ Saved to {output_file}")


In [ ]:
# === Aux — AMP1 boxplots 20 vs 50 Hz across Ca²⁺ ===
# Boxplots: AMP1 20Hz vs 50Hz for 1.5mM, 2.5mM, and 4mM
def _get_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'ns'

def _plot_amp1_comparison(ax, data20, data50, title, colors=('lightgray', 'gray'), ylims=None):
    res = plot_boxplot_analysis(ax, [make_box_group('20Hz', values=data20, color=colors[0]), make_box_group('50Hz', values=data50, color=colors[1])], paired=False, show_points=True, show_stats=True, parametric_stat=False, title=title, ylabel='Amplitude (ΔF/F)')
    p = res['p_value']
    tick_positions = ax.get_xticks()
    ax.set_xticks(tick_positions)
    ax.set_xticklabels([f'20Hz (n={len(data20)})', f'50Hz (n={len(data50)})'], fontsize=6)
    ax.set_ylabel('AMP1 (ΔF/F)', fontsize=8)
    ax.set_title(title, fontsize=8, fontweight='bold')
    ax.grid(False)
    if ylims is not None:
        ax.set_ylim(ylims)

# Extract AMP1 data
amp1_20_15 = PCA_Data_WT_Low_Ca['AMP1'].dropna()
amp1_50_15 = PCA_Data_50Hz_1_5_Ca['AMP1'].dropna()
amp1_20_25 = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50_25 = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()
amp1_20_4 = PCA_Data_WT_High_Ca['AMP1'].dropna()
amp1_50_4 = PCA_Data_50Hz_4_Ca['AMP1'].dropna()

# Compute global y-limits for all boxplots
all_amp1 = pd.concat([amp1_20_15, amp1_50_15, amp1_20_25, amp1_50_25, amp1_20_4, amp1_50_4])
ymin = all_amp1.min()
ymax = all_amp1.max()
yrange = ymax - ymin
ylims = (ymin - 0.1 * yrange, ymax + 0.1 * yrange)

fig, axes = make_figure_grid(1, 3, panel_kind='boxplot_compact')

_plot_amp1_comparison(axes[0], amp1_20_15, amp1_50_15, 'AMP1: 1.5mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('1.5mM'), get_50hz_ca_color('1.5mM')), ylims=ylims)
_plot_amp1_comparison(axes[1], amp1_20_25, amp1_50_25, 'AMP1: 2.5mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')), ylims=ylims)
_plot_amp1_comparison(axes[2], amp1_20_4,  amp1_50_4,  'AMP1: 4mM (20Hz vs 50Hz)', colors=(get_wt_ca_color('4mM'), get_50hz_ca_color('4mM')), ylims=ylims)

plt.tight_layout()
out = OUTPUT_DIR / "Aux_50hz_50hz_amp1_20hz_vs_50hz_all_calcium.pdf"
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

# Summary
def _summary(name, d20, d50):
    if len(d20) and len(d50):
        _, p = mannwhitneyu(d20, d50, alternative='two-sided')
        print(f"{name}: 20Hz {d20.mean():.3f}±{d20.std():.3f} (n={len(d20)}), "
              f"50Hz {d50.mean():.3f}±{d50.std():.3f} (n={len(d50)}) | p={p:.4g}")
    else:
        print(f"{name}: insufficient data")

print("=== AMP1 20Hz vs 50Hz by Ca ===")
_summary("1.5mM", amp1_20_15, amp1_50_15)
_summary("2.5mM", amp1_20_25, amp1_50_25)
_summary("4mM",   amp1_20_4,  amp1_50_4)
print(f"✓ Saved to {out}")

In [ ]:
# === Aux — Stars helper ===
# Reuse shared helper: _get_stars
# Boxplots: PPR2/1 20Hz vs 50Hz for 1.5mM, 2.5mM, 4mM
print(f"✓ Saved to {out}")

In [ ]:
# === Aux — A2 vs A1 pooled comparison ===
# Reuse shared helper: add_p_value

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare pooled AMP1 against AMP2 at 20 Hz and 50 Hz (2.5 mM Ca)
amp1_20hz = PCA_Data_WT_Pooled['AMP1'].dropna()
amp1_50hz = PCA_Data_50Hz_2_5_Ca['AMP1'].dropna()
amp1_combined = pd.concat([amp1_20hz, amp1_50hz], ignore_index=True)

amp2_20hz = PCA_Data_WT_Pooled['AMP2'].dropna()
amp2_50hz = PCA_Data_50Hz_2_5_Ca['AMP2'].dropna()

fig, ax = make_figure_grid(panel_kind='boxplot_compact')

data = [amp1_combined.values, amp2_20hz.values, amp2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [
    f'AMP1\n20Hz+50Hz\n(n={len(amp1_combined)})',
    f'AMP2\n20Hz\n(n={len(amp2_20hz)})',
    f'AMP2\n50Hz\n(n={len(amp2_50hz)})',
]

plot_boxplot_analysis(ax, [make_box_group(labels[0], values=amp1_combined, color=colors[0]), make_box_group(labels[1], values=amp2_20hz, color=colors[1]), make_box_group(labels[2], values=amp2_50hz, color=colors[2])], paired=False, show_points=True, show_stats=False, parametric_stat=False, title='AMP2 versus pooled AMP1 at 2.5 mM Ca', ylabel='Amplitude (ΔF/F)', point_alpha=0.4, point_size=20)

_, p_amp1_vs_amp2_20 = mannwhitneyu(amp1_combined, amp2_20hz, alternative='two-sided')
_, p_amp1_vs_amp2_50 = mannwhitneyu(amp1_combined, amp2_50hz, alternative='two-sided')
_, p_amp2_20_vs_50 = mannwhitneyu(amp2_20hz, amp2_50hz, alternative='two-sided')

ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
add_significance_bar(ax, 0, 1, p_amp1_vs_amp2_20, y=ymax + 0.06 * yrange, h=0.02 * yrange)
add_significance_bar(ax, 0, 2, p_amp1_vs_amp2_50, y=ymax + 0.15 * yrange, h=0.02 * yrange)
add_significance_bar(ax, 1, 2, p_amp2_20_vs_50, y=ymax + 0.24 * yrange, h=0.02 * yrange)

ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.35 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['AMP1 pooled', 'AMP2 20 Hz', 'AMP2 50 Hz'],
    loc='upper right',
)
ax.grid(False)

out = OUTPUT_DIR / 'Aux_50hz_50hz_amp2_vs_amp1_pooled.pdf'
plt.tight_layout()
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()

print('=== AMP2 versus pooled AMP1 @ 2.5 mM Ca ===')
print(f"AMP1 pooled: mean={amp1_combined.mean():.3f} ± {amp1_combined.std():.3f} (n={len(amp1_combined)})")
print(f"AMP2 20 Hz:  mean={amp2_20hz.mean():.3f} ± {amp2_20hz.std():.3f} (n={len(amp2_20hz)}) | p vs pooled AMP1={p_amp1_vs_amp2_20:.4g}")
print(f"AMP2 50 Hz:  mean={amp2_50hz.mean():.3f} ± {amp2_50hz.std():.3f} (n={len(amp2_50hz)}) | p vs pooled AMP1={p_amp1_vs_amp2_50:.4g}")
print(f"20 Hz vs 50 Hz AMP2: p={p_amp2_20_vs_50:.4g}")
print(f"✓ Saved to {out}")


In [ ]:
# === Aux — %Fail1 20 vs 50 Hz ===
# Reuse shared helper: _get_stars

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare %Fail1 between 20 Hz and 50 Hz at 2.5 mM Ca
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()

fig, ax = make_figure_grid(panel_kind='boxplot_compact')

data = [fail1_20hz.values, fail1_50hz.values]
positions = [0, 1]
colors = [get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [f'%Fail1 20Hz\n(n={len(fail1_20hz)})', f'%Fail1 50Hz\n(n={len(fail1_50hz)})']

plot_boxplot_analysis(ax, [make_box_group(labels[0], values=fail1_20hz, color=colors[0]), make_box_group(labels[1], values=fail1_50hz, color=colors[1])], paired=False, show_points=True, show_stats=False, parametric_stat=False, title='%Fail1: 20 Hz versus 50 Hz at 2.5 mM Ca', ylabel='Failure rate (%)')

_, p_fail1 = mannwhitneyu(fail1_20hz, fail1_50hz, alternative='two-sided')
ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
ax.plot([0, 0, 1, 1], [ymax + 0.05 * yrange, ymax + 0.08 * yrange, ymax + 0.08 * yrange, ymax + 0.05 * yrange], color='black', lw=1)
text_obj = ax.text(0.5, ymax + 0.09 * yrange, _get_stars(p_fail1), ha='center', va='bottom', fontsize=5)
setattr(text_obj, '_keep_in_panel_annotation', True)

ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.18 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['20 Hz', '50 Hz'],
    loc='upper right',
)
ax.grid(False)

output_file = OUTPUT_DIR / 'Aux_50hz_50hz_fail1_20hz_vs_50hz.pdf'
plt.tight_layout()
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print('=== %Fail1 comparison: 20 Hz versus 50 Hz @ 2.5 mM Ca ===')
print(f"20 Hz: {fail1_20hz.mean():.2f} ± {fail1_20hz.std():.2f} (n={len(fail1_20hz)})")
print(f"50 Hz: {fail1_50hz.mean():.2f} ± {fail1_50hz.std():.2f} (n={len(fail1_50hz)})")
print(f"Mann-Whitney p = {p_fail1:.4g}")
print(f"✓ Saved to {output_file}")


In [ ]:
# === Aux — %Fail2 vs %Fail1 pooled ===
# Reuse shared helper: add_p_value

from scipy.stats import mannwhitneyu
from matplotlib.patches import Patch

# Compare pooled %Fail1 against %Fail2 at 20 Hz and 50 Hz (2.5 mM Ca)
fail1_20hz = PCA_Data_WT_Pooled['%Fail1'].dropna()
fail1_50hz = PCA_Data_50Hz_2_5_Ca['%Fail1'].dropna()
fail1_combined = pd.concat([fail1_20hz, fail1_50hz], ignore_index=True)
fail2_20hz = PCA_Data_WT_Pooled['%Fail2'].dropna()
fail2_50hz = PCA_Data_50Hz_2_5_Ca['%Fail2'].dropna()

fig, ax = make_figure_grid(panel_kind='boxplot_compact')

data = [fail1_combined.values, fail2_20hz.values, fail2_50hz.values]
positions = [0, 1, 2]
colors = ['gray', get_wt_ca_color('2.5mM'), get_50hz_ca_color('2.5mM')]
labels = [
    f'%Fail1\n20Hz+50Hz\n(n={len(fail1_combined)})',
    f'%Fail2\n20Hz\n(n={len(fail2_20hz)})',
    f'%Fail2\n50Hz\n(n={len(fail2_50hz)})',
]

plot_boxplot_analysis(ax, [make_box_group(labels[0], values=fail1_combined, color=colors[0]), make_box_group(labels[1], values=fail2_20hz, color=colors[1]), make_box_group(labels[2], values=fail2_50hz, color=colors[2])], paired=False, show_points=True, show_stats=False, parametric_stat=False, title='%Fail2 versus pooled %Fail1 at 2.5 mM Ca', ylabel='Failure rate (%)', point_alpha=0.4, point_size=20)

_, p_fail1_vs_fail2_20 = mannwhitneyu(fail1_combined, fail2_20hz, alternative='two-sided')
_, p_fail1_vs_fail2_50 = mannwhitneyu(fail1_combined, fail2_50hz, alternative='two-sided')
_, p_fail2_20_vs_50 = mannwhitneyu(fail2_20hz, fail2_50hz, alternative='two-sided')

ymax = max(np.nanmax(vals) for vals in data)
ymin = min(np.nanmin(vals) for vals in data)
yrange = ymax - ymin if ymax > ymin else 1.0
add_significance_bar(ax, 0, 1, p_fail1_vs_fail2_20, y=ymax + 0.06 * yrange, h=0.02 * yrange)
add_significance_bar(ax, 0, 2, p_fail1_vs_fail2_50, y=ymax + 0.15 * yrange, h=0.02 * yrange)
add_significance_bar(ax, 1, 2, p_fail2_20_vs_50, y=ymax + 0.24 * yrange, h=0.02 * yrange)

ax.set_ylim(ymin - 0.08 * yrange, ymax + 0.35 * yrange)
add_legend(
    ax,
    handles=[Patch(facecolor=c, edgecolor='black', alpha=0.7) for c in colors],
    labels=['%Fail1 pooled', '%Fail2 20 Hz', '%Fail2 50 Hz'],
    loc='upper right',
)
ax.grid(False)

output_file = OUTPUT_DIR / 'Aux_50hz_50hz_fail2_vs_fail1_pooled.pdf'
plt.tight_layout()
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print('=== %Fail2 versus pooled %Fail1 @ 2.5 mM Ca ===')
print(f"%Fail1 pooled: mean={fail1_combined.mean():.2f} ± {fail1_combined.std():.2f} (n={len(fail1_combined)})")
print(f"%Fail2 20 Hz:  mean={fail2_20hz.mean():.2f} ± {fail2_20hz.std():.2f} (n={len(fail2_20hz)}) | p vs pooled %Fail1={p_fail1_vs_fail2_20:.4g}")
print(f"%Fail2 50 Hz:  mean={fail2_50hz.mean():.2f} ± {fail2_50hz.std():.2f} (n={len(fail2_50hz)}) | p vs pooled %Fail1={p_fail1_vs_fail2_50:.4g}")
print(f"20 Hz versus 50 Hz %Fail2: p={p_fail2_20_vs_50:.4g}")
print(f"✓ Saved to {output_file}")


In [ ]:
# === Aux — PPR2/PPR3 20 vs 50 Hz ===
# Extract PPR2/1 and PPR3/1 data for 20Hz WT and 50Hz WT at 2.5mM Ca
ppr2_1_20hz = PCA_Data_WT_Pooled['PPR2/1'].dropna()
ppr2_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR2/1'].dropna()

ppr3_1_20hz = PCA_Data_WT_Pooled['PPR3/1'].dropna()
ppr3_1_50hz = PCA_Data_50Hz_2_5_Ca['PPR3/1'].dropna()

# Create figure with two subplots
fig, (ax1, ax2) = make_figure_grid(1, 2, panel_kind='boxplot_compact')

groups_ppr2_1 = [
    make_box_group(f'PPR2/1 20Hz\n(n={len(ppr2_1_20hz)})', values=ppr2_1_20hz, color=get_wt_ca_color('2.5mM')),
    make_box_group(f'PPR2/1 50Hz\n(n={len(ppr2_1_50hz)})', values=ppr2_1_50hz, color=get_50hz_ca_color('2.5mM')),
]
groups_ppr3_1 = [
    make_box_group(f'PPR3/1 20Hz\n(n={len(ppr3_1_20hz)})', values=ppr3_1_20hz, color=get_wt_ca_color('2.5mM')),
    make_box_group(f'PPR3/1 50Hz\n(n={len(ppr3_1_50hz)})', values=ppr3_1_50hz, color=get_50hz_ca_color('2.5mM')),
]

res_ppr2_1 = plot_boxplot_analysis(ax1, groups_ppr2_1, show_points=True, show_stats=True, parametric_stat=False, title='PPR2/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', ylabel='Ratio')
res_ppr3_1 = plot_boxplot_analysis(ax2, groups_ppr3_1, show_points=True, show_stats=True, parametric_stat=False, title='PPR3/1 Comparison: 20Hz vs 50Hz @ 2.5mM Ca', ylabel='Ratio')
ax1.axhline(1, color='gray', linestyle='dotted', linewidth=1)
ax1.set_ylim(bottom=0)
ax2.axhline(1, color='gray', linestyle='dotted', linewidth=1)
ax2.set_ylim(bottom=0)
p_ppr2_1 = res_ppr2_1['p_value']
p_ppr3_1 = res_ppr3_1['p_value']

plt.tight_layout()
output_file = OUTPUT_DIR / "Aux_50hz_50hz_ppr2_1_ppr3_1_20hz_vs_50hz.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"=== PPR RATIO COMPARISON: 20Hz vs 50Hz @ 2.5mM Ca ===")
print(f"\nPPR2/1:")
print(f"  20Hz: {ppr2_1_20hz.mean():.3f} ± {ppr2_1_20hz.std():.3f} (n={len(ppr2_1_20hz)})")
print(f"  50Hz: {ppr2_1_50hz.mean():.3f} ± {ppr2_1_50hz.std():.3f} (n={len(ppr2_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr2_1:.4g}")

print(f"\nPPR3/1:")
print(f"  20Hz: {ppr3_1_20hz.mean():.3f} ± {ppr3_1_20hz.std():.3f} (n={len(ppr3_1_20hz)})")
print(f"  50Hz: {ppr3_1_50hz.mean():.3f} ± {ppr3_1_50hz.std():.3f} (n={len(ppr3_1_50hz)})")
print(f"  Mann-Whitney p = {p_ppr3_1:.4g}")

print(f"\n✓ Saved to {output_file}")

## Section 6 — Figure 6: Synapsin II perturbation


### Synapsin II perturbation

Synapsin II loss is examined after the WT reference, calcium, and frequency analyses so that genotype effects can be interpreted against the full WT framework. The retained SynII analyses focus on direct WT-versus-SynII comparisons and on the redistribution of SynII boutons within the WT reference space.

Several SynII panels in this notebook are more recent than the current manuscript wording and should be read as an extension of the same biological question: which subsets of bouton release properties depend on Synapsin II, and which parts of the WT state space persist without it?


#### Scope note

SynII-only HCPC figures are dropped. The analyses below keep only direct WT-versus-SynII comparisons and projection of SynII boutons into the WT reference PCA space.


In [ ]:
# === Fig 6a — WT vs SynII traces normalized to A1 ===
# Compare mean traces after scaling each bouton to its own A1 peak.
A1_START_LOCAL = float(globals().get('A1_START', 1.000))
A1_END_LOCAL = float(globals().get('A1_END', 1.049))


def _normalize_trace_rows_to_a1(rows_df):
    norm_rows = rows_df.copy(deep=True)
    norm_avg = []
    kept = []
    for _, row in rows_df.iterrows():
        tt = np.asarray(_trace_row_time(row, aligned=True), dtype=float)
        yy = np.asarray(row['Avg'], dtype=float)
        if tt.shape[0] != yy.shape[0]:
            norm_avg.append(yy)
            kept.append(False)
            continue
        a1_mask = (tt >= A1_START_LOCAL) & (tt <= A1_END_LOCAL)
        w = yy[a1_mask]
        if w.size == 0 or np.all(~np.isfinite(w)):
            norm_avg.append(yy)
            kept.append(False)
            continue
        a1 = float(np.nanmax(np.abs(w)))
        if not np.isfinite(a1) or abs(a1) < 1e-12:
            norm_avg.append(yy)
            kept.append(False)
            continue
        norm_avg.append(yy / a1)
        kept.append(True)
    norm_rows['Avg'] = norm_avg
    return norm_rows.loc[np.asarray(kept, dtype=bool)].reset_index(drop=True)

wt_rows = select_traces(trace_ids=PCA_Data_WT_Pooled_clustered['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_MEAN_SOURCE)
synii_rows = select_traces(trace_ids=PCA_Data_SynII['ID'], condition_names=get_synapsin_conditions(), source=TRACE_MEAN_SOURCE)
wt_rows_norm = _normalize_trace_rows_to_a1(wt_rows)
synii_rows_norm = _normalize_trace_rows_to_a1(synii_rows)
wt_norm_stats = compute_trace_stats(rows=wt_rows_norm, source=TRACE_MEAN_SOURCE)
synii_norm_stats = compute_trace_stats(rows=synii_rows_norm, source=TRACE_MEAN_SOURCE)

all_values = np.concatenate([
    np.asarray(wt_norm_stats['average'] - wt_norm_stats['sem'], float),
    np.asarray(wt_norm_stats['average'] + wt_norm_stats['sem'], float),
    np.asarray(synii_norm_stats['average'] - synii_norm_stats['sem'], float),
    np.asarray(synii_norm_stats['average'] + synii_norm_stats['sem'], float),
    np.array([1.0]),
])
y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
y_pad = max((y_max - y_min) * 0.05, 0.05)
y_lim = (y_min - y_pad, y_max + y_pad)
stim_times = [t for t in [1.0 + 0.05 * i for i in range(10)] if t <= 1.5]

fig, ax = make_figure_grid(figsize=(7, 5))
plot_trace_analysis(ax=ax, rows=wt_rows_norm, source=TRACE_MEAN_SOURCE, color=get_wt_ca_color('2.5mM'), label=f'WT pooled (n={wt_norm_stats["n"]})', display='mean_sem', style_axis=False)
plot_trace_analysis(ax=ax, rows=synii_rows_norm, source=TRACE_MEAN_SOURCE, color=SYNII_COLOR, label=f'SynII KO (n={synii_norm_stats["n"]})', display='mean_sem', stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.3}, zero_line=True, hlines=[1], hline_kwargs={'color': 'gray', 'linestyle': ':', 'linewidth': 1, 'alpha': 0.7}, xlim=TRACE_XLIM_20HZ, ylim=y_lim, xlabel='Time (s)', ylabel='ΔF/F (normalized to A1)', title='WT vs SynII mean traces normalized to A1', legend=True)
out_a = OUTPUT_DIR / 'Fig6_a_wt_vs_synii_mean_traces_a1_normalized.pdf'
finalize_figure(fig, title='WT vs SynII A1-normalized traces', save_path=out_a)
print(f"✓ WT normalized traces: n={wt_norm_stats['n']}")
print(f"✓ SynII normalized traces: n={synii_norm_stats['n']}")
print(f"✓ Saved to {out_a}")


In [ ]:
# === Fig 6b — WT vs SynII mean PPR profiles ===
# Compare train-normalized PPR profiles between WT pooled boutons and SynII KO boutons.
if 'ppr_profiles_matrix' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns and f'PPR{i}/1' in PCA_Data_SynII.columns]
wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
synii_profiles = ppr_profiles_matrix(PCA_Data_SynII, ppr_column_names=ppr_cols)
pulse_numbers = np.arange(1, wt_profiles.shape[1] + 1, dtype=float)

fig, ax = make_figure_grid(figsize=(7, 5))
plot_profile_group_comparison(
    ax,
    [
        {
            'matrix': wt_profiles,
            'color': get_wt_ca_color('2.5mM'),
            'marker': 'o',
            'label': f'WT pooled (n={wt_profiles.shape[0]})',
            'linewidth': 1.8,
            'sem_alpha': 0.18,
        },
        {
            'matrix': synii_profiles,
            'color': SYNII_COLOR,
            'marker': 's',
            'label': f'SynII KO (n={synii_profiles.shape[0]})',
            'linewidth': 1.8,
            'sem_alpha': 0.18,
        },
    ],
    pulse_numbers=pulse_numbers,
    stats_start_index=1,
    show_stats=True,
    parametric_stat=True,
    summary_label='WT pooled vs SynII mean PPR profile',
    title='WT vs SynII mean PPR profile',
    ylabel='Mean PPR (A$_n$/A$_1$)',
    finalize_kwargs={
        'ylim': (0.0, 1.65),
        'unity_kwargs': {'color': 'gray', 'linestyle': ':', 'linewidth': 1},
        'legend': True,
        'legend_loc': 'best',
    },
)
out_b = OUTPUT_DIR / 'Fig6_b_wt_vs_synii_mean_ppr_profile.pdf'
finalize_figure(fig, title='WT vs SynII mean PPR profile', save_path=out_b)
print(f'✓ WT profiles: {wt_profiles.shape[0]} | SynII profiles: {synii_profiles.shape[0]}')
print(f'✓ Saved to {out_b}')


#### SynII projection onto the WT PCA space

Projecting SynII boutons onto the WT PCA makes it possible to ask which parts of the WT reference manifold are lost, compressed, or overrepresented after Synapsin II deletion. This is the main state-space summary of the genotype effect.


In [ ]:
# === Fig 6c — SynII projection on WT PCA ===
# Project SynII data onto WT-trained PCA space.
make_figure(figsize=(8, 6))

synii_pca_coords = pca_data['SynII']
synii_centroid = np.mean(synii_pca_coords, axis=0)
plot_pca_analysis(
    plt.gca(),
    background={'use_cluster_colors': True, 'alpha': 0.18, 'label': 'WT pooled (2.5mM Ca)'},
    overlays=[
        make_pca_group(synii_pca_coords, label=f'SynII KO (n={len(synii_pca_coords)})', color=SYNII_COLOR, marker='*', alpha=0.82, linewidths=0.0),
        make_pca_group(np.array([synii_centroid]), label=f'SynII centroid ({synii_centroid[0]:.2f}, {synii_centroid[1]:.2f})', color=SYNII_COLOR, marker='X', linewidths=0.0, zorder=10),
    ],
    title='SynII KO projection onto WT PCA space',
    legend=False,
)
add_legend(plt.gca(), frameon=False)
plt.tight_layout()
out_c = OUTPUT_DIR / 'Fig6_c_synii_projection_on_wt_pca.pdf'
plt.savefig(out_c, dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space')
print(f'✓ SynII centroid: PC1={synii_centroid[0]:.3f}, PC2={synii_centroid[1]:.3f}')
print(f'✓ Saved to {out_c}')


In [ ]:
# === Fig 6d — WT vs SynII cumulative release normalized to A1 ===
# Build cumulative PPR profiles from the same train-normalized matrix used in Fig 6b.
if 'ppr_profiles_matrix' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns and f'PPR{i}/1' in PCA_Data_SynII.columns]
wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
synii_profiles = ppr_profiles_matrix(PCA_Data_SynII, ppr_column_names=ppr_cols)
pulse_numbers = np.arange(1, wt_profiles.shape[1] + 1, dtype=float)

fig6_cumulative_profiles = {
    'WT pooled': np.cumsum(wt_profiles, axis=1),
    'SynII KO': np.cumsum(synii_profiles, axis=1),
}

fig, ax = make_figure_grid(figsize=(7, 5))
plot_profile_group_comparison(
    ax,
    [
        {
            'matrix': fig6_cumulative_profiles['WT pooled'],
            'color': get_wt_ca_color('2.5mM'),
            'marker': 'o',
            'label': f'WT pooled (n={fig6_cumulative_profiles["WT pooled"].shape[0]})',
            'linewidth': 1.6,
            'sem_alpha': 0.16,
        },
        {
            'matrix': fig6_cumulative_profiles['SynII KO'],
            'color': SYNII_COLOR,
            'marker': 's',
            'label': f'SynII KO (n={fig6_cumulative_profiles["SynII KO"].shape[0]})',
            'linewidth': 1.6,
            'sem_alpha': 0.16,
        },
    ],
    pulse_numbers=pulse_numbers,
    stats_start_index=1,
    show_stats=True,
    parametric_stat=True,
    summary_label='WT vs SynII cumulative PPR profile',
    title='WT vs SynII cumulative PPR profile',
    ylabel='Normalized cumulative slope',
    finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'best'},
)
out_d = OUTPUT_DIR / 'Fig6_d_wt_vs_synii_cumulative_norm_a1.pdf'
finalize_figure(fig, title='WT vs SynII cumulative PPR profile', save_path=out_d)
print(f"✓ WT cumulative rows: {fig6_cumulative_profiles['WT pooled'].shape[0]}")
print(f"✓ SynII cumulative rows: {fig6_cumulative_profiles['SynII KO'].shape[0]}")
print(f"✓ Saved to {out_d}")


In [ ]:
# === Fig 6e — Final slope norm ===
# Compare the end-of-train slope from the A1-normalized cumulative curves.
from scipy.stats import linregress

if 'fig6_cumulative_profiles' not in globals() or 'pulse_numbers' not in globals():
    raise RuntimeError('Run Fig 6d first so cumulative profiles and pulse_numbers are available.')

fig6_tail_len = int(min(mat.shape[1] for mat in fig6_cumulative_profiles.values()))
fig6_nfit = int(min(globals().get('N_FIT_LAST', 4), fig6_tail_len, len(pulse_numbers)))
fig6_x_tail = pulse_numbers[-fig6_nfit:]
fig6_slope_arrays = {}
for label, mat in fig6_cumulative_profiles.items():
    local_slopes = np.full(mat.shape[0], np.nan)
    for i, row in enumerate(mat):
        y_tail = np.asarray(row[-fig6_nfit:], dtype=float)
        ok = np.isfinite(y_tail)
        if ok.sum() >= 2:
            local_slopes[i] = linregress(fig6_x_tail[ok], y_tail[ok]).slope
    fig6_slope_arrays[label] = local_slopes[np.isfinite(local_slopes)]

fig, ax = make_figure_grid(panel_kind='boxplot_compact')
res = plot_boxplot_analysis(
    ax,
    [
        make_box_group(f'WT\n(n={len(fig6_slope_arrays["WT pooled"])})', values=fig6_slope_arrays['WT pooled'], color=get_wt_ca_color('2.5mM')),
        make_box_group(f'SynII\n(n={len(fig6_slope_arrays["SynII KO"])})', values=fig6_slope_arrays['SynII KO'], color=SYNII_COLOR),
    ],
    paired=False,
    show_points=True,
    show_stats=True,
    parametric_stat=False,
    title='Final slope norm',
    ylabel='Final slope norm',
)
out_e = OUTPUT_DIR / 'Fig6_e_wt_vs_synii_cumulative_slope_boxplot.pdf'
finalize_figure(fig, title='WT vs SynII cumulative slope', save_path=out_e)
if 'p_value' in res and np.isfinite(res['p_value']):
    print(f"Final slope comparison p = {res['p_value']:.4g}")
print(f'✓ Saved to {out_e}')

In [ ]:
# === Fig 6f — WT vs SynII P MLE trajectories ===
# Restore the direct WT-versus-SynII binomial P comparison in the main Figure 6 block.
if 'cond_results_E' not in globals():
    raise RuntimeError('Run the shared WT condition MLE cell first (cell building cond_results_E).')

if 'F_P' not in globals() or 'F_N' not in globals() or not np.isfinite(np.asarray(F_P, dtype=float)).any():
    _synii_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
    _synii_trials = _synii_trials[_synii_trials['condition'] == 'SynII']
    tg_F = defaultdict(list)
    for _, r in _synii_trials.iterrows():
        key = extract_base_name(str(r['file']).strip())
        tg_F[key].append(get_count_row_with_failures(r, Q_D))

    synii_xy_all = np.array(pca_data['SynII'])
    synii_ids = PCA_Data_SynII['ID'].astype(str).str.strip().values
    n_synii_F = min(len(synii_xy_all), len(synii_ids))
    synii_xy = synii_xy_all[:n_synii_F]
    F_N = np.full((n_synii_F, N_STIM), np.nan)
    F_P = np.full((n_synii_F, N_STIM), np.nan)
    F_fail = np.full((n_synii_F, N_STIM), np.nan)
    n_matched_F, n_fit_F = 0, 0
    for i in range(n_synii_F):
        key = extract_base_name(synii_ids[i])
        if key not in tg_F:
            continue
        qc = np.array(tg_F[key])
        ok_rows = np.all(np.isfinite(qc), axis=1)
        qc = qc[ok_rows].astype(int)
        if len(qc) < MIN_TRIALS_D:
            continue
        n_matched_F += 1
        F_fail[i, :] = np.mean(qc == 0, axis=0)
        ntr = len(qc)
        if USE_BOOTSTRAP:
            rng_f = np.random.default_rng(1000 + i)
            bN = np.full((N_STIM, N_BOOT), np.nan)
            bP = np.full((N_STIM, N_BOOT), np.nan)
            for b in range(N_BOOT):
                qc_b = qc[rng_f.integers(0, ntr, ntr)]
                for k in range(N_STIM):
                    qk = qc_b[:, k].astype(int)
                    Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                    if np.isfinite(Nf) and Nf < N_MAX_D:
                        bN[k, b] = Nf
                        bP[k, b] = Pf
            for k in range(N_STIM):
                if np.isfinite(bN[k]).sum() >= N_BOOT * 0.5:
                    F_N[i, k] = np.nanmedian(bN[k])
                    F_P[i, k] = np.nanmedian(bP[k])
        else:
            for k in range(N_STIM):
                qk = qc[:, k].astype(int)
                Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    F_N[i, k] = Nf
                    F_P[i, k] = Pf
        if np.isfinite(F_N[i, 0]):
            n_fit_F += 1

wt_key_candidates = ['2.5 mM 20 Hz', 'WT 2.5 mM 20 Hz']
wt_key = next((k for k in wt_key_candidates if k in cond_results_E), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if ('2.5 mM 20 Hz' in str(k)) and ('WT' in str(k))), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if '2.5 mM 20 Hz' in str(k)), None)
if wt_key is None:
    raise RuntimeError(f"WT 2.5 mM 20 Hz key not found in cond_results_E. Available keys: {list(cond_results_E.keys())}")
WT_E_P = cond_results_E[wt_key]['E_P']
wt_color = cond_results_E[wt_key].get('color', get_wt_ca_color('2.5mM'))
x_stim = np.arange(1, N_STIM + 1)

def _format_profile_pvalues_synii(stats_result):
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
    if per_pulse_df.empty:
        return 'none'
    vals = []
    for _, row in per_pulse_df.iterrows():
        p_corr = row.get('p_value_corrected', np.nan)
        if np.isfinite(p_corr):
            vals.append(f"P{int(row['Pulse'])}={p_corr:.3g}")
    return ', '.join(vals) if vals else 'none'

stats_synii_p = summarize_profile_group_stats(
    [
        {'label': 'WT 2.5 mM 20 Hz', 'matrix': WT_E_P},
        {'label': 'SynII', 'matrix': F_P},
    ],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='WT vs SynII P trajectories',
    print_summary=False,
)

fig, ax = make_figure_grid(figsize=(6.5, 5))
plot_mean_sem_trace(ax, x_stim, F_P, color=SYNII_COLOR, label='SynII', marker='o', linestyle='-', ms=4, lw=1.4, fill_alpha=0.12)
plot_mean_sem_trace(ax, x_stim, WT_E_P, color=wt_color, label='WT 2.5 mM 20 Hz', marker='o', linestyle='-', ms=4, lw=1.2, fill_alpha=0.10)
ax.set_ylim(0, 1)
style_ax(ax, 'Stimulus', '$P_k$', 'WT vs SynII P MLE trajectory')
add_legend(ax, frameon=False, loc='best')
annotate_profile_significance(ax, stats_synii_p)
out_f = OUTPUT_DIR / 'Fig6_f_wt_vs_synii_p_mle_trajectory.pdf'
finalize_figure(fig, title='WT vs SynII P MLE trajectory', save_path=out_f)
wt_p_n = int(np.isfinite(np.asarray(WT_E_P, dtype=float)[:, 0]).sum()) if np.asarray(WT_E_P, dtype=float).ndim == 2 else 0
synii_p_n = int(np.isfinite(np.asarray(F_P, dtype=float)[:, 0]).sum()) if np.asarray(F_P, dtype=float).ndim == 2 else 0
def _nanmed_local(arr, idx):
    arr = np.asarray(arr, dtype=float)
    return float(np.nanmedian(arr[:, idx])) if arr.ndim == 2 and arr.shape[1] > idx else np.nan
print('=== WT vs SynII P MLE trajectory ===')
print(f"WT:    n={wt_p_n} | P1={_nanmed_local(WT_E_P, 0):.3f} P2={_nanmed_local(WT_E_P, 1):.3f} P10={_nanmed_local(WT_E_P, 9):.3f}")
print(f"SynII: n={synii_p_n} | P1={_nanmed_local(F_P, 0):.3f} P2={_nanmed_local(F_P, 1):.3f} P10={_nanmed_local(F_P, 9):.3f}")
print(stats_synii_p.get('summary_line', 'WT vs SynII P trajectories: no statistics available.'))
print(f"WT vs SynII P trajectories FDR-corrected p values: {_format_profile_pvalues_synii(stats_synii_p)}")
print(f'✓ Saved to {out_f}')



In [ ]:
# === Fig 6g — WT vs SynII N MLE trajectories ===
# Restore the direct WT-versus-SynII binomial N comparison in the main Figure 6 block.
if 'F_N' not in globals() or not np.isfinite(np.asarray(F_N, dtype=float)).any():
    raise RuntimeError('Run the Fig 6f cell first to compute SynII N/P MLE trajectories.')
if 'cond_results_E' not in globals():
    raise RuntimeError('Run the shared WT condition MLE cell first (cell building cond_results_E).')

wt_key_candidates = ['2.5 mM 20 Hz', 'WT 2.5 mM 20 Hz']
wt_key = next((k for k in wt_key_candidates if k in cond_results_E), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if ('2.5 mM 20 Hz' in str(k)) and ('WT' in str(k))), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if '2.5 mM 20 Hz' in str(k)), None)
if wt_key is None:
    raise RuntimeError(f"WT 2.5 mM 20 Hz key not found in cond_results_E. Available keys: {list(cond_results_E.keys())}")
WT_E_N = cond_results_E[wt_key]['E_N']
wt_color = cond_results_E[wt_key].get('color', get_wt_ca_color('2.5mM'))
x_stim = np.arange(1, N_STIM + 1)

if '_format_profile_pvalues_synii' not in globals():
    def _format_profile_pvalues_synii(stats_result):
        per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
        if per_pulse_df.empty:
            return 'none'
        vals = []
        for _, row in per_pulse_df.iterrows():
            p_corr = row.get('p_value_corrected', np.nan)
            if np.isfinite(p_corr):
                vals.append(f"P{int(row['Pulse'])}={p_corr:.3g}")
        return ', '.join(vals) if vals else 'none'

stats_synii_n = summarize_profile_group_stats(
    [
        {'label': 'WT 2.5 mM 20 Hz', 'matrix': WT_E_N},
        {'label': 'SynII', 'matrix': F_N},
    ],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='WT vs SynII N trajectories',
    print_summary=False,
)

fig, ax = make_figure_grid(figsize=(6.5, 5))
plot_mean_sem_trace(ax, x_stim, F_N, color=SYNII_COLOR, label='SynII', marker='o', linestyle='-', ms=4, lw=1.4, fill_alpha=0.12)
plot_mean_sem_trace(ax, x_stim, WT_E_N, color=wt_color, label='WT 2.5 mM 20 Hz', marker='o', linestyle='-', ms=4, lw=1.2, fill_alpha=0.10)
ax.set_ylim(bottom=0)
style_ax(ax, 'Stimulus', '$N_k$', 'WT vs SynII N MLE trajectory')
add_legend(ax, frameon=False, loc='best')
annotate_profile_significance(ax, stats_synii_n)
out_g = OUTPUT_DIR / 'Fig6_g_wt_vs_synii_n_mle_trajectory.pdf'
finalize_figure(fig, title='WT vs SynII N MLE trajectory', save_path=out_g)
wt_n_n = int(np.isfinite(np.asarray(WT_E_N, dtype=float)[:, 0]).sum()) if np.asarray(WT_E_N, dtype=float).ndim == 2 else 0
synii_n_n = int(np.isfinite(np.asarray(F_N, dtype=float)[:, 0]).sum()) if np.asarray(F_N, dtype=float).ndim == 2 else 0
def _nanmed_local_n(arr, idx):
    arr = np.asarray(arr, dtype=float)
    return float(np.nanmedian(arr[:, idx])) if arr.ndim == 2 and arr.shape[1] > idx else np.nan
print('=== WT vs SynII N MLE trajectory ===')
print(f"WT:    n={wt_n_n} | N1={_nanmed_local_n(WT_E_N, 0):.3f} N2={_nanmed_local_n(WT_E_N, 1):.3f} N10={_nanmed_local_n(WT_E_N, 9):.3f}")
print(f"SynII: n={synii_n_n} | N1={_nanmed_local_n(F_N, 0):.3f} N2={_nanmed_local_n(F_N, 1):.3f} N10={_nanmed_local_n(F_N, 9):.3f}")
print(stats_synii_n.get('summary_line', 'WT vs SynII N trajectories: no statistics available.'))
print(f"WT vs SynII N trajectories FDR-corrected p values: {_format_profile_pvalues_synii(stats_synii_n)}")
print(f'✓ Saved to {out_g}')



In [ ]:
# === Helper — Assign SynII to WT clusters ===
# Assign SynII samples to WT-derived clusters using PCA space.
def assign_clusters_robust(X_existing, labels_existing, X_new, method='centroid'):
    """Assign new samples to existing clusters using specified linkage method."""
    unique_labels = np.unique(labels_existing)
    if method == 'centroid':
        centroids = np.vstack([X_existing[labels_existing == lbl].mean(axis=0) for lbl in unique_labels])
        distances = cdist(X_new, centroids)
        return unique_labels[np.argmin(distances, axis=1)]
    if method == 'single':
        dist_matrix = np.empty((X_new.shape[0], len(unique_labels)))
        for j, lbl in enumerate(unique_labels):
            cluster_points = X_existing[labels_existing == lbl]
            dist_matrix[:, j] = np.min(cdist(X_new, cluster_points), axis=1)
        return unique_labels[np.argmin(dist_matrix, axis=1)]
    raise ValueError("method must be 'centroid' or 'single'")

synii_cluster_assignments = assign_clusters_robust(pca_coordinates, cluster_assignments, pca_data['SynII'], method='single')
print(f"✓ Assigned {len(synii_cluster_assignments)} SynII samples to WT clusters")
if '_format_cluster_count_summary' not in globals():
    def _format_cluster_count_summary(label, counts):
        items = [f"C{int(k)}={int(v)}" for k, v in pd.Series(counts).sort_index().items()]
        joined = '  '.join(items) if items else '-'
        print(f"{label}: {joined}")
_format_cluster_count_summary('SynII', pd.Series(synii_cluster_assignments).value_counts())


#### Additional SynII class-attribution analyses

The cells below keep the WT-cluster attribution view for SynII as an auxiliary follow-up. They are useful for interpretation, but they are not the main manuscript panels for Figure 6.


In [ ]:
# === Aux — SynII vs WT AMP1/PPR boxplots ===
# Boxplot comparison: WT pooled vs SynII for AMP1, PPR2/1, PPR3/1.
combined_data = pd.concat([
    PCA_Data_WT_Pooled[['AMP1', 'PPR2/1', 'PPR3/1']].assign(Genotype='WT'),
    PCA_Data_SynII[['AMP1', 'PPR2/1', 'PPR3/1']].assign(Genotype='SynII'),
], ignore_index=True)

params = ['AMP1', 'PPR2/1', 'PPR3/1']
fig, axes = make_figure_grid(1, 3, panel_kind='boxplot_compact')
results = []
for ax, param in zip(axes, params):
    wt_vals = combined_data.loc[combined_data['Genotype'] == 'WT', param].dropna()
    synii_vals = combined_data.loc[combined_data['Genotype'] == 'SynII', param].dropna()
    res = plot_boxplot_analysis(ax, [make_box_group('WT', values=wt_vals, color=WT_THEO_COLOR), make_box_group('SynII', values=synii_vals, color=SYNII_COLOR)], paired=False, show_points=True, show_stats=True, parametric_stat=False, title=None, ylabel=param)
    ax.set_title(param)
    if 'PPR' in param:
        ax.set_ylim(bottom=0)
    results.append({
        'param': param,
        'n_wt': len(wt_vals),
        'n_syn': len(synii_vals),
        'wt_median': float(np.nanmedian(wt_vals)) if len(wt_vals) else np.nan,
        'synii_median': float(np.nanmedian(synii_vals)) if len(synii_vals) else np.nan,
        'pval': float(res.get('p_value', np.nan)),
    })
plt.suptitle('WT vs SynII scalar comparisons', fontsize=8, fontweight='bold')
out_aux = OUTPUT_DIR / 'Fig6_aux_synii_vs_wt_amp1_ppr_boxplots.pdf'
plt.tight_layout()
plt.savefig(out_aux, dpi=300, bbox_inches='tight')
plt.show()
for result in results:
    pval = result['pval']
    if np.isfinite(pval):
        print(f"{result['param']}: WT n={result['n_wt']}, SynII n={result['n_syn']}, p={pval:.4g}")
    else:
        print(f"{result['param']}: WT n={result['n_wt']}, SynII n={result['n_syn']}")
for result in results:
    pval = result['pval']
    if np.isfinite(pval) and pval < 0.05:
        direction = 'reduced' if result['synii_median'] < result['wt_median'] else 'increased'
        print(f"{result['param']} was significantly {direction} in SynII relative to WT (Mann-Whitney U, p = {pval:.3f}).")
print(f'✓ Saved to {out_aux}')


## Extras — Synapsin II follow-up analyses (not in main figures)


### Fiber-level SynII controls

The per-fiber SynII projections test whether the genotype effect is homogeneous across fibers or whether it remains structured at the level of individual axons. This mirrors the same-fiber logic used for the WT diversity analysis.


In [ ]:
# === SynII — Projection by prefix ===
# Project SynII data onto WT-trained PCA space with ID-based coloring
make_figure(figsize=(8, 6))

# Plot WT pooled data with cluster colors (background)
plot_pca_background(plt.gca(), alpha=0.4,  label='WT pooled (2.5mM Ca)')

# Get SynII bouton IDs and extract first 14 characters
synii_ids = PCA_Data_SynII['ID'].values
synii_prefixes = [str(bouton_id)[:14] for bouton_id in synii_ids]

# Create unique prefix-to-color mapping with Set2 colormap
unique_prefixes = sorted(set(synii_prefixes))
set2_colormap = plt.get_cmap('Set1')
specified_colors = [set2_colormap(i) for i in range(len(unique_prefixes))]
prefix_color_map = dict(zip(unique_prefixes, specified_colors))

# Plot SynII points grouped by prefix
synii_pca_coords = pca_data['SynII']
overlay_groups = []
for prefix in unique_prefixes:
    mask = np.array([p == prefix for p in synii_prefixes])
    overlay_groups.append(make_pca_group(synii_pca_coords[mask], label=f'SynII {prefix}', color=prefix_color_map[prefix], marker='*', linewidths=0.0))
plot_pca_analysis(plt.gca(), overlays=overlay_groups, title=None, legend=False)

# Format plot
style_pca_axes(
    plt.gca(),
    title='SynII KO Projection onto WT PCA Space (colored by ID prefix)',
legend=False,
)
add_legend(plt.gca(), loc='best', fontsize=6, frameon=False)
plt.tight_layout()

# Save and display
output_file = OUTPUT_DIR / "Fig6_aux_synii_projection_by_prefix.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ SynII projection: {len(synii_pca_coords)} samples projected onto WT PCA space")
print(f"✓ Found {len(unique_prefixes)} unique ID prefixes: {unique_prefixes}")
print(f"✓ Saved to {output_file}")


### WT cluster attribution for SynII boutons

Assigning SynII boutons to WT-derived classes provides a direct way to see which WT phenotypes are depleted or enriched in the knockout. This keeps the genotype comparison grounded in the WT reference taxonomy.


In [ ]:
# === SynII — Cluster composition (header) ===
# Reuse shared helper: get_text_color
# Compare cluster distributions between WT and SynII
wt_cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
synii_cluster_counts = pd.Series(synii_cluster_assignments).value_counts()

# Ensure all clusters represented
all_clusters = sorted(wt_cluster_counts.index)
synii_complete = pd.Series([synii_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
wt_percentages = 100 * wt_cluster_counts / len(cluster_assignments)
synii_percentages = 100 * synii_complete / len(synii_cluster_assignments)

# Stacked bar plot
fig, ax = make_figure_grid(figsize=(8, 6))
bar_width = 0.6
x_positions = np.array([0, 1], dtype=float)
labels = ['WT pooled', 'SynII']

wt_vals = np.array([wt_percentages.get(c, 0.0) for c in all_clusters], dtype=float)
syn_vals = np.array([synii_percentages.get(c, 0.0) for c in all_clusters], dtype=float)

bottom = np.zeros(2, dtype=float)
legend_handles = []
legend_labels = []
for idx, cid in enumerate(all_clusters):
    heights = np.array([wt_vals[idx], syn_vals[idx]], dtype=float)
    color = get_cluster_color(cid)
    bars = ax.bar(
        x_positions,
        heights,
        width=bar_width,
        bottom=bottom,
        color=color,
        edgecolor='white',
        linewidth=0.8,
    )
    bottom += heights
    legend_handles.append(bars[0])
    legend_labels.append(f'C{cid}')

ax.set_xticks(x_positions)
ax.set_xticklabels([f'WT pooled\n(n={len(cluster_assignments)})', f'SynII\n(n={len(synii_cluster_assignments)})'])
style_ax(ax, 'Group', 'Cluster fraction (%)', 'WT versus SynII cluster composition')
ax.set_ylim(0, 100)
add_legend(ax, handles=legend_handles, labels=legend_labels, loc='center left', bbox_to_anchor=(1.01, 0.5), outside=True)
ax.grid(False)

output_file = OUTPUT_DIR / 'Fig6_aux_synii_cluster_composition_vs_wt.pdf'
fig.tight_layout()
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved cluster composition to {output_file}')

print('WT:    ' + '  '.join(f"C{int(c)}={int(wt_cluster_counts.get(c, 0))}" for c in all_clusters))
print('SynII: ' + '  '.join(f"C{int(c)}={int(synii_complete.get(c, 0))}" for c in all_clusters))

summary_df = pd.DataFrame({
    'Cluster': [f'C{int(c)}' for c in all_clusters],
    'WT_n': [int(wt_cluster_counts.get(c, 0)) for c in all_clusters],
    'WT_pct': [float(wt_percentages.get(c, 0.0)) for c in all_clusters],
    'SynII_n': [int(synii_complete.get(c, 0)) for c in all_clusters],
    'SynII_pct': [float(synii_percentages.get(c, 0.0)) for c in all_clusters],
})
print(summary_df.to_string(index=False, formatters={'WT_pct': '{:.1f}'.format, 'SynII_pct': '{:.1f}'.format}))

try:
    from scipy.stats import chi2_contingency
    contingency = np.vstack([
        [int(wt_cluster_counts.get(c, 0)) for c in all_clusters],
        [int(synii_complete.get(c, 0)) for c in all_clusters],
    ])
    chi2, p_val, dof, _ = chi2_contingency(contingency)
    print(f'Chi-square test: chi2={chi2:.3f}, dof={dof}, p={p_val:.4g}')
except Exception as exc:
    print(f'Chi-square test unavailable: {exc}')


### SynII descriptive statistics

Simple descriptive metrics are retained here because they show how the genotype reshapes amplitude and reliability even before considering the full PCA organization. They provide the scalar counterpart to the state-space comparison. 


In [ ]:
# === SynII — Descriptive statistics ===
# Calculate mean ± SD for Amp1, Amp2, and %Fail1 in SynII data
print("Mean ± SD for SynII data:")
print(f"AMP1: {PCA_Data_SynII['AMP1'].mean():.3f} ± {PCA_Data_SynII['AMP1'].std():.3f}")
print(f"AMP2: {PCA_Data_SynII['AMP2'].mean():.3f} ± {PCA_Data_SynII['AMP2'].std():.3f}")
print(f"%Fail1: {PCA_Data_SynII['%Fail1'].mean():.3f} ± {PCA_Data_SynII['%Fail1'].std():.3f}")

### SynII vs WT PPR profiles

The train-normalized SynII profiles test whether Synapsin II loss preferentially alters sustained facilitation, early paired-pulse behavior, or both. This directly supports the claim that Synapsin II contributes to specific subsets of STP phenotypes.


In [ ]:
# === SynII — Enriched cluster PPR comparison ===
# Identify and analyze SynII-enriched clusters

# PPR profiles comparison
if 'ppr_profiles_matrix' not in globals() or 'plot_profile_group_comparison' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

show_stats = True
parametric_stat = True

ppr_cols = [
    f'PPR{i}/1'
    for i in range(2, 11)
    if f'PPR{i}/1' in PCA_Data_WT_Pooled.columns and f'PPR{i}/1' in PCA_Data_SynII.columns
]

wt_profiles = ppr_profiles_matrix(PCA_Data_WT_Pooled, ppr_column_names=ppr_cols)
synii_profiles = ppr_profiles_matrix(PCA_Data_SynII, ppr_column_names=ppr_cols)
pulse_numbers = np.arange(1, wt_profiles.shape[1] + 1, dtype=float)

fig, ax = make_figure_grid(figsize=(8, 5))
stats_result = plot_profile_group_comparison(
    ax,
    [
        {
            'matrix': wt_profiles,
            'color': get_wt_ca_color('2.5mM'),
            'marker': 'o',
            'label': f'WT Pooled (n={wt_profiles.shape[0]})',
            'linewidth': 1.0,
            'sem_alpha': 0.2,
        },
        {
            'matrix': synii_profiles,
            'color': SYNII_COLOR,
            'marker': 's',
            'label': f'SynII KO (n={synii_profiles.shape[0]})',
            'linewidth': 1.0,
            'sem_alpha': 0.2,
        },
    ],
    pulse_numbers=pulse_numbers,
    stats_start_index=1,
    show_stats=show_stats,
    parametric_stat=parametric_stat,
    summary_label='WT vs SynII KO PPR profile',
    title='PPR Profiles: WT vs SynII KO',
    ylabel='Mean PPR (A_n/A_1)',
    finalize_kwargs={'legend': True, 'legend_loc': 'best', 'ylim': (0, None)},
)
plt.tight_layout()

output_file = OUTPUT_DIR / 'Fig6_aux_synii_enriched_cluster_ppr_comparison.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'✓ Saved enriched clusters comparison to {output_file}')

# Summary statistics
summary_stats = []
for group_name, group_df in [('WT', PCA_Data_WT_Pooled), ('SynII', PCA_Data_SynII)]:
    for param in ['AMP1', 'AMP2', '%Fail1'] + ppr_cols:
        if param in group_df.columns:
            vals = group_df[param].dropna()
            summary_stats.append({
                'Group': group_name,
                'Parameter': param,
                'Mean': vals.mean(),
                'SD': vals.std(),
                'SEM': vals.sem(),
                'N': len(vals),
                'Median': vals.median(),
                'Min': vals.min(),
                'Max': vals.max()
            })

summary_df = pd.DataFrame(summary_stats)
omnibus_df = stats_result.get('omnibus_df', pd.DataFrame()) if stats_result is not None else pd.DataFrame()
posthoc_df = stats_result.get('per_pulse_df', pd.DataFrame()) if stats_result is not None else pd.DataFrame()
metadata_df = pd.DataFrame([
    {
        'summary_label': 'WT vs SynII KO PPR profile',
        'show_stats': show_stats,
        'parametric_stat': parametric_stat,
        'summary_line': stats_result.get('summary_line', '') if stats_result is not None else '',
    }
])

# Save all statistics to Excel
stats_output_file = write_excel_sheets(
    OUTPUT_DIR / 'Fig6_aux_synii_enriched_cluster_ppr_statistics.xlsx',
    {
        'Mixed_ANOVA': omnibus_df,
        'PostHoc_PerPulse': posthoc_df,
        'Summary_Statistics': summary_df,
        'Analysis_Metadata': metadata_df,
    },
    engine='openpyxl',
    index=False,
)

print(f"\n✓ Saved statistics to {stats_output_file}")

# SynII summary statistics
print("\nSynII Summary Statistics:")
for param in ['AMP1', 'AMP2', '%Fail1']:
    if param in PCA_Data_SynII.columns:
        mean_val = PCA_Data_SynII[param].mean()
        std_val = PCA_Data_SynII[param].std()
        print(f"{param}: {mean_val:.3f} ± {std_val:.3f}")


### High-amplitude WT boutons and the SynII phenotype

The high-amplitude WT subset is separated here because it represents the upper end of the WT synaptic-weight distribution. Comparing this subset to SynII boutons helps determine whether the genotype specifically eliminates the largest-output states or broadly compresses the full distribution.

#### Identify high-amplitude WT boutons

This thresholding step isolates the strongest WT boutons in the original fluorescence space. It provides a concrete subset for testing whether SynII loss selectively removes high-output release states.

#### Mean trace comparison at the high-amplitude end

The mean-trace comparison asks whether the largest WT responses have a time-domain signature that resembles or diverges from the SynII phenotype. This adds temporal interpretation to the threshold-based subset analysis.


In [ ]:
# === Fig 6h — WT and SynII trace overlays with high-amplitude WT highlights ===
# Overlay all WT traces, all SynII traces, and highlight WT traces above the SynII threshold.
# Threshold sentence: the cutoff is computed as the chosen percentile of SynII single-trial peak amplitudes, using either only the A1 window or the maximum across the full train.

synII_threshold_value = 95  # percentile of SynII peaks used for the high-amplitude cutoff; e.g. 95 or 99
peak_used_for_thr = 'A1'    # 'A1' uses the first-response window; 'all train' uses the maximum across the displayed train

A1_START_LOCAL = globals().get('A1_START', 1.000)
A1_END_LOCAL = globals().get('A1_END', 1.049)

def _a1_window_amp_local(trace_time, trace_values):
    tt = np.asarray(trace_time, dtype=float)
    tv = np.asarray(trace_values, dtype=float)
    if tt.shape[0] != tv.shape[0]:
        return np.nan
    a1_mask_local = (tt >= A1_START_LOCAL) & (tt <= A1_END_LOCAL)
    w = tv[a1_mask_local]
    if w.size == 0 or np.all(~np.isfinite(w)):
        return np.nan
    return np.nanmax(np.abs(w))

def _train_peak_amp_local(trace_time, trace_values):
    tt = np.asarray(trace_time, dtype=float)
    tv = np.asarray(trace_values, dtype=float)
    if tt.shape[0] != tv.shape[0]:
        return np.nan
    train_mask = np.isfinite(tt) & np.isfinite(tv) & (tt >= A1_START_LOCAL) & (tt <= TRACE_XLIM_20HZ[1])
    w = tv[train_mask]
    if w.size == 0 or np.all(~np.isfinite(w)):
        return np.nan
    return np.nanmax(np.abs(w))

def _threshold_peak_amp_local(trace_time, trace_values):
    peak_mode = str(peak_used_for_thr).strip().lower().replace('_', ' ')
    if peak_mode in ('a1', 'first', 'first response'):
        return _a1_window_amp_local(trace_time, trace_values)
    if peak_mode in ('all train', 'all', 'train', 'max train'):
        return _train_peak_amp_local(trace_time, trace_values)
    raise ValueError("peak_used_for_thr must be 'A1' or 'all train'")

try:
    synII_threshold_percentile = float(synII_threshold_value)
except Exception as exc:
    raise ValueError('synII_threshold_value must be a numeric percentile, e.g. 95 or 99') from exc
if not (0 < synII_threshold_percentile <= 100):
    raise ValueError('synII_threshold_value must be in the interval (0, 100]')

synii_threshold_values_local = []
synii_rows_local = select_traces(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
for _, trace_row in synii_rows_local.iterrows():
    amp_peak = _threshold_peak_amp_local(_trace_row_time(trace_row, aligned=True), trace_row['Avg'])
    if np.isfinite(amp_peak):
        synii_threshold_values_local.append(amp_peak)
synii_amplitude_threshold = np.percentile(synii_threshold_values_local, synII_threshold_percentile) if len(synii_threshold_values_local) > 0 else np.nan
if not np.isfinite(synii_amplitude_threshold):
    raise ValueError('Could not compute SynII amplitude threshold for this cell')
threshold_peak_label = 'A1 window' if str(peak_used_for_thr).strip().lower().replace('_', ' ') in ('a1', 'first', 'first response') else 'full train'
wt_trace_data = []
wt_high_amplitude_traces = []
wt_regular_traces = []
wt_trace_rows = select_traces(trace_ids=PCA_Data_WT_Pooled_clustered['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
for _, trace_row in wt_trace_rows.iterrows():
    bouton_id = trace_row['ID']
    bouton_base_id = _normalize_bouton_id(bouton_id)
    trace_time = _trace_row_time(trace_row, aligned=True)
    trace_values = np.asarray(trace_row['Avg'], float)
    wt_trace_data.append((bouton_id, trace_time, trace_values))
    matching_feature = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['BaseID'] == bouton_base_id]
    if not matching_feature.empty:
        feature_row = matching_feature.iloc[0]
        canonical_id = feature_row['ID']
        max_amp = _threshold_peak_amp_local(trace_time, trace_values)
        if np.isfinite(max_amp):
            if max_amp > synii_amplitude_threshold:
                wt_high_amplitude_traces.append((canonical_id, trace_time, trace_values, max_amp))
            else:
                wt_regular_traces.append((canonical_id, trace_time, trace_values))
if 'synii_trace_data_a1' in globals() and len(synii_trace_data_a1) > 0:
    synii_trace_data_local = synii_trace_data_a1
else:
    synii_trace_data_local = []
    synii_rows_local = select_traces(condition_names=SYNAPSIN_CONDITIONS, source=TRACE_SINGLE_SOURCE)
    for _, trace_row in synii_rows_local.iterrows():
        synii_trace_data_local.append((trace_row['ID'], _trace_row_time(trace_row, aligned=True), trace_row['Avg']))

from scipy.signal import savgol_filter

fig, axes = make_figure_grid(1, 2, figsize=(11.5, 4.6), sharey=True)


def _smooth_trace_local(trace_values, window_length=11, polyorder=3):
    vals = np.asarray(trace_values, dtype=float)
    if vals.ndim != 1:
        return vals
    finite_mask = np.isfinite(vals)
    if finite_mask.sum() < 5:
        return vals
    filled = vals.copy()
    if not np.all(finite_mask):
        x = np.arange(vals.size, dtype=float)
        filled[~finite_mask] = np.interp(x[~finite_mask], x[finite_mask], vals[finite_mask])
    usable_window = min(int(window_length), int(vals.size) if int(vals.size) % 2 == 1 else int(vals.size) - 1)
    if usable_window < 5:
        return vals
    usable_poly = min(int(polyorder), usable_window - 2)
    if usable_poly < 1:
        return vals
    smoothed = savgol_filter(filled, window_length=usable_window, polyorder=usable_poly, mode='interp')
    smoothed[~finite_mask] = np.nan
    return smoothed


def _has_noisy_baseline_local(trace_time, trace_values, threshold=0.5):
    tt = np.asarray(trace_time, dtype=float)
    tv = _smooth_trace_local(trace_values)
    if tt.shape[0] != tv.shape[0]:
        return False
    baseline_mask = np.isfinite(tt) & np.isfinite(tv) & (tt < A1_START_LOCAL)
    if not np.any(baseline_mask):
        return False
    return bool(np.nanmax(np.abs(tv[baseline_mask])) > threshold)


def _valid_smoothed_traces(trace_collection):
    valid = []
    for _, trace_time, trace_values, *rest in trace_collection:
        if _has_noisy_baseline_local(trace_time, trace_values, threshold=0.5):
            continue
        tt = np.asarray(trace_time, dtype=float)
        yy = _smooth_trace_local(trace_values)
        if tt.shape[0] != yy.shape[0]:
            continue
        valid.append((tt, yy))
    return valid


def _plot_trace_collection(ax, trace_collection, color, alpha, linewidth, label, zorder):
    valid = _valid_smoothed_traces(trace_collection)
    for j, (trace_time, trace_values) in enumerate(valid):
        ax.plot(
            trace_time,
            trace_values,
            color=color,
            alpha=alpha,
            linewidth=linewidth,
            label=label if j == 0 else None,
            zorder=zorder,
        )
    return valid


def _plot_mean_trace(ax, valid_traces, label):
    if not valid_traces:
        return np.nan
    ref_time = valid_traces[0][0]
    aligned = []
    for trace_time, trace_values in valid_traces:
        if trace_time.shape == ref_time.shape and np.allclose(trace_time, ref_time, equal_nan=True):
            aligned.append(trace_values)
        else:
            ok = np.isfinite(trace_time) & np.isfinite(trace_values)
            interp = np.full(ref_time.shape, np.nan, dtype=float)
            if ok.sum() >= 2:
                interp = np.interp(ref_time, trace_time[ok], trace_values[ok], left=np.nan, right=np.nan)
            aligned.append(interp)
    mean_trace = np.nanmean(np.vstack(aligned), axis=0)
    ax.plot(ref_time, mean_trace, color='black', linewidth=1.7, alpha=0.98, label=label, zorder=8)
    return _a1_window_amp_local(ref_time, mean_trace)


ax_synii, ax_wt = axes
wt_color = get_wt_ca_color('2.5mM')

synii_valid = _plot_trace_collection(
    ax_synii,
    synii_trace_data_local,
    color=SYNII_COLOR,
    alpha=0.30,
    linewidth=0.45,
    label=f'SynII traces (n={len(synii_trace_data_local)})',
    zorder=2,
)
wt_valid = _plot_trace_collection(
    ax_wt,
    wt_trace_data,
    color='0.72',
    alpha=0.28,
    linewidth=0.40,
    label=f'WT all traces (n={len(wt_trace_data)})',
    zorder=1,
)
wt_high_valid = _plot_trace_collection(
    ax_wt,
    wt_high_amplitude_traces,
    color=wt_color,
    alpha=0.82,
    linewidth=0.65,
    label=f'WT > SynII threshold (n={len(wt_high_amplitude_traces)})',
    zorder=4,
)

synii_mean_a1 = _plot_mean_trace(ax_synii, synii_valid, label='SynII mean')
wt_mean_a1 = _plot_mean_trace(ax_wt, wt_valid, label='WT mean')

for ax in axes:
    ax.axhline(
        synii_amplitude_threshold,
        color=SYNII_COLOR,
        linestyle=':',
        linewidth=1.1,
        alpha=0.95,
        label='SynII high-amplitude threshold',
        zorder=6,
    )
    if np.isfinite(synii_mean_a1):
        ax.axhline(
            synii_mean_a1,
            color='black',
            linestyle='--',
            linewidth=1.0,
            alpha=0.75,
            label='SynII mean A1' if ax is ax_synii else None,
            zorder=5,
        )
    ax.set_xlim(*TRACE_XLIM_20HZ)
    ax.grid(False)

style_ax(ax_synii, 'Time (s)', 'ΔF/F', 'SynII traces')
style_ax(ax_wt, 'Time (s)', '', 'WT traces with high-amplitude highlights')
ax_synii.legend(frameon=False, fontsize=7, loc='upper right')
ax_wt.legend(frameon=False, fontsize=7, loc='upper right')
plt.tight_layout()
output_file = OUTPUT_DIR / 'Fig6_h_wt_synii_high_amplitude_trace_overlay.pdf'
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'✓ SynII threshold: {synII_threshold_percentile:g}th percentile of SynII {threshold_peak_label} peaks = {synii_amplitude_threshold:.4f} (n={len(synii_threshold_values_local)})')
print(f'  Threshold computation: values were extracted from SynII single-trial traces using peak_used_for_thr={peak_used_for_thr!r}; WT traces above this same cutoff are highlighted.')
print(f'✓ SynII mean A1 line: {synii_mean_a1:.4f}')
print(f'✓ WT mean A1: {wt_mean_a1:.4f}')
print(f'✓ WT above-threshold traces: {len(wt_high_amplitude_traces)} / {len(wt_trace_data)}')
print(f'✓ Saved to {output_file}')


#### Map high-amplitude WT boutons in PCA space

Mapping the high-amplitude WT boutons back onto the PCA shows whether they occupy a distinct sector of the WT state space. This is useful when interpreting which portions of that space are absent or compressed in SynII. 


In [ ]:
# === Fig 6i — High-amp WT boutons in PCA ===
# Show PCA locations of high-amplitude WT traces identified in the previous cell (ignoring target type)
if wt_high_amplitude_traces:
    high_amp_base_ids = [_normalize_bouton_id(item[0]) for item in wt_high_amplitude_traces]
    high_amp_mask = PCA_Data_WT_Pooled_clustered['BaseID'].isin(high_amp_base_ids)
    high_amp_coordinates = pca_coordinates[high_amp_mask]   

    make_figure(figsize=(8, 6)) 
    plot_pca_analysis(plt.gca(), background={'alpha': 0.4, 'label': 'WT pooled (2.5mM Ca)'}, overlays=[make_pca_group(high_amp_coordinates, label=f'High-Amplitude WT (n={len(high_amp_coordinates)})', color='red', marker='p', edgecolors='black', linewidths=0.0)], title='PCA Locations of High-Amplitude WT Traces', legend=False)
    add_legend(plt.gca(), frameon=False)
    plt.tight_layout()
    output_file = OUTPUT_DIR / "Fig6_i_wt_high_amplitude_pca_traces.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✓ High-amplitude WT PCA subset: n={len(high_amp_coordinates)}')
    print(f'✓ Saved to {output_file}')


### SynII N/P and target-identity follow-up

Target identity is revisited under SynII loss to test whether the genotype effect interacts with postsynaptic class. The logic parallels the WT target analysis while asking a more specific molecular question.

The remaining SynII analyses focus on direct comparison to WT and on WT-reference-space interpretation of SynII boutons, without a separate SynII-only HCPC layer.


In [ ]:
# === SynII — NP/refilling analyses ===
# Compute auxiliary SynII N/P, failure, and refilling analyses relative to the WT 2.5 mM reference.

from scipy.stats import linregress

# == Reload SynII trials from raw Excel (not in trials_all) ======
_synii_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
_synii_trials = _synii_trials[_synii_trials['condition'] == 'SynII']
print(f"SynII trials loaded from Excel: {len(_synii_trials)} rows")

# == Build count dict for SynII ===================================
tg_F = defaultdict(list)
for _, r in _synii_trials.iterrows():
    key = extract_base_name(str(r['file']).strip())
    tg_F[key].append(get_count_row_with_failures(r, Q_D))

# == Match SynII summary DF rows -> counts, run MLE ===============
synii_xy_all = np.array(pca_data['SynII'])
synii_ids    = PCA_Data_SynII['ID'].astype(str).str.strip().values
n_synii_F    = min(len(synii_xy_all), len(synii_ids))
synii_xy     = synii_xy_all[:n_synii_F]

F_N = np.full((n_synii_F, N_STIM), np.nan)
F_P = np.full((n_synii_F, N_STIM), np.nan)
F_fail = np.full((n_synii_F, N_STIM), np.nan)
n_matched_F, n_fit_F = 0, 0

for i in range(n_synii_F):
    key = extract_base_name(synii_ids[i])
    if key not in tg_F:
        continue
    qc = np.array(tg_F[key])
    ok_rows = np.all(np.isfinite(qc), axis=1)
    qc = qc[ok_rows].astype(int)
    if len(qc) < MIN_TRIALS_D:
        continue
    n_matched_F += 1
    F_fail[i, :] = np.mean(qc == 0, axis=0)
    ntr = len(qc)

    if USE_BOOTSTRAP:
        rng_f = np.random.default_rng(1000 + i)
        bN = np.full((N_STIM, N_BOOT), np.nan)
        bP = np.full((N_STIM, N_BOOT), np.nan)
        for b in range(N_BOOT):
            qc_b = qc[rng_f.integers(0, ntr, ntr)]
            for k in range(N_STIM):
                qk = qc_b[:, k].astype(int)
                Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
                if np.isfinite(Nf) and Nf < N_MAX_D:
                    bN[k, b] = Nf
                    bP[k, b] = Pf
        for k in range(N_STIM):
            if np.isfinite(bN[k]).sum() >= N_BOOT * 0.5:
                F_N[i, k] = np.nanmedian(bN[k])
                F_P[i, k] = np.nanmedian(bP[k])
    else:
        for k in range(N_STIM):
            qk = qc[:, k].astype(int)
            Nf, Pf, _ = fit_binom_mle(qk, N_MAX_D)
            if np.isfinite(Nf) and Nf < N_MAX_D:
                F_N[i, k] = Nf
                F_P[i, k] = Pf

    if np.isfinite(F_N[i, 0]):
        n_fit_F += 1

print(f"SynII: {n_synii_F} boutons, matched={n_matched_F}, fit={n_fit_F}")
if n_matched_F == 0 and len(tg_F) > 0:
    print(f"  trial keys sample : {sorted(tg_F.keys())[:3]}")
    print(f"  sdf keys sample   : {[extract_base_name(s) for s in synii_ids[:3]]}")

# == Derived: NP + refilling slope ================================
F_NP   = F_N * F_P
F_NP_1 = F_NP[:, 0]

F_NP_norm = np.where(
    np.isfinite(F_NP_1[:, None]) & (F_NP_1[:, None] > 0),
    F_NP / F_NP_1[:, None],
    np.nan
)

F_NFIT = int(min(globals().get('N_FIT_LAST', 4), N_STIM))
x_tail = np.arange(N_STIM - F_NFIT + 1, N_STIM + 1)

F_refill_slope = np.full(n_synii_F, np.nan)
for i in range(n_synii_F):
    tail = F_NP[i, -F_NFIT:]
    if not np.all(np.isfinite(tail)):
        continue
    cum_tail = np.cumsum(tail)
    sl, ic, r, p, se = linregress(x_tail, cum_tail)
    if np.isfinite(sl):
        F_refill_slope[i] = sl

F_refill_norm = np.where(
    np.isfinite(F_refill_slope) & np.isfinite(F_NP_1) & (F_NP_1 > 0),
    F_refill_slope / F_NP_1,
    np.nan
)

# == Reference WT 2.5 mM 20 Hz results ============================
wt_key_candidates = ['2.5 mM 20 Hz', '2.5 mM 20 Hz (WT)']
wt_key = next((k for k in wt_key_candidates if k in cond_results_E), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if ('2.5 mM 20 Hz' in str(k)) and ('WT' in str(k))), None)
if wt_key is None:
    raise RuntimeError(f"WT 2.5 mM 20 Hz key not found in cond_results_E. Available keys: {list(cond_results_E.keys())}")

WT_E_N = cond_results_E[wt_key]['E_N']
WT_E_P = cond_results_E[wt_key]['E_P']
WT_NP = WT_E_N * WT_E_P
WT_NP1 = WT_NP[:, 0]

WT_refill_slope = np.full(WT_NP.shape[0], np.nan)
for i in range(WT_NP.shape[0]):
    tail = WT_NP[i, -F_NFIT:]
    if not np.all(np.isfinite(tail)):
        continue
    cum_tail = np.cumsum(tail)
    sl, ic, r, p, se = linregress(x_tail, cum_tail)
    if np.isfinite(sl):
        WT_refill_slope[i] = sl

WT_refill_norm = np.where(
    np.isfinite(WT_refill_slope) & np.isfinite(WT_NP1) & (WT_NP1 > 0),
    WT_refill_slope / WT_NP1,
    np.nan
)

# Aux SynII N and P maps at stim 1, 2, 5, 10
F_STIM = STIM_SHOW
n_show_F = len(F_STIM)
nv_f, nxv_f = N_VLIM
pv_f, pxv_f = 0.0, 1.0

gx_f, gy_f, ext_f = setup_pca_grid(synii_xy, pad=0.5)

fig_f1, axes_f1 = make_figure_grid(2, n_show_F, panel_kind='pca')
for col_f, sk in enumerate(F_STIM):
    kidx = sk - 1
    for row_f, (vals, cmap, vlim, vname) in enumerate([
            (F_N, 'Spectral_r', (nv_f, nxv_f), 'N'),
            (F_P, 'coolwarm',   (pv_f, pxv_f), 'P')]):
        ok = np.isfinite(vals[:, kidx])
        ax = axes_f1[row_f, col_f]
        vlo, vhi = vlim
        render_pca_scalar_panel(
            ax,
            synii_xy,
            vals[:, kidx],
            cmap=cmap,
            vmin=vlo,
            vmax=vhi,
            title=f'SynII {vname}: stim {sk} (n={ok.sum()})',
            point_size=18,
            min_points=3,
            empty_label=f'n={ok.sum()} (too few)',
        )
        if col_f == n_show_F - 1:
            add_scalar_colorbar(fig_f1, axes_f1[row_f, :], cmap=cmap, vmin=vlo, vmax=vhi, label=vname, shrink=0.7, ticks=np.linspace(0.0, 1.0, 6) if vname == 'P' else None)

finalize_figure(
    fig_f1,
    title='Aux SynII binomial MLE N and P maps',
    rect=[0, 0, 0.94, 0.93],
    tight_layout_kwargs={'pad': 0.12, 'w_pad': 0.08, 'h_pad': 0.08},
    save_path=OUTPUT_DIR / 'Fig6_aux_synii_np_maps_sharedthr.pdf',
)

# Aux SynII NP map (mean quantal content) at stim 1, 2, 5, 10
_np_ok = np.isfinite(F_NP)
_np_vmin = np.nanpercentile(F_NP[_np_ok], 2) if _np_ok.any() else 0
_np_vmax = np.nanpercentile(F_NP[_np_ok], 98) if _np_ok.any() else 5

fig_f2, axes_f2 = make_figure_grid(1, n_show_F, panel_kind='pca', squeeze=False)
for col_f, sk in enumerate(F_STIM):
    kidx = sk - 1
    ok = np.isfinite(F_NP[:, kidx])
    ax = axes_f2[0, col_f]
    render_pca_scalar_panel(
        ax,
        synii_xy,
        F_NP[:, kidx],
        cmap='viridis',
        vmin=_np_vmin,
        vmax=_np_vmax,
        title=f'SynII NP: stim {sk} (n={ok.sum()})',
        point_size=18,
        min_points=3,
        empty_label=f'n={ok.sum()} (too few)',
    )
    if col_f == n_show_F - 1:
        add_scalar_colorbar(fig_f2, axes_f2[0, :], cmap='viridis', vmin=_np_vmin, vmax=_np_vmax, label='NP (quanta)', shrink=0.7)

finalize_figure(
    fig_f2,
    title='Aux SynII NP map',
    rect=[0, 0, 0.92, 0.91],
    tight_layout_kwargs={'pad': 0.12, 'w_pad': 0.08, 'h_pad': 0.08},
    save_path=OUTPUT_DIR / 'Fig6_aux_synii_np_refilling_sharedthr.pdf',
)

# Aux SynII refilling slope/NP1 map + isoline at 1.0
ok_rf = np.isfinite(F_refill_norm)
if ok_rf.sum() == 0:
    print("No finite SynII refilling slope/NP1 values to plot.")
else:
    v3, v4 = np.nanpercentile(F_refill_norm[ok_rf], [2, 98])
    v3 = 0.55
    v4 = 2.1

    fig_f3, ax = make_figure_grid(figsize=grid_size(1, 1, 'pca'))
    sg2, sig2 = smooth_field(synii_xy, F_refill_norm, ok_rf, gx_f, gy_f)
    Xg, Yg = np.meshgrid(gx_f, gy_f)

    im = ax.imshow(
        sg2,
        extent=ext_f,
        origin='lower',
        aspect='auto',
        cmap=_display_cmap('RdBu_r'),
        vmin=v3,
        vmax=v4,
        interpolation='bilinear',
        zorder=1,
        alpha=0.8,
    )

    if np.any(np.isfinite(sg2)):
        ax.contour(Xg, Yg, sg2, levels=[1.0], colors='black', linewidths=0.0, zorder=2)

    plot_pca_analysis(ax, overlays=[make_pca_group(synii_xy[ok_rf], label='SynII', color='gray', s=18, edgecolors='none', zorder=3)], title=None, legend=False)

    plt.colorbar(im, ax=ax, shrink=0.7, label='refilling slope / NP$_1$')
    style_pca_axes(ax, title=f'SynII slope/NP$_1$ (σ={sig2:.2f}, n={ok_rf.sum()})', legend=False)

    fig_f3.tight_layout()
    fig_f3.savefig(
        OUTPUT_DIR / 'Fig6_aux_synii_refilling_slope_norm_sharedthr.pdf',
        dpi=300,
        bbox_inches='tight'
    )
    plt.show()

# Aux SynII vs WT pooled N and P trajectories
fig_f4, axes_f4 = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
wt_color = cond_results_E[wt_key].get('color', 'tab:orange')

plot_mean_sem_trace(
    axes_f4[0, 0], x_stim, F_P,
    color=SYNII_COLOR, label='SynII ({n})',
    marker='o', linestyle='-', ms=5, lw=1.8, fill_alpha=0.15
)
plot_mean_sem_trace(
    axes_f4[0, 0], x_stim, WT_E_P,
    color=wt_color, label='WT 2.5 mM 20 Hz ({n})',
    marker='o', linestyle='-', ms=5, lw=1.6, fill_alpha=0.12
)
style_ax(axes_f4[0, 0], 'Stimulus', '$P_k$', 'P trajectory')
axes_f4[0, 0].set_ylim(0, 1)
add_legend(axes_f4[0, 0], loc='best')

plot_mean_sem_trace(
    axes_f4[0, 1], x_stim, F_N,
    color=SYNII_COLOR, label='SynII ({n})',
    marker='o', linestyle='-', ms=5, lw=1.8, fill_alpha=0.15
)
plot_mean_sem_trace(
    axes_f4[0, 1], x_stim, WT_E_N,
    color=wt_color, label='WT 2.5 mM 20 Hz ({n})',
    marker='o', linestyle='-', ms=5, lw=1.6, fill_alpha=0.12
)
style_ax(axes_f4[0, 1], 'Stimulus', '$N_k$', 'N trajectory')
axes_f4[0, 1].set_ylim(bottom=0)
add_legend(axes_f4[0, 1], loc='best')

finalize_figure(
    fig_f4,
    title='Aux SynII vs WT 2.5 mM 20 Hz binomial trajectories',
    save_path=OUTPUT_DIR / 'Fig6_aux_synii_vs_wt20_np_trajectories_sharedthr.pdf',
)

# Aux pooled per-event failure rate by condition
fig_fail, axes_fail = make_figure_grid(1, 1, panel_kind='simple', squeeze=False)
ax_fail = axes_fail[0, 0]
failure_plot_defs = [
    (cond_label, cond_results_E[cond_label]['E_fail'], cond_results_E[cond_label]['color'])
    for (cond_label, _, _, _, _) in COND_DEFS_E
    if cond_label in cond_results_E
]
failure_plot_defs.append(('SynII', F_fail, SYNII_COLOR))

for cond_label, fail_mat, color in failure_plot_defs:
    ls = '--' if '50 Hz' in cond_label else '-'
    lw = 2.0 if cond_label == 'SynII' else 1.4
    mk = 's' if cond_label == 'SynII' else 'o'
    plot_mean_sem_trace(
        ax_fail, x_stim, fail_mat, color=color,
        label=f'{cond_label} ' + '({n})',
        marker=mk, linestyle=ls, ms=4, lw=lw, fill_alpha=0.10
    )

ax_fail.set_ylim(-0.02, 1.02)
style_ax(ax_fail, 'Stimulus', 'Failure rate', 'Per-event failure rate by pooled condition')
add_legend(ax_fail, loc='best')
finalize_figure(
    fig_fail,
    save_path=OUTPUT_DIR / 'Fig6_aux_synii_failure_rate_per_event_sharedthr.pdf',
)

# Aux boxplot of normalized refilling slope
#          SynII vs WT 2.5 mM 20 Hz + statistics
from scipy.stats import mannwhitneyu

synii_refill_vals = F_refill_norm[np.isfinite(F_refill_norm)]
wt_refill_vals = WT_refill_norm[np.isfinite(WT_refill_norm)]

df_refill_box = pd.DataFrame({
    'Group': (['SynII'] * len(synii_refill_vals)) +
             (['WT 2.5mM 20Hz'] * len(wt_refill_vals)),
    'Refill_norm': np.concatenate([synii_refill_vals, wt_refill_vals])
})

if df_refill_box.empty:
    raise RuntimeError("No finite slope/NP1 values to plot.")

u_stat, p_val = mannwhitneyu(synii_refill_vals, wt_refill_vals, alternative='two-sided')

fig_f5, ax = make_figure_grid(panel_kind='boxplot_compact')
plot_boxplot_analysis(ax, [make_box_group('WT', values=wt_refill_vals, color='lightgray'), make_box_group('SynII', values=synii_refill_vals, color='gray')], paired=False, show_points=True, show_stats=True, parametric_stat=False, title=None, ylabel='Refill_norm', point_alpha=0.3, point_size=18)

ax.axhline(1.0, ls='--', color='gray', lw=1, alpha=0.7)

ax.set_ylabel('Normalised refilling slope (slope / NP$_1$)')
ax.set_xlabel('')
ax.set_title('SynII vs WT 2.5 mM 20 Hz: refilling slope')
ax.grid(False)

fig_f5.tight_layout()
out_f5 = OUTPUT_DIR / 'Fig6_aux_synii_refilling_slope_boxplot_sharedthr.pdf'
fig_f5.savefig(out_f5, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved to {out_f5}")
print(f"SynII n={len(synii_refill_vals)}, WT n={len(wt_refill_vals)}")
print(f"Mann-Whitney U test: U={u_stat:.3f}, p={p_val:.3e}")
print(f"Median slope/NP1: SynII={np.nanmedian(synii_refill_vals):.3f}, WT={np.nanmedian(wt_refill_vals):.3f}")


In [ ]:
# === SynII — vs WT refilling stats ===
# Report statistical comparisons for the auxiliary SynII-versus-WT refilling-slope analysis.
from scipy.stats import ttest_ind
# Reuse boxplot dataframe if present, else rebuild from arrays
if 'df_refill_box' in globals():
    syn = df_refill_box.loc[df_refill_box['Group'] == 'SynII', 'Refill_norm'].dropna().values
    wt  = df_refill_box.loc[df_refill_box['Group'] == 'WT 2.5mM 20Hz', 'Refill_norm'].dropna().values
else:
    syn = F_refill_norm[np.isfinite(F_refill_norm)]
    wt  = WT_refill_norm[np.isfinite(WT_refill_norm)]

if len(syn) < 3 or len(wt) < 3:
    raise RuntimeError(f"Not enough values for stats (SynII n={len(syn)}, WT n={len(wt)}).")

# Non-parametric (recommended)
u_stat, p_mwu = mannwhitneyu(syn, wt, alternative='two-sided')
rbc = (2 * u_stat) / (len(syn) * len(wt)) - 1  # rank-biserial correlation

# Parametric (optional)
t_stat, p_t = ttest_ind(syn, wt, equal_var=False, nan_policy='omit')

print("=== Refilling slope/NP1: SynII vs WT 2.5mM 20Hz ===")
print(f"n SynII={len(syn)}, n WT={len(wt)}")
print(f"median SynII={np.median(syn):.4f}, median WT={np.median(wt):.4f}")
print(f"Mann-Whitney U={u_stat:.2f}, p={p_mwu:.4g}, rank-biserial={rbc:.3f}")
print(f"Welch t-test: t={t_stat:.3f}, p={p_t:.4g}")


## Section 7 — Figure 7: Diversity along single PFs


### Diversity along individual parallel fibers

These analyses test whether heterogeneous bouton phenotypes coexist along single WT fibers rather than only across unrelated axons. The section combines one representative fiber example with population-level summaries of PCA coverage and class diversity.


#### Representative fiber example and overlay summaries

The example-fiber panels translate the population result back into the measured traces and bouton-wise PPR profiles. They show directly that neighboring boutons on one axon can express distinct short-term plasticity signatures.


In [ ]:
# === Helper — Example fiber trace and PPR helpers ===
# Build trace lookup from normalized traces
if 'build_trace_lookup' in globals():
    resampled_trace_lookup = build_trace_lookup_from_source(TRACE_SINGLE_SOURCE)
else:
    resampled_trace_lookup = {}
    for _, trace_row in NORM_TRACES_DATAFRAME.iterrows():
        resampled_trace_lookup[trace_row['ID']] = {'Time': _trace_row_time(trace_row, aligned=True), 'Avg': trace_row['Avg']}

from scipy.signal import savgol_filter


def analyze_single_fiber(fiber_prefix, window_length=9, poly_order=2):
    """Analyze all boutons from a specific fiber and return native raw/smoothed traces."""
    fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(fiber_prefix)]

    if len(fiber_boutons) == 0:
        print(f"No boutons found with prefix '{fiber_prefix}'")
        return None

    print(f"Found {len(fiber_boutons)} boutons from fiber '{fiber_prefix}'")

    fiber_trace_rows = select_traces(trace_ids=fiber_boutons['ID'], condition_names=WT_2_5_20HZ_CONDITIONS, source=TRACE_SINGLE_SOURCE)
    fiber_trace_by_base = {
        str(trace_row['BaseID']): trace_row
        for _, trace_row in fiber_trace_rows.iterrows()
    }

    fiber_traces = {}
    for _, bouton in fiber_boutons.iterrows():
        bouton_id = str(bouton['ID'])
        bouton_base = _normalize_bouton_id(bouton_id)
        trace_row = fiber_trace_by_base.get(str(bouton_base))
        if trace_row is None:
            continue

        trace_time = _trace_row_time(trace_row, aligned=True)

        fiber_traces[bouton_id] = {
            'time': trace_time,
            'raw': np.asarray(trace_row['Avg'], dtype=float),
            'cluster': bouton['HC_Cluster'],
            'condition': trace_row['Condition'],
        }

    if not fiber_traces:
        print(f"No trace data found for fiber '{fiber_prefix}'")
        return None

    for bouton_id in fiber_traces:
        raw_trace = np.asarray(fiber_traces[bouton_id]['raw'], dtype=float)

        if np.all(np.isnan(raw_trace)):
            fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
            continue

        trace_for_smoothing = raw_trace.copy()
        nan_mask = np.isnan(trace_for_smoothing)

        if np.any(nan_mask):
            valid_indices = np.where(~nan_mask)[0]
            valid_values = trace_for_smoothing[~nan_mask]
            if len(valid_indices) < 3:
                fiber_traces[bouton_id]['smoothed'] = raw_trace.copy()
                continue
            nan_indices = np.where(nan_mask)[0]
            trace_for_smoothing[nan_mask] = np.interp(nan_indices, valid_indices, valid_values)

        win_len = min(window_length, len(trace_for_smoothing))
        if win_len % 2 == 0:
            win_len -= 1
        if win_len < 3:
            smoothed_trace = trace_for_smoothing.copy()
        else:
            smoothed_trace = savgol_filter(
                trace_for_smoothing,
                window_length=win_len,
                polyorder=min(poly_order, win_len - 1),
                mode='interp',
            )

        if np.any(nan_mask):
            smoothed_trace[nan_mask] = np.nan

        fiber_traces[bouton_id]['smoothed'] = smoothed_trace

    return fiber_traces


def _fiber_trace_display_params(fiber_traces, n_pulses=10, train_start_s=None, trace_xlim=None):
    """Infer or override native train timing for one fiber instead of forcing the common 1.0 s window."""
    conditions = [str(data.get('condition', '')) for data in fiber_traces.values()]
    if train_start_s is not None:
        train_start = float(train_start_s)
    elif TRACE_SINGLE_SOURCE != 'normalized' and conditions and all(cond in EXCEPTIONAL_CONDITIONS for cond in conditions):
        train_start = float(EXCEPTIONAL_BASELINE_OFFSET)
    else:
        train_start = float(DEFAULT_TRAIN_START)
    stim_times = [train_start + i * DEFAULT_ISI for i in range(int(n_pulses))]
    all_times = []
    for data in fiber_traces.values():
        t = np.asarray(data['time'], dtype=float)
        all_times.extend(t[np.isfinite(t)])
    if not all_times:
        raise ValueError('No valid time values found for fiber trace display.')
    time_min = float(np.nanmin(all_times))
    time_max = float(np.nanmax(all_times))
    if trace_xlim is not None:
        x_min, x_max = map(float, trace_xlim)
    else:
        x_min = max(time_min, train_start - 0.05)
        x_max = min(time_max, stim_times[-1] + 0.10)
        if x_max <= x_min:
            x_min, x_max = time_min, time_max
    return train_start, stim_times, (x_min, x_max)


def plot_fiber_smoothed_traces_and_ppr(fiber_boutons, fiber_traces, fiber_prefix, figure_tag='05_06', train_start_s=None, trace_xlim=None):
    """Plot smoothed traces and bouton-wise colored PPR profiles side by side."""
    if 'plot_profile_analysis' not in globals():
        raise RuntimeError('Run the shared profile API helper cell first.')
    if fiber_traces is None or len(fiber_boutons) == 0:
        raise ValueError(f"No fiber data available for '{fiber_prefix}'")

    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in fiber_boutons.columns]
    if not ppr_cols:
        raise ValueError('No PPR columns found for this fiber selection')

    ordered_boutons = fiber_boutons.copy()
    n_boutons = len(ordered_boutons)
    fig, axes = make_figure_grid(n_boutons, 2, figsize=(12, max(4.0, 2.6 * n_boutons)), sharex='col')
    axes = np.asarray(axes)
    if n_boutons == 1:
        axes = axes.reshape(1, 2)

    all_trace_vals = []
    for bouton_id in ordered_boutons['ID']:
        if bouton_id not in fiber_traces:
            raise ValueError(f"Missing trace data for bouton '{bouton_id}'")
        trace_vals = np.asarray(fiber_traces[bouton_id]['smoothed'], dtype=float)
        all_trace_vals.extend(trace_vals[np.isfinite(trace_vals)])

    if not all_trace_vals:
        raise ValueError(f"No valid smoothed trace values found for '{fiber_prefix}'")

    y_min, y_max = np.min(all_trace_vals), np.max(all_trace_vals)
    y_padding = max((y_max - y_min) * 0.08, 0.05)
    y_lims = (y_min - y_padding, y_max + y_padding)
    y_tick = y_lims[1] - 0.06 * (y_lims[1] - y_lims[0])
    train_start, stim_times, trace_xlim = _fiber_trace_display_params(fiber_traces, n_pulses=len(ppr_cols) + 1, train_start_s=train_start_s, trace_xlim=trace_xlim)

    all_ppr_vals = [1.0]
    for _, bouton in ordered_boutons.iterrows():
        for col in ppr_cols:
            if pd.notna(bouton[col]):
                all_ppr_vals.append(float(bouton[col]))
    ppr_y_max = max(1.2, np.nanmax(all_ppr_vals) * 1.1)

    for idx, (_, bouton) in enumerate(ordered_boutons.iterrows()):
        bouton_id = str(bouton['ID'])
        data = fiber_traces[bouton_id]
        cluster_id = data['cluster']
        color = get_cluster_color(cluster_id)

        ax_trace = axes[idx, 0]
        ax_trace.plot(data['time'], data['smoothed'], color=color, linewidth=1.5)
        ax_trace.axhline(0, color='gray', linestyle='dotted', linewidth=0.7)
        ax_trace.axvline(train_start, color='red', linestyle='--', alpha=0.5)
        for stim_t in stim_times:
            ax_trace.plot(stim_t, y_tick, marker='|', color='black', markersize=7, markeredgewidth=0.0)
        ax_trace.set_xlim(*trace_xlim)
        ax_trace.set_ylim(y_lims)
        ax_trace.set_ylabel('ΔF/F')
        ax_trace.set_title(f'{bouton_id} (Cluster {cluster_id})', fontsize=8, loc='left')
        if 'style_trace_axis' in globals():
            style_trace_axis(ax_trace)

        ax_ppr = axes[idx, 1]
        plot_profile_analysis(
            ax_ppr,
            [make_profile_group(f'Cluster {cluster_id}', dataframe=bouton.to_frame().T, profile_kind='ppr', ppr_column_names=ppr_cols, color=color, marker='o', linewidth=2, markersize=5, sem_alpha=0.2)],
            pulse_numbers=np.arange(1, len(ppr_cols) + 2, dtype=float),
            show_stats=False,
            title=None,
            ylabel='PPR (A_n/A_1)',
            finalize_kwargs={'ylim': (0, ppr_y_max), 'unity_kwargs': {'color': 'gray', 'linestyle': '--', 'linewidth': 1}, 'legend': False},
        )
        ax_ppr.set_title('PPR profile', fontsize=8, loc='left')
        if 'style_trace_axis' in globals():
            style_trace_axis(ax_ppr)
        if ax_ppr.get_legend() is not None:
            ax_ppr.get_legend().remove()

    axes[-1, 0].set_xlabel('Time (s)')
    finalize_ppr_axis(axes[-1, 1], None, xlabel='Stimulus', ylabel=None, unity_line=False, legend=False)

    fig.subplots_adjust(top=0.97, hspace=0.45, wspace=0.28)
    plt.suptitle(f'Fiber {fiber_prefix} (n={n_boutons} boutons)', fontsize=8, fontweight='bold', y=0.995)

    safe_prefix = fiber_prefix.replace(':', '_').replace('/', '_')
    output_file = OUTPUT_DIR / f"{figure_tag}_fiber_{safe_prefix}_smoothed_traces_ppr.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()
    return output_file


In [ ]:
# === Fig 7a / 7b / 7c — Representative fiber example and overlays ===
# Use one representative WT fiber to generate the example panel and its corresponding trace/PPR overlays.
if 'analyze_single_fiber' not in globals() or 'plot_fiber_smoothed_traces_and_ppr' not in globals():
    raise RuntimeError('Run the shared fiber helper cell first (the cell defining analyze_single_fiber).')

EXAMPLE_FIBER_PREFIX = '20210304_linescan1'
EXAMPLE_FIBER_TRAIN_START_S = 1.0
EXAMPLE_FIBER_TRACE_XLIM = (0.9, 1.58)

fiber_boutons = PCA_Data_WT_Pooled_clustered[PCA_Data_WT_Pooled_clustered['ID'].str.startswith(EXAMPLE_FIBER_PREFIX)].copy()
fiber_traces = analyze_single_fiber(EXAMPLE_FIBER_PREFIX)

if fiber_traces is None:
    available_prefixes = PCA_Data_WT_Pooled_clustered['ID'].str[:25].value_counts()
    print(f"No analysis possible for fiber '{EXAMPLE_FIBER_PREFIX}'")
    print('\nAvailable fiber prefixes (showing top 10):')
    for prefix, count in available_prefixes.head(10).items():
        if count >= 3:
            print(f"  '{prefix}': {count} boutons")
else:
    # Panel a: keep the detailed worked example.
    out_a = plot_fiber_smoothed_traces_and_ppr(
        fiber_boutons=fiber_boutons,
        fiber_traces=fiber_traces,
        fiber_prefix=EXAMPLE_FIBER_PREFIX,
        figure_tag='Fig7_a',
        train_start_s=EXAMPLE_FIBER_TRAIN_START_S,
        trace_xlim=EXAMPLE_FIBER_TRACE_XLIM,
    )

    # Shared ordering and ranges.
    ordered_boutons = fiber_boutons.copy()
    ordered_ids = ordered_boutons['ID'].astype(str).tolist()
    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in ordered_boutons.columns]
    train_start, stim_times, trace_xlim = _fiber_trace_display_params(
        fiber_traces,
        n_pulses=len(ppr_cols) + 1,
        train_start_s=EXAMPLE_FIBER_TRAIN_START_S,
        trace_xlim=EXAMPLE_FIBER_TRACE_XLIM,
    )

    all_trace_vals = []
    for bouton_id in ordered_ids:
        smoothed = np.asarray(fiber_traces[bouton_id]['smoothed'], dtype=float)
        all_trace_vals.extend(smoothed[np.isfinite(smoothed)])
    y_min, y_max = np.min(all_trace_vals), np.max(all_trace_vals)
    y_padding = max((y_max - y_min) * 0.08, 0.05)
    y_lims = (y_min - y_padding, y_max + y_padding)
    stim_tick_low = y_lims[1] - 0.06 * (y_lims[1] - y_lims[0])
    stim_tick_high = y_lims[1] - 0.02 * (y_lims[1] - y_lims[0])

    # Panel b: overlay all bouton traces from the same fiber.
    fig_b, ax_b = make_figure_grid(figsize=(8, 5))
    for bouton_id in ordered_ids:
        data = fiber_traces[bouton_id]
        cluster_id = data['cluster']
        color = get_cluster_color(cluster_id)
        ax_b.plot(np.asarray(data['time'], dtype=float), np.asarray(data['smoothed'], dtype=float), color=color, linewidth=1.3, alpha=0.95, label=f'C{cluster_id}' if f'C{cluster_id}' not in ax_b.get_legend_handles_labels()[1] else '_nolegend_')
    ax_b.axhline(0, color='gray', linestyle='dotted', linewidth=0.8)
    ax_b.axvline(train_start, color='red', linestyle='--', alpha=0.5, linewidth=1)
    for stim_t in stim_times:
        ax_b.plot([stim_t, stim_t], [stim_tick_low, stim_tick_high], color='black', lw=1.0, solid_capstyle='butt')
    ax_b.set_xlim(*trace_xlim)
    ax_b.set_ylim(*y_lims)
    style_trace_axis(ax_b)
    ax_b.set_xlabel('Time (s)')
    ax_b.set_ylabel('ΔF/F')
    ax_b.set_title(f'Fiber {EXAMPLE_FIBER_PREFIX}: bouton trace overlays')
    add_legend(ax_b, frameon=False, loc='upper right', fontsize=6)
    out_b = OUTPUT_DIR / 'Fig7_b_example_fiber_trace_overlays.pdf'
    finalize_figure(fig_b, title='Example fiber trace overlays', save_path=out_b)

    # Panel c: corresponding bouton-wise PPR profiles.
    fig_c, ax_c = make_figure_grid(figsize=(7, 5))
    for _, bouton in ordered_boutons.iterrows():
        cluster_id = int(bouton['HC_Cluster'])
        color = get_cluster_color(cluster_id)
        plot_profile_analysis(
            ax_c,
            [make_profile_group(f'C{cluster_id}', dataframe=bouton.to_frame().T, profile_kind='ppr', ppr_column_names=ppr_cols, color=color, marker='o', linewidth=1.5, markersize=4, sem_alpha=0.0)],
            pulse_numbers=np.arange(1, len(ppr_cols) + 2, dtype=float),
            show_stats=False,
            title='Example fiber bouton-wise PPR profiles',
            ylabel='PPR (A$_n$/A$_1$)',
            finalize_kwargs={'ylim': (0, None), 'unity_kwargs': {'color': 'gray', 'linestyle': '--', 'linewidth': 1}, 'legend': False},
        )
    if ax_c.get_legend() is not None:
        ax_c.get_legend().remove()
    out_c = OUTPUT_DIR / 'Fig7_c_example_fiber_ppr_profiles.pdf'
    finalize_figure(fig_c, title='Example fiber PPR profiles', save_path=out_c)

    print(f"✓ Plotted {len(fiber_boutons)} boutons from fiber '{EXAMPLE_FIBER_PREFIX}'")
    print(f'✓ Saved panel a to {out_a}')
    print(f'✓ Saved panel b to {out_b}')
    print(f'✓ Saved panel c to {out_c}')

    cluster_distribution = fiber_boutons['HC_Cluster'].value_counts().sort_index()
    print('\nFiber cluster composition:')
    for cluster_id, count in cluster_distribution.items():
        print(f'  Cluster {cluster_id}: {count} boutons')


#### Population summaries of within-fiber diversity

The population-level analyses quantify how much PCA territory is covered by single fibers and how many WT classes co-occur along those fibers. Both metrics are compared against chance-level random bouton sets.


In [ ]:
# === Fig 7d — Fiber PCA coverage vs random bouton sets ===
# Extract fiber IDs from WT pooled data, collapsing PortionA/B into the same parent fiber.
PCA_Data_WT_Pooled['FiberID'] = PCA_Data_WT_Pooled['ID'].map(_extract_fiber_id)

fiber_ids = PCA_Data_WT_Pooled['FiberID'].dropna().unique()

# Filter fibers with more than 2 boutons
fiber_bouton_counts = PCA_Data_WT_Pooled.groupby('FiberID').size().to_dict()
eligible_fibers = [fiber for fiber, count in fiber_bouton_counts.items() if count > 2]

print(f"Fibers with >2 boutons: {len(eligible_fibers)} out of {len(fiber_ids)} total fibers")

# Calculate grid dimensions for all eligible fibers
n_fibers = len(eligible_fibers)
n_cols = 3
n_rows = int(np.ceil(n_fibers / n_cols))

# Create figure with subplots for all eligible fibers
fig, axes = make_figure_grid(n_rows, n_cols, figsize=(18, 6 * n_rows))
axes = axes.flatten()

# Color palette for fibers
set1_cmap = plt.get_cmap('Set1')

# Plot each eligible fiber in its own subplot
for idx, fiber_prefix in enumerate(eligible_fibers):
    ax = axes[idx]
    
    # Plot WT pooled with cluster colors (background) - lighter
    plot_pca_background(ax, alpha=0.15, s=20, label=None)
    
    # Find boutons belonging to this fiber
    fiber_mask = PCA_Data_WT_Pooled['FiberID'] == fiber_prefix
    fiber_indices = np.where(fiber_mask)[0]
    
    if len(fiber_indices) > 0:
        # Use consistent color for this fiber
        fiber_color = set1_cmap(idx % 9)  # Cycle through Set1 colors
        
        # Plot all boutons from this fiber with the same color
        plot_pca_overlay_points(ax, pca_coordinates[fiber_indices],
                   color=fiber_color,  marker='o', 
                   edgecolors='black', linewidths=0.0, alpha=0.9,
                   label=f'{fiber_prefix}\n(n={len(fiber_indices)} boutons)')
    
    # Format subplot
    style_pca_axes(ax, title=f'Fiber: {fiber_prefix}',  legend=False)
    add_legend(ax, loc='upper right', fontsize=6, frameon=False)

# Hide unused subplots
for idx in range(n_fibers, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.suptitle(f'PCA: All Fibers from WT Pooled with >2 boutons ({n_fibers} fibers)', 
             fontsize=8, fontweight='bold', y=1.002)
output_file_fibers = OUTPUT_DIR / 'Aux_wt_pooled_fibers_overview_gt2_boutons.pdf'
plt.savefig(output_file_fibers, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved fiber overview figure to {output_file_fibers}")

# Fiber area distribution
from scipy.spatial import ConvexHull
from scipy.stats import mannwhitneyu
from concurrent.futures import ThreadPoolExecutor
import multiprocessing

# Analyze PCA area coverage for individual fibers with 4+ boutons


def calculate_convex_hull_area(points):
    """Calculate area of convex hull for a set of 2D points."""
    if len(points) < 3:
        return 0.0
    try:
        hull = ConvexHull(points)
        return hull.volume  # In 2D, volume is area
    except Exception:
        return 0.0

def plot_convex_hull_outline(ax, points, color='black', fill_alpha=0.12, linewidth=1.2):
    """Plot a filled convex hull outline for a 2D point cloud."""
    if len(points) < 3:
        return
    try:
        hull = ConvexHull(points)
        hull_points = points[hull.vertices]
        hull_points = np.vstack([hull_points, hull_points[0]])
        ax.fill(hull_points[:, 0], hull_points[:, 1], color=color, alpha=fill_alpha, linewidth=0)
        ax.plot(hull_points[:, 0], hull_points[:, 1], color=color, linewidth=linewidth, alpha=0.95)
    except Exception:
        pass

# Extract fiber IDs from WT pooled data, collapsing PortionA/B into the same parent fiber.
PCA_Data_WT_Pooled['FiberID'] = PCA_Data_WT_Pooled['ID'].map(_extract_fiber_id)
fiber_ids = PCA_Data_WT_Pooled['FiberID'].dropna().unique()

# Filter fibers with 4+ boutons
fiber_bouton_counts = PCA_Data_WT_Pooled['FiberID'].value_counts().to_dict()
eligible_fibers = {fiber: count for fiber, count in fiber_bouton_counts.items() if count >= 4}

print(f"=== FIBER AREA ANALYSIS ===")
print(f"Total unique fibers in WT pooled: {len(fiber_ids)}")
print(f"Fibers with 4+ boutons: {len(eligible_fibers)}")

example_fiber_prefix = EXAMPLE_FIBER_PREFIX if 'EXAMPLE_FIBER_PREFIX' in globals() else '20210304_linescan1'

# Calculate real fiber areas
real_fiber_areas = []
fiber_size_distribution = {}  # Track how many boutons per fiber

for fiber_prefix, n_boutons in eligible_fibers.items():
    fiber_mask = PCA_Data_WT_Pooled['FiberID'] == fiber_prefix
    fiber_indices = np.where(fiber_mask)[0]
    fiber_coords = pca_coordinates[fiber_indices]
    
    area = calculate_convex_hull_area(fiber_coords)
    real_fiber_areas.append(area)
    
    # Track size distribution
    if n_boutons not in fiber_size_distribution:
        fiber_size_distribution[n_boutons] = 0
    fiber_size_distribution[n_boutons] += 1

print(f"\nFiber size distribution:")
for size in sorted(fiber_size_distribution.keys()):
    count = fiber_size_distribution[size]
    print(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)")

# Generate null distribution by random sampling
n_permutations = 10000
null_areas = []

print(f"\nGenerating null distribution with {n_permutations} permutations...")

# Get all WT pooled coordinates
all_wt_coords = pca_coordinates.copy()
n_total_boutons = len(all_wt_coords)

def generate_null_areas_for_permutation(args):
    """Generate null areas for a single permutation."""
    fiber_size_distribution, all_wt_coords, seed = args
    np.random.seed(seed)
    n_total_boutons = len(all_wt_coords)
    perm_areas = []
    
    for fiber_size, n_fibers in fiber_size_distribution.items():
        for _ in range(n_fibers):
            random_indices = np.random.choice(n_total_boutons, size=fiber_size, replace=False)
            random_coords = all_wt_coords[random_indices]
            area = calculate_convex_hull_area(random_coords)
            perm_areas.append(area)
    
    return perm_areas

# Prepare arguments for parallel execution
seeds = np.random.randint(0, 2**31, size=n_permutations)
args_list = [(fiber_size_distribution, all_wt_coords, seed) for seed in seeds]

# Use multiprocessing to parallelize
# Use threading to parallelize (ProcessPoolExecutor fails in notebooks due to pickling issues)
n_workers = min(multiprocessing.cpu_count(), 8)
print(f"Using {n_workers} workers for parallel computation...")

with ThreadPoolExecutor(max_workers=n_workers) as executor:
    results = list(executor.map(generate_null_areas_for_permutation, args_list))
for perm_areas in results:
    null_areas.extend(perm_areas)

print(f"  Completed {n_permutations} permutations")

# Statistical comparison
real_areas_array = np.array(real_fiber_areas)
null_areas_array = np.array(null_areas)

# Mann-Whitney U test
u_stat, p_value = mannwhitneyu(real_areas_array, null_areas_array, alternative='two-sided')

# Calculate summary statistics
real_median = np.median(real_areas_array)
real_mean = np.mean(real_areas_array)
null_median = np.median(null_areas_array)
null_mean = np.mean(null_areas_array)

print(f"\n=== STATISTICAL RESULTS ===")
print(f"Real fibers (n={len(real_areas_array)}):")
print(f"  Mean area: {real_mean:.3f}")
print(f"  Median area: {real_median:.3f}")
print(f"  Range: [{np.min(real_areas_array):.3f}, {np.max(real_areas_array):.3f}]")

print(f"\nNull distribution (n={len(null_areas_array)}):")
print(f"  Mean area: {null_mean:.3f}")
print(f"  Median area: {null_median:.3f}")
print(f"  Range: [{np.min(null_areas_array):.3f}, {np.max(null_areas_array):.3f}]")

print(f"\nMann-Whitney U test:")
print(f"  U-statistic: {u_stat:.2f}")
print(f"  p-value: {p_value:.6g}")
print(f"  Effect: Real fibers {'smaller' if real_median < null_median else 'larger'} than random")


example_fiber_mask = PCA_Data_WT_Pooled['ID'].astype(str).str.startswith(example_fiber_prefix)
example_fiber_indices = np.where(example_fiber_mask)[0]
example_fiber_area = np.nan
if len(example_fiber_indices) >= 3:
    example_fiber_coords = pca_coordinates[example_fiber_indices]
    example_fiber_area = calculate_convex_hull_area(example_fiber_coords)
    example_fiber_clusters = cluster_assignments[example_fiber_mask]

    fig_example, ax_example = make_figure_grid(figsize=(7, 6))
    plot_pca_background(ax_example, alpha=0.12, s=20, label=None)
    for cluster_id in sorted(np.unique(example_fiber_clusters)):
        cluster_mask = example_fiber_clusters == cluster_id
        plot_pca_overlay_points(
            ax_example,
            example_fiber_coords[cluster_mask],
            color=get_cluster_color(int(cluster_id)),
            marker='o',
            edgecolors='black',
            linewidths=0.0,
            alpha=0.95,
            label=f'C{int(cluster_id)} (n={int(np.sum(cluster_mask))})',
        )
    plot_convex_hull_outline(ax_example, example_fiber_coords, color='black', fill_alpha=0.10, linewidth=1.0)
    style_pca_axes(
        ax_example,
        title=f'Example fiber: {example_fiber_prefix}\nConvex hull area = {example_fiber_area:.3f} PCA units²',
        legend=False,
    )
    add_legend(ax_example, loc='upper right', fontsize=7, frameon=False)
    output_file_example = OUTPUT_DIR / 'Fig7_d_example_fiber_pca_area_inset.pdf'
    finalize_figure(fig_example, title='Fig 7d example fiber PCA area inset', save_path=output_file_example)
    print(f"✓ Saved example fiber PCA area overlay to {output_file_example}")
else:
    print(f"Example fiber PCA area overlay not created: prefix '{example_fiber_prefix}' matched {len(example_fiber_indices)} boutons in PCA_Data_WT_Pooled")

# Visualization: overlay comparison only
all_areas = np.concatenate([real_areas_array, null_areas_array])
common_bins = np.linspace(0, np.percentile(all_areas, 99), 30)

make_figure(figsize=(10, 6))

plt.hist(null_areas_array, bins=common_bins, alpha=0.5, color='lightcoral', 
         edgecolor='black', label=f'Null (n={len(null_areas_array)})', density=True)
plt.hist(real_areas_array, bins=common_bins, alpha=0.7, color='steelblue', 
         edgecolor='black', label=f'Real fibers (n={len(real_areas_array)})', density=True)

plt.axvline(null_median, color='darkred', linestyle='--', linewidth=1, 
            label=f'Null median: {null_median:.3f}')
plt.axvline(real_median, color='darkblue', linestyle='--', linewidth=1, 
            label=f'Real median: {real_median:.3f}')

plt.xlabel('Convex Hull Area (PCA units²)')
plt.ylabel('Probability Density')
plt.title(f'Fiber Area Distribution Comparison\nMann-Whitney U={u_stat:.0f}, p={p_value:.6g}')
add_legend(plt.gca(), frameon=False)
if 'style_hist_axis' in globals():
    style_hist_axis(plt.gca())
plt.grid(False)
plt.tight_layout()

output_file_overlay = OUTPUT_DIR / "Fig7_d_fiber_area_overlay_comparison.pdf"
plt.savefig(output_file_overlay, dpi=300, bbox_inches='tight')
plt.show()
print(f"\n✓ Saved area distribution plot:")
print(f"  {output_file_overlay}")

# Save detailed results to text file
stats_output = OUTPUT_DIR / "fiber_area_statistics.txt"
with open(stats_output, 'w') as f:
    f.write("FIBER AREA ANALYSIS: REAL vs NULL DISTRIBUTION\n")
    f.write("=" * 60 + "\n\n")
    
    f.write(f"Dataset: WT pooled (fibers with 4+ boutons)\n")
    f.write(f"Total fibers analyzed: {len(eligible_fibers)}\n\n")
    
    f.write("FIBER SIZE DISTRIBUTION:\n")
    for size in sorted(fiber_size_distribution.keys()):
        count = fiber_size_distribution[size]
        f.write(f"  {size} boutons: {count} fibers ({100*count/len(eligible_fibers):.1f}%)\n")
    
    f.write(f"\nREAL FIBER AREAS (n={len(real_areas_array)}):\n")
    f.write(f"  Mean:   {real_mean:.6f}\n")
    f.write(f"  Median: {real_median:.6f}\n")
    f.write(f"  SD:     {np.std(real_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(real_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(real_areas_array):.6f}\n")
    
    f.write(f"\nNULL DISTRIBUTION (n={len(null_areas_array)}):\n")
    f.write(f"  Permutations: {n_permutations}\n")
    f.write(f"  Mean:   {null_mean:.6f}\n")
    f.write(f"  Median: {null_median:.6f}\n")
    f.write(f"  SD:     {np.std(null_areas_array):.6f}\n")
    f.write(f"  Min:    {np.min(null_areas_array):.6f}\n")
    f.write(f"  Max:    {np.max(null_areas_array):.6f}\n")
    
    f.write(f"\nSTATISTICAL TEST:\n")
    f.write(f"  Test: Mann-Whitney U (two-sided)\n")
    f.write(f"  U-statistic: {u_stat:.6f}\n")
    f.write(f"  p-value: {p_value:.10f}\n")
    f.write(f"  Effect: Real fibers are {'smaller' if real_median < null_median else 'larger'} than random\n")
    f.write(f"  Median difference: {real_median - null_median:.6f}\n")

print(f"✓ Saved detailed statistics to {stats_output}")

if np.isfinite(example_fiber_area):
    print(f"Example fiber area ({example_fiber_prefix}): {example_fiber_area:.3f} PCA units²")


In [ ]:
# === Fig 7e — Number of classes per fiber vs chance ===
# Analyze cluster diversity within individual fibers (boutons from same axon)

def extract_fiber_id(bouton_id):
    """Extract fiber ID from bouton ID for both WT linescan and SynII Fibre naming."""
    if '_extract_fiber_id' in globals():
        return _extract_fiber_id(bouton_id)
    s = str(bouton_id).strip().replace('_traces_converted', '')
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    return m.group(1).rstrip('_ ') if m else s

def analyze_fiber_diversity(min_boutons_per_fiber=4):
    """Analyze how clusters are distributed within individual fibers."""

    fiber_data = PCA_Data_WT_Pooled_clustered.copy()
    fiber_data['FiberID'] = fiber_data['ID'].apply(extract_fiber_id)
    fiber_data['Fiber_ID'] = fiber_data['FiberID']

    fiber_bouton_counts = fiber_data.groupby('Fiber_ID').size()

    valid_fibers  = fiber_bouton_counts[fiber_bouton_counts >= min_boutons_per_fiber].index
    filtered_data = fiber_data[fiber_data['Fiber_ID'].isin(valid_fibers)]

    print(f"Fiber analysis: {len(valid_fibers)} fibers with {min_boutons_per_fiber}+ boutons")
    print(f"Total boutons analyzed: {len(filtered_data)}")

    return filtered_data, valid_fibers

def calculate_cluster_diversity(filtered_data):
    """Calculate number of different clusters per fiber."""
    fiber_diversity = {}

    for fiber_id in filtered_data['Fiber_ID'].unique():
        fiber_boutons        = filtered_data[filtered_data['Fiber_ID'] == fiber_id]
        unique_clusters      = fiber_boutons['HC_Cluster'].nunique()
        total_boutons        = len(fiber_boutons)
        cluster_distribution = fiber_boutons['HC_Cluster'].value_counts(normalize=True)

        fiber_diversity[fiber_id] = {
            'num_cluster_types': unique_clusters,
            'total_boutons': total_boutons,
            'cluster_props': cluster_distribution.to_dict()
        }

    return fiber_diversity

def diversity_category_label(num_types):
    return str(num_types) if num_types <= 3 else '4+'

def bootstrap_diversity_chance(filtered_data, fiber_diversity, diversity_order, n_boot=2000, seed=42):
    """Chance expectation preserving fiber sizes and global cluster frequencies."""
    rng = np.random.default_rng(seed)
    cluster_freq = filtered_data['HC_Cluster'].value_counts(normalize=True).sort_index()
    cluster_ids = cluster_freq.index.to_numpy(dtype=int)
    cluster_probs = cluster_freq.to_numpy(dtype=float)
    fiber_sizes = np.asarray([info['total_boutons'] for info in fiber_diversity.values()], dtype=int)
    n_fibers = fiber_sizes.size
    chance_pct = np.zeros((n_boot, len(diversity_order)), dtype=float)

    for boot_idx in range(n_boot):
        counts = dict.fromkeys(diversity_order, 0)
        for fiber_size in fiber_sizes:
            sampled_clusters = rng.choice(cluster_ids, size=int(fiber_size), replace=True, p=cluster_probs)
            category = diversity_category_label(np.unique(sampled_clusters).size)
            counts[category] += 1
        chance_pct[boot_idx] = [100.0 * counts[category] / n_fibers for category in diversity_order]

    return {
        'mean': chance_pct.mean(axis=0),
        'ci_low': np.percentile(chance_pct, 2.5, axis=0),
        'ci_high': np.percentile(chance_pct, 97.5, axis=0),
    }

filtered_data, valid_fibers = analyze_fiber_diversity(min_boutons_per_fiber=4)
fiber_diversity             = calculate_cluster_diversity(filtered_data)

diversity_categories = {'1': [], '2': [], '3': [], '4+': []}
for fiber_id, info in fiber_diversity.items():
    category = diversity_category_label(info['num_cluster_types'])
    diversity_categories[category].append(info)

diversity_means  = {}
diversity_counts = {}
diversity_order  = ['1', '2', '3', '4+']

for category, fiber_list in diversity_categories.items():
    diversity_counts[category] = len(fiber_list)
    if len(fiber_list) > 0:
        cluster_means = {}
        for cluster_id in range(1, N_CLUSTERS + 1):
            proportions = [fiber['cluster_props'].get(cluster_id, 0) for fiber in fiber_list]
            cluster_means[cluster_id] = np.mean(proportions)
        diversity_means[category] = cluster_means
    else:
        diversity_means[category] = {i: 0 for i in range(1, N_CLUSTERS + 1)}

chance_summary = bootstrap_diversity_chance(
    filtered_data,
    fiber_diversity,
    diversity_order,
    n_boot=2000,
    seed=42,
)

fig, ax = make_figure_grid(figsize=(4.8, 3.8))

x_positions   = np.arange(len(diversity_order))
bottom_values = np.zeros(len(diversity_order))
total_fibers  = len(valid_fibers)

for cluster_id in range(1, N_CLUSTERS + 1):
    cluster_heights = []
    for category in diversity_order:
        fiber_percentage   = (diversity_counts[category] / total_fibers) * 100 if total_fibers else 0.0
        cluster_proportion = diversity_means[category][cluster_id]
        height             = fiber_percentage * cluster_proportion
        cluster_heights.append(height)

    ax.bar(
        x_positions,
        cluster_heights,
        width=0.55,
        bottom=bottom_values,
        color=get_cluster_color(cluster_id),
        label=f'C{cluster_id}',
        alpha=0.78,
        zorder=2,
    )
    bottom_values += cluster_heights

chance_mean = np.asarray(chance_summary['mean'], dtype=float)
chance_lo = np.asarray(chance_summary['ci_low'], dtype=float)
chance_hi = np.asarray(chance_summary['ci_high'], dtype=float)
chance_err = np.vstack([chance_mean - chance_lo, chance_hi - chance_mean])
chance_overlay = ax.errorbar(
    x_positions,
    chance_mean,
    yerr=chance_err,
    color='0.35',
    ecolor='0.35',
    linewidth=1.5,
    elinewidth=1.2,
    capsize=3,
    marker='o',
    markersize=4.5,
    markerfacecolor='white',
    markeredgewidth=1.0,
    label='Chance mean ± 95% CI',
    zorder=5,
)

ax.set_xlabel('Number of cluster types per fiber')
ax.set_ylabel('Percentage of fibers (%)')
ax.set_title('Fiber cluster diversity (fibers with ≥4 boutons)')
ax.set_xticks(x_positions)
ax.set_xticklabels(diversity_order)
ax.grid(False)
ax.set_ylim(0, 105)

handles, labels = ax.get_legend_handles_labels()
cluster_handles = handles[:N_CLUSTERS]
cluster_labels = labels[:N_CLUSTERS]
legend_handles = cluster_handles + [chance_overlay]
legend_labels = cluster_labels + ['Chance mean ± 95% CI']
add_legend(ax, legend_handles, legend_labels, bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()

output_file = OUTPUT_DIR / "Fig7_e_fiber_cluster_diversity_summary.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Saved to {output_file}")

print('\nFiber cluster diversity summary:')
print(f"{'Types/fiber':<12} {'n fibers':>8} {'% fibers':>8} {'Chance mean':>12} {'95% CI':>18} {'Mean cluster composition':<60}")
for idx, category in enumerate(diversity_order):
    count = diversity_counts[category]
    pct = (100.0 * count / total_fibers) if total_fibers else np.nan
    comp = '  '.join(
        f"C{cluster_id}={100 * diversity_means[category][cluster_id]:.1f}%"
        for cluster_id in range(1, N_CLUSTERS + 1)
        if diversity_means[category][cluster_id] > 0
    )
    if not comp:
        comp = '-'
    ci_txt = f"[{chance_lo[idx]:.1f}, {chance_hi[idx]:.1f}]"
    print(f"{category:<12} {count:>8} {pct:>7.1f}% {chance_mean[idx]:>11.1f}% {ci_txt:>18}  {comp}")

print(f"\nTotal fibers: {total_fibers}")
print(f"Total boutons: {len(filtered_data)}")


In [ ]:
# === Aux — Bouton-neighbor similarity along fibers ===
# Builds all same-fiber bouton pairs, then asks whether physically closer pairs
# are more often in the same class and closer in WT PCA space.

MIN_BOUTONS_NEIGHBOR_ANALYSIS = 3


def extract_fiber_segment_id(bouton_id):
    """Return the continuous imaged segment ID, keeping PortionA/B when present."""
    s = str(bouton_id).strip().replace('_traces_converted', '')
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    return m.group(1).rstrip('_ ') if m else s


def extract_fiber_portion_label(bouton_id):
    m = re.search(r'[_\s]*Portion([A-Za-z0-9]+)', str(bouton_id), flags=re.IGNORECASE)
    return m.group(1).upper() if m else ''


def _portion_sort_value(label):
    label = str(label).strip().upper()
    if not label:
        return 0
    if label.isdigit():
        return int(label)
    value = 0
    for char in label:
        if 'A' <= char <= 'Z':
            value = value * 26 + (ord(char) - ord('A') + 1)
    return value if value else 0


def extract_bouton_number(bouton_id):
    m = re.search(r'(?:Bouton[_\s]*(\d+)|bouton(\d+))', str(bouton_id), flags=re.IGNORECASE)
    if not m:
        return np.nan
    for group in m.groups():
        if group is not None:
            return float(group)
    return np.nan


def build_fiber_neighbor_pair_table(min_boutons_per_fiber=MIN_BOUTONS_NEIGHBOR_ANALYSIS):
    data = PCA_Data_WT_Pooled_clustered.reset_index(drop=True).copy()
    data['Fiber_ID'] = data['ID'].apply(extract_fiber_id)
    data['Fiber_Segment_ID'] = data['ID'].apply(extract_fiber_segment_id)
    data['Fiber_Portion'] = data['ID'].apply(extract_fiber_portion_label)
    data['Fiber_Portion_Order'] = data['Fiber_Portion'].map(_portion_sort_value)
    data['Bouton_Order'] = data['ID'].apply(extract_bouton_number)

    coords = np.asarray(pca_coordinates, dtype=float)[:, :2]
    if len(coords) != len(data):
        raise RuntimeError(f"PCA coords/rows mismatch: {len(coords)} vs {len(data)}")
    data['PC1'] = coords[:, 0]
    data['PC2'] = coords[:, 1]

    needed = ['Fiber_ID', 'Fiber_Segment_ID', 'Bouton_Order', 'HC_Cluster', 'PC1', 'PC2']
    usable = data.dropna(subset=needed).copy()
    rows = []

    for fiber_id, fiber_df in usable.groupby('Fiber_ID', sort=False):
        if len(fiber_df) < int(min_boutons_per_fiber):
            continue
        ordered = fiber_df.sort_values(['Fiber_Portion_Order', 'Bouton_Order', 'ID']).reset_index(drop=True)
        for i in range(len(ordered) - 1):
            for j in range(i + 1, len(ordered)):
                a = ordered.iloc[i]
                b = ordered.iloc[j]
                same_segment = str(a['Fiber_Segment_ID']) == str(b['Fiber_Segment_ID'])
                neighbor_order = abs(float(b['Bouton_Order']) - float(a['Bouton_Order'])) if same_segment else np.nan
                pca_distance = float(np.linalg.norm([b['PC1'] - a['PC1'], b['PC2'] - a['PC2']]))
                rows.append({
                    'Fiber_ID': fiber_id,
                    'Bouton_A': a['ID'],
                    'Bouton_B': b['ID'],
                    'Segment_A': a['Fiber_Segment_ID'],
                    'Segment_B': b['Fiber_Segment_ID'],
                    'Same_Continuous_Segment': bool(same_segment),
                    'Neighbor_Order': neighbor_order,
                    'Global_Order_Difference': int(abs(j - i)),
                    'Cluster_A': int(a['HC_Cluster']),
                    'Cluster_B': int(b['HC_Cluster']),
                    'Same_Class': int(a['HC_Cluster'] == b['HC_Cluster']),
                    'PC1_A': float(a['PC1']),
                    'PC2_A': float(a['PC2']),
                    'PC1_B': float(b['PC1']),
                    'PC2_B': float(b['PC2']),
                    'PCA_Distance': pca_distance,
                })

    return pd.DataFrame(rows)


fiber_neighbor_pair_df = build_fiber_neighbor_pair_table()
fiber_neighbor_contiguous_df = fiber_neighbor_pair_df[np.isfinite(fiber_neighbor_pair_df['Neighbor_Order'])].copy()
fiber_neighbor_contiguous_df['Neighbor_Order'] = fiber_neighbor_contiguous_df['Neighbor_Order'].astype(int)

neighbor_summary = (
    fiber_neighbor_contiguous_df
    .groupby('Neighbor_Order')
    .agg(
        n_pairs=('Same_Class', 'size'),
        n_fibers=('Fiber_ID', 'nunique'),
        same_class_probability=('Same_Class', 'mean'),
        pca_distance_mean=('PCA_Distance', 'mean'),
        pca_distance_sem=('PCA_Distance', lambda x: np.nanstd(x, ddof=1) / np.sqrt(max(np.sum(np.isfinite(x)), 1))),
    )
    .reset_index()
)

if len(neighbor_summary) == 0:
    raise RuntimeError('No contiguous same-segment bouton pairs available for neighbor-order analysis.')


def _bh_fdr_numpy(p_values):
    p_values = np.asarray(p_values, dtype=float)
    corrected = np.full(p_values.shape, np.nan, dtype=float)
    valid = np.isfinite(p_values)
    if not np.any(valid):
        return corrected
    p = p_values[valid]
    order = np.argsort(p)
    ranked = p[order]
    m = float(len(ranked))
    adj = ranked * m / np.arange(1, len(ranked) + 1, dtype=float)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    adj = np.clip(adj, 0.0, 1.0)
    out = np.empty_like(adj)
    out[order] = adj
    corrected[valid] = out
    return corrected


def _fisher_exact_two_sided_standard(a, b, c, d):
    """Two-sided Fisher exact p value for a 2x2 table using only stdlib comb."""
    from math import comb
    a, b, c, d = int(a), int(b), int(c), int(d)
    row1 = a + b
    row2 = c + d
    col1 = a + c
    total = row1 + row2
    if total == 0:
        return np.nan
    lo = max(0, col1 - row2)
    hi = min(row1, col1)
    denom = comb(total, col1)
    if denom == 0:
        return np.nan
    def prob(x):
        return comb(row1, x) * comb(row2, col1 - x) / denom
    p_obs = prob(a)
    return float(sum(prob(x) for x in range(lo, hi + 1) if prob(x) <= p_obs + 1e-12))


def pointwise_neighbor_order_stats(pair_df, reference_order=None):
    """Compare each neighbor order against the reference order, usually N+1."""
    available_orders = sorted(int(v) for v in pair_df['Neighbor_Order'].dropna().unique())
    if not available_orders:
        return pd.DataFrame(), np.nan
    ref_order = int(reference_order) if reference_order is not None else available_orders[0]
    if ref_order not in available_orders:
        ref_order = available_orders[0]
    ref = pair_df.loc[pair_df['Neighbor_Order'] == ref_order].copy()
    rows = []
    for order in available_orders:
        cur = pair_df.loc[pair_df['Neighbor_Order'] == order].copy()
        same_cur = int(cur['Same_Class'].sum())
        diff_cur = int(len(cur) - same_cur)
        same_ref = int(ref['Same_Class'].sum())
        diff_ref = int(len(ref) - same_ref)
        if order == ref_order:
            class_p = np.nan
            dist_u = np.nan
            dist_p = np.nan
        else:
            class_p = _fisher_exact_two_sided_standard(same_cur, diff_cur, same_ref, diff_ref)
            dist_u = np.nan
            dist_p = np.nan
            if 'mannwhitneyu' in globals():
                x = cur['PCA_Distance'].dropna().to_numpy(dtype=float)
                y = ref['PCA_Distance'].dropna().to_numpy(dtype=float)
                if len(x) and len(y):
                    dist_u, dist_p = mannwhitneyu(x, y, alternative='two-sided')
        rows.append({
            'Neighbor_Order': int(order),
            'Reference_Order': int(ref_order),
            'n_pairs': int(len(cur)),
            'n_ref_pairs': int(len(ref)),
            'same_class_pct': float(100.0 * cur['Same_Class'].mean()) if len(cur) else np.nan,
            'ref_same_class_pct': float(100.0 * ref['Same_Class'].mean()) if len(ref) else np.nan,
            'same_class_fisher_p': class_p,
            'pca_distance_mean': float(cur['PCA_Distance'].mean()) if len(cur) else np.nan,
            'ref_pca_distance_mean': float(ref['PCA_Distance'].mean()) if len(ref) else np.nan,
            'pca_distance_U': float(dist_u) if np.isfinite(dist_u) else np.nan,
            'pca_distance_mwu_p': float(dist_p) if np.isfinite(dist_p) else np.nan,
        })
    stats_df = pd.DataFrame(rows)
    stats_df['same_class_fisher_p_fdr'] = _bh_fdr_numpy(stats_df['same_class_fisher_p'].to_numpy(dtype=float))
    stats_df['pca_distance_mwu_p_fdr'] = _bh_fdr_numpy(stats_df['pca_distance_mwu_p'].to_numpy(dtype=float))
    return stats_df, ref_order


neighbor_pointwise_stats_df, neighbor_reference_order = pointwise_neighbor_order_stats(fiber_neighbor_contiguous_df, reference_order=1)

fig, axes = make_figure_grid(1, 2, panel_kind='simple', figsize=(7.2, 3.0))
x = neighbor_summary['Neighbor_Order'].to_numpy(dtype=float)
same_pct = 100.0 * neighbor_summary['same_class_probability'].to_numpy(dtype=float)
axes[0].plot(x, same_pct, marker='o', color='black', linewidth=1.2)
axes[0].set_xlabel('Neighbor order on same segment')
axes[0].set_ylabel('Same class (%)')
axes[0].set_ylim(0, 100)
axes[0].set_title('Class similarity')

dist_mean = neighbor_summary['pca_distance_mean'].to_numpy(dtype=float)
dist_sem = neighbor_summary['pca_distance_sem'].to_numpy(dtype=float)
axes[1].errorbar(x, dist_mean, yerr=dist_sem, marker='o', color='black', linewidth=1.2, capsize=2)
axes[1].set_xlabel('Neighbor order on same segment')
axes[1].set_ylabel('PCA distance')
axes[1].set_title('Profile-space distance')

for ax in axes:
    ax.set_xticks(x)
    ax.grid(False)
    if '_apply_clean_axes_style' in globals():
        _apply_clean_axes_style(ax, panel_kind='simple')

out_neighbor = OUTPUT_DIR / 'Aux_fiber_neighbor_similarity_by_order.pdf'
finalize_figure(fig, title='Within-fiber neighbor similarity', save_path=out_neighbor)

print('Within-fiber bouton-pair table:')
print(f"  all same-fiber pairs: {len(fiber_neighbor_pair_df)}")
print(f"  contiguous same-segment pairs used for neighbor order: {len(fiber_neighbor_contiguous_df)}")
print(f"  cross-portion same-fiber pairs flagged but excluded from neighbor order: {int((~fiber_neighbor_pair_df['Same_Continuous_Segment']).sum())}")
print('\nNeighbor-order summary:')
print(neighbor_summary.to_string(index=False, formatters={
    'same_class_probability': lambda v: f'{100.0 * v:.1f}%',
    'pca_distance_mean': lambda v: f'{v:.3f}',
    'pca_distance_sem': lambda v: f'{v:.3f}',
}))

print(f"\nPointwise stats versus N+{neighbor_reference_order} reference:")
if len(neighbor_pointwise_stats_df):
    print(neighbor_pointwise_stats_df.to_string(index=False, formatters={
        'same_class_pct': lambda v: f'{v:.1f}%',
        'ref_same_class_pct': lambda v: f'{v:.1f}%',
        'same_class_fisher_p': lambda v: 'ref' if not np.isfinite(v) else f'{v:.3g}',
        'same_class_fisher_p_fdr': lambda v: 'ref' if not np.isfinite(v) else f'{v:.3g}',
        'pca_distance_mean': lambda v: f'{v:.3f}',
        'ref_pca_distance_mean': lambda v: f'{v:.3f}',
        'pca_distance_U': lambda v: 'ref' if not np.isfinite(v) else f'{v:.0f}',
        'pca_distance_mwu_p': lambda v: 'ref' if not np.isfinite(v) else f'{v:.3g}',
        'pca_distance_mwu_p_fdr': lambda v: 'ref' if not np.isfinite(v) else f'{v:.3g}',
    }))
    significant_rows = neighbor_pointwise_stats_df[
        (neighbor_pointwise_stats_df['same_class_fisher_p_fdr'] < 0.05) |
        (neighbor_pointwise_stats_df['pca_distance_mwu_p_fdr'] < 0.05)
    ]
    if len(significant_rows):
        sig_orders = ', '.join(f"N+{int(v)}" for v in significant_rows['Neighbor_Order'])
        print(f"  Significant after FDR at: {sig_orders}")
    else:
        print('  No neighbor order differs from reference after FDR correction at alpha=0.05.')
else:
    print('  No pointwise stats available.')
if 'mannwhitneyu' not in globals():
    print('  PCA distance tests skipped: mannwhitneyu is not currently imported.')

print(f"✓ Saved to {out_neighbor}")


In [ ]:
# === Aux — Fiber diversity hulls (real vs shuffle) ===
# Illustrate real and label-shuffled within-fiber convex hulls after recentering each fiber in PCA space.
# Two-panel figure illustrating per-fiber spread in WT PCA space:
#   (A) each fiber's bouton cloud as a translucent convex hull, recentred
#       on the fiber centroid (origin), so spread/shape is comparable.
#   (B) same after shuffling the bouton -> fiber assignment (preserving
#       fiber sizes), as a null reference for the diversity in (A).
# Uses min_boutons_per_fiber=4 to match `analyze_fiber_diversity`.

from scipy.spatial import ConvexHull

MIN_BOUTONS_FIBER_HULL = 4
N_SHUFFLE_REPLICATES = 1  # 1 shuffle realisation for the null panel

_fiber_df = PCA_Data_WT_Pooled.reset_index(drop=True).copy()
_fiber_df['FiberID'] = _fiber_df['ID'].apply(extract_fiber_id)

_pc_xy = np.asarray(pca_coordinates)[:, :2]
if len(_pc_xy) != len(_fiber_df):
    raise RuntimeError(f"PCA coords/rows mismatch: {len(_pc_xy)} vs {len(_fiber_df)}")
_fiber_ids = _fiber_df['FiberID'].to_numpy()

_counts = pd.Series(_fiber_ids).value_counts()
_valid = set(_counts[_counts >= MIN_BOUTONS_FIBER_HULL].index)


def _centred_hulls(coords, fibers, valid_set):
    out = []
    for fid in valid_set:
        m = (fibers == fid)
        pts = coords[m]
        if len(pts) < 3:
            continue
        c = pts.mean(axis=0)
        centred = pts - c
        try:
            hull = ConvexHull(centred)
            out.append(centred[hull.vertices])
        except Exception:
            continue
    return out


def _average_hull(hulls, n_angles=360):
    """Compute the average convex hull outline by averaging radial distances.
    Returns closed (x, y) arrays for plotting."""
    if not hulls:
        return None, None
    angles = np.linspace(0, 2 * np.pi, n_angles, endpoint=False)
    radii = np.zeros((len(hulls), n_angles))
    for i, verts in enumerate(hulls):
        closed = np.vstack([verts, verts[0]])
        for j, theta in enumerate(angles):
            d = np.cos(theta) * closed[:, 0] + np.sin(theta) * closed[:, 1]
            radii[i, j] = np.max(d)
    avg_r = radii.mean(axis=0)
    x = avg_r * np.cos(angles)
    y = avg_r * np.sin(angles)
    return np.append(x, x[0]), np.append(y, y[0])


_real_hulls = _centred_hulls(_pc_xy, _fiber_ids, _valid)

rng = np.random.default_rng(42)
_shuffled = _fiber_ids.copy()
rng.shuffle(_shuffled)
_null_hulls = _centred_hulls(_pc_xy, _shuffled, _valid)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(8.5, 4.2), sharex=True, sharey=True)

def _draw(ax, hulls, title):
    for verts in hulls:
        poly = plt.Polygon(verts, closed=True,
                           facecolor='0.25', edgecolor='0.15',
                           alpha=0.15, linewidth=0.6)
        ax.add_patch(poly)
    # Average surface overlay
    avg_x, avg_y = _average_hull(hulls)
    if avg_x is not None:
        ax.plot(avg_x, avg_y, linestyle='--', linewidth=1.4,
                color='tab:red', alpha=0.85, label='average hull')
    ax.axhline(0, color='k', lw=0.5, alpha=0.4)
    ax.axvline(0, color='k', lw=0.5, alpha=0.4)
    ax.set_title(f"{title} (n={len(hulls)} fibers, " + r"$\geq$" + f"{MIN_BOUTONS_FIBER_HULL} boutons)")
    ax.set_xlabel('PC1 — fiber centroid')
    ax.set_aspect('equal', adjustable='box')
    ax.legend(fontsize=7, loc='upper right')

_draw(axes[0], _real_hulls, 'Real fibers')
_draw(axes[1], _null_hulls, 'Shuffled (label permutation null)')
axes[0].set_ylabel('PC2 — fiber centroid')

# Use a symmetric range that fits both panels
all_pts = np.vstack(_real_hulls + _null_hulls) if (_real_hulls and _null_hulls) else None
if all_pts is not None and len(all_pts):
    rmax = np.max(np.abs(all_pts)) * 1.05
    axes[0].set_xlim(-rmax, rmax); axes[0].set_ylim(-rmax, rmax)

fig.tight_layout()
output_file = OUTPUT_DIR / 'Aux_fiber_diversity_hulls_real_vs_shuffle.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Real hulls drawn: {len(_real_hulls)}; shuffle hulls drawn: {len(_null_hulls)}")
print(f"✓ Saved to {output_file}")


## Section 8 — Figure 8: Postsynaptic target identity


### Postsynaptic target identity

These cells test whether bouton diversity can be reduced to the identity of the postsynaptic target. WT PCA classes are kept fixed while PC- and interneuron-associated boutons are projected onto the same reference space.

This section addresses the manuscript result that target identity can influence apparent release probability or failures, yet does not explain the full diversity of STP profiles along single fibers.


#### Target-aligned trace comparison

The main target-identity trace panel compares the average WT responses of PC- and interneuron-associated boutons in the original fluorescence domain. This keeps the target effect anchored to the measured train response before the PCA and MLE summaries.


In [ ]:
# === Fig 8c — PC vs IN mean traces ===
# Overlay average WT traces for boutons targeting PCs vs interneurons.
target_series = PCA_Data_WT_Pooled_clustered['Target'].astype(str).str.upper()
pc_ids = PCA_Data_WT_Pooled_clustered.loc[target_series == 'PC', 'ID'].tolist()
in_ids = PCA_Data_WT_Pooled_clustered.loc[target_series == 'IN', 'ID'].tolist()

pc_stats = compute_trace_stats(trace_ids=pc_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
in_stats = compute_trace_stats(trace_ids=in_ids, source=TRACE_MEAN_SOURCE, condition_names=WT_2_5_20HZ_CONDITIONS)
all_values = np.concatenate([
    np.asarray(pc_stats['average'] - pc_stats['sem'], float),
    np.asarray(pc_stats['average'] + pc_stats['sem'], float),
    np.asarray(in_stats['average'] - in_stats['sem'], float),
    np.asarray(in_stats['average'] + in_stats['sem'], float),
])
y_min, y_max = np.nanmin(all_values), np.nanmax(all_values)
y_pad = max((y_max - y_min) * 0.05, 0.05)
y_lim = (y_min - y_pad, y_max + y_pad)
stim_times = [1.0 + 0.05 * i for i in range(10)]

fig, ax = make_figure_grid(figsize=(7, 5))
plot_trace_analysis(ax=ax, rows=pc_stats['rows'], source=TRACE_MEAN_SOURCE, color=PC_TARGET_COLOR, label=f'PC (n={pc_stats["n"]})', display='mean_sem', style_axis=False)
plot_trace_analysis(ax=ax, rows=in_stats['rows'], source=TRACE_MEAN_SOURCE, color=IN_TARGET_COLOR, label=f'IN (n={in_stats["n"]})', display='mean_sem', stim_times=stim_times, stim_kwargs={'mode': 'top', 'linewidth': 1.3}, zero_line=True, event_time=1.0, event_kwargs={'color': 'gray', 'linestyle': '--', 'alpha': 0.5}, xlim=TRACE_XLIM_20HZ, ylim=y_lim, xlabel='Time (s)', ylabel='ΔF/F', title='PC vs IN mean traces', legend=True)
out_c = OUTPUT_DIR / 'Fig8_c_pc_vs_in_mean_traces.pdf'
finalize_figure(fig, title='PC vs IN mean traces', save_path=out_c)
print(f"✓ PC traces: n={pc_stats['n']}")
print(f"✓ IN traces: n={in_stats['n']}")
print(f"✓ Saved to {out_c}")


In [ ]:
# === Fig 8d — Target overlay on PCA scatter ===
# Scatter plot: WT pooled points with PC (dark green) and IN (purple) overlays
# Assumes pca_coordinates (np.ndarray), PCA_Data_WT_Pooled (DataFrame) and PCA_RESULTS are available

make_figure(figsize=(9, 7))

# Plot original PCA
plot_pca_background(plt.gca(), alpha=0.4,  marker='o', linewidths=0.0, label='WT pooled')


# Target masks
targets = PCA_Data_WT_Pooled['Target'].values
pc_mask = targets == 'PC'
in_mask = targets == 'IN'

# Calculate medians for PC and IN groups
pc_coordinates = pca_coordinates[pc_mask]
in_coordinates = pca_coordinates[in_mask]

pc_median = np.median(pc_coordinates, axis=0) if len(pc_coordinates) > 0 else None
in_median = np.median(in_coordinates, axis=0) if len(in_coordinates) > 0 else None

# Calculate distance between medians
if pc_median is not None and in_median is not None:
    median_distance = np.linalg.norm(pc_median - in_median)
else:
    median_distance = None

overlay_groups = []
if np.any(pc_mask):
    overlay_groups.append(make_pca_group(pca_coordinates[pc_mask], label='PC', color=PC_TARGET_COLOR, marker='o', edgecolors='black', linewidths=0.0))
if np.any(in_mask):
    overlay_groups.append(make_pca_group(pca_coordinates[in_mask], label='IN', color=IN_TARGET_COLOR, marker='^', edgecolors='black', linewidths=0.0))
if pc_median is not None and in_median is not None:
    overlay_groups.append(make_pca_group(np.array([pc_median]), label='_nolegend_', color=PC_TARGET_MEDIAN_COLOR, marker='X', edgecolors='black', linewidths=0.0, s=35, alpha=0.9))
    overlay_groups.append(make_pca_group(np.array([in_median]), label='_nolegend_', color=IN_TARGET_MEDIAN_COLOR, marker='X', edgecolors='black', linewidths=0.0, s=35, alpha=0.9))
plot_pca_analysis(plt.gca(), overlays=overlay_groups, title=None, legend=False)
if pc_median is not None and in_median is not None:
    plt.plot([pc_median[0], in_median[0]], [pc_median[1], in_median[1]],
             color='black', linestyle='--', linewidth=1.0, alpha=0.5,
             label='_nolegend_')


# Axis labels with explained variance
style_pca_axes(plt.gca(), title='PCA: WT pooled colored by Target (PC vs IN)',  legend=False)
add_legend(plt.gca(), loc='best', fontsize=6, frameon=False)
plt.tight_layout()

# Save and show
output_path = OUTPUT_DIR / "Fig8_d_target_overlay_pca_scatter.pdf"
plt.savefig(output_path, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved PCA PC/IN scatter to {output_path}")

# Summary statistics
print("=== PC vs IN SEPARATION ANALYSIS ===")
print(f"Purkinje Cells (PC): {np.sum(pc_mask):2d} samples")
print(f"Interneurons (IN):   {np.sum(in_mask):2d} samples")




if pc_median is not None and in_median is not None:
    print(f"PC median coordinates:     ({pc_median[0]:.3f}, {pc_median[1]:.3f})")
    print(f"IN median coordinates:     ({in_median[0]:.3f}, {in_median[1]:.3f})")
    print(f"Median-to-median distance: {median_distance:.3f}")
else:
    print("Cannot calculate median distance - insufficient data")


#### Bootstrap MLE trajectories by target

The same bootstrap binomial fits used for the WT calcium comparisons are split here by postsynaptic target identity. This restores the target-specific P and N trajectories that were present before the figure reordering.


In [ ]:
# === Fig 8e — PC vs IN final slope norm ===
# Compare the end-of-train slope from A1-normalized cumulative profiles for WT boutons by postsynaptic target.
from scipy.stats import linregress

if 'ppr_profiles_matrix' not in globals():
    raise RuntimeError('Run the shared PPR helper cell first (cell defining ppr_profiles_matrix).')

target_series = PCA_Data_WT_Pooled_clustered['Target'].astype(str).str.strip().str.upper()
pc_target_df = PCA_Data_WT_Pooled_clustered.loc[target_series == 'PC'].copy()
in_target_df = PCA_Data_WT_Pooled_clustered.loc[target_series == 'IN'].copy()
ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_WT_Pooled_clustered.columns]
if len(ppr_cols) == 0:
    raise RuntimeError('No PPR columns found for target cumulative slope panel.')

pc_profiles = ppr_profiles_matrix(pc_target_df, ppr_column_names=ppr_cols)
in_profiles = ppr_profiles_matrix(in_target_df, ppr_column_names=ppr_cols)
fig8_target_cumulative_profiles = {
    'PC': np.cumsum(pc_profiles, axis=1),
    'IN': np.cumsum(in_profiles, axis=1),
}
fig8_pulse_numbers = np.arange(1, pc_profiles.shape[1] + 1, dtype=float)
fig8_tail_len = int(min(mat.shape[1] for mat in fig8_target_cumulative_profiles.values()))
fig8_nfit = int(min(globals().get('N_FIT_LAST', 4), fig8_tail_len, len(fig8_pulse_numbers)))
fig8_x_tail = fig8_pulse_numbers[-fig8_nfit:]

fig_profile, ax_profile = make_figure_grid(figsize=(7, 5))
plot_profile_group_comparison(
    ax_profile,
    [
        {
            'matrix': fig8_target_cumulative_profiles['PC'],
            'color': PC_TARGET_COLOR,
            'marker': 'o',
            'label': f'PC (n={fig8_target_cumulative_profiles["PC"].shape[0]})',
            'linewidth': 1.6,
            'sem_alpha': 0.16,
        },
        {
            'matrix': fig8_target_cumulative_profiles['IN'],
            'color': IN_TARGET_COLOR,
            'marker': 's',
            'label': f'IN (n={fig8_target_cumulative_profiles["IN"].shape[0]})',
            'linewidth': 1.6,
            'sem_alpha': 0.16,
        },
    ],
    pulse_numbers=fig8_pulse_numbers,
    stats_start_index=1,
    show_stats=True,
    parametric_stat=True,
    summary_label='PC vs IN cumulative PPR profile',
    title='PC vs IN cumulative PPR profile',
    ylabel='Normalized cumulative slope',
    finalize_kwargs={'unity_line': False, 'legend': True, 'legend_loc': 'best'},
)
out_e_profile = OUTPUT_DIR / 'Fig8_e_pc_vs_in_cumulative_norm_a1.pdf'
finalize_figure(fig_profile, title='PC vs IN cumulative PPR profile', save_path=out_e_profile)
print(f"✓ Saved PC vs IN cumulative profile to {out_e_profile}")

fig8_slope_arrays = {}
for label, mat in fig8_target_cumulative_profiles.items():
    local_slopes = np.full(mat.shape[0], np.nan)
    for i, row in enumerate(mat):
        y_tail = np.asarray(row[-fig8_nfit:], dtype=float)
        ok = np.isfinite(y_tail)
        if ok.sum() >= 2:
            local_slopes[i] = linregress(fig8_x_tail[ok], y_tail[ok]).slope
    fig8_slope_arrays[label] = local_slopes[np.isfinite(local_slopes)]

fig_box, ax_box = make_figure_grid(panel_kind='boxplot_compact')
res = plot_boxplot_analysis(
    ax_box,
    [
        make_box_group(f'PC\n(n={len(fig8_slope_arrays["PC"])})', values=fig8_slope_arrays['PC'], color=PC_TARGET_COLOR),
        make_box_group(f'IN\n(n={len(fig8_slope_arrays["IN"])})', values=fig8_slope_arrays['IN'], color=IN_TARGET_COLOR),
    ],
    paired=False,
    show_points=True,
    show_stats=True,
    parametric_stat=False,
    title='PC vs IN final slope norm',
    ylabel='Final slope norm',
)
out_e = OUTPUT_DIR / 'Fig8_e_pc_vs_in_cumulative_slope_boxplot.pdf'
finalize_figure(fig_box, title='PC vs IN final slope norm', save_path=out_e)
if 'p_value' in res and np.isfinite(res['p_value']):
    print(f"PC vs IN final slope norm p = {res['p_value']:.4g}")
print(f"✓ Saved PC vs IN final slope norm to {out_e}")


In [ ]:
# === Fig 8f / 8g — PC vs IN bootstrap MLE trajectories ===
# Target-specific WT bootstrap MLE trajectories using the WT 2.5 mM / 20 Hz fits.
wt_key_candidates = ['2.5 mM 20 Hz', '2.5 mM 20 Hz (WT)']
wt_key = next((k for k in wt_key_candidates if k in cond_results_E), None)
if wt_key is None:
    wt_key = next((k for k in cond_results_E.keys() if ('2.5 mM 20 Hz' in str(k)) and ('WT' in str(k))), None)
if wt_key is None:
    raise RuntimeError(f"WT 2.5 mM 20 Hz key not found in cond_results_E. Available keys: {list(cond_results_E.keys())}")

target_bootstrap_df = PCA_Data_WT_Pooled_clustered.copy().reset_index(drop=True)
target_bootstrap_df['Target'] = target_bootstrap_df['Target'].astype(str).str.strip().str.upper()
E_P_all = np.asarray(cond_results_E[wt_key]['E_P'], dtype=float)
E_N_all = np.asarray(cond_results_E[wt_key]['E_N'], dtype=float)
n_rows = min(len(target_bootstrap_df), E_P_all.shape[0], E_N_all.shape[0])
target_bootstrap_df = target_bootstrap_df.iloc[:n_rows].copy()
E_P_all = E_P_all[:n_rows]
E_N_all = E_N_all[:n_rows]

target_masks = {
    'PC': target_bootstrap_df['Target'].eq('PC').to_numpy(),
    'IN': target_bootstrap_df['Target'].eq('IN').to_numpy(),
}
target_colors = {'PC': PC_TARGET_COLOR, 'IN': IN_TARGET_COLOR}

meth = 'bootstrap MLE (explicit failures)' if USE_BOOTSTRAP else 'point MLE (explicit failures)'
summary_rows = []

figp, axp_tgt = make_figure_grid(figsize=(6.5, 5))
fign, axn_tgt = make_figure_grid(figsize=(6.5, 5))
for target_label in ['PC', 'IN']:
    mask = target_masks[target_label]
    vals_p = E_P_all[mask]
    vals_n = E_N_all[mask]
    n_fit = int(np.isfinite(vals_n[:, 0]).sum()) if vals_n.ndim == 2 and vals_n.shape[1] else 0
    plot_mean_sem_trace(axp_tgt, x_stim, vals_p, color=target_colors[target_label], label=f'{target_label} ({n_fit})', marker='o', linestyle='-', ms=4, lw=1.4, fill_alpha=0.12)
    plot_mean_sem_trace(axn_tgt, x_stim, vals_n, color=target_colors[target_label], label=f'{target_label} ({n_fit})', marker='o', linestyle='-', ms=4, lw=1.4, fill_alpha=0.12)
    summary_rows.append((target_label, int(mask.sum()), n_fit, np.nanmedian(vals_p[:, 0]), np.nanmedian(vals_p[:, 1]), np.nanmedian(vals_p[:, 9]), np.nanmedian(vals_n[:, 0]), np.nanmedian(vals_n[:, 1]), np.nanmedian(vals_n[:, 9])))

def _format_profile_pvalues(stats_result):
    per_pulse_df = stats_result.get('per_pulse_df', pd.DataFrame()) if isinstance(stats_result, dict) else pd.DataFrame()
    if per_pulse_df.empty:
        return 'none'
    return ', '.join(
        f"P{int(row['Pulse'])}={row['p_value_corrected']:.3g}"
        for _, row in per_pulse_df.iterrows()
        if np.isfinite(row.get('p_value_corrected', np.nan))
    ) or 'none'

stats_p = summarize_profile_group_stats(
    [
        {'label': 'PC', 'matrix': E_P_all[target_masks['PC']]},
        {'label': 'IN', 'matrix': E_P_all[target_masks['IN']]},
    ],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='PC vs IN P trajectories',
    print_summary=False,
)
stats_n = summarize_profile_group_stats(
    [
        {'label': 'PC', 'matrix': E_N_all[target_masks['PC']]},
        {'label': 'IN', 'matrix': E_N_all[target_masks['IN']]},
    ],
    pulse_numbers=x_stim,
    paired_groups=False,
    parametric_stat=False,
    alpha=0.05,
    summary_label='PC vs IN N trajectories',
    print_summary=False,
)

style_ax(axp_tgt, 'Stimulus', '$P_k$', 'PC vs IN bootstrap MLE: P')
axp_tgt.set_ylim(0, 1)
add_legend(axp_tgt, loc='best')
annotate_profile_significance(axp_tgt, stats_p)
out_f = OUTPUT_DIR / 'Fig8_f_pc_in_bootstrap_mle_p_trajectories.pdf'
finalize_figure(figp, title=f'{meth}: PC vs IN P trajectories ({wt_key})', save_path=out_f)

style_ax(axn_tgt, 'Stimulus', '$N_k$', 'PC vs IN bootstrap MLE: N')
axn_tgt.set_ylim(bottom=0)
add_legend(axn_tgt, loc='best')
annotate_profile_significance(axn_tgt, stats_n)
out_g = OUTPUT_DIR / 'Fig8_g_pc_in_bootstrap_mle_n_trajectories.pdf'
finalize_figure(fign, title=f'{meth}: PC vs IN N trajectories ({wt_key})', save_path=out_g)

print('=== PC vs IN BOOTSTRAP MLE TRAJECTORIES ===')
print(f"{'Target':<8} {'n':>4} {'fit':>4} {'P1':>6} {'P2':>6} {'P10':>6} {'N1':>6} {'N2':>6} {'N10':>6}")
for row in summary_rows:
    target_label, n_total, n_fit, p1, p2, p10, n1, n2, n10 = row
    print(f"{target_label:<8} {n_total:>4} {n_fit:>4} {p1:>6.3f} {p2:>6.3f} {p10:>6.3f} {n1:>6.3f} {n2:>6.3f} {n10:>6.3f}")
if isinstance(stats_p, dict):
    print(stats_p.get('summary_line', 'PC vs IN P trajectories: no statistics available.'))
    print(f"PC vs IN P trajectories FDR-corrected p values: {_format_profile_pvalues(stats_p)}")
if isinstance(stats_n, dict):
    print(stats_n.get('summary_line', 'PC vs IN N trajectories: no statistics available.'))
    print(f"PC vs IN N trajectories FDR-corrected p values: {_format_profile_pvalues(stats_n)}")
print(f"✓ Saved PC vs IN P trajectories to {out_f}")
print(f"✓ Saved PC vs IN N trajectories to {out_g}")



#### Additional target-identity follow-up analyses

The cells below retain the within-fiber paired PCA summary and target-composition view as auxiliary follow-up analyses. They help interpret the main target comparison, but they are not the core manuscript panels for Figure 8.


In [ ]:
# === Aux — Target composition across WT clusters ===
# Compare cluster distributions between Purkinje Cells, Interneurons, and overall WT
target_cell_types = PCA_Data_WT_Pooled_clustered['Target'].values
pc_mask = target_cell_types == 'PC'
in_mask = target_cell_types == 'IN'

# Get cluster assignments for each target type
pc_cluster_assignments = cluster_assignments[pc_mask]
in_cluster_assignments = cluster_assignments[in_mask]
wt_cluster_assignments = cluster_assignments  # All WT pooled

# Count clusters for each group
pc_cluster_counts = pd.Series(pc_cluster_assignments).value_counts().sort_index()
in_cluster_counts = pd.Series(in_cluster_assignments).value_counts().sort_index()
wt_cluster_counts = pd.Series(wt_cluster_assignments).value_counts().sort_index()

# Ensure all clusters represented
all_clusters = sorted(range(1, N_CLUSTERS + 1))
pc_complete = pd.Series([pc_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
in_complete = pd.Series([in_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)
wt_complete = pd.Series([wt_cluster_counts.get(c, 0) for c in all_clusters], index=all_clusters)

# Calculate percentages
pc_percentages = 100 * pc_complete / len(pc_cluster_assignments)
in_percentages = 100 * in_complete / len(in_cluster_assignments)
wt_percentages = 100 * wt_complete / len(wt_cluster_assignments)

# Stacked bar plot with 3 bars
fig, ax = make_figure_grid(figsize=(10, 6))
bar_width = 0.6

def get_text_color(rgb):
    r, g, b = rgb[:3]
    return 'white' if 0.2126*r + 0.7152*g + 0.0722*b < 0.55 else 'black'

bottom_pc = bottom_in = bottom_wt = 0
for cluster_id in all_clusters:
    color = get_cluster_color(cluster_id)
    pc_pct = pc_percentages.iloc[cluster_id - 1]
    in_pct = in_percentages.iloc[cluster_id - 1]
    wt_pct = wt_percentages.iloc[cluster_id - 1]

    # PC bar
    ax.bar(0, pc_pct, bar_width, bottom=bottom_pc, color=color,
           label=f'Cluster {cluster_id}' if cluster_id == 1 else None)
    bottom_pc += pc_pct

    # IN bar
    ax.bar(1, in_pct, bar_width, bottom=bottom_in, color=color)
    bottom_in += in_pct
    
    # WT bar
    ax.bar(2, wt_pct, bar_width, bottom=bottom_wt, color=color)
    bottom_wt += wt_pct

ax.set_ylabel('Percentage (%)')
ax.set_title('Cluster Distribution: PC vs IN vs WT Pooled')
ax.set_xticks([0, 1, 2])
ax.set_xticklabels(['Purkinje Cells', 'Interneurons', 'WT Pooled'])
ax.set_ylim(0, 110)

# Legend
handles = [plt.Rectangle((0,0),1,1, color=get_cluster_color(c)) for c in all_clusters]
add_legend(ax, handles, [f'Cluster {c}' for c in all_clusters],
          bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
output_file = OUTPUT_DIR / "Aux_target_cluster_distribution.pdf"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f"✓ Saved cluster distribution to {output_file}")

if '_format_cluster_count_summary' not in globals():
    def _format_cluster_count_summary(label, counts):
        items = [f"C{int(k)}={int(v)}" for k, v in pd.Series(counts).sort_index().items()]
        joined = '  '.join(items) if items else '-'
        print(f"{label}: {joined}")

print("Target cluster counts:")
_format_cluster_count_summary('PC', pc_cluster_counts)
_format_cluster_count_summary('IN', in_cluster_counts)
_format_cluster_count_summary('WT', wt_cluster_counts)
print(f"n: PC={len(pc_cluster_assignments)}  IN={len(in_cluster_assignments)}  WT={len(wt_cluster_assignments)}")


## Section S2 — Supp Fig S2: NNLS method overview

This supplementary methods figure condenses the extraction logic used throughout the notebook into one representative bouton. It shows how the first-event template is obtained, how the NNLS template/jitter search is scored, how amplitudes are read on the Savitzky-Golay trace while being corrected by the NNLS model, and how residual-driven null amplitudes define an empirical noise floor.


In [ ]:
# === Supp S2 — NNLS method overview ===
# Compact methods figure: template fit, NNLS variant grid, SavGol peak measurement, and residual/null-amplitude noise construction.

import os
import subprocess
import sys
from pathlib import Path

from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.signal import savgol_filter

FIGS2_METHOD_PULSE = 3
FIGS2_METHOD_TRACE_XLIM = (0.94, 1.18)
FIGS2_METHOD_PULSE_WINDOW = (-0.010, 0.028)
FIGS2_METHOD_TEMPLATE_XLIM_MS = (-3.0, 35.0)
FIGS2_METHOD_OUTPUT = OUTPUT_DIR / 'FigS2_nnls_method_overview.pdf'
FIGS2_METHOD_NPZ = OUTPUT_DIR / 'FigS2_nnls_method_overlay.npz'


def _figs2_method_overlay_matches(npz_path, condition, file_stem, trial_input_col_1based):
    if not Path(npz_path).exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        required = {
            'condition', 'file_stem', 'time_s', 'train_start_s', 'isi_s',
            'average_yproc', 'average_yhat', 'average_amp_savgol', 'average_amp_savgol_corr',
            'selected_trial_null_amps_nnls', 'amplitudes_3d', 'recut_t_rel', 'recut_avg'
        }
        ok = (
            required.issubset(set(d.files))
            and str(d['condition']) == str(condition)
            and str(d['file_stem']) == str(file_stem)
            and int(d['trial_input_col_1based']) == int(trial_input_col_1based)
        )
        d.close()
        return ok
    except Exception:
        return False


def _figs2_flat_ratio_label(value):
    arr = np.asarray(value, dtype=object).reshape(-1)
    if arr.size == 0:
        return 'n/a'
    first = arr[0]
    if isinstance(first, (tuple, list, np.ndarray)):
        vals = np.asarray(first, float).ravel()
        if vals.size >= 2:
            return f'variant=({vals[0]:.2f}, {vals[1]:.2f})'
        if vals.size == 1:
            return f'variant={vals[0]:.2f}'
        return 'n/a'
    try:
        return f'variant={float(first):.2f}'
    except Exception:
        return str(first)


def _figs2_mad_sigma(values):
    arr = np.asarray(values, float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.nan
    med = float(np.nanmedian(arr))
    mad = float(np.nanmedian(np.abs(arr - med)))
    return 1.4826 * mad


if 'example_condition' in globals() and 'example_file_stem' in globals():
    figs2_condition = str(example_condition)
    figs2_file_stem = str(example_file_stem)
    figs2_trial_1based = int(globals().get('EXAMPLE_LARGE_2P5_TRIAL_INPUT_COL_1BASED', 1))
else:
    if 'example_trials' not in globals():
        example_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
    _candidate_df = PCA_Data_WT_Pooled.sort_values(['AMP1', 'PPR2/1'], ascending=[False, False]).copy()
    figs2_condition = None
    figs2_file_stem = None
    for _, _row in _candidate_df.iterrows():
        _cond = str(_row['Condition'])
        try:
            _stem = _resolve_workbook_stem_local(_cond, _row['ID'], example_trials)
        except Exception:
            continue
        if (BASE_DIR / _cond / f'{_stem}.xlsx').exists():
            figs2_condition = _cond
            figs2_file_stem = _stem
            break
    figs2_trial_1based = 1
    if figs2_condition is None or figs2_file_stem is None:
        raise RuntimeError('Could not resolve a representative WT workbook for Fig S2.')

REPO_ROOT = Path.cwd()
figs2_helper_script = REPO_ROOT / 'Feature_extraction' / 'tmp_selected_recording_nnls_overlay.py'
if not _figs2_method_overlay_matches(FIGS2_METHOD_NPZ, figs2_condition, figs2_file_stem, figs2_trial_1based):
    if not figs2_helper_script.exists():
        raise FileNotFoundError(f'Helper script not found: {figs2_helper_script}')
    env = os.environ.copy()
    env['PYTHONIOENCODING'] = 'utf-8'
    subprocess.run(
        [
            sys.executable,
            str(figs2_helper_script),
            '--condition', figs2_condition,
            '--file', f'{figs2_file_stem}.xlsx',
            '--trial', str(int(figs2_trial_1based)),
            '--out', str(FIGS2_METHOD_NPZ),
        ],
        check=True,
        cwd=str(REPO_ROOT),
        env=env,
    )

overlay_data = np.load(FIGS2_METHOD_NPZ, allow_pickle=True)
overlay_time = np.asarray(overlay_data['time_s'], float)
overlay_train_start = float(np.asarray(overlay_data['train_start_s']).reshape(-1)[0])
overlay_isi = float(np.asarray(overlay_data['isi_s']).reshape(-1)[0])
overlay_plot_time = overlay_time + (1.0 - overlay_train_start)
stim_times = 1.0 + overlay_isi * np.arange(10)

average_yproc = np.asarray(overlay_data['average_yproc'], float)
average_yhat = np.asarray(overlay_data['average_yhat'], float)
average_sg = savgol_filter(average_yproc, 9, 2, mode='interp')
average_amp_savgol = np.asarray(overlay_data['average_amp_savgol'], float)
average_amp_savgol_corr = np.asarray(overlay_data['average_amp_savgol_corr'], float)
average_amp_nnls_corr = np.asarray(overlay_data['average_amp_nnls_corr'], float)

selected_trial_yproc = np.asarray(overlay_data['selected_trial_yproc'], float)
selected_trial_yhat = np.asarray(overlay_data['selected_trial_yhat'], float)
selected_trial_resid = np.asarray(selected_trial_yproc - selected_trial_yhat, float)
selected_trial_null_amps_nnls = np.asarray(overlay_data['selected_trial_null_amps_nnls'], float)
selected_trial_null_amps_nnls = selected_trial_null_amps_nnls[np.isfinite(selected_trial_null_amps_nnls)]

recut_t_rel = np.asarray(overlay_data['recut_t_rel'], float)
recut_avg = np.asarray(overlay_data['recut_avg'], float)
tau_r_ms = float(np.asarray(overlay_data['tau_r_s']).reshape(-1)[0]) * 1000.0
tau_d_vec_ms = np.asarray(overlay_data['tau_d_s'], float) * 1000.0
amplitudes_3d = np.asarray(overlay_data['amplitudes_3d'], float)
jitter_variant_ms = np.asarray(overlay_data['jitter_variant_ms'], float)
dominant_template_ratio = np.asarray(overlay_data['dominant_template_ratio'], dtype=object)
dominant_jitter_ms = np.asarray(overlay_data['dominant_jitter_ms'], float)
overlay_data.close()

pulse_idx = max(0, min(int(FIGS2_METHOD_PULSE) - 1, len(stim_times) - 1))
pulse_time = float(stim_times[pulse_idx])
event_window = (
    (overlay_plot_time >= pulse_time + float(FIGS2_METHOD_PULSE_WINDOW[0]))
    & (overlay_plot_time <= pulse_time + float(FIGS2_METHOD_PULSE_WINDOW[1]))
)
if not np.any(event_window):
    raise RuntimeError('No samples found in the requested pulse window for Fig S2.')

peak_local_idx = int(np.nanargmax(average_sg[event_window]))
peak_time = float(overlay_plot_time[event_window][peak_local_idx])
peak_value = float(average_sg[event_window][peak_local_idx])
amp_sg_raw = float(average_amp_savgol[pulse_idx]) if pulse_idx < len(average_amp_savgol) else np.nan
amp_sg_corr = float(average_amp_savgol_corr[pulse_idx]) if pulse_idx < len(average_amp_savgol_corr) else np.nan
baseline_raw = peak_value - amp_sg_raw if np.isfinite(amp_sg_raw) else np.nan
baseline_corr = peak_value - amp_sg_corr if np.isfinite(amp_sg_corr) else np.nan

fig_s2, axes_s2 = make_figure_grid(2, 2, figsize=(8.2, 6.2))
axes_s2 = np.ravel(axes_s2)
ax_template, ax_grid, ax_peak, ax_noise = axes_s2

first_stim = float(stim_times[0])
template_mask = ((overlay_plot_time - first_stim) * 1000.0 >= float(FIGS2_METHOD_TEMPLATE_XLIM_MS[0])) & ((overlay_plot_time - first_stim) * 1000.0 <= float(FIGS2_METHOD_TEMPLATE_XLIM_MS[1]))
template_fit_t_ms = (overlay_plot_time[template_mask] - first_stim) * 1000.0
template_fit_y = np.asarray(average_yhat[template_mask], float)
if recut_t_rel.size and recut_avg.size:
    recut_mask = (recut_t_rel * 1000.0 >= float(FIGS2_METHOD_TEMPLATE_XLIM_MS[0])) & (recut_t_rel * 1000.0 <= float(FIGS2_METHOD_TEMPLATE_XLIM_MS[1]))
    recut_t_ms = recut_t_rel[recut_mask] * 1000.0
    recut_y = np.asarray(recut_avg[recut_mask], float)
    if np.nanmax(recut_y) > 0:
        recut_y = recut_y / np.nanmax(recut_y)
    if np.nanmax(template_fit_y) > 0:
        template_fit_y = template_fit_y / np.nanmax(template_fit_y)
    ax_template.plot(recut_t_ms, recut_y, color='black', lw=1.7, label='Recut average event')
    ax_template.plot(template_fit_t_ms, template_fit_y, color='#f28e2b', lw=1.6, ls='--', label='Fitted template')
else:
    ax_template.plot(template_fit_t_ms, template_fit_y, color='#f28e2b', lw=1.8, label='Event-1 NNLS fit')
ax_template.axvline(0.0, color='0.6', ls=':', lw=0.8)
ax_template.set_xlabel('Time from stimulus 1 (ms)')
ax_template.set_ylabel('Normalized amplitude')
ax_template.set_title('Template fitting')
ax_template.grid(False)
ax_template.legend(loc='upper right', fontsize=6)
ax_template.text(
    0.03, 0.97,
    f'τrise={tau_r_ms:.2f} ms\nτdecay(1)={tau_d_vec_ms[0]:.2f} ms' if tau_d_vec_ms.size else f'τrise={tau_r_ms:.2f} ms',
    transform=ax_template.transAxes,
    va='top', ha='left', fontsize=6,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.8')
)

if amplitudes_3d.ndim == 3 and amplitudes_3d.shape[0] > pulse_idx and amplitudes_3d.shape[1] and amplitudes_3d.shape[2]:
    search_map = np.asarray(amplitudes_3d[pulse_idx], float)
    im = ax_grid.imshow(
        search_map,
        origin='lower',
        aspect='auto',
        extent=[float(np.nanmin(jitter_variant_ms)), float(np.nanmax(jitter_variant_ms)), 0, search_map.shape[0] - 1],
        cmap='viridis'
    )
    best_flat = int(np.nanargmax(search_map))
    best_ratio_idx, best_jitter_idx = np.unravel_index(best_flat, search_map.shape)
    best_jitter = float(jitter_variant_ms[min(best_jitter_idx, len(jitter_variant_ms) - 1)]) if jitter_variant_ms.size else 0.0
    ax_grid.plot(best_jitter, best_ratio_idx, marker='*', color='red', markersize=10)
    plt.colorbar(im, ax=ax_grid, fraction=0.046, pad=0.04, label='Variant weight')
    ratio_label = _figs2_flat_ratio_label(dominant_template_ratio[pulse_idx:pulse_idx + 1]) if dominant_template_ratio.size else 'n/a'
    jitter_label = float(dominant_jitter_ms[pulse_idx]) if dominant_jitter_ms.size > pulse_idx and np.isfinite(dominant_jitter_ms[pulse_idx]) else best_jitter
    ax_grid.text(
        0.03, 0.97,
        f'Event {pulse_idx + 1}\n{ratio_label}\njitter={jitter_label:+.2f} ms',
        transform=ax_grid.transAxes,
        va='top', ha='left', fontsize=6,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.8')
    )
else:
    ax_grid.text(0.5, 0.5, 'Variant grid unavailable', ha='center', va='center', transform=ax_grid.transAxes)
ax_grid.set_xlabel('Jitter (ms)')
ax_grid.set_ylabel('Template index')
ax_grid.set_title('NNLS search grid')
ax_grid.grid(False)

ax_peak.plot(overlay_plot_time, average_yproc, color='0.80', lw=1.0, label='Average trace')
ax_peak.plot(overlay_plot_time, average_sg, color='#4C78A8', lw=1.8, label='SavGol trace')
ax_peak.plot(overlay_plot_time, average_yhat, color='#f28e2b', lw=1.4, ls='--', label='NNLS fit')
ax_peak.axvline(pulse_time, color='black', ls=':', lw=0.8)
if np.isfinite(baseline_raw):
    ax_peak.hlines(baseline_raw, pulse_time + FIGS2_METHOD_PULSE_WINDOW[0], peak_time, color='#4C78A8', ls=':', lw=1.0)
    ax_peak.vlines(peak_time, baseline_raw, peak_value, color='#4C78A8', lw=1.2)
if np.isfinite(baseline_corr):
    ax_peak.hlines(baseline_corr, pulse_time + FIGS2_METHOD_PULSE_WINDOW[0], peak_time, color='#f28e2b', ls=':', lw=1.0)
    ax_peak.vlines(peak_time + 0.0015, baseline_corr, peak_value, color='#f28e2b', lw=1.2)
ax_peak.set_xlim(pulse_time + FIGS2_METHOD_PULSE_WINDOW[0], pulse_time + FIGS2_METHOD_PULSE_WINDOW[1])
ax_peak.set_xlabel('Time (s)')
ax_peak.set_ylabel('ΔF/F0')
ax_peak.set_title('SavGol peak + NNLS residual correction')
ax_peak.grid(False)
ax_peak.legend(loc='upper right', fontsize=6)
ax_peak.text(
    0.03, 0.97,
    f'A{pulse_idx + 1} SavGol={amp_sg_raw:.3f}\nA{pulse_idx + 1} corrected={amp_sg_corr:.3f}\nA{pulse_idx + 1} NNLS={average_amp_nnls_corr[pulse_idx]:.3f}',
    transform=ax_peak.transAxes,
    va='top', ha='left', fontsize=6,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.8')
)

ax_noise.plot(overlay_plot_time, selected_trial_resid, color='#9467bd', lw=1.1)
ax_noise.axhline(0.0, color='0.6', ls=':', lw=0.8)
ax_noise.axvspan(0.80, 1.00, color='0.93', zorder=0)
ax_noise.set_xlim(*FIGS2_METHOD_TRACE_XLIM)
ax_noise.set_xlabel('Time (s)')
ax_noise.set_ylabel('Residual')
ax_noise.set_title('Residuals build the empirical noise model')
ax_noise.grid(False)
ax_noise.text(
    0.03, 0.97,
    f'Selected trial {int(figs2_trial_1based)}\npre-stim window shaded',
    transform=ax_noise.transAxes,
    va='top', ha='left', fontsize=6,
    bbox=dict(boxstyle='round', facecolor='white', alpha=0.75, edgecolor='0.8')
)
ax_noise_inset = inset_axes(ax_noise, width='42%', height='55%', loc='upper right', borderpad=1.0)
if selected_trial_null_amps_nnls.size:
    nnls_null_sigma = _figs2_mad_sigma(selected_trial_null_amps_nnls)
    nnls_null_median = float(np.nanmedian(selected_trial_null_amps_nnls))
    nnls_null_thr = nnls_null_median + nnls_null_sigma if np.isfinite(nnls_null_sigma) else np.nan
    ax_noise_inset.hist(selected_trial_null_amps_nnls, bins=min(24, max(8, int(np.sqrt(selected_trial_null_amps_nnls.size)) + 1)), color='0.70', edgecolor='0.25')
    if np.isfinite(nnls_null_thr):
        ax_noise_inset.axvline(nnls_null_thr, color='red', ls='--', lw=1.0)
    ax_noise_inset.axvline(nnls_null_median, color='black', ls=':', lw=0.9)
    ax_noise_inset.set_title('Sliding NNLS null amplitudes', fontsize=6)
    ax_noise_inset.tick_params(labelsize=6)
    ax_noise_inset.set_xlabel('A1 null', fontsize=6)
    ax_noise_inset.set_ylabel('n', fontsize=6)
    ax_noise_inset.text(
        0.98, 0.97,
        f'MADσ={nnls_null_sigma:.3f}\nmedian={nnls_null_median:.3f}\nthr≈{nnls_null_thr:.3f}',
        transform=ax_noise_inset.transAxes,
        va='top', ha='right', fontsize=5,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.70, edgecolor='0.8')
    )
else:
    ax_noise_inset.text(0.5, 0.5, 'No null amplitudes', ha='center', va='center', transform=ax_noise_inset.transAxes, fontsize=6)

plt.tight_layout()
fig_s2.savefig(FIGS2_METHOD_OUTPUT, dpi=300, bbox_inches='tight')
plt.show()

print('Fig S2 method summary:')
print(f'  Example bouton: {figs2_condition} / {figs2_file_stem}')
print(f'  Representative pulse for measurement panel: {pulse_idx + 1}')
if amplitudes_3d.ndim == 3:
    print(f'  NNLS variant grid: {amplitudes_3d.shape[1]} template variants × {amplitudes_3d.shape[2]} jitter variants per event')
print('  Peak panel: SavGol local maximum is displayed together with the NNLS model-based correction.')
print('  Noise panel: histogram shows sliding NNLS null amplitudes from the pre-stimulus residual/noise window.')
print(f'  Exported figure: {FIGS2_METHOD_OUTPUT}')


## Section S6 — Supp Fig S6: Stability of glutamate release with time


### Stability control

Stability measurements are placed near the end of the notebook because they validate the persistence of WT bouton properties without redefining the main biological axes. The goal is to show that the major WT features are not explained by recording drift over time.

These controls matter for both the diversity and the mechanistic analyses: if amplitudes, PPR profiles, or PCA positions drift strongly over minutes, then bouton-to-bouton heterogeneity would be harder to interpret as a stable biological property.


#### Stability configuration


In [ ]:
# === Supp S6 — Stability config + helpers ===
# Config + minimal helpers for the retained stability control

ELLIPSE_CTRL_ALPHA = 0.80   # reference ellipse quantile; must stay in (0, 1)
ELLIPSE_CTRL_SCALE = 2.0    # multiplicative slack applied to the chi² threshold for stability scoring

import numpy as np
from scipy.stats import chi2


def build_ellipse_models(X, y, n_clusters, alpha=0.80, ridge=1e-6):
    """Fit one Gaussian ellipse model per cluster and return its chi² threshold."""
    if not (0.0 < float(alpha) < 1.0):
        raise ValueError('Ellipse alpha must be strictly between 0 and 1.')
    thr = chi2.ppf(float(alpha), df=2)
    models = []
    for k in range(1, n_clusters + 1):
        pts = X[y == k]
        if len(pts) > 2:
            mu = pts.mean(axis=0)
            cov = np.cov(pts.T) + np.eye(2) * ridge
            inv = np.linalg.pinv(cov)
            models.append({'cluster': k, 'center': mu, 'cov': cov, 'inv_cov': inv})
    return models, thr


def mahalanobis_sq(x, mu, inv):
    d = x - mu
    return float(d.T @ inv @ d)


#### Stability of mean traces


In [ ]:
# === Supp S6a — Stability mean traces before/after ===
# Compare mean WT traces before and after the stability manipulation.

before_stats = compute_trace_stats(condition_names=STABILITY_BEFORE_CONDITIONS, source=TRACE_MEAN_SOURCE)
after_stats = compute_trace_stats(condition_names=STABILITY_AFTER_CONDITIONS, source=TRACE_MEAN_SOURCE)

fig, ax = make_figure_grid(figsize=(8.0, 5.0))
plot_trace_analysis(
    ax=ax,
    condition_names=STABILITY_BEFORE_CONDITIONS,
    source=TRACE_MEAN_SOURCE,
    display='mean_sem',
    color=STABILITY_BEFORE_COLOR,
    label=f"Before (n={before_stats['n']})",
    linewidth=2.0,
    linestyle='-',
    sem_alpha=0.18,
    zero_line=False,
    style_axis=False,
)
plot_trace_analysis(
    ax=ax,
    condition_names=STABILITY_AFTER_CONDITIONS,
    source=TRACE_MEAN_SOURCE,
    display='mean_sem',
    color=STABILITY_AFTER_COLOR,
    label=f"After (n={after_stats['n']})",
    linewidth=2.0,
    linestyle='-',
    sem_alpha=0.18,
    zero_line=False,
    xlim=TRACE_XLIM_20HZ,
    xlabel='Time (s)',
    ylabel='ΔF/F',
    title='Stability conditions: mean traces (±SEM)',
    legend=True,
    legend_kwargs={'fontsize': 9},
    style_axis=True,
)
ax.axvspan(*TRACE_XLIM_20HZ, color='gray', alpha=0.08)
plt.tight_layout()
out = OUTPUT_DIR / 'FigS6_stability_mean_traces_before_after.pdf'
plt.savefig(out, dpi=300, bbox_inches='tight')
plt.show()
print('Counts:', {'Before': before_stats['n'], 'After': after_stats['n']})
print(f'✓ Saved {out}')


#### Stability of WT plasticity profiles


In [ ]:
# === Supp S6b — Stability PPR profile ===
# Compare PPR profiles before vs after treatment


def plot_stability_ppr_profile():
    """Plot paired stability PPR profile with shared omnibus/profile stats."""
    if 'plot_profile_group_comparison' not in globals():
        raise RuntimeError('Run the shared PPR helper cell first (cell defining plot_profile_group_comparison).')

    ppr_cols = [f'PPR{i}/1' for i in range(2, 11) if f'PPR{i}/1' in PCA_Data_Stability_Before.columns and f'PPR{i}/1' in PCA_Data_Stability_After.columns]
    show_stats = True
    parametric_stat = True

    def _build_paired_stability_profiles(before_df, after_df, ppr_columns):
        before = before_df.copy().reset_index(drop=True)
        after = after_df.copy().reset_index(drop=True)
        idx_before, idx_after, pair_ids = find_valid_pairs(
            before['ID'].astype(str).tolist(),
            after['ID'].astype(str).tolist(),
        )
        if len(pair_ids) == 0:
            raise ValueError('No paired stability boutons found between Before and After.')
        paired_before = before.iloc[idx_before].reset_index(drop=True)
        paired_after = after.iloc[idx_after].reset_index(drop=True)
        before_mat = paired_before[ppr_columns].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
        after_mat = paired_after[ppr_columns].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
        valid_mask = np.isfinite(before_mat).all(axis=1) & np.isfinite(after_mat).all(axis=1)
        before_mat = before_mat[valid_mask]
        after_mat = after_mat[valid_mask]
        pair_info = pd.DataFrame({'PairID': np.asarray(pair_ids, dtype=object)[valid_mask]})
        ones = np.ones((len(pair_info), 1), dtype=float)
        return np.hstack([ones, before_mat]), np.hstack([ones, after_mat]), pair_info

    before_profiles, after_profiles, paired_df = _build_paired_stability_profiles(
        PCA_Data_Stability_Before,
        PCA_Data_Stability_After,
        ppr_cols,
    )
    pulse_numbers = np.arange(1, before_profiles.shape[1] + 1, dtype=float)

    fig, ax = make_figure_grid(figsize=(8, 6))
    stats_result = plot_profile_group_comparison(
        ax,
        [
            {
                'matrix': before_profiles,
                'color': STABILITY_BEFORE_COLOR,
                'marker': 'o',
                'label': f'Before (n={before_profiles.shape[0]})',
                'linewidth': 2.0,
                'sem_alpha': 0.2,
            },
            {
                'matrix': after_profiles,
                'color': STABILITY_AFTER_COLOR,
                'marker': 's',
                'label': f'After (n={after_profiles.shape[0]})',
                'linewidth': 2.0,
                'sem_alpha': 0.2,
            },
        ],
        pulse_numbers=pulse_numbers,
        stats_start_index=1,
        paired_groups=True,
        show_stats=show_stats,
        parametric_stat=parametric_stat,
        summary_label='Stability PPR profile: Before vs After',
        title='PPR Profile: Before vs After',
        ylabel='PPR (A_n/A_1)',
        finalize_kwargs={
            'legend': True,
            'legend_loc': 'best',
            'unity_kwargs': {'color': 'gray', 'linestyle': 'dotted', 'linewidth': 1},
            'ylim': (0, None),
        },
    )
    plt.tight_layout()

    output_file = OUTPUT_DIR / 'FigS6_stability_ppr_profile.pdf'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print(f'Paired stability curves included: {len(paired_df)}')
    return output_file, stats_result


plot_output, stability_ppr_stats = plot_stability_ppr_profile()
print(f'✓ Saved PPR profile to {plot_output}')


#### Paired WT metric comparisons


In [ ]:
# === Supp S6cde — Stability statistical comparisons ===
# Compare paired WT scalar metrics before and after the stability manipulation with summary statistics.
from scipy.stats import wilcoxon, ttest_rel


def _paired_stability_rows(before_df, after_df):
    before = before_df.copy().reset_index(drop=True)
    after = after_df.copy().reset_index(drop=True)
    idx_before, idx_after, pair_ids = find_valid_pairs(
        before['ID'].astype(str).tolist(),
        after['ID'].astype(str).tolist(),
    )
    if len(pair_ids) == 0:
        raise ValueError('No paired stability boutons found between Before and After.')
    paired_before = before.iloc[idx_before].reset_index(drop=True).copy()
    paired_after = after.iloc[idx_after].reset_index(drop=True).copy()
    paired_before['PairID'] = pair_ids
    paired_after['PairID'] = pair_ids
    return paired_before, paired_after


def _pair_stability_metric(before_df, after_df, value_col):
    paired_before, paired_after = _paired_stability_rows(before_df, after_df)
    paired = pd.DataFrame({
        'PairID': paired_before['PairID'],
        f'{value_col}_before': pd.to_numeric(paired_before[value_col], errors='coerce'),
        f'{value_col}_after': pd.to_numeric(paired_after[value_col], errors='coerce'),
    })
    return paired.dropna(subset=[f'{value_col}_before', f'{value_col}_after']).reset_index(drop=True)


def _paired_failure1_metric(before_df, after_df):
    candidates = [
        ('FailRate1', 'Failure1 (%)', 100.0),
        ('%Fail1', 'Failure1 (%)', 1.0),
    ]
    for column, ylabel, scale in candidates:
        if column in before_df.columns and column in after_df.columns:
            paired = _pair_stability_metric(before_df, after_df, column)
            if scale != 1.0 and not paired.empty:
                paired[f'{column}_before'] = paired[f'{column}_before'] * scale
                paired[f'{column}_after'] = paired[f'{column}_after'] * scale
            return paired, column, ylabel
    raise KeyError('No Failure1 column found in stability data. Expected FailRate1 or %Fail1.')


def _paired_test(before_values, after_values, *, parametric=True):
    if parametric:
        stat, p_value = ttest_rel(before_values, after_values, nan_policy='omit')
        return 'paired t-test', stat, p_value
    diffs = np.asarray(after_values, dtype=float) - np.asarray(before_values, dtype=float)
    diffs = diffs[np.isfinite(diffs)]
    if diffs.size == 0 or np.allclose(diffs, 0.0):
        return 'Wilcoxon', 0.0, 1.0
    stat, p_value = wilcoxon(before_values, after_values)
    return 'Wilcoxon', stat, p_value


def plot_stability_comparisons():
    """Create paired before/after scalar comparisons for AMP1, PPR2/1, and Failure1."""

    STAB_GROUP_SPACING = cluster_group_spacing if 'cluster_group_spacing' in globals() else BOXPLOT_GROUP_SPACING_DEFAULT
    STAB_CLUSTER_EQUIV_COUNT = 4
    STAB_BASE_X_PADDING = cluster_x_padding if 'cluster_x_padding' in globals() else BOXPLOT_X_PADDING_DEFAULT
    STAB_X_PADDING = 0.5 * ((STAB_CLUSTER_EQUIV_COUNT - 1) * float(STAB_GROUP_SPACING) + 2.0 * float(STAB_BASE_X_PADDING) - float(STAB_GROUP_SPACING))
    STAB_BOX_WIDTH = min(0.36, max(0.10, 0.55 * float(STAB_GROUP_SPACING)))
    STAB_POSITIONS = np.arange(2, dtype=float) * float(STAB_GROUP_SPACING)

    def _add_paired_lines(ax, before_values, after_values, positions=STAB_POSITIONS, color='gray', alpha=0.35, linewidth=0.9):
        before_arr = np.asarray(before_values, dtype=float)
        after_arr = np.asarray(after_values, dtype=float)
        valid = np.isfinite(before_arr) & np.isfinite(after_arr)
        for before_value, after_value in zip(before_arr[valid], after_arr[valid]):
            ax.plot(positions, [before_value, after_value], color=color, alpha=alpha, linewidth=linewidth, zorder=1)

    metric_specs = [
        ('AMP1', 'AMP1', True),
        ('PPR2/1', 'PPR2/1', True),
    ]
    paired_metrics = []
    for column, ylabel, parametric in metric_specs:
        paired = _pair_stability_metric(PCA_Data_Stability_Before, PCA_Data_Stability_After, column)
        paired_metrics.append((column, ylabel, paired, parametric))

    paired_failure, failure_col, failure_ylabel = _paired_failure1_metric(PCA_Data_Stability_Before, PCA_Data_Stability_After)
    paired_metrics.append((failure_col, failure_ylabel, paired_failure, False))

    fig, axes = make_figure_grid(1, 3, panel_kind='boxplot_compact')
    stat_summaries = []

    for ax, (column, ylabel, paired, parametric) in zip(np.ravel(axes), paired_metrics):
        before_values = paired[f'{column}_before']
        after_values = paired[f'{column}_after']
        plot_boxplot_analysis(
            ax,
            [
                make_box_group('Before', values=before_values, color=STABILITY_BEFORE_COLOR),
                make_box_group('After', values=after_values, color=STABILITY_AFTER_COLOR),
            ],
            paired=True,
            show_points=True,
            show_stats=False,
            parametric_stat=parametric,
            title=None,
            ylabel=ylabel,
            point_alpha=0.5,
            point_size=18,
            group_spacing=STAB_GROUP_SPACING,
            x_padding=STAB_X_PADDING,
            box_width=STAB_BOX_WIDTH,
        )
        _add_paired_lines(ax, before_values, after_values)
        test_name, stat_value, p_value = _paired_test(before_values, after_values, parametric=parametric)
        if 'PPR' in ylabel:
            ax.axhline(1, color='gray', linestyle='dotted', linewidth=1)
            y_top = ax.get_ylim()[1]
            ax.set_ylim(0, y_top)
        else:
            ax.set_ylim(bottom=0)
        add_significance_bar(ax, STAB_POSITIONS[0], STAB_POSITIONS[1], p_value, linewidth=1.0, fontsize=5, y_pad_frac=0.05, h_frac=0.0, text_pad_frac=0.015)
        ax.set_title(f'{ylabel}: Before vs After (paired n={len(paired)})')
        stat_summaries.append((ylabel, test_name, stat_value, p_value, len(paired), float(np.nanmean(before_values)), float(np.nanmean(after_values))))

    plt.tight_layout()
    output_file = OUTPUT_DIR / 'FigS6_stability_statistical_comparisons.pdf'
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    print('=== STABILITY SCALAR COMPARISONS ===')
    for ylabel, test_name, stat_value, p_value, n_pairs, mean_before, mean_after in stat_summaries:
        print(f'{ylabel}: {test_name}, stat={stat_value:.3f}, p={p_value:.4g}, paired n={n_pairs}, mean Before={mean_before:.3f}, mean After={mean_after:.3f}')

    return output_file, stat_summaries


comparison_output = plot_stability_comparisons()
print(f'Comparisons saved to {comparison_output[0]}')


#### Stability ellipse control


In [ ]:
# === Supp S6 — Stability matched vs shuffle ellipses ===
# Stability ellipse control: matched pairs vs random full-dataset pairs

from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse
import numpy as np
import matplotlib.pyplot as plt

if 'pca_data' not in globals() or 'stab_before' not in pca_data or 'stab_after' not in pca_data:
    raise RuntimeError("pca_data['stab_before'] and pca_data['stab_after'] are required.")

ELLIPSE_CTRL_ALPHA = float(globals().get('ELLIPSE_CTRL_ALPHA', globals().get('ELLIPSE_ALPHA', 0.80)))
ELLIPSE_CTRL_SCALE = float(globals().get('ELLIPSE_CTRL_SCALE', 2.0))
if not (0.0 < ELLIPSE_CTRL_ALPHA < 1.0):
    raise ValueError('ELLIPSE_CTRL_ALPHA must be strictly between 0 and 1.')
if ELLIPSE_CTRL_SCALE <= 0.0:
    raise ValueError('ELLIPSE_CTRL_SCALE must be > 0.')

SHUFFLE_SEED = 852
SHUFFLE_EVAL_N = int(globals().get('SHUFFLE_EVAL_N', 1000))
SHUFFLE_BOOTSTRAP_N = int(globals().get('N_BOOTSTRAP', 2000))

n_pairs = int(min(len(pca_data['stab_before']), len(pca_data['stab_after'])))
if n_pairs == 0:
    raise ValueError('No paired stability PCA points available.')

before = np.asarray(pca_data['stab_before'], dtype=float)[:n_pairs]
after = np.asarray(pca_data['stab_after'], dtype=float)[:n_pairs]
total_cloud = np.asarray(pca_coordinates, dtype=float)
if len(total_cloud) < 2:
    raise ValueError('Need at least two WT PCA points to build the shuffle control.')

ellipse_models, thr_hard = build_ellipse_models(
    pca_coordinates,
    cluster_assignments,
    N_CLUSTERS,
    alpha=ELLIPSE_CTRL_ALPHA,
)
thr_score = thr_hard * ELLIPSE_CTRL_SCALE


def _md2_point(x, model):
    return mahalanobis_sq(np.asarray(x, dtype=float), model['center'], model['inv_cov'])


def _cluster_labels(x, thr):
    inside = {m['cluster'] for m in ellipse_models if _md2_point(x, m) <= thr}
    if inside:
        return inside
    distances = [(m['cluster'], np.sqrt(_md2_point(x, m) / thr) - 1.0) for m in ellipse_models]
    return {min(distances, key=lambda item: item[1])[0]}


def _pair_stability_mask(before_mat, after_mat, thr):
    stable = []
    for idx in range(len(before_mat)):
        stable.append(bool(_cluster_labels(before_mat[idx], thr) & _cluster_labels(after_mat[idx], thr)))
    return np.asarray(stable, dtype=bool)


def _draw_ellipse_boundaries(ax):
    for model in ellipse_models:
        color = get_cluster_color(model['cluster'])
        u, s, _ = np.linalg.svd(model['cov'])
        angle = np.degrees(np.arctan2(u[1, 0], u[0, 0]))
        w_score, h_score = 2.0 * np.sqrt(thr_score * s)
        w_hard, h_hard = 2.0 * np.sqrt(thr_hard * s)
        ax.add_patch(Ellipse(model['center'], w_score, h_score, angle=angle, facecolor=color, edgecolor=color, alpha=0.10, lw=0.0))
        ax.add_patch(Ellipse(model['center'], w_hard, h_hard, angle=angle, facecolor='none', edgecolor=color, alpha=1.0, lw=2.0))


def _draw_pair_lines(ax, before_mat, after_mat, stable_mask):
    for idx in range(len(before_mat)):
        color = 'gray' if stable_mask[idx] else 'red'
        alpha = 0.55 if stable_mask[idx] else 0.85
        linewidth = 1.0 if stable_mask[idx] else 1.2
        ax.plot([before_mat[idx, 0], after_mat[idx, 0]], [before_mat[idx, 1], after_mat[idx, 1]], color=color, alpha=alpha, lw=linewidth)


def _draw_mean_vector(ax, before_mat, after_mat, color, label):
    start = np.nanmean(before_mat, axis=0)
    end = np.nanmean(after_mat, axis=0)
    delta = end - start
    ax.scatter([start[0]], [start[1]], s=52, color=color, edgecolors='white', linewidths=0.0, zorder=6)
    ax.annotate('', xy=(end[0], end[1]), xytext=(start[0], start[1]), arrowprops={'arrowstyle': '-|>', 'lw': 2.2, 'color': color, 'mutation_scale': 16}, zorder=6)
    ax.text(end[0], end[1], label, color=color, fontsize=5, fontweight='bold', ha='left', va='bottom')


rng = np.random.default_rng(SHUFFLE_SEED)
shuffle_plot_n = n_pairs
shuffle_eval_pairs = rng.choice(len(total_cloud), size=(SHUFFLE_EVAL_N, 2), replace=True)
shuffle_before_eval = total_cloud[shuffle_eval_pairs[:, 0]]
shuffle_after_eval = total_cloud[shuffle_eval_pairs[:, 1]]
shuffle_plot_pairs = rng.choice(len(total_cloud), size=(shuffle_plot_n, 2), replace=False if (2 * shuffle_plot_n) <= len(total_cloud) else True)
shuffle_before_plot = total_cloud[shuffle_plot_pairs[:, 0]]
shuffle_after_plot = total_cloud[shuffle_plot_pairs[:, 1]]

observed_mask = _pair_stability_mask(before, after, thr_score)
shuffle_mask_eval = _pair_stability_mask(shuffle_before_eval, shuffle_after_eval, thr_score)
shuffle_mask_plot = _pair_stability_mask(shuffle_before_plot, shuffle_after_plot, thr_score)

observed_stable = int(observed_mask.sum())
observed_stable_pct = 100.0 * observed_stable / n_pairs

bootstrap_stabilities = []
for _ in range(SHUFFLE_BOOTSTRAP_N):
    draw_idx = rng.choice(len(total_cloud), size=(SHUFFLE_EVAL_N, 2), replace=True)
    draw_before = total_cloud[draw_idx[:, 0]]
    draw_after = total_cloud[draw_idx[:, 1]]
    bootstrap_stabilities.append(100.0 * np.mean(_pair_stability_mask(draw_before, draw_after, thr_score)))
bootstrap_stabilities = np.asarray(bootstrap_stabilities, dtype=float)
shuffle_stable_pct_mean = float(np.mean(bootstrap_stabilities))
shuffle_stable_pct_ci = (
    float(np.percentile(bootstrap_stabilities, 2.5)),
    float(np.percentile(bootstrap_stabilities, 97.5)),
)

observed_distances = np.linalg.norm(after - before, axis=1)
shuffle_distances = np.linalg.norm(shuffle_after_eval - shuffle_before_eval, axis=1)
all_distances = np.concatenate([observed_distances, shuffle_distances])
dist_x_max = float(np.nanmax(all_distances)) if all_distances.size else 1.0
if not np.isfinite(dist_x_max) or dist_x_max <= 0:
    dist_x_max = 1.0
hist_bin_count = min(40, max(20, int(3 * np.sqrt(len(all_distances)))))
hist_bin_edges = np.linspace(0.0, dist_x_max, hist_bin_count + 1)

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=STABILITY_BEFORE_COLOR, markeredgecolor='k', markersize=8, label='Before'),
    Line2D([0], [0], marker='s', color='w', markerfacecolor=STABILITY_AFTER_COLOR, markeredgecolor='k', markersize=8, label='After'),
    Line2D([0], [0], color='gray', alpha=0.55, lw=1.0, label='Stable pairs'),
    Line2D([0], [0], color='red', alpha=0.85, lw=1.2, label='Unstable pairs'),
    Line2D([0], [0], color='black', lw=2.2, label='Mean displacement'),
]

fig, axes = make_figure_grid(2, 2, figsize=(15, 11))
panel_specs = [
    (
        axes[0, 0],
        f'Matched pairs\nα={ELLIPSE_CTRL_ALPHA:.2f}, threshold ×{ELLIPSE_CTRL_SCALE:.2f} | stable={observed_stable}/{n_pairs} ({observed_stable_pct:.1f}%)',
        before,
        after,
        observed_mask,
    ),
    (
        axes[0, 1],
        f'Random pairs from full WT PCA cloud\n{shuffle_plot_n} plotted; chance estimated on {SHUFFLE_EVAL_N} pairs',
        shuffle_before_plot,
        shuffle_after_plot,
        shuffle_mask_plot,
    ),
]
for ax, title, before_mat, after_mat, stable_mask in panel_specs:
    plot_pca_background(ax, alpha=0.30, s=20, label='WT')
    _draw_ellipse_boundaries(ax)
    _draw_pair_lines(ax, before_mat, after_mat, stable_mask)
    plot_pca_analysis(
        ax,
        overlays=[
            make_pca_group(before_mat, label='Before', color=STABILITY_BEFORE_COLOR, edgecolors='k', linewidths=0.0),
            make_pca_group(after_mat, label='After', color=STABILITY_AFTER_COLOR, marker='s', edgecolors='k', linewidths=0.0),
        ],
        title=None,
        legend=False,
    )
    if ax is axes[0, 0]:
        _draw_mean_vector(ax, before, after, color='black', label='Mean Δ real')
    else:
        _draw_mean_vector(ax, shuffle_before_eval, shuffle_after_eval, color='black', label='Mean Δ bootstrap')
    style_pca_axes(ax, title=title, legend=False)
    add_legend(ax, handles=legend_elements, frameon=False)
    ax.grid(False)

hist_specs = [
    (axes[1, 0], observed_distances, STABILITY_BEFORE_COLOR, 'Matched pair distances'),
    (axes[1, 1], shuffle_distances, STABILITY_AFTER_COLOR, 'Random-pair distances from full WT cloud'),
]
for ax, distances, color, title in hist_specs:
    ax.hist(distances, bins=hist_bin_edges, color=color, alpha=0.75, edgecolor='black')
    ax.axvline(np.mean(distances), color='black', linestyle='--', linewidth=1, label=f'Mean = {np.mean(distances):.2f}')
    ax.set_title(title)
    ax.set_xlabel('Euclidean distance in PCA space')
    ax.set_ylabel('Count')
    ax.set_xlim(0.0, dist_x_max)
    add_legend(ax, frameon=False)
    if 'style_hist_axis' in globals():
        style_hist_axis(ax)
    ax.grid(False)

plt.tight_layout()
out_pca = OUTPUT_DIR / 'FigS6_stability_matched_vs_shuffle_pca.pdf'
plt.savefig(out_pca, dpi=300, bbox_inches='tight')
plt.show()

fig, axes = make_figure_grid(1, 2, figsize=(12, 5.8))
axes[0].pie(
    [observed_stable, n_pairs - observed_stable],
    labels=[f'Stable ({observed_stable})', f'Unstable ({n_pairs - observed_stable})'],
    colors=['#4CAF50', '#FF5722'],
    autopct='%.1f%%',
    startangle=90,
    textprops={'fontsize': 11},
)
axes[0].set_title(f'Matched pairs\n{observed_stable_pct:.1f}% stable')

axes[1].pie(
    [shuffle_stable_pct_mean, 100.0 - shuffle_stable_pct_mean],
    labels=[f'Stable ({shuffle_stable_pct_mean:.1f}%)', f'Unstable ({100.0 - shuffle_stable_pct_mean:.1f}%)'],
    colors=['#4CAF50', '#FF5722'],
    autopct='%.1f%%',
    startangle=90,
    textprops={'fontsize': 11},
)
axes[1].set_title(
    f'Bootstrap chance level\nmean stable = {shuffle_stable_pct_mean:.1f}% | 95% CI [{shuffle_stable_pct_ci[0]:.1f}, {shuffle_stable_pct_ci[1]:.1f}]'
)
plt.tight_layout()
out_pie = OUTPUT_DIR / 'FigS6_stability_matched_vs_shuffle_pies.pdf'
plt.savefig(out_pie, dpi=300, bbox_inches='tight')
plt.show()

print('=== Stability ellipse control ===')
print(f'Matched pairs: {observed_stable}/{n_pairs} stable ({observed_stable_pct:.1f}%)')
print(f'Random full-cloud draw: {int(shuffle_mask_eval.sum())}/{SHUFFLE_EVAL_N} stable ({100.0 * np.mean(shuffle_mask_eval):.1f}%)')
print(f'Chance level from {SHUFFLE_BOOTSTRAP_N} full-cloud draws: {shuffle_stable_pct_mean:.1f}% stable (95% CI [{shuffle_stable_pct_ci[0]:.1f}, {shuffle_stable_pct_ci[1]:.1f}])')
print(f'Ellipse parameters: alpha={ELLIPSE_CTRL_ALPHA:.2f}, threshold scale={ELLIPSE_CTRL_SCALE:.2f}')
print(f'Shuffle seed: {SHUFFLE_SEED}')
print(f'✓ Saved PCA comparison to {out_pca}')
print(f'✓ Saved pie comparison to {out_pie}')


#### Global trace overview across conditions


In [ ]:
# === Supp S6 — Global trace overview (all conditions) ===
# Plot the processed traces across all pooled conditions as a global visual stability control.

condition_names_plot = list(NORM_TRACES_DATAFRAME['Condition'].unique())
n_conditions = len(condition_names_plot)
n_rows = (n_conditions + 4) // 5
fig, axes = make_figure_grid(n_rows, 5, figsize=(20, 4 * n_rows))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for plot_idx, condition_name in enumerate(condition_names_plot):
    if plot_idx >= len(axes):
        break
    ax = axes[plot_idx]
    stats = plot_trace_analysis(ax=ax, condition_names=condition_name, source='normalized', display='mean', show_individuals=True, alignment='grid', color='black', individual_color='gray', linewidth=2.0, individual_alpha=0.2, individual_lw=0.5, zero_line=True, zero_kwargs={'color': 'black', 'linestyle': '--', 'alpha': 0.3}, event_time=1.0, event_kwargs={'color': 'red', 'linestyle': '--', 'alpha': 0.5}, xlim=(0, CROP_END), style_axis=True)
    ax.set_title(f'{condition_name}\n(n={stats["n"]})', fontsize=8, pad=10)
    if plot_idx >= (n_rows - 1) * 5:
        ax.set_xlabel('Time (s)')
    else:
        ax.set_xlabel('')
    if plot_idx % 5 == 0:
        ax.set_ylabel('ΔF/F')
    else:
        ax.set_ylabel('')

for empty_idx in range(n_conditions, len(axes)):
    fig.delaxes(axes[empty_idx])

fig.tight_layout()
fig.suptitle('Preprocessed bouton traces by condition (from extract_metrics)', y=1.02, fontsize=8, fontweight='bold')
output_file = OUTPUT_DIR / 'FigS6_global_trace_overview_all_conditions.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved to {output_file}')
print(f"Time range: {COMMON_TIME[0]:.2f} - {COMMON_TIME[-1]:.2f}s ({len(COMMON_TIME)} points)")
print(f"Conditions: {', '.join(condition_names_plot)}")


## Section S7 — Supp Fig S7: HCPC dendrogram & Random Forest validation


### Hierarchical clustering dendrogram

The dendrogram shows how WT boutons partition when the reduced WT representation is submitted to Ward hierarchical clustering. It is the structural bridge between the continuous PCA space and the discrete class definition used later in the notebook.


In [ ]:
# === Supp S7c — HCPC dendrogram ===
# Create dendrogram visualization of hierarchical clustering

from matplotlib.lines import Line2D

# Reuse consistent cluster colors for dendrogram branches
if 'cluster_hex_colors' not in globals():
    dendrogram_colormap = plt.get_cmap('Set1')
    cluster_hex_colors  = [to_hex(dendrogram_colormap(i)) for i in range(N_CLUSTERS)]
set_link_color_palette(cluster_hex_colors)

# Create sample labels for dendrogram leaves
try:
    sample_labels = PCA_Data_WT_Pooled.index.tolist()
except AttributeError:
    sample_labels = [f'Sample_{i+1}' for i in range(len(pca_coordinates))]

# Calculate clustering threshold for specified number of clusters
clustering_threshold = linkage_matrix[-N_CLUSTERS+1, 2]

# Generate dendrogram plot
fig, ax = make_figure_grid(figsize=(12, 6))
dendrogram(
    linkage_matrix,
    color_threshold=clustering_threshold,
    labels=sample_labels,
    leaf_rotation=90,
    leaf_font_size=8,
    ax=ax,
)

cluster_handles = [
    Line2D([0], [0], color=cluster_hex_colors[cid - 1], lw=2.5, label=f'C{cid}')
    for cid in range(1, N_CLUSTERS + 1)
]
cluster_labels = [f'C{cid}' for cid in range(1, N_CLUSTERS + 1)]

ax.set_title(f'Hierarchical Clustering Dendrogram (k={N_CLUSTERS})')
ax.set_xlabel('Samples')
ax.set_ylabel('Ward Distance')
add_legend(ax, handles=cluster_handles, labels=cluster_labels, loc='upper right')
fig.tight_layout()

# Save dendrogram
output_file = OUTPUT_DIR / "FigS7_c_wt_hcpc_dendrogram.pdf"
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()

# Display clustering information
print(f"Clustering threshold for {N_CLUSTERS} clusters: {clustering_threshold:.3f}")
print(f"Dendrogram branches colored by cluster membership")
print(f"✓ Saved dendrogram to {output_file}")


## Section S8 — Supp Fig S8: Sex distribution control


### Sex distribution control in WT data

Sex is examined here as a descriptive control to verify that the WT reference structure is not trivially explained by cohort composition.


In [ ]:
# === Supp S8 — Sex distribution control ===
# Single panel: full WT_pooled cloud as background, individual M/F boutons on top,
# sex centroids marked with X, Mann-Whitney test on PC1 and PC2.

from scipy.stats import mannwhitneyu

n_pca = len(pca_coordinates)
_features_source = pd.read_csv(BASE_DIR / 'summary.csv')
_features_source.columns = [c.strip() for c in _features_source.columns]
if 'Condition' not in _features_source.columns and 'condition' in _features_source.columns:
    _features_source = _features_source.rename(columns={'condition': 'Condition'})
_features_source['Condition'] = _features_source['Condition'].astype(str).str.strip()
_features_source['ID'] = _features_source['ID'].astype(str).str.strip()

_rebuilt_features = enrich_features_dataframe(make_pooled_features_dataframe(_features_source, CONDITION_POOLS))
sex_plot_df = _rebuilt_features[_rebuilt_features['Condition'] == 'WT_pooled'].copy()
if 'OUTLIER_IDS_HIGH_PC1' in globals() and len(OUTLIER_IDS_HIGH_PC1):
    sex_plot_df = _exclude_ids(sex_plot_df, OUTLIER_IDS_HIGH_PC1)
sex_plot_df = sex_plot_df.reset_index(drop=True)

_sex_path_local = BASE_DIR / SEX_FILENAME
if not _sex_path_local.exists():
    raise FileNotFoundError(f'Sex annotation file not found: {_sex_path_local}')
_sex_df_local = pd.read_csv(_sex_path_local)
_sex_df_local.columns = [c.strip() for c in _sex_df_local.columns]
_id_col_local = _sex_df_local.columns[1]
_sex_col_local = _sex_df_local.columns[2]
_sex_map_local = (_sex_df_local[[_id_col_local, _sex_col_local]]
                  .dropna(subset=[_id_col_local])
                  .drop_duplicates(subset=[_id_col_local])
                  .rename(columns={_id_col_local: 'ID', _sex_col_local: 'Sexe'}))
_sex_map_local['ID'] = _sex_map_local['ID'].astype(str).str.strip()
sex_plot_df = sex_plot_df.drop(columns=['Sexe'], errors='ignore').merge(_sex_map_local, on='ID', how='left')

if len(sex_plot_df) != n_pca:
    raise ValueError(f'WT_pooled rebuilt rows ({len(sex_plot_df)}) do not match PCA rows ({n_pca}).')

sex_data = sex_plot_df['Sexe'].astype('object').fillna('').values
male_mask = sex_data == 'M'
female_mask = sex_data == 'F'
n_m = int(male_mask.sum())
n_f = int(female_mask.sum())
print(f'Male: {n_m}  Female: {n_f}  (other/unknown: {n_pca - n_m - n_f})')

if n_m == 0 or n_f == 0:
    raise ValueError('Sex matching failed: one of the plotted groups is empty.')

coords_m = pca_coordinates[male_mask, :2]
coords_f = pca_coordinates[female_mask, :2]
centroid_m = coords_m.mean(axis=0)
centroid_f = coords_f.mean(axis=0)

_, p_pc1 = mannwhitneyu(coords_m[:, 0], coords_f[:, 0], alternative='two-sided')
_, p_pc2 = mannwhitneyu(coords_m[:, 1], coords_f[:, 1], alternative='two-sided')

fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_pca_analysis(
    ax,
    background={'alpha': 0.35, 'label': 'WT pooled'},
    overlays=[
        make_pca_group(coords_m, label=f'Male (n={n_m})', color='tab:blue', marker='o', alpha=0.55, linewidths=0.0),
        make_pca_group(coords_f, label=f'Female (n={n_f})', color='tab:red', marker='o', alpha=0.55, linewidths=0.0),
        make_pca_group(centroid_m[np.newaxis], label='M centroid', color='tab:blue', marker='X', s=120, alpha=1.0, edgecolors='black', linewidths=0.8, zorder=10),
        make_pca_group(centroid_f[np.newaxis], label='F centroid', color='tab:red', marker='X', s=120, alpha=1.0, edgecolors='black', linewidths=0.8, zorder=10),
    ],
    title=f'WT PCA by sex  |  Mann-Whitney  PC1 p={p_pc1:.3g}  PC2 p={p_pc2:.3g}',
    legend=False,
)
add_legend(ax, frameon=False)

fig.tight_layout()
output_file = OUTPUT_DIR / 'FigS8_wt_pca_by_sex.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {output_file}')
print(f'Mann-Whitney  PC1: p={p_pc1:.4g}   PC2: p={p_pc2:.4g}')
print(f'Male centroid:   PC1={centroid_m[0]:.3f}  PC2={centroid_m[1]:.3f}')
print(f'Female centroid: PC1={centroid_f[0]:.3f}  PC2={centroid_f[1]:.3f}')


## Section S9 — Supp Fig S9: PPR vs A1 & Fail correlations across Ca²⁺


### Calcium correlation structure

We correlate paired-pulse facilitation with amplitude and failure metrics under each calcium condition to test whether changes in release strength and apparent reliability are aligned with changes in short-term plasticity. These scatterplots help determine whether calcium effects can be explained by a single release-probability axis or instead require an additional occupancy/refilling component.


In [ ]:
# === Supp S9 — A1/Fail vs PPR2 across Ca²⁺ ===
# Correlation analysis: AMP1 and %Fail1 vs PPR2/1 across calcium conditions
from scipy.stats import pearsonr, t

def calcium_scatter_analysis(x_param, panel_suffix, y_param='PPR2/1'):
    """Create scatter plot with regression analysis for calcium conditions."""

    # Extract data for both conditions
    low_ca_data  = PCA_Data_WT_Low_Ca[[x_param, y_param]].dropna()
    high_ca_data = PCA_Data_WT_High_Ca[[x_param, y_param]].dropna()

    x_low, y_low   = low_ca_data[x_param].values, low_ca_data[y_param].values
    x_high, y_high = high_ca_data[x_param].values, high_ca_data[y_param].values

    # Calculate separate correlations
    r_low, p_low   = pearsonr(x_low, y_low) if len(x_low) > 1 else (float('nan'), float('nan'))
    r_high, p_high = pearsonr(x_high, y_high) if len(x_high) > 1 else (float('nan'), float('nan'))

    # Pooled analysis
    x_pool = np.concatenate([x_low, x_high])
    y_pool = np.concatenate([y_low, y_high])

    slope = intercept = margin = None
    if len(x_pool) > 2:
        slope, intercept = np.polyfit(x_pool, y_pool, 1)
        r_pool, p_pool = pearsonr(x_pool, y_pool)

        # Simplified CI calculation
        residuals = y_pool - (intercept + slope * x_pool)
        mse = np.sum(residuals**2) / (len(x_pool) - 2)
        se = np.sqrt(mse)

        t_crit = t.ppf(0.975, len(x_pool) - 2)
        margin = t_crit * se
    else:
        r_pool = p_pool = float('nan')

    # Create plot
    make_figure(figsize=(7, 5))
    plt.scatter(x_low, y_low, c=get_wt_ca_color('1.5mM'), alpha=0.7, edgecolor='black',
                label=f'1.5mM Ca (n={len(x_low)})')
    plt.scatter(x_high, y_high, c=get_wt_ca_color('4mM'), alpha=0.7, edgecolor='black',
                label=f'4mM Ca (n={len(x_high)})')

    # Set axis limits first so the fit spans the whole plotted range
    if len(x_pool) > 0:
        if x_param == 'AMP1':
            x_min, x_max = 0, max(3, x_pool.max() * 1.1)
        else:
            x_span = x_pool.max() - x_pool.min()
            pad = max(0.1 * x_span, 0.05)
            x_min, x_max = x_pool.min() - pad, x_pool.max() + pad
        y_max = max(3, y_pool.max() * 1.1)
        plt.xlim(x_min, x_max)
        plt.ylim(0, y_max)

    # Add regression line and confidence band across the full plot width
    if slope is not None:
        x_grid = np.linspace(*plt.gca().get_xlim(), 200)
        y_fit = intercept + slope * x_grid
        plt.plot(x_grid, y_fit, color='black', linewidth=2, label='Pooled regression')
        plt.fill_between(x_grid, y_fit - margin, y_fit + margin,
                         color='black', alpha=0.15, label='95% CI')

    # Format plot
    plt.xlabel(x_param)
    plt.ylabel(y_param)
    plt.title(f'{x_param} vs {y_param}: Calcium Comparison')
    plt.grid(False)
    add_legend(plt.gca(), )
    plt.tight_layout()

    # Save results
    safe_param = x_param.replace('%', 'pct').replace('/', '_')
    output_file = OUTPUT_DIR / f"FigS9_{panel_suffix}_{safe_param}_vs_ppr2_1_across_calcium.pdf"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    plt.show()

    # Save statistics
    stats_file = OUTPUT_DIR / f"FigS9_{panel_suffix}_{safe_param}_vs_ppr2_1_across_calcium_stats.txt"
    with open(stats_file, 'w') as f:
        f.write(f"Correlation: {x_param} vs {y_param}\n")
        f.write(f"1.5mM Ca: r={r_low:.4f}, p={p_low:.6g}, n={len(x_low)}\n")
        f.write(f"4mM Ca: r={r_high:.4f}, p={p_high:.6g}, n={len(x_high)}\n")
        f.write(f"Pooled: r={r_pool:.4f}, p={p_pool:.6g}, n={len(x_pool)}\n")

    # Print statistics instead of writing them inside the figure
    print(f"\n{x_param} vs {y_param}")
    print(f"  1.5mM Ca: r={r_low:.2f}, p={p_low:.2g}, n={len(x_low)}")
    print(f"  4mM Ca:   r={r_high:.2f}, p={p_high:.2g}, n={len(x_high)}")
    print(f"  Pooled:   r={r_pool:.2f}, p={p_pool:.2g}, n={len(x_pool)}")
    print(f"  Saved figure: {output_file}")
    print(f"  Saved stats:  {stats_file}")

    return output_file

# Run both analyses
amp1_output = calcium_scatter_analysis('AMP1', 'a')
fail1_output = calcium_scatter_analysis('%Fail1', 'b')

print(f"\nScatter analyses complete:")
print(f"  AMP1 vs PPR2/1: {amp1_output}")
print(f"  %Fail1 vs PPR2/1: {fail1_output}")


## Section S4 — Control AMP1 / PPR2 vs baseline proxy


In [ ]:
# === Supp S4 — Control AMP1 / PPR2 vs baseline proxy ===
# No true bouton F0 is exported in the current summary tables, so this panel uses
# the per-bouton median of the trial-wise null baseline median as an explicit proxy.
from scipy.stats import pearsonr, t

_s4_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
_s4_trials['Condition'] = _s4_trials['condition'].astype(str).str.strip()
_s4_trials['BaseID'] = _s4_trials['file'].map(_normalize_bouton_id)
_s4_trials['baseline_proxy'] = pd.to_numeric(_s4_trials['baseline_null_median_including_zero'], errors='coerce')

_s4_features = PCA_Data_WT_Pooled.copy()
if 'Condition' not in _s4_features.columns and 'condition' in _s4_features.columns:
    _s4_features['Condition'] = _s4_features['condition'].astype(str).str.strip()
elif 'Condition' in _s4_features.columns:
    _s4_features['Condition'] = _s4_features['Condition'].astype(str).str.strip()
else:
    raise KeyError('Supp S4 requires a Condition/condition column in PCA_Data_WT_Pooled')

if 'BaseID' not in _s4_features.columns:
    _s4_features['BaseID'] = _s4_features['ID'].map(_normalize_bouton_id)

_s4_feature_baseids = set(_s4_features['BaseID'].dropna().astype(str))
_s4_control_conditions = sorted(_s4_features['Condition'].dropna().astype(str).unique())
_s4_trials_subset = _s4_trials[_s4_trials['BaseID'].astype(str).isin(_s4_feature_baseids)].copy()
_s4_baseline = (
    _s4_trials_subset
    .groupby('BaseID', as_index=False)['baseline_proxy']
    .median()
)

_s4_df = _s4_features.merge(_s4_baseline, on='BaseID', how='left')
print(f'Supp S4 feature conditions: {_s4_control_conditions}')
print(f'Supp S4 feature BaseIDs: {len(_s4_feature_baseids)}')
print(f'Supp S4 trial rows matching WT pooled BaseIDs: {len(_s4_trials_subset)}')
print(f'Supp S4 merged rows with baseline proxy: {_s4_df["baseline_proxy"].notna().sum()} / {len(_s4_df)}')
if _s4_df['baseline_proxy'].notna().sum() == 0:
    sample_feature_ids = sorted(list(_s4_feature_baseids))[:5]
    sample_trial_ids = sorted(_s4_trials['BaseID'].dropna().astype(str).unique().tolist())[:5]
    raise ValueError(f'Supp S4 merge produced no baseline proxy values. Sample feature BaseIDs={sample_feature_ids}; sample trial BaseIDs={sample_trial_ids}')

def _plot_s4_scatter(ax, dataframe, x_col, y_col, panel_title, y_label):
    panel_df = dataframe[[x_col, y_col]].copy()
    panel_df[x_col] = pd.to_numeric(panel_df[x_col], errors='coerce')
    panel_df[y_col] = pd.to_numeric(panel_df[y_col], errors='coerce')
    panel_df = panel_df[np.isfinite(panel_df[x_col]) & np.isfinite(panel_df[y_col])]

    x = panel_df[x_col].to_numpy(dtype=float)
    y = panel_df[y_col].to_numpy(dtype=float)

    ax.scatter(x, y, s=22, alpha=0.80, color=get_wt_ca_color('2.5mM'), edgecolors='black', linewidths=0.0)

    slope = intercept = r_val = p_val = margin = np.nan
    if len(panel_df) >= 3 and np.nanstd(x) > 0:
        slope, intercept = np.polyfit(x, y, 1)
        r_val, p_val = pearsonr(x, y)
        y_hat = intercept + slope * x
        resid = y - y_hat
        mse = np.sum(resid**2) / max(len(x) - 2, 1)
        se = np.sqrt(mse)
        t_crit = t.ppf(0.975, len(x) - 2)
        margin = t_crit * se if np.isfinite(t_crit) else np.nan

        x_span = x.max() - x.min()
        x_pad = max(0.08 * x_span, 1e-9)
        x_grid = np.linspace(x.min() - x_pad, x.max() + x_pad, 200)
        y_fit = intercept + slope * x_grid
        ax.plot(x_grid, y_fit, color='black', linewidth=1.8)
        if np.isfinite(margin):
            ax.fill_between(x_grid, y_fit - margin, y_fit + margin, color='black', alpha=0.12)

    style_ax(ax, xlabel='Baseline proxy (median null baseline)', ylabel=y_label, title=panel_title)
    ax.text(
        0.03,
        0.97,
        f'r={r_val:.2f}\np={p_val:.3g}\nn={len(panel_df)}',
        transform=ax.transAxes,
        ha='left',
        va='top',
        fontsize=6,
        bbox=dict(facecolor='white', edgecolor='0.85', alpha=0.85, pad=0.2),
    )
    return {
        'n': int(len(panel_df)),
        'r': float(r_val) if np.isfinite(r_val) else np.nan,
        'p': float(p_val) if np.isfinite(p_val) else np.nan,
        'slope': float(slope) if np.isfinite(slope) else np.nan,
        'intercept': float(intercept) if np.isfinite(intercept) else np.nan,
    }

fig_s4, axes_s4 = make_figure_grid(1, 2, panel_kind='simple', squeeze=False, figsize=(8.0, 3.6))
ax_s4a, ax_s4b = axes_s4[0, 0], axes_s4[0, 1]

s4_stats_amp1 = _plot_s4_scatter(ax_s4a, _s4_df, 'baseline_proxy', 'AMP1', 'Control AMP1 vs baseline proxy', 'AMP1 (ΔF/F)')
s4_stats_ppr = _plot_s4_scatter(ax_s4b, _s4_df, 'baseline_proxy', 'PPR2/1', 'Control PPR2/1 vs baseline proxy', 'PPR2/1')
ax_s4b.axhline(1.0, color='gray', linestyle='--', linewidth=1.0, alpha=0.7)

finalize_figure(
    fig_s4,
    title='Fig S4 — Control bouton metrics vs baseline proxy',
    rect=[0, 0, 1, 0.95],
    tight_layout_kwargs={'pad': 0.25, 'w_pad': 0.35},
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
fig_s4_file = OUTPUT_DIR / 'FigS4_control_amp1_ppr2_1_vs_baseline_proxy.pdf'
fig_s4_stats_file = OUTPUT_DIR / 'FigS4_control_amp1_ppr2_1_vs_baseline_proxy_stats.txt'
fig_s4.savefig(fig_s4_file, dpi=300, bbox_inches='tight')
plt.show()

with open(fig_s4_stats_file, 'w', encoding='utf-8') as f:
    f.write('Fig S4 — WT pooled 2.5mM / 20Hz control boutons\n')
    f.write('Baseline metric: per-bouton median of trial baseline_null_median_including_zero from summary_trials.xlsx\n\n')
    f.write(
        'AMP1 vs baseline proxy: '
        f"r={s4_stats_amp1['r']:.6g}, p={s4_stats_amp1['p']:.6g}, n={s4_stats_amp1['n']}, "
        f"slope={s4_stats_amp1['slope']:.6g}, intercept={s4_stats_amp1['intercept']:.6g}\n"
    )
    f.write(
        'PPR2/1 vs baseline proxy: '
        f"r={s4_stats_ppr['r']:.6g}, p={s4_stats_ppr['p']:.6g}, n={s4_stats_ppr['n']}, "
        f"slope={s4_stats_ppr['slope']:.6g}, intercept={s4_stats_ppr['intercept']:.6g}\n"
    )

print(f'Fig S4 saved: {fig_s4_file}')
print(f'Fig S4 stats saved: {fig_s4_stats_file}')


## Section S10 — Supp Fig S10: Sensor saturation control


### Sensor saturation control

The saturation-control panel checks that the 50 Hz interpretation is not driven by sensor ceiling effects, especially at high calcium. This is essential because the mechanistic conclusions depend on changes in fluorescence remaining proportional to changes in glutamate release.


In [ ]:
# === Supp S10 addendum — Paired 20 Hz / 50 Hz vs saturation recordings ===
# Build paired 20 Hz and, when available, 50 Hz versus high-frequency saturation controls separately for 2.5 and 4 mM.
from scipy.stats import ttest_rel, wilcoxon

SATURATION_WORKBOOK = BASE_DIR / 'Saturation_data.xlsx'
SATURATION_TRACE_SHEETS = {'2.5 mM': '2.5mMCa_traces', '4 mM': '4mMCa_traces'}
SATURATION_AMP_SHEETS = {'2.5 mM': '2.5mMCa_amps', '4 mM': '4mMCa_amps'}
SATURATION_CA_KEYS = {'2.5 mM': '2.5mM', '4 mM': '4mM'}
SATURATION_PANEL_BY_CA_20HZ = {'2.5 mM': 'c', '4 mM': 'd'}
SATURATION_PANEL_BY_CA_50HZ = {'2.5 mM': 'e', '4 mM': 'f'}

saturation_trace_dfs = {
    ca_label: pd.read_excel(SATURATION_WORKBOOK, sheet_name=sheet_name)
    for ca_label, sheet_name in SATURATION_TRACE_SHEETS.items()
}
saturation_amp_dfs = {
    ca_label: pd.read_excel(SATURATION_WORKBOOK, sheet_name=sheet_name)
    for ca_label, sheet_name in SATURATION_AMP_SHEETS.items()
}

_saturation_trace_re = re.compile(
    r'^(?P<date>\d{8})_linescan(?P<linescan>\d+)_(?P<freq>\d+-\d+Hz)_'
    r'(?P<stim_start>\d+)-(?P<n_pulses>\d+)pulses_(?P<ca>2\.5mMCa|4mMCa)_'
    r'(?P<bouton>butt?on\d+)_dF_F0_(?P<kind>trace|time)$'
)
_saturation_amp_re = re.compile(
    r'^(?P<date>\d{8})_linescan(?P<linescan>\d+)_(?P<freq>\d+-\d+Hz)_'
    r'(?P<stim_start>\d+)-(?P<n_pulses>\d+)pulses_(?P<ca>2\.5mMCa|4mMCa)_'
    r'(?P<bouton>butt?on\d+)_dF_F0_Amp$'
)
_regular_freq_re = re.compile(
    r'^(?P<date>\d{8})_linescan(?P<linescan>\d+)_(?P<freq>20Hz|50Hz)_(?P<n_pulses>\d+)pulses_'
    r'(?P<ca>2\.5mMCa|4mMCa)_(?P<bouton>bouton\d+)'
)


def _canon_bouton_name(value):
    return str(value).strip().replace('button', 'bouton')


def _saturation_pair_key(date, linescan, bouton, stim_protocol):
    return (str(date), int(linescan), _canon_bouton_name(bouton), str(stim_protocol))


def _lookup_key(date, linescan, bouton):
    return (str(date), int(linescan), _canon_bouton_name(bouton))


def _parse_saturation_amp_id(record_id, ca_label):
    match = _saturation_amp_re.match(str(record_id).strip())
    if match is None:
        return None
    info = match.groupdict()
    info['ca_label'] = ca_label
    info['linescan'] = int(info['linescan'])
    info['stim_start'] = int(info['stim_start'])
    info['n_pulses'] = int(info['n_pulses'])
    info['bouton'] = _canon_bouton_name(info['bouton'])
    info['stim_protocol'] = f"{info['freq']}_{info['stim_start']}-{info['n_pulses']}pulses"
    info['pair_key'] = _saturation_pair_key(info['date'], info['linescan'], info['bouton'], info['stim_protocol'])
    info['lookup_key'] = _lookup_key(info['date'], info['linescan'], info['bouton'])
    return info


def _parse_saturation_trace_columns(trace_df, ca_label):
    records = []
    columns = {str(col): col for col in trace_df.columns}
    for col in trace_df.columns:
        match = _saturation_trace_re.match(str(col))
        if match is None or match.group('kind') != 'trace':
            continue
        info = match.groupdict()
        time_col_name = str(col).replace('_trace', '_time')
        if time_col_name not in columns:
            continue
        trace = np.asarray(trace_df[col], dtype=float)
        finite_trace = trace[np.isfinite(trace)]
        if finite_trace.size == 0:
            continue
        bouton = _canon_bouton_name(info['bouton'])
        stim_start = int(info['stim_start'])
        n_pulses = int(info['n_pulses'])
        stim_protocol = f"{info['freq']}_{stim_start}-{n_pulses}pulses"
        records.append({
            'ca_label': ca_label,
            'source_column': str(col),
            'time_column': time_col_name,
            'date': info['date'],
            'linescan': int(info['linescan']),
            'freq': info['freq'],
            'stim_start': stim_start,
            'n_pulses': n_pulses,
            'stim_protocol': stim_protocol,
            'bouton': bouton,
            'pair_key': _saturation_pair_key(info['date'], info['linescan'], bouton, stim_protocol),
            'lookup_key': _lookup_key(info['date'], info['linescan'], bouton),
            'trace_max': float(np.nanmax(finite_trace)),
            'trace_min': float(np.nanmin(finite_trace)),
            'trace_mean': float(np.nanmean(finite_trace)),
            'n_samples': int(finite_trace.size),
        })
    return pd.DataFrame(records)


def _build_regular_freq_lookup(freq_target_hz):
    feature_df = globals().get('FEATURES_DATAFRAME_RAW', FEATURES_DATAFRAME).copy()
    records = []
    for _, row in feature_df.iterrows():
        condition = str(row.get('Condition', row.get('condition', ''))).strip()
        meta = condition_meta.get(condition, {})
        ca_norm = _normalize_calcium(meta.get('Ca_mM'))
        freq_hz = meta.get('Freq_Hz', np.nan)
        if ca_norm not in {'2.5mM', '4mM'}:
            continue
        if not np.isfinite(freq_hz) or int(freq_hz) != int(freq_target_hz):
            continue
        bouton_id = str(row.get('ID', '')).strip()
        match = _regular_freq_re.match(bouton_id)
        if match is None:
            continue
        info = match.groupdict()
        ca_label = '2.5 mM' if info['ca'] == '2.5mMCa' else '4 mM'
        records.append({
            'ca_label': ca_label,
            'condition': condition,
            'regular_id': bouton_id,
            'date': info['date'],
            'linescan': int(info['linescan']),
            'freq_label': info['freq'],
            'n_pulses_regular': int(info['n_pulses']),
            'bouton': _canon_bouton_name(info['bouton']),
            'lookup_key': _lookup_key(info['date'], info['linescan'], info['bouton']),
            'amp1_regular': float(row['AMP1']) if pd.notna(row.get('AMP1', np.nan)) else np.nan,
            'amp2_regular': float(row['AMP2']) if pd.notna(row.get('AMP2', np.nan)) else np.nan,
            'amp3_regular': float(row['AMP3']) if pd.notna(row.get('AMP3', np.nan)) else np.nan,
        })
    if not records:
        return pd.DataFrame(columns=['ca_label', 'lookup_key', 'regular_id', 'condition', 'amp1_regular'])
    return pd.DataFrame(records)


def _paired_stats(x_values, y_values):
    x = np.asarray(x_values, dtype=float)
    y = np.asarray(y_values, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    stats = {
        'n': int(x.size),
        'mean_left': float(np.nanmean(x)) if x.size else np.nan,
        'sd_left': float(np.nanstd(x, ddof=1)) if x.size > 1 else np.nan,
        'mean_right': float(np.nanmean(y)) if y.size else np.nan,
        'sd_right': float(np.nanstd(y, ddof=1)) if y.size > 1 else np.nan,
        'mean_delta': float(np.nanmean(y - x)) if x.size else np.nan,
        'sd_delta': float(np.nanstd(y - x, ddof=1)) if x.size > 1 else np.nan,
        't_stat': np.nan,
        'p_ttest': np.nan,
        'w_stat': np.nan,
        'p_wilcoxon': np.nan,
    }
    if x.size >= 2:
        t_stat, p_ttest = ttest_rel(x, y, nan_policy='omit')
        stats['t_stat'] = float(t_stat)
        stats['p_ttest'] = float(p_ttest)
        deltas = y - x
        if np.any(np.abs(deltas) > 0):
            try:
                w_stat, p_wilcoxon = wilcoxon(x, y)
                stats['w_stat'] = float(w_stat)
                stats['p_wilcoxon'] = float(p_wilcoxon)
            except ValueError:
                pass
    return stats


def _format_pvalue(value):
    if not np.isfinite(value):
        return 'nan'
    if value < 1e-4:
        return f"{value:.2e}"
    return f"{value:.4f}"


def _plot_saturation_pairs(pair_df, ca_label, left_col, right_col, left_label, right_label, left_source_col, panel, title, output_stem):
    ca_key = SATURATION_CA_KEYS[ca_label]
    stats = _paired_stats(pair_df[left_col], pair_df[right_col])
    fig, ax = make_figure_grid(1, 1, figsize=(4.7, 4.3))
    x_positions = np.array([0.0, 1.0], dtype=float)
    left = pair_df[left_col].to_numpy(float)
    right = pair_df[right_col].to_numpy(float)
    means = [float(np.nanmean(left)), float(np.nanmean(right))]
    sems = [
        float(np.nanstd(values, ddof=1) / np.sqrt(np.isfinite(values).sum())) if np.isfinite(values).sum() > 1 else 0.0
        for values in (left, right)
    ]
    left_color = get_wt_ca_color(ca_key) if left_label == '20 Hz' else get_50hz_ca_color(ca_key)
    right_color = get_50hz_ca_color(ca_key)
    ax.bar(x_positions, means, yerr=sems, color=[left_color, right_color], edgecolor='black', linewidth=0.8, capsize=3, width=0.62, alpha=0.85)
    for _, row in pair_df.iterrows():
        ax.plot(x_positions, [row[left_col], row[right_col]], color='0.35', lw=0.9, alpha=0.7, zorder=2)
        ax.scatter(x_positions, [row[left_col], row[right_col]], color='0.2', s=16, zorder=3)
    source_counts = pair_df[left_source_col].value_counts(dropna=False).to_dict()
    source_text = ', '.join(f"{key}={int(val)}" for key, val in source_counts.items()) if source_counts else 'none'
    ax.text(
        0.98,
        0.98,
        (
            f"n={stats['n']} pairs\n"
            f"paired t p={_format_pvalue(stats['p_ttest'])}\n"
            f"{left_label} source: {source_text}"
        ),
        transform=ax.transAxes,
        ha='right',
        va='top',
        fontsize=7,
        bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='0.8', alpha=0.9),
    )
    y_max = float(np.nanmax(np.concatenate([left[np.isfinite(left)], right[np.isfinite(right)]]))) if stats['n'] else 1.0
    ax.set_ylim(0, max(1.0, y_max * 1.18))
    ax.set_xticks(x_positions)
    ax.set_xticklabels([left_label, right_label])
    style_ax(ax, xlabel='', ylabel='Amplitude (ΔF/F0)', title=title)
    fig.tight_layout()
    pdf_path = OUTPUT_DIR / f'FigS10{panel}_{output_stem}_{ca_key}.pdf'
    png_path = OUTPUT_DIR / f'FigS10{panel}_{output_stem}_{ca_key}.png'
    fig.savefig(pdf_path, dpi=300, bbox_inches='tight')
    fig.savefig(png_path, dpi=300, bbox_inches='tight')
    plt.show()
    return stats, pdf_path, png_path


saturation_records_df = pd.concat(
    [_parse_saturation_trace_columns(df, ca_label) for ca_label, df in saturation_trace_dfs.items()],
    ignore_index=True,
)
if saturation_records_df.empty:
    raise RuntimeError(f'No trace columns parsed from {SATURATION_WORKBOOK}')

trace_summary_df = (
    saturation_records_df
    .sort_values(['ca_label', 'date', 'linescan', 'bouton', 'stim_protocol'])
    .drop_duplicates(subset=['pair_key'])
    .copy()
)
regular_20hz_df = _build_regular_freq_lookup(20)
regular_50hz_df = _build_regular_freq_lookup(50)

paired_20hz_by_ca = {}
paired_50hz_by_ca = {}
saturation_stats_by_key = {}
saturation_output_paths = {}
parsed_amp_tables = []

for ca_label, amp_df in saturation_amp_dfs.items():
    parsed_amp_records = []
    for _, row in amp_df.iterrows():
        info = _parse_saturation_amp_id(row['ID'], ca_label)
        if info is None:
            continue
        parsed_amp_records.append({
            **info,
            'saturation_id': str(row['ID']).strip(),
            'amp20_workbook': float(row['AMP1']) if pd.notna(row.get('AMP1', np.nan)) else np.nan,
            'amp2_workbook': float(row['AMP2']) if pd.notna(row.get('AMP2', np.nan)) else np.nan,
            'amp3_workbook': float(row['AMP3']) if pd.notna(row.get('AMP3', np.nan)) else np.nan,
            'fmax200': float(row['Fmax']) if pd.notna(row.get('Fmax', np.nan)) else np.nan,
        })
    parsed_amp_df = pd.DataFrame(parsed_amp_records)
    if parsed_amp_df.empty:
        raise RuntimeError(f'No amplitude rows parsed from saturation sheet for {ca_label}')
    parsed_amp_tables.append(parsed_amp_df.copy())

    ca_trace_df = trace_summary_df[trace_summary_df['ca_label'] == ca_label].copy()
    pair_df = parsed_amp_df.merge(
        ca_trace_df[['pair_key', 'source_column', 'time_column', 'trace_max', 'trace_min', 'trace_mean', 'n_samples']],
        on='pair_key',
        how='left',
    )

    ca_regular_20 = regular_20hz_df[regular_20hz_df['ca_label'] == ca_label].copy()
    if not ca_regular_20.empty:
        pair_df = pair_df.merge(
            ca_regular_20[['lookup_key', 'regular_id', 'condition', 'amp1_regular', 'amp2_regular', 'amp3_regular']],
            on='lookup_key',
            how='left',
            suffixes=('', '_20hz'),
        )
    else:
        pair_df['regular_id'] = np.nan
        pair_df['condition'] = np.nan
        pair_df['amp1_regular'] = np.nan
        pair_df['amp2_regular'] = np.nan
        pair_df['amp3_regular'] = np.nan

    ca_regular_50 = regular_50hz_df[regular_50hz_df['ca_label'] == ca_label].copy()
    if not ca_regular_50.empty:
        pair_df = pair_df.merge(
            ca_regular_50[['lookup_key', 'regular_id', 'condition', 'amp1_regular', 'amp2_regular', 'amp3_regular']].rename(columns={
                'regular_id': 'regular_50hz_id',
                'condition': 'condition_50hz',
                'amp1_regular': 'amp1_regular_50hz',
                'amp2_regular': 'amp2_regular_50hz',
                'amp3_regular': 'amp3_regular_50hz',
            }),
            on='lookup_key',
            how='left',
        )
    else:
        pair_df['regular_50hz_id'] = np.nan
        pair_df['condition_50hz'] = np.nan
        pair_df['amp1_regular_50hz'] = np.nan
        pair_df['amp2_regular_50hz'] = np.nan
        pair_df['amp3_regular_50hz'] = np.nan

    pair_df['amp20'] = np.where(np.isfinite(pair_df['amp1_regular']), pair_df['amp1_regular'], pair_df['amp20_workbook'])
    pair_df['amp20_source'] = np.where(np.isfinite(pair_df['amp1_regular']), 'regular_dataset', 'saturation_workbook_amp1')
    pair_df['amp50'] = pair_df['amp1_regular_50hz']
    pair_df['amp50_source'] = pd.Series(np.where(np.isfinite(pair_df['amp1_regular_50hz']), 'regular_dataset', None), index=pair_df.index, dtype=object)
    pair_df['amp200'] = pair_df['fmax200']
    pair_df['trace_max_minus_fmax'] = pair_df['trace_max'] - pair_df['fmax200']
    pair_df['pair_label'] = pair_df.apply(
        lambda row: f"{row['date']}_linescan{int(row['linescan'])}_{row['bouton']}_{row['stim_protocol']}",
        axis=1,
    )

    pair20_df = pair_df[np.isfinite(pair_df['amp20']) & np.isfinite(pair_df['amp200'])].copy()
    pair20_df = pair20_df.sort_values(['date', 'linescan', 'bouton', 'stim_protocol']).reset_index(drop=True)
    if pair20_df.empty:
        raise RuntimeError(f'No valid 20 Hz / 200 Hz pairs assembled for {ca_label}')

    stats20, pdf20, png20 = _plot_saturation_pairs(
        pair20_df,
        ca_label,
        left_col='amp20',
        right_col='amp200',
        left_label='20 Hz',
        right_label='200 Hz sat.',
        left_source_col='amp20_source',
        panel=SATURATION_PANEL_BY_CA_20HZ[ca_label],
        title=f'{ca_label}: paired 20 Hz vs saturation',
        output_stem='paired_20Hz_vs_200Hz',
    )
    paired_20hz_by_ca[ca_label] = pair20_df
    saturation_stats_by_key[(ca_label, '20Hz')] = stats20
    saturation_output_paths[(ca_label, '20Hz')] = {'pdf': pdf20, 'png': png20}

    pair50_df = pair_df[np.isfinite(pair_df['amp50']) & np.isfinite(pair_df['amp200'])].copy()
    pair50_df = pair50_df.sort_values(['date', 'linescan', 'bouton', 'stim_protocol']).reset_index(drop=True)
    if not pair50_df.empty:
        stats50, pdf50, png50 = _plot_saturation_pairs(
            pair50_df,
            ca_label,
            left_col='amp50',
            right_col='amp200',
            left_label='50 Hz',
            right_label='200 Hz sat.',
            left_source_col='amp50_source',
            panel=SATURATION_PANEL_BY_CA_50HZ[ca_label],
            title=f'{ca_label}: paired 50 Hz vs saturation',
            output_stem='paired_50Hz_vs_200Hz',
        )
        paired_50hz_by_ca[ca_label] = pair50_df
        saturation_stats_by_key[(ca_label, '50Hz')] = stats50
        saturation_output_paths[(ca_label, '50Hz')] = {'pdf': pdf50, 'png': png50}

saturation_pair_table = OUTPUT_DIR / 'FigS10_paired_20Hz_50Hz_vs_200Hz_tables.xlsx'
with pd.ExcelWriter(saturation_pair_table) as writer:
    for ca_label, pair_df in paired_20hz_by_ca.items():
        pair_df.to_excel(writer, sheet_name=f"paired20_{SATURATION_CA_KEYS[ca_label]}", index=False)
    for ca_label, pair_df in paired_50hz_by_ca.items():
        pair_df.to_excel(writer, sheet_name=f"paired50_{SATURATION_CA_KEYS[ca_label]}", index=False)
    trace_summary_df.to_excel(writer, sheet_name='parsed_traces', index=False)
    pd.concat(parsed_amp_tables, ignore_index=True).to_excel(writer, sheet_name='parsed_amp_rows', index=False)
    regular_20hz_df.to_excel(writer, sheet_name='regular_20hz_lookup', index=False)
    regular_50hz_df.to_excel(writer, sheet_name='regular_50hz_lookup', index=False)
    pd.DataFrame([
        {'ca_label': ca_label, 'comparison': comparison, **stats}
        for (ca_label, comparison), stats in saturation_stats_by_key.items()
    ]).to_excel(writer, sheet_name='stats_summary', index=False)

for ca_label in ['2.5 mM', '4 mM']:
    pair20_df = paired_20hz_by_ca[ca_label]
    stats20 = saturation_stats_by_key[(ca_label, '20Hz')]
    matched_regular_20 = int(pair20_df['amp20_source'].eq('regular_dataset').sum())
    fallback_workbook_20 = int(pair20_df['amp20_source'].eq('saturation_workbook_amp1').sum())
    print(f"=== {ca_label}: paired 20 Hz vs saturation ===")
    print(
        f"n={stats20['n']} | "
        f"20 Hz mean±SD={stats20['mean_left']:.4f}±{stats20['sd_left']:.4f} | "
        f"200 Hz mean±SD={stats20['mean_right']:.4f}±{stats20['sd_right']:.4f} | "
        f"Δ(200-20) mean±SD={stats20['mean_delta']:.4f}±{stats20['sd_delta']:.4f}"
    )
    print(
        f"paired t-test: t={stats20['t_stat']:.4f}, p={_format_pvalue(stats20['p_ttest'])}; "
        f"Wilcoxon p={_format_pvalue(stats20['p_wilcoxon'])}"
    )
    print(f"20 Hz source counts: regular dataset={matched_regular_20}, workbook AMP1 fallback={fallback_workbook_20}")
    print(pair20_df[['pair_label', 'amp20', 'amp200', 'amp20_source', 'regular_id', 'source_column']].to_string(index=False))
    print(f"Saved figure: {saturation_output_paths[(ca_label, '20Hz')]['pdf']}")
    print()

    pair50_df = paired_50hz_by_ca.get(ca_label, pd.DataFrame())
    if pair50_df.empty:
        print(f"=== {ca_label}: paired 50 Hz vs saturation ===")
        print('No exact 50 Hz pairs found in the regular dataset for this calcium condition.')
        print()
        continue

    stats50 = saturation_stats_by_key[(ca_label, '50Hz')]
    matched_regular_50 = int(pair50_df['amp50_source'].eq('regular_dataset').sum())
    print(f"=== {ca_label}: paired 50 Hz vs saturation ===")
    print(
        f"n={stats50['n']} | "
        f"50 Hz mean±SD={stats50['mean_left']:.4f}±{stats50['sd_left']:.4f} | "
        f"200 Hz mean±SD={stats50['mean_right']:.4f}±{stats50['sd_right']:.4f} | "
        f"Δ(200-50) mean±SD={stats50['mean_delta']:.4f}±{stats50['sd_delta']:.4f}"
    )
    print(
        f"paired t-test: t={stats50['t_stat']:.4f}, p={_format_pvalue(stats50['p_ttest'])}; "
        f"Wilcoxon p={_format_pvalue(stats50['p_wilcoxon'])}"
    )
    print(f"50 Hz source counts: regular dataset={matched_regular_50}")
    print(pair50_df[['pair_label', 'amp50', 'amp200', 'amp50_source', 'regular_50hz_id', 'source_column']].to_string(index=False))
    print(f"Saved figure: {saturation_output_paths[(ca_label, '50Hz')]['pdf']}")
    print()

print(f'Saved paired saturation tables: {saturation_pair_table}')


In [ ]:
# === Supp S10 — 50 Hz saturation diagnostic histograms ===
# Plot the pre-filter 50 Hz saturation diagnostic after the main 50 Hz analyses
trace_source_df = globals().get('NORM_TRACES_DATAFRAME_PRE_50HZ_FILTER', NORM_TRACES_DATAFRAME)
cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
cond_2_5_50hz = get_calcium_conditions('2.5mM', '50Hz')[0]

traces_4_50Hz = trace_source_df[trace_source_df['Condition'] == cond_4_50hz]['Avg']
times_4_50Hz = trace_source_df[trace_source_df['Condition'] == cond_4_50hz]['Time']
traces_2_5_50Hz = trace_source_df[trace_source_df['Condition'] == cond_2_5_50hz]['Avg']
times_2_5_50Hz = trace_source_df[trace_source_df['Condition'] == cond_2_5_50hz]['Time']

max_values_4_50Hz, max_times_4_50Hz = get_max_and_time(traces_4_50Hz, times_4_50Hz)
max_values_2_5_50Hz, max_times_2_5_50Hz = get_max_and_time(traces_2_5_50Hz, times_2_5_50Hz)

all_max_values = max_values_4_50Hz + max_values_2_5_50Hz
bins_max_values = np.linspace(min(all_max_values), max(all_max_values), 50)
all_max_times = max_times_4_50Hz + max_times_2_5_50Hz
bins_max_times = np.linspace(min(all_max_times), max(all_max_times), 50)

fig, axs = make_figure_grid(2, 1)

axs[0].hist(max_values_4_50Hz, bins=bins_max_values, color=get_50hz_ca_color('4mM'), alpha=0.6, edgecolor='black', label=cond_4_50hz)
axs[0].hist(max_values_2_5_50Hz, bins=bins_max_values, color=get_50hz_ca_color('2.5mM'), alpha=0.6, edgecolor='black', label=cond_2_5_50hz)
axs[0].set_title('Histogram of maxima (overlaid)')
axs[0].set_xlabel('Maximum value')
axs[0].set_ylabel('Frequency')
add_legend(axs[0])
if 'style_hist_axis' in globals():
    style_hist_axis(axs[0])

axs[1].hist(max_times_4_50Hz, bins=bins_max_times, color=get_50hz_ca_color('4mM'), alpha=0.6, edgecolor='black', label=cond_4_50hz)
axs[1].hist(max_times_2_5_50Hz, bins=bins_max_times, color=get_50hz_ca_color('2.5mM'), alpha=0.6, edgecolor='black', label=cond_2_5_50hz)
axs[1].set_title('Histogram of max-time values (overlaid)')
axs[1].set_xlabel('Time')
axs[1].set_ylabel('Frequency')
add_legend(axs[1])
if 'style_hist_axis' in globals():
    style_hist_axis(axs[1])

fig.tight_layout()
output_file = OUTPUT_DIR / 'FigS10_50hz_saturation_diagnostic_histograms.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {output_file}')


In [ ]:
# === Supp S10 — 50 Hz 4 mM raw trace grid ===
# Retrieve traces and times for the 4mM 50Hz condition
cond_4_50hz = get_calcium_conditions('4mM', '50Hz')[0]
traces_theo_4_50hz = select_traces(condition_names=cond_4_50hz, source=TRACE_SINGLE_SOURCE)
if TRACE_SINGLE_SOURCE != 'normalized' and cond_4_50hz in EXCEPTIONAL_CONDITIONS:
    traces_theo_4_50hz['Time'] = [np.asarray(t, dtype=float) + EXCEPTIONAL_BASELINE_OFFSET for t in traces_theo_4_50hz['Time']]
traces = traces_theo_4_50hz['Avg']
times = traces_theo_4_50hz['Time']
num_traces = len(traces)
num_cols = 5
num_rows = int(np.ceil(num_traces / num_cols))
fig, axes = make_figure_grid(num_rows, num_cols, figsize=(20, 4 * num_rows))
axes = axes.flatten()
for i, (ax, trace, time) in enumerate(zip(axes, traces, times)):
    ax.plot(time, trace, color=get_50hz_ca_color('4mM'))
    ax.set_xlim(0.8, 1.7)
    ax.set_title(f'Trace {i+1}')
    ax.set_xlabel('Time')
    ax.set_ylabel('Value')
for j in range(i + 1, len(axes)):
    axes[j].axis('off')
fig.tight_layout()
output_file = OUTPUT_DIR / 'FigS10_50hz_4mM_raw_trace_grid.pdf'
fig.savefig(output_file, dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {output_file}')


In [ ]:
# === Supp S10 addendum — Shared stimulation-failure events by fiber ===
# Reviewer control: if a stimulus itself fails, the same trial/stimulus event
# should fail in every bouton recorded from that fiber segment.

STIM_FAILURE_GROUPS = OrderedDict([
    ('1.5 mM', get_calcium_conditions('1.5mM', 'all')),
    ('2.5 mM', get_calcium_conditions('2.5mM', 'all')),
    ('4 mM', get_calcium_conditions('4mM', 'all')),
    ('SynII', get_synapsin_conditions()),
])
STIM_FAILURE_GROUP_COLORS = OrderedDict([
    ('1.5 mM', get_wt_ca_color('1.5mM')),
    ('2.5 mM', get_wt_ca_color('2.5mM')),
    ('4 mM', get_wt_ca_color('4mM')),
    ('SynII', SYNII_COLOR),
])
STIM_FAILURE_CONDITIONS = _ordered_unique([cond for group in STIM_FAILURE_GROUPS.values() for cond in group])
STIM_FAILURE_PULSES = list(range(1, int(globals().get('N_STIM', 10)) + 1))
MIN_BOUTONS_PER_FIBER_STIM_FAILURE = 2
STIM_FAILURE_THRESHOLD_COL = 'thr_shared'
STIM_FAILURE_NULL_BOOT = 5000
STIM_FAILURE_NULL_SEED = 42
STIM_FAILURE_NULL_WORKERS = min(20, (__import__('os').cpu_count() or 1))
STIM_FAILURE_NULL_CHUNK = 250


def _extract_recording_segment_id_for_stim_failure(bouton_id):
    """Fiber/recording segment for trial-synchronous stimulation-failure checks.
    PortionA/B are intentionally kept separate because trial identities are only
    comparable within one simultaneously imaged recording segment.
    """
    s = str(bouton_id).strip().replace('_traces_converted', '')
    m = re.match(r'^(.*?)(?:[_\s]*Bouton[_\s]*\d+.*|[_\s]*bouton\d+.*)$', s, flags=re.IGNORECASE)
    return m.group(1).rstrip('_ ') if m else s


def _trial_id_column_for_stim_failure(trials_df):
    for col in ('trial_input_col_1based', 'trial', 'Trial'):
        if col in trials_df.columns:
            return col
    raise KeyError('Could not find a trial identifier column in the trial table.')


def build_shared_stimulation_failure_event_table(
    trials_df,
    conditions,
    *,
    min_boutons_per_fiber=MIN_BOUTONS_PER_FIBER_STIM_FAILURE,
    threshold_col=STIM_FAILURE_THRESHOLD_COL,
):
    d = trials_df[trials_df['condition'].isin(list(conditions))].copy()
    if d.empty:
        raise RuntimeError(f'No trial rows found for conditions: {conditions}')
    trial_col = _trial_id_column_for_stim_failure(d)
    d['BoutonID'] = d['file'].map(_normalize_bouton_id)
    d['StimFiberID'] = d['file'].map(_extract_recording_segment_id_for_stim_failure)
    d['TrialID'] = d[trial_col]

    fiber_bouton_counts = (
        d.groupby(['condition', 'StimFiberID'])['BoutonID']
        .nunique()
        .rename('ExpectedBoutons')
        .reset_index()
    )
    eligible = fiber_bouton_counts[fiber_bouton_counts['ExpectedBoutons'] >= int(min_boutons_per_fiber)].copy()
    if eligible.empty:
        raise RuntimeError(f'No fiber segments with at least {min_boutons_per_fiber} boutons were found.')

    d = d.merge(eligible, on=['condition', 'StimFiberID'], how='inner')
    event_rows = []
    for _, row in d.iterrows():
        fail_mask, _ = get_failure_mask_row(row, threshold_col=threshold_col)
        for pulse_num in STIM_FAILURE_PULSES:
            k = int(pulse_num) - 1
            if k >= len(fail_mask) or not np.isfinite(fail_mask[k]):
                continue
            event_rows.append({
                'condition': row['condition'],
                'StimFiberID': row['StimFiberID'],
                'TrialID': row['TrialID'],
                'Stim': int(pulse_num),
                'BoutonID': row['BoutonID'],
                'ExpectedBoutons': int(row['ExpectedBoutons']),
                'BoutonFailed': int(fail_mask[k] > 0),
            })

    event_bouton_df = pd.DataFrame(event_rows)
    if event_bouton_df.empty:
        raise RuntimeError('No analyzable bouton-level trial/stimulus events were found.')

    event_summary = (
        event_bouton_df
        .groupby(['condition', 'StimFiberID', 'TrialID', 'Stim', 'ExpectedBoutons'], as_index=False)
        .agg(
            ObservedBoutons=('BoutonID', 'nunique'),
            FailedBoutons=('BoutonFailed', 'sum'),
        )
    )
    event_summary['CompleteFiberEvent'] = event_summary['ObservedBoutons'] == event_summary['ExpectedBoutons']
    event_summary['AllBoutonsFailed'] = event_summary['CompleteFiberEvent'] & (event_summary['FailedBoutons'] == event_summary['ExpectedBoutons'])
    event_summary['NoBoutonsFailed'] = event_summary['CompleteFiberEvent'] & (event_summary['FailedBoutons'] == 0)
    event_summary['PartialBoutonFailure'] = event_summary['CompleteFiberEvent'] & (event_summary['FailedBoutons'] > 0) & (event_summary['FailedBoutons'] < event_summary['ExpectedBoutons'])
    return event_bouton_df, event_summary, eligible


stim_trials = pd.read_excel(BASE_DIR / PPR_TRIALS_FILENAME)
stim_event_bouton_df, stim_event_summary_df, stim_eligible_fibers_df = build_shared_stimulation_failure_event_table(
    stim_trials,
    STIM_FAILURE_CONDITIONS,
)

stim_complete_events = stim_event_summary_df[stim_event_summary_df['CompleteFiberEvent']].copy()
stim_incomplete_events = stim_event_summary_df[~stim_event_summary_df['CompleteFiberEvent']].copy()
if stim_complete_events.empty:
    raise RuntimeError('No complete fiber/trial/stimulus events were available after requiring every bouton in the fiber segment.')

condition_to_stim_group = {}
for group_label, group_conditions in STIM_FAILURE_GROUPS.items():
    for condition_name in group_conditions:
        condition_to_stim_group[str(condition_name)] = str(group_label)

stim_event_bouton_df['StimGroup'] = stim_event_bouton_df['condition'].astype(str).map(condition_to_stim_group)
stim_event_summary_df['StimGroup'] = stim_event_summary_df['condition'].astype(str).map(condition_to_stim_group)
stim_complete_events['StimGroup'] = stim_complete_events['condition'].astype(str).map(condition_to_stim_group)

n_all_failed = int(stim_complete_events['AllBoutonsFailed'].sum())
n_not_all_failed = int(len(stim_complete_events) - n_all_failed)
n_none_failed = int(stim_complete_events['NoBoutonsFailed'].sum())
n_partial_failed = int(stim_complete_events['PartialBoutonFailure'].sum())
all_failed_pct = 100.0 * n_all_failed / len(stim_complete_events)

fig, ax = make_figure_grid(1, 1, panel_kind='simple')
pie_values = [n_all_failed, n_not_all_failed]
pie_labels = ['All boutons failed', 'Not all boutons failed']
pie_colors = ['#c43c39', '#b8b8b8']
ax.pie(
    pie_values,
    labels=pie_labels,
    colors=pie_colors,
    autopct=lambda pct: f'{pct:.1f}%' if pct > 0 else '',
    startangle=90,
    counterclock=False,
    wedgeprops={'edgecolor': 'white', 'linewidth': 0.7},
    textprops={'fontsize': clamp_fontsize(7.0)},
)
ax.set_title('Fiber-wide trial/stim failures')
ax.text(
    0.5,
    -0.10,
    f'n={len(stim_complete_events)} complete fiber-events',
    transform=ax.transAxes,
    ha='center',
    va='top',
    fontsize=clamp_fontsize(6.0),
)

out_stim_failure = OUTPUT_DIR / 'FigS10b_shared_stimulation_failure_events_pie.pdf'
finalize_figure(fig, title='Shared stimulation-failure event control', save_path=out_stim_failure)

stim_by_condition = (
    stim_complete_events
    .groupby('condition')
    .agg(
        complete_events=('AllBoutonsFailed', 'size'),
        all_boutons_failed=('AllBoutonsFailed', 'sum'),
        partial_bouton_failure=('PartialBoutonFailure', 'sum'),
        no_bouton_failure=('NoBoutonsFailed', 'sum'),
        fiber_segments=('StimFiberID', 'nunique'),
    )
    .reset_index()
)
stim_by_condition['all_boutons_failed_pct'] = 100.0 * stim_by_condition['all_boutons_failed'] / stim_by_condition['complete_events']

stim_group_rows = []
for group_label, group_conditions in STIM_FAILURE_GROUPS.items():
    group_events = stim_complete_events[stim_complete_events['condition'].isin(list(group_conditions))].copy()
    n_group_events = int(len(group_events))
    n_group_all_failed = int(group_events['AllBoutonsFailed'].sum()) if n_group_events else 0
    n_group_partial = int(group_events['PartialBoutonFailure'].sum()) if n_group_events else 0
    n_group_none = int(group_events['NoBoutonsFailed'].sum()) if n_group_events else 0
    stim_group_rows.append({
        'group': group_label,
        'conditions': ', '.join(map(str, group_conditions)),
        'complete_events': n_group_events,
        'all_boutons_failed': n_group_all_failed,
        'partial_bouton_failure': n_group_partial,
        'no_bouton_failure': n_group_none,
        'all_boutons_failed_pct': (100.0 * n_group_all_failed / n_group_events) if n_group_events else np.nan,
        'fiber_segments': int(group_events['StimFiberID'].nunique()) if n_group_events else 0,
    })
stim_by_group = pd.DataFrame(stim_group_rows)


def build_size_matched_stim_failure_null(event_bouton_df, complete_events, *, n_boot=STIM_FAILURE_NULL_BOOT, seed=STIM_FAILURE_NULL_SEED):
    """Expected all-bouton failures preserving exact condition, stimulus number, and fiber size."""
    pool_df = event_bouton_df.merge(
        complete_events[['condition', 'StimFiberID', 'TrialID', 'Stim']],
        on=['condition', 'StimFiberID', 'TrialID', 'Stim'],
        how='inner',
    ).copy()
    pool_df = pool_df[np.isfinite(pool_df['BoutonFailed']) & pool_df['StimGroup'].notna()].copy()
    eval_events = complete_events[complete_events['StimGroup'].notna()].copy()
    rate_df = (
        pool_df
        .groupby(['StimGroup', 'condition', 'Stim'], as_index=False)
        .agg(
            bouton_events=('BoutonFailed', 'size'),
            failure_rate=('BoutonFailed', 'mean'),
        )
    )
    rate_lookup = {
        (str(row['condition']), int(row['Stim'])): float(row['failure_rate'])
        for _, row in rate_df.iterrows()
    }
    sample_lookup = {
        (str(condition), int(stim)): sub['BoutonFailed'].to_numpy(dtype=int)
        for (condition, stim), sub in pool_df.groupby(['condition', 'Stim'])
    }

    expected_by_group = {str(group): 0.0 for group in STIM_FAILURE_GROUPS.keys()}
    observed_by_group = {str(group): 0 for group in STIM_FAILURE_GROUPS.keys()}
    n_events_by_group = {str(group): 0 for group in STIM_FAILURE_GROUPS.keys()}
    evaluable_rows = []
    for _, event in eval_events.iterrows():
        key = (str(event['condition']), int(event['Stim']))
        p_fail = rate_lookup.get(key, np.nan)
        n_boutons = int(event['ExpectedBoutons'])
        if not np.isfinite(p_fail) or n_boutons <= 0:
            continue
        expected_prob = float(p_fail ** n_boutons)
        group = str(event['StimGroup'])
        expected_by_group[group] = expected_by_group.get(group, 0.0) + expected_prob
        observed_by_group[group] = observed_by_group.get(group, 0) + int(bool(event['AllBoutonsFailed']))
        n_events_by_group[group] = n_events_by_group.get(group, 0) + 1
        evaluable_rows.append({
            'StimGroup': group,
            'condition': str(event['condition']),
            'Stim': int(event['Stim']),
            'ExpectedBoutons': n_boutons,
            'AllBoutonsFailed': int(bool(event['AllBoutonsFailed'])),
            'p_fail_matched': p_fail,
            'expected_all_failed_prob': expected_prob,
        })

    event_null_df = pd.DataFrame(evaluable_rows)
    group_labels = [str(group) for group in STIM_FAILURE_GROUPS.keys()]
    group_index = {group: idx for idx, group in enumerate(group_labels)}
    if event_null_df.empty:
        boot_matrix = np.zeros((int(n_boot), len(group_labels)), dtype=float)
    else:
        event_probs = event_null_df['expected_all_failed_prob'].to_numpy(dtype=float)
        event_groups = event_null_df['StimGroup'].map(group_index).to_numpy(dtype=int)
        n_groups = len(group_labels)
        n_boot = int(n_boot)
        chunk_size = max(1, int(globals().get('STIM_FAILURE_NULL_CHUNK', 250)))
        n_workers = max(1, int(globals().get('STIM_FAILURE_NULL_WORKERS', 1)))
        seed_seq = np.random.SeedSequence(seed)
        n_chunks = int(np.ceil(n_boot / chunk_size))
        child_seeds = seed_seq.spawn(n_chunks)

        def _simulate_null_chunk(chunk_idx):
            start = chunk_idx * chunk_size
            stop = min(start + chunk_size, n_boot)
            size = stop - start
            rng_local = np.random.default_rng(child_seeds[chunk_idx])
            draws = rng_local.random((size, len(event_probs))) < event_probs[None, :]
            out = np.zeros((size, n_groups), dtype=float)
            for group_idx in range(n_groups):
                mask = event_groups == group_idx
                if np.any(mask):
                    out[:, group_idx] = draws[:, mask].sum(axis=1)
            return start, out

        boot_matrix = np.zeros((n_boot, n_groups), dtype=float)
        if n_workers > 1 and n_chunks > 1:
            from concurrent.futures import ThreadPoolExecutor
            with ThreadPoolExecutor(max_workers=min(n_workers, n_chunks)) as pool:
                for start, chunk_out in pool.map(_simulate_null_chunk, range(n_chunks)):
                    boot_matrix[start:start + len(chunk_out), :] = chunk_out
        else:
            for chunk_idx in range(n_chunks):
                start, chunk_out = _simulate_null_chunk(chunk_idx)
                boot_matrix[start:start + len(chunk_out), :] = chunk_out

    boot_counts = {group: boot_matrix[:, idx] for group, idx in group_index.items()}
    total_boot = boot_matrix.sum(axis=1)

    rows = []
    for group in STIM_FAILURE_GROUPS.keys():
        group = str(group)
        boot = boot_counts.get(group, np.array([], dtype=float))
        observed = int(observed_by_group.get(group, 0))
        expected = float(expected_by_group.get(group, 0.0))
        if boot.size:
            p_ge = float((np.sum(boot >= observed) + 1) / (boot.size + 1))
            null_mean = float(np.mean(boot))
            null_lo, null_hi = np.percentile(boot, [2.5, 97.5])
        else:
            p_ge = null_mean = null_lo = null_hi = np.nan
        rows.append({
            'group': group,
            'complete_events': int(n_events_by_group.get(group, 0)),
            'observed_all_failed': observed,
            'expected_all_failed_analytic': expected,
            'expected_all_failed_boot_mean': null_mean,
            'null_95ci_low': float(null_lo) if np.isfinite(null_lo) else np.nan,
            'null_95ci_high': float(null_hi) if np.isfinite(null_hi) else np.nan,
            'observed_minus_expected': float(observed - expected),
            'upper_tail_p_observed_ge_null': p_ge,
        })

    observed_total = int(sum(observed_by_group.values()))
    expected_total = float(sum(expected_by_group.values()))
    total_summary = {
        'observed_total': observed_total,
        'expected_total_analytic': expected_total,
        'expected_total_boot_mean': float(np.mean(total_boot)) if total_boot.size else np.nan,
        'null_95ci_low': float(np.percentile(total_boot, 2.5)) if total_boot.size else np.nan,
        'null_95ci_high': float(np.percentile(total_boot, 97.5)) if total_boot.size else np.nan,
        'upper_tail_p_observed_ge_null': float((np.sum(total_boot >= observed_total) + 1) / (total_boot.size + 1)) if total_boot.size else np.nan,
        'n_boot': int(n_boot),
    }
    return pd.DataFrame(rows), rate_df, pd.DataFrame(evaluable_rows), total_summary


stim_null_by_group, stim_failure_rate_by_group_stim, stim_null_event_table, stim_null_total = build_size_matched_stim_failure_null(
    stim_event_bouton_df,
    stim_complete_events,
)

fig_group, axes_group = make_figure_grid(1, len(STIM_FAILURE_GROUPS), panel_kind='simple')
axes_group = np.asarray(axes_group).reshape(-1)
for ax_g, row_g in zip(axes_group, stim_group_rows):
    values_g = [int(row_g['all_boutons_failed']), int(row_g['complete_events'] - row_g['all_boutons_failed'])]
    if int(row_g['complete_events']) == 0:
        ax_g.text(0.5, 0.5, 'No complete\nevents', ha='center', va='center', transform=ax_g.transAxes, fontsize=clamp_fontsize(7.0))
        ax_g.set_title(str(row_g['group']))
        ax_g.axis('off')
        continue
    group_color = STIM_FAILURE_GROUP_COLORS.get(str(row_g['group']), pie_colors[0])
    ax_g.pie(
        values_g,
        labels=None,
        colors=[group_color, pie_colors[1]],
        autopct=lambda pct: f'{pct:.1f}%' if pct > 0 else '',
        startangle=90,
        counterclock=False,
        wedgeprops={'edgecolor': 'white', 'linewidth': 0.7},
        textprops={'fontsize': clamp_fontsize(6.0)},
    )
    ax_g.set_title(f"{row_g['group']}\nn={int(row_g['complete_events'])}")
group_legend_handles = [
    plt.Line2D([0], [0], marker='o', color='none', markerfacecolor='0.35', markersize=6, label='All boutons failed (group color)'),
    plt.Line2D([0], [0], marker='o', color='none', markerfacecolor=pie_colors[1], markersize=6, label='Not all boutons failed'),
]
fig_group.legend(handles=group_legend_handles, loc='lower center', ncol=2, frameon=False, fontsize=clamp_fontsize(6.0))
out_stim_failure_groups = OUTPUT_DIR / 'FigS10b_shared_stimulation_failure_events_by_group_pies.pdf'
finalize_figure(
    fig_group,
    title='Shared stimulation-failure events by condition group',
    rect=[0, 0.12, 1, 0.92],
    save_path=out_stim_failure_groups,
)

fig_null, ax_null = make_figure_grid(1, 1, panel_kind='simple')
x_null = np.arange(len(stim_null_by_group), dtype=float)
obs_null = stim_null_by_group['observed_all_failed'].to_numpy(dtype=float)
exp_null = stim_null_by_group['expected_all_failed_boot_mean'].to_numpy(dtype=float)
lo_null = stim_null_by_group['null_95ci_low'].to_numpy(dtype=float)
hi_null = stim_null_by_group['null_95ci_high'].to_numpy(dtype=float)
group_colors_null = [STIM_FAILURE_GROUP_COLORS.get(str(g), '0.35') for g in stim_null_by_group['group']]
ax_null.bar(x_null, exp_null, color='0.82', edgecolor='0.35', linewidth=0.6, label='size-matched null')
ax_null.errorbar(x_null, exp_null, yerr=[exp_null - lo_null, hi_null - exp_null], fmt='none', ecolor='0.25', elinewidth=0.8, capsize=2)
ax_null.scatter(x_null, obs_null, c=group_colors_null, s=28, edgecolors='black', linewidths=0.4, zorder=3, label='observed')
ax_null.set_xticks(x_null)
ax_null.set_xticklabels(stim_null_by_group['group'].astype(str).tolist(), rotation=0)
ax_null.set_ylabel('All-bouton failed events')
ax_null.set_title('Observed vs matched null')
if '_apply_clean_axes_style' in globals():
    _apply_clean_axes_style(ax_null, panel_kind='simple')
add_legend(ax_null, loc='best')
out_stim_failure_null = OUTPUT_DIR / 'FigS10b_shared_stimulation_failure_events_matched_null.pdf'
finalize_figure(fig_null, title='Shared stimulation-failure matched null', save_path=out_stim_failure_null)

print('=== Shared stimulation-failure event control ===')
print(f"Conditions: {', '.join(map(str, STIM_FAILURE_CONDITIONS))}")
print(f"Events included: all listed stimuli per trial ({', '.join('A' + str(p) for p in STIM_FAILURE_PULSES)}), not only A1.")
print(f"Matched-null simulation: {STIM_FAILURE_NULL_BOOT} draws, chunk={STIM_FAILURE_NULL_CHUNK}, workers={STIM_FAILURE_NULL_WORKERS}")
print('Fiber key: recording segment; PortionA/B are kept separate for trial alignment.')
print(f"Eligible fiber segments: {len(stim_eligible_fibers_df)} with >= {MIN_BOUTONS_PER_FIBER_STIM_FAILURE} boutons")
print(f"Complete fiber/trial/stim events: {len(stim_complete_events)}")
print(f"Incomplete events excluded: {len(stim_incomplete_events)}")
print(f"All-bouton failed events: {n_all_failed} / {len(stim_complete_events)} ({all_failed_pct:.3f}%)")
print(f"Partial-bouton failure events: {n_partial_failed} / {len(stim_complete_events)} ({100.0 * n_partial_failed / len(stim_complete_events):.3f}%)")
print(f"No-bouton failure events: {n_none_failed} / {len(stim_complete_events)} ({100.0 * n_none_failed / len(stim_complete_events):.3f}%)")
print('\nBy condition:')
print(stim_by_condition.to_string(index=False, formatters={'all_boutons_failed_pct': lambda v: f'{v:.3f}%'}))
print('\nBy group, pooling 20 Hz and 50 Hz where available:')
print(stim_by_group.to_string(index=False, formatters={'all_boutons_failed_pct': lambda v: 'nan' if not np.isfinite(v) else f'{v:.3f}%'}))
print('\nSize-matched null, preserving exact-condition failure rate at each stimulus A1-A10:')
print(stim_null_by_group.to_string(index=False, formatters={
    'expected_all_failed_analytic': lambda v: f'{v:.3f}',
    'expected_all_failed_boot_mean': lambda v: f'{v:.3f}',
    'null_95ci_low': lambda v: f'{v:.1f}',
    'null_95ci_high': lambda v: f'{v:.1f}',
    'observed_minus_expected': lambda v: f'{v:.3f}',
    'upper_tail_p_observed_ge_null': lambda v: f'{v:.4g}',
}))
print(
    f"\nTotal observed all-bouton failed events={stim_null_total['observed_total']} | "
    f"expected={stim_null_total['expected_total_analytic']:.3f} | "
    f"bootstrap mean={stim_null_total['expected_total_boot_mean']:.3f} "
    f"[95% CI {stim_null_total['null_95ci_low']:.1f}, {stim_null_total['null_95ci_high']:.1f}] | "
    f"upper-tail p={stim_null_total['upper_tail_p_observed_ge_null']:.4g} "
    f"({stim_null_total['n_boot']} size-matched draws)"
)
print('\nFailure rates used by null, by exact condition and stimulus:')
print(stim_failure_rate_by_group_stim.to_string(index=False, formatters={'failure_rate': lambda v: f'{100.0 * v:.2f}%'}))
print(f"Saved pooled pie to {out_stim_failure}")
print(f"Saved grouped pies to {out_stim_failure_groups}")
print(f"Saved matched-null panel to {out_stim_failure_null}")


## Supp — Quantal & failure detection method (technical figures)


In [ ]:
# === Supp method — Binomial grid search space ===
# Visualize the binomial grid-search space used by the methodological fitting procedure.

from math import comb
from matplotlib.colors import ListedColormap, BoundaryNorm

N_vals = np.arange(1, 16)                # 1 .. 15
P_vals = np.linspace(0.01, 0.50, 200)    # 0.01 .. 0.50

median_success_grid = np.full((len(P_vals), len(N_vals)), np.nan)

for i, p in enumerate(P_vals):
    for j, N in enumerate(N_vals):
        k = np.arange(0, N + 1)
        pmf = np.array([comb(N, kk) * (p ** kk) * ((1 - p) ** (N - kk)) for kk in k], float)

        succ_mask = k >= 1
        succ_k = k[succ_mask]
        succ_pmf = pmf[succ_mask]

        if succ_pmf.sum() <= 0:
            continue

        succ_pmf = succ_pmf / succ_pmf.sum()
        succ_cdf = np.cumsum(succ_pmf)
        median_success_k = succ_k[np.searchsorted(succ_cdf, 0.5)]

        median_success_grid[i, j] = min(median_success_k, 10)

fig, ax = plt.subplots(figsize=(7.2, 4.8))

x_edges = np.arange(N_vals.min() - 0.5, N_vals.max() + 1.5, 1.0)
p_step = P_vals[1] - P_vals[0]
y_edges = np.r_[P_vals - 0.5 * p_step, P_vals[-1] + 0.5 * p_step]

colors = [
    '#2e7d32',  # 1
    '#66bb6a',  # 2
    '#c0ca33',  # 3
    '#fdd835',  # 4
    '#ffb300',  # 5
    '#fb8c00',  # 6
    '#f4511e',  # 7
    '#e53935',  # 8
    '#8e24aa',  # 9
    '#546e7a',  # 10
]
cmap = ListedColormap(colors)
bounds = np.arange(0.5, 10.6, 1.0)
norm = BoundaryNorm(bounds, cmap.N)

mesh = ax.pcolormesh(
    x_edges,
    y_edges,
    median_success_grid,
    cmap=cmap,
    norm=norm,
    shading='auto'
)

cbar = plt.colorbar(mesh, ax=ax, ticks=np.arange(1, 11))
cbar.set_label('Median success count | K≥1')

style_ax(
    ax,
    'N',
    'Release probability p',
    'Median success count conditioned on success'
)

ax.set_xlim(N_vals.min() - 0.5, N_vals.max() + 0.5)
ax.set_ylim(0, 0.5)

plt.tight_layout()
plt.show()

fig.savefig(OUTPUT_DIR / 'Supp_method_binomial_grid_search_space.pdf', dpi=300, bbox_inches='tight')


In [ ]:
# === Supp method — Q & failure method technical figure ===
# Assemble the methodological panels that explain the Q-estimation and failure-calling procedure.

import json
import os
import subprocess
import sys
from pathlib import Path

from scipy.signal import savgol_filter
from scipy.stats import norm

panel1_condition = 'Theo_1_5Ca'
panel1_file = '20200909_linescan1_20Hz_10pulses_1.5mMCa_bouton4_traces_converted'
panel1_trial_col = '0'
panel1_trial_input_col_1based = int(panel1_trial_col) + 1

DISPLAY_STIMS = 'stim1'   # 'stim1' or 'all'

def _sanitize_token(value):
    return ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in str(value))

def _amp_min_from_row(row, k0):
    vals = []
    cc = _amp_corr[k0]
    uc = _amp_uncorr[k0]
    if cc in row.index and np.isfinite(row[cc]):
        vals.append(float(row[cc]))
    if uc in row.index and np.isfinite(row[uc]):
        vals.append(float(row[uc]))
    return float(np.min(vals)) if vals else np.nan

def _amp_uncorr_from_row(row, k0):
    uc = _amp_uncorr[k0]
    if uc in row.index and np.isfinite(row[uc]):
        return float(row[uc])
    cc = _amp_corr[k0]
    if cc in row.index and np.isfinite(row[cc]):
        return float(row[cc])
    return np.nan

REPO_ROOT = Path.cwd()
helper_script = REPO_ROOT / 'Feature_extraction' / 'tmp_selected_recording_nnls_overlay.py'
overlay_npz = REPO_ROOT / (
    f"_tmp_nnls_overlay__{_sanitize_token(panel1_condition)}__"
    f"{_sanitize_token(panel1_file)}.npz"
)

def _overlay_npz_matches(npz_path, condition, file_stem):
    if not npz_path.exists():
        return False
    try:
        d = np.load(npz_path, allow_pickle=True)
        required = {
            'condition', 'file_stem', 'time_s',
            'all_trial_input_col_1based', 'all_trial_yproc', 'all_trial_yhat',
            'all_trial_null_amps_nnls',
            'average_yproc', 'average_yhat',
        }
        ok = (
            required.issubset(set(d.files))
            and str(d['condition']) == str(condition)
            and str(d['file_stem']) == str(file_stem)
        )
        d.close()
        return ok
    except Exception:
        return False

if not _overlay_npz_matches(overlay_npz, panel1_condition, panel1_file):
    env = dict(os.environ)
    env['PYTHONIOENCODING'] = 'utf-8'
    subprocess.run(
        [
            sys.executable,
            str(helper_script),
            '--condition', panel1_condition,
            '--file', f'{panel1_file}.xlsx',
            '--trial', '1',
            '--out', str(overlay_npz),
        ],
        check=True,
        cwd=str(REPO_ROOT),
        env=env,
    )

overlay_data = np.load(overlay_npz, allow_pickle=True)
overlay_time = np.asarray(overlay_data['time_s'], float)
all_trial_input_col_1based = np.asarray(overlay_data['all_trial_input_col_1based'], int)
all_trial_yproc = np.asarray(overlay_data['all_trial_yproc'], float)
all_trial_yhat = np.asarray(overlay_data['all_trial_yhat'], float)
all_trial_null_amps_nnls = np.asarray(overlay_data['all_trial_null_amps_nnls'], dtype=object)
panel1_average_yproc = np.asarray(overlay_data['average_yproc'], float)
panel1_average_yhat = np.asarray(overlay_data['average_yhat'], float)
overlay_data.close()

trial_yproc_map = {
    int(k): np.asarray(v, float)
    for k, v in zip(all_trial_input_col_1based, all_trial_yproc)
}
trial_yhat_map = {
    int(k): np.asarray(v, float)
    for k, v in zip(all_trial_input_col_1based, all_trial_yhat)
}

panel1_trial_yproc = trial_yproc_map[panel1_trial_input_col_1based]
panel1_trial_yhat = trial_yhat_map[panel1_trial_input_col_1based]
panel1_trial_sg = savgol_filter(panel1_trial_yproc, 9, 2, mode='interp')

panel1_path = BASE_DIR / panel1_condition / f'{panel1_file}.xlsx'
panel1_df = pd.read_excel(panel1_path, sheet_name='Traces DF_F0')
panel1_time_raw = panel1_df['Time'].to_numpy(float)
panel1_trace_raw = panel1_df[panel1_trial_col].to_numpy(float)

bouton_trials = (
    trials_for_q[
        (trials_for_q['condition'] == panel1_condition) &
        (trials_for_q['file'] == panel1_file)
    ]
    .copy()
    .sort_values('trial_input_col_1based')
)

trial_row = bouton_trials[
    bouton_trials['trial_input_col_1based'] == panel1_trial_input_col_1based
].iloc[0]

thr_selected = float(trial_row['thr_shared'])
bouton_thr = bouton_trials['thr_shared'].to_numpy(float)
bouton_thr = bouton_thr[np.isfinite(bouton_thr)]
thr_bouton = float(np.nanmedian(bouton_thr)) if bouton_thr.size else np.nan

stim_times = 0.5 + 0.05 * np.arange(N_STIM)
pulse_indices = [0] if str(DISPLAY_STIMS).lower() == 'stim1' else list(range(N_STIM))
title_suffix = 'stim 1 only' if pulse_indices == [0] else 'all stimuli'
pool_label = 'stim 1' if pulse_indices == [0] else 'all events'

# Selected trial exact event classification
selected_fail_mask, _ = get_failure_mask_row(trial_row)
selected_event_fail = np.full(N_STIM, False)
for k in range(N_STIM):
    selected_event_fail[k] = bool(np.isfinite(selected_fail_mask[k]) and (selected_fail_mask[k] > 0))

# All-trial structures
bouton_trial_traces_sg = []
pooled_amp = []
pooled_is_fail = []
success_pct_by_pulse = np.full(N_STIM, np.nan)

all_fail_masks = []
for _, row in bouton_trials.iterrows():
    trial_1based = int(row['trial_input_col_1based'])
    if trial_1based not in trial_yproc_map:
        continue

    trace_proc = trial_yproc_map[trial_1based]
    trace_sg = savgol_filter(trace_proc, 9, 2, mode='interp')
    fail_mask, _ = get_failure_mask_row(row)

    bouton_trial_traces_sg.append(trace_sg)
    all_fail_masks.append(np.asarray(fail_mask, float))

    for k in pulse_indices:
        amp_k = _amp_min_from_row(row, k)
        is_fail_k = bool(np.isfinite(fail_mask[k]) and (fail_mask[k] > 0)) if len(fail_mask) > k else False
        if np.isfinite(amp_k):
            pooled_amp.append(amp_k)
            pooled_is_fail.append(is_fail_k)

bouton_trial_traces_sg = np.asarray(bouton_trial_traces_sg, float)
pooled_amp = np.asarray(pooled_amp, float)
pooled_is_fail = np.asarray(pooled_is_fail, bool)

if len(all_fail_masks):
    all_fail_masks = np.asarray(all_fail_masks, float)
    for k in range(N_STIM):
        valid = np.isfinite(all_fail_masks[:, k])
        if np.any(valid):
            success_pct_by_pulse[k] = 100.0 * np.mean(all_fail_masks[valid, k] == 0)

success_amps = pooled_amp[np.isfinite(pooled_amp) & (~pooled_is_fail)]
failure_amps = pooled_amp[np.isfinite(pooled_amp) & pooled_is_fail]
median_success_amp = float(np.nanmedian(success_amps)) if success_amps.size else np.nan

# Pooled NNLS null from all trials of the same bouton workbook
panel2_null = []
for arr in all_trial_null_amps_nnls:
    arr = np.asarray(arr, float)
    arr = arr[np.isfinite(arr)]
    if arr.size:
        panel2_null.append(arr)
panel2_null = np.concatenate(panel2_null) if panel2_null else np.array([], float)

# Panel 4 source rebuilt directly from trials
if pulse_indices == [0]:
    q_source_df = trials_for_q[
        trials_for_q['condition'].isin(get_calcium_conditions('1.5mM', 'all'))
    ].copy()
    q_source_df = q_source_df[q_source_df['status'].astype(str).str.strip().str.lower() == 'success']
    q_vals = q_source_df['AMP1_UNCORR'].dropna().to_numpy(float)
    q_title = f'Q from 1.5 mM stim 1 successes : n={len(q_vals)}'
    fit_cap = 0.75
else:
    q_vals = []
    q_source_df = trials_for_q[
        trials_for_q['condition'].isin(get_calcium_conditions('1.5mM', 'all'))
    ].copy()

    for _, row in q_source_df.iterrows():
        fail_mask, _ = get_failure_mask_row(row)
        for k in range(N_STIM):
            if k >= len(fail_mask):
                continue
            if not np.isfinite(fail_mask[k]) or fail_mask[k] > 0:
                continue
            amp_k = _amp_uncorr_from_row(row, k)
            if np.isfinite(amp_k) and amp_k > 0:
                q_vals.append(float(amp_k))

    q_vals = np.asarray(q_vals, float)
    q_title = f'Aggregated successful 1.5 mM events : n={len(q_vals)}'
    fit_cap = float(np.nanpercentile(q_vals, 95)) if q_vals.size else np.nan

q_vals = q_vals[np.isfinite(q_vals) & (q_vals > 0)]
q_mean_panel = float(np.nanmean(q_vals)) if q_vals.size else np.nan
q_median_panel = float(np.nanmedian(q_vals)) if q_vals.size else np.nan

fig = plt.figure(figsize=(10.8, 7.8))
gs = fig.add_gridspec(2, 2)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])

baseline_kw = dict(color='black', lw=1.3, linestyle=(0, (1.2, 2.0)), alpha=1.0, zorder=20)
thr_kw = dict(color='#c62828', lw=1.5, linestyle=(0, (1.2, 2.0)), alpha=1.0, zorder=21)

# Panel 1: selected trial
ax1.plot(panel1_time_raw, panel1_trace_raw, color='0.78', lw=0.9, alpha=0.95, zorder=1)
ax1.plot(overlay_time, panel1_trial_sg, color='black', lw=1.1, zorder=2)
ax1.plot(overlay_time, panel1_trial_yhat, color='#f28e2b', lw=1.8, zorder=8)
ax1.axvspan(0.4, 0.5, color='0.94', zorder=0)

ymin1 = np.nanmin(np.r_[panel1_trace_raw, panel1_trial_sg, panel1_trial_yhat])
ymax1 = np.nanmax(np.r_[panel1_trace_raw, panel1_trial_sg, panel1_trial_yhat])
ypad1 = 0.08 * (ymax1 - ymin1 if ymax1 > ymin1 else 1.0)
ax1.set_ylim(ymin1 - ypad1, ymax1 + ypad1)
ax1.set_xlim(0.4, 1.0)

tick_base1 = ax1.get_ylim()[0] + 0.04 * (ax1.get_ylim()[1] - ax1.get_ylim()[0])
tick_top1 = ax1.get_ylim()[0] + 0.10 * (ax1.get_ylim()[1] - ax1.get_ylim()[0])
for stim_time in stim_times:
    ax1.plot([stim_time, stim_time], [tick_base1, tick_top1], color='black', lw=1.0, solid_capstyle='butt', zorder=5)

event_pre_s = 0.003
event_post_s = 0.030
for k in pulse_indices:
    stim_time = stim_times[k]
    mask = (overlay_time >= stim_time - event_pre_s) & (overlay_time <= stim_time + event_post_s)
    if not np.any(mask):
        continue
    color = '#c62828' if selected_event_fail[k] else '#2e7d32'
    ax1.plot(overlay_time[mask], panel1_trial_sg[mask], color=color, lw=2.2, zorder=7)

style_ax(ax1, 'Time (s)', 'ΔF/F0', f'Single trial with NNLS fit (trial {panel1_trial_input_col_1based}, {title_suffix})')
ax1.hlines(0.0, xmin=0.4, xmax=1.0, **baseline_kw)
ax1.hlines(thr_selected, xmin=0.4, xmax=1.0, **thr_kw)

# Panel 2: all trials, SavGol only + success percentages
ax2.axvspan(0.4, 0.5, color='0.94', zorder=0)
for trace_sg in bouton_trial_traces_sg:
    ax2.plot(overlay_time, trace_sg, color='0.75', lw=0.8, alpha=0.55, zorder=1)

avg_sg = savgol_filter(panel1_average_yproc, 9, 2, mode='interp')
ax2.plot(overlay_time, avg_sg, color='black', lw=1.3, zorder=6)
ax2.plot(overlay_time, panel1_average_yhat, color='#f28e2b', lw=2.0, zorder=7)

ymin2 = np.nanmin(np.r_[bouton_trial_traces_sg.ravel(), panel1_average_yhat])
ymax2 = np.nanmax(np.r_[bouton_trial_traces_sg.ravel(), panel1_average_yhat])
ypad2 = 0.08 * (ymax2 - ymin2 if ymax2 > ymin2 else 1.0)
ax2.set_ylim(ymin2 - ypad2, ymax2 + ypad2)
ax2.set_xlim(0.4, 1.0)

tick_base2 = ax2.get_ylim()[0] + 0.04 * (ax2.get_ylim()[1] - ax2.get_ylim()[0])
tick_top2 = ax2.get_ylim()[0] + 0.10 * (ax2.get_ylim()[1] - ax2.get_ylim()[0])
for stim_time in stim_times:
    ax2.plot([stim_time, stim_time], [tick_base2, tick_top2], color='black', lw=1.0, solid_capstyle='butt', zorder=5)

for k in pulse_indices:
    if not np.isfinite(success_pct_by_pulse[k]):
        continue
    win = (overlay_time >= stim_times[k] - event_pre_s) & (overlay_time <= stim_times[k] + event_post_s)
    if not np.any(win):
        continue
    y_peak = float(np.nanmax(panel1_average_yhat[win]))
    ax2.annotate(
        f'{success_pct_by_pulse[k]:.0f}%',
        xy=(stim_times[k], y_peak),
        xytext=(0, 6),
        textcoords='offset points',
        ha='center',
        va='bottom',
        fontsize=6,
        color='black',
        zorder=9,
        clip_on=False,
        bbox=dict(facecolor='white', edgecolor='none', alpha=0.7, pad=0.2),
    )

style_ax(ax2, 'Time (s)', 'ΔF/F0', f'All trials from same bouton ({title_suffix})')
ax2.hlines(0.0, xmin=0.4, xmax=1.0, **baseline_kw)
if np.isfinite(thr_bouton):
    ax2.hlines(thr_bouton, xmin=0.4, xmax=1.0, **thr_kw)

# Panel 3: pooled amplitudes
hist_vals = np.r_[panel2_null, pooled_amp[np.isfinite(pooled_amp)]]
xmax3 = max(
    np.nanmax(hist_vals) if hist_vals.size else 1.0,
    thr_bouton if np.isfinite(thr_bouton) else 0.0,
    median_success_amp if np.isfinite(median_success_amp) else 0.0,
) * 1.08
bins3 = np.linspace(0.0, xmax3, 24)

if panel2_null.size:
    ax3.hist(panel2_null, bins=bins3, density=True, color='0.75', edgecolor='none', alpha=0.70,
             label=f'NNLS null all trials (n={len(panel2_null)})')
if failure_amps.size:
    ax3.hist(failure_amps, bins=bins3, density=True, color='#c62828', edgecolor='none', alpha=0.45,
             label=f'{pool_label} failures (n={len(failure_amps)})')
if success_amps.size:
    ax3.hist(success_amps, bins=bins3, density=True, color='#2e7d32', edgecolor='none', alpha=0.45,
             label=f'{pool_label} successes (n={len(success_amps)})')

style_ax(ax3, 'Amplitude (ΔF/F0)', 'Density', f'Pooled bouton null and {pool_label} amplitudes')
ax3.set_xlim(-0.01, xmax3)
ax3.vlines(0.0, ymin=0, ymax=ax3.get_ylim()[1], **baseline_kw)
if np.isfinite(thr_bouton):
    ax3.vlines(thr_bouton, ymin=0, ymax=ax3.get_ylim()[1], **thr_kw)
if np.isfinite(median_success_amp):
    ax3.vlines(
        median_success_amp,
        ymin=0,
        ymax=ax3.get_ylim()[1],
        color='#1b5e20',
        lw=1.7,
        linestyle='-',
        alpha=1.0,
        zorder=22,
    )
add_legend(ax3)

# Panel 4: Q distribution rebuilt directly from trials
if q_vals.size:
    ax4.hist(q_vals, bins=15, color='steelblue', edgecolor='none', alpha=0.8, density=True)

fit_data = q_vals[q_vals <= fit_cap] if np.isfinite(fit_cap) else np.array([], float)
if len(fit_data) > 1:
    mu, std = norm.fit(fit_data)
    xf = np.linspace(0, q_vals.max() * 1.2, 200)
    ax4.plot(
        xf,
        norm.pdf(xf, mu, std),
        color='#0b4f8a',
        lw=2.0,
        alpha=1.0,
        zorder=10,
        solid_capstyle='round',
        label=f'Gauss μ={mu:.4f} σ={std:.4f}',
    )

if np.isfinite(q_mean_panel):
    ax4.axvline(q_mean_panel, color='red', lw=1.2, ls='--', label=f'mean={q_mean_panel:.4f}')
if np.isfinite(q_median_panel):
    ax4.axvline(q_median_panel, color='orange', lw=1.2, ls='-.', label=f'med={q_median_panel:.4f}')
if q_vals.size:
    ax4.plot(q_vals, -0.12 * np.ones_like(q_vals), '|k', ms=6, mew=0.8)
    ax4.set_xlim(0, q_vals.max() * 1.2)

style_ax(ax4, None, 'Density', q_title)
ax4.set_xlabel('Quantal size Q (ΔF/F₀)')
ax4.set_ylim(bottom=0)
add_legend(ax4)
if 'style_hist_axis' in globals():
    style_hist_axis(ax4)

# Re-assert the Gaussian style after global styling helpers
for line in ax4.lines:
    if 'Gauss μ=' in str(line.get_label()):
        line.set_color('#0b4f8a')
        line.set_alpha(1.0)
        line.set_linewidth(2.0)
        line.set_zorder(10)
        break


plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'Supp_method_q_failure_method_technical.pdf', dpi=300, bbox_inches='tight')
plt.show()



In [ ]:
# === Manuscript value ledger: recoverable values for paper update ===
# This cell is intentionally placed after the regular analysis sections and before Random Forest.

def _ledger_values(values):
    arr = np.asarray(list(values), dtype=float)
    return arr[np.isfinite(arr)]


def _ledger_mean_sd_text(values, unit=''):
    arr = _ledger_values(values)
    if arr.size == 0:
        return 'not available'
    suffix = f' {unit}' if unit else ''
    if arr.size == 1:
        return f'{arr[0]:.3g}{suffix}, n=1'
    return f'{np.nanmean(arr):.3g} +/- {np.nanstd(arr, ddof=1):.3g}{suffix}, n={arr.size}'


def _ledger_count(df, column='ID'):
    if df is None or column not in getattr(df, 'columns', []):
        return len(df) if df is not None else 0
    return int(df[column].dropna().astype(str).nunique())


def _ledger_col_summary(df, column, label=None, unit=''):
    if df is None or column not in getattr(df, 'columns', []):
        return f'{label or column}: not available'
    return f'{label or column}: {_ledger_mean_sd_text(df[column], unit=unit)}'


def _ledger_target_counts(df):
    if df is None:
        return 'target counts: not available'
    for column in ['Target', 'target', 'TargetIdentity', 'target_identity', 'PostSynapticTarget']:
        if column in df.columns:
            counts = df[column].dropna().astype(str).str.upper().value_counts()
            return 'target counts: ' + ', '.join(f'{idx}={int(val)}' for idx, val in counts.items())
    return 'target counts: not available'


MANUSCRIPT_VALUE_LEDGER = {}

wt_df = globals().get('PCA_Data_WT_Pooled')
if wt_df is None and 'pca_data' in globals() and isinstance(pca_data, dict):
    wt_df = pca_data.get('WT_pooled')

print('=== Manuscript value ledger: values recoverable from this notebook ===')

if 'DATASET_N_SUMMARY' in globals():
    MANUSCRIPT_VALUE_LEDGER['dataset_n_summary'] = DATASET_N_SUMMARY
    print('\nDataset N summary:')
    try:
        display(DATASET_N_SUMMARY)
    except Exception:
        print(DATASET_N_SUMMARY)
else:
    print('\nDataset N summary: not available')

print('\nWT pooled reference:')
if wt_df is not None:
    wt_n = _ledger_count(wt_df, 'ID')
    MANUSCRIPT_VALUE_LEDGER['wt_pooled_n'] = wt_n
    print(f'WT pooled boutons: n={wt_n}')
    for column, label in [('AMP1', 'A1'), ('AMP2', 'A2'), ('PPR2/1', 'PPR2/1'), ('PPR5/1', 'PPR5/1'), ('%Fail1', 'Fail1 percent')]:
        text = _ledger_col_summary(wt_df, column, label=label)
        MANUSCRIPT_VALUE_LEDGER[f'wt_{column}'] = text
        print(text)
    if '%Fail1' in wt_df.columns:
        fail1 = pd.to_numeric(wt_df['%Fail1'], errors='coerce')
        active = fail1 < 100
        valid = fail1.notna()
        if valid.any():
            active_pct = 100.0 * active[valid].mean()
            inactive_pct = 100.0 - active_pct
            MANUSCRIPT_VALUE_LEDGER['wt_active_percent'] = active_pct
            MANUSCRIPT_VALUE_LEDGER['wt_inactive_percent'] = inactive_pct
            print(f'Active by Fail1 < 100%: {active_pct:.1f}% active, {inactive_pct:.1f}% inactive, n={int(valid.sum())}')
    print(_ledger_target_counts(wt_df))
    if 'FiberID' in wt_df.columns:
        fiber_sizes = wt_df.groupby('FiberID')['ID'].nunique() if 'ID' in wt_df.columns else wt_df.groupby('FiberID').size()
        print(f'Fiber IDs represented: n={len(fiber_sizes)} fibers; boutons per fiber: {_ledger_mean_sd_text(fiber_sizes)}')
else:
    print('WT pooled dataframe: not available')

print('\nQ estimate:')
if 'Q_estimates' in globals():
    q_values = _ledger_values(Q_estimates.values())
    MANUSCRIPT_VALUE_LEDGER['q_n'] = int(q_values.size)
    MANUSCRIPT_VALUE_LEDGER['q_mean'] = float(np.nanmean(q_values)) if q_values.size else np.nan
    MANUSCRIPT_VALUE_LEDGER['q_median'] = float(np.nanmedian(q_values)) if q_values.size else np.nan
    MANUSCRIPT_VALUE_LEDGER['global_q'] = float(globals().get('Global_Q', np.nan))
    print(f"Q boutons: n={q_values.size}; mean={MANUSCRIPT_VALUE_LEDGER['q_mean']:.4f}; median={MANUSCRIPT_VALUE_LEDGER['q_median']:.4f}; Global_Q={MANUSCRIPT_VALUE_LEDGER['global_q']:.4f}")
else:
    print('Q_estimates: not available')

print('\nSignal quality and kinetics:')
kinetics_df = globals().get('FIGS2_GLU_T_METRICS_DF')
if kinetics_df is not None:
    for column, label, unit in [('SNR', 'SNR', ''), ('RiseTime_ms', 'rise time', 'ms'), ('DecayTau_ms', 'decay tau', 'ms')]:
        text = _ledger_col_summary(kinetics_df, column, label=label, unit=unit)
        MANUSCRIPT_VALUE_LEDGER[f'kinetics_{column}'] = text
        print(text)
else:
    print('FIGS2_GLU_T_METRICS_DF: not available')

print('\nPCA and clustering:')
if 'PCA_RESULTS' in globals() and 'explained_variance' in PCA_RESULTS:
    ev = np.asarray(PCA_RESULTS['explained_variance'], dtype=float)
    MANUSCRIPT_VALUE_LEDGER['pca_pc1_variance'] = float(ev[0]) if ev.size > 0 else np.nan
    MANUSCRIPT_VALUE_LEDGER['pca_pc2_variance'] = float(ev[1]) if ev.size > 1 else np.nan
    print(f'PC1 variance={100*ev[0]:.1f}%, PC2 variance={100*ev[1]:.1f}%, PC1+PC2={100*np.nansum(ev[:2]):.1f}%')
else:
    print('PCA explained variance: not available')
if 'N_CLUSTERS' in globals():
    MANUSCRIPT_VALUE_LEDGER['n_clusters'] = int(N_CLUSTERS)
    print(f'Cluster count used in notebook: {N_CLUSTERS}')
if 'cluster_assignments' in globals():
    cluster_counts = pd.Series(cluster_assignments).value_counts().sort_index()
    MANUSCRIPT_VALUE_LEDGER['cluster_counts'] = cluster_counts.to_dict()
    print('Cluster sizes: ' + ', '.join(f'C{int(k)}={int(v)}' for k, v in cluster_counts.items()))

print('\nPerturbation dataset counts:')
if 'pca_data' in globals() and isinstance(pca_data, dict):
    for key, value in pca_data.items():
        try:
            n_rows = int(len(value))
            print(f'{key}: n={n_rows}')
        except Exception:
            pass
else:
    print('pca_data dictionary: not available')

print('\nExplicitly external or not refreshed in this run:')
print('Morphology, immunohistochemistry, mouse/slice counts, and Random Forest validation are not recomputed by this notebook refresh.')

In [ ]:
# === Manuscript value ledger addendum: newly recovered missing stats ===
# Collect stats added during the thorough recovery pass.

MANUSCRIPT_VALUE_LEDGER.setdefault('recovered_missing_stats', {})
recovered = MANUSCRIPT_VALUE_LEDGER['recovered_missing_stats']

print('=== Manuscript value ledger addendum: newly recovered missing stats ===')

print('\nSupp Fig S4 basal fluorescence control:')
if 'SUPP_S4_F0_STATS' in globals():
    recovered['supp_s4_f0_stats'] = SUPP_S4_F0_STATS
    for metric_name, stat in SUPP_S4_F0_STATS.items():
        print(f"F0 vs {metric_name}: Pearson r={stat['pearson_r']:.3f}, p={stat['pearson_p']:.4g}, n={stat['n']}; Spearman rho={stat['spearman_r']:.3f}, p={stat['spearman_p']:.4g}")
else:
    print('SUPP_S4_F0_STATS: not available')

print('\nPCA PC2 feature-correlation manuscript stats:')
if 'PCA_PC2_MANUSCRIPT_STATS' in globals():
    recovered['pc2_manuscript_stats'] = PCA_PC2_MANUSCRIPT_STATS
    raw_amp = PCA_PC2_MANUSCRIPT_STATS['raw_PC2']['amplitudes']
    raw_fail = PCA_PC2_MANUSCRIPT_STATS['raw_PC2']['failures']
    oriented_amp = PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['amplitudes']
    oriented_fail = PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['failures']
    orient = PCA_PC2_MANUSCRIPT_STATS['synaptic_weight_axis']['orientation_multiplier']
    print(f"raw PC2 amplitudes: r={raw_amp['mean_r']:.3f} ± {raw_amp['sd_r']:.3f}; features={raw_amp['r_values']}")
    print(f"raw PC2 failures:   r={raw_fail['mean_r']:.3f} ± {raw_fail['sd_r']:.3f}; features={raw_fail['r_values']}")
    print(f"synaptic-weight axis multiplier: {orient:+.0f}")
    print(f"oriented amplitudes: r={oriented_amp['mean_r']:.3f} ± {oriented_amp['sd_r']:.3f}")
    print(f"oriented failures:   r={oriented_fail['mean_r']:.3f} ± {oriented_fail['sd_r']:.3f}")
else:
    print('PCA_PC2_MANUSCRIPT_STATS: not available')

print('\nFig 4 cumulative release, Q-normalized SV equivalents:')
if 'FIG4_QNORM_CUMULATIVE_STATS' in globals():
    recovered['fig4_qnorm_cumulative_stats'] = FIG4_QNORM_CUMULATIVE_STATS
    for lab, stat in FIG4_QNORM_CUMULATIVE_STATS.items():
        print(f"{lab}: released={stat['released_sv_mean']:.2f} ± {stat['released_sv_sd']:.2f} SV equivalents; refill={stat['refill_sv_per_stim']:.3f} SV/stim = {stat['refill_sv_per_ms']:.5f} SV/ms; n={stat['n']}")
else:
    print('FIG4_QNORM_CUMULATIVE_STATS: not available')

print('\nCluster framework:')
if 'N_CLUSTERS' in globals():
    recovered['n_clusters'] = int(N_CLUSTERS)
    print(f'Current notebook cluster framework: C1-C{int(N_CLUSTERS)}')
else:
    print('N_CLUSTERS: not available')

## Random Forest Validation

Random-forest classification is kept at the end as a validation step rather than as a primary biological analysis. It quantifies how reproducibly the WT bouton classes can be discriminated from the feature set used for clustering and provides a classifier-based check that the WT classes are not arbitrary partitions of the PCA space.


In [ ]:
# Random Forest validation is intentionally skipped for this manuscript value refresh.
SKIP_RANDOM_FOREST = True
print('Random Forest validation skipped for this notebook refresh.')

In [ ]:
# === Supp S7d / S7e / S7f — RF detailed panels at optimal k ===
# Run RF validation across cluster counts, but export the detailed RF figure only for the manuscript cluster count.

from scipy.cluster.hierarchy import fcluster
from scipy.stats import ttest_ind
from Func_RF import random_forest_classification
RF_RANDOM_STATE = 42

if not SKIP_RANDOM_FOREST:

    # Feature columns for RF classification
    feature_cols = ["AMP1", "AMP2", 
                    "PPR2/1", "PPR3/1", "PPR4/1", "PPR5/1", "PPR6/1",
                    "PPR7/1", "PPR8/1", "PPR9/1", "PPR10/1",
                    "%Fail1", "%Fail2"]

    # Store results for comparison
    rf_comparison = {}

    # Test cluster numbers from 2 to 10
    n_clusters_list = list(range(2, 11))  # [2, 3, 4, 5, 6, 7, 8, 9, 10]
    #n_clusters_list = [6]  # For quick testing, use only 6 clusters (comment out to run all)
    # Get AMP1 values for reordering clusters
    amp1_values = PCA_Data_WT_Pooled["AMP1"].values

    for n_clust in n_clusters_list:
        print(f"\n{'='*60}")
        print(f"  Running Random Forest with {n_clust} clusters")
        print(f"{'='*60}")
        
        # Create cluster assignments for this number of clusters
        cluster_raw = fcluster(linkage_matrix, n_clust, criterion='maxclust')
        
        # Reorder clusters by ascending mean AMP1 (same as cell 191)
        cluster_amp1_means = {}
        for cid in range(1, n_clust + 1):
            mask = cluster_raw == cid
            cluster_amp1_means[cid] = np.nanmean(amp1_values[mask])
        sorted_clusters = sorted(cluster_amp1_means.keys(), key=lambda c: cluster_amp1_means[c])
        cluster_map = {old: new for new, old in enumerate(sorted_clusters, start=1)}
        cluster_assignments = np.array([cluster_map[c] for c in cluster_raw])
        
        # Create temporary DataFrame with new cluster assignments
        df_temp = PCA_Data_WT_Pooled.copy()
        df_temp['HC_Cluster'] = cluster_assignments
        
        # Run RF classification - show the detailed figure only for the manuscript cluster count
        show_figure = (n_clust == N_CLUSTERS)
        
        fig, results = random_forest_classification(
            df_temp,
            feature_cols=feature_cols,
            target_col="HC_Cluster",
            random_state=RF_RANDOM_STATE,
            n_splits=5,
            n_iter=3,
            verbose=show_figure  # Show detailed output only for key cluster counts
        )
        
        # Save only the manuscript detailed RF panel set (normal, shuffled, score matrix)
        if n_clust == N_CLUSTERS:
            rf_detail_output = OUTPUT_DIR / f"FigS7_def_rf_detailed_k{n_clust}.pdf"
            fig.savefig(rf_detail_output, format='pdf', dpi=300, bbox_inches='tight')
            print(f"  Saved Supp S7d/e/f detailed RF figure to: {rf_detail_output}")
            plt.show()
        else:
            plt.close(fig)

        
        # Store results (compute std from scores if not in results)
        std_actual = results.get('std_actual', np.std(results['scores_actual']))
        std_shuffled = results.get('std_shuffled', np.std(results['scores_shuffled']))
        p_value = results.get('p_value', ttest_ind(results['scores_actual'], results['scores_shuffled'])[1])
        
        rf_comparison[n_clust] = {
            'accuracy_actual': results['accuracy_actual'],
            'accuracy_shuffled': results['accuracy_shuffled'],
            'std_actual': std_actual,
            'std_shuffled': std_shuffled,
            'p_value': p_value,
            'scores_actual': results['scores_actual'],
            'scores_shuffled': results['scores_shuffled'],
            'figure': fig
        }
        
        # Print short summary for all
        improvement = results['accuracy_actual'] - results['accuracy_shuffled']
        print(f"\n  Summary: {n_clust} clusters - Accuracy = {results['accuracy_actual']:.1%} ± {std_actual:.1%}")
        print(f"           Improvement = +{improvement:.1%}, p = {p_value:.2e}")

    print("\n" + "="*60)
    print("  All RF classifications completed (2-10 clusters)")
    if 'rf_detail_output' in globals():
        print(f"  Detailed manuscript RF figure saved to: {rf_detail_output}")
    print("="*60)


In [ ]:
# === Supp S7g — RF accuracy comparison bar plot and summary table ===
# Summarize random-forest accuracy metrics across tested cluster counts; the bar plot is the manuscript S7g panel.

if not SKIP_RANDOM_FOREST:

    fig_comparison, axes = make_figure_grid(1, 2, figsize=(12, 5))

    # --- Subplot 1: Bar chart of accuracies ---
    ax1 = axes[0]
    x_pos = np.arange(len(n_clusters_list))
    bar_width = 0.35

    # Actual accuracies
    acc_actual = [rf_comparison[n]['accuracy_actual'] for n in n_clusters_list]
    std_actual = [rf_comparison[n]['std_actual'] for n in n_clusters_list]
    bars1 = ax1.bar(x_pos - bar_width/2, acc_actual, bar_width, 
                    yerr=std_actual, label='Actual', color='#4CAF50', capsize=5, alpha=0.8)

    # Shuffled accuracies
    acc_shuffled = [rf_comparison[n]['accuracy_shuffled'] for n in n_clusters_list]
    std_shuffled = [rf_comparison[n]['std_shuffled'] for n in n_clusters_list]
    bars2 = ax1.bar(x_pos + bar_width/2, acc_shuffled, bar_width,
                    yerr=std_shuffled, label='Shuffled', color='#9E9E9E', capsize=5, alpha=0.8)

    ax1.set_xlabel('Number of Clusters')
    ax1.set_ylabel('Classification Accuracy')
    ax1.set_title('RF Classification: Actual vs Shuffled')
    ax1.set_xticks(x_pos)
    ax1.set_xticklabels([str(n) for n in n_clusters_list])
    add_legend(ax1, )
    ax1.set_ylim(0, 1.0)
    ax1.axhline(y=1/4, color='gray', linestyle='--', alpha=0.5, label='Chance (4cl)')
    ax1.axhline(y=1/5, color='gray', linestyle=':', alpha=0.5, label='Chance (5cl)')
    ax1.axhline(y=1/6, color='gray', linestyle='-.', alpha=0.5, label='Chance (6cl)')

    # Add value labels on bars
    for bar, val in zip(bars1, acc_actual):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03, 
                f'{val:.0%}', ha='center', va='bottom', fontsize=5)

    # --- Subplot 2: Improvement over chance ---
    ax2 = axes[1]
    improvement = [rf_comparison[n]['accuracy_actual'] - rf_comparison[n]['accuracy_shuffled'] 
                for n in n_clusters_list]
    colors_imp = ['#2196F3' if imp > 0 else '#F44336' for imp in improvement]
    bars3 = ax2.bar(x_pos, improvement, 0.6, color=colors_imp, alpha=0.8)

    ax2.set_xlabel('Number of Clusters')
    ax2.set_ylabel('Improvement over Shuffled')
    ax2.set_title('Classification Improvement')
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels([str(n) for n in n_clusters_list])
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

    for bar, val in zip(bars3, improvement):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
                f'+{val:.0%}', ha='center', va='bottom', fontsize=5, fontweight='bold')

    plt.tight_layout()

    # Save comparison figure to PDF
    fig_comparison.savefig(OUTPUT_DIR / "FigS7_g_rf_cluster_count_comparison.pdf", 
                        format='pdf', dpi=300, bbox_inches='tight')
    print(f"✓ Supp S7g comparison figure saved: {OUTPUT_DIR / 'FigS7_g_rf_cluster_count_comparison.pdf'}")

    plt.show()

    # --- Separate figure for summary table ---
    fig_table, ax_table = make_figure_grid(figsize=(10, 4))
    ax_table.axis('off')

    # Create summary table
    table_data = []
    for n in n_clusters_list:
        res = rf_comparison[n]
        row = [
            f"{n}",
            f"{res['accuracy_actual']:.1%} ± {res['std_actual']:.1%}",
            f"{res['accuracy_shuffled']:.1%} ± {res['std_shuffled']:.1%}",
            f"+{res['accuracy_actual'] - res['accuracy_shuffled']:.1%}",
            f"{res['p_value']:.2e}"
        ]
        table_data.append(row)

    columns = ['Clusters', 'Actual', 'Shuffled', 'Improvement', 'p-value']
    table = ax_table.table(cellText=table_data, colLabels=columns, loc='center', 
                        cellLoc='center', colColours=['#E0E0E0']*5)
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.4, 1.8)

    # Highlight best result
    best_idx = np.argmax(improvement)
    ax_table.set_title(f'Summary: Best = {n_clusters_list[best_idx]} clusters', 
                    fontsize=8, fontweight='bold', pad=20)

    plt.tight_layout()

    # Save table figure to PDF
    fig_table.savefig(OUTPUT_DIR / "TableS7_rf_summary_table.pdf", 
                    format='pdf', dpi=300, bbox_inches='tight')
    print(f"✓ RF summary table saved: {OUTPUT_DIR / 'TableS7_rf_summary_table.pdf'}")

    plt.show()

    # Print conclusion
    print("\n" + "="*60)
    print("  CONCLUSION")
    print("="*60)
    best_n = n_clusters_list[best_idx]
    best_acc = rf_comparison[best_n]['accuracy_actual']
    best_imp = rf_comparison[best_n]['accuracy_actual'] - rf_comparison[best_n]['accuracy_shuffled']
    print(f"\n  : Best number of clusters: {best_n}")
    print(f"  : Classification accuracy: {best_acc:.1%}")
    print(f"  : Improvement over chance: +{best_imp:.1%}")
    print(f"  : p-value: {rf_comparison[best_n]['p_value']:.4e}")
    print("\n  The higher the improvement over shuffled, the better the cluster separation.")
    print(f"\n  All PDF figures saved to: {OUTPUT_DIR}")


In [ ]:
# === Supp S7h (optional) ? RF focus on k = 5 and 6 cluster solutions ===
# Optional focused figure reusing rf_comparison to show that strong RF separation persists for the 5- and 6-cluster solutions.

if SKIP_RANDOM_FOREST:
    print('Optional RF focus figure skipped because Random Forest validation is disabled in this notebook refresh.')
elif 'rf_comparison' not in globals():
    raise RuntimeError('Run the RF screening cell first so rf_comparison is available.')
else:
    focus_clusters = [5, 6]
    missing_focus = [k for k in focus_clusters if k not in rf_comparison]
    if missing_focus:
        raise RuntimeError(f'Missing RF results for clusters {missing_focus}. Run the RF screening cell first.')

    summary_clusters = [k for k in (4, 5, 6) if k in rf_comparison]
    fig_focus, axes = make_figure_grid(1, 3, figsize=(12, 3.8))
    axes = np.ravel(axes)
    ax_summary, ax_k5, ax_k6 = axes

    actual_color = '#4CAF50'
    shuffled_color = '#9E9E9E'
    improvement_color = '#1F77B4'

    actual_vals = [rf_comparison[k]['accuracy_actual'] for k in summary_clusters]
    shuffled_vals = [rf_comparison[k]['accuracy_shuffled'] for k in summary_clusters]
    improvement_vals = [actual - shuffled for actual, shuffled in zip(actual_vals, shuffled_vals)]
    x_pos = np.arange(len(summary_clusters))

    ax_summary.plot(x_pos, actual_vals, marker='o', ms=6, lw=1.8, color=actual_color, label='Actual')
    ax_summary.plot(x_pos, shuffled_vals, marker='o', ms=6, lw=1.8, color=shuffled_color, label='Shuffled')
    ax_summary.bar(x_pos, improvement_vals, width=0.35, color=improvement_color, alpha=0.18)
    for idx, k in enumerate(summary_clusters):
        ax_summary.text(x_pos[idx], actual_vals[idx] + 0.03, f'{actual_vals[idx]:.0%}', ha='center', va='bottom', fontsize=6)
        ax_summary.text(x_pos[idx], shuffled_vals[idx] - 0.05, f'{shuffled_vals[idx]:.0%}', ha='center', va='top', fontsize=6, color='0.35')
        ax_summary.text(x_pos[idx], improvement_vals[idx] + 0.02, f'+{improvement_vals[idx]:.0%}', ha='center', va='bottom', fontsize=6, color=improvement_color)
    ax_summary.set_xticks(x_pos)
    ax_summary.set_xticklabels([str(k) for k in summary_clusters])
    ax_summary.set_xlabel('Cluster count')
    ax_summary.set_ylabel('Balanced accuracy')
    ax_summary.set_title('RF signal stays high at k = 5-6')
    ax_summary.set_ylim(0, 1.0)
    ax_summary.grid(False)
    add_legend(ax_summary)

    for ax, n_clust in zip((ax_k5, ax_k6), focus_clusters):
        actual_scores = np.asarray(rf_comparison[n_clust]['scores_actual'], dtype=float)
        shuffled_scores = np.asarray(rf_comparison[n_clust]['scores_shuffled'], dtype=float)
        box = ax.boxplot(
            [actual_scores, shuffled_scores],
            positions=[1, 2],
            widths=0.55,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color='black', linewidth=1.0),
            whiskerprops=dict(color='0.35', linewidth=1.0),
            capprops=dict(color='0.35', linewidth=1.0),
        )
        for patch, color in zip(box['boxes'], (actual_color, shuffled_color)):
            patch.set_facecolor(color)
            patch.set_alpha(0.55)
            patch.set_edgecolor('0.35')

        jitter_rng = np.random.default_rng(RF_RANDOM_STATE + n_clust)
        for xpos, scores, color in ((1, actual_scores, actual_color), (2, shuffled_scores, shuffled_color)):
            jitter = jitter_rng.uniform(-0.08, 0.08, size=len(scores))
            ax.scatter(np.full(len(scores), xpos) + jitter, scores, s=14, alpha=0.8, color=color, edgecolors='none')

        chance_level = 1.0 / n_clust
        improvement = rf_comparison[n_clust]['accuracy_actual'] - rf_comparison[n_clust]['accuracy_shuffled']
        p_value = rf_comparison[n_clust]['p_value']
        ax.axhline(chance_level, color='0.45', linestyle=':', linewidth=1.0)
        ax.set_xticks([1, 2])
        ax.set_xticklabels(['Actual', 'Shuffled'])
        ax.set_ylim(0, 1.0)
        ax.set_title(f'k = {n_clust}')
        ax.set_ylabel('Balanced accuracy')
        ax.grid(False)
        ax.text(
            0.04,
            0.96,
            f'Improvement = +{improvement:.1%}\np = {p_value:.1e}\nchance = {chance_level:.1%}',
            transform=ax.transAxes,
            va='top',
            ha='left',
            fontsize=6,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='0.8'),
        )

    plt.tight_layout()
    out_focus = OUTPUT_DIR / 'FigS7_h_optional_rf_focus_k5_k6.pdf'
    fig_focus.savefig(out_focus, format='pdf', dpi=300, bbox_inches='tight')
    plt.show()

    print('Focused RF summary for the 4/5/6-cluster comparison:')
    for n_clust in summary_clusters:
        res = rf_comparison[n_clust]
        print(
            f"  k = {n_clust}: actual = {res['accuracy_actual']:.1%} ? {res['std_actual']:.1%}, "
            f"shuffled = {res['accuracy_shuffled']:.1%} ? {res['std_shuffled']:.1%}, "
            f"improvement = +{res['accuracy_actual'] - res['accuracy_shuffled']:.1%}, p = {res['p_value']:.2e}"
        )
    print(f'? Optional RF focus figure saved to {out_focus}')


In [ ]:
# === Aux — WT Fail1 vs binomial predicted zero-probability ===
# Compare the empirical stim-1 failure map to the binomial model's predicted zero-count probability.

if 'PCA_Data_WT_Pooled_clustered' not in globals() or 'D_N' not in globals() or 'D_P' not in globals():
    raise RuntimeError('Run the WT pooled PCA and MLE helper cells first.')

fail_comp = PCA_Data_WT_Pooled_clustered[['ID', '%Fail1']].copy().reset_index(drop=True)
fail_comp['ObservedFail1'] = pd.to_numeric(fail_comp['%Fail1'], errors='coerce') / 100.0
fail_comp['PredFail1'] = np.where(
    np.isfinite(D_N[:, 0]) & np.isfinite(D_P[:, 0]),
    np.power(1.0 - D_P[:, 0], D_N[:, 0]),
    np.nan,
)
fail_comp['Pred/Obs'] = np.where(
    fail_comp['ObservedFail1'] > 0,
    fail_comp['PredFail1'] / fail_comp['ObservedFail1'],
    np.nan,
)
fail_comp['Δ(pred-obs)'] = fail_comp['PredFail1'] - fail_comp['ObservedFail1']

ok_all = np.isfinite(fail_comp['ObservedFail1']) & np.isfinite(fail_comp['PredFail1'])
ok_ratio = ok_all & (fail_comp['ObservedFail1'] > 0)

print(f"Stim 1 valid boutons: {int(ok_all.sum())}")
print(f"Observed %Fail1 mean: {100*np.nanmean(fail_comp.loc[ok_all, 'ObservedFail1']):.1f}%")
print(f"Predicted (1-P)^N mean: {100*np.nanmean(fail_comp.loc[ok_all, 'PredFail1']):.1f}%")
print(f"Mean Δ(pred-obs): {100*np.nanmean(fail_comp.loc[ok_all, 'Δ(pred-obs)']):+.1f} percentage points")
print(f"Median pred/obs (excluding observed 0): {np.nanmedian(fail_comp.loc[ok_ratio, 'Pred/Obs']):.3f}")
print(f"Pearson r(obs, pred): {fail_comp.loc[ok_all, ['ObservedFail1', 'PredFail1']].corr().iloc[0,1]:.3f}")

fig_cmp, axes_cmp = make_figure_grid(1, 2, panel_kind='simple', squeeze=False)
ax1, ax2 = axes_cmp[0, 0], axes_cmp[0, 1]

ax1.scatter(
    fail_comp.loc[ok_all, 'ObservedFail1'],
    fail_comp.loc[ok_all, 'PredFail1'],
    s=10,
    alpha=0.8,
    color=get_wt_ca_color('2.5mM'),
    edgecolors='none',
)
lims = [0, max(0.01, float(np.nanmax(np.r_[fail_comp.loc[ok_all, 'ObservedFail1'], fail_comp.loc[ok_all, 'PredFail1']])))]
ax1.plot(lims, lims, color='k', lw=0.8, ls=':')
style_ax(ax1, xlabel='Observed %Fail1', ylabel='Predicted (1-P)^N', title='Stim 1 failure comparison')
ax1.set_xlim(lims)
ax1.set_ylim(lims)

ratio_vals = fail_comp.loc[ok_ratio, 'Pred/Obs'].to_numpy(dtype=float)
ratio_hi = max(1.8, float(np.nanpercentile(ratio_vals, 95))) if ratio_vals.size else 1.8
ax2.hist(ratio_vals, bins=np.linspace(0, ratio_hi, 18), color=get_wt_ca_color('2.5mM'), alpha=0.85)
ax2.axvline(1.0, color='k', lw=0.8, ls=':')
ax2.axvline(np.nanmedian(ratio_vals), color='k', lw=1.0)
style_ax(ax2, xlabel='Predicted / observed fail1', ylabel='Boutons', title='Stim 1 ratio')

finalize_figure(
    fig_cmp,
    title='Aux WT Fail1 vs binomial zero-probability',
    rect=[0, 0, 1, 0.96],
    tight_layout_kwargs={'pad': 0.18, 'w_pad': 0.18},
)
plt.show()
